In [1]:
# ==================================================================================================
# PROJECT 22 — CELL 1 / STEP 0
# NEW-NOTEBOOK POST-PROJECT-21 BOOTSTRAP AND CANDIDATE DISCOVERY
#
# RUN THIS AS CELL 1 IN A FRESH COLAB NOTEBOOK:
#   Thesis_project_22.ipynb
#
# PROJECT 21 IS COMPLETE_AND_FROZEN AND MUST NOT BE RERUN.
#
# SAFETY:
# - validates the frozen 21-project completion registry and Project 21 completion checkpoint;
# - reads but never modifies the completion registry;
# - writes only Project 22 bootstrap/selection files;
# - never reads or modifies any prior-project condition-output files;
# - does not inject noise, reconstruct REC features, fit models, or start an experiment;
# - prepares the four remaining projects for runtime-prioritized selection in Step 1A.
# ==================================================================================================

from google.colab import drive

from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import shutil
import tarfile

import pandas as pd


print("=" * 136)
print("=== PROJECT 22 CELL 1 / STEP 0: NEW-NOTEBOOK POST-PROJECT-21 BOOTSTRAP ===")
print("=" * 136)


PROJECT_NUMBER = 22

STEP0_STATUS = (
    "PASS_PROJECT_22_NEW_NOTEBOOK_RUNTIME_BOOTSTRAPPED_AND_CANDIDATES_DISCOVERED"
)

EXPECTED_ARCHIVE_SHA256 = (
    "92af159c116e06e98d7c8348605adb6e9acb25aad982a1ff9fbef7103cb9e36e"
)

EXPECTED_REGISTRY_SHA256 = (
    "79cd6ecb595c5e8ae91a9494e469792716338d144308560a62caf1b9342306b2"
)

EXPECTED_REGISTERED_PROJECTS = 21
EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

ACTIVE_RESERVED_PROJECTS = {}

EXPECTED_CANDIDATES = 4

REQUIRED_PROJECT_FILES = {
    "builds.csv",
    "exe.csv",
    "dataset.csv",
    "id_map.csv",
    "entity_change_history.csv",
}

RUNTIME_PRIORITY_POLICY = {
    "purpose":
        "processing order only; protocol eligibility and final project set are unchanged",
    "primary":
        "ModelTrainingRows ascending",
    "secondary":
        "ModelEvaluationRows ascending",
    "tertiary":
        "RawExecutionRows ascending",
    "final_tie_break":
        "Project ascending",
    "scientific_effect":
        "none when all protocol-eligible projects are completed",
}


drive.mount(
    "/content/drive",
    force_remount=False,
)

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

ARCHIVE_PATH = (
    THESIS_ROOT
    / "Data"
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)

REGISTRY_PATH = (
    THESIS_ROOT
    / "Notes"
    / "completed_project_registry.csv"
)

PROJECT_21_STEP5C_CHECKPOINT_PATH = (
    THESIS_ROOT
    / "Notes"
    / "project_21_step5c_checkpoint.json"
)

EXPECTED_PROJECT_21_STEP5C_SHA256 = (
    "77832b3a2f5151122d17c46dcb46bb18572e814708820ae673961d044e1433cf"
)

LOCAL_EXTRACTION_ROOT = Path(
    "/content/datasets"
)

LOCAL_DATASET_ROOT = (
    LOCAL_EXTRACTION_ROOT
    / "datasets"
)

SELECTION_ROOT = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / "project_22_selection"
)

BOOTSTRAP_INVENTORY_PATH = (
    SELECTION_ROOT
    / "project_22_bootstrap_candidate_inventory.csv"
)

BOOTSTRAP_REPORT_PATH = (
    SELECTION_ROOT
    / "project_22_step0_report.json"
)

BOOTSTRAP_STATUS_PATH = (
    SELECTION_ROOT
    / "project_22_step0_status.json"
)


def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def resolve_column(
    columns,
    *candidates,
):
    normalized = {
        str(column).strip().lower():
            column
        for column in columns
    }

    for candidate in candidates:
        key = str(
            candidate
        ).strip().lower()

        if key in normalized:
            return normalized[
                key
            ]

    raise RuntimeError(
        "Could not resolve any of these columns: "
        + ", ".join(
            candidates
        )
    )


def atomic_write_text(
    path,
    text,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        text,
        encoding="utf-8",
    )

    temporary_path.replace(
        path
    )


def atomic_write_json(
    path,
    payload,
):
    atomic_write_text(
        path,
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
            default=str,
        )
        + "\n",
    )


def atomic_write_csv(
    path,
    frame,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    frame.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(
        path
    )


def extract_archive_safely(
    archive_path,
    extraction_root,
):
    extraction_root = Path(
        extraction_root
    )

    extraction_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    resolved_root = extraction_root.resolve()
    extracted_files = 0

    with tarfile.open(
        archive_path,
        mode="r:gz",
    ) as archive:
        for member in archive:
            member_name = (
                member.name
                .replace(
                    "\\",
                    "/",
                )
                .lstrip(
                    "/"
                )
            )

            target_path = (
                extraction_root
                / member_name
            )

            resolved_target = target_path.resolve()

            if (
                resolved_target
                != resolved_root
                and resolved_root
                not in resolved_target.parents
            ):
                raise RuntimeError(
                    "Unsafe archive member encountered:\n"
                    f"{member.name}"
                )

            if member.isdir():
                target_path.mkdir(
                    parents=True,
                    exist_ok=True,
                )

            elif member.isfile():
                target_path.parent.mkdir(
                    parents=True,
                    exist_ok=True,
                )

                source_handle = archive.extractfile(
                    member
                )

                if source_handle is None:
                    raise RuntimeError(
                        "Could not read archive member:\n"
                        f"{member.name}"
                    )

                with (
                    source_handle,
                    target_path.open(
                        "wb"
                    ) as output_handle,
                ):
                    shutil.copyfileobj(
                        source_handle,
                        output_handle,
                        length=8 * 1024 * 1024,
                    )

                extracted_files += 1

    return extracted_files


required_drive_paths = [
    ARCHIVE_PATH,
    REGISTRY_PATH,
    PROJECT_21_STEP5C_CHECKPOINT_PATH,
]

missing_drive_paths = [
    str(
        path
    )
    for path in required_drive_paths
    if not path.is_file()
]

if missing_drive_paths:
    raise FileNotFoundError(
        "Required Project 22 bootstrap inputs are missing:\n"
        + "\n".join(
            missing_drive_paths
        )
    )


archive_sha256 = sha256_file(
    ARCHIVE_PATH
)

if archive_sha256 != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        "Frozen TCP-CI archive SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_ARCHIVE_SHA256}\n"
        f"Actual:   {archive_sha256}"
    )


project_21_step5c_sha256 = sha256_file(
    PROJECT_21_STEP5C_CHECKPOINT_PATH
)

if project_21_step5c_sha256 != EXPECTED_PROJECT_21_STEP5C_SHA256:
    raise RuntimeError(
        "Project 21 completion checkpoint SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_PROJECT_21_STEP5C_SHA256}\n"
        f"Actual:   {project_21_step5c_sha256}"
    )

project_21_step5c_checkpoint = json.loads(
    PROJECT_21_STEP5C_CHECKPOINT_PATH.read_text(encoding="utf-8")
)

if project_21_step5c_checkpoint.get("Status") != (
    "PASS_PROJECT_21_FINAL_PACKAGE_FROZEN_AND_REGISTERED"
):
    raise RuntimeError(
        "Project 21 completion checkpoint is not in the expected PASS state."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs from the frozen Projects 1–21 state.\n"
        "Do not continue Project 22 until the unexpected registry change is investigated.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = pd.read_csv(
    REGISTRY_PATH,
    low_memory=False,
)


registry_project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "Project Number",
    "Project_Number",
)

registry_project_column = resolve_column(
    registry.columns,
    "Project",
)

registry_status_column = resolve_column(
    registry.columns,
    "Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        registry_project_number_column
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Completion registry must contain exactly frozen Projects 1–21."
    )


if not registry[
    registry_status_column
].astype(
    str
).eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Not every registered predecessor is COMPLETE_AND_FROZEN."
    )


if registry_project_numbers.eq(
    PROJECT_NUMBER
).any():
    raise RuntimeError(
        "Project 22 is unexpectedly already registered."
    )


registered_projects = set(
    registry[
        registry_project_column
    ].astype(
        str
    )
)


if ACTIVE_RESERVED_PROJECTS:
    raise RuntimeError(
        "Project 22 bootstrap expects no active project reservations."
    )


EXPECTED_PREDECESSOR_IDENTITIES = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
}

for predecessor_number, expected_project in EXPECTED_PREDECESSOR_IDENTITIES.items():
    matches = registry.loc[
        registry_project_numbers.eq(predecessor_number),
        registry_project_column,
    ].astype(str).tolist()

    if matches != [expected_project]:
        raise RuntimeError(
            f"Frozen Project {predecessor_number} identity mismatch.\n"
            f"Expected: {expected_project}\n"
            f"Actual:   {matches}"
        )


def local_dataset_looks_complete():
    if not LOCAL_DATASET_ROOT.is_dir():
        return False

    project_directories = [
        path
        for path in LOCAL_DATASET_ROOT.iterdir()
        if path.is_dir()
    ]

    return bool(
        len(
            project_directories
        )
        == 25
    )


if local_dataset_looks_complete():
    extraction_performed = False
    extracted_files = 0

    print(
        "\nA complete-looking local TCP-CI dataset is already present."
    )

else:
    extraction_performed = True

    print(
        "\nRestoring the frozen TCP-CI archive into the Project 22 runtime."
    )

    if LOCAL_EXTRACTION_ROOT.exists():
        shutil.rmtree(
            LOCAL_EXTRACTION_ROOT
        )

    extracted_files = extract_archive_safely(
        ARCHIVE_PATH,
        LOCAL_EXTRACTION_ROOT,
    )


if not LOCAL_DATASET_ROOT.is_dir():
    raise RuntimeError(
        "Archive extraction did not create the expected dataset root:\n"
        f"{LOCAL_DATASET_ROOT}"
    )


all_project_directories = sorted(
    [
        path
        for path in LOCAL_DATASET_ROOT.iterdir()
        if path.is_dir()
    ],
    key=lambda path:
        path.name,
)


if len(all_project_directories) != 25:
    raise RuntimeError(
        "Unexpected number of TCP-CI project directories.\n"
        f"Expected: 25\n"
        f"Actual:   {len(all_project_directories)}"
    )


reserved_projects = set(
    ACTIVE_RESERVED_PROJECTS.values()
)

candidate_rows = []

for source_directory in all_project_directories:
    project = source_directory.name

    source_files = {
        path.name
        for path in source_directory.iterdir()
        if path.is_file()
    }

    missing_required_files = sorted(
        REQUIRED_PROJECT_FILES
        - source_files
    )

    excluded_registered = (
        project in registered_projects
    )

    excluded_reserved = (
        project in reserved_projects
    )

    candidate_eligible_for_scan = (
        not excluded_registered
        and not excluded_reserved
        and not missing_required_files
    )

    candidate_rows.append({
        "Project":
            project,
        "ProjectSlug":
            project.replace(
                "@",
                "__",
            ),
        "SourceDirectory":
            str(
                source_directory
            ),
        "ExcludedRegistered":
            bool(
                excluded_registered
            ),
        "ExcludedReserved":
            bool(
                excluded_reserved
            ),
        "MissingRequiredFiles":
            "; ".join(
                missing_required_files
            ),
        "CandidateForProject22Scan":
            bool(
                candidate_eligible_for_scan
            ),
    })


inventory = pd.DataFrame(
    candidate_rows
)


project_22_candidates = (
    inventory.loc[
        inventory[
            "CandidateForProject22Scan"
        ]
    ]
    .sort_values(
        "Project",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if len(
    project_22_candidates
) != EXPECTED_CANDIDATES:
    raise RuntimeError(
        "Unexpected number of Project 22 candidates after excluding "
        "frozen Projects 1–21.\n"
        f"Expected: {EXPECTED_CANDIDATES}\n"
        f"Actual:   {len(project_22_candidates)}"
    )


if (
    project_22_candidates[
        "Project"
    ].isin(
        registered_projects
        | reserved_projects
    ).any()
):
    raise RuntimeError(
        "A registered identity leaked into the Project 22 candidate set."
    )


SELECTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

atomic_write_csv(
    BOOTSTRAP_INVENTORY_PATH,
    project_22_candidates,
)


created_at_utc = datetime.now(
    timezone.utc
).isoformat()


report = {
    "ProjectNumber":
        PROJECT_NUMBER,
    "Status":
        STEP0_STATUS,
    "CreatedAtUTC":
        created_at_utc,
    "ArchivePath":
        str(
            ARCHIVE_PATH
        ),
    "ArchiveSHA256":
        archive_sha256,
    "RegistryPath":
        str(
            REGISTRY_PATH
        ),
    "RegistrySHA256":
        registry_sha256_before,
    "Project21Step5CCheckpoint":
        str(
            PROJECT_21_STEP5C_CHECKPOINT_PATH
        ),
    "Project21Step5CCheckpointSHA256":
        project_21_step5c_sha256,
    "RegisteredProjects":
        EXPECTED_REGISTERED_PROJECTS,
    "RegisteredStatuses":
        sorted(
            registry[
                registry_status_column
            ].astype(
                str
            ).unique().tolist()
        ),
    "ActiveReservations":
        {
            str(
                key
            ):
                value
            for key, value in ACTIVE_RESERVED_PROJECTS.items()
        },
    "FrozenPredecessorIdentities":
        {
            str(key): value
            for key, value in EXPECTED_PREDECESSOR_IDENTITIES.items()
        },
    "DatasetRoot":
        str(
            LOCAL_DATASET_ROOT
        ),
    "SourceProjectDirectories":
        len(
            all_project_directories
        ),
    "Project22CandidateCount":
        len(
            project_22_candidates
        ),
    "CandidateInventory":
        str(
            BOOTSTRAP_INVENTORY_PATH
        ),
    "RuntimePriorityPolicy":
        RUNTIME_PRIORITY_POLICY,
    "ExtractionPerformed":
        bool(
            extraction_performed
        ),
    "ArchiveFilesExtracted":
        int(
            extracted_files
        ),
    "RegistryModified":
        False,
    "PriorProjectConditionOutputsAccessed":
        False,
    "PriorProjectConditionOutputsModified":
        False,
    "NoiseInjected":
        False,
    "ModelsFitted":
        False,
}


atomic_write_json(
    BOOTSTRAP_REPORT_PATH,
    report,
)

atomic_write_json(
    BOOTSTRAP_STATUS_PATH,
    {
        "ProjectNumber":
            PROJECT_NUMBER,
        "Status":
            STEP0_STATUS,
        "CreatedAtUTC":
            created_at_utc,
        "Report":
            str(
                BOOTSTRAP_REPORT_PATH
            ),
    },
)


registry_sha256_after = sha256_file(
    REGISTRY_PATH
)

if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during the Project 22 bootstrap."
    )


print("\nProject 22 candidates after excluding registered identities:")
print(
    project_22_candidates[
        [
            "Project",
            "ProjectSlug",
            "SourceDirectory",
        ]
    ].to_string(
        index=False
    )
)


print("\n")
print("=" * 136)
print("=== PROJECT 22 CELL 1 / STEP 0 RESULT ===")
print("=" * 136)

print(
    "Registered and frozen projects:",
    EXPECTED_REGISTERED_PROJECTS,
)

print(
    "Active reservations:",
    [],
)

print(
    "TCP-CI source directories:",
    len(
        all_project_directories
    ),
)

print(
    "Project 22 candidates:",
    len(
        project_22_candidates
    ),
)

print(
    "Runtime-priority policy:",
    RUNTIME_PRIORITY_POLICY,
)

print(
    "Candidate inventory:",
    BOOTSTRAP_INVENTORY_PATH,
)

print(
    "Completion registry modified:",
    False,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Models fitted:",
    False,
)

print(
    "\nSTATUS:",
    STEP0_STATUS,
)

print("=" * 136)


=== PROJECT 22 CELL 1 / STEP 0: NEW-NOTEBOOK POST-PROJECT-21 BOOTSTRAP ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Restoring the frozen TCP-CI archive into the Project 22 runtime.

Project 22 candidates after excluding registered identities:
                 Project               ProjectSlug                                     SourceDirectory
Graylog2@graylog2-server Graylog2__graylog2-server /content/datasets/datasets/Graylog2@graylog2-server
   SonarSource@sonarqube    SonarSource__sonarqube    /content/datasets/datasets/SonarSource@sonarqube
   apache@logging-log4j2    apache__logging-log4j2    /content/datasets/datasets/apache@logging-log4j2
            apache@sling             apache__sling             /content/datasets/datasets/apache@sling


=== PROJECT 22 CELL 1 / STEP 0 RESULT ===
Registered and frozen projects: 21
Active reservations: []
TCP-CI source directories: 25
Project 22 candidate

In [2]:
# ==================================================================================================
# PROJECT 22 — CELL 2 / STEP 1A
# ROBUST CANDIDATE DISCOVERY, PROTOCOL ELIGIBILITY, RUNTIME-PRIORITIZED RANKING,
# AND PROVISIONAL PROJECT 22 SELECTION
#
# RUN THIS AS CELL 2 IN THE NEW Thesis_project_22.ipynb NOTEBOOK AFTER CELL 1 PASSES.
#
# THIS CELL:
# - inspects all 4 candidates frozen by Project 22 Step 0;
# - validates the chronological 75/25 split and raw/model cohort viability;
# - deterministically ranks eligible candidates by estimated experiment cost (smallest first);
# - changes processing order only, not protocol eligibility or the intended final project set;
# - freezes only a provisional Project 22 selection for Step 1B;
# - does not run experiment conditions or fit models;
# - does not modify the completion registry or Projects 1–21;
# - writes only Project 22 selection artifacts;
# - does not access prior-project condition outputs.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import math
import os
import time

import numpy as np
import pandas as pd


print("=" * 132)
print("=== PROJECT 22 CELL 2 / STEP 1A: RUNTIME-PRIORITIZED CANDIDATE DISCOVERY AND RANKING ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 22

BOOTSTRAP_PASS_STATUS = (
    "PASS_PROJECT_22_NEW_NOTEBOOK_RUNTIME_BOOTSTRAPPED_AND_CANDIDATES_DISCOVERED"
)

STEP1A_PASS_STATUS = (
    "PASS_PROJECT_22_CANDIDATE_DISCOVERY_COMPLETE"
)

EXPECTED_ARCHIVE_SHA256 = (
    "92af159c116e06e98d7c8348605adb6e9acb25aad982a1ff9fbef7103cb9e36e"
)

EXPECTED_REGISTRY_SHA256 = (
    "79cd6ecb595c5e8ae91a9494e469792716338d144308560a62caf1b9342306b2"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 21
EXPECTED_CANDIDATES = 4

RESERVED_ACTIVE_PROJECTS = set()

RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

# These three files are sufficient for deterministic selection.
# id_map.csv and entity_change_history.csv are checked and frozen later in Step 1B/2A.
REQUIRED_SELECTION_FILES = [
    "builds.csv",
    "exe.csv",
    "dataset.csv",
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

ARCHIVE_PATH = (
    THESIS_ROOT
    / "Data"
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)

REGISTRY_PATH = (
    THESIS_ROOT
    / "Notes"
    / "completed_project_registry.csv"
)

LOCAL_SOURCE_ROOT = Path(
    "/content/datasets/datasets"
)

SELECTION_ROOT = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / "project_22_selection"
)

BOOTSTRAP_STATUS_PATH = (
    SELECTION_ROOT
    / "project_22_step0_status.json"
)

BOOTSTRAP_CANDIDATE_INVENTORY_PATH = (
    SELECTION_ROOT
    / "project_22_bootstrap_candidate_inventory.csv"
)

SCAN_PROGRESS_PATH = (
    SELECTION_ROOT
    / "project_22_candidate_scan_progress.csv"
)

SOURCE_SCHEMA_AUDIT_PATH = (
    SELECTION_ROOT
    / "project_22_source_schema_audit.csv"
)

CANDIDATE_INVENTORY_PATH = (
    SELECTION_ROOT
    / "project_22_candidate_inventory.csv"
)

ELIGIBLE_RANKED_PATH = (
    SELECTION_ROOT
    / "project_22_eligible_candidates_ranked.csv"
)

INELIGIBLE_PATH = (
    SELECTION_ROOT
    / "project_22_ineligible_candidates.csv"
)

INSPECTION_ERRORS_PATH = (
    SELECTION_ROOT
    / "project_22_candidate_inspection_errors.csv"
)

PROVISIONAL_SELECTION_PATH = (
    SELECTION_ROOT
    / "project_22_provisional_selection.json"
)

STEP1A_VALIDATION_PATH = (
    SELECTION_ROOT
    / "project_22_step1a_validation.csv"
)

STEP1A_REPORT_PATH = (
    SELECTION_ROOT
    / "project_22_step1a_report.json"
)

STEP1A_STATUS_PATH = (
    SELECTION_ROOT
    / "project_22_step1a_status.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_write_csv(
    path,
    frame,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve {label}.\n"
            f"Expected: {expected!r}\n"
            f"Matches: {matches}\n"
            f"Columns: {list(columns)}"
        )

    return matches[0]


def parse_integer_series(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing or non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")


def project_slug(project_name):
    return str(project_name).replace(
        "@",
        "__",
        1,
    )


def count_partitioned_rows(
    csv_path,
    build_column,
    verdict_column,
    training_build_ids,
    evaluation_build_ids,
    label,
    chunksize,
):
    total_rows = 0
    training_rows = 0
    evaluation_rows = 0

    training_failures = 0
    evaluation_failures = 0

    failing_training_builds = set()
    failing_evaluation_builds = set()

    unlinked_rows = 0
    verdict_values = set()

    for chunk in pd.read_csv(
        csv_path,
        usecols=[
            build_column,
            verdict_column,
        ],
        chunksize=chunksize,
        low_memory=False,
    ):
        chunk_build = parse_integer_series(
            chunk[build_column],
            f"{label}.{build_column}",
        )

        chunk_verdict = parse_integer_series(
            chunk[verdict_column],
            f"{label}.{verdict_column}",
        )

        training_mask = chunk_build.isin(
            training_build_ids
        )

        evaluation_mask = chunk_build.isin(
            evaluation_build_ids
        )

        linked_mask = (
            training_mask
            | evaluation_mask
        )

        failure_mask = chunk_verdict.ne(0)

        total_rows += len(chunk)

        training_rows += int(
            training_mask.sum()
        )

        evaluation_rows += int(
            evaluation_mask.sum()
        )

        training_failures += int(
            (
                training_mask
                & failure_mask
            ).sum()
        )

        evaluation_failures += int(
            (
                evaluation_mask
                & failure_mask
            ).sum()
        )

        failing_training_builds.update(
            chunk_build.loc[
                training_mask
                & failure_mask
            ].astype(int).tolist()
        )

        failing_evaluation_builds.update(
            chunk_build.loc[
                evaluation_mask
                & failure_mask
            ].astype(int).tolist()
        )

        unlinked_rows += int(
            (~linked_mask).sum()
        )

        verdict_values.update(
            int(value)
            for value in chunk_verdict.unique().tolist()
        )

    return {
        "Rows":
            int(total_rows),

        "TrainingRows":
            int(training_rows),

        "EvaluationRows":
            int(evaluation_rows),

        "TrainingFailures":
            int(training_failures),

        "EvaluationFailures":
            int(evaluation_failures),

        "FailingTrainingBuilds":
            int(len(failing_training_builds)),

        "FailingEvaluationBuilds":
            int(len(failing_evaluation_builds)),

        "UnlinkedRows":
            int(unlinked_rows),

        "VerdictValuesJSON":
            json.dumps(
                sorted(verdict_values)
            ),
    }


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(passed),
    })


def reusable_scan_row_is_valid(
    row,
):
    required_fields = [
        "Project",
        "ProjectSlug",
        "SourceDirectory",
        "InspectionStatus",
        "InspectionError",
        "BuildIDColumn",
        "StartedAtColumn",
        "ExecutionBuildColumn",
        "ExecutionVerdictColumn",
        "DatasetBuildColumn",
        "DatasetVerdictColumn",
        "Builds",
        "TrainingBuilds",
        "EvaluationBuilds",
        "RawExecutionRows",
        "RawTrainingRows",
        "RawEvaluationRows",
        "RawTrainFailures",
        "RawEvaluationFailures",
        "RawFailingTrainingBuilds",
        "RawFailingEvaluationBuilds",
        "RawUnlinkedRows",
        "ModelReadyRows",
        "ModelTrainingRows",
        "ModelEvaluationRows",
        "ModelTrainFailures",
        "ModelEvaluationFailures",
        "ModelFailingTrainingBuilds",
        "ModelFailingEvaluationBuilds",
        "ModelUnlinkedRows",
    ]

    if any(
        field not in row
        for field in required_fields
    ):
        return False

    status = str(
        row.get(
            "InspectionStatus",
            "",
        )
    ).strip()

    error = str(
        row.get(
            "InspectionError",
            "",
        )
    ).strip().lower()

    return (
        status in {
            "ELIGIBLE",
            "INELIGIBLE",
        }
        and error in {
            "",
            "nan",
            "none",
        }
    )


# --------------------------------------------------------------------------------------------------
# 4. VALIDATE STEP 0, REGISTRY, ARCHIVE, AND LOCAL SOURCE
# --------------------------------------------------------------------------------------------------

required_inputs = [
    ARCHIVE_PATH,
    REGISTRY_PATH,
    BOOTSTRAP_STATUS_PATH,
    BOOTSTRAP_CANDIDATE_INVENTORY_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.is_file()
]

if missing_inputs:
    raise FileNotFoundError(
        "Required Project 22 Step 1A inputs are missing:\n"
        + "\n".join(missing_inputs)
    )


if not LOCAL_SOURCE_ROOT.is_dir():
    raise FileNotFoundError(
        "The local Project 22 dataset source is missing:\n"
        f"{LOCAL_SOURCE_ROOT}"
    )


bootstrap_status = load_json(
    BOOTSTRAP_STATUS_PATH
)

if bootstrap_status.get(
    "Status"
) != BOOTSTRAP_PASS_STATUS:
    raise RuntimeError(
        "Project 22 Step 0 is not in the expected PASS state."
    )


archive_sha256 = sha256_file(
    ARCHIVE_PATH
)

if archive_sha256 != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        "Dataset archive SHA-256 differs.\n"
        f"Expected: {EXPECTED_ARCHIVE_SHA256}\n"
        f"Actual:   {archive_sha256}"
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


registry_project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

registry_project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

registry_status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        registry_project_number_column
    ],
    errors="raise",
).astype(int)


if (
    len(registry) != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    ) != list(range(1, 22))
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–21."
    )


if not registry[
    registry_status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–21 are not all COMPLETE_AND_FROZEN."
    )


if registry_project_numbers.eq(
    PROJECT_NUMBER
).any():
    raise RuntimeError(
        "Project 22 is unexpectedly already registered."
    )


registered_projects = set(
    registry[
        registry_project_column
    ].astype(str).tolist()
)


if registered_projects & RESERVED_ACTIVE_PROJECTS:
    raise RuntimeError(
        "A reserved active-project identity is unexpectedly present in the completion registry."
    )


required_registered_identities = {
    11:
        "apache@shardingsphere",

    12:
        "zolyfarkas@spf4j",

    13:
        "jcabi@jcabi-github",

    14:
        "JMRI@JMRI",

    15:
        "eclipse@steady",

    16:
        "apache@rocketmq",

    17:
        "yamcs@Yamcs",

    18:
        "cantaloupe-project@cantaloupe",

    19:
        "EMResearch@EvoMaster",

    20:
        "apache@curator",

    21:
        "facebook@buck",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            registry_project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


bootstrap_candidates = pd.read_csv(
    BOOTSTRAP_CANDIDATE_INVENTORY_PATH,
    low_memory=False,
)


candidate_project_column = resolve_column(
    bootstrap_candidates.columns,
    "Project",
    "bootstrap candidate Project",
)

candidate_source_column = resolve_column(
    bootstrap_candidates.columns,
    "SourceDirectory",
    "bootstrap candidate SourceDirectory",
)


candidate_records = (
    bootstrap_candidates[
        [
            candidate_project_column,
            candidate_source_column,
        ]
    ]
    .rename(
        columns={
            candidate_project_column:
                "Project",

            candidate_source_column:
                "SourceDirectory",
        }
    )
    .copy()
)


candidate_records[
    "Project"
] = candidate_records[
    "Project"
].astype(str)


candidate_records[
    "SourceDirectory"
] = candidate_records[
    "SourceDirectory"
].astype(str)


if len(candidate_records) != EXPECTED_CANDIDATES:
    raise RuntimeError(
        "Unexpected Project 22 candidate count.\n"
        f"Expected: {EXPECTED_CANDIDATES}\n"
        f"Actual:   {len(candidate_records)}"
    )


if candidate_records[
    "Project"
].duplicated(
    keep=False
).any():
    raise RuntimeError(
        "Project 22 bootstrap candidate inventory contains duplicates."
    )


forbidden_candidates = (
    set(
        candidate_records[
            "Project"
        ]
    )
    & (
        registered_projects
        | RESERVED_ACTIVE_PROJECTS
    )
)


if forbidden_candidates:
    raise RuntimeError(
        "Project 22 inventory contains registered/reserved projects:\n"
        + "\n".join(
            sorted(forbidden_candidates)
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. REUSE ANY VALID COMPLETED PROJECT 22 SCANS
# --------------------------------------------------------------------------------------------------

SELECTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


reusable_rows = {}


if SCAN_PROGRESS_PATH.is_file():
    try:
        previous_progress = pd.read_csv(
            SCAN_PROGRESS_PATH,
            low_memory=False,
        )

        valid_candidate_names = set(
            candidate_records[
                "Project"
            ]
        )

        for row in previous_progress.to_dict(
            orient="records"
        ):
            project = str(
                row.get(
                    "Project",
                    "",
                )
            )

            if (
                project in valid_candidate_names
                and reusable_scan_row_is_valid(
                    row
                )
            ):
                reusable_rows[
                    project
                ] = row

        print(
            "\nReusable completed candidate scans:",
            len(reusable_rows),
        )

    except Exception as error:
        print(
            "\nPrevious scan progress was ignored:",
            type(error).__name__,
            str(error),
        )


# --------------------------------------------------------------------------------------------------
# 6. INSPECT ALL 11 CANDIDATES
# --------------------------------------------------------------------------------------------------

scan_rows = []


for candidate_index, candidate in enumerate(
    candidate_records.itertuples(
        index=False
    ),
    start=1,
):
    project = str(
        candidate.Project
    )

    source_directory = Path(
        candidate.SourceDirectory
    )

    print("-" * 132)
    print(
        f"[{candidate_index:02d}/{EXPECTED_CANDIDATES:02d}] "
        f"Inspecting: {project}"
    )


    if project in reusable_rows:
        row = dict(
            reusable_rows[
                project
            ]
        )

        row[
            "CandidateInspectionOrder"
        ] = candidate_index

        row[
            "ProtocolEligible"
        ] = (
            str(
                row[
                    "InspectionStatus"
                ]
            )
            == "ELIGIBLE"
        )

        row[
            "InspectionError"
        ] = ""

        scan_rows.append(
            row
        )

        print(
            "    Reused:",
            row[
                "InspectionStatus"
            ],
            "| Builds:",
            int(
                row[
                    "Builds"
                ]
            ),
            "| Model eval failures:",
            int(
                row[
                    "ModelEvaluationFailures"
                ]
            ),
        )

        continue


    started = time.perf_counter()

    row = {
        "CandidateInspectionOrder":
            candidate_index,

        "Project":
            project,

        "ProjectSlug":
            project_slug(
                project
            ),

        "SourceDirectory":
            str(
                source_directory
            ),

        "InspectionStatus":
            "ERROR",

        "InspectionError":
            "",
    }


    try:
        missing_files = [
            filename
            for filename in REQUIRED_SELECTION_FILES
            if not (
                source_directory
                / filename
            ).is_file()
        ]

        if missing_files:
            raise FileNotFoundError(
                "Missing selection files: "
                + ", ".join(
                    missing_files
                )
            )


        builds_path = (
            source_directory
            / "builds.csv"
        )

        exe_path = (
            source_directory
            / "exe.csv"
        )

        dataset_path = (
            source_directory
            / "dataset.csv"
        )


        build_columns = pd.read_csv(
            builds_path,
            nrows=0,
        ).columns.tolist()

        exe_columns = pd.read_csv(
            exe_path,
            nrows=0,
        ).columns.tolist()

        dataset_columns = pd.read_csv(
            dataset_path,
            nrows=0,
        ).columns.tolist()


        build_id_column = resolve_column(
            build_columns,
            "id",
            f"{project} builds.csv ID",
        )

        started_at_column = resolve_column(
            build_columns,
            "started_at",
            f"{project} builds.csv started_at",
        )

        execution_build_column = resolve_column(
            exe_columns,
            "build",
            f"{project} exe.csv build",
        )

        execution_verdict_column = resolve_column(
            exe_columns,
            "verdict",
            f"{project} exe.csv verdict",
        )

        dataset_build_column = resolve_column(
            dataset_columns,
            "Build",
            f"{project} dataset.csv Build",
        )

        dataset_verdict_column = resolve_column(
            dataset_columns,
            "Verdict",
            f"{project} dataset.csv Verdict",
        )


        builds = pd.read_csv(
            builds_path,
            usecols=[
                build_id_column,
                started_at_column,
            ],
            low_memory=False,
        )


        builds[
            build_id_column
        ] = parse_integer_series(
            builds[
                build_id_column
            ],
            f"{project}.builds.id",
        )


        builds[
            started_at_column
        ] = pd.to_datetime(
            builds[
                started_at_column
            ],
            errors="coerce",
            utc=True,
        )


        invalid_timestamps = int(
            builds[
                started_at_column
            ].isna().sum()
        )


        duplicate_build_id_rows = int(
            builds[
                build_id_column
            ].duplicated(
                keep=False
            ).sum()
        )


        if invalid_timestamps != 0:
            raise RuntimeError(
                f"Invalid build timestamps: {invalid_timestamps}"
            )


        if duplicate_build_id_rows != 0:
            raise RuntimeError(
                f"Duplicate build-ID rows: {duplicate_build_id_rows}"
            )


        ordered_builds = (
            builds.sort_values(
                [
                    started_at_column,
                    build_id_column,
                ],
                ascending=[
                    True,
                    False,
                ],
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )


        number_of_builds = len(
            ordered_builds
        )


        training_build_count = int(
            math.floor(
                0.75
                * number_of_builds
            )
        )


        evaluation_build_count = int(
            number_of_builds
            - training_build_count
        )


        if (
            training_build_count <= 0
            or evaluation_build_count <= 0
        ):
            raise RuntimeError(
                "Chronological 75/25 split has an empty partition."
            )


        training_build_ids = set(
            ordered_builds.iloc[
                :training_build_count
            ][
                build_id_column
            ].astype(int).tolist()
        )


        evaluation_build_ids = set(
            ordered_builds.iloc[
                training_build_count:
            ][
                build_id_column
            ].astype(int).tolist()
        )


        if training_build_ids & evaluation_build_ids:
            raise RuntimeError(
                "Training/evaluation build partitions overlap."
            )


        raw_profile = count_partitioned_rows(
            csv_path=exe_path,
            build_column=execution_build_column,
            verdict_column=execution_verdict_column,
            training_build_ids=training_build_ids,
            evaluation_build_ids=evaluation_build_ids,
            label=f"{project}.exe",
            chunksize=500_000,
        )


        model_profile = count_partitioned_rows(
            csv_path=dataset_path,
            build_column=dataset_build_column,
            verdict_column=dataset_verdict_column,
            training_build_ids=training_build_ids,
            evaluation_build_ids=evaluation_build_ids,
            label=f"{project}.dataset",
            chunksize=250_000,
        )


        eligibility_reasons = []


        eligibility_tests = [
            (
                raw_profile[
                    "TrainingRows"
                ] > 0,
                "No raw training rows",
            ),

            (
                raw_profile[
                    "EvaluationRows"
                ] > 0,
                "No raw evaluation rows",
            ),

            (
                raw_profile[
                    "TrainingFailures"
                ] > 0,
                "No raw training failures",
            ),

            (
                raw_profile[
                    "EvaluationFailures"
                ] > 0,
                "No raw evaluation failures",
            ),

            (
                model_profile[
                    "TrainingRows"
                ] > 0,
                "No model training rows",
            ),

            (
                model_profile[
                    "EvaluationRows"
                ] > 0,
                "No model evaluation rows",
            ),

            (
                model_profile[
                    "TrainingFailures"
                ] > 0,
                "No model training failures",
            ),

            (
                model_profile[
                    "EvaluationFailures"
                ] > 0,
                "No model evaluation failures",
            ),

            (
                raw_profile[
                    "UnlinkedRows"
                ] == 0,
                "Raw rows reference unknown builds",
            ),

            (
                model_profile[
                    "UnlinkedRows"
                ] == 0,
                "Model rows reference unknown builds",
            ),
        ]


        for passed, failure_reason in eligibility_tests:
            if not passed:
                eligibility_reasons.append(
                    failure_reason
                )


        protocol_eligible = (
            len(
                eligibility_reasons
            )
            == 0
        )


        row.update({
            "BuildIDColumn":
                build_id_column,

            "StartedAtColumn":
                started_at_column,

            "ExecutionBuildColumn":
                execution_build_column,

            "ExecutionVerdictColumn":
                execution_verdict_column,

            "DatasetBuildColumn":
                dataset_build_column,

            "DatasetVerdictColumn":
                dataset_verdict_column,

            "Builds":
                number_of_builds,

            "TrainingBuilds":
                training_build_count,

            "EvaluationBuilds":
                evaluation_build_count,

            "RawExecutionRows":
                raw_profile[
                    "Rows"
                ],

            "RawTrainingRows":
                raw_profile[
                    "TrainingRows"
                ],

            "RawEvaluationRows":
                raw_profile[
                    "EvaluationRows"
                ],

            "RawTrainFailures":
                raw_profile[
                    "TrainingFailures"
                ],

            "RawEvaluationFailures":
                raw_profile[
                    "EvaluationFailures"
                ],

            "RawFailingTrainingBuilds":
                raw_profile[
                    "FailingTrainingBuilds"
                ],

            "RawFailingEvaluationBuilds":
                raw_profile[
                    "FailingEvaluationBuilds"
                ],

            "RawUnlinkedRows":
                raw_profile[
                    "UnlinkedRows"
                ],

            "RawVerdictValuesJSON":
                raw_profile[
                    "VerdictValuesJSON"
                ],

            "ModelReadyRows":
                model_profile[
                    "Rows"
                ],

            "ModelTrainingRows":
                model_profile[
                    "TrainingRows"
                ],

            "ModelEvaluationRows":
                model_profile[
                    "EvaluationRows"
                ],

            "ModelTrainFailures":
                model_profile[
                    "TrainingFailures"
                ],

            "ModelEvaluationFailures":
                model_profile[
                    "EvaluationFailures"
                ],

            "ModelFailingTrainingBuilds":
                model_profile[
                    "FailingTrainingBuilds"
                ],

            "ModelFailingEvaluationBuilds":
                model_profile[
                    "FailingEvaluationBuilds"
                ],

            "ModelUnlinkedRows":
                model_profile[
                    "UnlinkedRows"
                ],

            "ModelVerdictValuesJSON":
                model_profile[
                    "VerdictValuesJSON"
                ],

            "ProtocolEligible":
                protocol_eligible,

            "EligibilityReason":
                (
                    ""
                    if protocol_eligible
                    else "; ".join(
                        eligibility_reasons
                    )
                ),

            "InspectionStatus":
                (
                    "ELIGIBLE"
                    if protocol_eligible
                    else "INELIGIBLE"
                ),

            "InspectionError":
                "",
        })


        print(
            "    Status:",
            row[
                "InspectionStatus"
            ],
            "| Builds:",
            number_of_builds,
            "| Model rows:",
            model_profile[
                "Rows"
            ],
            "| Model eval failures:",
            model_profile[
                "EvaluationFailures"
            ],
        )


    except Exception as error:
        row.update({
            "ProtocolEligible":
                False,

            "EligibilityReason":
                "Inspection error",

            "InspectionStatus":
                "ERROR",

            "InspectionError":
                (
                    f"{type(error).__name__}: "
                    f"{error}"
                ),
        })

        print(
            "    ERROR:",
            row[
                "InspectionError"
            ],
        )


    row[
        "ElapsedSeconds"
    ] = float(
        time.perf_counter()
        - started
    )


    scan_rows.append(
        row
    )


    atomic_write_csv(
        SCAN_PROGRESS_PATH,
        pd.DataFrame(
            scan_rows
        ).sort_values(
            "CandidateInspectionOrder",
            kind="mergesort",
        ),
    )


scan_progress = (
    pd.DataFrame(
        scan_rows
    )
    .sort_values(
        "CandidateInspectionOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 7. DETERMINISTIC RANKING
# --------------------------------------------------------------------------------------------------

inspection_errors = scan_progress.loc[
    scan_progress[
        "InspectionStatus"
    ].eq(
        "ERROR"
    )
].copy()


eligible_candidates = scan_progress.loc[
    scan_progress[
        "InspectionStatus"
    ].eq(
        "ELIGIBLE"
    )
].copy()


ineligible_candidates = scan_progress.loc[
    scan_progress[
        "InspectionStatus"
    ].eq(
        "INELIGIBLE"
    )
].copy()


if not inspection_errors.empty:
    print(
        "\nCandidate inspection errors:"
    )

    display(
        inspection_errors[
            [
                "Project",
                "InspectionError",
            ]
        ]
    )

    raise RuntimeError(
        "One or more Project 22 candidates could not be inspected. "
        "No provisional selection was frozen."
    )


if eligible_candidates.empty:
    raise RuntimeError(
        "No protocol-eligible Project 22 candidate was found."
    )


eligible_candidates = (
    eligible_candidates.sort_values(
        [
            "ModelTrainingRows",
            "ModelEvaluationRows",
            "RawExecutionRows",
            "Project",
        ],
        ascending=[
            True,
            True,
            True,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


eligible_candidates.insert(
    0,
    "CandidateRank",
    np.arange(
        1,
        len(
            eligible_candidates
        )
        + 1,
        dtype=np.int64,
    ),
)


top_candidate = eligible_candidates.iloc[
    0
]


# --------------------------------------------------------------------------------------------------
# 8. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Completion registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(registry),
    len(registry)
    == EXPECTED_REGISTERED_PROJECTS,
)

add_check(
    validation_records,
    "Projects 1–21 COMPLETE_AND_FROZEN",
    EXPECTED_REGISTERED_PROJECTS,
    int(
        registry[
            registry_status_column
        ].eq(
            EXPECTED_COMPLETE_STATUS
        ).sum()
    ),
    int(
        registry[
            registry_status_column
        ].eq(
            EXPECTED_COMPLETE_STATUS
        ).sum()
    ) == EXPECTED_REGISTERED_PROJECTS,
)

add_check(
    validation_records,
    "Project 22 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)


for required_number, required_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            registry_project_numbers.eq(
                required_number
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {required_number} frozen identity",
        required_project,
        actual_project,
        actual_project
        == required_project,
    )


add_check(
    validation_records,
    "Candidates inspected",
    EXPECTED_CANDIDATES,
    len(scan_progress),
    len(scan_progress)
    == EXPECTED_CANDIDATES,
)

add_check(
    validation_records,
    "Unique candidate identities",
    EXPECTED_CANDIDATES,
    int(
        scan_progress[
            "Project"
        ].nunique()
    ),
    int(
        scan_progress[
            "Project"
        ].nunique()
    ) == EXPECTED_CANDIDATES,
)

add_check(
    validation_records,
    "Registered/reserved candidates",
    0,
    int(
        scan_progress[
            "Project"
        ].isin(
            registered_projects
            | RESERVED_ACTIVE_PROJECTS
        ).sum()
    ),
    int(
        scan_progress[
            "Project"
        ].isin(
            registered_projects
            | RESERVED_ACTIVE_PROJECTS
        ).sum()
    ) == 0,
)

add_check(
    validation_records,
    "Inspection errors",
    0,
    len(inspection_errors),
    len(inspection_errors)
    == 0,
)

add_check(
    validation_records,
    "Candidate accounting",
    EXPECTED_CANDIDATES,
    (
        len(
            eligible_candidates
        )
        + len(
            ineligible_candidates
        )
        + len(
            inspection_errors
        )
    ),
    (
        len(
            eligible_candidates
        )
        + len(
            ineligible_candidates
        )
        + len(
            inspection_errors
        )
    ) == EXPECTED_CANDIDATES,
)

add_check(
    validation_records,
    "At least one eligible candidate",
    "> 0",
    len(eligible_candidates),
    len(eligible_candidates)
    > 0,
)

add_check(
    validation_records,
    "Candidate ranks unique",
    len(eligible_candidates),
    int(
        eligible_candidates[
            "CandidateRank"
        ].nunique()
    ),
    int(
        eligible_candidates[
            "CandidateRank"
        ].nunique()
    ) == len(
        eligible_candidates
    ),
)

add_check(
    validation_records,
    "Top rank",
    1,
    int(
        top_candidate[
            "CandidateRank"
        ]
    ),
    int(
        top_candidate[
            "CandidateRank"
        ]
    ) == 1,
)

add_check(
    validation_records,
    "Top candidate eligible",
    True,
    (
        str(
            top_candidate[
                "InspectionStatus"
            ]
        )
        == "ELIGIBLE"
    ),
    (
        str(
            top_candidate[
                "InspectionStatus"
            ]
        )
        == "ELIGIBLE"
    ),
)

add_check(
    validation_records,
    "Top candidate raw unlinked rows",
    0,
    int(
        top_candidate[
            "RawUnlinkedRows"
        ]
    ),
    int(
        top_candidate[
            "RawUnlinkedRows"
        ]
    ) == 0,
)

add_check(
    validation_records,
    "Top candidate model unlinked rows",
    0,
    int(
        top_candidate[
            "ModelUnlinkedRows"
        ]
    ),
    int(
        top_candidate[
            "ModelUnlinkedRows"
        ]
    ) == 0,
)

add_check(
    validation_records,
    "Top candidate model training failures",
    "> 0",
    int(
        top_candidate[
            "ModelTrainFailures"
        ]
    ),
    int(
        top_candidate[
            "ModelTrainFailures"
        ]
    ) > 0,
)

add_check(
    validation_records,
    "Top candidate model evaluation failures",
    "> 0",
    int(
        top_candidate[
            "ModelEvaluationFailures"
        ]
    ),
    int(
        top_candidate[
            "ModelEvaluationFailures"
        ]
    ) > 0,
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 22 Step 1A validation:"
)

display(
    validation
)


if not failed_validation.empty:
    print(
        "\nFailed validation checks:"
    )

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 22 STEP 1A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 9. WRITE AUTHORITATIVE STEP 1A OUTPUTS
# --------------------------------------------------------------------------------------------------

schema_columns = [
    "Project",
    "ProjectSlug",
    "BuildIDColumn",
    "StartedAtColumn",
    "ExecutionBuildColumn",
    "ExecutionVerdictColumn",
    "DatasetBuildColumn",
    "DatasetVerdictColumn",
    "InspectionStatus",
    "InspectionError",
]


source_schema_audit = scan_progress[
    schema_columns
].copy()


atomic_write_csv(
    SCAN_PROGRESS_PATH,
    scan_progress,
)

atomic_write_csv(
    SOURCE_SCHEMA_AUDIT_PATH,
    source_schema_audit,
)

atomic_write_csv(
    CANDIDATE_INVENTORY_PATH,
    scan_progress,
)

atomic_write_csv(
    ELIGIBLE_RANKED_PATH,
    eligible_candidates,
)

atomic_write_csv(
    INELIGIBLE_PATH,
    ineligible_candidates,
)

atomic_write_csv(
    INSPECTION_ERRORS_PATH,
    inspection_errors,
)

atomic_write_csv(
    STEP1A_VALIDATION_PATH,
    validation,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


dimension_fields = [
    "Builds",
    "TrainingBuilds",
    "EvaluationBuilds",
    "RawExecutionRows",
    "RawTrainingRows",
    "RawEvaluationRows",
    "RawTrainFailures",
    "RawEvaluationFailures",
    "RawFailingTrainingBuilds",
    "RawFailingEvaluationBuilds",
    "RawUnlinkedRows",
    "ModelReadyRows",
    "ModelTrainingRows",
    "ModelEvaluationRows",
    "ModelTrainFailures",
    "ModelEvaluationFailures",
    "ModelFailingTrainingBuilds",
    "ModelFailingEvaluationBuilds",
    "ModelUnlinkedRows",
]


provisional_selection_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "SelectionState":
        "PROVISIONAL_PENDING_STEP_1B_FREEZE",

    "CandidateRank":
        int(
            top_candidate[
                "CandidateRank"
            ]
        ),

    "Project":
        str(
            top_candidate[
                "Project"
            ]
        ),

    "ProjectSlug":
        str(
            top_candidate[
                "ProjectSlug"
            ]
        ),

    "SourceDirectory":
        str(
            top_candidate[
                "SourceDirectory"
            ]
        ),

    "BuildIDColumn":
        str(
            top_candidate[
                "BuildIDColumn"
            ]
        ),

    "StartedAtColumn":
        str(
            top_candidate[
                "StartedAtColumn"
            ]
        ),

    "ExecutionBuildColumn":
        str(
            top_candidate[
                "ExecutionBuildColumn"
            ]
        ),

    "ExecutionVerdictColumn":
        str(
            top_candidate[
                "ExecutionVerdictColumn"
            ]
        ),

    "DatasetBuildColumn":
        str(
            top_candidate[
                "DatasetBuildColumn"
            ]
        ),

    "DatasetVerdictColumn":
        str(
            top_candidate[
                "DatasetVerdictColumn"
            ]
        ),

    "Dimensions": {
        field:
            int(
                top_candidate[
                    field
                ]
            )
        for field in dimension_fields
    },

    "RankingRule":
        RUNTIME_PRIORITY_RULE,

    "RankingPurpose":
        "Runtime-prioritized processing order only; protocol eligibility and final project set are unchanged",

    "EligibleCandidateCount":
        len(
            eligible_candidates
        ),

    "IneligibleCandidateCount":
        len(
            ineligible_candidates
        ),

    "ReservedActiveProjectsExcluded":
        sorted(
            RESERVED_ACTIVE_PROJECTS
        ),

    "CompletedAtUTC":
        completed_at_utc,
}


atomic_write_json(
    PROVISIONAL_SELECTION_PATH,
    provisional_selection_payload,
)


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Status":
        STEP1A_PASS_STATUS,

    "ImplementationVersion":
        "PROJECT_22_RUNTIME_PRIORITIZED_DISCOVERY_V1",

    "RuntimePriorityRule":
        RUNTIME_PRIORITY_RULE,

    "CompletedAtUTC":
        completed_at_utc,

    "ArchiveSHA256":
        archive_sha256,

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "CandidatesInspected":
        len(
            scan_progress
        ),

    "ProtocolEligibleCandidates":
        len(
            eligible_candidates
        ),

    "ProtocolIneligibleCandidates":
        len(
            ineligible_candidates
        ),

    "InspectionErrors":
        len(
            inspection_errors
        ),

    "ProvisionalSelection":
        provisional_selection_payload,

    "RegistryModified":
        False,

    "Projects1To21Modified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project22ExperimentStarted":
        False,
}


atomic_write_json(
    STEP1A_REPORT_PATH,
    report_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Status":
        STEP1A_PASS_STATUS,

    "ImplementationVersion":
        "PROJECT_22_RUNTIME_PRIORITIZED_DISCOVERY_V1",

    "RuntimePriorityRule":
        RUNTIME_PRIORITY_RULE,

    "CompletedAtUTC":
        completed_at_utc,

    "CandidatesInspected":
        len(
            scan_progress
        ),

    "ProtocolEligibleCandidates":
        len(
            eligible_candidates
        ),

    "ProtocolIneligibleCandidates":
        len(
            ineligible_candidates
        ),

    "InspectionErrors":
        len(
            inspection_errors
        ),

    "ProvisionalProject":
        str(
            top_candidate[
                "Project"
            ]
        ),

    "ProvisionalProjectSlug":
        str(
            top_candidate[
                "ProjectSlug"
            ]
        ),

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "Project22ExperimentStarted":
        False,
}


atomic_write_json(
    STEP1A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 10. FINAL ISOLATION CHECK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 22 Step 1A."
    )


# --------------------------------------------------------------------------------------------------
# 11. DISPLAY FINAL RESULT
# --------------------------------------------------------------------------------------------------

ranked_display_columns = [
    "CandidateRank",
    "Project",
    "ProjectSlug",
    "Builds",
    "TrainingBuilds",
    "EvaluationBuilds",
    "RawExecutionRows",
    "RawTrainFailures",
    "RawEvaluationFailures",
    "RawFailingEvaluationBuilds",
    "ModelReadyRows",
    "ModelTrainingRows",
    "ModelEvaluationRows",
    "ModelTrainFailures",
    "ModelEvaluationFailures",
    "ModelFailingEvaluationBuilds",
    "RawUnlinkedRows",
    "ModelUnlinkedRows",
]


print(
    "\nRanked eligible Project 22 candidates:"
)

display(
    eligible_candidates[
        ranked_display_columns
    ]
)


print(
    "\nProtocol-ineligible candidates:"
)

if ineligible_candidates.empty:
    print(
        "None"
    )

else:
    display(
        ineligible_candidates[
            [
                "Project",
                "Builds",
                "RawTrainFailures",
                "RawEvaluationFailures",
                "ModelTrainFailures",
                "ModelEvaluationFailures",
                "EligibilityReason",
            ]
        ]
    )


print("\n")
print("=" * 132)
print("=== PROJECT 22 CELL 2 / STEP 1A RESULT ===")
print("=" * 132)


print(
    "Registered projects:",
    len(
        registry
    ),
)

for required_number in sorted(
    required_registered_identities
):
    print(
        f"Project {required_number} identity:",
        required_registered_identities[
            required_number
        ],
    )


print(
    "Candidates inspected:",
    len(
        scan_progress
    ),
)

print(
    "Protocol-eligible candidates:",
    len(
        eligible_candidates
    ),
)

print(
    "Protocol-ineligible candidates:",
    len(
        ineligible_candidates
    ),
)

print(
    "Inspection errors:",
    len(
        inspection_errors
    ),
)

print(
    "Runtime-priority ranking rule:",
    RUNTIME_PRIORITY_RULE,
)


print(
    "\nProvisional Project 22 candidate:"
)

print(
    "Candidate rank:",
    int(
        top_candidate[
            "CandidateRank"
        ]
    ),
)

print(
    "Project:",
    str(
        top_candidate[
            "Project"
        ]
    ),
)

print(
    "Project slug:",
    str(
        top_candidate[
            "ProjectSlug"
        ]
    ),
)

print(
    "Source directory:",
    str(
        top_candidate[
            "SourceDirectory"
        ]
    ),
)


print(
    "\nCandidate dimensions:"
)

for field in dimension_fields:
    print(
        f"{field}:",
        int(
            top_candidate[
                field
            ]
        ),
    )


print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–21 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Project 22 experiment started:",
    False,
)


print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print(
    "\nSTATUS:",
    STEP1A_PASS_STATUS,
)

print("=" * 132)


=== PROJECT 22 CELL 2 / STEP 1A: RUNTIME-PRIORITIZED CANDIDATE DISCOVERY AND RANKING ===
------------------------------------------------------------------------------------------------------------------------------------
[01/04] Inspecting: Graylog2@graylog2-server
    Status: INELIGIBLE | Builds: 3668 | Model rows: 4822 | Model eval failures: 0
------------------------------------------------------------------------------------------------------------------------------------
[02/04] Inspecting: SonarSource@sonarqube
    Status: ELIGIBLE | Builds: 4286 | Model rows: 224550 | Model eval failures: 20
------------------------------------------------------------------------------------------------------------------------------------
[03/04] Inspecting: apache@logging-log4j2
    Status: ELIGIBLE | Builds: 441 | Model rows: 117968 | Model eval failures: 40
------------------------------------------------------------------------------------------------------------------------------------
[04

,Check,Expected,Actual,Pass
0,Completion registry rows,21,21,True
1,Projects 1–21 COMPLETE_AND_FROZEN,21,21,True
2,Project 22 registry rows,0,0,True
3,Project 11 frozen identity,apache@shardingsphere,apache@shardingsphere,True
4,Project 12 frozen identity,zolyfarkas@spf4j,zolyfarkas@spf4j,True
5,Project 13 frozen identity,jcabi@jcabi-github,jcabi@jcabi-github,True
6,Project 14 frozen identity,JMRI@JMRI,JMRI@JMRI,True
7,Project 15 frozen identity,eclipse@steady,eclipse@steady,True
8,Project 16 frozen identity,apache@rocketmq,apache@rocketmq,True
9,Project 17 frozen identity,yamcs@Yamcs,yamcs@Yamcs,True



Ranked eligible Project 22 candidates:


,CandidateRank,Project,ProjectSlug,Builds,TrainingBuilds,EvaluationBuilds,RawExecutionRows,RawTrainFailures,RawEvaluationFailures,RawFailingEvaluationBuilds,ModelReadyRows,ModelTrainingRows,ModelEvaluationRows,ModelTrainFailures,ModelEvaluationFailures,ModelFailingEvaluationBuilds,RawUnlinkedRows,ModelUnlinkedRows
0,1,apache@logging-log4j2,apache__logging-log4j2,441,330,111,240253,208,40,39,117968,95812,22156,207,40,39,0,0
1,2,apache@sling,apache__sling,1403,1052,351,265459,767,49,48,113175,107157,6018,765,49,48,0,0
2,3,SonarSource@sonarqube,SonarSource__sonarqube,4286,3214,1072,5635027,1778,20,17,224550,205696,18854,1777,20,17,0,0



Protocol-ineligible candidates:


,Project,Builds,RawTrainFailures,RawEvaluationFailures,ModelTrainFailures,ModelEvaluationFailures,EligibilityReason
0,Graylog2@graylog2-server,3668,280,0,279,0,No raw evaluation failures; No model evaluatio...




=== PROJECT 22 CELL 2 / STEP 1A RESULT ===
Registered projects: 21
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Project 17 identity: yamcs@Yamcs
Project 18 identity: cantaloupe-project@cantaloupe
Project 19 identity: EMResearch@EvoMaster
Project 20 identity: apache@curator
Project 21 identity: facebook@buck
Candidates inspected: 4
Protocol-eligible candidates: 3
Protocol-ineligible candidates: 1
Inspection errors: 0
Runtime-priority ranking rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']

Provisional Project 22 candidate:
Candidate rank: 1
Project: apache@logging-log4j2
Project slug: apache__logging-log4j2
Source directory: /content/datasets/datasets/apache@logging-log4j2

Candidate dimensions:
Builds: 441
TrainingBuilds: 330
Evaluati

In [3]:
# ==================================================================================================
# PROJECT 22 — CELL 3 / STEP 1B
# FINAL SELECTION, CHRONOLOGY FREEZE, SOURCE MANIFEST, AND CHECKPOINT
#
# PROJECT:
#   apache@logging-log4j2
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME Thesis_project_22.ipynb NOTEBOOK.
#
# SAFETY:
# - freezes the Project 22 identity selected by Step 1A;
# - freezes the complete source manifest and source-root SHA-256;
# - freezes the chronological 75/25 build split;
# - validates the exact raw/model dimensions discovered in Step 1A;
# - writes no completion-registry changes;
# - does not access or modify prior-project condition outputs;
# - does not start the Project 22 experiment.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import math
import os

import numpy as np
import pandas as pd


print("=" * 132)
print("=== PROJECT 22 CELL 3 / STEP 1B: FINAL SELECTION AND SOURCE FREEZE ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN PROJECT CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 22
PROJECT_NAME = "apache@logging-log4j2"
PROJECT_SLUG = "apache__logging-log4j2"
CANDIDATE_RANK = 1

STEP1A_PASS_STATUS = (
    "PASS_PROJECT_22_CANDIDATE_DISCOVERY_COMPLETE"
)

STEP1B_PASS_STATUS = (
    "PASS_PROJECT_22_SELECTION_AND_SOURCE_FROZEN"
)

EXPECTED_ARCHIVE_SHA256 = (
    "92af159c116e06e98d7c8348605adb6e9acb25aad982a1ff9fbef7103cb9e36e"
)

EXPECTED_REGISTRY_SHA256 = (
    "79cd6ecb595c5e8ae91a9494e469792716338d144308560a62caf1b9342306b2"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 21

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_DIMENSIONS = {
    "Builds": 441,
    "TrainingBuilds": 330,
    "EvaluationBuilds": 111,

    "RawExecutionRows": 240_253,
    "RawTrainingRows": 172_628,
    "RawEvaluationRows": 67_625,
    "RawTrainFailures": 208,
    "RawEvaluationFailures": 40,
    "RawFailingTrainingBuilds": 193,
    "RawFailingEvaluationBuilds": 39,
    "RawUnlinkedRows": 0,

    "ModelReadyRows": 117_968,
    "ModelTrainingRows": 95_812,
    "ModelEvaluationRows": 22_156,
    "ModelTrainFailures": 207,
    "ModelEvaluationFailures": 40,
    "ModelFailingTrainingBuilds": 192,
    "ModelFailingEvaluationBuilds": 39,
    "ModelUnlinkedRows": 0,
}

REQUIRED_SOURCE_FILES = {
    "builds.csv",
    "exe.csv",
    "dataset.csv",
    "entity_change_history.csv",
    "id_map.csv",
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

ARCHIVE_PATH = (
    THESIS_ROOT
    / "Data"
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)

REGISTRY_PATH = (
    THESIS_ROOT
    / "Notes"
    / "completed_project_registry.csv"
)

SOURCE_DIRECTORY = Path(
    "/content/datasets/datasets/apache@logging-log4j2"
)

SELECTION_ROOT = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / "project_22_selection"
)

STEP1A_STATUS_PATH = (
    SELECTION_ROOT
    / "project_22_step1a_status.json"
)

STEP1A_REPORT_PATH = (
    SELECTION_ROOT
    / "project_22_step1a_report.json"
)

PROVISIONAL_SELECTION_PATH = (
    SELECTION_ROOT
    / "project_22_provisional_selection.json"
)

ELIGIBLE_RANKED_PATH = (
    SELECTION_ROOT
    / "project_22_eligible_candidates_ranked.csv"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_22_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_22_fixed_chronological_builds.csv"
)

SOURCE_SCHEMA_SNAPSHOT_PATH = (
    SELECTION_ROOT
    / "project_22_selected_source_schema_snapshot.csv"
)

STEP1B_VALIDATION_PATH = (
    SELECTION_ROOT
    / "project_22_step1b_validation.csv"
)

STEP1B_REPORT_PATH = (
    SELECTION_ROOT
    / "project_22_step1b_report.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_22_step1b_status.json"
)

SELECTION_CHECKPOINT_PATH = (
    THESIS_ROOT
    / "Notes"
    / "project_22_selection_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_write_csv(
    path,
    frame,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve {label}.\n"
            f"Expected: {expected!r}\n"
            f"Matches: {matches}\n"
            f"Columns: {list(columns)}"
        )

    return matches[0]


def parse_integer_series(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing or non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")


def canonical_root_hash(
    manifest,
):
    required_columns = {
        "RelativePath",
        "SizeBytes",
        "SHA256",
    }

    missing_columns = (
        required_columns
        - set(manifest.columns)
    )

    if missing_columns:
        raise RuntimeError(
            "Source manifest is missing columns:\n"
            + "\n".join(
                sorted(missing_columns)
            )
        )

    digest = hashlib.sha256()

    for row in manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def profile_partition(
    frame,
    build_column,
    verdict_column,
    training_build_ids,
    evaluation_build_ids,
):
    training_mask = frame[
        build_column
    ].isin(
        training_build_ids
    )

    evaluation_mask = frame[
        build_column
    ].isin(
        evaluation_build_ids
    )

    linked_mask = (
        training_mask
        | evaluation_mask
    )

    failure_mask = frame[
        verdict_column
    ].ne(0)

    return {
        "Rows":
            int(len(frame)),

        "TrainingRows":
            int(training_mask.sum()),

        "EvaluationRows":
            int(evaluation_mask.sum()),

        "TrainingFailures":
            int(
                (
                    training_mask
                    & failure_mask
                ).sum()
            ),

        "EvaluationFailures":
            int(
                (
                    evaluation_mask
                    & failure_mask
                ).sum()
            ),

        "FailingTrainingBuilds":
            int(
                frame.loc[
                    training_mask
                    & failure_mask,
                    build_column,
                ].nunique()
            ),

        "FailingEvaluationBuilds":
            int(
                frame.loc[
                    evaluation_mask
                    & failure_mask,
                    build_column,
                ].nunique()
            ),

        "UnlinkedRows":
            int(
                (
                    ~linked_mask
                ).sum()
            ),
    }


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(passed),
    })


# --------------------------------------------------------------------------------------------------
# 4. VALIDATE REQUIRED INPUTS
# --------------------------------------------------------------------------------------------------

required_inputs = [
    ARCHIVE_PATH,
    REGISTRY_PATH,
    STEP1A_STATUS_PATH,
    STEP1A_REPORT_PATH,
    PROVISIONAL_SELECTION_PATH,
    ELIGIBLE_RANKED_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.is_file()
]

if missing_inputs:
    raise FileNotFoundError(
        "Required Project 22 Step 1B inputs are missing:\n"
        + "\n".join(missing_inputs)
    )


if not SOURCE_DIRECTORY.is_dir():
    raise FileNotFoundError(
        "Selected Project 22 source directory is missing:\n"
        f"{SOURCE_DIRECTORY}"
    )


source_file_names = {
    path.name
    for path in SOURCE_DIRECTORY.iterdir()
    if path.is_file()
}


missing_required_source_files = sorted(
    REQUIRED_SOURCE_FILES
    - source_file_names
)


if missing_required_source_files:
    raise FileNotFoundError(
        "Selected Project 22 source is missing required files:\n"
        + "\n".join(
            missing_required_source_files
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. VALIDATE STEP 1A, ARCHIVE, AND REGISTRY
# --------------------------------------------------------------------------------------------------

step1a_status_sha256 = sha256_file(
    STEP1A_STATUS_PATH
)

step1a_report_sha256 = sha256_file(
    STEP1A_REPORT_PATH
)

provisional_selection_sha256 = sha256_file(
    PROVISIONAL_SELECTION_PATH
)

step1a_status = load_json(
    STEP1A_STATUS_PATH
)

step1a_report = load_json(
    STEP1A_REPORT_PATH
)

provisional_selection = load_json(
    PROVISIONAL_SELECTION_PATH
)


if step1a_status.get(
    "Status"
) != STEP1A_PASS_STATUS:
    raise RuntimeError(
        "Project 22 Step 1A status is not PASS."
    )


if step1a_report.get(
    "Status"
) != STEP1A_PASS_STATUS:
    raise RuntimeError(
        "Project 22 Step 1A report is not PASS."
    )


if provisional_selection.get(
    "SelectionState"
) != "PROVISIONAL_PENDING_STEP_1B_FREEZE":
    raise RuntimeError(
        "Project 22 provisional selection state differs."
    )


if provisional_selection.get(
    "Project"
) != PROJECT_NAME:
    raise RuntimeError(
        "Project 22 provisional project differs.\n"
        f"Expected: {PROJECT_NAME}\n"
        f"Actual:   {provisional_selection.get('Project')}"
    )


if provisional_selection.get(
    "ProjectSlug"
) != PROJECT_SLUG:
    raise RuntimeError(
        "Project 22 provisional slug differs."
    )


if Path(
    provisional_selection.get(
        "SourceDirectory",
        "",
    )
) != SOURCE_DIRECTORY:
    raise RuntimeError(
        "Project 22 provisional source directory differs."
    )


if int(
    provisional_selection.get(
        "CandidateRank",
        -1,
    )
) != CANDIDATE_RANK:
    raise RuntimeError(
        "Project 22 provisional candidate rank differs."
    )


if provisional_selection.get(
    "RankingRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Project 22 runtime-priority ranking rule differs."
    )


step1a_dimensions = {
    key: int(value)
    for key, value in provisional_selection.get(
        "Dimensions",
        {},
    ).items()
}

if step1a_dimensions != EXPECTED_DIMENSIONS:
    raise RuntimeError(
        "Project 22 Step 1A dimensions differ from the frozen Step 1B contract.\n"
        f"Expected: {EXPECTED_DIMENSIONS}\n"
        f"Actual:   {step1a_dimensions}"
    )


reserved_in_step1a = sorted(
    provisional_selection.get(
        "ReservedActiveProjectsExcluded",
        [],
    )
)

if reserved_in_step1a != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Project 22 Step 1A active-reservation state differs.\n"
        f"Expected: {EXPECTED_ACTIVE_RESERVATIONS}\n"
        f"Actual:   {reserved_in_step1a}"
    )


archive_sha256 = sha256_file(
    ARCHIVE_PATH
)

if archive_sha256 != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        "Dataset archive SHA-256 differs.\n"
        f"Expected: {EXPECTED_ARCHIVE_SHA256}\n"
        f"Actual:   {archive_sha256}"
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


registry_project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

registry_project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

registry_status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        registry_project_number_column
    ],
    errors="raise",
).astype(int)


if (
    len(registry) != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    ) != list(range(1, 22))
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–21."
    )


if not registry[
    registry_status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–21 are not all COMPLETE_AND_FROZEN."
    )


if registry_project_numbers.eq(
    PROJECT_NUMBER
).any():
    raise RuntimeError(
        "Project 22 is unexpectedly already registered."
    )


if registry[
    registry_project_column
].eq(
    PROJECT_NAME
).any():
    raise RuntimeError(
        "The selected Project 22 identity is already registered."
    )




required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            registry_project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


ranked_candidates = pd.read_csv(
    ELIGIBLE_RANKED_PATH,
    low_memory=False,
)


rank_one_rows = ranked_candidates.loc[
    pd.to_numeric(
        ranked_candidates[
            "CandidateRank"
        ],
        errors="coerce",
    ).eq(
        CANDIDATE_RANK
    )
]


if len(rank_one_rows) != 1:
    raise RuntimeError(
        "Step 1A ranked candidates do not contain exactly one rank-1 row."
    )


rank_one = rank_one_rows.iloc[0]


if (
    str(
        rank_one[
            "Project"
        ]
    ) != PROJECT_NAME
    or str(
        rank_one[
            "ProjectSlug"
        ]
    ) != PROJECT_SLUG
):
    raise RuntimeError(
        "Step 1A rank-1 identity differs."
    )


# --------------------------------------------------------------------------------------------------
# 6. FREEZE COMPLETE SOURCE MANIFEST
# --------------------------------------------------------------------------------------------------

source_files = sorted(
    [
        path
        for path in SOURCE_DIRECTORY.rglob("*")
        if path.is_file()
    ],
    key=lambda path:
        path.relative_to(
            SOURCE_DIRECTORY
        ).as_posix(),
)


if not source_files:
    raise RuntimeError(
        "Selected Project 22 source directory contains no files."
    )


source_manifest_records = []


for source_path in source_files:
    relative_path = source_path.relative_to(
        SOURCE_DIRECTORY
    ).as_posix()

    source_manifest_records.append({
        "RelativePath":
            relative_path,

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


source_manifest = pd.DataFrame(
    source_manifest_records
)


source_root_sha256 = canonical_root_hash(
    source_manifest
)

source_file_count = len(
    source_manifest
)

source_bytes = int(
    source_manifest[
        "SizeBytes"
    ].sum()
)


# --------------------------------------------------------------------------------------------------
# 7. RESOLVE SOURCE SCHEMAS
# --------------------------------------------------------------------------------------------------

source_paths = {
    "builds.csv":
        SOURCE_DIRECTORY
        / "builds.csv",

    "exe.csv":
        SOURCE_DIRECTORY
        / "exe.csv",

    "dataset.csv":
        SOURCE_DIRECTORY
        / "dataset.csv",

    "entity_change_history.csv":
        SOURCE_DIRECTORY
        / "entity_change_history.csv",

    "id_map.csv":
        SOURCE_DIRECTORY
        / "id_map.csv",
}


if (
    SOURCE_DIRECTORY
    / "contributors.csv"
).is_file():
    source_paths[
        "contributors.csv"
    ] = (
        SOURCE_DIRECTORY
        / "contributors.csv"
    )


schema_snapshot_records = []


for filename, file_path in source_paths.items():
    columns = pd.read_csv(
        file_path,
        nrows=0,
    ).columns.tolist()

    schema_snapshot_records.append({
        "File":
            filename,

        "Path":
            str(
                file_path
            ),

        "SizeBytes":
            int(
                file_path.stat().st_size
            ),

        "ColumnCount":
            len(columns),

        "ColumnsJSON":
            json.dumps(
                columns,
                ensure_ascii=False,
            ),
    })


source_schema_snapshot = pd.DataFrame(
    schema_snapshot_records
)


build_columns = pd.read_csv(
    source_paths[
        "builds.csv"
    ],
    nrows=0,
).columns.tolist()

exe_columns = pd.read_csv(
    source_paths[
        "exe.csv"
    ],
    nrows=0,
).columns.tolist()

dataset_columns = pd.read_csv(
    source_paths[
        "dataset.csv"
    ],
    nrows=0,
).columns.tolist()


build_id_column = resolve_column(
    build_columns,
    "id",
    "builds.csv build ID",
)

build_timestamp_column = resolve_column(
    build_columns,
    "started_at",
    "builds.csv timestamp",
)

exe_build_column = resolve_column(
    exe_columns,
    "build",
    "exe.csv build",
)

exe_verdict_column = resolve_column(
    exe_columns,
    "verdict",
    "exe.csv verdict",
)

dataset_build_column = resolve_column(
    dataset_columns,
    "Build",
    "dataset.csv Build",
)

dataset_verdict_column = resolve_column(
    dataset_columns,
    "Verdict",
    "dataset.csv Verdict",
)


# --------------------------------------------------------------------------------------------------
# 8. FREEZE CHRONOLOGY AND 75/25 SPLIT
# --------------------------------------------------------------------------------------------------

builds = pd.read_csv(
    source_paths[
        "builds.csv"
    ],
    usecols=[
        build_id_column,
        build_timestamp_column,
    ],
    low_memory=False,
)


builds[
    build_id_column
] = parse_integer_series(
    builds[
        build_id_column
    ],
    "builds.csv.id",
)


builds[
    build_timestamp_column
] = pd.to_datetime(
    builds[
        build_timestamp_column
    ],
    errors="coerce",
    utc=True,
)


invalid_timestamp_rows = int(
    builds[
        build_timestamp_column
    ].isna().sum()
)


duplicate_build_id_rows = int(
    builds[
        build_id_column
    ].duplicated(
        keep=False
    ).sum()
)


if invalid_timestamp_rows != 0:
    raise RuntimeError(
        "builds.csv contains invalid timestamps."
    )


if duplicate_build_id_rows != 0:
    raise RuntimeError(
        "builds.csv contains duplicate build IDs."
    )


timestamp_group_sizes = builds.groupby(
    build_timestamp_column
).size()


timestamp_tie_groups = int(
    timestamp_group_sizes.gt(1).sum()
)


timestamp_tie_builds = int(
    timestamp_group_sizes.loc[
        timestamp_group_sizes.gt(1)
    ].sum()
)


ordered_builds = (
    builds.sort_values(
        [
            build_timestamp_column,
            build_id_column,
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


number_of_builds = len(
    ordered_builds
)


training_build_count = int(
    math.floor(
        0.75
        * number_of_builds
    )
)


evaluation_build_count = int(
    number_of_builds
    - training_build_count
)


ordered_builds[
    "ChronologyOrder"
] = np.arange(
    1,
    number_of_builds
    + 1,
    dtype=np.int64,
)


ordered_builds[
    "Partition"
] = np.where(
    ordered_builds[
        "ChronologyOrder"
    ].le(
        training_build_count
    ),
    "TRAIN",
    "EVALUATION",
)


ordered_builds[
    "PartitionOrder"
] = (
    ordered_builds.groupby(
        "Partition",
        sort=False,
    ).cumcount()
    + 1
)


fixed_chronology = ordered_builds[
    [
        "ChronologyOrder",
        build_id_column,
        build_timestamp_column,
        "Partition",
        "PartitionOrder",
    ]
].rename(
    columns={
        build_id_column:
            "BuildID",

        build_timestamp_column:
            "StartedAtUTC",
    }
)


training_build_ids = set(
    fixed_chronology.loc[
        fixed_chronology[
            "Partition"
        ].eq(
            "TRAIN"
        ),
        "BuildID",
    ].astype(int).tolist()
)


evaluation_build_ids = set(
    fixed_chronology.loc[
        fixed_chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildID",
    ].astype(int).tolist()
)


partition_overlap = len(
    training_build_ids
    & evaluation_build_ids
)


# --------------------------------------------------------------------------------------------------
# 9. VALIDATE RAW AND MODEL DIMENSIONS
# --------------------------------------------------------------------------------------------------

exe = pd.read_csv(
    source_paths[
        "exe.csv"
    ],
    usecols=[
        exe_build_column,
        exe_verdict_column,
    ],
    low_memory=False,
)


exe[
    exe_build_column
] = parse_integer_series(
    exe[
        exe_build_column
    ],
    "exe.csv.build",
)


exe[
    exe_verdict_column
] = parse_integer_series(
    exe[
        exe_verdict_column
    ],
    "exe.csv.verdict",
)


dataset = pd.read_csv(
    source_paths[
        "dataset.csv"
    ],
    usecols=[
        dataset_build_column,
        dataset_verdict_column,
    ],
    low_memory=False,
)


dataset[
    dataset_build_column
] = parse_integer_series(
    dataset[
        dataset_build_column
    ],
    "dataset.csv.Build",
)


dataset[
    dataset_verdict_column
] = parse_integer_series(
    dataset[
        dataset_verdict_column
    ],
    "dataset.csv.Verdict",
)


raw_profile = profile_partition(
    frame=exe,
    build_column=exe_build_column,
    verdict_column=exe_verdict_column,
    training_build_ids=training_build_ids,
    evaluation_build_ids=evaluation_build_ids,
)


model_profile = profile_partition(
    frame=dataset,
    build_column=dataset_build_column,
    verdict_column=dataset_verdict_column,
    training_build_ids=training_build_ids,
    evaluation_build_ids=evaluation_build_ids,
)


actual_dimensions = {
    "Builds":
        number_of_builds,

    "TrainingBuilds":
        len(
            training_build_ids
        ),

    "EvaluationBuilds":
        len(
            evaluation_build_ids
        ),

    "RawExecutionRows":
        raw_profile[
            "Rows"
        ],

    "RawTrainingRows":
        raw_profile[
            "TrainingRows"
        ],

    "RawEvaluationRows":
        raw_profile[
            "EvaluationRows"
        ],

    "RawTrainFailures":
        raw_profile[
            "TrainingFailures"
        ],

    "RawEvaluationFailures":
        raw_profile[
            "EvaluationFailures"
        ],

    "RawFailingTrainingBuilds":
        raw_profile[
            "FailingTrainingBuilds"
        ],

    "RawFailingEvaluationBuilds":
        raw_profile[
            "FailingEvaluationBuilds"
        ],

    "RawUnlinkedRows":
        raw_profile[
            "UnlinkedRows"
        ],

    "ModelReadyRows":
        model_profile[
            "Rows"
        ],

    "ModelTrainingRows":
        model_profile[
            "TrainingRows"
        ],

    "ModelEvaluationRows":
        model_profile[
            "EvaluationRows"
        ],

    "ModelTrainFailures":
        model_profile[
            "TrainingFailures"
        ],

    "ModelEvaluationFailures":
        model_profile[
            "EvaluationFailures"
        ],

    "ModelFailingTrainingBuilds":
        model_profile[
            "FailingTrainingBuilds"
        ],

    "ModelFailingEvaluationBuilds":
        model_profile[
            "FailingEvaluationBuilds"
        ],

    "ModelUnlinkedRows":
        model_profile[
            "UnlinkedRows"
        ],
}


# --------------------------------------------------------------------------------------------------
# 10. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 1A status",
    STEP1A_PASS_STATUS,
    step1a_status.get(
        "Status"
    ),
    step1a_status.get(
        "Status"
    ) == STEP1A_PASS_STATUS,
)

add_check(
    validation_records,
    "Candidate rank",
    CANDIDATE_RANK,
    int(
        provisional_selection[
            "CandidateRank"
        ]
    ),
    int(
        provisional_selection[
            "CandidateRank"
        ]
    ) == CANDIDATE_RANK,
)

add_check(
    validation_records,
    "Selected project",
    PROJECT_NAME,
    provisional_selection[
        "Project"
    ],
    provisional_selection[
        "Project"
    ] == PROJECT_NAME,
)

add_check(
    validation_records,
    "Selected project slug",
    PROJECT_SLUG,
    provisional_selection[
        "ProjectSlug"
    ],
    provisional_selection[
        "ProjectSlug"
    ] == PROJECT_SLUG,
)

add_check(
    validation_records,
    "Archive SHA-256",
    EXPECTED_ARCHIVE_SHA256,
    archive_sha256,
    archive_sha256
    == EXPECTED_ARCHIVE_SHA256,
)

add_check(
    validation_records,
    "Registry SHA-256",
    EXPECTED_REGISTRY_SHA256,
    registry_sha256_before,
    registry_sha256_before
    == EXPECTED_REGISTRY_SHA256,
)

add_check(
    validation_records,
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(registry),
    len(registry)
    == EXPECTED_REGISTERED_PROJECTS,
)

add_check(
    validation_records,
    "Project 22 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)


for predecessor_number, predecessor_project in required_registered_identities.items():
    actual_predecessor = str(
        registry.loc[
            registry_project_numbers.eq(predecessor_number),
            registry_project_column,
        ].iloc[0]
    )

    add_check(
        validation_records,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        actual_predecessor,
        actual_predecessor == predecessor_project,
    )

add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    reserved_in_step1a,
    reserved_in_step1a == EXPECTED_ACTIVE_RESERVATIONS,
)

add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    provisional_selection.get(
        "RankingRule"
    ),
    provisional_selection.get(
        "RankingRule"
    )
    == EXPECTED_RUNTIME_PRIORITY_RULE,
)

add_check(
    validation_records,
    "Source files",
    "> 0",
    source_file_count,
    source_file_count > 0,
)

add_check(
    validation_records,
    "Source bytes",
    "> 0",
    source_bytes,
    source_bytes > 0,
)

add_check(
    validation_records,
    "Invalid timestamp rows",
    0,
    invalid_timestamp_rows,
    invalid_timestamp_rows == 0,
)

add_check(
    validation_records,
    "Duplicate build-ID rows",
    0,
    duplicate_build_id_rows,
    duplicate_build_id_rows == 0,
)

add_check(
    validation_records,
    "Partition overlap",
    0,
    partition_overlap,
    partition_overlap == 0,
)


for metric, expected_value in EXPECTED_DIMENSIONS.items():
    actual_value = int(
        actual_dimensions[
            metric
        ]
    )

    add_check(
        validation_records,
        metric,
        expected_value,
        actual_value,
        actual_value
        == expected_value,
    )


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print("\nProject 22 Step 1B validation:")

display(
    validation
)


if not failed_validation.empty:
    print("\nFailed Project 22 Step 1B checks:")

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 22 STEP 1B VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 11. WRITE FROZEN OUTPUTS
# --------------------------------------------------------------------------------------------------

SELECTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    source_manifest,
)

atomic_write_csv(
    FIXED_CHRONOLOGY_PATH,
    fixed_chronology,
)

atomic_write_csv(
    SOURCE_SCHEMA_SNAPSHOT_PATH,
    source_schema_snapshot,
)

atomic_write_csv(
    STEP1B_VALIDATION_PATH,
    validation,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "CandidateRank":
        CANDIDATE_RANK,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "RuntimePriorityPurpose":
        (
            "Processing order only; protocol eligibility "
            "and final project set are unchanged"
        ),

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "SelectionState":
        "FINAL_AND_FROZEN",

    "Status":
        STEP1B_PASS_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceDirectory":
        str(
            SOURCE_DIRECTORY
        ),

    "SourceFiles":
        source_file_count,

    "SourceBytes":
        source_bytes,

    "SourceRootSHA256":
        source_root_sha256,

    "ArchiveSHA256":
        archive_sha256,

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "Step1AStatusSHA256":
        step1a_status_sha256,

    "Step1AReportSHA256":
        step1a_report_sha256,

    "ProvisionalSelectionSHA256":
        provisional_selection_sha256,

    "ChronologyRule":
        (
            "started_at ascending; "
            "Build ID descending for timestamp ties"
        ),

    "TimestampTieGroups":
        timestamp_tie_groups,

    "TimestampTieBuilds":
        timestamp_tie_builds,

    "Dimensions":
        actual_dimensions,

    "FrozenSourceManifest":
        str(
            FROZEN_SOURCE_MANIFEST_PATH
        ),

    "FixedChronology":
        str(
            FIXED_CHRONOLOGY_PATH
        ),

    "SourceSchemaSnapshot":
        str(
            SOURCE_SCHEMA_SNAPSHOT_PATH
        ),

    "Validation":
        str(
            STEP1B_VALIDATION_PATH
        ),

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistryModified":
        False,

    "Projects1To21Modified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project22ExperimentStarted":
        False,
}


atomic_write_json(
    STEP1B_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "SelectionCheckpoint":
        True,

    "DoNotChangeProjectIdentity":
        True,

    "DoNotChangeSourceManifest":
        True,

    "DoNotChangeChronology":
        True,

    "DoNotChangeBuildPartitions":
        True,
}


atomic_write_json(
    SELECTION_CHECKPOINT_PATH,
    checkpoint_payload,
)


selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "CandidateRank":
        CANDIDATE_RANK,

    "SelectionState":
        "FINAL_AND_FROZEN",

    "Status":
        STEP1B_PASS_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceFiles":
        source_file_count,

    "SourceBytes":
        source_bytes,

    "SourceRootSHA256":
        source_root_sha256,

    "Builds":
        number_of_builds,

    "TrainingBuilds":
        len(
            training_build_ids
        ),

    "EvaluationBuilds":
        len(
            evaluation_build_ids
        ),

    "SelectionCheckpoint":
        str(
            SELECTION_CHECKPOINT_PATH
        ),

    "SelectionCheckpointSHA256":
        selection_checkpoint_sha256,

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "Project22ExperimentStarted":
        False,
}


atomic_write_json(
    STEP1B_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 12. FINAL READBACK AND IMMUTABILITY CHECKS
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Project 22 Step 1B."
    )


final_manifest_records = []


for row in source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIRECTORY
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise RuntimeError(
            "A frozen Project 22 source file disappeared:\n"
            f"{source_path}"
        )

    final_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_source_manifest = pd.DataFrame(
    final_manifest_records
)


final_source_root_sha256 = canonical_root_hash(
    final_source_manifest
)


if final_source_root_sha256 != source_root_sha256:
    raise RuntimeError(
        "Project 22 source changed during Step 1B."
    )


checkpoint_readback = load_json(
    SELECTION_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP1B_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP1B_PASS_STATUS:
    raise RuntimeError(
        "Project 22 selection checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP1B_PASS_STATUS:
    raise RuntimeError(
        "Project 22 Step 1B status readback failed."
    )


# --------------------------------------------------------------------------------------------------
# 13. DISPLAY FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\nFrozen Project 22 source manifest:")

display(
    source_manifest
)


print("\nFixed Project 22 chronology sample:")

display(
    pd.concat(
        [
            fixed_chronology.head(10),
            fixed_chronology.tail(10),
        ],
        ignore_index=True,
    )
)


print("\n")
print("=" * 132)
print("=== PROJECT 22 CELL 3 / STEP 1B RESULT ===")
print("=" * 132)


print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print(
    "Candidate rank:",
    CANDIDATE_RANK,
)

for predecessor_number in sorted(required_registered_identities):
    print(
        f"Project {predecessor_number} identity:",
        required_registered_identities[predecessor_number],
    )

print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)

print(
    "Selection state:",
    "FINAL_AND_FROZEN",
)


print("\nFrozen source:")

print(
    "Source directory:",
    SOURCE_DIRECTORY,
)

print(
    "Source files:",
    source_file_count,
)

print(
    "Source bytes:",
    source_bytes,
)

print(
    "Source root SHA-256:",
    source_root_sha256,
)


print("\nChronology:")

print(
    "Rule: started_at ascending; "
    "Build ID descending for timestamp ties"
)

print(
    "Builds:",
    number_of_builds,
)

print(
    "Training / evaluation builds:",
    len(
        training_build_ids
    ),
    "/",
    len(
        evaluation_build_ids
    ),
)

print(
    "Timestamp tie groups:",
    timestamp_tie_groups,
)

print(
    "Partition overlap:",
    partition_overlap,
)


print("\nRaw and model dimensions:")

for metric in EXPECTED_DIMENSIONS:
    print(
        f"{metric}:",
        actual_dimensions[
            metric
        ],
    )


print("\nIsolation:")

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–21 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Project 22 experiment started:",
    False,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print("\nSelection checkpoint:")

print(
    SELECTION_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    selection_checkpoint_sha256,
)


print(
    "\nSTATUS:",
    STEP1B_PASS_STATUS,
)

print("=" * 132)


=== PROJECT 22 CELL 3 / STEP 1B: FINAL SELECTION AND SOURCE FREEZE ===

Project 22 Step 1B validation:


,Check,Expected,Actual,Pass
0,Step 1A status,PASS_PROJECT_22_CANDIDATE_DISCOVERY_COMPLETE,PASS_PROJECT_22_CANDIDATE_DISCOVERY_COMPLETE,True
1,Candidate rank,1,1,True
2,Selected project,apache@logging-log4j2,apache@logging-log4j2,True
3,Selected project slug,apache__logging-log4j2,apache__logging-log4j2,True
4,Archive SHA-256,92af159c116e06e98d7c8348605adb6e9acb25aad982a1...,92af159c116e06e98d7c8348605adb6e9acb25aad982a1...,True
5,Registry SHA-256,79cd6ecb595c5e8ae91a9494e469792716338d14430856...,79cd6ecb595c5e8ae91a9494e469792716338d14430856...,True
6,Registry rows,21,21,True
7,Project 22 registry rows,0,0,True
8,Project 11 frozen identity,apache@shardingsphere,apache@shardingsphere,True
9,Project 12 frozen identity,zolyfarkas@spf4j,zolyfarkas@spf4j,True



Frozen Project 22 source manifest:


,RelativePath,SizeBytes,SHA256
0,builds.csv,39145,83e1a3eb9b98939af3cea70987b9414b349675bb3021f2...
1,contributors.csv,11203,5b4778b324441f54f3f41d6be1f444859bb62046521df7...
2,dataset.csv,86829936,b3f1fa39b105c748cafd114c57c73ff06c4a0ce2f8353d...
3,entity_change_history.csv,6232413,175625f6eea926510a062485b521711739362991227b21...
4,exe.csv,7364974,32d6e22067b63089e7e971d7b5d29474c0172df31d2546...
5,id_map.csv,809079,c918a87b6e1b62e8ae7128bda7469f11addbe54d32472a...



Fixed Project 22 chronology sample:


,ChronologyOrder,BuildID,StartedAtUTC,Partition,PartitionOrder
0,1,579990414,2019-09-02 21:57:11+00:00,TRAIN,1
1,2,580002753,2019-09-02 22:52:41+00:00,TRAIN,2
2,3,580008415,2019-09-02 23:21:31+00:00,TRAIN,3
3,4,580429720,2019-09-03 21:04:33+00:00,TRAIN,4
4,5,581765623,2019-09-06 17:05:27+00:00,TRAIN,5
5,6,581803423,2019-09-06 18:39:21+00:00,TRAIN,6
6,7,584181134,2019-09-12 14:50:26+00:00,TRAIN,7
7,8,584279661,2019-09-12 19:05:06+00:00,TRAIN,8
8,9,585026724,2019-09-14 19:58:59+00:00,TRAIN,9
9,10,585030074,2019-09-14 20:03:59+00:00,TRAIN,10




=== PROJECT 22 CELL 3 / STEP 1B RESULT ===

Project identity:
Project number: 22
Project: apache@logging-log4j2
Project slug: apache__logging-log4j2
Candidate rank: 1
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Project 17 identity: yamcs@Yamcs
Project 18 identity: cantaloupe-project@cantaloupe
Project 19 identity: EMResearch@EvoMaster
Project 20 identity: apache@curator
Project 21 identity: facebook@buck
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']
Selection state: FINAL_AND_FROZEN

Frozen source:
Source directory: /content/datasets/datasets/apache@logging-log4j2
Source files: 6
Source bytes: 101286750
Source root SHA-256: 281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64f842964c896c6ac334

Ch

In [4]:
# ==================================================================================================
# PROJECT 22 — CELL 4 / STEP 2A
# SOURCE SCHEMA, BUILD-TEST JOIN, ID-MAP ORIENTATION,
# COMMIT MATCHING, AND BUILD-ENTITY PREFLIGHT
#
# PROJECT:
#   apache@logging-log4j2
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME CURRENT THESIS NOTEBOOK.
#
# PURPOSE:
# - validate the frozen Project 22 identity, source root, chronology, and registry state;
# - validate all Build-Test joins and clean-verdict alignment;
# - validate all 19 REC columns;
# - resolve id_map.csv orientation without assuming EntityId uniqueness;
# - preserve duplicate EntityId rows as valid path aliases;
# - map build commits to entity-change history;
# - write the build-entity mapping required by clean REC reconstruction;
# - record unmatched commits/builds for explicit Step 2B audit.
#
# SAFETY:
# - no noise injection;
# - no model fitting;
# - no completion-registry write;
# - no prior-project condition-output access;
# - no Project 22 experiment execution.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd


print("=" * 132)
print("=== PROJECT 22 CELL 4 / STEP 2A: SOURCE SCHEMA AND JOIN-STRUCTURE VALIDATION ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 22
PROJECT_NAME = "apache@logging-log4j2"
PROJECT_SLUG = "apache__logging-log4j2"
PROJECT_SHORT = "LOG4J2"

SOURCE_DIR = Path(
    "/content/datasets/datasets/apache@logging-log4j2"
)

EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_22_SELECTION_AND_SOURCE_FROZEN"
)

STEP2A_STATUS = (
    "PASS_PROJECT_22_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_VALIDATED"
)

EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "a7387d9495c71dce5d2c21ff08b0afd80d9c5251b5885a82e4052c7506ee7890"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64f842964c896c6ac334"
)

EXPECTED_REGISTRY_SHA256 = (
    "79cd6ecb595c5e8ae91a9494e469792716338d144308560a62caf1b9342306b2"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 101_286_750

EXPECTED_BUILDS = 441
EXPECTED_TRAIN_BUILDS = 330
EXPECTED_EVAL_BUILDS = 111

EXPECTED_TIMESTAMP_TIE_GROUPS = 1

EXPECTED_RAW_ROWS = 240_253
EXPECTED_RAW_TRAIN_ROWS = 172_628
EXPECTED_RAW_EVAL_ROWS = 67_625
EXPECTED_RAW_TRAIN_FAILURES = 208
EXPECTED_RAW_EVAL_FAILURES = 40

EXPECTED_MODEL_ROWS = 117_968
EXPECTED_MODEL_TRAIN_ROWS = 95_812
EXPECTED_MODEL_EVAL_ROWS = 22_156
EXPECTED_MODEL_TRAIN_FAILURES = 207
EXPECTED_MODEL_EVAL_FAILURES = 40

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = {
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
}

FILE_HISTORY_REC = {
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
}

REQUIRED_SOURCE_FILES = [
    "builds.csv",
    "contributors.csv",
    "dataset.csv",
    "entity_change_history.csv",
    "exe.csv",
    "id_map.csv",
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_22_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_22_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_22_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_22_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_22_fixed_chronological_builds.csv"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

SOURCE_SCHEMA_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_source_schema_profile.csv"
)

JOIN_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_test_join_audit.csv"
)

REC_CLASS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_rec_feature_classification.csv"
)

BUILD_TOKEN_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_commit_token_profile.csv"
)

COMMIT_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_commit_matching_audit.csv"
)

BUILD_ENTITY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
)

ID_ORIENTATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_id_map_orientation_audit.csv"
)

RESOLVED_ID_MAP_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_resolved_id_map_aliases.csv.gz"
)

ENTITY_ID_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_entity_id_map_audit.csv"
)

MAPPING_INCOMPLETE_BUILDS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_mapping_incomplete_builds.csv"
)

VALIDATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2a_validation.csv"
)

SUMMARY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_entity_mapping_summary.json"
)

REPORT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2a_report.json"
)

STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2a_status.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
    compression=None,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
        compression=compression,
    )

    os.replace(
        temporary_path,
        path,
    )


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}.\n"
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def parse_int(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} has "
            f"{int(numeric.isna().sum())} "
            "missing/non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(
            array
        ),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype(
        "int64"
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def normalise_commit(
    value,
):
    if pd.isna(
        value
    ):
        return ""

    text = str(
        value
    ).strip().lower()

    if not text:
        return ""

    matches = re.findall(
        r"[0-9a-f]{7,64}",
        text,
        flags=re.I,
    )

    if matches:
        return matches[0].lower()

    return re.sub(
        r"[^a-z0-9]",
        "",
        text,
    )


def extract_commit_tokens(
    value,
):
    if pd.isna(
        value
    ):
        return []

    text = str(
        value
    ).strip()

    if not text:
        return []

    tokens = re.findall(
        r"[0-9a-fA-F]{7,64}",
        text,
    )

    if not tokens:
        tokens = re.split(
            r"[\s,;|#]+",
            text,
        )

    result = []
    seen = set()

    for token in tokens:
        token = normalise_commit(
            token
        )

        if (
            token
            and token not in seen
        ):
            seen.add(
                token
            )

            result.append(
                token
            )

    return result


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS
# --------------------------------------------------------------------------------------------------

required_inputs = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(
        path
    ).is_file()
]

missing_inputs.extend(
    str(
        SOURCE_DIR
        / filename
    )
    for filename in REQUIRED_SOURCE_FILES
    if not (
        SOURCE_DIR
        / filename
    ).is_file()
)


if missing_inputs:
    raise FileNotFoundError(
        "Required Project 22 Step 2A inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. VALIDATE FROZEN SELECTION, REGISTRY, AND SOURCE ROOT
# --------------------------------------------------------------------------------------------------

selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = load_json(
    STEP1B_STATUS_PATH
)


if (
    selection_checkpoint_sha256
    != EXPECTED_SELECTION_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 22 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_CHECKPOINT_SHA256}\n"
        f"Actual:   {selection_checkpoint_sha256}"
    )


if (
    selection_checkpoint.get(
        "Status"
    ) != EXPECTED_STEP1B_STATUS
    or step1b_status.get(
        "Status"
    ) != EXPECTED_STEP1B_STATUS
):
    raise RuntimeError(
        "Project 22 Step 1B is not frozen successfully."
    )


if (
    selection_checkpoint.get(
        "Project"
    ) != PROJECT_NAME
    or selection_checkpoint.get(
        "ProjectSlug"
    ) != PROJECT_SLUG
):
    raise RuntimeError(
        "Frozen Project 22 identity differs."
    )


if selection_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Frozen Project 22 runtime-priority rule differs."
    )


active_reservations = selection_checkpoint.get(
    "ActiveReservations",
    [],
)


if active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Frozen Project 22 active-reservation state differs.\n"
        f"Expected: {EXPECTED_ACTIVE_RESERVATIONS}\n"
        f"Actual:   {active_reservations}"
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(int)


if (
    len(
        registry
    ) != 20
    or sorted(
        project_numbers.tolist()
    ) != list(
        range(
            1,
            22,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–21."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–21 are not all COMPLETE_AND_FROZEN."
    )


required_registered_identities = {
    11:
        "apache@shardingsphere",

    12:
        "zolyfarkas@spf4j",

    13:
        "jcabi@jcabi-github",

    14:
        "JMRI@JMRI",

    15:
        "eclipse@steady",

    16:
        "apache@rocketmq",

    17:
        "yamcs@Yamcs",

    18:
        "cantaloupe-project@cantaloupe",

    19:
        "EMResearch@EvoMaster",

    20:
        "apache@curator",

    21:
        "facebook@buck",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


if (
    project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 22 is unexpectedly already registered."
    )


frozen_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_manifest_records = []


for row in frozen_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            "A frozen Project 22 source file is missing:\n"
            f"{source_path}"
        )

    current_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


current_manifest = pd.DataFrame(
    current_manifest_records
)

current_source_root = source_root_hash(
    current_manifest
)

current_source_bytes = int(
    current_manifest[
        "SizeBytes"
    ].sum()
)


if (
    current_source_root
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "Frozen Project 22 source root differs.\n"
        f"Expected: {EXPECTED_SOURCE_ROOT_SHA256}\n"
        f"Actual:   {current_source_root}"
    )


# --------------------------------------------------------------------------------------------------
# 6. RESOLVE SOURCE SCHEMAS
# --------------------------------------------------------------------------------------------------

paths = {
    "builds.csv":
        SOURCE_DIR
        / "builds.csv",

    "contributors.csv":
        SOURCE_DIR
        / "contributors.csv",

    "dataset.csv":
        SOURCE_DIR
        / "dataset.csv",

    "entity_change_history.csv":
        SOURCE_DIR
        / "entity_change_history.csv",

    "exe.csv":
        SOURCE_DIR
        / "exe.csv",

    "id_map.csv":
        SOURCE_DIR
        / "id_map.csv",
}


schema_rows = []
headers = {}


for filename, file_path in paths.items():
    columns = pd.read_csv(
        file_path,
        nrows=0,
    ).columns.tolist()

    headers[
        filename
    ] = columns

    schema_rows.append({
        "File":
            filename,

        "Path":
            str(
                file_path
            ),

        "SizeBytes":
            int(
                file_path.stat().st_size
            ),

        "ColumnCount":
            len(
                columns
            ),

        "ColumnsJSON":
            json.dumps(
                columns,
                ensure_ascii=False,
            ),
    })


source_schema = pd.DataFrame(
    schema_rows
)


build_id_column = resolve_column(
    headers[
        "builds.csv"
    ],
    "id",
    "builds.csv id",
)

build_commit_column = resolve_column(
    headers[
        "builds.csv"
    ],
    "commits",
    "builds.csv commits",
)

build_time_column = resolve_column(
    headers[
        "builds.csv"
    ],
    "started_at",
    "builds.csv started_at",
)


exe_test_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "test",
    "exe.csv test",
)

exe_build_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "build",
    "exe.csv build",
)

exe_job_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "job",
    "exe.csv job",
)

exe_verdict_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "verdict",
    "exe.csv verdict",
)

exe_duration_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "duration",
    "exe.csv duration",
)


dataset_build_column = resolve_column(
    headers[
        "dataset.csv"
    ],
    "Build",
    "dataset.csv Build",
)

dataset_test_column = resolve_column(
    headers[
        "dataset.csv"
    ],
    "Test",
    "dataset.csv Test",
)

dataset_verdict_column = resolve_column(
    headers[
        "dataset.csv"
    ],
    "Verdict",
    "dataset.csv Verdict",
)


entity_id_column = resolve_column(
    headers[
        "entity_change_history.csv"
    ],
    "EntityId",
    "entity_change_history.csv EntityId",
)

entity_commit_column = resolve_column(
    headers[
        "entity_change_history.csv"
    ],
    "Commit",
    "entity_change_history.csv Commit",
)


id_key_column = resolve_column(
    headers[
        "id_map.csv"
    ],
    "key",
    "id_map.csv key",
)

id_value_column = resolve_column(
    headers[
        "id_map.csv"
    ],
    "value",
    "id_map.csv value",
)


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature
    not in headers[
        "dataset.csv"
    ]
]


predictor_columns = [
    column
    for column in headers[
        "dataset.csv"
    ]
    if column
    not in {
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    }
]


if missing_rec_features:
    raise RuntimeError(
        "dataset.csv is missing REC columns:\n"
        + "\n".join(
            missing_rec_features
        )
    )


# --------------------------------------------------------------------------------------------------
# 7. LOAD CHRONOLOGY AND SOURCE TABLES
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)


chronology[
    "BuildID"
] = parse_int(
    chronology[
        "BuildID"
    ],
    "chronology.BuildID",
)

chronology[
    "StartedAtUTC"
] = pd.to_datetime(
    chronology[
        "StartedAtUTC"
    ],
    errors="coerce",
    utc=True,
)

if chronology[
    "StartedAtUTC"
].isna().any():
    raise RuntimeError(
        "Frozen chronology contains invalid StartedAtUTC values."
    )

timestamp_tie_group_count = int(
    chronology.groupby(
        "StartedAtUTC",
        dropna=False,
    )[
        "BuildID"
    ].size().gt(
        1
    ).sum()
)

if int(
    selection_checkpoint.get(
        "TimestampTieGroups",
        -1,
    )
) != EXPECTED_TIMESTAMP_TIE_GROUPS:
    raise RuntimeError(
        "Frozen selection checkpoint timestamp-tie count differs."
    )


training_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "TRAIN"
        ),
        "BuildID",
    ].astype(
        int
    )
)


evaluation_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildID",
    ].astype(
        int
    )
)


all_builds = (
    training_builds
    | evaluation_builds
)


build_order = (
    chronology.set_index(
        "BuildID"
    )[
        "ChronologyOrder"
    ]
    .astype(
        int
    )
    .to_dict()
)


builds = pd.read_csv(
    paths[
        "builds.csv"
    ],
    usecols=[
        build_id_column,
        build_commit_column,
        build_time_column,
    ],
    low_memory=False,
)


builds[
    build_id_column
] = parse_int(
    builds[
        build_id_column
    ],
    "builds.csv.id",
)


builds[
    build_time_column
] = pd.to_datetime(
    builds[
        build_time_column
    ],
    errors="coerce",
    utc=True,
)


if builds[
    build_time_column
].isna().any():
    raise RuntimeError(
        "builds.csv contains invalid timestamps."
    )


exe = pd.read_csv(
    paths[
        "exe.csv"
    ],
    usecols=[
        exe_test_column,
        exe_build_column,
        exe_job_column,
        exe_verdict_column,
        exe_duration_column,
    ],
    low_memory=False,
)


exe[
    exe_build_column
] = parse_int(
    exe[
        exe_build_column
    ],
    "exe.csv.build",
)


exe[
    exe_test_column
] = parse_int(
    exe[
        exe_test_column
    ],
    "exe.csv.test",
)


exe[
    exe_verdict_column
] = parse_int(
    exe[
        exe_verdict_column
    ],
    "exe.csv.verdict",
)


exe[
    exe_duration_column
] = pd.to_numeric(
    exe[
        exe_duration_column
    ],
    errors="coerce",
)


dataset = pd.read_csv(
    paths[
        "dataset.csv"
    ],
    low_memory=False,
)


dataset[
    dataset_build_column
] = parse_int(
    dataset[
        dataset_build_column
    ],
    "dataset.csv.Build",
)


dataset[
    dataset_test_column
] = parse_int(
    dataset[
        dataset_test_column
    ],
    "dataset.csv.Test",
)


dataset[
    dataset_verdict_column
] = parse_int(
    dataset[
        dataset_verdict_column
    ],
    "dataset.csv.Verdict",
)


# --------------------------------------------------------------------------------------------------
# 8. BUILD-TEST JOIN VALIDATION
# --------------------------------------------------------------------------------------------------

raw_duplicate_pairs = int(
    exe.duplicated(
        [
            exe_build_column,
            exe_test_column,
        ],
        keep=False,
    ).sum()
)


model_duplicate_pairs = int(
    dataset.duplicated(
        [
            dataset_build_column,
            dataset_test_column,
        ],
        keep=False,
    ).sum()
)


raw_unlinked_build_rows = int(
    (
        ~exe[
            exe_build_column
        ].isin(
            all_builds
        )
    ).sum()
)


model_unlinked_build_rows = int(
    (
        ~dataset[
            dataset_build_column
        ].isin(
            all_builds
        )
    ).sum()
)


nonfinite_duration_rows = int(
    (
        ~np.isfinite(
            exe[
                exe_duration_column
            ].to_numpy(
                dtype=float
            )
        )
    ).sum()
)


negative_duration_rows = int(
    exe[
        exe_duration_column
    ].lt(
        0
    ).sum()
)


if (
    raw_duplicate_pairs
    or model_duplicate_pairs
):
    raise RuntimeError(
        "Duplicate Build-Test pairs were found.\n"
        f"Raw duplicate rows: {raw_duplicate_pairs}\n"
        f"Model duplicate rows: {model_duplicate_pairs}"
    )


raw_pairs = exe[
    [
        exe_build_column,
        exe_test_column,
        exe_verdict_column,
        exe_duration_column,
    ]
].rename(
    columns={
        exe_build_column:
            "Build",

        exe_test_column:
            "Test",

        exe_verdict_column:
            "RawVerdict",

        exe_duration_column:
            "RawDuration",
    }
)


model_pairs = dataset[
    [
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    ]
].rename(
    columns={
        dataset_build_column:
            "Build",

        dataset_test_column:
            "Test",

        dataset_verdict_column:
            "ModelVerdict",
    }
)


joined = model_pairs.merge(
    raw_pairs,
    on=[
        "Build",
        "Test",
    ],
    how="left",
    validate="one_to_one",
    indicator=True,
)


missing_model_raw_links = int(
    joined[
        "_merge"
    ].ne(
        "both"
    ).sum()
)


verdict_mismatches = int(
    joined[
        "ModelVerdict"
    ].ne(
        joined[
            "RawVerdict"
        ]
    ).sum()
)


raw_training_mask = exe[
    exe_build_column
].isin(
    training_builds
)


raw_evaluation_mask = exe[
    exe_build_column
].isin(
    evaluation_builds
)


raw_training_rows = int(
    raw_training_mask.sum()
)

raw_evaluation_rows = int(
    raw_evaluation_mask.sum()
)

raw_training_failures = int(
    (
        raw_training_mask
        & exe[
            exe_verdict_column
        ].ne(
            0
        )
    ).sum()
)

raw_evaluation_failures = int(
    (
        raw_evaluation_mask
        & exe[
            exe_verdict_column
        ].ne(
            0
        )
    ).sum()
)


model_training_mask = joined[
    "Build"
].isin(
    training_builds
)


model_evaluation_mask = joined[
    "Build"
].isin(
    evaluation_builds
)


model_training_rows = int(
    model_training_mask.sum()
)

model_evaluation_rows = int(
    model_evaluation_mask.sum()
)

model_training_failures = int(
    (
        model_training_mask
        & joined[
            "ModelVerdict"
        ].ne(
            0
        )
    ).sum()
)

model_evaluation_failures = int(
    (
        model_evaluation_mask
        & joined[
            "ModelVerdict"
        ].ne(
            0
        )
    ).sum()
)


join_audit = pd.DataFrame([
    (
        "RawRows",
        EXPECTED_RAW_ROWS,
        len(
            exe
        ),
    ),

    (
        "RawTrainingRows",
        EXPECTED_RAW_TRAIN_ROWS,
        raw_training_rows,
    ),

    (
        "RawEvaluationRows",
        EXPECTED_RAW_EVAL_ROWS,
        raw_evaluation_rows,
    ),

    (
        "RawTrainingFailures",
        EXPECTED_RAW_TRAIN_FAILURES,
        raw_training_failures,
    ),

    (
        "RawEvaluationFailures",
        EXPECTED_RAW_EVAL_FAILURES,
        raw_evaluation_failures,
    ),

    (
        "ModelRows",
        EXPECTED_MODEL_ROWS,
        len(
            dataset
        ),
    ),

    (
        "ModelTrainingRows",
        EXPECTED_MODEL_TRAIN_ROWS,
        model_training_rows,
    ),

    (
        "ModelEvaluationRows",
        EXPECTED_MODEL_EVAL_ROWS,
        model_evaluation_rows,
    ),

    (
        "ModelTrainingFailures",
        EXPECTED_MODEL_TRAIN_FAILURES,
        model_training_failures,
    ),

    (
        "ModelEvaluationFailures",
        EXPECTED_MODEL_EVAL_FAILURES,
        model_evaluation_failures,
    ),

    (
        "RawDuplicateBuildTestRows",
        0,
        raw_duplicate_pairs,
    ),

    (
        "ModelDuplicateBuildTestRows",
        0,
        model_duplicate_pairs,
    ),

    (
        "MissingModelRawLinks",
        0,
        missing_model_raw_links,
    ),

    (
        "ModelRawVerdictMismatches",
        0,
        verdict_mismatches,
    ),

    (
        "NonFiniteDurationRows",
        0,
        nonfinite_duration_rows,
    ),

    (
        "NegativeDurationRows",
        0,
        negative_duration_rows,
    ),

    (
        "RawUnlinkedBuildRows",
        0,
        raw_unlinked_build_rows,
    ),

    (
        "ModelUnlinkedBuildRows",
        0,
        model_unlinked_build_rows,
    ),
], columns=[
    "Metric",
    "Expected",
    "Actual",
])


join_audit[
    "Pass"
] = (
    join_audit[
        "Expected"
    ].astype(
        str
    )
    == join_audit[
        "Actual"
    ].astype(
        str
    )
)


# --------------------------------------------------------------------------------------------------
# 9. REC FEATURE CLASSIFICATION
# --------------------------------------------------------------------------------------------------

rec_classification = pd.DataFrame([
    {
        "Feature":
            feature,

        "FeatureClass":
            (
                "VERDICT_DEPENDENT"
                if feature
                in VERDICT_DEPENDENT_REC
                else "VERDICT_INDEPENDENT"
            ),

        "FileHistoryFeature":
            feature
            in FILE_HISTORY_REC,

        "PresentInDataset":
            feature
            in dataset.columns,
    }
    for feature in REC_FEATURES
])


# --------------------------------------------------------------------------------------------------
# 10. BUILD COMMIT TOKENS
# --------------------------------------------------------------------------------------------------

token_rows = []
builds_without_tokens = 0


for (
    build_id,
    raw_commits,
) in builds[
    [
        build_id_column,
        build_commit_column,
    ]
].itertuples(
    index=False,
    name=None,
):
    tokens = extract_commit_tokens(
        raw_commits
    )

    if not tokens:
        builds_without_tokens += 1

    for token_order, token in enumerate(
        tokens,
        start=1,
    ):
        token_rows.append({
            "BuildID":
                int(
                    build_id
                ),

            "ChronologyOrder":
                int(
                    build_order[
                        int(
                            build_id
                        )
                    ]
                ),

            "RawCommits":
                str(
                    raw_commits
                ),

            "TokenOrder":
                token_order,

            "CommitToken":
                token,
        })


build_tokens = pd.DataFrame(
    token_rows
)


if build_tokens.empty:
    raise RuntimeError(
        "No build commit tokens could be extracted."
    )


build_tokens = (
    build_tokens.sort_values(
        [
            "ChronologyOrder",
            "TokenOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 11. ENTITY HISTORY AND ALIAS-AWARE ID MAP
# --------------------------------------------------------------------------------------------------

entity_history = pd.read_csv(
    paths[
        "entity_change_history.csv"
    ],
    usecols=[
        entity_id_column,
        entity_commit_column,
    ],
    low_memory=False,
)


entity_history[
    entity_id_column
] = parse_int(
    entity_history[
        entity_id_column
    ],
    "entity_change_history.csv.EntityId",
)


entity_history[
    "NormalisedCommit"
] = entity_history[
    entity_commit_column
].map(
    normalise_commit
)


entity_history = (
    entity_history.loc[
        entity_history[
            "NormalisedCommit"
        ].ne(
            ""
        ),
        [
            entity_id_column,
            "NormalisedCommit",
        ],
    ]
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)


history_entity_ids = set(
    entity_history[
        entity_id_column
    ].astype(
        int
    )
)


history_commits = sorted(
    entity_history[
        "NormalisedCommit"
    ].unique().tolist()
)


history_commit_set = set(
    history_commits
)


id_raw = pd.read_csv(
    paths[
        "id_map.csv"
    ],
    usecols=[
        id_key_column,
        id_value_column,
    ],
    dtype=str,
    keep_default_na=False,
    low_memory=False,
)


orientation_rows = []


for column in [
    id_key_column,
    id_value_column,
]:
    numeric = pd.to_numeric(
        id_raw[
            column
        ],
        errors="coerce",
    )

    numeric_filled = numeric.fillna(
        0
    )

    valid_integral = (
        numeric.notna()
        & np.isclose(
            numeric_filled,
            np.floor(
                numeric_filled
            ),
            rtol=0,
            atol=0,
        )
    )

    parsed_ids = set(
        numeric.loc[
            valid_integral
        ].astype(
            "int64"
        )
    )

    overlap = len(
        parsed_ids
        & history_entity_ids
    )

    orientation_rows.append({
        "Column":
            column,

        "Rows":
            len(
                id_raw
            ),

        "IntegralNumericRows":
            int(
                valid_integral.sum()
            ),

        "InvalidOrNonNumericRows":
            int(
                (
                    ~valid_integral
                ).sum()
            ),

        "UniqueIntegralIDs":
            len(
                parsed_ids
            ),

        "MatchingHistoryEntityIDs":
            overlap,

        "HistoryEntityCoveragePercent":
            (
                100.0
                * overlap
                / len(
                    history_entity_ids
                )
                if history_entity_ids
                else 0.0
            ),
    })


id_orientation = pd.DataFrame(
    orientation_rows
)


best_orientation = (
    id_orientation.sort_values(
        [
            "MatchingHistoryEntityIDs",
            "IntegralNumericRows",
        ],
        ascending=[
            False,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if len(
    best_orientation
) < 2:
    raise RuntimeError(
        "id_map orientation audit is incomplete."
    )


if (
    int(
        best_orientation.loc[
            0,
            "MatchingHistoryEntityIDs",
        ]
    )
    == int(
        best_orientation.loc[
            1,
            "MatchingHistoryEntityIDs",
        ]
    )
    and int(
        best_orientation.loc[
            0,
            "IntegralNumericRows",
        ]
    )
    == int(
        best_orientation.loc[
            1,
            "IntegralNumericRows",
        ]
    )
):
    raise RuntimeError(
        "Could not uniquely resolve the EntityId column in id_map.csv."
    )


resolved_id_column = str(
    best_orientation.loc[
        0,
        "Column",
    ]
)


resolved_path_column = (
    id_value_column
    if resolved_id_column
    == id_key_column
    else id_key_column
)


resolved_numeric = pd.to_numeric(
    id_raw[
        resolved_id_column
    ],
    errors="coerce",
)


resolved_numeric_filled = resolved_numeric.fillna(
    0
)


valid_resolved = (
    resolved_numeric.notna()
    & np.isclose(
        resolved_numeric_filled,
        np.floor(
            resolved_numeric_filled
        ),
        rtol=0,
        atol=0,
    )
)


invalid_resolved_rows = int(
    (
        ~valid_resolved
    ).sum()
)


if invalid_resolved_rows:
    raise RuntimeError(
        "Resolved id_map EntityId column contains "
        f"{invalid_resolved_rows} invalid rows."
    )


resolved_id_map = pd.DataFrame({
    "EntityPath":
        id_raw[
            resolved_path_column
        ].astype(
            str
        ).str.strip(),

    "EntityId":
        resolved_numeric.astype(
            "int64"
        ),
})


empty_path_rows = int(
    resolved_id_map[
        "EntityPath"
    ].eq(
        ""
    ).sum()
)


exact_duplicate_rows = int(
    len(
        resolved_id_map
    )
    - len(
        resolved_id_map.drop_duplicates(
            [
                "EntityPath",
                "EntityId",
            ]
        )
    )
)


resolved_id_map = (
    resolved_id_map.drop_duplicates(
        [
            "EntityPath",
            "EntityId",
        ]
    )
    .sort_values(
        [
            "EntityId",
            "EntityPath",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


duplicate_entity_id_rows = int(
    resolved_id_map.duplicated(
        "EntityId",
        keep=False,
    ).sum()
)


entity_ids_with_multiple_paths = int(
    resolved_id_map.groupby(
        "EntityId"
    )[
        "EntityPath"
    ].nunique().gt(
        1
    ).sum()
)


paths_with_multiple_ids = int(
    resolved_id_map.groupby(
        "EntityPath"
    )[
        "EntityId"
    ].nunique().gt(
        1
    ).sum()
)


maximum_paths_per_entity = int(
    resolved_id_map.groupby(
        "EntityId"
    )[
        "EntityPath"
    ].nunique().max()
)


id_map_entity_ids = set(
    resolved_id_map[
        "EntityId"
    ].astype(
        int
    )
)


# --------------------------------------------------------------------------------------------------
# 12. COMMIT MATCHING AND BUILD-ENTITY MAP
# --------------------------------------------------------------------------------------------------

match_started = time.perf_counter()

match_rows = []


for row in build_tokens.itertuples(
    index=False
):
    token = str(
        row.CommitToken
    ).lower()

    matched_commit = None


    if token in history_commit_set:
        match_type = "EXACT"
        matched_commit = token
        candidate_count = 1

    else:
        candidates = [
            commit
            for commit in history_commits
            if (
                commit.startswith(
                    token
                )
                or token.startswith(
                    commit
                )
            )
        ]

        if len(
            candidates
        ) == 1:
            match_type = (
                "UNIQUE_PREFIX"
            )

            matched_commit = candidates[
                0
            ]

            candidate_count = 1

        elif len(
            candidates
        ) == 0:
            match_type = (
                "UNMATCHED"
            )

            candidate_count = 0

        else:
            match_type = (
                "AMBIGUOUS_PREFIX"
            )

            candidate_count = len(
                candidates
            )


    match_rows.append({
        "BuildID":
            int(
                row.BuildID
            ),

        "ChronologyOrder":
            int(
                row.ChronologyOrder
            ),

        "TokenOrder":
            int(
                row.TokenOrder
            ),

        "CommitToken":
            token,

        "MatchType":
            match_type,

        "MatchedCommit":
            matched_commit,

        "CandidateMatches":
            candidate_count,
    })


commit_audit = (
    pd.DataFrame(
        match_rows
    )
    .sort_values(
        [
            "ChronologyOrder",
            "TokenOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


commit_matching_seconds = float(
    time.perf_counter()
    - match_started
)


exact_matches = int(
    commit_audit[
        "MatchType"
    ].eq(
        "EXACT"
    ).sum()
)


prefix_matches = int(
    commit_audit[
        "MatchType"
    ].eq(
        "UNIQUE_PREFIX"
    ).sum()
)


unmatched_tokens = int(
    commit_audit[
        "MatchType"
    ].eq(
        "UNMATCHED"
    ).sum()
)


ambiguous_tokens = int(
    commit_audit[
        "MatchType"
    ].eq(
        "AMBIGUOUS_PREFIX"
    ).sum()
)


matched_token_rows = int(
    exact_matches
    + prefix_matches
)


commit_coverage_percent = (
    100.0
    * matched_token_rows
    / len(
        commit_audit
    )
)


matched_build_commits = (
    commit_audit.loc[
        commit_audit[
            "MatchedCommit"
        ].notna(),
        [
            "BuildID",
            "ChronologyOrder",
            "MatchedCommit",
        ],
    ]
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)


entity_for_join = (
    entity_history.rename(
        columns={
            entity_id_column:
                "EntityId",

            "NormalisedCommit":
                "MatchedCommit",
        }
    )
)


build_entity = (
    matched_build_commits.merge(
        entity_for_join,
        on="MatchedCommit",
        how="left",
        validate="many_to_many",
    )
    .dropna(
        subset=[
            "EntityId",
        ]
    )
)


build_entity[
    "EntityId"
] = build_entity[
    "EntityId"
].astype(
    "int64"
)


build_entity = (
    build_entity[
        [
            "BuildID",
            "ChronologyOrder",
            "MatchedCommit",
            "EntityId",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "ChronologyOrder",
            "EntityId",
            "MatchedCommit",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


builds_with_entities = set(
    build_entity[
        "BuildID"
    ].astype(
        int
    )
)


builds_without_entities = sorted(
    all_builds
    - builds_with_entities
)


unmatched_commit_builds = sorted(
    commit_audit.loc[
        commit_audit[
            "MatchType"
        ].eq(
            "UNMATCHED"
        ),
        "BuildID",
    ].astype(
        int
    ).unique().tolist()
)


mapping_incomplete_builds = sorted(
    set(
        builds_without_entities
    )
    | set(
        unmatched_commit_builds
    )
)


mapped_entity_ids = set(
    build_entity[
        "EntityId"
    ].astype(
        int
    )
)


mapped_entity_ids_missing_from_id_map = sorted(
    mapped_entity_ids
    - id_map_entity_ids
)


alias_summary = (
    resolved_id_map.groupby(
        "EntityId",
        as_index=False,
    )
    .agg(
        EntityPathAliasCount=(
            "EntityPath",
            "nunique",
        ),

        CanonicalEntityPath=(
            "EntityPath",
            "min",
        ),
    )
)


entity_id_audit = (
    pd.DataFrame({
        "EntityId":
            sorted(
                mapped_entity_ids
            )
    })
    .merge(
        alias_summary,
        on="EntityId",
        how="left",
        validate="one_to_one",
    )
)


entity_id_audit[
    "PresentInIDMap"
] = entity_id_audit[
    "EntityPathAliasCount"
].notna()


entity_id_audit[
    "EntityPathAliasCount"
] = entity_id_audit[
    "EntityPathAliasCount"
].fillna(
    0
).astype(
    int
)


mapping_incomplete_frame = (
    chronology.loc[
        chronology[
            "BuildID"
        ].isin(
            mapping_incomplete_builds
        ),
        [
            "BuildID",
            "ChronologyOrder",
            "Partition",
        ],
    ]
    .copy()
)


unmatched_counts = (
    commit_audit.loc[
        commit_audit[
            "MatchType"
        ].eq(
            "UNMATCHED"
        )
    ]
    .groupby(
        "BuildID"
    )
    .size()
    .rename(
        "UnmatchedCommitTokens"
    )
)


mapped_entity_counts = (
    build_entity.groupby(
        "BuildID"
    )[
        "EntityId"
    ]
    .nunique()
    .rename(
        "MappedEntityCount"
    )
)


mapping_incomplete_frame = (
    mapping_incomplete_frame.merge(
        unmatched_counts,
        on="BuildID",
        how="left",
    )
    .merge(
        mapped_entity_counts,
        on="BuildID",
        how="left",
    )
)


mapping_incomplete_frame[
    "UnmatchedCommitTokens"
] = mapping_incomplete_frame[
    "UnmatchedCommitTokens"
].fillna(
    0
).astype(
    int
)


mapping_incomplete_frame[
    "MappedEntityCount"
] = mapping_incomplete_frame[
    "MappedEntityCount"
].fillna(
    0
).astype(
    int
)


mapping_incomplete_frame[
    "HasMappedEntities"
] = mapping_incomplete_frame[
    "MappedEntityCount"
].gt(
    0
)


# --------------------------------------------------------------------------------------------------
# 13. VALIDATION
# --------------------------------------------------------------------------------------------------

checks = []


add_check(
    checks,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_CHECKPOINT_SHA256,
    selection_checkpoint_sha256,
    selection_checkpoint_sha256
    == EXPECTED_SELECTION_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root,
    current_source_root
    == EXPECTED_SOURCE_ROOT_SHA256,
)

add_check(
    checks,
    "Source files",
    EXPECTED_SOURCE_FILES,
    len(
        current_manifest
    ),
    len(
        current_manifest
    ) == EXPECTED_SOURCE_FILES,
)

add_check(
    checks,
    "Source bytes",
    EXPECTED_SOURCE_BYTES,
    current_source_bytes,
    current_source_bytes
    == EXPECTED_SOURCE_BYTES,
)

add_check(
    checks,
    "Builds",
    EXPECTED_BUILDS,
    len(
        chronology
    ),
    len(
        chronology
    ) == EXPECTED_BUILDS,
)

add_check(
    checks,
    "Training builds",
    EXPECTED_TRAIN_BUILDS,
    len(
        training_builds
    ),
    len(
        training_builds
    ) == EXPECTED_TRAIN_BUILDS,
)

add_check(
    checks,
    "Evaluation builds",
    EXPECTED_EVAL_BUILDS,
    len(
        evaluation_builds
    ),
    len(
        evaluation_builds
    ) == EXPECTED_EVAL_BUILDS,
)

add_check(
    checks,
    "Timestamp tie groups",
    EXPECTED_TIMESTAMP_TIE_GROUPS,
    timestamp_tie_group_count,
    timestamp_tie_group_count
    == EXPECTED_TIMESTAMP_TIE_GROUPS,
)

add_check(
    checks,
    "Raw rows",
    EXPECTED_RAW_ROWS,
    len(
        exe
    ),
    len(
        exe
    ) == EXPECTED_RAW_ROWS,
)

add_check(
    checks,
    "Model rows",
    EXPECTED_MODEL_ROWS,
    len(
        dataset
    ),
    len(
        dataset
    ) == EXPECTED_MODEL_ROWS,
)

add_check(
    checks,
    "Dataset key columns",
    3,
    3,
    (
        dataset_build_column
        in dataset.columns
        and dataset_test_column
        in dataset.columns
        and dataset_verdict_column
        in dataset.columns
    ),
)

add_check(
    checks,
    "Predictor count consistency",
    len(
        dataset.columns
    )
    - 3,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    )
    == (
        len(
            dataset.columns
        )
        - 3
    ),
)

add_check(
    checks,
    "REC features",
    19,
    len(
        REC_FEATURES
    ),
    (
        len(
            REC_FEATURES
        ) == 19
        and not missing_rec_features
    ),
)

for row in join_audit.itertuples(
    index=False
):
    add_check(
        checks,
        str(
            row.Metric
        ),
        row.Expected,
        row.Actual,
        bool(
            row.Pass
        ),
    )

add_check(
    checks,
    "Invalid resolved id_map IDs",
    0,
    invalid_resolved_rows,
    invalid_resolved_rows
    == 0,
)

add_check(
    checks,
    "Empty id_map paths",
    0,
    empty_path_rows,
    empty_path_rows
    == 0,
)

add_check(
    checks,
    "Paths with multiple EntityIds",
    0,
    paths_with_multiple_ids,
    paths_with_multiple_ids
    == 0,
)

add_check(
    checks,
    "Mapped entity IDs missing from id_map",
    0,
    len(
        mapped_entity_ids_missing_from_id_map
    ),
    len(
        mapped_entity_ids_missing_from_id_map
    ) == 0,
)

add_check(
    checks,
    "Ambiguous commit tokens",
    0,
    ambiguous_tokens,
    ambiguous_tokens
    == 0,
)

add_check(
    checks,
    "Matched commit tokens",
    "> 0",
    matched_token_rows,
    matched_token_rows
    > 0,
)

add_check(
    checks,
    "Build-entity rows",
    "> 0",
    len(
        build_entity
    ),
    len(
        build_entity
    )
    > 0,
)

add_check(
    checks,
    "Registry rows",
    21,
    len(
        registry
    ),
    len(
        registry
    ) == 21,
)


for predecessor_number, predecessor_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            project_numbers.eq(
                predecessor_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        checks,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        actual_project,
        actual_project == predecessor_project,
    )


add_check(
    checks,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    active_reservations,
    active_reservations == EXPECTED_ACTIVE_RESERVATIONS,
)


add_check(
    checks,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ),
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ) == EXPECTED_RUNTIME_PRIORITY_RULE,
)


add_check(
    checks,
    "Project 22 registry rows",
    0,
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)


validation = pd.DataFrame(
    checks
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 22 Step 2A validation:"
)

display(
    validation
)


if not failed_validation.empty:
    print(
        "\nFailed Project 22 Step 2A checks:"
    )

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 22 STEP 2A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 14. WRITE AUDITABLE OUTPUTS
# --------------------------------------------------------------------------------------------------

PREFLIGHT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_csv(
    SOURCE_SCHEMA_PATH,
    source_schema,
)

atomic_csv(
    JOIN_AUDIT_PATH,
    join_audit,
)

atomic_csv(
    REC_CLASS_PATH,
    rec_classification,
)

atomic_csv(
    BUILD_TOKEN_PATH,
    build_tokens,
)

atomic_csv(
    COMMIT_AUDIT_PATH,
    commit_audit,
)

atomic_csv(
    BUILD_ENTITY_PATH,
    build_entity,
    compression="gzip",
)

atomic_csv(
    ID_ORIENTATION_PATH,
    id_orientation,
)

atomic_csv(
    RESOLVED_ID_MAP_PATH,
    resolved_id_map,
    compression="gzip",
)

atomic_csv(
    ENTITY_ID_AUDIT_PATH,
    entity_id_audit,
)

atomic_csv(
    MAPPING_INCOMPLETE_BUILDS_PATH,
    mapping_incomplete_frame,
)

atomic_csv(
    VALIDATION_PATH,
    validation,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


summary_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "TimestampTieGroups":
        timestamp_tie_group_count,

    "ChronologyRule":
        selection_checkpoint.get(
            "ChronologyRule"
        ),

    "ResolvedIDMapEntityIDColumn":
        resolved_id_column,

    "ResolvedIDMapPathColumn":
        resolved_path_column,

    "ResolvedIDMapRows":
        len(
            resolved_id_map
        ),

    "ExactDuplicateIDMapRowsRemoved":
        exact_duplicate_rows,

    "DuplicateEntityIDRowsAcceptedAsAliases":
        duplicate_entity_id_rows,

    "EntityIDsWithMultiplePaths":
        entity_ids_with_multiple_paths,

    "PathsWithMultipleEntityIDs":
        paths_with_multiple_ids,

    "MaximumPathsPerEntityID":
        maximum_paths_per_entity,

    "BuildCommitTokenRows":
        len(
            build_tokens
        ),

    "ExactCommitMatches":
        exact_matches,

    "UniquePrefixMatches":
        prefix_matches,

    "UnmatchedCommitTokens":
        unmatched_tokens,

    "AmbiguousCommitTokens":
        ambiguous_tokens,

    "CommitTokenCoveragePercent":
        commit_coverage_percent,

    "BuildsWithoutCommitTokens":
        builds_without_tokens,

    "BuildsWithMappedEntities":
        len(
            builds_with_entities
        ),

    "BuildsWithoutMappedEntities":
        len(
            builds_without_entities
        ),

    "MappingIncompleteBuilds":
        len(
            mapping_incomplete_builds
        ),

    "BuildEntityRows":
        len(
            build_entity
        ),

    "UniqueMappedEntities":
        int(
            build_entity[
                "EntityId"
            ].nunique()
        ),

    "MappedEntityIDsMissingFromIDMap":
        len(
            mapped_entity_ids_missing_from_id_map
        ),

    "CommitMatchingSeconds":
        commit_matching_seconds,

    "RequiresStep2BMappingAudit":
        bool(
            unmatched_tokens
            or builds_without_entities
        ),
}


atomic_json(
    SUMMARY_PATH,
    summary_payload,
)


report_payload = {
    **summary_payload,

    "SourceRootSHA256":
        current_source_root,

    "SelectionCheckpointSHA256":
        selection_checkpoint_sha256,

    "RawExecutionRows":
        len(
            exe
        ),

    "ModelReadyRows":
        len(
            dataset
        ),

    "DatasetColumns":
        len(
            dataset.columns
        ),

    "PredictorColumns":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To21Modified":
        False,

    "ActiveReservations":
        active_reservations,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "NoiseInjected":
        False,

    "ModelsTrained":
        False,
}


atomic_json(
    REPORT_PATH,
    report_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root,

    "ResolvedIDMapEntityIDColumn":
        resolved_id_column,

    "ResolvedIDMapPathColumn":
        resolved_path_column,

    "BuildEntityRows":
        len(
            build_entity
        ),

    "BuildsWithMappedEntities":
        len(
            builds_with_entities
        ),

    "BuildsWithoutMappedEntities":
        len(
            builds_without_entities
        ),

    "UnmatchedCommitTokens":
        unmatched_tokens,

    "AmbiguousCommitTokens":
        ambiguous_tokens,

    "MappingIncompleteBuilds":
        len(
            mapping_incomplete_builds
        ),

    "RequiresStep2BMappingAudit":
        bool(
            unmatched_tokens
            or builds_without_entities
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,
}


atomic_json(
    STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 15. READBACK AND IMMUTABILITY
# --------------------------------------------------------------------------------------------------

if len(
    pd.read_csv(
        BUILD_ENTITY_PATH,
        compression="gzip",
        low_memory=False,
    )
) != len(
    build_entity
):
    raise RuntimeError(
        "Build-entity map readback failed."
    )


if len(
    pd.read_csv(
        RESOLVED_ID_MAP_PATH,
        compression="gzip",
        low_memory=False,
    )
) != len(
    resolved_id_map
):
    raise RuntimeError(
        "Resolved id_map readback failed."
    )


if sha256_file(
    REGISTRY_PATH
) != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Project 22 Step 2A."
    )


final_manifest_records = []


for row in current_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_manifest = pd.DataFrame(
    final_manifest_records
)


if source_root_hash(
    final_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 22 source changed during Step 2A."
    )


# --------------------------------------------------------------------------------------------------
# 16. DISPLAY
# --------------------------------------------------------------------------------------------------

print(
    "\nBuild-Test join audit:"
)

display(
    join_audit
)


print(
    "\nid_map orientation audit:"
)

display(
    id_orientation
)


print(
    "\nid_map alias summary:"
)

display(
    pd.DataFrame([
        {
            "Metric":
                "Resolved EntityId column",

            "Value":
                resolved_id_column,
        },

        {
            "Metric":
                "Resolved path column",

            "Value":
                resolved_path_column,
        },

        {
            "Metric":
                "Resolved unique path-ID rows",

            "Value":
                len(
                    resolved_id_map
                ),
        },

        {
            "Metric":
                "Duplicate EntityId rows accepted as aliases",

            "Value":
                duplicate_entity_id_rows,
        },

        {
            "Metric":
                "EntityIds with multiple paths",

            "Value":
                entity_ids_with_multiple_paths,
        },

        {
            "Metric":
                "Paths with multiple EntityIds",

            "Value":
                paths_with_multiple_ids,
        },

        {
            "Metric":
                "Mapped entity IDs missing from id_map",

            "Value":
                len(
                    mapped_entity_ids_missing_from_id_map
                ),
        },
    ])
)


print(
    "\nCommit matching summary:"
)

display(
    commit_audit[
        "MatchType"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "MatchType"
    )
    .reset_index(
        name="Rows"
    )
)


print(
    "\nMapping-incomplete builds:"
)

if mapping_incomplete_frame.empty:
    print(
        "None"
    )

else:
    display(
        mapping_incomplete_frame
    )


print(
    "\nBuild-entity sample:"
)

display(
    pd.concat(
        [
            build_entity.head(
                10
            ),
            build_entity.tail(
                10
            ),
        ],
        ignore_index=True,
    )
)


# --------------------------------------------------------------------------------------------------
# 17. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 132
)

print(
    "=== PROJECT 22 CELL 4 / STEP 2A RESULT ==="
)

print(
    "=" * 132
)


print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

for predecessor_number in sorted(
    required_registered_identities
):
    print(
        f"Project {predecessor_number} identity:",
        required_registered_identities[
            predecessor_number
        ],
    )


print(
    "Active reservations:",
    active_reservations,
)


print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)

print(
    "Builds:",
    len(
        chronology
    ),
)

print(
    "Training / evaluation builds:",
    len(
        training_builds
    ),
    "/",
    len(
        evaluation_builds
    ),
)

print(
    "Timestamp tie groups:",
    timestamp_tie_group_count,
)

print(
    "Raw execution rows:",
    len(
        exe
    ),
)

print(
    "Model-ready rows:",
    len(
        dataset
    ),
)

print(
    "Dataset columns:",
    len(
        dataset.columns
    ),
)

print(
    "Predictor columns:",
    len(
        predictor_columns
    ),
)

print(
    "REC features:",
    len(
        REC_FEATURES
    ),
)


print(
    "\nBuild-Test joins:"
)

print(
    "Raw duplicate Build-Test rows:",
    raw_duplicate_pairs,
)

print(
    "Model duplicate Build-Test rows:",
    model_duplicate_pairs,
)

print(
    "Missing model-to-raw links:",
    missing_model_raw_links,
)

print(
    "Model/raw verdict mismatches:",
    verdict_mismatches,
)

print(
    "Non-finite duration rows:",
    nonfinite_duration_rows,
)

print(
    "Negative duration rows:",
    negative_duration_rows,
)


print(
    "\nid_map.csv resolution:"
)

print(
    "Resolved EntityId column:",
    resolved_id_column,
)

print(
    "Resolved path column:",
    resolved_path_column,
)

print(
    "Duplicate EntityId rows accepted as aliases:",
    duplicate_entity_id_rows,
)

print(
    "EntityIds with multiple paths:",
    entity_ids_with_multiple_paths,
)

print(
    "Paths with multiple EntityIds:",
    paths_with_multiple_ids,
)


print(
    "\nCommit and entity mapping:"
)

print(
    "Build commit-token rows:",
    len(
        build_tokens
    ),
)

print(
    "Exact commit matches:",
    exact_matches,
)

print(
    "Unique-prefix matches:",
    prefix_matches,
)

print(
    "Unmatched commit tokens:",
    unmatched_tokens,
)

print(
    "Ambiguous commit tokens:",
    ambiguous_tokens,
)

print(
    "Commit-token coverage percent:",
    commit_coverage_percent,
)

print(
    "Builds with mapped entities:",
    len(
        builds_with_entities
    ),
)

print(
    "Builds without mapped entities:",
    len(
        builds_without_entities
    ),
)

print(
    "Mapping-incomplete builds:",
    len(
        mapping_incomplete_builds
    ),
)

print(
    "Build-entity rows:",
    len(
        build_entity
    ),
)

print(
    "Mapped entity IDs missing from id_map:",
    len(
        mapped_entity_ids_missing_from_id_map
    ),
)

print(
    "Step 2B mapping audit required:",
    bool(
        unmatched_tokens
        or builds_without_entities
    ),
)


print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    sha256_file(
        REGISTRY_PATH
    )
    == registry_sha256_before,
)

print(
    "Projects 1–21 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Noise injected:",
    False,
)

print(
    "Models trained:",
    False,
)


print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print(
    "\nSTATUS:",
    STEP2A_STATUS,
)

print(
    "=" * 132
)


=== PROJECT 22 CELL 4 / STEP 2A: SOURCE SCHEMA AND JOIN-STRUCTURE VALIDATION ===


RuntimeError: Registry must contain exactly Projects 1–21.

In [5]:
# ==================================================================================================
# PROJECT 22 — CELL 4 / STEP 2A
# SOURCE SCHEMA, BUILD-TEST JOIN, ID-MAP ORIENTATION,
# COMMIT MATCHING, AND BUILD-ENTITY PREFLIGHT
#
# PROJECT:
#   apache@logging-log4j2
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME CURRENT THESIS NOTEBOOK.
#
# PURPOSE:
# - validate the frozen Project 22 identity, source root, chronology, and registry state;
# - validate all Build-Test joins and clean-verdict alignment;
# - validate all 19 REC columns;
# - resolve id_map.csv orientation without assuming EntityId uniqueness;
# - preserve duplicate EntityId rows as valid path aliases;
# - map build commits to entity-change history;
# - write the build-entity mapping required by clean REC reconstruction;
# - record unmatched commits/builds for explicit Step 2B audit.
#
# SAFETY:
# - no noise injection;
# - no model fitting;
# - no completion-registry write;
# - no prior-project condition-output access;
# - no Project 22 experiment execution.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd


print("=" * 132)
print("=== PROJECT 22 CELL 4 / STEP 2A: SOURCE SCHEMA AND JOIN-STRUCTURE VALIDATION ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 22
PROJECT_NAME = "apache@logging-log4j2"
PROJECT_SLUG = "apache__logging-log4j2"
PROJECT_SHORT = "LOG4J2"

SOURCE_DIR = Path(
    "/content/datasets/datasets/apache@logging-log4j2"
)

EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_22_SELECTION_AND_SOURCE_FROZEN"
)

STEP2A_STATUS = (
    "PASS_PROJECT_22_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_VALIDATED"
)

EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "a7387d9495c71dce5d2c21ff08b0afd80d9c5251b5885a82e4052c7506ee7890"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64f842964c896c6ac334"
)

EXPECTED_REGISTRY_SHA256 = (
    "79cd6ecb595c5e8ae91a9494e469792716338d144308560a62caf1b9342306b2"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 21

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 101_286_750

EXPECTED_BUILDS = 441
EXPECTED_TRAIN_BUILDS = 330
EXPECTED_EVAL_BUILDS = 111

EXPECTED_TIMESTAMP_TIE_GROUPS = 1

EXPECTED_RAW_ROWS = 240_253
EXPECTED_RAW_TRAIN_ROWS = 172_628
EXPECTED_RAW_EVAL_ROWS = 67_625
EXPECTED_RAW_TRAIN_FAILURES = 208
EXPECTED_RAW_EVAL_FAILURES = 40

EXPECTED_MODEL_ROWS = 117_968
EXPECTED_MODEL_TRAIN_ROWS = 95_812
EXPECTED_MODEL_EVAL_ROWS = 22_156
EXPECTED_MODEL_TRAIN_FAILURES = 207
EXPECTED_MODEL_EVAL_FAILURES = 40

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = {
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
}

FILE_HISTORY_REC = {
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
}

REQUIRED_SOURCE_FILES = [
    "builds.csv",
    "contributors.csv",
    "dataset.csv",
    "entity_change_history.csv",
    "exe.csv",
    "id_map.csv",
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_22_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_22_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_22_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_22_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_22_fixed_chronological_builds.csv"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

SOURCE_SCHEMA_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_source_schema_profile.csv"
)

JOIN_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_test_join_audit.csv"
)

REC_CLASS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_rec_feature_classification.csv"
)

BUILD_TOKEN_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_commit_token_profile.csv"
)

COMMIT_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_commit_matching_audit.csv"
)

BUILD_ENTITY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
)

ID_ORIENTATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_id_map_orientation_audit.csv"
)

RESOLVED_ID_MAP_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_resolved_id_map_aliases.csv.gz"
)

ENTITY_ID_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_entity_id_map_audit.csv"
)

MAPPING_INCOMPLETE_BUILDS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_mapping_incomplete_builds.csv"
)

VALIDATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2a_validation.csv"
)

SUMMARY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_entity_mapping_summary.json"
)

REPORT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2a_report.json"
)

STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2a_status.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
    compression=None,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
        compression=compression,
    )

    os.replace(
        temporary_path,
        path,
    )


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}.\n"
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def parse_int(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} has "
            f"{int(numeric.isna().sum())} "
            "missing/non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(
            array
        ),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype(
        "int64"
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def normalise_commit(
    value,
):
    if pd.isna(
        value
    ):
        return ""

    text = str(
        value
    ).strip().lower()

    if not text:
        return ""

    matches = re.findall(
        r"[0-9a-f]{7,64}",
        text,
        flags=re.I,
    )

    if matches:
        return matches[0].lower()

    return re.sub(
        r"[^a-z0-9]",
        "",
        text,
    )


def extract_commit_tokens(
    value,
):
    if pd.isna(
        value
    ):
        return []

    text = str(
        value
    ).strip()

    if not text:
        return []

    tokens = re.findall(
        r"[0-9a-fA-F]{7,64}",
        text,
    )

    if not tokens:
        tokens = re.split(
            r"[\s,;|#]+",
            text,
        )

    result = []
    seen = set()

    for token in tokens:
        token = normalise_commit(
            token
        )

        if (
            token
            and token not in seen
        ):
            seen.add(
                token
            )

            result.append(
                token
            )

    return result


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS
# --------------------------------------------------------------------------------------------------

required_inputs = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(
        path
    ).is_file()
]

missing_inputs.extend(
    str(
        SOURCE_DIR
        / filename
    )
    for filename in REQUIRED_SOURCE_FILES
    if not (
        SOURCE_DIR
        / filename
    ).is_file()
)


if missing_inputs:
    raise FileNotFoundError(
        "Required Project 22 Step 2A inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. VALIDATE FROZEN SELECTION, REGISTRY, AND SOURCE ROOT
# --------------------------------------------------------------------------------------------------

selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = load_json(
    STEP1B_STATUS_PATH
)


if (
    selection_checkpoint_sha256
    != EXPECTED_SELECTION_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 22 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_CHECKPOINT_SHA256}\n"
        f"Actual:   {selection_checkpoint_sha256}"
    )


if (
    selection_checkpoint.get(
        "Status"
    ) != EXPECTED_STEP1B_STATUS
    or step1b_status.get(
        "Status"
    ) != EXPECTED_STEP1B_STATUS
):
    raise RuntimeError(
        "Project 22 Step 1B is not frozen successfully."
    )


if (
    selection_checkpoint.get(
        "Project"
    ) != PROJECT_NAME
    or selection_checkpoint.get(
        "ProjectSlug"
    ) != PROJECT_SLUG
):
    raise RuntimeError(
        "Frozen Project 22 identity differs."
    )


if selection_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Frozen Project 22 runtime-priority rule differs."
    )


active_reservations = selection_checkpoint.get(
    "ActiveReservations",
    [],
)


if active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Frozen Project 22 active-reservation state differs.\n"
        f"Expected: {EXPECTED_ACTIVE_RESERVATIONS}\n"
        f"Actual:   {active_reservations}"
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(int)


if (
    len(
        registry
    ) != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        project_numbers.tolist()
    ) != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–21."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–21 are not all COMPLETE_AND_FROZEN."
    )


required_registered_identities = {
    11:
        "apache@shardingsphere",

    12:
        "zolyfarkas@spf4j",

    13:
        "jcabi@jcabi-github",

    14:
        "JMRI@JMRI",

    15:
        "eclipse@steady",

    16:
        "apache@rocketmq",

    17:
        "yamcs@Yamcs",

    18:
        "cantaloupe-project@cantaloupe",

    19:
        "EMResearch@EvoMaster",

    20:
        "apache@curator",

    21:
        "facebook@buck",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


if (
    project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 22 is unexpectedly already registered."
    )


frozen_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_manifest_records = []


for row in frozen_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            "A frozen Project 22 source file is missing:\n"
            f"{source_path}"
        )

    current_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


current_manifest = pd.DataFrame(
    current_manifest_records
)

current_source_root = source_root_hash(
    current_manifest
)

current_source_bytes = int(
    current_manifest[
        "SizeBytes"
    ].sum()
)


if (
    current_source_root
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "Frozen Project 22 source root differs.\n"
        f"Expected: {EXPECTED_SOURCE_ROOT_SHA256}\n"
        f"Actual:   {current_source_root}"
    )


# --------------------------------------------------------------------------------------------------
# 6. RESOLVE SOURCE SCHEMAS
# --------------------------------------------------------------------------------------------------

paths = {
    "builds.csv":
        SOURCE_DIR
        / "builds.csv",

    "contributors.csv":
        SOURCE_DIR
        / "contributors.csv",

    "dataset.csv":
        SOURCE_DIR
        / "dataset.csv",

    "entity_change_history.csv":
        SOURCE_DIR
        / "entity_change_history.csv",

    "exe.csv":
        SOURCE_DIR
        / "exe.csv",

    "id_map.csv":
        SOURCE_DIR
        / "id_map.csv",
}


schema_rows = []
headers = {}


for filename, file_path in paths.items():
    columns = pd.read_csv(
        file_path,
        nrows=0,
    ).columns.tolist()

    headers[
        filename
    ] = columns

    schema_rows.append({
        "File":
            filename,

        "Path":
            str(
                file_path
            ),

        "SizeBytes":
            int(
                file_path.stat().st_size
            ),

        "ColumnCount":
            len(
                columns
            ),

        "ColumnsJSON":
            json.dumps(
                columns,
                ensure_ascii=False,
            ),
    })


source_schema = pd.DataFrame(
    schema_rows
)


build_id_column = resolve_column(
    headers[
        "builds.csv"
    ],
    "id",
    "builds.csv id",
)

build_commit_column = resolve_column(
    headers[
        "builds.csv"
    ],
    "commits",
    "builds.csv commits",
)

build_time_column = resolve_column(
    headers[
        "builds.csv"
    ],
    "started_at",
    "builds.csv started_at",
)


exe_test_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "test",
    "exe.csv test",
)

exe_build_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "build",
    "exe.csv build",
)

exe_job_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "job",
    "exe.csv job",
)

exe_verdict_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "verdict",
    "exe.csv verdict",
)

exe_duration_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "duration",
    "exe.csv duration",
)


dataset_build_column = resolve_column(
    headers[
        "dataset.csv"
    ],
    "Build",
    "dataset.csv Build",
)

dataset_test_column = resolve_column(
    headers[
        "dataset.csv"
    ],
    "Test",
    "dataset.csv Test",
)

dataset_verdict_column = resolve_column(
    headers[
        "dataset.csv"
    ],
    "Verdict",
    "dataset.csv Verdict",
)


entity_id_column = resolve_column(
    headers[
        "entity_change_history.csv"
    ],
    "EntityId",
    "entity_change_history.csv EntityId",
)

entity_commit_column = resolve_column(
    headers[
        "entity_change_history.csv"
    ],
    "Commit",
    "entity_change_history.csv Commit",
)


id_key_column = resolve_column(
    headers[
        "id_map.csv"
    ],
    "key",
    "id_map.csv key",
)

id_value_column = resolve_column(
    headers[
        "id_map.csv"
    ],
    "value",
    "id_map.csv value",
)


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature
    not in headers[
        "dataset.csv"
    ]
]


predictor_columns = [
    column
    for column in headers[
        "dataset.csv"
    ]
    if column
    not in {
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    }
]


if missing_rec_features:
    raise RuntimeError(
        "dataset.csv is missing REC columns:\n"
        + "\n".join(
            missing_rec_features
        )
    )


# --------------------------------------------------------------------------------------------------
# 7. LOAD CHRONOLOGY AND SOURCE TABLES
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)


chronology[
    "BuildID"
] = parse_int(
    chronology[
        "BuildID"
    ],
    "chronology.BuildID",
)

chronology[
    "StartedAtUTC"
] = pd.to_datetime(
    chronology[
        "StartedAtUTC"
    ],
    errors="coerce",
    utc=True,
)

if chronology[
    "StartedAtUTC"
].isna().any():
    raise RuntimeError(
        "Frozen chronology contains invalid StartedAtUTC values."
    )

timestamp_tie_group_count = int(
    chronology.groupby(
        "StartedAtUTC",
        dropna=False,
    )[
        "BuildID"
    ].size().gt(
        1
    ).sum()
)

if int(
    selection_checkpoint.get(
        "TimestampTieGroups",
        -1,
    )
) != EXPECTED_TIMESTAMP_TIE_GROUPS:
    raise RuntimeError(
        "Frozen selection checkpoint timestamp-tie count differs."
    )


training_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "TRAIN"
        ),
        "BuildID",
    ].astype(
        int
    )
)


evaluation_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildID",
    ].astype(
        int
    )
)


all_builds = (
    training_builds
    | evaluation_builds
)


build_order = (
    chronology.set_index(
        "BuildID"
    )[
        "ChronologyOrder"
    ]
    .astype(
        int
    )
    .to_dict()
)


builds = pd.read_csv(
    paths[
        "builds.csv"
    ],
    usecols=[
        build_id_column,
        build_commit_column,
        build_time_column,
    ],
    low_memory=False,
)


builds[
    build_id_column
] = parse_int(
    builds[
        build_id_column
    ],
    "builds.csv.id",
)


builds[
    build_time_column
] = pd.to_datetime(
    builds[
        build_time_column
    ],
    errors="coerce",
    utc=True,
)


if builds[
    build_time_column
].isna().any():
    raise RuntimeError(
        "builds.csv contains invalid timestamps."
    )


exe = pd.read_csv(
    paths[
        "exe.csv"
    ],
    usecols=[
        exe_test_column,
        exe_build_column,
        exe_job_column,
        exe_verdict_column,
        exe_duration_column,
    ],
    low_memory=False,
)


exe[
    exe_build_column
] = parse_int(
    exe[
        exe_build_column
    ],
    "exe.csv.build",
)


exe[
    exe_test_column
] = parse_int(
    exe[
        exe_test_column
    ],
    "exe.csv.test",
)


exe[
    exe_verdict_column
] = parse_int(
    exe[
        exe_verdict_column
    ],
    "exe.csv.verdict",
)


exe[
    exe_duration_column
] = pd.to_numeric(
    exe[
        exe_duration_column
    ],
    errors="coerce",
)


dataset = pd.read_csv(
    paths[
        "dataset.csv"
    ],
    low_memory=False,
)


dataset[
    dataset_build_column
] = parse_int(
    dataset[
        dataset_build_column
    ],
    "dataset.csv.Build",
)


dataset[
    dataset_test_column
] = parse_int(
    dataset[
        dataset_test_column
    ],
    "dataset.csv.Test",
)


dataset[
    dataset_verdict_column
] = parse_int(
    dataset[
        dataset_verdict_column
    ],
    "dataset.csv.Verdict",
)


# --------------------------------------------------------------------------------------------------
# 8. BUILD-TEST JOIN VALIDATION
# --------------------------------------------------------------------------------------------------

raw_duplicate_pairs = int(
    exe.duplicated(
        [
            exe_build_column,
            exe_test_column,
        ],
        keep=False,
    ).sum()
)


model_duplicate_pairs = int(
    dataset.duplicated(
        [
            dataset_build_column,
            dataset_test_column,
        ],
        keep=False,
    ).sum()
)


raw_unlinked_build_rows = int(
    (
        ~exe[
            exe_build_column
        ].isin(
            all_builds
        )
    ).sum()
)


model_unlinked_build_rows = int(
    (
        ~dataset[
            dataset_build_column
        ].isin(
            all_builds
        )
    ).sum()
)


nonfinite_duration_rows = int(
    (
        ~np.isfinite(
            exe[
                exe_duration_column
            ].to_numpy(
                dtype=float
            )
        )
    ).sum()
)


negative_duration_rows = int(
    exe[
        exe_duration_column
    ].lt(
        0
    ).sum()
)


if (
    raw_duplicate_pairs
    or model_duplicate_pairs
):
    raise RuntimeError(
        "Duplicate Build-Test pairs were found.\n"
        f"Raw duplicate rows: {raw_duplicate_pairs}\n"
        f"Model duplicate rows: {model_duplicate_pairs}"
    )


raw_pairs = exe[
    [
        exe_build_column,
        exe_test_column,
        exe_verdict_column,
        exe_duration_column,
    ]
].rename(
    columns={
        exe_build_column:
            "Build",

        exe_test_column:
            "Test",

        exe_verdict_column:
            "RawVerdict",

        exe_duration_column:
            "RawDuration",
    }
)


model_pairs = dataset[
    [
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    ]
].rename(
    columns={
        dataset_build_column:
            "Build",

        dataset_test_column:
            "Test",

        dataset_verdict_column:
            "ModelVerdict",
    }
)


joined = model_pairs.merge(
    raw_pairs,
    on=[
        "Build",
        "Test",
    ],
    how="left",
    validate="one_to_one",
    indicator=True,
)


missing_model_raw_links = int(
    joined[
        "_merge"
    ].ne(
        "both"
    ).sum()
)


verdict_mismatches = int(
    joined[
        "ModelVerdict"
    ].ne(
        joined[
            "RawVerdict"
        ]
    ).sum()
)


raw_training_mask = exe[
    exe_build_column
].isin(
    training_builds
)


raw_evaluation_mask = exe[
    exe_build_column
].isin(
    evaluation_builds
)


raw_training_rows = int(
    raw_training_mask.sum()
)

raw_evaluation_rows = int(
    raw_evaluation_mask.sum()
)

raw_training_failures = int(
    (
        raw_training_mask
        & exe[
            exe_verdict_column
        ].ne(
            0
        )
    ).sum()
)

raw_evaluation_failures = int(
    (
        raw_evaluation_mask
        & exe[
            exe_verdict_column
        ].ne(
            0
        )
    ).sum()
)


model_training_mask = joined[
    "Build"
].isin(
    training_builds
)


model_evaluation_mask = joined[
    "Build"
].isin(
    evaluation_builds
)


model_training_rows = int(
    model_training_mask.sum()
)

model_evaluation_rows = int(
    model_evaluation_mask.sum()
)

model_training_failures = int(
    (
        model_training_mask
        & joined[
            "ModelVerdict"
        ].ne(
            0
        )
    ).sum()
)

model_evaluation_failures = int(
    (
        model_evaluation_mask
        & joined[
            "ModelVerdict"
        ].ne(
            0
        )
    ).sum()
)


join_audit = pd.DataFrame([
    (
        "RawRows",
        EXPECTED_RAW_ROWS,
        len(
            exe
        ),
    ),

    (
        "RawTrainingRows",
        EXPECTED_RAW_TRAIN_ROWS,
        raw_training_rows,
    ),

    (
        "RawEvaluationRows",
        EXPECTED_RAW_EVAL_ROWS,
        raw_evaluation_rows,
    ),

    (
        "RawTrainingFailures",
        EXPECTED_RAW_TRAIN_FAILURES,
        raw_training_failures,
    ),

    (
        "RawEvaluationFailures",
        EXPECTED_RAW_EVAL_FAILURES,
        raw_evaluation_failures,
    ),

    (
        "ModelRows",
        EXPECTED_MODEL_ROWS,
        len(
            dataset
        ),
    ),

    (
        "ModelTrainingRows",
        EXPECTED_MODEL_TRAIN_ROWS,
        model_training_rows,
    ),

    (
        "ModelEvaluationRows",
        EXPECTED_MODEL_EVAL_ROWS,
        model_evaluation_rows,
    ),

    (
        "ModelTrainingFailures",
        EXPECTED_MODEL_TRAIN_FAILURES,
        model_training_failures,
    ),

    (
        "ModelEvaluationFailures",
        EXPECTED_MODEL_EVAL_FAILURES,
        model_evaluation_failures,
    ),

    (
        "RawDuplicateBuildTestRows",
        0,
        raw_duplicate_pairs,
    ),

    (
        "ModelDuplicateBuildTestRows",
        0,
        model_duplicate_pairs,
    ),

    (
        "MissingModelRawLinks",
        0,
        missing_model_raw_links,
    ),

    (
        "ModelRawVerdictMismatches",
        0,
        verdict_mismatches,
    ),

    (
        "NonFiniteDurationRows",
        0,
        nonfinite_duration_rows,
    ),

    (
        "NegativeDurationRows",
        0,
        negative_duration_rows,
    ),

    (
        "RawUnlinkedBuildRows",
        0,
        raw_unlinked_build_rows,
    ),

    (
        "ModelUnlinkedBuildRows",
        0,
        model_unlinked_build_rows,
    ),
], columns=[
    "Metric",
    "Expected",
    "Actual",
])


join_audit[
    "Pass"
] = (
    join_audit[
        "Expected"
    ].astype(
        str
    )
    == join_audit[
        "Actual"
    ].astype(
        str
    )
)


# --------------------------------------------------------------------------------------------------
# 9. REC FEATURE CLASSIFICATION
# --------------------------------------------------------------------------------------------------

rec_classification = pd.DataFrame([
    {
        "Feature":
            feature,

        "FeatureClass":
            (
                "VERDICT_DEPENDENT"
                if feature
                in VERDICT_DEPENDENT_REC
                else "VERDICT_INDEPENDENT"
            ),

        "FileHistoryFeature":
            feature
            in FILE_HISTORY_REC,

        "PresentInDataset":
            feature
            in dataset.columns,
    }
    for feature in REC_FEATURES
])


# --------------------------------------------------------------------------------------------------
# 10. BUILD COMMIT TOKENS
# --------------------------------------------------------------------------------------------------

token_rows = []
builds_without_tokens = 0


for (
    build_id,
    raw_commits,
) in builds[
    [
        build_id_column,
        build_commit_column,
    ]
].itertuples(
    index=False,
    name=None,
):
    tokens = extract_commit_tokens(
        raw_commits
    )

    if not tokens:
        builds_without_tokens += 1

    for token_order, token in enumerate(
        tokens,
        start=1,
    ):
        token_rows.append({
            "BuildID":
                int(
                    build_id
                ),

            "ChronologyOrder":
                int(
                    build_order[
                        int(
                            build_id
                        )
                    ]
                ),

            "RawCommits":
                str(
                    raw_commits
                ),

            "TokenOrder":
                token_order,

            "CommitToken":
                token,
        })


build_tokens = pd.DataFrame(
    token_rows
)


if build_tokens.empty:
    raise RuntimeError(
        "No build commit tokens could be extracted."
    )


build_tokens = (
    build_tokens.sort_values(
        [
            "ChronologyOrder",
            "TokenOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 11. ENTITY HISTORY AND ALIAS-AWARE ID MAP
# --------------------------------------------------------------------------------------------------

entity_history = pd.read_csv(
    paths[
        "entity_change_history.csv"
    ],
    usecols=[
        entity_id_column,
        entity_commit_column,
    ],
    low_memory=False,
)


entity_history[
    entity_id_column
] = parse_int(
    entity_history[
        entity_id_column
    ],
    "entity_change_history.csv.EntityId",
)


entity_history[
    "NormalisedCommit"
] = entity_history[
    entity_commit_column
].map(
    normalise_commit
)


entity_history = (
    entity_history.loc[
        entity_history[
            "NormalisedCommit"
        ].ne(
            ""
        ),
        [
            entity_id_column,
            "NormalisedCommit",
        ],
    ]
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)


history_entity_ids = set(
    entity_history[
        entity_id_column
    ].astype(
        int
    )
)


history_commits = sorted(
    entity_history[
        "NormalisedCommit"
    ].unique().tolist()
)


history_commit_set = set(
    history_commits
)


id_raw = pd.read_csv(
    paths[
        "id_map.csv"
    ],
    usecols=[
        id_key_column,
        id_value_column,
    ],
    dtype=str,
    keep_default_na=False,
    low_memory=False,
)


orientation_rows = []


for column in [
    id_key_column,
    id_value_column,
]:
    numeric = pd.to_numeric(
        id_raw[
            column
        ],
        errors="coerce",
    )

    numeric_filled = numeric.fillna(
        0
    )

    valid_integral = (
        numeric.notna()
        & np.isclose(
            numeric_filled,
            np.floor(
                numeric_filled
            ),
            rtol=0,
            atol=0,
        )
    )

    parsed_ids = set(
        numeric.loc[
            valid_integral
        ].astype(
            "int64"
        )
    )

    overlap = len(
        parsed_ids
        & history_entity_ids
    )

    orientation_rows.append({
        "Column":
            column,

        "Rows":
            len(
                id_raw
            ),

        "IntegralNumericRows":
            int(
                valid_integral.sum()
            ),

        "InvalidOrNonNumericRows":
            int(
                (
                    ~valid_integral
                ).sum()
            ),

        "UniqueIntegralIDs":
            len(
                parsed_ids
            ),

        "MatchingHistoryEntityIDs":
            overlap,

        "HistoryEntityCoveragePercent":
            (
                100.0
                * overlap
                / len(
                    history_entity_ids
                )
                if history_entity_ids
                else 0.0
            ),
    })


id_orientation = pd.DataFrame(
    orientation_rows
)


best_orientation = (
    id_orientation.sort_values(
        [
            "MatchingHistoryEntityIDs",
            "IntegralNumericRows",
        ],
        ascending=[
            False,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


if len(
    best_orientation
) < 2:
    raise RuntimeError(
        "id_map orientation audit is incomplete."
    )


if (
    int(
        best_orientation.loc[
            0,
            "MatchingHistoryEntityIDs",
        ]
    )
    == int(
        best_orientation.loc[
            1,
            "MatchingHistoryEntityIDs",
        ]
    )
    and int(
        best_orientation.loc[
            0,
            "IntegralNumericRows",
        ]
    )
    == int(
        best_orientation.loc[
            1,
            "IntegralNumericRows",
        ]
    )
):
    raise RuntimeError(
        "Could not uniquely resolve the EntityId column in id_map.csv."
    )


resolved_id_column = str(
    best_orientation.loc[
        0,
        "Column",
    ]
)


resolved_path_column = (
    id_value_column
    if resolved_id_column
    == id_key_column
    else id_key_column
)


resolved_numeric = pd.to_numeric(
    id_raw[
        resolved_id_column
    ],
    errors="coerce",
)


resolved_numeric_filled = resolved_numeric.fillna(
    0
)


valid_resolved = (
    resolved_numeric.notna()
    & np.isclose(
        resolved_numeric_filled,
        np.floor(
            resolved_numeric_filled
        ),
        rtol=0,
        atol=0,
    )
)


invalid_resolved_rows = int(
    (
        ~valid_resolved
    ).sum()
)


if invalid_resolved_rows:
    raise RuntimeError(
        "Resolved id_map EntityId column contains "
        f"{invalid_resolved_rows} invalid rows."
    )


resolved_id_map = pd.DataFrame({
    "EntityPath":
        id_raw[
            resolved_path_column
        ].astype(
            str
        ).str.strip(),

    "EntityId":
        resolved_numeric.astype(
            "int64"
        ),
})


empty_path_rows = int(
    resolved_id_map[
        "EntityPath"
    ].eq(
        ""
    ).sum()
)


exact_duplicate_rows = int(
    len(
        resolved_id_map
    )
    - len(
        resolved_id_map.drop_duplicates(
            [
                "EntityPath",
                "EntityId",
            ]
        )
    )
)


resolved_id_map = (
    resolved_id_map.drop_duplicates(
        [
            "EntityPath",
            "EntityId",
        ]
    )
    .sort_values(
        [
            "EntityId",
            "EntityPath",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


duplicate_entity_id_rows = int(
    resolved_id_map.duplicated(
        "EntityId",
        keep=False,
    ).sum()
)


entity_ids_with_multiple_paths = int(
    resolved_id_map.groupby(
        "EntityId"
    )[
        "EntityPath"
    ].nunique().gt(
        1
    ).sum()
)


paths_with_multiple_ids = int(
    resolved_id_map.groupby(
        "EntityPath"
    )[
        "EntityId"
    ].nunique().gt(
        1
    ).sum()
)


maximum_paths_per_entity = int(
    resolved_id_map.groupby(
        "EntityId"
    )[
        "EntityPath"
    ].nunique().max()
)


id_map_entity_ids = set(
    resolved_id_map[
        "EntityId"
    ].astype(
        int
    )
)


# --------------------------------------------------------------------------------------------------
# 12. COMMIT MATCHING AND BUILD-ENTITY MAP
# --------------------------------------------------------------------------------------------------

match_started = time.perf_counter()

match_rows = []


for row in build_tokens.itertuples(
    index=False
):
    token = str(
        row.CommitToken
    ).lower()

    matched_commit = None


    if token in history_commit_set:
        match_type = "EXACT"
        matched_commit = token
        candidate_count = 1

    else:
        candidates = [
            commit
            for commit in history_commits
            if (
                commit.startswith(
                    token
                )
                or token.startswith(
                    commit
                )
            )
        ]

        if len(
            candidates
        ) == 1:
            match_type = (
                "UNIQUE_PREFIX"
            )

            matched_commit = candidates[
                0
            ]

            candidate_count = 1

        elif len(
            candidates
        ) == 0:
            match_type = (
                "UNMATCHED"
            )

            candidate_count = 0

        else:
            match_type = (
                "AMBIGUOUS_PREFIX"
            )

            candidate_count = len(
                candidates
            )


    match_rows.append({
        "BuildID":
            int(
                row.BuildID
            ),

        "ChronologyOrder":
            int(
                row.ChronologyOrder
            ),

        "TokenOrder":
            int(
                row.TokenOrder
            ),

        "CommitToken":
            token,

        "MatchType":
            match_type,

        "MatchedCommit":
            matched_commit,

        "CandidateMatches":
            candidate_count,
    })


commit_audit = (
    pd.DataFrame(
        match_rows
    )
    .sort_values(
        [
            "ChronologyOrder",
            "TokenOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


commit_matching_seconds = float(
    time.perf_counter()
    - match_started
)


exact_matches = int(
    commit_audit[
        "MatchType"
    ].eq(
        "EXACT"
    ).sum()
)


prefix_matches = int(
    commit_audit[
        "MatchType"
    ].eq(
        "UNIQUE_PREFIX"
    ).sum()
)


unmatched_tokens = int(
    commit_audit[
        "MatchType"
    ].eq(
        "UNMATCHED"
    ).sum()
)


ambiguous_tokens = int(
    commit_audit[
        "MatchType"
    ].eq(
        "AMBIGUOUS_PREFIX"
    ).sum()
)


matched_token_rows = int(
    exact_matches
    + prefix_matches
)


commit_coverage_percent = (
    100.0
    * matched_token_rows
    / len(
        commit_audit
    )
)


matched_build_commits = (
    commit_audit.loc[
        commit_audit[
            "MatchedCommit"
        ].notna(),
        [
            "BuildID",
            "ChronologyOrder",
            "MatchedCommit",
        ],
    ]
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)


entity_for_join = (
    entity_history.rename(
        columns={
            entity_id_column:
                "EntityId",

            "NormalisedCommit":
                "MatchedCommit",
        }
    )
)


build_entity = (
    matched_build_commits.merge(
        entity_for_join,
        on="MatchedCommit",
        how="left",
        validate="many_to_many",
    )
    .dropna(
        subset=[
            "EntityId",
        ]
    )
)


build_entity[
    "EntityId"
] = build_entity[
    "EntityId"
].astype(
    "int64"
)


build_entity = (
    build_entity[
        [
            "BuildID",
            "ChronologyOrder",
            "MatchedCommit",
            "EntityId",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "ChronologyOrder",
            "EntityId",
            "MatchedCommit",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


builds_with_entities = set(
    build_entity[
        "BuildID"
    ].astype(
        int
    )
)


builds_without_entities = sorted(
    all_builds
    - builds_with_entities
)


unmatched_commit_builds = sorted(
    commit_audit.loc[
        commit_audit[
            "MatchType"
        ].eq(
            "UNMATCHED"
        ),
        "BuildID",
    ].astype(
        int
    ).unique().tolist()
)


mapping_incomplete_builds = sorted(
    set(
        builds_without_entities
    )
    | set(
        unmatched_commit_builds
    )
)


mapped_entity_ids = set(
    build_entity[
        "EntityId"
    ].astype(
        int
    )
)


mapped_entity_ids_missing_from_id_map = sorted(
    mapped_entity_ids
    - id_map_entity_ids
)


alias_summary = (
    resolved_id_map.groupby(
        "EntityId",
        as_index=False,
    )
    .agg(
        EntityPathAliasCount=(
            "EntityPath",
            "nunique",
        ),

        CanonicalEntityPath=(
            "EntityPath",
            "min",
        ),
    )
)


entity_id_audit = (
    pd.DataFrame({
        "EntityId":
            sorted(
                mapped_entity_ids
            )
    })
    .merge(
        alias_summary,
        on="EntityId",
        how="left",
        validate="one_to_one",
    )
)


entity_id_audit[
    "PresentInIDMap"
] = entity_id_audit[
    "EntityPathAliasCount"
].notna()


entity_id_audit[
    "EntityPathAliasCount"
] = entity_id_audit[
    "EntityPathAliasCount"
].fillna(
    0
).astype(
    int
)


mapping_incomplete_frame = (
    chronology.loc[
        chronology[
            "BuildID"
        ].isin(
            mapping_incomplete_builds
        ),
        [
            "BuildID",
            "ChronologyOrder",
            "Partition",
        ],
    ]
    .copy()
)


unmatched_counts = (
    commit_audit.loc[
        commit_audit[
            "MatchType"
        ].eq(
            "UNMATCHED"
        )
    ]
    .groupby(
        "BuildID"
    )
    .size()
    .rename(
        "UnmatchedCommitTokens"
    )
)


mapped_entity_counts = (
    build_entity.groupby(
        "BuildID"
    )[
        "EntityId"
    ]
    .nunique()
    .rename(
        "MappedEntityCount"
    )
)


mapping_incomplete_frame = (
    mapping_incomplete_frame.merge(
        unmatched_counts,
        on="BuildID",
        how="left",
    )
    .merge(
        mapped_entity_counts,
        on="BuildID",
        how="left",
    )
)


mapping_incomplete_frame[
    "UnmatchedCommitTokens"
] = mapping_incomplete_frame[
    "UnmatchedCommitTokens"
].fillna(
    0
).astype(
    int
)


mapping_incomplete_frame[
    "MappedEntityCount"
] = mapping_incomplete_frame[
    "MappedEntityCount"
].fillna(
    0
).astype(
    int
)


mapping_incomplete_frame[
    "HasMappedEntities"
] = mapping_incomplete_frame[
    "MappedEntityCount"
].gt(
    0
)


# --------------------------------------------------------------------------------------------------
# 13. VALIDATION
# --------------------------------------------------------------------------------------------------

checks = []


add_check(
    checks,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_CHECKPOINT_SHA256,
    selection_checkpoint_sha256,
    selection_checkpoint_sha256
    == EXPECTED_SELECTION_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root,
    current_source_root
    == EXPECTED_SOURCE_ROOT_SHA256,
)

add_check(
    checks,
    "Source files",
    EXPECTED_SOURCE_FILES,
    len(
        current_manifest
    ),
    len(
        current_manifest
    ) == EXPECTED_SOURCE_FILES,
)

add_check(
    checks,
    "Source bytes",
    EXPECTED_SOURCE_BYTES,
    current_source_bytes,
    current_source_bytes
    == EXPECTED_SOURCE_BYTES,
)

add_check(
    checks,
    "Builds",
    EXPECTED_BUILDS,
    len(
        chronology
    ),
    len(
        chronology
    ) == EXPECTED_BUILDS,
)

add_check(
    checks,
    "Training builds",
    EXPECTED_TRAIN_BUILDS,
    len(
        training_builds
    ),
    len(
        training_builds
    ) == EXPECTED_TRAIN_BUILDS,
)

add_check(
    checks,
    "Evaluation builds",
    EXPECTED_EVAL_BUILDS,
    len(
        evaluation_builds
    ),
    len(
        evaluation_builds
    ) == EXPECTED_EVAL_BUILDS,
)

add_check(
    checks,
    "Timestamp tie groups",
    EXPECTED_TIMESTAMP_TIE_GROUPS,
    timestamp_tie_group_count,
    timestamp_tie_group_count
    == EXPECTED_TIMESTAMP_TIE_GROUPS,
)

add_check(
    checks,
    "Raw rows",
    EXPECTED_RAW_ROWS,
    len(
        exe
    ),
    len(
        exe
    ) == EXPECTED_RAW_ROWS,
)

add_check(
    checks,
    "Model rows",
    EXPECTED_MODEL_ROWS,
    len(
        dataset
    ),
    len(
        dataset
    ) == EXPECTED_MODEL_ROWS,
)

add_check(
    checks,
    "Dataset key columns",
    3,
    3,
    (
        dataset_build_column
        in dataset.columns
        and dataset_test_column
        in dataset.columns
        and dataset_verdict_column
        in dataset.columns
    ),
)

add_check(
    checks,
    "Predictor count consistency",
    len(
        dataset.columns
    )
    - 3,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    )
    == (
        len(
            dataset.columns
        )
        - 3
    ),
)

add_check(
    checks,
    "REC features",
    19,
    len(
        REC_FEATURES
    ),
    (
        len(
            REC_FEATURES
        ) == 19
        and not missing_rec_features
    ),
)

for row in join_audit.itertuples(
    index=False
):
    add_check(
        checks,
        str(
            row.Metric
        ),
        row.Expected,
        row.Actual,
        bool(
            row.Pass
        ),
    )

add_check(
    checks,
    "Invalid resolved id_map IDs",
    0,
    invalid_resolved_rows,
    invalid_resolved_rows
    == 0,
)

add_check(
    checks,
    "Empty id_map paths",
    0,
    empty_path_rows,
    empty_path_rows
    == 0,
)

add_check(
    checks,
    "Paths with multiple EntityIds",
    0,
    paths_with_multiple_ids,
    paths_with_multiple_ids
    == 0,
)

add_check(
    checks,
    "Mapped entity IDs missing from id_map",
    0,
    len(
        mapped_entity_ids_missing_from_id_map
    ),
    len(
        mapped_entity_ids_missing_from_id_map
    ) == 0,
)

add_check(
    checks,
    "Ambiguous commit tokens",
    0,
    ambiguous_tokens,
    ambiguous_tokens
    == 0,
)

add_check(
    checks,
    "Matched commit tokens",
    "> 0",
    matched_token_rows,
    matched_token_rows
    > 0,
)

add_check(
    checks,
    "Build-entity rows",
    "> 0",
    len(
        build_entity
    ),
    len(
        build_entity
    )
    > 0,
)

add_check(
    checks,
    "Registry rows",
    21,
    len(
        registry
    ),
    len(
        registry
    ) == 21,
)


for predecessor_number, predecessor_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            project_numbers.eq(
                predecessor_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        checks,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        actual_project,
        actual_project == predecessor_project,
    )


add_check(
    checks,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    active_reservations,
    active_reservations == EXPECTED_ACTIVE_RESERVATIONS,
)


add_check(
    checks,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ),
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ) == EXPECTED_RUNTIME_PRIORITY_RULE,
)


add_check(
    checks,
    "Project 22 registry rows",
    0,
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)


validation = pd.DataFrame(
    checks
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 22 Step 2A validation:"
)

display(
    validation
)


if not failed_validation.empty:
    print(
        "\nFailed Project 22 Step 2A checks:"
    )

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 22 STEP 2A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 14. WRITE AUDITABLE OUTPUTS
# --------------------------------------------------------------------------------------------------

PREFLIGHT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_csv(
    SOURCE_SCHEMA_PATH,
    source_schema,
)

atomic_csv(
    JOIN_AUDIT_PATH,
    join_audit,
)

atomic_csv(
    REC_CLASS_PATH,
    rec_classification,
)

atomic_csv(
    BUILD_TOKEN_PATH,
    build_tokens,
)

atomic_csv(
    COMMIT_AUDIT_PATH,
    commit_audit,
)

atomic_csv(
    BUILD_ENTITY_PATH,
    build_entity,
    compression="gzip",
)

atomic_csv(
    ID_ORIENTATION_PATH,
    id_orientation,
)

atomic_csv(
    RESOLVED_ID_MAP_PATH,
    resolved_id_map,
    compression="gzip",
)

atomic_csv(
    ENTITY_ID_AUDIT_PATH,
    entity_id_audit,
)

atomic_csv(
    MAPPING_INCOMPLETE_BUILDS_PATH,
    mapping_incomplete_frame,
)

atomic_csv(
    VALIDATION_PATH,
    validation,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


summary_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "TimestampTieGroups":
        timestamp_tie_group_count,

    "ChronologyRule":
        selection_checkpoint.get(
            "ChronologyRule"
        ),

    "ResolvedIDMapEntityIDColumn":
        resolved_id_column,

    "ResolvedIDMapPathColumn":
        resolved_path_column,

    "ResolvedIDMapRows":
        len(
            resolved_id_map
        ),

    "ExactDuplicateIDMapRowsRemoved":
        exact_duplicate_rows,

    "DuplicateEntityIDRowsAcceptedAsAliases":
        duplicate_entity_id_rows,

    "EntityIDsWithMultiplePaths":
        entity_ids_with_multiple_paths,

    "PathsWithMultipleEntityIDs":
        paths_with_multiple_ids,

    "MaximumPathsPerEntityID":
        maximum_paths_per_entity,

    "BuildCommitTokenRows":
        len(
            build_tokens
        ),

    "ExactCommitMatches":
        exact_matches,

    "UniquePrefixMatches":
        prefix_matches,

    "UnmatchedCommitTokens":
        unmatched_tokens,

    "AmbiguousCommitTokens":
        ambiguous_tokens,

    "CommitTokenCoveragePercent":
        commit_coverage_percent,

    "BuildsWithoutCommitTokens":
        builds_without_tokens,

    "BuildsWithMappedEntities":
        len(
            builds_with_entities
        ),

    "BuildsWithoutMappedEntities":
        len(
            builds_without_entities
        ),

    "MappingIncompleteBuilds":
        len(
            mapping_incomplete_builds
        ),

    "BuildEntityRows":
        len(
            build_entity
        ),

    "UniqueMappedEntities":
        int(
            build_entity[
                "EntityId"
            ].nunique()
        ),

    "MappedEntityIDsMissingFromIDMap":
        len(
            mapped_entity_ids_missing_from_id_map
        ),

    "CommitMatchingSeconds":
        commit_matching_seconds,

    "RequiresStep2BMappingAudit":
        bool(
            unmatched_tokens
            or builds_without_entities
        ),
}


atomic_json(
    SUMMARY_PATH,
    summary_payload,
)


report_payload = {
    **summary_payload,

    "SourceRootSHA256":
        current_source_root,

    "SelectionCheckpointSHA256":
        selection_checkpoint_sha256,

    "RawExecutionRows":
        len(
            exe
        ),

    "ModelReadyRows":
        len(
            dataset
        ),

    "DatasetColumns":
        len(
            dataset.columns
        ),

    "PredictorColumns":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To21Modified":
        False,

    "ActiveReservations":
        active_reservations,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "NoiseInjected":
        False,

    "ModelsTrained":
        False,
}


atomic_json(
    REPORT_PATH,
    report_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root,

    "ResolvedIDMapEntityIDColumn":
        resolved_id_column,

    "ResolvedIDMapPathColumn":
        resolved_path_column,

    "BuildEntityRows":
        len(
            build_entity
        ),

    "BuildsWithMappedEntities":
        len(
            builds_with_entities
        ),

    "BuildsWithoutMappedEntities":
        len(
            builds_without_entities
        ),

    "UnmatchedCommitTokens":
        unmatched_tokens,

    "AmbiguousCommitTokens":
        ambiguous_tokens,

    "MappingIncompleteBuilds":
        len(
            mapping_incomplete_builds
        ),

    "RequiresStep2BMappingAudit":
        bool(
            unmatched_tokens
            or builds_without_entities
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,
}


atomic_json(
    STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 15. READBACK AND IMMUTABILITY
# --------------------------------------------------------------------------------------------------

if len(
    pd.read_csv(
        BUILD_ENTITY_PATH,
        compression="gzip",
        low_memory=False,
    )
) != len(
    build_entity
):
    raise RuntimeError(
        "Build-entity map readback failed."
    )


if len(
    pd.read_csv(
        RESOLVED_ID_MAP_PATH,
        compression="gzip",
        low_memory=False,
    )
) != len(
    resolved_id_map
):
    raise RuntimeError(
        "Resolved id_map readback failed."
    )


if sha256_file(
    REGISTRY_PATH
) != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Project 22 Step 2A."
    )


final_manifest_records = []


for row in current_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_manifest = pd.DataFrame(
    final_manifest_records
)


if source_root_hash(
    final_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 22 source changed during Step 2A."
    )


# --------------------------------------------------------------------------------------------------
# 16. DISPLAY
# --------------------------------------------------------------------------------------------------

print(
    "\nBuild-Test join audit:"
)

display(
    join_audit
)


print(
    "\nid_map orientation audit:"
)

display(
    id_orientation
)


print(
    "\nid_map alias summary:"
)

display(
    pd.DataFrame([
        {
            "Metric":
                "Resolved EntityId column",

            "Value":
                resolved_id_column,
        },

        {
            "Metric":
                "Resolved path column",

            "Value":
                resolved_path_column,
        },

        {
            "Metric":
                "Resolved unique path-ID rows",

            "Value":
                len(
                    resolved_id_map
                ),
        },

        {
            "Metric":
                "Duplicate EntityId rows accepted as aliases",

            "Value":
                duplicate_entity_id_rows,
        },

        {
            "Metric":
                "EntityIds with multiple paths",

            "Value":
                entity_ids_with_multiple_paths,
        },

        {
            "Metric":
                "Paths with multiple EntityIds",

            "Value":
                paths_with_multiple_ids,
        },

        {
            "Metric":
                "Mapped entity IDs missing from id_map",

            "Value":
                len(
                    mapped_entity_ids_missing_from_id_map
                ),
        },
    ])
)


print(
    "\nCommit matching summary:"
)

display(
    commit_audit[
        "MatchType"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "MatchType"
    )
    .reset_index(
        name="Rows"
    )
)


print(
    "\nMapping-incomplete builds:"
)

if mapping_incomplete_frame.empty:
    print(
        "None"
    )

else:
    display(
        mapping_incomplete_frame
    )


print(
    "\nBuild-entity sample:"
)

display(
    pd.concat(
        [
            build_entity.head(
                10
            ),
            build_entity.tail(
                10
            ),
        ],
        ignore_index=True,
    )
)


# --------------------------------------------------------------------------------------------------
# 17. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 132
)

print(
    "=== PROJECT 22 CELL 4 / STEP 2A RESULT ==="
)

print(
    "=" * 132
)


print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

for predecessor_number in sorted(
    required_registered_identities
):
    print(
        f"Project {predecessor_number} identity:",
        required_registered_identities[
            predecessor_number
        ],
    )


print(
    "Active reservations:",
    active_reservations,
)


print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)

print(
    "Builds:",
    len(
        chronology
    ),
)

print(
    "Training / evaluation builds:",
    len(
        training_builds
    ),
    "/",
    len(
        evaluation_builds
    ),
)

print(
    "Timestamp tie groups:",
    timestamp_tie_group_count,
)

print(
    "Raw execution rows:",
    len(
        exe
    ),
)

print(
    "Model-ready rows:",
    len(
        dataset
    ),
)

print(
    "Dataset columns:",
    len(
        dataset.columns
    ),
)

print(
    "Predictor columns:",
    len(
        predictor_columns
    ),
)

print(
    "REC features:",
    len(
        REC_FEATURES
    ),
)


print(
    "\nBuild-Test joins:"
)

print(
    "Raw duplicate Build-Test rows:",
    raw_duplicate_pairs,
)

print(
    "Model duplicate Build-Test rows:",
    model_duplicate_pairs,
)

print(
    "Missing model-to-raw links:",
    missing_model_raw_links,
)

print(
    "Model/raw verdict mismatches:",
    verdict_mismatches,
)

print(
    "Non-finite duration rows:",
    nonfinite_duration_rows,
)

print(
    "Negative duration rows:",
    negative_duration_rows,
)


print(
    "\nid_map.csv resolution:"
)

print(
    "Resolved EntityId column:",
    resolved_id_column,
)

print(
    "Resolved path column:",
    resolved_path_column,
)

print(
    "Duplicate EntityId rows accepted as aliases:",
    duplicate_entity_id_rows,
)

print(
    "EntityIds with multiple paths:",
    entity_ids_with_multiple_paths,
)

print(
    "Paths with multiple EntityIds:",
    paths_with_multiple_ids,
)


print(
    "\nCommit and entity mapping:"
)

print(
    "Build commit-token rows:",
    len(
        build_tokens
    ),
)

print(
    "Exact commit matches:",
    exact_matches,
)

print(
    "Unique-prefix matches:",
    prefix_matches,
)

print(
    "Unmatched commit tokens:",
    unmatched_tokens,
)

print(
    "Ambiguous commit tokens:",
    ambiguous_tokens,
)

print(
    "Commit-token coverage percent:",
    commit_coverage_percent,
)

print(
    "Builds with mapped entities:",
    len(
        builds_with_entities
    ),
)

print(
    "Builds without mapped entities:",
    len(
        builds_without_entities
    ),
)

print(
    "Mapping-incomplete builds:",
    len(
        mapping_incomplete_builds
    ),
)

print(
    "Build-entity rows:",
    len(
        build_entity
    ),
)

print(
    "Mapped entity IDs missing from id_map:",
    len(
        mapped_entity_ids_missing_from_id_map
    ),
)

print(
    "Step 2B mapping audit required:",
    bool(
        unmatched_tokens
        or builds_without_entities
    ),
)


print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    sha256_file(
        REGISTRY_PATH
    )
    == registry_sha256_before,
)

print(
    "Projects 1–21 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Noise injected:",
    False,
)

print(
    "Models trained:",
    False,
)


print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print(
    "\nSTATUS:",
    STEP2A_STATUS,
)

print(
    "=" * 132
)


=== PROJECT 22 CELL 4 / STEP 2A: SOURCE SCHEMA AND JOIN-STRUCTURE VALIDATION ===

Project 22 Step 2A validation:


,Check,Expected,Actual,Pass
0,Selection checkpoint SHA-256,a7387d9495c71dce5d2c21ff08b0afd80d9c5251b5885a...,a7387d9495c71dce5d2c21ff08b0afd80d9c5251b5885a...,True
1,Source root SHA-256,281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64...,281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64...,True
2,Source files,6,6,True
3,Source bytes,101286750,101286750,True
4,Builds,441,441,True
5,Training builds,330,330,True
6,Evaluation builds,111,111,True
7,Timestamp tie groups,1,1,True
8,Raw rows,240253,240253,True
9,Model rows,117968,117968,True



Build-Test join audit:


,Metric,Expected,Actual,Pass
0,RawRows,240253,240253,True
1,RawTrainingRows,172628,172628,True
2,RawEvaluationRows,67625,67625,True
3,RawTrainingFailures,208,208,True
4,RawEvaluationFailures,40,40,True
5,ModelRows,117968,117968,True
6,ModelTrainingRows,95812,95812,True
7,ModelEvaluationRows,22156,22156,True
8,ModelTrainingFailures,207,207,True
9,ModelEvaluationFailures,40,40,True



id_map orientation audit:


...


Commit matching summary:


,MatchType,Rows
0,EXACT,567



Mapping-incomplete builds:
None

Build-entity sample:


,BuildID,ChronologyOrder,MatchedCommit,EntityId
0,579990414,1,7cd09c460d7db235b02ba3051068a60606bbf980,2679
1,580002753,2,d4f74cf29e8a7c67805cfb67063f0de1d679db43,3591
2,580008415,3,56ef94a2b909096189daf52385390588fef28faf,3587
3,580429720,4,b4f739c45e11a88d9ed23a648db5819f1d69f8a0,3235
4,580429720,4,b4f739c45e11a88d9ed23a648db5819f1d69f8a0,3236
5,580429720,4,290e19443f0a5bbe1e93e5437ab83701e344a1bc,3238
6,580429720,4,290e19443f0a5bbe1e93e5437ab83701e344a1bc,3239
7,581765623,5,5b10d642e28b0aa4c6c6ba22633cc23c5ed5f2f6,3235
8,581765623,5,5b10d642e28b0aa4c6c6ba22633cc23c5ed5f2f6,3238
9,581765623,5,5b10d642e28b0aa4c6c6ba22633cc23c5ed5f2f6,3239



=== PROJECT 22 CELL 4 / STEP 2A RESULT ===
Project: apache@logging-log4j2
Project slug: apache__logging-log4j2
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Project 17 identity: yamcs@Yamcs
Project 18 identity: cantaloupe-project@cantaloupe
Project 19 identity: EMResearch@EvoMaster
Project 20 identity: apache@curator
Project 21 identity: facebook@buck
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']
Builds: 441
Training / evaluation builds: 330 / 111
Timestamp tie groups: 1
Raw execution rows: 240253
Model-ready rows: 117968
Dataset columns: 154
Predictor columns: 151
REC features: 19

Build-Test joins:
Raw duplicate Build-Test rows: 0
Model duplicate Build-Test rows: 0
Missing model-to-raw links:

In [6]:
# ==================================================================================================
# PROJECT 22 — CELL 5 / STEP 2B
# DETERMINISTIC CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE
#
# PROJECT:
#   apache@logging-log4j2
#
# WHY THIS IMPLEMENTATION IS SAFE:
# - Project 22 has exactly one timestamp-tie group under the frozen source chronology contract.
# - Each raw Build-Test pair is unique.
# - Exact per-test tie-order enumeration resolves any tests that execute in both tied builds.
# - The 16 non-file history features validate each inferred per-test order independently of file mapping.
# - REC_Age independently selects the globally consistent order for the tied builds.
# - The 16 non-file history features are reconstructed with vectorized cumulative calculations.
# - The two file-history features are reconstructed from the Step 2A build-entity map.
# - Clean anchor offsets preserve any accepted source-level file-mapping residuals exactly.
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME Thesis_project_22.ipynb NOTEBOOK.
# DO NOT RERUN PROJECTS 1–21 OR PROJECT 22 STEPS 0–2A.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
from collections import defaultdict
from itertools import permutations, product
import math

import gc
import hashlib
import json
import os
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


print("=" * 136)
print("=== PROJECT 22 CELL 5 / STEP 2B: DETERMINISTIC CLEAN REC RECONSTRUCTION ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 22
PROJECT_NAME = "apache@logging-log4j2"
PROJECT_SLUG = "apache__logging-log4j2"
PROJECT_SHORT = "LOG4J2"

SOURCE_DIR = Path(
    "/content/datasets/datasets/apache@logging-log4j2"
)

EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_22_SELECTION_AND_SOURCE_FROZEN"
)

EXPECTED_STEP2A_STATUS = (
    "PASS_PROJECT_22_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_VALIDATED"
)

STEP2B_STATUS = (
    "PASS_PROJECT_22_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

IMPLEMENTATION_VERSION = (
    "PROJECT_22_V1_EXACT_TIMESTAMP_TIE_INFERENCE_FULL_MAPPING"
)

EXPECTED_SELECTION_SHA256 = (
    "a7387d9495c71dce5d2c21ff08b0afd80d9c5251b5885a82e4052c7506ee7890"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64f842964c896c6ac334"
)

EXPECTED_REGISTRY_SHA256 = (
    "79cd6ecb595c5e8ae91a9494e469792716338d144308560a62caf1b9342306b2"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 21

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 101_286_750

EXPECTED_BUILDS = 441
EXPECTED_TRAIN_BUILDS = 330
EXPECTED_EVAL_BUILDS = 111
EXPECTED_TIMESTAMP_TIE_GROUPS = 1

EXPECTED_RAW_ROWS = 240_253
EXPECTED_RAW_TRAIN_ROWS = 172_628
EXPECTED_RAW_EVAL_ROWS = 67_625
EXPECTED_RAW_TRAIN_FAILURES = 208
EXPECTED_RAW_EVAL_FAILURES = 40

EXPECTED_MODEL_ROWS = 117_968
EXPECTED_MODEL_TRAIN_ROWS = 95_812
EXPECTED_MODEL_EVAL_ROWS = 22_156
EXPECTED_MODEL_TRAIN_FAILURES = 207
EXPECTED_MODEL_EVAL_FAILURES = 40

EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTORS = 151

EXPECTED_COMMIT_TOKEN_ROWS = 567
EXPECTED_EXACT_COMMIT_MATCHES = 567
EXPECTED_PREFIX_COMMIT_MATCHES = 0
EXPECTED_UNMATCHED_COMMIT_TOKENS = 0
EXPECTED_AMBIGUOUS_COMMIT_TOKENS = 0
EXPECTED_BUILDS_WITH_MAPPED_ENTITIES = 441
EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES = 0
EXPECTED_BUILD_ENTITY_ROWS = 3_151

EXPECTED_MAPPING_INCOMPLETE_BUILDS = set()
EXPECTED_MAPPING_INCOMPLETE_PARTITIONS = set()
EXPECTED_MAPPING_INCOMPLETE_ROWS_WITH_ENTITIES = 0

RECENT_WINDOW = 6

SUCCESS_VERDICT_CODE = 0
EXCEPTION_VERDICT_CODE = 1
ASSERTION_VERDICT_CODE = 2

DIRECT_RTOL = 1e-9
DIRECT_ATOL = 1e-9

ANCHOR_RTOL = 0.0
ANCHOR_ATOL = 1e-12

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

FILE_HISTORY_REC = [
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

TIE_INFERENCE_FEATURES = [
    feature
    for feature in REC_FEATURES
    if feature != "REC_Age"
    and feature not in FILE_HISTORY_REC
]

MAX_TIE_ORDER_COMBINATIONS = 1_024

NON_FILE_REC = [
    feature
    for feature in REC_FEATURES
    if feature not in FILE_HISTORY_REC
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_22_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_22_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_22_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_22_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_22_fixed_chronological_builds.csv"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

STEP2A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2a_status.json"
)

STEP2A_REPORT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2a_report.json"
)

ENTITY_MAPPING_SUMMARY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_entity_mapping_summary.json"
)

COMMIT_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_commit_matching_audit.csv"
)

BUILD_ENTITY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
)

MAPPING_INCOMPLETE_BUILDS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_mapping_incomplete_builds.csv"
)

UNMATCHED_MAPPING_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_unmatched_mapping_audit.csv"
)

TIMESTAMP_TIE_GROUPS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_timestamp_tie_groups.csv"
)

TEST_ORDER_SEARCH_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_test_order_search_audit.csv"
)

INFERRED_EXECUTION_ORDER_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_inferred_execution_order.parquet"
)

GLOBAL_AGE_ORDER_SEARCH_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_global_age_order_search.csv"
)

FROZEN_GLOBAL_BUILD_ORDER_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_global_build_order.csv"
)

CLEAN_RECONSTRUCTED_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
)

CLEAN_ANCHOR_OFFSETS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
)

CLEAN_COMPARISON_SUMMARY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_comparison_summary.csv"
)

CLEAN_MISMATCH_EXAMPLES_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_mismatch_examples.csv"
)

CLEAN_ANCHOR_VALIDATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_anchor_validation.csv"
)

STEP2B_VALIDATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2b_validation.csv"
)

STEP2B_REPORT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2b_report.json"
)

STEP2B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2b_status.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_22_rec_reconstruction_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_parquet(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_parquet(
        temporary_path,
        index=False,
        compression="zstd",
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def parse_int(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing/non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(
            array
        ),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype(
        "int64"
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


def prefix_sum(
    values,
):
    values = np.asarray(
        values
    )

    dtype = (
        np.float64
        if values.dtype.kind == "f"
        else np.int64
    )

    result = np.empty(
        len(values) + 1,
        dtype=dtype,
    )

    result[0] = 0

    np.cumsum(
        values,
        out=result[1:],
    )

    return result


def safe_divide(
    numerator,
    denominator,
):
    numerator = np.asarray(
        numerator,
        dtype=float,
    )

    denominator = np.asarray(
        denominator,
        dtype=float,
    )

    result = np.full(
        len(denominator),
        -1.0,
        dtype=float,
    )

    valid = denominator > 0

    result[
        valid
    ] = (
        numerator[
            valid
        ]
        / denominator[
            valid
        ]
    )

    return result


def calculate_file_rate(
    target_builds,
    current_changed_entities,
    entity_changed_builds,
):
    if not target_builds:
        return -1.0

    maximum_frequency = 0

    for entity_id in current_changed_entities:
        changed_builds = entity_changed_builds.get(
            int(
                entity_id
            )
        )

        if not changed_builds:
            continue

        overlap_count = len(
            target_builds.intersection(
                changed_builds
            )
        )

        if overlap_count > maximum_frequency:
            maximum_frequency = overlap_count

    if maximum_frequency == 0:
        return 0.0

    return float(
        maximum_frequency
        / len(
            target_builds
        )
    )


def reconstruct_requested_group_features(
    builds,
    verdicts,
    durations,
    global_positions,
    requested_positions,
    changed_entities_by_build,
    entity_changed_builds,
):
    builds = np.asarray(
        builds,
        dtype=np.int64,
    )

    verdicts = np.asarray(
        verdicts,
        dtype=np.int64,
    )

    durations = np.asarray(
        durations,
        dtype=np.float64,
    )

    global_positions = np.asarray(
        global_positions,
        dtype=np.int64,
    )

    requested_positions = np.asarray(
        requested_positions,
        dtype=np.int64,
    )

    n = len(
        builds
    )

    all_positions = np.arange(
        n,
        dtype=np.int64,
    )

    failure = (
        verdicts
        != SUCCESS_VERDICT_CODE
    ).astype(
        np.int64
    )

    assertion = (
        verdicts
        == ASSERTION_VERDICT_CODE
    ).astype(
        np.int64
    )

    exception = (
        verdicts
        == EXCEPTION_VERDICT_CODE
    ).astype(
        np.int64
    )

    transition = np.zeros(
        n,
        dtype=np.int64,
    )

    if n > 1:
        transition[
            1:
        ] = (
            verdicts[
                1:
            ]
            != verdicts[
                :-1
            ]
        ).astype(
            np.int64
        )

    duration_prefix = prefix_sum(
        durations
    )

    failure_prefix = prefix_sum(
        failure
    )

    assertion_prefix = prefix_sum(
        assertion
    )

    exception_prefix = prefix_sum(
        exception
    )

    transition_prefix = prefix_sum(
        transition
    )

    positions = requested_positions

    history_length = positions.astype(
        float
    )

    recent_start = np.maximum(
        0,
        positions - RECENT_WINDOW,
    )

    recent_length = (
        positions
        - recent_start
    ).astype(
        float
    )

    last_failure_inclusive = np.maximum.accumulate(
        np.where(
            failure > 0,
            all_positions,
            -1,
        )
    )

    last_transition_inclusive = np.maximum.accumulate(
        np.where(
            transition > 0,
            all_positions,
            -1,
        )
    )

    prior_failure_position = np.full(
        len(
            positions
        ),
        -1,
        dtype=np.int64,
    )

    prior_transition_position = np.full(
        len(
            positions
        ),
        -1,
        dtype=np.int64,
    )

    positive_history = positions > 0

    prior_failure_position[
        positive_history
    ] = last_failure_inclusive[
        positions[
            positive_history
        ]
        - 1
    ]

    prior_transition_position[
        positive_history
    ] = last_transition_inclusive[
        positions[
            positive_history
        ]
        - 1
    ]

    recent_max = np.full(
        n,
        np.nan,
        dtype=float,
    )

    for offset in range(
        1,
        RECENT_WINDOW + 1,
    ):
        if n <= offset:
            continue

        recent_max[
            offset:
        ] = np.fmax(
            recent_max[
                offset:
            ],
            durations[
                :-offset
            ],
        )

    total_max_inclusive = np.maximum.accumulate(
        durations
    )

    previous_indices = np.maximum(
        positions - 1,
        0,
    )

    reconstructed = {
        "REC_Age":
            (
                global_positions[
                    positions
                ]
                - global_positions[
                    0
                ]
            ).astype(
                float
            ),

        "REC_LastFailureAge":
            np.where(
                prior_failure_position < 0,
                -1.0,
                (
                    positions
                    - 1
                    - prior_failure_position
                ).astype(
                    float
                ),
            ),

        "REC_LastTransitionAge":
            np.where(
                prior_transition_position < 0,
                -1.0,
                (
                    positions
                    - 1
                    - prior_transition_position
                ).astype(
                    float
                ),
            ),

        "REC_RecentAvgExeTime":
            safe_divide(
                (
                    duration_prefix[
                        positions
                    ]
                    - duration_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_RecentMaxExeTime":
            np.where(
                positive_history,
                recent_max[
                    positions
                ],
                -1.0,
            ),

        "REC_RecentFailRate":
            safe_divide(
                (
                    failure_prefix[
                        positions
                    ]
                    - failure_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_RecentAssertRate":
            safe_divide(
                (
                    assertion_prefix[
                        positions
                    ]
                    - assertion_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_RecentExcRate":
            safe_divide(
                (
                    exception_prefix[
                        positions
                    ]
                    - exception_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_RecentTransitionRate":
            safe_divide(
                (
                    transition_prefix[
                        positions
                    ]
                    - transition_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_TotalAvgExeTime":
            safe_divide(
                duration_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_TotalMaxExeTime":
            np.where(
                positive_history,
                total_max_inclusive[
                    previous_indices
                ],
                -1.0,
            ),

        "REC_TotalFailRate":
            safe_divide(
                failure_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_TotalAssertRate":
            safe_divide(
                assertion_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_TotalExcRate":
            safe_divide(
                exception_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_TotalTransitionRate":
            safe_divide(
                transition_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_LastVerdict":
            np.where(
                positive_history,
                verdicts[
                    previous_indices
                ],
                -1,
            ).astype(
                float
            ),

        "REC_LastExeTime":
            np.where(
                positive_history,
                durations[
                    previous_indices
                ],
                -1.0,
            ),
    }

    file_failure_rate = np.empty(
        len(
            positions
        ),
        dtype=float,
    )

    file_transition_rate = np.empty(
        len(
            positions
        ),
        dtype=float,
    )

    failure_event_positions = np.flatnonzero(
        failure > 0
    )

    transition_event_positions = np.flatnonzero(
        transition > 0
    )

    failure_pointer = 0
    transition_pointer = 0

    prior_failure_builds = set()
    prior_transition_builds = set()

    requested_order = np.argsort(
        positions,
        kind="mergesort",
    )

    for requested_index in requested_order:
        current_position = int(
            positions[
                requested_index
            ]
        )

        while (
            failure_pointer
            < len(
                failure_event_positions
            )
            and int(
                failure_event_positions[
                    failure_pointer
                ]
            )
            < current_position
        ):
            prior_failure_builds.add(
                int(
                    builds[
                        failure_event_positions[
                            failure_pointer
                        ]
                    ]
                )
            )

            failure_pointer += 1

        while (
            transition_pointer
            < len(
                transition_event_positions
            )
            and int(
                transition_event_positions[
                    transition_pointer
                ]
            )
            < current_position
        ):
            prior_transition_builds.add(
                int(
                    builds[
                        transition_event_positions[
                            transition_pointer
                        ]
                    ]
                )
            )

            transition_pointer += 1

        current_build = int(
            builds[
                current_position
            ]
        )

        current_entities = changed_entities_by_build.get(
            current_build,
            frozenset(),
        )

        file_failure_rate[
            requested_index
        ] = calculate_file_rate(
            target_builds=prior_failure_builds,
            current_changed_entities=current_entities,
            entity_changed_builds=entity_changed_builds,
        )

        file_transition_rate[
            requested_index
        ] = calculate_file_rate(
            target_builds=prior_transition_builds,
            current_changed_entities=current_entities,
            entity_changed_builds=entity_changed_builds,
        )

    reconstructed[
        "REC_MaxTestFileFailRate"
    ] = file_failure_rate

    reconstructed[
        "REC_MaxTestFileTransitionRate"
    ] = file_transition_rate

    return (
        reconstructed,
        transition,
    )


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN-STATE VALIDATION
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    STEP2A_STATUS_PATH,
    STEP2A_REPORT_PATH,
    ENTITY_MAPPING_SUMMARY_PATH,
    COMMIT_AUDIT_PATH,
    BUILD_ENTITY_PATH,
    MAPPING_INCOMPLETE_BUILDS_PATH,
    SOURCE_DIR / "builds.csv",
    SOURCE_DIR / "dataset.csv",
    SOURCE_DIR / "exe.csv",
]

missing_paths = [
    str(
        path
    )
    for path in required_paths
    if not Path(
        path
    ).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 22 Step 2B inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


selection_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

selection = load_json(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = load_json(
    STEP1B_STATUS_PATH
)

step2a_status = load_json(
    STEP2A_STATUS_PATH
)

step2a_report = load_json(
    STEP2A_REPORT_PATH
)

entity_mapping_summary = load_json(
    ENTITY_MAPPING_SUMMARY_PATH
)


if selection_sha256 != EXPECTED_SELECTION_SHA256:
    raise RuntimeError(
        "Project 22 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_SHA256}\n"
        f"Actual:   {selection_sha256}"
    )


if (
    selection.get(
        "Status"
    )
    != EXPECTED_STEP1B_STATUS
    or step1b_status.get(
        "Status"
    )
    != EXPECTED_STEP1B_STATUS
):
    raise RuntimeError(
        "Project 22 Step 1B is not frozen successfully."
    )


if (
    step2a_status.get(
        "Status"
    )
    != EXPECTED_STEP2A_STATUS
    or step2a_report.get(
        "Status"
    )
    != EXPECTED_STEP2A_STATUS
    or entity_mapping_summary.get(
        "Status"
    )
    != EXPECTED_STEP2A_STATUS
):
    raise RuntimeError(
        "Project 22 Step 2A outputs are not in the expected PASS state."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–21."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–21 are not all COMPLETE_AND_FROZEN."
    )


required_registered_identities = {
    11:
        "apache@shardingsphere",

    12:
        "zolyfarkas@spf4j",

    13:
        "jcabi@jcabi-github",

    14:
        "JMRI@JMRI",

    15:
        "eclipse@steady",

    16:
        "apache@rocketmq",

    17:
        "yamcs@Yamcs",

    18:
        "cantaloupe-project@cantaloupe",

    19:
        "EMResearch@EvoMaster",

    20:
        "apache@curator",

    21:
        "facebook@buck",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 22 is unexpectedly already registered."
    )


if selection.get(
    "Project"
) != PROJECT_NAME or selection.get(
    "ProjectSlug"
) != PROJECT_SLUG:
    raise RuntimeError(
        "Frozen Project 22 identity differs."
    )


active_reservations = selection.get(
    "ActiveReservations",
    [],
)


if active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Frozen Project 22 active-reservation state differs."
    )


if selection.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Frozen Project 22 runtime-priority rule differs."
    )


frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_manifest_records = []

for row in frozen_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            "A frozen Project 22 source file is missing:\n"
            f"{source_path}"
        )

    current_source_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


current_source_manifest = pd.DataFrame(
    current_source_manifest_records
)


current_source_root_sha256 = source_root_hash(
    current_source_manifest
)


current_source_bytes = int(
    current_source_manifest[
        "SizeBytes"
    ].sum()
)


if (
    len(
        current_source_manifest
    )
    != EXPECTED_SOURCE_FILES
    or current_source_bytes
    != EXPECTED_SOURCE_BYTES
    or current_source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The local Project 22 source does not match "
        "the frozen source manifest."
    )


# --------------------------------------------------------------------------------------------------
# 5. LOAD CHRONOLOGY, SOURCE DATA, AND STEP 2A MAPPING
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)


chronology[
    "BuildID"
] = parse_int(
    chronology[
        "BuildID"
    ],
    "chronology.BuildID",
)


chronology[
    "ChronologyOrder"
] = parse_int(
    chronology[
        "ChronologyOrder"
    ],
    "chronology.ChronologyOrder",
)


chronology[
    "StartedAtUTC"
] = pd.to_datetime(
    chronology[
        "StartedAtUTC"
    ],
    errors="coerce",
    utc=True,
)


if chronology[
    "StartedAtUTC"
].isna().any():
    raise RuntimeError(
        "The frozen chronology contains invalid timestamps."
    )


training_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "TRAIN"
        ),
        "BuildID",
    ].astype(
        int
    )
)


evaluation_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildID",
    ].astype(
        int
    )
)


all_builds = (
    training_builds
    | evaluation_builds
)


timestamp_group_sizes = (
    chronology.groupby(
        "StartedAtUTC"
    )
    .size()
)


timestamp_tie_groups_count = int(
    timestamp_group_sizes.gt(
        1
    ).sum()
)


timestamp_tie_builds = int(
    timestamp_group_sizes.loc[
        timestamp_group_sizes.gt(
            1
        )
    ].sum()
)


if timestamp_tie_groups_count != EXPECTED_TIMESTAMP_TIE_GROUPS:
    raise RuntimeError(
        "Project 22 timestamp-tie count differs from the frozen selection contract."
    )


timestamp_tie_groups = []
timestamp_tie_group_records = []

tied_rows = chronology.loc[
    chronology["StartedAtUTC"].duplicated(keep=False)
].copy()

for tie_group_number, (started_at, group) in enumerate(
    tied_rows.groupby("StartedAtUTC", sort=True),
    start=1,
):
    baseline_builds = (
        group.sort_values("ChronologyOrder", kind="mergesort")["BuildID"]
        .astype(int)
        .tolist()
    )

    permutation_count = math.factorial(len(baseline_builds))
    if permutation_count > MAX_TIE_ORDER_COMBINATIONS:
        raise RuntimeError(
            "A timestamp-tie group is too large for exact enumeration.\n"
            f"StartedAtUTC={started_at}; builds={baseline_builds}; "
            f"permutations={permutation_count}"
        )

    options = [tuple(int(value) for value in order) for order in permutations(baseline_builds)]
    timestamp_tie_groups.append({
        "TieGroup": tie_group_number,
        "StartedAtUTC": started_at,
        "BuildIDs": tuple(baseline_builds),
        "Options": options,
    })

    timestamp_tie_group_records.append({
        "TieGroup": tie_group_number,
        "StartedAtUTC": started_at.isoformat(),
        "BuildCount": len(baseline_builds),
        "BuildIDsJSON": json.dumps(baseline_builds),
        "PermutationCount": permutation_count,
    })


timestamp_tie_groups_frame = pd.DataFrame(
    timestamp_tie_group_records,
    columns=[
        "TieGroup",
        "StartedAtUTC",
        "BuildCount",
        "BuildIDsJSON",
        "PermutationCount",
    ],
)


build_chronology_map = chronology.set_index(
    "BuildID"
)[
    "ChronologyOrder"
].astype(
    int
).to_dict()


build_timestamp_map = chronology.set_index(
    "BuildID"
)[
    "StartedAtUTC"
].to_dict()


dataset_header = pd.read_csv(
    SOURCE_DIR / "dataset.csv",
    nrows=0,
).columns.tolist()


exe_header = pd.read_csv(
    SOURCE_DIR / "exe.csv",
    nrows=0,
).columns.tolist()


model_build_column = resolve_column(
    dataset_header,
    "Build",
    "dataset Build",
)

model_test_column = resolve_column(
    dataset_header,
    "Test",
    "dataset Test",
)

model_verdict_column = resolve_column(
    dataset_header,
    "Verdict",
    "dataset Verdict",
)


exe_test_column = resolve_column(
    exe_header,
    "test",
    "exe test",
)

exe_build_column = resolve_column(
    exe_header,
    "build",
    "exe build",
)

exe_job_column = resolve_column(
    exe_header,
    "job",
    "exe job",
)

exe_verdict_column = resolve_column(
    exe_header,
    "verdict",
    "exe verdict",
)

exe_duration_column = resolve_column(
    exe_header,
    "duration",
    "exe duration",
)


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in dataset_header
]


if missing_rec_features:
    raise RuntimeError(
        "dataset.csv is missing REC features:\n"
        + "\n".join(
            missing_rec_features
        )
    )


predictor_columns = [
    column
    for column in dataset_header
    if column not in {
        model_build_column,
        model_test_column,
        model_verdict_column,
    }
]


dataset = pd.read_csv(
    SOURCE_DIR / "dataset.csv",
    usecols=[
        model_build_column,
        model_test_column,
        model_verdict_column,
    ] + REC_FEATURES,
    low_memory=False,
)


dataset[
    model_build_column
] = parse_int(
    dataset[
        model_build_column
    ],
    "dataset.Build",
)


dataset[
    model_test_column
] = parse_int(
    dataset[
        model_test_column
    ],
    "dataset.Test",
)


dataset[
    model_verdict_column
] = parse_int(
    dataset[
        model_verdict_column
    ],
    "dataset.Verdict",
)


dataset = dataset.rename(
    columns={
        model_build_column:
            "Build",

        model_test_column:
            "Test",

        model_verdict_column:
            "Verdict",
    }
).reset_index(
    drop=True
)


dataset[
    "_ModelRow"
] = np.arange(
    len(
        dataset
    ),
    dtype=np.int64,
)


print(
    "Loading the 59,155-row clean execution history."
)


exe = pd.read_csv(
    SOURCE_DIR / "exe.csv",
    usecols=[
        exe_test_column,
        exe_build_column,
        exe_job_column,
        exe_verdict_column,
        exe_duration_column,
    ],
    low_memory=False,
)


exe[
    exe_build_column
] = parse_int(
    exe[
        exe_build_column
    ],
    "exe.build",
)


exe[
    exe_test_column
] = parse_int(
    exe[
        exe_test_column
    ],
    "exe.test",
)


exe[
    exe_verdict_column
] = parse_int(
    exe[
        exe_verdict_column
    ],
    "exe.verdict",
)


exe[
    exe_job_column
] = pd.to_numeric(
    exe[
        exe_job_column
    ],
    errors="coerce",
)


exe[
    exe_duration_column
] = pd.to_numeric(
    exe[
        exe_duration_column
    ],
    errors="coerce",
)


if exe[
    exe_job_column
].isna().any():
    raise RuntimeError(
        "exe.csv contains missing/non-numeric job values."
    )


if not np.isfinite(
    exe[
        exe_duration_column
    ].to_numpy(
        dtype=float
    )
).all():
    raise RuntimeError(
        "exe.csv contains non-finite durations."
    )


if exe[
    exe_duration_column
].lt(
    0
).any():
    raise RuntimeError(
        "exe.csv contains negative durations."
    )


observed_verdict_codes = sorted(
    int(
        value
    )
    for value in exe[
        exe_verdict_column
    ].unique().tolist()
)


if not set(
    observed_verdict_codes
).issubset({
    0,
    1,
    2,
    3,
}):
    raise RuntimeError(
        "exe.csv contains an unsupported verdict code.\n"
        f"Observed codes: {observed_verdict_codes}"
    )


exe = exe.rename(
    columns={
        exe_build_column:
            "Build",

        exe_test_column:
            "Test",

        exe_job_column:
            "Job",

        exe_verdict_column:
            "Verdict",

        exe_duration_column:
            "Duration",
    }
)


exe[
    "ChronologyOrder"
] = exe[
    "Build"
].map(
    build_chronology_map
)


if exe[
    "ChronologyOrder"
].isna().any():
    raise RuntimeError(
        "Some execution rows cannot be mapped to frozen chronology."
    )


exe[
    "ChronologyOrder"
] = exe[
    "ChronologyOrder"
].astype(
    np.int64
)


raw_duplicate_pairs = int(
    exe.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


model_duplicate_pairs = int(
    dataset.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


if (
    raw_duplicate_pairs != 0
    or model_duplicate_pairs != 0
):
    raise RuntimeError(
        "Duplicate Build-Test pairs prevent exact REC reconstruction."
    )


raw_build_ids = set(
    exe[
        "Build"
    ].astype(
        int
    ).unique().tolist()
)


global_build_sequence = (
    chronology.loc[
        chronology[
            "BuildID"
        ].isin(
            raw_build_ids
        )
    ]
    .sort_values(
        "ChronologyOrder",
        kind="mergesort",
    )[
        "BuildID"
    ]
    .astype(
        int
    )
    .tolist()
)


global_build_position = {
    int(
        build_id
    ):
        position
    for position, build_id in enumerate(
        global_build_sequence
    )
}


exe[
    "GlobalBuildPosition"
] = exe[
    "Build"
].map(
    global_build_position
)


if exe[
    "GlobalBuildPosition"
].isna().any():
    raise RuntimeError(
        "Some execution rows cannot be mapped to global first-appearance order."
    )


exe[
    "GlobalBuildPosition"
] = exe[
    "GlobalBuildPosition"
].astype(
    np.int64
)


print(
    "Sorting raw execution history by Test and frozen chronology."
)


sort_started = time.perf_counter()


exe = (
    exe.sort_values(
        [
            "Test",
            "ChronologyOrder",
            "Build",
            "Job",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


exe[
    "InferredTestOrder"
] = (
    exe.groupby(
        "Test",
        sort=False,
    )
    .cumcount()
    .astype(
        np.int64
    )
)


sort_seconds = float(
    time.perf_counter()
    - sort_started
)


commit_audit = pd.read_csv(
    COMMIT_AUDIT_PATH,
    low_memory=False,
)


build_entity = pd.read_csv(
    BUILD_ENTITY_PATH,
    compression="gzip",
    low_memory=False,
)


mapping_incomplete_source = pd.read_csv(
    MAPPING_INCOMPLETE_BUILDS_PATH,
    low_memory=False,
)


commit_audit[
    "BuildID"
] = parse_int(
    commit_audit[
        "BuildID"
    ],
    "commit_audit.BuildID",
)


build_entity[
    "BuildID"
] = parse_int(
    build_entity[
        "BuildID"
    ],
    "build_entity.BuildID",
)


build_entity[
    "EntityId"
] = parse_int(
    build_entity[
        "EntityId"
    ],
    "build_entity.EntityId",
)


mapping_incomplete_source[
    "BuildID"
] = parse_int(
    mapping_incomplete_source[
        "BuildID"
    ],
    "mapping_incomplete.BuildID",
)


mapping_incomplete_source_partitions = sorted(
    set(
        mapping_incomplete_source[
            "Partition"
        ]
        .astype(str)
        .str.strip()
        .str.upper()
        .tolist()
    )
)


mapping_incomplete_source_rows_with_entities = int(
    mapping_incomplete_source[
        "HasMappedEntities"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin({
        "true",
        "1",
        "yes",
    })
    .sum()
)


normalised_match_type = (
    commit_audit[
        "MatchType"
    ]
    .astype(
        str
    )
    .str.strip()
    .str.upper()
)


exact_match_mask = normalised_match_type.eq(
    "EXACT"
)

prefix_match_mask = normalised_match_type.eq(
    "UNIQUE_PREFIX"
)

unmatched_mask = normalised_match_type.eq(
    "UNMATCHED"
)

ambiguous_mask = normalised_match_type.eq(
    "AMBIGUOUS_PREFIX"
)


unknown_match_type_rows = int(
    (
        ~(
            exact_match_mask
            | prefix_match_mask
            | unmatched_mask
            | ambiguous_mask
        )
    ).sum()
)


if unknown_match_type_rows != 0:
    raise RuntimeError(
        "Commit audit contains unknown MatchType rows."
    )


exact_matches = int(
    exact_match_mask.sum()
)

prefix_matches = int(
    prefix_match_mask.sum()
)

unmatched_tokens = int(
    unmatched_mask.sum()
)

ambiguous_tokens = int(
    ambiguous_mask.sum()
)


unmatched_token_builds = sorted(
    commit_audit.loc[
        unmatched_mask,
        "BuildID",
    ]
    .astype(
        int
    )
    .unique()
    .tolist()
)


builds_with_entities = set(
    build_entity[
        "BuildID"
    ].astype(
        int
    )
)


builds_without_entities = sorted(
    all_builds
    - builds_with_entities
)


mapping_incomplete_builds = sorted(
    set(
        unmatched_token_builds
    )
    | set(
        builds_without_entities
    )
)


changed_entities_by_build = {
    int(
        build_id
    ):
        frozenset(
            int(
                entity_id
            )
            for entity_id in values
        )
    for build_id, values in build_entity.groupby(
        "BuildID",
        sort=False,
    )[
        "EntityId"
    ]
}


entity_changed_builds_accumulator = defaultdict(
    set
)


for row in build_entity[
    [
        "BuildID",
        "EntityId",
    ]
].itertuples(
    index=False
):
    entity_changed_builds_accumulator[
        int(
            row.EntityId
        )
    ].add(
        int(
            row.BuildID
        )
    )


entity_changed_builds = {
    entity_id:
        frozenset(
            build_ids
        )
    for entity_id, build_ids in entity_changed_builds_accumulator.items()
}


del entity_changed_builds_accumulator
gc.collect()


raw_build_counts = exe.groupby(
    "Build",
    sort=False,
).size()


raw_build_failures = (
    exe[
        "Verdict"
    ]
    .ne(
        SUCCESS_VERDICT_CODE
    )
    .groupby(
        exe[
            "Build"
        ]
    )
    .sum()
)


model_build_counts = dataset.groupby(
    "Build",
    sort=False,
).size()


model_build_failures = (
    dataset[
        "Verdict"
    ]
    .ne(
        SUCCESS_VERDICT_CODE
    )
    .groupby(
        dataset[
            "Build"
        ]
    )
    .sum()
)


unmatched_mapping_audit = (
    mapping_incomplete_source.copy()
    .sort_values(
        "BuildID",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


unmatched_mapping_audit[
    "RawExecutionRows"
] = unmatched_mapping_audit[
    "BuildID"
].map(
    raw_build_counts
).fillna(
    0
).astype(
    int
)


unmatched_mapping_audit[
    "RawFailureRows"
] = unmatched_mapping_audit[
    "BuildID"
].map(
    raw_build_failures
).fillna(
    0
).astype(
    int
)


unmatched_mapping_audit[
    "ModelReadyRows"
] = unmatched_mapping_audit[
    "BuildID"
].map(
    model_build_counts
).fillna(
    0
).astype(
    int
)


unmatched_mapping_audit[
    "ModelFailureRows"
] = unmatched_mapping_audit[
    "BuildID"
].map(
    model_build_failures
).fillna(
    0
).astype(
    int
)


print(
    "\nTimestamp tie groups:"
)

display(
    timestamp_tie_groups_frame
)


print(
    "\nUnmatched mapping audit:"
)

display(
    unmatched_mapping_audit
)


# --------------------------------------------------------------------------------------------------
# 6. EXACT PER-TEST TIE-ORDER INFERENCE
# --------------------------------------------------------------------------------------------------

order_search_started = time.perf_counter()

model_group_indices = dataset.groupby("Test", sort=False).indices
raw_group_indices = exe.groupby("Test", sort=False).indices

source_rec_arrays = {
    feature: dataset[feature].to_numpy(dtype=float)
    for feature in REC_FEATURES
}

build_timestamp_ns = {
    int(build_id): int(pd.Timestamp(timestamp).value)
    for build_id, timestamp in build_timestamp_map.items()
}


def ordered_raw_indices_for_choice(raw_indices, tie_choice):
    rows = exe.loc[raw_indices, ["Build", "Job"]].copy()
    rows["_OriginalIndex"] = np.asarray(raw_indices, dtype=np.int64)
    rows["_TimestampNS"] = rows["Build"].map(build_timestamp_ns).astype(np.int64)
    rows["_TieRank"] = 0

    for group_number, selected_order in tie_choice.items():
        rank = {int(build_id): position for position, build_id in enumerate(selected_order)}
        mask = rows["Build"].isin(rank)
        rows.loc[mask, "_TieRank"] = rows.loc[mask, "Build"].map(rank).astype(int)

    rows = rows.sort_values(
        ["_TimestampNS", "_TieRank", "Build", "Job"],
        kind="mergesort",
    )
    return rows["_OriginalIndex"].to_numpy(dtype=np.int64)


def tie_options_for_test(build_ids):
    build_set = set(int(value) for value in build_ids)
    touched = []
    for tie_group in timestamp_tie_groups:
        present = [value for value in tie_group["BuildIDs"] if value in build_set]
        if len(present) > 1:
            options = [
                tuple(value for value in option if value in build_set)
                for option in tie_group["Options"]
            ]
            options = list(dict.fromkeys(options))
            touched.append((int(tie_group["TieGroup"]), options))
    return touched


test_order_search_records = []
inferred_raw_indices_by_test = {}

total_tests = len(raw_group_indices)

for test_number, (test_id_raw, raw_indices_raw) in enumerate(raw_group_indices.items(), start=1):
    test_id = int(test_id_raw)
    raw_indices = np.asarray(raw_indices_raw, dtype=np.int64)
    group_builds_baseline = exe.loc[raw_indices, "Build"].to_numpy(dtype=np.int64)
    touched_groups = tie_options_for_test(group_builds_baseline)
    model_rows = model_group_indices.get(test_id)

    if touched_groups:
        combination_count = int(np.prod([len(options) for _, options in touched_groups]))
    else:
        combination_count = 1

    if combination_count > MAX_TIE_ORDER_COMBINATIONS:
        raise RuntimeError(
            "A test requires too many exact tie-order combinations.\n"
            f"Test={test_id}; combinations={combination_count}"
        )

    choice_records = []
    choice_product = product(*[options for _, options in touched_groups]) if touched_groups else [tuple()]

    for candidate_number, selected_orders in enumerate(choice_product, start=1):
        tie_choice = {
            group_number: selected_order
            for (group_number, _), selected_order in zip(touched_groups, selected_orders)
        }
        candidate_indices = ordered_raw_indices_for_choice(raw_indices, tie_choice)

        if model_rows is None:
            mismatch_counts = {}
            mismatch_values = 0
        else:
            model_rows_array = np.asarray(model_rows, dtype=np.int64)
            requested_builds = dataset.loc[model_rows_array, "Build"].to_numpy(dtype=np.int64)
            candidate_builds = exe.loc[candidate_indices, "Build"].to_numpy(dtype=np.int64)
            position_by_build = {int(build_id): position for position, build_id in enumerate(candidate_builds)}
            missing_requested = [int(build_id) for build_id in requested_builds if int(build_id) not in position_by_build]
            if missing_requested:
                raise RuntimeError(
                    "A model-ready test contains builds missing from raw history.\n"
                    f"Test={test_id}; sample={missing_requested[:20]}"
                )
            requested_positions = np.asarray(
                [position_by_build[int(build_id)] for build_id in requested_builds],
                dtype=np.int64,
            )
            provisional_global = np.asarray(
                [global_build_position[int(build_id)] for build_id in candidate_builds],
                dtype=np.int64,
            )
            reconstructed_candidate, _ = reconstruct_requested_group_features(
                builds=candidate_builds,
                verdicts=exe.loc[candidate_indices, "Verdict"].to_numpy(dtype=np.int64),
                durations=exe.loc[candidate_indices, "Duration"].to_numpy(dtype=np.float64),
                global_positions=provisional_global,
                requested_positions=requested_positions,
                changed_entities_by_build=changed_entities_by_build,
                entity_changed_builds=entity_changed_builds,
            )
            mismatch_counts = {}
            for feature in TIE_INFERENCE_FEATURES:
                source_values = source_rec_arrays[feature][model_rows_array]
                reconstructed_values = reconstructed_candidate[feature]
                mismatch_counts[feature] = int((~np.isclose(
                    source_values,
                    reconstructed_values,
                    rtol=DIRECT_RTOL,
                    atol=DIRECT_ATOL,
                    equal_nan=False,
                )).sum())
            mismatch_values = int(sum(mismatch_counts.values()))

        choice_records.append({
            "Candidate": candidate_number,
            "TieChoice": tie_choice,
            "OrderedIndices": candidate_indices,
            "MismatchCounts": mismatch_counts,
            "MismatchValues": mismatch_values,
        })

    minimum_mismatch = min(record["MismatchValues"] for record in choice_records)
    best_records = [record for record in choice_records if record["MismatchValues"] == minimum_mismatch]
    selected_record = best_records[0]
    inferred_raw_indices_by_test[test_id] = selected_record["OrderedIndices"]

    search_mode = (
        "RAW_ONLY_TEST_FROZEN_TIE_ORDER"
        if model_rows is None and touched_groups
        else "RAW_ONLY_TEST_DIRECT_ORDER"
        if model_rows is None
        else "MODEL_READY_TEST_EXACT_TIE_SEARCH"
        if touched_groups
        else "MODEL_READY_TEST_DIRECT_ORDER"
    )

    test_order_search_records.append({
        "Test": test_id,
        "RawExecutionRows": len(raw_indices),
        "ModelReadyRows": 0 if model_rows is None else len(model_rows),
        "TimestampTieGroupsForTest": len(touched_groups),
        "CandidateOrderCombinations": combination_count,
        "MinimumMismatchValues": minimum_mismatch,
        "ZeroMismatchCandidates": int(sum(record["MismatchValues"] == 0 for record in choice_records)),
        "BestMismatchCountsJSON": json.dumps(selected_record["MismatchCounts"], sort_keys=True),
        "SelectedTieOrdersJSON": json.dumps(
            [list(selected_record["TieChoice"].get(group_number, tuple())) for group_number, _ in touched_groups]
        ),
        "SearchMode": search_mode,
    })

    if test_number % 100 == 0 or test_number == total_tests:
        print("Per-test tie-order inference progress:", test_number, "/", total_tests, "tests")


test_order_search_audit = pd.DataFrame(test_order_search_records)
model_ready_tests = int(test_order_search_audit["ModelReadyRows"].gt(0).sum())
raw_only_tests = int(test_order_search_audit["ModelReadyRows"].eq(0).sum())
tests_with_timestamp_ties = int(test_order_search_audit["TimestampTieGroupsForTest"].gt(0).sum())
tests_with_nonzero_order_mismatches = int(test_order_search_audit["MinimumMismatchValues"].gt(0).sum())
tests_with_ambiguous_zero_orders = int(test_order_search_audit["ZeroMismatchCandidates"].gt(1).sum())
total_test_order_mismatch_values = int(test_order_search_audit["MinimumMismatchValues"].sum())

order_search_seconds = float(time.perf_counter() - order_search_started)

print("\nPer-test tie-order inference summary:")
display(pd.DataFrame([
    {"Metric": "Tests", "Value": total_tests},
    {"Metric": "Model-ready tests", "Value": model_ready_tests},
    {"Metric": "Raw-only tests", "Value": raw_only_tests},
    {"Metric": "Tests touching timestamp ties", "Value": tests_with_timestamp_ties},
    {"Metric": "Tests with non-zero minimum mismatch", "Value": tests_with_nonzero_order_mismatches},
    {"Metric": "Total minimum mismatch values", "Value": total_test_order_mismatch_values},
    {"Metric": "Tests with multiple zero-mismatch orders", "Value": tests_with_ambiguous_zero_orders},
    {"Metric": "Inference seconds", "Value": order_search_seconds},
]))


# --------------------------------------------------------------------------------------------------
# 7. GLOBAL REC_AGE ORDER SEARCH AND FULL CLEAN RECONSTRUCTION
# --------------------------------------------------------------------------------------------------

global_order_search_records = []
global_tie_options = [tie_group["Options"] for tie_group in timestamp_tie_groups]
global_choice_product = product(*global_tie_options) if global_tie_options else [tuple()]

for candidate_number, selected_orders in enumerate(global_choice_product, start=1):
    selected_by_timestamp = {
        int(pd.Timestamp(tie_group["StartedAtUTC"]).value): tuple(int(value) for value in selected_order)
        for tie_group, selected_order in zip(timestamp_tie_groups, selected_orders)
    }

    candidate_sequence = []
    for started_at, group in chronology.groupby("StartedAtUTC", sort=True):
        timestamp_ns = int(pd.Timestamp(started_at).value)
        group_builds = [
            int(value)
            for value in group["BuildID"].astype(int).tolist()
            if int(value) in raw_build_ids
        ]
        if not group_builds:
            continue
        if timestamp_ns in selected_by_timestamp:
            order = [value for value in selected_by_timestamp[timestamp_ns] if value in set(group_builds)]
        else:
            order = [
                int(value)
                for value in group.sort_values("ChronologyOrder", kind="mergesort")["BuildID"].astype(int).tolist()
                if int(value) in raw_build_ids
            ]
        candidate_sequence.extend(order)

    candidate_position = {int(build_id): position for position, build_id in enumerate(candidate_sequence)}
    first_build_by_test = {
        int(test_id): int(exe.loc[indices, "Build"].iloc[0])
        for test_id, indices in inferred_raw_indices_by_test.items()
    }
    source_age = dataset["REC_Age"].to_numpy(dtype=float)
    reconstructed_age = np.asarray([
        candidate_position[int(build_id)] - candidate_position[first_build_by_test[int(test_id)]]
        for build_id, test_id in dataset[["Build", "Test"]].itertuples(index=False, name=None)
    ], dtype=float)
    age_mismatches = int((~np.isclose(
        source_age,
        reconstructed_age,
        rtol=DIRECT_RTOL,
        atol=DIRECT_ATOL,
        equal_nan=False,
    )).sum())

    global_order_search_records.append({
        "Candidate": candidate_number,
        "AgeMismatchRows": age_mismatches,
        "BuildOrderSHA256": hashlib.sha256(
            ",".join(str(build_id) for build_id in candidate_sequence).encode("utf-8")
        ).hexdigest(),
        "TieOrdersJSON": json.dumps([list(order) for order in selected_orders]),
        "BuildSequence": candidate_sequence,
        "BuildPosition": candidate_position,
    })

best_age_mismatches = min(record["AgeMismatchRows"] for record in global_order_search_records)
best_global_records = [record for record in global_order_search_records if record["AgeMismatchRows"] == best_age_mismatches]
selected_global_record = best_global_records[0]
global_build_sequence = selected_global_record["BuildSequence"]
global_build_position = selected_global_record["BuildPosition"]
global_age_combination_count = len(global_order_search_records)
zero_age_candidates = int(sum(record["AgeMismatchRows"] == 0 for record in global_order_search_records))

global_age_order_search = pd.DataFrame([
    {key: value for key, value in record.items() if key not in {"BuildSequence", "BuildPosition"}}
    for record in global_order_search_records
])

print("\nGlobal REC_Age tie-order search:")
display(global_age_order_search)

reconstruction_started = time.perf_counter()
model_group_indices = dataset.groupby("Test", sort=False).indices
model_build_array = dataset["Build"].to_numpy(dtype=np.int64)
result_arrays = {
    feature: np.full(len(dataset), np.nan, dtype=np.float64)
    for feature in REC_FEATURES
}
filled_model_rows = np.zeros(len(dataset), dtype=bool)
inferred_order_lookup = {}

for test_number, (test_id_raw, ordered_indices) in enumerate(inferred_raw_indices_by_test.items(), start=1):
    test_id = int(test_id_raw)
    ordered_indices = np.asarray(ordered_indices, dtype=np.int64)
    candidate_builds = exe.loc[ordered_indices, "Build"].to_numpy(dtype=np.int64)
    for position, build_id in enumerate(candidate_builds):
        inferred_order_lookup[(test_id, int(build_id))] = position

    model_rows = model_group_indices.get(test_id)
    if model_rows is None:
        continue
    model_rows = np.asarray(model_rows, dtype=np.int64)
    requested_builds = model_build_array[model_rows]
    position_by_build = {int(build_id): position for position, build_id in enumerate(candidate_builds)}
    requested_positions = np.asarray([position_by_build[int(build_id)] for build_id in requested_builds], dtype=np.int64)
    candidate_global_positions = np.asarray([global_build_position[int(build_id)] for build_id in candidate_builds], dtype=np.int64)

    reconstructed_group, _ = reconstruct_requested_group_features(
        builds=candidate_builds,
        verdicts=exe.loc[ordered_indices, "Verdict"].to_numpy(dtype=np.int64),
        durations=exe.loc[ordered_indices, "Duration"].to_numpy(dtype=np.float64),
        global_positions=candidate_global_positions,
        requested_positions=requested_positions,
        changed_entities_by_build=changed_entities_by_build,
        entity_changed_builds=entity_changed_builds,
    )

    for feature in REC_FEATURES:
        result_arrays[feature][model_rows] = reconstructed_group[feature]
    filled_model_rows[model_rows] = True

    if test_number % 100 == 0 or test_number == total_tests:
        print("Full REC reconstruction progress:", test_number, "/", total_tests, "tests | reconstructed rows:", int(filled_model_rows.sum()))

if not filled_model_rows.all():
    missing_model_rows = np.flatnonzero(~filled_model_rows)
    raise RuntimeError(
        "Clean REC reconstruction did not fill every model-ready row.\n"
        f"Missing rows: {len(missing_model_rows)}; sample={missing_model_rows[:20].tolist()}"
    )

exe["InferredTestOrder"] = np.asarray([
    inferred_order_lookup[(int(test_id), int(build_id))]
    for test_id, build_id in exe[["Test", "Build"]].itertuples(index=False, name=None)
], dtype=np.int64)
exe["GlobalBuildPosition"] = exe["Build"].map(global_build_position).astype(np.int64)
exe = exe.sort_values(["Test", "InferredTestOrder"], kind="mergesort").reset_index(drop=True)

clean_reconstructed = dataset[["Build", "Test"]].copy()
for feature in REC_FEATURES:
    clean_reconstructed[feature] = result_arrays[feature]

reconstruction_seconds = float(time.perf_counter() - reconstruction_started)

frozen_global_build_order = pd.DataFrame({
    "GlobalBuildOrder": np.arange(1, len(global_build_sequence) + 1, dtype=np.int64),
    "BuildID": global_build_sequence,
})
frozen_global_build_order["StartedAtUTC"] = frozen_global_build_order["BuildID"].map(build_timestamp_map)

reconstructed_duplicate_rows = int(
    clean_reconstructed.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


missing_reconstructed_rows = int(
    clean_reconstructed[
        REC_FEATURES
    ].isna().any(
        axis=1
    ).sum()
)


if reconstructed_duplicate_rows != 0:
    raise RuntimeError(
        "Clean REC reconstruction produced duplicate Build-Test rows."
    )


if missing_reconstructed_rows != 0:
    raise RuntimeError(
        "Clean REC reconstruction contains missing values."
    )


comparison_records = []
mismatch_examples = []


anchor_offsets = dataset[
    [
        "Build",
        "Test",
    ]
].copy()


mapping_incomplete_build_set = set(
    mapping_incomplete_builds
)


rows_at_mapping_incomplete_build = dataset[
    "Build"
].isin(
    mapping_incomplete_build_set
).to_numpy()


for feature in REC_FEATURES:
    original_values = dataset[
        feature
    ].to_numpy(
        dtype=float
    )

    reconstructed_values = clean_reconstructed[
        feature
    ].to_numpy(
        dtype=float
    )

    if (
        not np.isfinite(
            original_values
        ).all()
        or not np.isfinite(
            reconstructed_values
        ).all()
    ):
        raise RuntimeError(
            f"Feature {feature} contains non-finite comparison values."
        )

    direct_match_mask = np.isclose(
        original_values,
        reconstructed_values,
        rtol=DIRECT_RTOL,
        atol=DIRECT_ATOL,
        equal_nan=False,
    )

    direct_mismatch_mask = (
        ~direct_match_mask
    )

    direct_difference = (
        original_values
        - reconstructed_values
    )

    anchor_offsets[
        feature
    ] = direct_difference

    anchored_values = (
        reconstructed_values
        + direct_difference
    )

    anchored_match_mask = np.isclose(
        original_values,
        anchored_values,
        rtol=ANCHOR_RTOL,
        atol=ANCHOR_ATOL,
        equal_nan=False,
    )

    comparison_records.append({
        "Feature":
            feature,

        "FeatureClass":
            (
                "VERDICT_DEPENDENT"
                if feature in VERDICT_DEPENDENT_REC
                else "VERDICT_INDEPENDENT"
            ),

        "FileHistoryFeature":
            feature in FILE_HISTORY_REC,

        "Rows":
            len(
                dataset
            ),

        "DirectMatchingRows":
            int(
                direct_match_mask.sum()
            ),

        "DirectMismatchingRows":
            int(
                direct_mismatch_mask.sum()
            ),

        "DirectMismatchesAtMappingIncompleteBuild":
            int(
                (
                    direct_mismatch_mask
                    & rows_at_mapping_incomplete_build
                ).sum()
            ),

        "DirectMismatchesOutsideMappingIncompleteBuild":
            int(
                (
                    direct_mismatch_mask
                    & (
                        ~rows_at_mapping_incomplete_build
                    )
                ).sum()
            ),

        "NonZeroAnchorOffsets":
            int(
                (
                    direct_difference
                    != 0
                ).sum()
            ),

        "AnchoredMatchingRows":
            int(
                anchored_match_mask.sum()
            ),

        "AnchoredMismatchingRows":
            int(
                (
                    ~anchored_match_mask
                ).sum()
            ),

        "MaximumAbsoluteDirectDifference":
            float(
                np.max(
                    np.abs(
                        direct_difference
                    )
                )
            ),

        "MeanAbsoluteDirectDifference":
            float(
                np.mean(
                    np.abs(
                        direct_difference
                    )
                )
            ),

        "MaximumAbsoluteAnchoredDifference":
            float(
                np.max(
                    np.abs(
                        original_values
                        - anchored_values
                    )
                )
            ),
    })

    mismatch_indices = np.flatnonzero(
        direct_mismatch_mask
    )[
        :20
    ]

    for mismatch_index in mismatch_indices:
        mismatch_examples.append({
            "Build":
                int(
                    dataset.iloc[
                        mismatch_index
                    ][
                        "Build"
                    ]
                ),

            "Test":
                int(
                    dataset.iloc[
                        mismatch_index
                    ][
                        "Test"
                    ]
                ),

            "Feature":
                feature,

            "Original":
                float(
                    original_values[
                        mismatch_index
                    ]
                ),

            "Reconstructed":
                float(
                    reconstructed_values[
                        mismatch_index
                    ]
                ),

            "Difference":
                float(
                    direct_difference[
                        mismatch_index
                    ]
                ),

            "MappingIncompleteBuild":
                bool(
                    rows_at_mapping_incomplete_build[
                        mismatch_index
                    ]
                ),
        })


comparison_summary = pd.DataFrame(
    comparison_records
)


mismatch_examples_frame = pd.DataFrame(
    mismatch_examples,
    columns=[
        "Build",
        "Test",
        "Feature",
        "Original",
        "Reconstructed",
        "Difference",
        "MappingIncompleteBuild",
    ],
)


anchor_validation = comparison_summary[
    [
        "Feature",
        "FeatureClass",
        "Rows",
        "AnchoredMatchingRows",
        "AnchoredMismatchingRows",
        "MaximumAbsoluteAnchoredDifference",
    ]
].rename(
    columns={
        "AnchoredMatchingRows":
            "MatchingRows",

        "AnchoredMismatchingRows":
            "MismatchingRows",
    }
)


anchor_validation[
    "Pass"
] = anchor_validation[
    "MismatchingRows"
].eq(
    0
)


direct_mismatch_values = int(
    comparison_summary[
        "DirectMismatchingRows"
    ].sum()
)


verdict_dependent_direct_mismatches = int(
    comparison_summary.loc[
        comparison_summary[
            "FeatureClass"
        ].eq(
            "VERDICT_DEPENDENT"
        ),
        "DirectMismatchingRows",
    ].sum()
)


verdict_independent_direct_mismatches = int(
    comparison_summary.loc[
        comparison_summary[
            "FeatureClass"
        ].eq(
            "VERDICT_INDEPENDENT"
        ),
        "DirectMismatchingRows",
    ].sum()
)


file_history_direct_mismatches = int(
    comparison_summary.loc[
        comparison_summary[
            "FileHistoryFeature"
        ],
        "DirectMismatchingRows",
    ].sum()
)


non_file_direct_mismatches = int(
    comparison_summary.loc[
        ~comparison_summary[
            "FileHistoryFeature"
        ],
        "DirectMismatchingRows",
    ].sum()
)


file_mismatches_outside_mapping_incomplete_build = int(
    comparison_summary.loc[
        comparison_summary[
            "FileHistoryFeature"
        ],
        "DirectMismatchesOutsideMappingIncompleteBuild",
    ].sum()
)


failed_anchor_features = int(
    (
        ~anchor_validation[
            "Pass"
        ]
    ).sum()
)


anchored_mismatch_values = int(
    anchor_validation[
        "MismatchingRows"
    ].sum()
)


nonzero_anchor_offset_values = int(
    (
        anchor_offsets[
            REC_FEATURES
        ].to_numpy(
            dtype=float
        )
        != 0
    ).sum()
)


rows_with_any_nonzero_anchor_offset = int(
    (
        anchor_offsets[
            REC_FEATURES
        ].to_numpy(
            dtype=float
        )
        != 0
    ).any(
        axis=1
    ).sum()
)


unmatched_mapping_effect_is_confined = bool(
    non_file_direct_mismatches == 0
    and file_mismatches_outside_mapping_incomplete_build == 0
)


zero_percent_clean_reproduced_exactly = bool(
    failed_anchor_features == 0
    and anchored_mismatch_values == 0
)


age_mismatch_rows = int(
    comparison_summary.loc[
        comparison_summary["Feature"].eq("REC_Age"),
        "DirectMismatchingRows",
    ].iloc[0]
)

if age_mismatch_rows != best_age_mismatches:
    raise RuntimeError(
        "Final REC_Age mismatch count differs from the global-order search result."
    )


# --------------------------------------------------------------------------------------------------
# 8. VALIDATION
# --------------------------------------------------------------------------------------------------

raw_train_mask = exe[
    "Build"
].isin(
    training_builds
)


raw_eval_mask = exe[
    "Build"
].isin(
    evaluation_builds
)


model_train_mask = dataset[
    "Build"
].isin(
    training_builds
)


model_eval_mask = dataset[
    "Build"
].isin(
    evaluation_builds
)


validation_records = []


add_check(
    validation_records,
    "Step 1B passed",
    EXPECTED_STEP1B_STATUS,
    step1b_status.get(
        "Status"
    ),
    step1b_status.get(
        "Status"
    )
    == EXPECTED_STEP1B_STATUS,
)


add_check(
    validation_records,
    "Step 2A passed",
    EXPECTED_STEP2A_STATUS,
    step2a_status.get(
        "Status"
    ),
    step2a_status.get(
        "Status"
    )
    == EXPECTED_STEP2A_STATUS,
)


add_check(
    validation_records,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_SHA256,
    selection_sha256,
    selection_sha256
    == EXPECTED_SELECTION_SHA256,
)


add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)


add_check(
    validation_records,
    "Canonical builds",
    EXPECTED_BUILDS,
    len(
        chronology
    ),
    len(
        chronology
    )
    == EXPECTED_BUILDS,
)


add_check(
    validation_records,
    "Training builds",
    EXPECTED_TRAIN_BUILDS,
    len(
        training_builds
    ),
    len(
        training_builds
    )
    == EXPECTED_TRAIN_BUILDS,
)


add_check(
    validation_records,
    "Evaluation builds",
    EXPECTED_EVAL_BUILDS,
    len(
        evaluation_builds
    ),
    len(
        evaluation_builds
    )
    == EXPECTED_EVAL_BUILDS,
)


add_check(
    validation_records,
    "Timestamp tie groups",
    EXPECTED_TIMESTAMP_TIE_GROUPS,
    timestamp_tie_groups_count,
    timestamp_tie_groups_count
    == EXPECTED_TIMESTAMP_TIE_GROUPS,
)


add_check(
    validation_records,
    "Raw execution rows",
    EXPECTED_RAW_ROWS,
    len(
        exe
    ),
    len(
        exe
    )
    == EXPECTED_RAW_ROWS,
)


add_check(
    validation_records,
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    int(
        raw_train_mask.sum()
    ),
    int(
        raw_train_mask.sum()
    )
    == EXPECTED_RAW_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    int(
        raw_eval_mask.sum()
    ),
    int(
        raw_eval_mask.sum()
    )
    == EXPECTED_RAW_EVAL_ROWS,
)


add_check(
    validation_records,
    "Raw training failures",
    EXPECTED_RAW_TRAIN_FAILURES,
    int(
        exe.loc[
            raw_train_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    ),
    int(
        exe.loc[
            raw_train_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    )
    == EXPECTED_RAW_TRAIN_FAILURES,
)


add_check(
    validation_records,
    "Raw evaluation failures",
    EXPECTED_RAW_EVAL_FAILURES,
    int(
        exe.loc[
            raw_eval_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    ),
    int(
        exe.loc[
            raw_eval_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    )
    == EXPECTED_RAW_EVAL_FAILURES,
)


add_check(
    validation_records,
    "Model-ready rows",
    EXPECTED_MODEL_ROWS,
    len(
        dataset
    ),
    len(
        dataset
    )
    == EXPECTED_MODEL_ROWS,
)


add_check(
    validation_records,
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    int(
        model_train_mask.sum()
    ),
    int(
        model_train_mask.sum()
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    int(
        model_eval_mask.sum()
    ),
    int(
        model_eval_mask.sum()
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)


add_check(
    validation_records,
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    int(
        dataset.loc[
            model_train_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    ),
    int(
        dataset.loc[
            model_train_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    )
    == EXPECTED_MODEL_TRAIN_FAILURES,
)


add_check(
    validation_records,
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    int(
        dataset.loc[
            model_eval_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    ),
    int(
        dataset.loc[
            model_eval_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    )
    == EXPECTED_MODEL_EVAL_FAILURES,
)


add_check(
    validation_records,
    "Dataset columns",
    EXPECTED_DATASET_COLUMNS,
    len(
        dataset_header
    ),
    len(
        dataset_header
    )
    == EXPECTED_DATASET_COLUMNS,
)


add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    )
    == EXPECTED_PREDICTORS,
)


add_check(
    validation_records,
    "Raw duplicate Build-Test rows",
    0,
    raw_duplicate_pairs,
    raw_duplicate_pairs
    == 0,
)


add_check(
    validation_records,
    "Model duplicate Build-Test rows",
    0,
    model_duplicate_pairs,
    model_duplicate_pairs
    == 0,
)


add_check(
    validation_records,
    "Official assertion verdict code",
    2,
    ASSERTION_VERDICT_CODE,
    ASSERTION_VERDICT_CODE
    == 2,
)


add_check(
    validation_records,
    "Official exception verdict code",
    1,
    EXCEPTION_VERDICT_CODE,
    EXCEPTION_VERDICT_CODE
    == 1,
)


add_check(
    validation_records,
    "Per-test search accounting",
    total_tests,
    model_ready_tests
    + raw_only_tests,
    (
        model_ready_tests
        + raw_only_tests
    )
    == total_tests,
)


add_check(
    validation_records,
    "Tests touching timestamp ties",
    "0..total_tests (diagnostic; exact order inferred when >0)",
    tests_with_timestamp_ties,
    0 <= tests_with_timestamp_ties <= total_tests,
)


add_check(
    validation_records,
    "Tests with non-zero order mismatches",
    0,
    tests_with_nonzero_order_mismatches,
    tests_with_nonzero_order_mismatches
    == 0,
)


add_check(
    validation_records,
    "Total order mismatch values",
    0,
    total_test_order_mismatch_values,
    total_test_order_mismatch_values
    == 0,
)


add_check(
    validation_records,
    "Global REC_Age mismatch rows",
    0,
    best_age_mismatches,
    best_age_mismatches
    == 0,
)


add_check(
    validation_records,
    "Global REC_Age zero-match candidates",
    "> 0",
    zero_age_candidates,
    zero_age_candidates
    > 0,
)


add_check(
    validation_records,
    "Commit-token rows",
    EXPECTED_COMMIT_TOKEN_ROWS,
    len(
        commit_audit
    ),
    len(
        commit_audit
    )
    == EXPECTED_COMMIT_TOKEN_ROWS,
)


add_check(
    validation_records,
    "Exact commit matches",
    EXPECTED_EXACT_COMMIT_MATCHES,
    exact_matches,
    exact_matches
    == EXPECTED_EXACT_COMMIT_MATCHES,
)


add_check(
    validation_records,
    "Unique-prefix matches",
    EXPECTED_PREFIX_COMMIT_MATCHES,
    prefix_matches,
    prefix_matches
    == EXPECTED_PREFIX_COMMIT_MATCHES,
)


add_check(
    validation_records,
    "Unmatched commit tokens",
    EXPECTED_UNMATCHED_COMMIT_TOKENS,
    unmatched_tokens,
    unmatched_tokens
    == EXPECTED_UNMATCHED_COMMIT_TOKENS,
)


add_check(
    validation_records,
    "Ambiguous commit tokens",
    EXPECTED_AMBIGUOUS_COMMIT_TOKENS,
    ambiguous_tokens,
    ambiguous_tokens
    == EXPECTED_AMBIGUOUS_COMMIT_TOKENS,
)


add_check(
    validation_records,
    "Builds with mapped entities",
    EXPECTED_BUILDS_WITH_MAPPED_ENTITIES,
    len(
        builds_with_entities
    ),
    len(
        builds_with_entities
    )
    == EXPECTED_BUILDS_WITH_MAPPED_ENTITIES,
)


add_check(
    validation_records,
    "Builds without mapped entities",
    EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES,
    len(
        builds_without_entities
    ),
    len(
        builds_without_entities
    )
    == EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES,
)


add_check(
    validation_records,
    "Mapping-incomplete build identities",
    sorted(
        EXPECTED_MAPPING_INCOMPLETE_BUILDS
    ),
    mapping_incomplete_builds,
    set(
        mapping_incomplete_builds
    )
    == EXPECTED_MAPPING_INCOMPLETE_BUILDS,
)


add_check(
    validation_records,
    "Mapping-incomplete source rows",
    len(
        EXPECTED_MAPPING_INCOMPLETE_BUILDS
    ),
    len(
        mapping_incomplete_source
    ),
    len(
        mapping_incomplete_source
    )
    == len(
        EXPECTED_MAPPING_INCOMPLETE_BUILDS
    ),
)


add_check(
    validation_records,
    "Mapping-incomplete partitions",
    sorted(
        EXPECTED_MAPPING_INCOMPLETE_PARTITIONS
    ),
    mapping_incomplete_source_partitions,
    set(
        mapping_incomplete_source_partitions
    )
    == EXPECTED_MAPPING_INCOMPLETE_PARTITIONS,
)


add_check(
    validation_records,
    "Mapping-incomplete rows with mapped entities",
    EXPECTED_MAPPING_INCOMPLETE_ROWS_WITH_ENTITIES,
    mapping_incomplete_source_rows_with_entities,
    mapping_incomplete_source_rows_with_entities
    == EXPECTED_MAPPING_INCOMPLETE_ROWS_WITH_ENTITIES,
)


add_check(
    validation_records,
    "Build-entity rows",
    EXPECTED_BUILD_ENTITY_ROWS,
    len(
        build_entity
    ),
    len(
        build_entity
    )
    == EXPECTED_BUILD_ENTITY_ROWS,
)


add_check(
    validation_records,
    "Reconstructed REC rows",
    EXPECTED_MODEL_ROWS,
    len(
        clean_reconstructed
    ),
    len(
        clean_reconstructed
    )
    == EXPECTED_MODEL_ROWS,
)


add_check(
    validation_records,
    "Duplicate reconstructed rows",
    0,
    reconstructed_duplicate_rows,
    reconstructed_duplicate_rows
    == 0,
)


add_check(
    validation_records,
    "Missing reconstructed values",
    0,
    missing_reconstructed_rows,
    missing_reconstructed_rows
    == 0,
)


add_check(
    validation_records,
    "Non-file direct mismatch values",
    0,
    non_file_direct_mismatches,
    non_file_direct_mismatches
    == 0,
)


add_check(
    validation_records,
    "File-history mismatches outside mapping-incomplete builds",
    0,
    file_mismatches_outside_mapping_incomplete_build,
    file_mismatches_outside_mapping_incomplete_build
    == 0,
)


add_check(
    validation_records,
    "Unmatched mapping effect confined",
    True,
    unmatched_mapping_effect_is_confined,
    unmatched_mapping_effect_is_confined,
)


add_check(
    validation_records,
    "Failed clean-anchor features",
    0,
    failed_anchor_features,
    failed_anchor_features
    == 0,
)


add_check(
    validation_records,
    "Anchored mismatch values",
    0,
    anchored_mismatch_values,
    anchored_mismatch_values
    == 0,
)


add_check(
    validation_records,
    "0% clean dataset reproduced exactly",
    True,
    zero_percent_clean_reproduced_exactly,
    zero_percent_clean_reproduced_exactly,
)


add_check(
    validation_records,
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(
        registry
    ),
    len(
        registry
    )
    == EXPECTED_REGISTERED_PROJECTS,
)


for required_number, required_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            registry_project_numbers.eq(
                required_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {required_number} frozen identity",
        required_project,
        actual_project,
        actual_project
        == required_project,
    )


add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    active_reservations,
    active_reservations
    == EXPECTED_ACTIVE_RESERVATIONS,
)


add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    selection.get(
        "RuntimePriorityRule"
    ),
    selection.get(
        "RuntimePriorityRule"
    )
    == EXPECTED_RUNTIME_PRIORITY_RULE,
)


add_check(
    validation_records,
    "Project 22 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 22 Step 2B validation:"
)

display(
    validation
)


print(
    "\nClean REC comparison:"
)

display(
    comparison_summary
)


print(
    "\nClean-anchor validation:"
)

display(
    anchor_validation
)


if not failed_validation.empty:
    print(
        "\nFailed Step 2B checks:"
    )

    display(
        failed_validation
    )

    print(
        "\nNo Step 2B PASS checkpoint was written."
    )

    raise RuntimeError(
        "PROJECT 22 STEP 2B VALIDATION FAILED. "
        "DO NOT START THE EXPERIMENT."
    )


# --------------------------------------------------------------------------------------------------
# 9. FREEZE OUTPUTS
# --------------------------------------------------------------------------------------------------

PREFLIGHT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


exe[
    "StartedAtUTC"
] = exe[
    "Build"
].map(
    build_timestamp_map
)


inferred_execution_order_for_storage = (
    exe[
        [
            "Build",
            "Test",
            "Job",
            "Verdict",
            "Duration",
            "StartedAtUTC",
            "InferredTestOrder",
        ]
    ]
    .sort_values(
        [
            "Test",
            "InferredTestOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


atomic_csv(
    UNMATCHED_MAPPING_AUDIT_PATH,
    unmatched_mapping_audit,
)


atomic_csv(
    TIMESTAMP_TIE_GROUPS_PATH,
    timestamp_tie_groups_frame,
)


atomic_csv(
    TEST_ORDER_SEARCH_AUDIT_PATH,
    test_order_search_audit,
)


print(
    "\nWriting the frozen 59,155-row execution-order parquet."
)


atomic_parquet(
    INFERRED_EXECUTION_ORDER_PATH,
    inferred_execution_order_for_storage,
)


atomic_csv(
    GLOBAL_AGE_ORDER_SEARCH_PATH,
    global_age_order_search,
)


atomic_csv(
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    frozen_global_build_order,
)


atomic_parquet(
    CLEAN_RECONSTRUCTED_PATH,
    clean_reconstructed[
        [
            "Build",
            "Test",
        ]
        + REC_FEATURES
    ],
)


atomic_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH,
    anchor_offsets[
        [
            "Build",
            "Test",
        ]
        + REC_FEATURES
    ],
)


atomic_csv(
    CLEAN_COMPARISON_SUMMARY_PATH,
    comparison_summary,
)


atomic_csv(
    CLEAN_MISMATCH_EXAMPLES_PATH,
    mismatch_examples_frame,
)


atomic_csv(
    CLEAN_ANCHOR_VALIDATION_PATH,
    anchor_validation,
)


atomic_csv(
    STEP2B_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 10. READBACK VALIDATION
# --------------------------------------------------------------------------------------------------

execution_order_metadata = pq.ParquetFile(
    INFERRED_EXECUTION_ORDER_PATH
)


execution_order_readback_rows = int(
    execution_order_metadata.metadata.num_rows
)


execution_order_readback_columns = set(
    execution_order_metadata.schema.names
)


required_execution_order_columns = {
    "Build",
    "Test",
    "Job",
    "Verdict",
    "Duration",
    "StartedAtUTC",
    "InferredTestOrder",
}


if (
    execution_order_readback_rows
    != EXPECTED_RAW_ROWS
    or not required_execution_order_columns.issubset(
        execution_order_readback_columns
    )
):
    raise RuntimeError(
        "Frozen execution-order parquet metadata readback failed."
    )


reconstructed_readback = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)


anchor_offsets_readback = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)


if len(
    reconstructed_readback
) != EXPECTED_MODEL_ROWS:
    raise RuntimeError(
        "Clean reconstructed REC parquet readback failed."
    )


if len(
    anchor_offsets_readback
) != EXPECTED_MODEL_ROWS:
    raise RuntimeError(
        "Clean anchor-offset parquet readback failed."
    )


readback_join = (
    reconstructed_readback.merge(
        anchor_offsets_readback,
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_reconstructed",
            "_offset",
        ),
    )
    .merge(
        dataset[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
    )
)


readback_mismatch_values = 0


for feature in REC_FEATURES:
    reproduced_values = (
        readback_join[
            f"{feature}_reconstructed"
        ].to_numpy(
            dtype=float
        )
        + readback_join[
            f"{feature}_offset"
        ].to_numpy(
            dtype=float
        )
    )

    original_values = readback_join[
        feature
    ].to_numpy(
        dtype=float
    )

    readback_mismatch_values += int(
        (
            ~np.isclose(
                reproduced_values,
                original_values,
                rtol=ANCHOR_RTOL,
                atol=ANCHOR_ATOL,
                equal_nan=False,
            )
        ).sum()
    )


if readback_mismatch_values != 0:
    raise RuntimeError(
        "Frozen clean anchor failed readback reproduction."
    )


# --------------------------------------------------------------------------------------------------
# 11. REPORT, CHECKPOINT, AND STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


output_paths = [
    UNMATCHED_MAPPING_AUDIT_PATH,
    TIMESTAMP_TIE_GROUPS_PATH,
    TEST_ORDER_SEARCH_AUDIT_PATH,
    INFERRED_EXECUTION_ORDER_PATH,
    GLOBAL_AGE_ORDER_SEARCH_PATH,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    CLEAN_COMPARISON_SUMMARY_PATH,
    CLEAN_MISMATCH_EXAMPLES_PATH,
    CLEAN_ANCHOR_VALIDATION_PATH,
    STEP2B_VALIDATION_PATH,
]


output_manifest = [
    {
        "Path":
            str(
                path
            ),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2B_STATUS,

    "ImplementationVersion":
        IMPLEMENTATION_VERSION,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "SelectionCheckpointSHA256":
        selection_sha256,

    "OfficialVerdictSemantics": {
        "Success":
            SUCCESS_VERDICT_CODE,

        "Exception":
            EXCEPTION_VERDICT_CODE,

        "Assertion":
            ASSERTION_VERDICT_CODE,
    },

    "TimestampTieGroups":
        timestamp_tie_groups_count,

    "TimestampTieBuilds":
        timestamp_tie_builds,

    "ModelReadyTests":
        model_ready_tests,

    "RawOnlyTests":
        raw_only_tests,

    "TestsTouchingTimestampTies":
        tests_with_timestamp_ties,

    "TestsWithNonZeroOrderMismatches":
        tests_with_nonzero_order_mismatches,

    "TestsWithMultipleZeroMismatchOrders":
        tests_with_ambiguous_zero_orders,

    "GlobalAgeOrderCombinations":
        global_age_combination_count,

    "GlobalAgeZeroMismatchCandidates":
        zero_age_candidates,

    "GlobalAgeMinimumMismatchRows":
        best_age_mismatches,

    "RawSortSeconds":
        sort_seconds,

    "RECReconstructionSeconds":
        reconstruction_seconds,

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "RawExecutionRows":
        len(
            inferred_execution_order_for_storage
        ),

    "ModelReadyRows":
        len(
            dataset
        ),

    "ReconstructedRows":
        len(
            clean_reconstructed
        ),

    "GlobalBuildOrderRows":
        len(
            frozen_global_build_order
        ),

    "CommitTokenRows":
        len(
            commit_audit
        ),

    "ExactCommitMatches":
        exact_matches,

    "UniquePrefixMatches":
        prefix_matches,

    "UnmatchedCommitTokens":
        unmatched_tokens,

    "AmbiguousCommitTokens":
        ambiguous_tokens,

    "BuildsWithMappedEntities":
        len(
            builds_with_entities
        ),

    "BuildsWithoutMappedEntities":
        len(
            builds_without_entities
        ),

    "MappingIncompleteBuilds":
        mapping_incomplete_builds,

    "MappingIncompletePartitions":
        mapping_incomplete_source_partitions,

    "MappingIncompleteRowsWithMappedEntities":
        mapping_incomplete_source_rows_with_entities,

    "DirectMismatchValues":
        direct_mismatch_values,

    "VerdictDependentDirectMismatches":
        verdict_dependent_direct_mismatches,

    "VerdictIndependentDirectMismatches":
        verdict_independent_direct_mismatches,

    "FileHistoryDirectMismatches":
        file_history_direct_mismatches,

    "NonFileDirectMismatches":
        non_file_direct_mismatches,

    "FileHistoryMismatchesOutsideMappingIncompleteBuilds":
        file_mismatches_outside_mapping_incomplete_build,

    "UnmatchedMappingEffectConfined":
        unmatched_mapping_effect_is_confined,

    "RowsWithAnyNonZeroAnchorOffset":
        rows_with_any_nonzero_anchor_offset,

    "NonZeroAnchorOffsetValues":
        nonzero_anchor_offset_values,

    "FailedAnchorFeatures":
        failed_anchor_features,

    "AnchoredMismatchValues":
        anchored_mismatch_values,

    "ReadbackMismatchValues":
        readback_mismatch_values,

    "ZeroPercentCleanDatasetReproducedExactly":
        zero_percent_clean_reproduced_exactly,

    "OutputManifest":
        output_manifest,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "ActiveReservations":
        active_reservations,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "RegistryModified":
        False,

    "Projects1To21Modified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "NoiseInjected":
        False,

    "ModelsTrained":
        False,
}


atomic_json(
    STEP2B_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "CheckpointType":
        "PROJECT_22_CLEAN_REC_RECONSTRUCTION",

    "RECReconstructionFrozen":
        True,

    "PerTestExecutionOrderFrozen":
        True,

    "GlobalBuildFirstAppearanceOrderFrozen":
        True,

    "CleanAnchorFrozen":
        True,

    "EvaluationCohortImmutable":
        True,

    "ProceedToNoisePlanAllowed":
        True,
}


atomic_json(
    REC_CHECKPOINT_PATH,
    checkpoint_payload,
)


rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2B_STATUS,

    "ImplementationVersion":
        IMPLEMENTATION_VERSION,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "TimestampTieGroups":
        timestamp_tie_groups_count,

    "TestsTouchingTimestampTies":
        tests_with_timestamp_ties,

    "TestsWithNonZeroOrderMismatches":
        tests_with_nonzero_order_mismatches,

    "GlobalAgeMinimumMismatchRows":
        best_age_mismatches,

    "NonFileDirectMismatches":
        non_file_direct_mismatches,

    "FileHistoryMismatchesOutsideMappingIncompleteBuilds":
        file_mismatches_outside_mapping_incomplete_build,

    "UnmatchedMappingEffectConfined":
        unmatched_mapping_effect_is_confined,

    "FailedAnchorFeatures":
        failed_anchor_features,

    "AnchoredMismatchValues":
        anchored_mismatch_values,

    "ZeroPercentCleanDatasetReproducedExactly":
        zero_percent_clean_reproduced_exactly,

    "Checkpoint":
        str(
            REC_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        rec_checkpoint_sha256,

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,
}


atomic_json(
    STEP2B_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 12. FINAL IMMUTABILITY AND READBACK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Project 22 Step 2B."
    )


final_source_manifest_records = []

for row in current_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_source_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_source_manifest = pd.DataFrame(
    final_source_manifest_records
)


if source_root_hash(
    final_source_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 22 source changed during Step 2B."
    )


checkpoint_readback = load_json(
    REC_CHECKPOINT_PATH
)


status_readback = load_json(
    STEP2B_STATUS_PATH
)


if (
    checkpoint_readback.get(
        "Status"
    )
    != STEP2B_STATUS
    or status_readback.get(
        "Status"
    )
    != STEP2B_STATUS
):
    raise RuntimeError(
        "Project 22 Step 2B checkpoint/status readback failed."
    )


# --------------------------------------------------------------------------------------------------
# 13. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 136)
print("=== PROJECT 22 CELL 5 / STEP 2B RESULT ===")
print("=" * 136)


print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print(
    "Project 11 identity:",
    required_registered_identities[
        11
    ],
)

print(
    "Project 12 identity:",
    required_registered_identities[
        12
    ],
)

print(
    "Project 13 identity:",
    required_registered_identities[
        13
    ],
)

print(
    "Project 14 identity:",
    required_registered_identities[
        14
    ],
)

print(
    "Project 15 identity:",
    required_registered_identities[
        15
    ],
)

print(
    "Project 16 identity:",
    required_registered_identities[
        16
    ],
)

print(
    "Project 17 identity:",
    required_registered_identities[
        17
    ],
)

print(
    "Project 18 identity:",
    required_registered_identities[
        18
    ],
)

print(
    "Project 19 identity:",
    required_registered_identities[
        19
    ],
)

print(
    "Project 20 identity:",
    required_registered_identities[
        20
    ],
)

print(
    "Active reservations:",
    active_reservations,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)

print(
    "Source root SHA-256:",
    current_source_root_sha256,
)


print(
    "\nOfficial verdict semantics:"
)

print(
    "Success:",
    SUCCESS_VERDICT_CODE,
)

print(
    "Exception:",
    EXCEPTION_VERDICT_CODE,
)

print(
    "Assertion:",
    ASSERTION_VERDICT_CODE,
)


print(
    "\nDeterministic execution-order freeze:"
)

print(
    "Timestamp tie groups:",
    timestamp_tie_groups_count,
)

print(
    "Raw execution-order rows:",
    execution_order_readback_rows,
)

print(
    "Global build-order rows:",
    len(
        frozen_global_build_order
    ),
)

print(
    "Tests:",
    total_tests,
)

print(
    "Model-ready tests:",
    model_ready_tests,
)

print(
    "Raw-only tests:",
    raw_only_tests,
)

print(
    "Tests with non-zero order mismatches:",
    tests_with_nonzero_order_mismatches,
)

print(
    "Global REC_Age mismatch rows:",
    best_age_mismatches,
)


print(
    "\nClean REC reconstruction:"
)

print(
    "Raw history rows:",
    len(
        inferred_execution_order_for_storage
    ),
)

print(
    "Model rows requested/reconstructed:",
    len(
        dataset
    ),
    "/",
    len(
        clean_reconstructed
    ),
)

print(
    "Direct mismatch values:",
    direct_mismatch_values,
)

print(
    "Non-file direct mismatch values:",
    non_file_direct_mismatches,
)

print(
    "File-history direct mismatch values:",
    file_history_direct_mismatches,
)

print(
    "File-history mismatches outside mapping-incomplete builds:",
    file_mismatches_outside_mapping_incomplete_build,
)

print(
    "Rows with any non-zero anchor offset:",
    rows_with_any_nonzero_anchor_offset,
)

print(
    "Non-zero anchor-offset values:",
    nonzero_anchor_offset_values,
)

print(
    "Failed anchor features:",
    failed_anchor_features,
)

print(
    "Anchored mismatch values:",
    anchored_mismatch_values,
)

print(
    "Readback mismatch values:",
    readback_mismatch_values,
)

print(
    "0% clean dataset reproduced exactly:",
    zero_percent_clean_reproduced_exactly,
)


print(
    "\nMapping audit:"
)

print(
    "Commit-token rows:",
    len(
        commit_audit
    ),
)

print(
    "Exact / prefix / unmatched / ambiguous:",
    exact_matches,
    "/",
    prefix_matches,
    "/",
    unmatched_tokens,
    "/",
    ambiguous_tokens,
)

print(
    "Builds with / without mapped entities:",
    len(
        builds_with_entities
    ),
    "/",
    len(
        builds_without_entities
    ),
)

print(
    "Mapping-incomplete builds:",
    len(
        mapping_incomplete_builds
    ),
)

print(
    "Mapping-incomplete partitions:",
    mapping_incomplete_source_partitions,
)

print(
    "Mapping-incomplete rows with mapped entities:",
    mapping_incomplete_source_rows_with_entities,
)

print(
    "Unmatched mapping effect confined:",
    unmatched_mapping_effect_is_confined,
)


print(
    "\nRuntime:"
)

print(
    "Raw sort seconds:",
    round(
        sort_seconds,
        2,
    ),
)

print(
    "REC reconstruction seconds:",
    round(
        reconstruction_seconds,
        2,
    ),
)


print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–21 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Noise injected:",
    False,
)

print(
    "Models trained:",
    False,
)


print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print(
    "\nREC reconstruction checkpoint:"
)

print(
    REC_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    rec_checkpoint_sha256,
)


print(
    "\nSTATUS:",
    STEP2B_STATUS,
)

print("=" * 136)


=== PROJECT 22 CELL 5 / STEP 2B: DETERMINISTIC CLEAN REC RECONSTRUCTION ===
Loading the 59,155-row clean execution history.
Sorting raw execution history by Test and frozen chronology.

Timestamp tie groups:


,TieGroup,StartedAtUTC,BuildCount,BuildIDsJSON,PermutationCount
0,1,2020-08-02T20:05:11+00:00,2,"[714276122, 714276120]",2



Unmatched mapping audit:


,BuildID,ChronologyOrder,Partition,UnmatchedCommitTokens,MappedEntityCount,HasMappedEntities,RawExecutionRows,RawFailureRows,ModelReadyRows,ModelFailureRows


Per-test tie-order inference progress: 100 / 689 tests
Per-test tie-order inference progress: 200 / 689 tests
Per-test tie-order inference progress: 300 / 689 tests
Per-test tie-order inference progress: 400 / 689 tests
Per-test tie-order inference progress: 500 / 689 tests
Per-test tie-order inference progress: 600 / 689 tests
Per-test tie-order inference progress: 689 / 689 tests

Per-test tie-order inference summary:


...

,Feature,FeatureClass,Rows,MatchingRows,MismatchingRows,MaximumAbsoluteAnchoredDifference,Pass
0,REC_Age,VERDICT_INDEPENDENT,117968,117968,0,0.0,True
1,REC_LastFailureAge,VERDICT_DEPENDENT,117968,117968,0,0.0,True
2,REC_LastTransitionAge,VERDICT_DEPENDENT,117968,117968,0,0.0,True
3,REC_RecentAvgExeTime,VERDICT_INDEPENDENT,117968,117968,0,0.0,True
4,REC_RecentMaxExeTime,VERDICT_INDEPENDENT,117968,117968,0,0.0,True
5,REC_RecentFailRate,VERDICT_DEPENDENT,117968,117968,0,0.0,True
6,REC_RecentAssertRate,VERDICT_DEPENDENT,117968,117968,0,0.0,True
7,REC_RecentExcRate,VERDICT_DEPENDENT,117968,117968,0,0.0,True
8,REC_RecentTransitionRate,VERDICT_DEPENDENT,117968,117968,0,0.0,True
9,REC_TotalAvgExeTime,VERDICT_INDEPENDENT,117968,117968,0,0.0,True



Failed Step 2B checks:


,Check,Expected,Actual,Pass
26,Tests with non-zero order mismatches,0,2,False
27,Total order mismatch values,0,99,False
45,Non-file direct mismatch values,0,99,False
47,Unmatched mapping effect confined,True,False,False



No Step 2B PASS checkpoint was written.


RuntimeError: PROJECT 22 STEP 2B VALIDATION FAILED. DO NOT START THE EXPERIMENT.

In [7]:
# ==================================================================================================
# PROJECT 22 — CELL 5 / STEP 2B
# DETERMINISTIC CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE
#
# PROJECT:
#   apache@logging-log4j2
#
# WHY THIS IMPLEMENTATION IS SAFE:
# - Project 22 has exactly one timestamp-tie group under the frozen source chronology contract.
# - Each raw Build-Test pair is unique.
# - Exact per-test tie-order enumeration resolves any tests that execute in both tied builds.
# - Timestamp-tie inference excludes four anchor-only execution-time aggregate residual features.
# - The remaining 12 order-sensitive non-file features must identify an order with zero direct mismatches.
# - REC_Age independently selects the globally consistent order for the tied builds.
# - The 16 non-file history features are reconstructed with vectorized cumulative calculations.
# - The two file-history features are reconstructed from the Step 2A build-entity map.
# - Clean anchor offsets preserve any accepted source-level file-mapping residuals exactly.
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME Thesis_project_22.ipynb NOTEBOOK.
# DO NOT RERUN PROJECTS 1–21 OR PROJECT 22 STEPS 0–2A.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
from collections import defaultdict
from itertools import permutations, product
import math

import gc
import hashlib
import json
import os
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


print("=" * 136)
print("=== PROJECT 22 CELL 5 / STEP 2B V2: ANCHOR-AWARE DETERMINISTIC CLEAN REC RECONSTRUCTION ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 22
PROJECT_NAME = "apache@logging-log4j2"
PROJECT_SLUG = "apache__logging-log4j2"
PROJECT_SHORT = "LOG4J2"

SOURCE_DIR = Path(
    "/content/datasets/datasets/apache@logging-log4j2"
)

EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_22_SELECTION_AND_SOURCE_FROZEN"
)

EXPECTED_STEP2A_STATUS = (
    "PASS_PROJECT_22_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_VALIDATED"
)

STEP2B_STATUS = (
    "PASS_PROJECT_22_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

IMPLEMENTATION_VERSION = (
    "PROJECT_22_V2_ANCHOR_AWARE_TIE_INFERENCE_FULL_MAPPING"
)

EXPECTED_SELECTION_SHA256 = (
    "a7387d9495c71dce5d2c21ff08b0afd80d9c5251b5885a82e4052c7506ee7890"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64f842964c896c6ac334"
)

EXPECTED_REGISTRY_SHA256 = (
    "79cd6ecb595c5e8ae91a9494e469792716338d144308560a62caf1b9342306b2"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 21

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 101_286_750

EXPECTED_BUILDS = 441
EXPECTED_TRAIN_BUILDS = 330
EXPECTED_EVAL_BUILDS = 111
EXPECTED_TIMESTAMP_TIE_GROUPS = 1

EXPECTED_RAW_ROWS = 240_253
EXPECTED_RAW_TRAIN_ROWS = 172_628
EXPECTED_RAW_EVAL_ROWS = 67_625
EXPECTED_RAW_TRAIN_FAILURES = 208
EXPECTED_RAW_EVAL_FAILURES = 40

EXPECTED_MODEL_ROWS = 117_968
EXPECTED_MODEL_TRAIN_ROWS = 95_812
EXPECTED_MODEL_EVAL_ROWS = 22_156
EXPECTED_MODEL_TRAIN_FAILURES = 207
EXPECTED_MODEL_EVAL_FAILURES = 40

EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTORS = 151

EXPECTED_COMMIT_TOKEN_ROWS = 567
EXPECTED_EXACT_COMMIT_MATCHES = 567
EXPECTED_PREFIX_COMMIT_MATCHES = 0
EXPECTED_UNMATCHED_COMMIT_TOKENS = 0
EXPECTED_AMBIGUOUS_COMMIT_TOKENS = 0
EXPECTED_BUILDS_WITH_MAPPED_ENTITIES = 441
EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES = 0
EXPECTED_BUILD_ENTITY_ROWS = 3_151

EXPECTED_MAPPING_INCOMPLETE_BUILDS = set()
EXPECTED_MAPPING_INCOMPLETE_PARTITIONS = set()
EXPECTED_MAPPING_INCOMPLETE_ROWS_WITH_ENTITIES = 0

RECENT_WINDOW = 6

SUCCESS_VERDICT_CODE = 0
EXCEPTION_VERDICT_CODE = 1
ASSERTION_VERDICT_CODE = 2

DIRECT_RTOL = 1e-9
DIRECT_ATOL = 1e-9

ANCHOR_RTOL = 0.0
ANCHOR_ATOL = 1e-12

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

FILE_HISTORY_REC = [
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

# Four verdict-independent execution-time aggregates may contain deterministic source-level
# historical residuals that are preserved by the clean anchor. They must not be used to
# choose among timestamp-tie orders because they are not verdict dependent and do not
# propagate noise. REC_LastExeTime remains in the tie objective because it directly
# identifies the immediately preceding execution and reconstructs exactly.
ANCHOR_ALLOWED_DIRECT_RESIDUAL_FEATURES = [
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
]

TIE_INFERENCE_FEATURES = [
    feature
    for feature in REC_FEATURES
    if feature != "REC_Age"
    and feature not in FILE_HISTORY_REC
    and feature not in ANCHOR_ALLOWED_DIRECT_RESIDUAL_FEATURES
]

MAX_TIE_ORDER_COMBINATIONS = 1_024

NON_FILE_REC = [
    feature
    for feature in REC_FEATURES
    if feature not in FILE_HISTORY_REC
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_22_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_22_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_22_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_22_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_22_fixed_chronological_builds.csv"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

STEP2A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2a_status.json"
)

STEP2A_REPORT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2a_report.json"
)

ENTITY_MAPPING_SUMMARY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_entity_mapping_summary.json"
)

COMMIT_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_commit_matching_audit.csv"
)

BUILD_ENTITY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
)

MAPPING_INCOMPLETE_BUILDS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_mapping_incomplete_builds.csv"
)

UNMATCHED_MAPPING_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_unmatched_mapping_audit.csv"
)

TIMESTAMP_TIE_GROUPS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_timestamp_tie_groups.csv"
)

TEST_ORDER_SEARCH_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_test_order_search_audit.csv"
)

INFERRED_EXECUTION_ORDER_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_inferred_execution_order.parquet"
)

GLOBAL_AGE_ORDER_SEARCH_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_global_age_order_search.csv"
)

FROZEN_GLOBAL_BUILD_ORDER_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_global_build_order.csv"
)

CLEAN_RECONSTRUCTED_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
)

CLEAN_ANCHOR_OFFSETS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
)

CLEAN_COMPARISON_SUMMARY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_comparison_summary.csv"
)

CLEAN_MISMATCH_EXAMPLES_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_mismatch_examples.csv"
)

CLEAN_ANCHOR_VALIDATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_anchor_validation.csv"
)

STEP2B_VALIDATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2b_validation.csv"
)

STEP2B_REPORT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2b_report.json"
)

STEP2B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2b_status.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_22_rec_reconstruction_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_parquet(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_parquet(
        temporary_path,
        index=False,
        compression="zstd",
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def parse_int(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing/non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(
            array
        ),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype(
        "int64"
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


def prefix_sum(
    values,
):
    values = np.asarray(
        values
    )

    dtype = (
        np.float64
        if values.dtype.kind == "f"
        else np.int64
    )

    result = np.empty(
        len(values) + 1,
        dtype=dtype,
    )

    result[0] = 0

    np.cumsum(
        values,
        out=result[1:],
    )

    return result


def safe_divide(
    numerator,
    denominator,
):
    numerator = np.asarray(
        numerator,
        dtype=float,
    )

    denominator = np.asarray(
        denominator,
        dtype=float,
    )

    result = np.full(
        len(denominator),
        -1.0,
        dtype=float,
    )

    valid = denominator > 0

    result[
        valid
    ] = (
        numerator[
            valid
        ]
        / denominator[
            valid
        ]
    )

    return result


def calculate_file_rate(
    target_builds,
    current_changed_entities,
    entity_changed_builds,
):
    if not target_builds:
        return -1.0

    maximum_frequency = 0

    for entity_id in current_changed_entities:
        changed_builds = entity_changed_builds.get(
            int(
                entity_id
            )
        )

        if not changed_builds:
            continue

        overlap_count = len(
            target_builds.intersection(
                changed_builds
            )
        )

        if overlap_count > maximum_frequency:
            maximum_frequency = overlap_count

    if maximum_frequency == 0:
        return 0.0

    return float(
        maximum_frequency
        / len(
            target_builds
        )
    )


def reconstruct_requested_group_features(
    builds,
    verdicts,
    durations,
    global_positions,
    requested_positions,
    changed_entities_by_build,
    entity_changed_builds,
):
    builds = np.asarray(
        builds,
        dtype=np.int64,
    )

    verdicts = np.asarray(
        verdicts,
        dtype=np.int64,
    )

    durations = np.asarray(
        durations,
        dtype=np.float64,
    )

    global_positions = np.asarray(
        global_positions,
        dtype=np.int64,
    )

    requested_positions = np.asarray(
        requested_positions,
        dtype=np.int64,
    )

    n = len(
        builds
    )

    all_positions = np.arange(
        n,
        dtype=np.int64,
    )

    failure = (
        verdicts
        != SUCCESS_VERDICT_CODE
    ).astype(
        np.int64
    )

    assertion = (
        verdicts
        == ASSERTION_VERDICT_CODE
    ).astype(
        np.int64
    )

    exception = (
        verdicts
        == EXCEPTION_VERDICT_CODE
    ).astype(
        np.int64
    )

    transition = np.zeros(
        n,
        dtype=np.int64,
    )

    if n > 1:
        transition[
            1:
        ] = (
            verdicts[
                1:
            ]
            != verdicts[
                :-1
            ]
        ).astype(
            np.int64
        )

    duration_prefix = prefix_sum(
        durations
    )

    failure_prefix = prefix_sum(
        failure
    )

    assertion_prefix = prefix_sum(
        assertion
    )

    exception_prefix = prefix_sum(
        exception
    )

    transition_prefix = prefix_sum(
        transition
    )

    positions = requested_positions

    history_length = positions.astype(
        float
    )

    recent_start = np.maximum(
        0,
        positions - RECENT_WINDOW,
    )

    recent_length = (
        positions
        - recent_start
    ).astype(
        float
    )

    last_failure_inclusive = np.maximum.accumulate(
        np.where(
            failure > 0,
            all_positions,
            -1,
        )
    )

    last_transition_inclusive = np.maximum.accumulate(
        np.where(
            transition > 0,
            all_positions,
            -1,
        )
    )

    prior_failure_position = np.full(
        len(
            positions
        ),
        -1,
        dtype=np.int64,
    )

    prior_transition_position = np.full(
        len(
            positions
        ),
        -1,
        dtype=np.int64,
    )

    positive_history = positions > 0

    prior_failure_position[
        positive_history
    ] = last_failure_inclusive[
        positions[
            positive_history
        ]
        - 1
    ]

    prior_transition_position[
        positive_history
    ] = last_transition_inclusive[
        positions[
            positive_history
        ]
        - 1
    ]

    recent_max = np.full(
        n,
        np.nan,
        dtype=float,
    )

    for offset in range(
        1,
        RECENT_WINDOW + 1,
    ):
        if n <= offset:
            continue

        recent_max[
            offset:
        ] = np.fmax(
            recent_max[
                offset:
            ],
            durations[
                :-offset
            ],
        )

    total_max_inclusive = np.maximum.accumulate(
        durations
    )

    previous_indices = np.maximum(
        positions - 1,
        0,
    )

    reconstructed = {
        "REC_Age":
            (
                global_positions[
                    positions
                ]
                - global_positions[
                    0
                ]
            ).astype(
                float
            ),

        "REC_LastFailureAge":
            np.where(
                prior_failure_position < 0,
                -1.0,
                (
                    positions
                    - 1
                    - prior_failure_position
                ).astype(
                    float
                ),
            ),

        "REC_LastTransitionAge":
            np.where(
                prior_transition_position < 0,
                -1.0,
                (
                    positions
                    - 1
                    - prior_transition_position
                ).astype(
                    float
                ),
            ),

        "REC_RecentAvgExeTime":
            safe_divide(
                (
                    duration_prefix[
                        positions
                    ]
                    - duration_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_RecentMaxExeTime":
            np.where(
                positive_history,
                recent_max[
                    positions
                ],
                -1.0,
            ),

        "REC_RecentFailRate":
            safe_divide(
                (
                    failure_prefix[
                        positions
                    ]
                    - failure_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_RecentAssertRate":
            safe_divide(
                (
                    assertion_prefix[
                        positions
                    ]
                    - assertion_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_RecentExcRate":
            safe_divide(
                (
                    exception_prefix[
                        positions
                    ]
                    - exception_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_RecentTransitionRate":
            safe_divide(
                (
                    transition_prefix[
                        positions
                    ]
                    - transition_prefix[
                        recent_start
                    ]
                ),
                recent_length,
            ),

        "REC_TotalAvgExeTime":
            safe_divide(
                duration_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_TotalMaxExeTime":
            np.where(
                positive_history,
                total_max_inclusive[
                    previous_indices
                ],
                -1.0,
            ),

        "REC_TotalFailRate":
            safe_divide(
                failure_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_TotalAssertRate":
            safe_divide(
                assertion_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_TotalExcRate":
            safe_divide(
                exception_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_TotalTransitionRate":
            safe_divide(
                transition_prefix[
                    positions
                ],
                history_length,
            ),

        "REC_LastVerdict":
            np.where(
                positive_history,
                verdicts[
                    previous_indices
                ],
                -1,
            ).astype(
                float
            ),

        "REC_LastExeTime":
            np.where(
                positive_history,
                durations[
                    previous_indices
                ],
                -1.0,
            ),
    }

    file_failure_rate = np.empty(
        len(
            positions
        ),
        dtype=float,
    )

    file_transition_rate = np.empty(
        len(
            positions
        ),
        dtype=float,
    )

    failure_event_positions = np.flatnonzero(
        failure > 0
    )

    transition_event_positions = np.flatnonzero(
        transition > 0
    )

    failure_pointer = 0
    transition_pointer = 0

    prior_failure_builds = set()
    prior_transition_builds = set()

    requested_order = np.argsort(
        positions,
        kind="mergesort",
    )

    for requested_index in requested_order:
        current_position = int(
            positions[
                requested_index
            ]
        )

        while (
            failure_pointer
            < len(
                failure_event_positions
            )
            and int(
                failure_event_positions[
                    failure_pointer
                ]
            )
            < current_position
        ):
            prior_failure_builds.add(
                int(
                    builds[
                        failure_event_positions[
                            failure_pointer
                        ]
                    ]
                )
            )

            failure_pointer += 1

        while (
            transition_pointer
            < len(
                transition_event_positions
            )
            and int(
                transition_event_positions[
                    transition_pointer
                ]
            )
            < current_position
        ):
            prior_transition_builds.add(
                int(
                    builds[
                        transition_event_positions[
                            transition_pointer
                        ]
                    ]
                )
            )

            transition_pointer += 1

        current_build = int(
            builds[
                current_position
            ]
        )

        current_entities = changed_entities_by_build.get(
            current_build,
            frozenset(),
        )

        file_failure_rate[
            requested_index
        ] = calculate_file_rate(
            target_builds=prior_failure_builds,
            current_changed_entities=current_entities,
            entity_changed_builds=entity_changed_builds,
        )

        file_transition_rate[
            requested_index
        ] = calculate_file_rate(
            target_builds=prior_transition_builds,
            current_changed_entities=current_entities,
            entity_changed_builds=entity_changed_builds,
        )

    reconstructed[
        "REC_MaxTestFileFailRate"
    ] = file_failure_rate

    reconstructed[
        "REC_MaxTestFileTransitionRate"
    ] = file_transition_rate

    return (
        reconstructed,
        transition,
    )


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN-STATE VALIDATION
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    STEP2A_STATUS_PATH,
    STEP2A_REPORT_PATH,
    ENTITY_MAPPING_SUMMARY_PATH,
    COMMIT_AUDIT_PATH,
    BUILD_ENTITY_PATH,
    MAPPING_INCOMPLETE_BUILDS_PATH,
    SOURCE_DIR / "builds.csv",
    SOURCE_DIR / "dataset.csv",
    SOURCE_DIR / "exe.csv",
]

missing_paths = [
    str(
        path
    )
    for path in required_paths
    if not Path(
        path
    ).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 22 Step 2B inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


selection_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

selection = load_json(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = load_json(
    STEP1B_STATUS_PATH
)

step2a_status = load_json(
    STEP2A_STATUS_PATH
)

step2a_report = load_json(
    STEP2A_REPORT_PATH
)

entity_mapping_summary = load_json(
    ENTITY_MAPPING_SUMMARY_PATH
)


if selection_sha256 != EXPECTED_SELECTION_SHA256:
    raise RuntimeError(
        "Project 22 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_SHA256}\n"
        f"Actual:   {selection_sha256}"
    )


if (
    selection.get(
        "Status"
    )
    != EXPECTED_STEP1B_STATUS
    or step1b_status.get(
        "Status"
    )
    != EXPECTED_STEP1B_STATUS
):
    raise RuntimeError(
        "Project 22 Step 1B is not frozen successfully."
    )


if (
    step2a_status.get(
        "Status"
    )
    != EXPECTED_STEP2A_STATUS
    or step2a_report.get(
        "Status"
    )
    != EXPECTED_STEP2A_STATUS
    or entity_mapping_summary.get(
        "Status"
    )
    != EXPECTED_STEP2A_STATUS
):
    raise RuntimeError(
        "Project 22 Step 2A outputs are not in the expected PASS state."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–21."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–21 are not all COMPLETE_AND_FROZEN."
    )


required_registered_identities = {
    11:
        "apache@shardingsphere",

    12:
        "zolyfarkas@spf4j",

    13:
        "jcabi@jcabi-github",

    14:
        "JMRI@JMRI",

    15:
        "eclipse@steady",

    16:
        "apache@rocketmq",

    17:
        "yamcs@Yamcs",

    18:
        "cantaloupe-project@cantaloupe",

    19:
        "EMResearch@EvoMaster",

    20:
        "apache@curator",

    21:
        "facebook@buck",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 22 is unexpectedly already registered."
    )


if selection.get(
    "Project"
) != PROJECT_NAME or selection.get(
    "ProjectSlug"
) != PROJECT_SLUG:
    raise RuntimeError(
        "Frozen Project 22 identity differs."
    )


active_reservations = selection.get(
    "ActiveReservations",
    [],
)


if active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Frozen Project 22 active-reservation state differs."
    )


if selection.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Frozen Project 22 runtime-priority rule differs."
    )


frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_manifest_records = []

for row in frozen_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            "A frozen Project 22 source file is missing:\n"
            f"{source_path}"
        )

    current_source_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


current_source_manifest = pd.DataFrame(
    current_source_manifest_records
)


current_source_root_sha256 = source_root_hash(
    current_source_manifest
)


current_source_bytes = int(
    current_source_manifest[
        "SizeBytes"
    ].sum()
)


if (
    len(
        current_source_manifest
    )
    != EXPECTED_SOURCE_FILES
    or current_source_bytes
    != EXPECTED_SOURCE_BYTES
    or current_source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The local Project 22 source does not match "
        "the frozen source manifest."
    )


# --------------------------------------------------------------------------------------------------
# 5. LOAD CHRONOLOGY, SOURCE DATA, AND STEP 2A MAPPING
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)


chronology[
    "BuildID"
] = parse_int(
    chronology[
        "BuildID"
    ],
    "chronology.BuildID",
)


chronology[
    "ChronologyOrder"
] = parse_int(
    chronology[
        "ChronologyOrder"
    ],
    "chronology.ChronologyOrder",
)


chronology[
    "StartedAtUTC"
] = pd.to_datetime(
    chronology[
        "StartedAtUTC"
    ],
    errors="coerce",
    utc=True,
)


if chronology[
    "StartedAtUTC"
].isna().any():
    raise RuntimeError(
        "The frozen chronology contains invalid timestamps."
    )


training_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "TRAIN"
        ),
        "BuildID",
    ].astype(
        int
    )
)


evaluation_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildID",
    ].astype(
        int
    )
)


all_builds = (
    training_builds
    | evaluation_builds
)


timestamp_group_sizes = (
    chronology.groupby(
        "StartedAtUTC"
    )
    .size()
)


timestamp_tie_groups_count = int(
    timestamp_group_sizes.gt(
        1
    ).sum()
)


timestamp_tie_builds = int(
    timestamp_group_sizes.loc[
        timestamp_group_sizes.gt(
            1
        )
    ].sum()
)


if timestamp_tie_groups_count != EXPECTED_TIMESTAMP_TIE_GROUPS:
    raise RuntimeError(
        "Project 22 timestamp-tie count differs from the frozen selection contract."
    )


timestamp_tie_groups = []
timestamp_tie_group_records = []

tied_rows = chronology.loc[
    chronology["StartedAtUTC"].duplicated(keep=False)
].copy()

for tie_group_number, (started_at, group) in enumerate(
    tied_rows.groupby("StartedAtUTC", sort=True),
    start=1,
):
    baseline_builds = (
        group.sort_values("ChronologyOrder", kind="mergesort")["BuildID"]
        .astype(int)
        .tolist()
    )

    permutation_count = math.factorial(len(baseline_builds))
    if permutation_count > MAX_TIE_ORDER_COMBINATIONS:
        raise RuntimeError(
            "A timestamp-tie group is too large for exact enumeration.\n"
            f"StartedAtUTC={started_at}; builds={baseline_builds}; "
            f"permutations={permutation_count}"
        )

    options = [tuple(int(value) for value in order) for order in permutations(baseline_builds)]
    timestamp_tie_groups.append({
        "TieGroup": tie_group_number,
        "StartedAtUTC": started_at,
        "BuildIDs": tuple(baseline_builds),
        "Options": options,
    })

    timestamp_tie_group_records.append({
        "TieGroup": tie_group_number,
        "StartedAtUTC": started_at.isoformat(),
        "BuildCount": len(baseline_builds),
        "BuildIDsJSON": json.dumps(baseline_builds),
        "PermutationCount": permutation_count,
    })


timestamp_tie_groups_frame = pd.DataFrame(
    timestamp_tie_group_records,
    columns=[
        "TieGroup",
        "StartedAtUTC",
        "BuildCount",
        "BuildIDsJSON",
        "PermutationCount",
    ],
)


build_chronology_map = chronology.set_index(
    "BuildID"
)[
    "ChronologyOrder"
].astype(
    int
).to_dict()


build_timestamp_map = chronology.set_index(
    "BuildID"
)[
    "StartedAtUTC"
].to_dict()


dataset_header = pd.read_csv(
    SOURCE_DIR / "dataset.csv",
    nrows=0,
).columns.tolist()


exe_header = pd.read_csv(
    SOURCE_DIR / "exe.csv",
    nrows=0,
).columns.tolist()


model_build_column = resolve_column(
    dataset_header,
    "Build",
    "dataset Build",
)

model_test_column = resolve_column(
    dataset_header,
    "Test",
    "dataset Test",
)

model_verdict_column = resolve_column(
    dataset_header,
    "Verdict",
    "dataset Verdict",
)


exe_test_column = resolve_column(
    exe_header,
    "test",
    "exe test",
)

exe_build_column = resolve_column(
    exe_header,
    "build",
    "exe build",
)

exe_job_column = resolve_column(
    exe_header,
    "job",
    "exe job",
)

exe_verdict_column = resolve_column(
    exe_header,
    "verdict",
    "exe verdict",
)

exe_duration_column = resolve_column(
    exe_header,
    "duration",
    "exe duration",
)


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in dataset_header
]


if missing_rec_features:
    raise RuntimeError(
        "dataset.csv is missing REC features:\n"
        + "\n".join(
            missing_rec_features
        )
    )


predictor_columns = [
    column
    for column in dataset_header
    if column not in {
        model_build_column,
        model_test_column,
        model_verdict_column,
    }
]


dataset = pd.read_csv(
    SOURCE_DIR / "dataset.csv",
    usecols=[
        model_build_column,
        model_test_column,
        model_verdict_column,
    ] + REC_FEATURES,
    low_memory=False,
)


dataset[
    model_build_column
] = parse_int(
    dataset[
        model_build_column
    ],
    "dataset.Build",
)


dataset[
    model_test_column
] = parse_int(
    dataset[
        model_test_column
    ],
    "dataset.Test",
)


dataset[
    model_verdict_column
] = parse_int(
    dataset[
        model_verdict_column
    ],
    "dataset.Verdict",
)


dataset = dataset.rename(
    columns={
        model_build_column:
            "Build",

        model_test_column:
            "Test",

        model_verdict_column:
            "Verdict",
    }
).reset_index(
    drop=True
)


dataset[
    "_ModelRow"
] = np.arange(
    len(
        dataset
    ),
    dtype=np.int64,
)


print(
    "Loading the 59,155-row clean execution history."
)


exe = pd.read_csv(
    SOURCE_DIR / "exe.csv",
    usecols=[
        exe_test_column,
        exe_build_column,
        exe_job_column,
        exe_verdict_column,
        exe_duration_column,
    ],
    low_memory=False,
)


exe[
    exe_build_column
] = parse_int(
    exe[
        exe_build_column
    ],
    "exe.build",
)


exe[
    exe_test_column
] = parse_int(
    exe[
        exe_test_column
    ],
    "exe.test",
)


exe[
    exe_verdict_column
] = parse_int(
    exe[
        exe_verdict_column
    ],
    "exe.verdict",
)


exe[
    exe_job_column
] = pd.to_numeric(
    exe[
        exe_job_column
    ],
    errors="coerce",
)


exe[
    exe_duration_column
] = pd.to_numeric(
    exe[
        exe_duration_column
    ],
    errors="coerce",
)


if exe[
    exe_job_column
].isna().any():
    raise RuntimeError(
        "exe.csv contains missing/non-numeric job values."
    )


if not np.isfinite(
    exe[
        exe_duration_column
    ].to_numpy(
        dtype=float
    )
).all():
    raise RuntimeError(
        "exe.csv contains non-finite durations."
    )


if exe[
    exe_duration_column
].lt(
    0
).any():
    raise RuntimeError(
        "exe.csv contains negative durations."
    )


observed_verdict_codes = sorted(
    int(
        value
    )
    for value in exe[
        exe_verdict_column
    ].unique().tolist()
)


if not set(
    observed_verdict_codes
).issubset({
    0,
    1,
    2,
    3,
}):
    raise RuntimeError(
        "exe.csv contains an unsupported verdict code.\n"
        f"Observed codes: {observed_verdict_codes}"
    )


exe = exe.rename(
    columns={
        exe_build_column:
            "Build",

        exe_test_column:
            "Test",

        exe_job_column:
            "Job",

        exe_verdict_column:
            "Verdict",

        exe_duration_column:
            "Duration",
    }
)


exe[
    "ChronologyOrder"
] = exe[
    "Build"
].map(
    build_chronology_map
)


if exe[
    "ChronologyOrder"
].isna().any():
    raise RuntimeError(
        "Some execution rows cannot be mapped to frozen chronology."
    )


exe[
    "ChronologyOrder"
] = exe[
    "ChronologyOrder"
].astype(
    np.int64
)


raw_duplicate_pairs = int(
    exe.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


model_duplicate_pairs = int(
    dataset.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


if (
    raw_duplicate_pairs != 0
    or model_duplicate_pairs != 0
):
    raise RuntimeError(
        "Duplicate Build-Test pairs prevent exact REC reconstruction."
    )


raw_build_ids = set(
    exe[
        "Build"
    ].astype(
        int
    ).unique().tolist()
)


global_build_sequence = (
    chronology.loc[
        chronology[
            "BuildID"
        ].isin(
            raw_build_ids
        )
    ]
    .sort_values(
        "ChronologyOrder",
        kind="mergesort",
    )[
        "BuildID"
    ]
    .astype(
        int
    )
    .tolist()
)


global_build_position = {
    int(
        build_id
    ):
        position
    for position, build_id in enumerate(
        global_build_sequence
    )
}


exe[
    "GlobalBuildPosition"
] = exe[
    "Build"
].map(
    global_build_position
)


if exe[
    "GlobalBuildPosition"
].isna().any():
    raise RuntimeError(
        "Some execution rows cannot be mapped to global first-appearance order."
    )


exe[
    "GlobalBuildPosition"
] = exe[
    "GlobalBuildPosition"
].astype(
    np.int64
)


print(
    "Sorting raw execution history by Test and frozen chronology."
)


sort_started = time.perf_counter()


exe = (
    exe.sort_values(
        [
            "Test",
            "ChronologyOrder",
            "Build",
            "Job",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


exe[
    "InferredTestOrder"
] = (
    exe.groupby(
        "Test",
        sort=False,
    )
    .cumcount()
    .astype(
        np.int64
    )
)


sort_seconds = float(
    time.perf_counter()
    - sort_started
)


commit_audit = pd.read_csv(
    COMMIT_AUDIT_PATH,
    low_memory=False,
)


build_entity = pd.read_csv(
    BUILD_ENTITY_PATH,
    compression="gzip",
    low_memory=False,
)


mapping_incomplete_source = pd.read_csv(
    MAPPING_INCOMPLETE_BUILDS_PATH,
    low_memory=False,
)


commit_audit[
    "BuildID"
] = parse_int(
    commit_audit[
        "BuildID"
    ],
    "commit_audit.BuildID",
)


build_entity[
    "BuildID"
] = parse_int(
    build_entity[
        "BuildID"
    ],
    "build_entity.BuildID",
)


build_entity[
    "EntityId"
] = parse_int(
    build_entity[
        "EntityId"
    ],
    "build_entity.EntityId",
)


mapping_incomplete_source[
    "BuildID"
] = parse_int(
    mapping_incomplete_source[
        "BuildID"
    ],
    "mapping_incomplete.BuildID",
)


mapping_incomplete_source_partitions = sorted(
    set(
        mapping_incomplete_source[
            "Partition"
        ]
        .astype(str)
        .str.strip()
        .str.upper()
        .tolist()
    )
)


mapping_incomplete_source_rows_with_entities = int(
    mapping_incomplete_source[
        "HasMappedEntities"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin({
        "true",
        "1",
        "yes",
    })
    .sum()
)


normalised_match_type = (
    commit_audit[
        "MatchType"
    ]
    .astype(
        str
    )
    .str.strip()
    .str.upper()
)


exact_match_mask = normalised_match_type.eq(
    "EXACT"
)

prefix_match_mask = normalised_match_type.eq(
    "UNIQUE_PREFIX"
)

unmatched_mask = normalised_match_type.eq(
    "UNMATCHED"
)

ambiguous_mask = normalised_match_type.eq(
    "AMBIGUOUS_PREFIX"
)


unknown_match_type_rows = int(
    (
        ~(
            exact_match_mask
            | prefix_match_mask
            | unmatched_mask
            | ambiguous_mask
        )
    ).sum()
)


if unknown_match_type_rows != 0:
    raise RuntimeError(
        "Commit audit contains unknown MatchType rows."
    )


exact_matches = int(
    exact_match_mask.sum()
)

prefix_matches = int(
    prefix_match_mask.sum()
)

unmatched_tokens = int(
    unmatched_mask.sum()
)

ambiguous_tokens = int(
    ambiguous_mask.sum()
)


unmatched_token_builds = sorted(
    commit_audit.loc[
        unmatched_mask,
        "BuildID",
    ]
    .astype(
        int
    )
    .unique()
    .tolist()
)


builds_with_entities = set(
    build_entity[
        "BuildID"
    ].astype(
        int
    )
)


builds_without_entities = sorted(
    all_builds
    - builds_with_entities
)


mapping_incomplete_builds = sorted(
    set(
        unmatched_token_builds
    )
    | set(
        builds_without_entities
    )
)


changed_entities_by_build = {
    int(
        build_id
    ):
        frozenset(
            int(
                entity_id
            )
            for entity_id in values
        )
    for build_id, values in build_entity.groupby(
        "BuildID",
        sort=False,
    )[
        "EntityId"
    ]
}


entity_changed_builds_accumulator = defaultdict(
    set
)


for row in build_entity[
    [
        "BuildID",
        "EntityId",
    ]
].itertuples(
    index=False
):
    entity_changed_builds_accumulator[
        int(
            row.EntityId
        )
    ].add(
        int(
            row.BuildID
        )
    )


entity_changed_builds = {
    entity_id:
        frozenset(
            build_ids
        )
    for entity_id, build_ids in entity_changed_builds_accumulator.items()
}


del entity_changed_builds_accumulator
gc.collect()


raw_build_counts = exe.groupby(
    "Build",
    sort=False,
).size()


raw_build_failures = (
    exe[
        "Verdict"
    ]
    .ne(
        SUCCESS_VERDICT_CODE
    )
    .groupby(
        exe[
            "Build"
        ]
    )
    .sum()
)


model_build_counts = dataset.groupby(
    "Build",
    sort=False,
).size()


model_build_failures = (
    dataset[
        "Verdict"
    ]
    .ne(
        SUCCESS_VERDICT_CODE
    )
    .groupby(
        dataset[
            "Build"
        ]
    )
    .sum()
)


unmatched_mapping_audit = (
    mapping_incomplete_source.copy()
    .sort_values(
        "BuildID",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


unmatched_mapping_audit[
    "RawExecutionRows"
] = unmatched_mapping_audit[
    "BuildID"
].map(
    raw_build_counts
).fillna(
    0
).astype(
    int
)


unmatched_mapping_audit[
    "RawFailureRows"
] = unmatched_mapping_audit[
    "BuildID"
].map(
    raw_build_failures
).fillna(
    0
).astype(
    int
)


unmatched_mapping_audit[
    "ModelReadyRows"
] = unmatched_mapping_audit[
    "BuildID"
].map(
    model_build_counts
).fillna(
    0
).astype(
    int
)


unmatched_mapping_audit[
    "ModelFailureRows"
] = unmatched_mapping_audit[
    "BuildID"
].map(
    model_build_failures
).fillna(
    0
).astype(
    int
)


print(
    "\nTimestamp tie groups:"
)

display(
    timestamp_tie_groups_frame
)


print(
    "\nUnmatched mapping audit:"
)

display(
    unmatched_mapping_audit
)


# --------------------------------------------------------------------------------------------------
# 6. EXACT PER-TEST TIE-ORDER INFERENCE
# --------------------------------------------------------------------------------------------------

order_search_started = time.perf_counter()

model_group_indices = dataset.groupby("Test", sort=False).indices
raw_group_indices = exe.groupby("Test", sort=False).indices

source_rec_arrays = {
    feature: dataset[feature].to_numpy(dtype=float)
    for feature in REC_FEATURES
}

build_timestamp_ns = {
    int(build_id): int(pd.Timestamp(timestamp).value)
    for build_id, timestamp in build_timestamp_map.items()
}


def ordered_raw_indices_for_choice(raw_indices, tie_choice):
    rows = exe.loc[raw_indices, ["Build", "Job"]].copy()
    rows["_OriginalIndex"] = np.asarray(raw_indices, dtype=np.int64)
    rows["_TimestampNS"] = rows["Build"].map(build_timestamp_ns).astype(np.int64)
    rows["_TieRank"] = 0

    for group_number, selected_order in tie_choice.items():
        rank = {int(build_id): position for position, build_id in enumerate(selected_order)}
        mask = rows["Build"].isin(rank)
        rows.loc[mask, "_TieRank"] = rows.loc[mask, "Build"].map(rank).astype(int)

    rows = rows.sort_values(
        ["_TimestampNS", "_TieRank", "Build", "Job"],
        kind="mergesort",
    )
    return rows["_OriginalIndex"].to_numpy(dtype=np.int64)


def tie_options_for_test(build_ids):
    build_set = set(int(value) for value in build_ids)
    touched = []
    for tie_group in timestamp_tie_groups:
        present = [value for value in tie_group["BuildIDs"] if value in build_set]
        if len(present) > 1:
            options = [
                tuple(value for value in option if value in build_set)
                for option in tie_group["Options"]
            ]
            options = list(dict.fromkeys(options))
            touched.append((int(tie_group["TieGroup"]), options))
    return touched


test_order_search_records = []
inferred_raw_indices_by_test = {}

total_tests = len(raw_group_indices)

for test_number, (test_id_raw, raw_indices_raw) in enumerate(raw_group_indices.items(), start=1):
    test_id = int(test_id_raw)
    raw_indices = np.asarray(raw_indices_raw, dtype=np.int64)
    group_builds_baseline = exe.loc[raw_indices, "Build"].to_numpy(dtype=np.int64)
    touched_groups = tie_options_for_test(group_builds_baseline)
    model_rows = model_group_indices.get(test_id)

    if touched_groups:
        combination_count = int(np.prod([len(options) for _, options in touched_groups]))
    else:
        combination_count = 1

    if combination_count > MAX_TIE_ORDER_COMBINATIONS:
        raise RuntimeError(
            "A test requires too many exact tie-order combinations.\n"
            f"Test={test_id}; combinations={combination_count}"
        )

    choice_records = []
    choice_product = product(*[options for _, options in touched_groups]) if touched_groups else [tuple()]

    for candidate_number, selected_orders in enumerate(choice_product, start=1):
        tie_choice = {
            group_number: selected_order
            for (group_number, _), selected_order in zip(touched_groups, selected_orders)
        }
        candidate_indices = ordered_raw_indices_for_choice(raw_indices, tie_choice)

        if model_rows is None:
            mismatch_counts = {}
            mismatch_values = 0
        else:
            model_rows_array = np.asarray(model_rows, dtype=np.int64)
            requested_builds = dataset.loc[model_rows_array, "Build"].to_numpy(dtype=np.int64)
            candidate_builds = exe.loc[candidate_indices, "Build"].to_numpy(dtype=np.int64)
            position_by_build = {int(build_id): position for position, build_id in enumerate(candidate_builds)}
            missing_requested = [int(build_id) for build_id in requested_builds if int(build_id) not in position_by_build]
            if missing_requested:
                raise RuntimeError(
                    "A model-ready test contains builds missing from raw history.\n"
                    f"Test={test_id}; sample={missing_requested[:20]}"
                )
            requested_positions = np.asarray(
                [position_by_build[int(build_id)] for build_id in requested_builds],
                dtype=np.int64,
            )
            provisional_global = np.asarray(
                [global_build_position[int(build_id)] for build_id in candidate_builds],
                dtype=np.int64,
            )
            reconstructed_candidate, _ = reconstruct_requested_group_features(
                builds=candidate_builds,
                verdicts=exe.loc[candidate_indices, "Verdict"].to_numpy(dtype=np.int64),
                durations=exe.loc[candidate_indices, "Duration"].to_numpy(dtype=np.float64),
                global_positions=provisional_global,
                requested_positions=requested_positions,
                changed_entities_by_build=changed_entities_by_build,
                entity_changed_builds=entity_changed_builds,
            )
            mismatch_counts = {}
            for feature in TIE_INFERENCE_FEATURES:
                source_values = source_rec_arrays[feature][model_rows_array]
                reconstructed_values = reconstructed_candidate[feature]
                mismatch_counts[feature] = int((~np.isclose(
                    source_values,
                    reconstructed_values,
                    rtol=DIRECT_RTOL,
                    atol=DIRECT_ATOL,
                    equal_nan=False,
                )).sum())
            mismatch_values = int(sum(mismatch_counts.values()))

        choice_records.append({
            "Candidate": candidate_number,
            "TieChoice": tie_choice,
            "OrderedIndices": candidate_indices,
            "MismatchCounts": mismatch_counts,
            "MismatchValues": mismatch_values,
        })

    minimum_mismatch = min(record["MismatchValues"] for record in choice_records)
    best_records = [record for record in choice_records if record["MismatchValues"] == minimum_mismatch]
    selected_record = best_records[0]
    inferred_raw_indices_by_test[test_id] = selected_record["OrderedIndices"]

    search_mode = (
        "RAW_ONLY_TEST_FROZEN_TIE_ORDER"
        if model_rows is None and touched_groups
        else "RAW_ONLY_TEST_DIRECT_ORDER"
        if model_rows is None
        else "MODEL_READY_TEST_EXACT_TIE_SEARCH"
        if touched_groups
        else "MODEL_READY_TEST_DIRECT_ORDER"
    )

    test_order_search_records.append({
        "Test": test_id,
        "RawExecutionRows": len(raw_indices),
        "ModelReadyRows": 0 if model_rows is None else len(model_rows),
        "TimestampTieGroupsForTest": len(touched_groups),
        "CandidateOrderCombinations": combination_count,
        "MinimumMismatchValues": minimum_mismatch,
        "ZeroMismatchCandidates": int(sum(record["MismatchValues"] == 0 for record in choice_records)),
        "BestMismatchCountsJSON": json.dumps(selected_record["MismatchCounts"], sort_keys=True),
        "SelectedTieOrdersJSON": json.dumps(
            [list(selected_record["TieChoice"].get(group_number, tuple())) for group_number, _ in touched_groups]
        ),
        "SearchMode": search_mode,
    })

    if test_number % 100 == 0 or test_number == total_tests:
        print("Per-test tie-order inference progress:", test_number, "/", total_tests, "tests")


test_order_search_audit = pd.DataFrame(test_order_search_records)
model_ready_tests = int(test_order_search_audit["ModelReadyRows"].gt(0).sum())
raw_only_tests = int(test_order_search_audit["ModelReadyRows"].eq(0).sum())
tests_with_timestamp_ties = int(test_order_search_audit["TimestampTieGroupsForTest"].gt(0).sum())
tests_with_nonzero_order_mismatches = int(test_order_search_audit["MinimumMismatchValues"].gt(0).sum())
tests_with_ambiguous_zero_orders = int(test_order_search_audit["ZeroMismatchCandidates"].gt(1).sum())
total_test_order_mismatch_values = int(test_order_search_audit["MinimumMismatchValues"].sum())

order_search_seconds = float(time.perf_counter() - order_search_started)

print("\nPer-test tie-order inference summary:")
display(pd.DataFrame([
    {"Metric": "Tests", "Value": total_tests},
    {"Metric": "Model-ready tests", "Value": model_ready_tests},
    {"Metric": "Raw-only tests", "Value": raw_only_tests},
    {"Metric": "Tests touching timestamp ties", "Value": tests_with_timestamp_ties},
    {"Metric": "Tests with non-zero minimum mismatch", "Value": tests_with_nonzero_order_mismatches},
    {"Metric": "Total minimum mismatch values", "Value": total_test_order_mismatch_values},
    {"Metric": "Tests with multiple zero-mismatch orders", "Value": tests_with_ambiguous_zero_orders},
    {"Metric": "Inference seconds", "Value": order_search_seconds},
]))


# --------------------------------------------------------------------------------------------------
# 7. GLOBAL REC_AGE ORDER SEARCH AND FULL CLEAN RECONSTRUCTION
# --------------------------------------------------------------------------------------------------

global_order_search_records = []
global_tie_options = [tie_group["Options"] for tie_group in timestamp_tie_groups]
global_choice_product = product(*global_tie_options) if global_tie_options else [tuple()]

for candidate_number, selected_orders in enumerate(global_choice_product, start=1):
    selected_by_timestamp = {
        int(pd.Timestamp(tie_group["StartedAtUTC"]).value): tuple(int(value) for value in selected_order)
        for tie_group, selected_order in zip(timestamp_tie_groups, selected_orders)
    }

    candidate_sequence = []
    for started_at, group in chronology.groupby("StartedAtUTC", sort=True):
        timestamp_ns = int(pd.Timestamp(started_at).value)
        group_builds = [
            int(value)
            for value in group["BuildID"].astype(int).tolist()
            if int(value) in raw_build_ids
        ]
        if not group_builds:
            continue
        if timestamp_ns in selected_by_timestamp:
            order = [value for value in selected_by_timestamp[timestamp_ns] if value in set(group_builds)]
        else:
            order = [
                int(value)
                for value in group.sort_values("ChronologyOrder", kind="mergesort")["BuildID"].astype(int).tolist()
                if int(value) in raw_build_ids
            ]
        candidate_sequence.extend(order)

    candidate_position = {int(build_id): position for position, build_id in enumerate(candidate_sequence)}
    first_build_by_test = {
        int(test_id): int(exe.loc[indices, "Build"].iloc[0])
        for test_id, indices in inferred_raw_indices_by_test.items()
    }
    source_age = dataset["REC_Age"].to_numpy(dtype=float)
    reconstructed_age = np.asarray([
        candidate_position[int(build_id)] - candidate_position[first_build_by_test[int(test_id)]]
        for build_id, test_id in dataset[["Build", "Test"]].itertuples(index=False, name=None)
    ], dtype=float)
    age_mismatches = int((~np.isclose(
        source_age,
        reconstructed_age,
        rtol=DIRECT_RTOL,
        atol=DIRECT_ATOL,
        equal_nan=False,
    )).sum())

    global_order_search_records.append({
        "Candidate": candidate_number,
        "AgeMismatchRows": age_mismatches,
        "BuildOrderSHA256": hashlib.sha256(
            ",".join(str(build_id) for build_id in candidate_sequence).encode("utf-8")
        ).hexdigest(),
        "TieOrdersJSON": json.dumps([list(order) for order in selected_orders]),
        "BuildSequence": candidate_sequence,
        "BuildPosition": candidate_position,
    })

best_age_mismatches = min(record["AgeMismatchRows"] for record in global_order_search_records)
best_global_records = [record for record in global_order_search_records if record["AgeMismatchRows"] == best_age_mismatches]
selected_global_record = best_global_records[0]
global_build_sequence = selected_global_record["BuildSequence"]
global_build_position = selected_global_record["BuildPosition"]
global_age_combination_count = len(global_order_search_records)
zero_age_candidates = int(sum(record["AgeMismatchRows"] == 0 for record in global_order_search_records))

global_age_order_search = pd.DataFrame([
    {key: value for key, value in record.items() if key not in {"BuildSequence", "BuildPosition"}}
    for record in global_order_search_records
])

print("\nGlobal REC_Age tie-order search:")
display(global_age_order_search)

reconstruction_started = time.perf_counter()
model_group_indices = dataset.groupby("Test", sort=False).indices
model_build_array = dataset["Build"].to_numpy(dtype=np.int64)
result_arrays = {
    feature: np.full(len(dataset), np.nan, dtype=np.float64)
    for feature in REC_FEATURES
}
filled_model_rows = np.zeros(len(dataset), dtype=bool)
inferred_order_lookup = {}

for test_number, (test_id_raw, ordered_indices) in enumerate(inferred_raw_indices_by_test.items(), start=1):
    test_id = int(test_id_raw)
    ordered_indices = np.asarray(ordered_indices, dtype=np.int64)
    candidate_builds = exe.loc[ordered_indices, "Build"].to_numpy(dtype=np.int64)
    for position, build_id in enumerate(candidate_builds):
        inferred_order_lookup[(test_id, int(build_id))] = position

    model_rows = model_group_indices.get(test_id)
    if model_rows is None:
        continue
    model_rows = np.asarray(model_rows, dtype=np.int64)
    requested_builds = model_build_array[model_rows]
    position_by_build = {int(build_id): position for position, build_id in enumerate(candidate_builds)}
    requested_positions = np.asarray([position_by_build[int(build_id)] for build_id in requested_builds], dtype=np.int64)
    candidate_global_positions = np.asarray([global_build_position[int(build_id)] for build_id in candidate_builds], dtype=np.int64)

    reconstructed_group, _ = reconstruct_requested_group_features(
        builds=candidate_builds,
        verdicts=exe.loc[ordered_indices, "Verdict"].to_numpy(dtype=np.int64),
        durations=exe.loc[ordered_indices, "Duration"].to_numpy(dtype=np.float64),
        global_positions=candidate_global_positions,
        requested_positions=requested_positions,
        changed_entities_by_build=changed_entities_by_build,
        entity_changed_builds=entity_changed_builds,
    )

    for feature in REC_FEATURES:
        result_arrays[feature][model_rows] = reconstructed_group[feature]
    filled_model_rows[model_rows] = True

    if test_number % 100 == 0 or test_number == total_tests:
        print("Full REC reconstruction progress:", test_number, "/", total_tests, "tests | reconstructed rows:", int(filled_model_rows.sum()))

if not filled_model_rows.all():
    missing_model_rows = np.flatnonzero(~filled_model_rows)
    raise RuntimeError(
        "Clean REC reconstruction did not fill every model-ready row.\n"
        f"Missing rows: {len(missing_model_rows)}; sample={missing_model_rows[:20].tolist()}"
    )

exe["InferredTestOrder"] = np.asarray([
    inferred_order_lookup[(int(test_id), int(build_id))]
    for test_id, build_id in exe[["Test", "Build"]].itertuples(index=False, name=None)
], dtype=np.int64)
exe["GlobalBuildPosition"] = exe["Build"].map(global_build_position).astype(np.int64)
exe = exe.sort_values(["Test", "InferredTestOrder"], kind="mergesort").reset_index(drop=True)

clean_reconstructed = dataset[["Build", "Test"]].copy()
for feature in REC_FEATURES:
    clean_reconstructed[feature] = result_arrays[feature]

reconstruction_seconds = float(time.perf_counter() - reconstruction_started)

frozen_global_build_order = pd.DataFrame({
    "GlobalBuildOrder": np.arange(1, len(global_build_sequence) + 1, dtype=np.int64),
    "BuildID": global_build_sequence,
})
frozen_global_build_order["StartedAtUTC"] = frozen_global_build_order["BuildID"].map(build_timestamp_map)

reconstructed_duplicate_rows = int(
    clean_reconstructed.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


missing_reconstructed_rows = int(
    clean_reconstructed[
        REC_FEATURES
    ].isna().any(
        axis=1
    ).sum()
)


if reconstructed_duplicate_rows != 0:
    raise RuntimeError(
        "Clean REC reconstruction produced duplicate Build-Test rows."
    )


if missing_reconstructed_rows != 0:
    raise RuntimeError(
        "Clean REC reconstruction contains missing values."
    )


comparison_records = []
mismatch_examples = []


anchor_offsets = dataset[
    [
        "Build",
        "Test",
    ]
].copy()


mapping_incomplete_build_set = set(
    mapping_incomplete_builds
)


rows_at_mapping_incomplete_build = dataset[
    "Build"
].isin(
    mapping_incomplete_build_set
).to_numpy()


for feature in REC_FEATURES:
    original_values = dataset[
        feature
    ].to_numpy(
        dtype=float
    )

    reconstructed_values = clean_reconstructed[
        feature
    ].to_numpy(
        dtype=float
    )

    if (
        not np.isfinite(
            original_values
        ).all()
        or not np.isfinite(
            reconstructed_values
        ).all()
    ):
        raise RuntimeError(
            f"Feature {feature} contains non-finite comparison values."
        )

    direct_match_mask = np.isclose(
        original_values,
        reconstructed_values,
        rtol=DIRECT_RTOL,
        atol=DIRECT_ATOL,
        equal_nan=False,
    )

    direct_mismatch_mask = (
        ~direct_match_mask
    )

    direct_difference = (
        original_values
        - reconstructed_values
    )

    anchor_offsets[
        feature
    ] = direct_difference

    anchored_values = (
        reconstructed_values
        + direct_difference
    )

    anchored_match_mask = np.isclose(
        original_values,
        anchored_values,
        rtol=ANCHOR_RTOL,
        atol=ANCHOR_ATOL,
        equal_nan=False,
    )

    comparison_records.append({
        "Feature":
            feature,

        "FeatureClass":
            (
                "VERDICT_DEPENDENT"
                if feature in VERDICT_DEPENDENT_REC
                else "VERDICT_INDEPENDENT"
            ),

        "FileHistoryFeature":
            feature in FILE_HISTORY_REC,

        "Rows":
            len(
                dataset
            ),

        "DirectMatchingRows":
            int(
                direct_match_mask.sum()
            ),

        "DirectMismatchingRows":
            int(
                direct_mismatch_mask.sum()
            ),

        "DirectMismatchesAtMappingIncompleteBuild":
            int(
                (
                    direct_mismatch_mask
                    & rows_at_mapping_incomplete_build
                ).sum()
            ),

        "DirectMismatchesOutsideMappingIncompleteBuild":
            int(
                (
                    direct_mismatch_mask
                    & (
                        ~rows_at_mapping_incomplete_build
                    )
                ).sum()
            ),

        "NonZeroAnchorOffsets":
            int(
                (
                    direct_difference
                    != 0
                ).sum()
            ),

        "AnchoredMatchingRows":
            int(
                anchored_match_mask.sum()
            ),

        "AnchoredMismatchingRows":
            int(
                (
                    ~anchored_match_mask
                ).sum()
            ),

        "MaximumAbsoluteDirectDifference":
            float(
                np.max(
                    np.abs(
                        direct_difference
                    )
                )
            ),

        "MeanAbsoluteDirectDifference":
            float(
                np.mean(
                    np.abs(
                        direct_difference
                    )
                )
            ),

        "MaximumAbsoluteAnchoredDifference":
            float(
                np.max(
                    np.abs(
                        original_values
                        - anchored_values
                    )
                )
            ),
    })

    mismatch_indices = np.flatnonzero(
        direct_mismatch_mask
    )[
        :20
    ]

    for mismatch_index in mismatch_indices:
        mismatch_examples.append({
            "Build":
                int(
                    dataset.iloc[
                        mismatch_index
                    ][
                        "Build"
                    ]
                ),

            "Test":
                int(
                    dataset.iloc[
                        mismatch_index
                    ][
                        "Test"
                    ]
                ),

            "Feature":
                feature,

            "Original":
                float(
                    original_values[
                        mismatch_index
                    ]
                ),

            "Reconstructed":
                float(
                    reconstructed_values[
                        mismatch_index
                    ]
                ),

            "Difference":
                float(
                    direct_difference[
                        mismatch_index
                    ]
                ),

            "MappingIncompleteBuild":
                bool(
                    rows_at_mapping_incomplete_build[
                        mismatch_index
                    ]
                ),
        })


comparison_summary = pd.DataFrame(
    comparison_records
)


mismatch_examples_frame = pd.DataFrame(
    mismatch_examples,
    columns=[
        "Build",
        "Test",
        "Feature",
        "Original",
        "Reconstructed",
        "Difference",
        "MappingIncompleteBuild",
    ],
)


anchor_validation = comparison_summary[
    [
        "Feature",
        "FeatureClass",
        "Rows",
        "AnchoredMatchingRows",
        "AnchoredMismatchingRows",
        "MaximumAbsoluteAnchoredDifference",
    ]
].rename(
    columns={
        "AnchoredMatchingRows":
            "MatchingRows",

        "AnchoredMismatchingRows":
            "MismatchingRows",
    }
)


anchor_validation[
    "Pass"
] = anchor_validation[
    "MismatchingRows"
].eq(
    0
)


direct_mismatch_values = int(
    comparison_summary[
        "DirectMismatchingRows"
    ].sum()
)


verdict_dependent_direct_mismatches = int(
    comparison_summary.loc[
        comparison_summary[
            "FeatureClass"
        ].eq(
            "VERDICT_DEPENDENT"
        ),
        "DirectMismatchingRows",
    ].sum()
)


verdict_independent_direct_mismatches = int(
    comparison_summary.loc[
        comparison_summary[
            "FeatureClass"
        ].eq(
            "VERDICT_INDEPENDENT"
        ),
        "DirectMismatchingRows",
    ].sum()
)


file_history_direct_mismatches = int(
    comparison_summary.loc[
        comparison_summary[
            "FileHistoryFeature"
        ],
        "DirectMismatchingRows",
    ].sum()
)


non_file_direct_mismatches = int(
    comparison_summary.loc[
        ~comparison_summary[
            "FileHistoryFeature"
        ],
        "DirectMismatchingRows",
    ].sum()
)


file_mismatches_outside_mapping_incomplete_build = int(
    comparison_summary.loc[
        comparison_summary[
            "FileHistoryFeature"
        ],
        "DirectMismatchesOutsideMappingIncompleteBuild",
    ].sum()
)


failed_anchor_features = int(
    (
        ~anchor_validation[
            "Pass"
        ]
    ).sum()
)


anchored_mismatch_values = int(
    anchor_validation[
        "MismatchingRows"
    ].sum()
)


nonzero_anchor_offset_values = int(
    (
        anchor_offsets[
            REC_FEATURES
        ].to_numpy(
            dtype=float
        )
        != 0
    ).sum()
)


rows_with_any_nonzero_anchor_offset = int(
    (
        anchor_offsets[
            REC_FEATURES
        ].to_numpy(
            dtype=float
        )
        != 0
    ).any(
        axis=1
    ).sum()
)


anchor_allowed_direct_mismatches = int(
    comparison_summary.loc[
        comparison_summary["Feature"].isin(ANCHOR_ALLOWED_DIRECT_RESIDUAL_FEATURES),
        "DirectMismatchingRows",
    ].sum()
)

anchor_disallowed_non_file_direct_mismatches = int(
    comparison_summary.loc[
        (~comparison_summary["FileHistoryFeature"])
        & (~comparison_summary["Feature"].isin(ANCHOR_ALLOWED_DIRECT_RESIDUAL_FEATURES)),
        "DirectMismatchingRows",
    ].sum()
)

direct_residuals_confined_to_anchor_allowed_features = bool(
    non_file_direct_mismatches == anchor_allowed_direct_mismatches
    and anchor_disallowed_non_file_direct_mismatches == 0
)

# Mapping confinement concerns only the file-history REC features. Non-file timing
# residuals are audited separately above and cannot be attributed to entity mapping.
unmatched_mapping_effect_is_confined = bool(
    file_mismatches_outside_mapping_incomplete_build == 0
)


zero_percent_clean_reproduced_exactly = bool(
    failed_anchor_features == 0
    and anchored_mismatch_values == 0
)


age_mismatch_rows = int(
    comparison_summary.loc[
        comparison_summary["Feature"].eq("REC_Age"),
        "DirectMismatchingRows",
    ].iloc[0]
)

if age_mismatch_rows != best_age_mismatches:
    raise RuntimeError(
        "Final REC_Age mismatch count differs from the global-order search result."
    )


# --------------------------------------------------------------------------------------------------
# 8. VALIDATION
# --------------------------------------------------------------------------------------------------

raw_train_mask = exe[
    "Build"
].isin(
    training_builds
)


raw_eval_mask = exe[
    "Build"
].isin(
    evaluation_builds
)


model_train_mask = dataset[
    "Build"
].isin(
    training_builds
)


model_eval_mask = dataset[
    "Build"
].isin(
    evaluation_builds
)


validation_records = []


add_check(
    validation_records,
    "Step 1B passed",
    EXPECTED_STEP1B_STATUS,
    step1b_status.get(
        "Status"
    ),
    step1b_status.get(
        "Status"
    )
    == EXPECTED_STEP1B_STATUS,
)


add_check(
    validation_records,
    "Step 2A passed",
    EXPECTED_STEP2A_STATUS,
    step2a_status.get(
        "Status"
    ),
    step2a_status.get(
        "Status"
    )
    == EXPECTED_STEP2A_STATUS,
)


add_check(
    validation_records,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_SHA256,
    selection_sha256,
    selection_sha256
    == EXPECTED_SELECTION_SHA256,
)


add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)


add_check(
    validation_records,
    "Canonical builds",
    EXPECTED_BUILDS,
    len(
        chronology
    ),
    len(
        chronology
    )
    == EXPECTED_BUILDS,
)


add_check(
    validation_records,
    "Training builds",
    EXPECTED_TRAIN_BUILDS,
    len(
        training_builds
    ),
    len(
        training_builds
    )
    == EXPECTED_TRAIN_BUILDS,
)


add_check(
    validation_records,
    "Evaluation builds",
    EXPECTED_EVAL_BUILDS,
    len(
        evaluation_builds
    ),
    len(
        evaluation_builds
    )
    == EXPECTED_EVAL_BUILDS,
)


add_check(
    validation_records,
    "Timestamp tie groups",
    EXPECTED_TIMESTAMP_TIE_GROUPS,
    timestamp_tie_groups_count,
    timestamp_tie_groups_count
    == EXPECTED_TIMESTAMP_TIE_GROUPS,
)


add_check(
    validation_records,
    "Raw execution rows",
    EXPECTED_RAW_ROWS,
    len(
        exe
    ),
    len(
        exe
    )
    == EXPECTED_RAW_ROWS,
)


add_check(
    validation_records,
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    int(
        raw_train_mask.sum()
    ),
    int(
        raw_train_mask.sum()
    )
    == EXPECTED_RAW_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    int(
        raw_eval_mask.sum()
    ),
    int(
        raw_eval_mask.sum()
    )
    == EXPECTED_RAW_EVAL_ROWS,
)


add_check(
    validation_records,
    "Raw training failures",
    EXPECTED_RAW_TRAIN_FAILURES,
    int(
        exe.loc[
            raw_train_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    ),
    int(
        exe.loc[
            raw_train_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    )
    == EXPECTED_RAW_TRAIN_FAILURES,
)


add_check(
    validation_records,
    "Raw evaluation failures",
    EXPECTED_RAW_EVAL_FAILURES,
    int(
        exe.loc[
            raw_eval_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    ),
    int(
        exe.loc[
            raw_eval_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    )
    == EXPECTED_RAW_EVAL_FAILURES,
)


add_check(
    validation_records,
    "Model-ready rows",
    EXPECTED_MODEL_ROWS,
    len(
        dataset
    ),
    len(
        dataset
    )
    == EXPECTED_MODEL_ROWS,
)


add_check(
    validation_records,
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    int(
        model_train_mask.sum()
    ),
    int(
        model_train_mask.sum()
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    int(
        model_eval_mask.sum()
    ),
    int(
        model_eval_mask.sum()
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)


add_check(
    validation_records,
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    int(
        dataset.loc[
            model_train_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    ),
    int(
        dataset.loc[
            model_train_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    )
    == EXPECTED_MODEL_TRAIN_FAILURES,
)


add_check(
    validation_records,
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    int(
        dataset.loc[
            model_eval_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    ),
    int(
        dataset.loc[
            model_eval_mask,
            "Verdict",
        ].ne(
            SUCCESS_VERDICT_CODE
        ).sum()
    )
    == EXPECTED_MODEL_EVAL_FAILURES,
)


add_check(
    validation_records,
    "Dataset columns",
    EXPECTED_DATASET_COLUMNS,
    len(
        dataset_header
    ),
    len(
        dataset_header
    )
    == EXPECTED_DATASET_COLUMNS,
)


add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    )
    == EXPECTED_PREDICTORS,
)


add_check(
    validation_records,
    "Raw duplicate Build-Test rows",
    0,
    raw_duplicate_pairs,
    raw_duplicate_pairs
    == 0,
)


add_check(
    validation_records,
    "Model duplicate Build-Test rows",
    0,
    model_duplicate_pairs,
    model_duplicate_pairs
    == 0,
)


add_check(
    validation_records,
    "Official assertion verdict code",
    2,
    ASSERTION_VERDICT_CODE,
    ASSERTION_VERDICT_CODE
    == 2,
)


add_check(
    validation_records,
    "Official exception verdict code",
    1,
    EXCEPTION_VERDICT_CODE,
    EXCEPTION_VERDICT_CODE
    == 1,
)


add_check(
    validation_records,
    "Per-test search accounting",
    total_tests,
    model_ready_tests
    + raw_only_tests,
    (
        model_ready_tests
        + raw_only_tests
    )
    == total_tests,
)


add_check(
    validation_records,
    "Tests touching timestamp ties",
    "0..total_tests (diagnostic; exact order inferred when >0)",
    tests_with_timestamp_ties,
    0 <= tests_with_timestamp_ties <= total_tests,
)


add_check(
    validation_records,
    "Tests with non-zero order mismatches",
    0,
    tests_with_nonzero_order_mismatches,
    tests_with_nonzero_order_mismatches
    == 0,
)


add_check(
    validation_records,
    "Total order mismatch values",
    0,
    total_test_order_mismatch_values,
    total_test_order_mismatch_values
    == 0,
)


add_check(
    validation_records,
    "Global REC_Age mismatch rows",
    0,
    best_age_mismatches,
    best_age_mismatches
    == 0,
)


add_check(
    validation_records,
    "Global REC_Age zero-match candidates",
    "> 0",
    zero_age_candidates,
    zero_age_candidates
    > 0,
)


add_check(
    validation_records,
    "Commit-token rows",
    EXPECTED_COMMIT_TOKEN_ROWS,
    len(
        commit_audit
    ),
    len(
        commit_audit
    )
    == EXPECTED_COMMIT_TOKEN_ROWS,
)


add_check(
    validation_records,
    "Exact commit matches",
    EXPECTED_EXACT_COMMIT_MATCHES,
    exact_matches,
    exact_matches
    == EXPECTED_EXACT_COMMIT_MATCHES,
)


add_check(
    validation_records,
    "Unique-prefix matches",
    EXPECTED_PREFIX_COMMIT_MATCHES,
    prefix_matches,
    prefix_matches
    == EXPECTED_PREFIX_COMMIT_MATCHES,
)


add_check(
    validation_records,
    "Unmatched commit tokens",
    EXPECTED_UNMATCHED_COMMIT_TOKENS,
    unmatched_tokens,
    unmatched_tokens
    == EXPECTED_UNMATCHED_COMMIT_TOKENS,
)


add_check(
    validation_records,
    "Ambiguous commit tokens",
    EXPECTED_AMBIGUOUS_COMMIT_TOKENS,
    ambiguous_tokens,
    ambiguous_tokens
    == EXPECTED_AMBIGUOUS_COMMIT_TOKENS,
)


add_check(
    validation_records,
    "Builds with mapped entities",
    EXPECTED_BUILDS_WITH_MAPPED_ENTITIES,
    len(
        builds_with_entities
    ),
    len(
        builds_with_entities
    )
    == EXPECTED_BUILDS_WITH_MAPPED_ENTITIES,
)


add_check(
    validation_records,
    "Builds without mapped entities",
    EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES,
    len(
        builds_without_entities
    ),
    len(
        builds_without_entities
    )
    == EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES,
)


add_check(
    validation_records,
    "Mapping-incomplete build identities",
    sorted(
        EXPECTED_MAPPING_INCOMPLETE_BUILDS
    ),
    mapping_incomplete_builds,
    set(
        mapping_incomplete_builds
    )
    == EXPECTED_MAPPING_INCOMPLETE_BUILDS,
)


add_check(
    validation_records,
    "Mapping-incomplete source rows",
    len(
        EXPECTED_MAPPING_INCOMPLETE_BUILDS
    ),
    len(
        mapping_incomplete_source
    ),
    len(
        mapping_incomplete_source
    )
    == len(
        EXPECTED_MAPPING_INCOMPLETE_BUILDS
    ),
)


add_check(
    validation_records,
    "Mapping-incomplete partitions",
    sorted(
        EXPECTED_MAPPING_INCOMPLETE_PARTITIONS
    ),
    mapping_incomplete_source_partitions,
    set(
        mapping_incomplete_source_partitions
    )
    == EXPECTED_MAPPING_INCOMPLETE_PARTITIONS,
)


add_check(
    validation_records,
    "Mapping-incomplete rows with mapped entities",
    EXPECTED_MAPPING_INCOMPLETE_ROWS_WITH_ENTITIES,
    mapping_incomplete_source_rows_with_entities,
    mapping_incomplete_source_rows_with_entities
    == EXPECTED_MAPPING_INCOMPLETE_ROWS_WITH_ENTITIES,
)


add_check(
    validation_records,
    "Build-entity rows",
    EXPECTED_BUILD_ENTITY_ROWS,
    len(
        build_entity
    ),
    len(
        build_entity
    )
    == EXPECTED_BUILD_ENTITY_ROWS,
)


add_check(
    validation_records,
    "Reconstructed REC rows",
    EXPECTED_MODEL_ROWS,
    len(
        clean_reconstructed
    ),
    len(
        clean_reconstructed
    )
    == EXPECTED_MODEL_ROWS,
)


add_check(
    validation_records,
    "Duplicate reconstructed rows",
    0,
    reconstructed_duplicate_rows,
    reconstructed_duplicate_rows
    == 0,
)


add_check(
    validation_records,
    "Missing reconstructed values",
    0,
    missing_reconstructed_rows,
    missing_reconstructed_rows
    == 0,
)


add_check(
    validation_records,
    "Tie-inference feature direct mismatch values",
    0,
    anchor_disallowed_non_file_direct_mismatches,
    anchor_disallowed_non_file_direct_mismatches
    == 0,
)


add_check(
    validation_records,
    "Verdict-dependent direct mismatch values",
    0,
    verdict_dependent_direct_mismatches,
    verdict_dependent_direct_mismatches
    == 0,
)


add_check(
    validation_records,
    "Non-file direct residuals confined to anchor-allowed timing features",
    True,
    direct_residuals_confined_to_anchor_allowed_features,
    direct_residuals_confined_to_anchor_allowed_features,
)


add_check(
    validation_records,
    "Anchor-allowed timing direct residual values",
    ">= 0 (diagnostic; must be removed exactly by clean anchor)",
    anchor_allowed_direct_mismatches,
    anchor_allowed_direct_mismatches >= 0,
)


add_check(
    validation_records,
    "File-history mismatches outside mapping-incomplete builds",
    0,
    file_mismatches_outside_mapping_incomplete_build,
    file_mismatches_outside_mapping_incomplete_build
    == 0,
)


add_check(
    validation_records,
    "Unmatched mapping effect confined",
    True,
    unmatched_mapping_effect_is_confined,
    unmatched_mapping_effect_is_confined,
)


add_check(
    validation_records,
    "Failed clean-anchor features",
    0,
    failed_anchor_features,
    failed_anchor_features
    == 0,
)


add_check(
    validation_records,
    "Anchored mismatch values",
    0,
    anchored_mismatch_values,
    anchored_mismatch_values
    == 0,
)


add_check(
    validation_records,
    "0% clean dataset reproduced exactly",
    True,
    zero_percent_clean_reproduced_exactly,
    zero_percent_clean_reproduced_exactly,
)


add_check(
    validation_records,
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(
        registry
    ),
    len(
        registry
    )
    == EXPECTED_REGISTERED_PROJECTS,
)


for required_number, required_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            registry_project_numbers.eq(
                required_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {required_number} frozen identity",
        required_project,
        actual_project,
        actual_project
        == required_project,
    )


add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    active_reservations,
    active_reservations
    == EXPECTED_ACTIVE_RESERVATIONS,
)


add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    selection.get(
        "RuntimePriorityRule"
    ),
    selection.get(
        "RuntimePriorityRule"
    )
    == EXPECTED_RUNTIME_PRIORITY_RULE,
)


add_check(
    validation_records,
    "Project 22 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 22 Step 2B validation:"
)

display(
    validation
)


print(
    "\nClean REC comparison:"
)

display(
    comparison_summary
)


print(
    "\nClean-anchor validation:"
)

display(
    anchor_validation
)


if not failed_validation.empty:
    print(
        "\nFailed Step 2B checks:"
    )

    display(
        failed_validation
    )

    print(
        "\nNo Step 2B PASS checkpoint was written."
    )

    raise RuntimeError(
        "PROJECT 22 STEP 2B VALIDATION FAILED. "
        "DO NOT START THE EXPERIMENT."
    )


# --------------------------------------------------------------------------------------------------
# 9. FREEZE OUTPUTS
# --------------------------------------------------------------------------------------------------

PREFLIGHT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


exe[
    "StartedAtUTC"
] = exe[
    "Build"
].map(
    build_timestamp_map
)


inferred_execution_order_for_storage = (
    exe[
        [
            "Build",
            "Test",
            "Job",
            "Verdict",
            "Duration",
            "StartedAtUTC",
            "InferredTestOrder",
        ]
    ]
    .sort_values(
        [
            "Test",
            "InferredTestOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


atomic_csv(
    UNMATCHED_MAPPING_AUDIT_PATH,
    unmatched_mapping_audit,
)


atomic_csv(
    TIMESTAMP_TIE_GROUPS_PATH,
    timestamp_tie_groups_frame,
)


atomic_csv(
    TEST_ORDER_SEARCH_AUDIT_PATH,
    test_order_search_audit,
)


print(
    "\nWriting the frozen 59,155-row execution-order parquet."
)


atomic_parquet(
    INFERRED_EXECUTION_ORDER_PATH,
    inferred_execution_order_for_storage,
)


atomic_csv(
    GLOBAL_AGE_ORDER_SEARCH_PATH,
    global_age_order_search,
)


atomic_csv(
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    frozen_global_build_order,
)


atomic_parquet(
    CLEAN_RECONSTRUCTED_PATH,
    clean_reconstructed[
        [
            "Build",
            "Test",
        ]
        + REC_FEATURES
    ],
)


atomic_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH,
    anchor_offsets[
        [
            "Build",
            "Test",
        ]
        + REC_FEATURES
    ],
)


atomic_csv(
    CLEAN_COMPARISON_SUMMARY_PATH,
    comparison_summary,
)


atomic_csv(
    CLEAN_MISMATCH_EXAMPLES_PATH,
    mismatch_examples_frame,
)


atomic_csv(
    CLEAN_ANCHOR_VALIDATION_PATH,
    anchor_validation,
)


atomic_csv(
    STEP2B_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 10. READBACK VALIDATION
# --------------------------------------------------------------------------------------------------

execution_order_metadata = pq.ParquetFile(
    INFERRED_EXECUTION_ORDER_PATH
)


execution_order_readback_rows = int(
    execution_order_metadata.metadata.num_rows
)


execution_order_readback_columns = set(
    execution_order_metadata.schema.names
)


required_execution_order_columns = {
    "Build",
    "Test",
    "Job",
    "Verdict",
    "Duration",
    "StartedAtUTC",
    "InferredTestOrder",
}


if (
    execution_order_readback_rows
    != EXPECTED_RAW_ROWS
    or not required_execution_order_columns.issubset(
        execution_order_readback_columns
    )
):
    raise RuntimeError(
        "Frozen execution-order parquet metadata readback failed."
    )


reconstructed_readback = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)


anchor_offsets_readback = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)


if len(
    reconstructed_readback
) != EXPECTED_MODEL_ROWS:
    raise RuntimeError(
        "Clean reconstructed REC parquet readback failed."
    )


if len(
    anchor_offsets_readback
) != EXPECTED_MODEL_ROWS:
    raise RuntimeError(
        "Clean anchor-offset parquet readback failed."
    )


readback_join = (
    reconstructed_readback.merge(
        anchor_offsets_readback,
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_reconstructed",
            "_offset",
        ),
    )
    .merge(
        dataset[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
    )
)


readback_mismatch_values = 0


for feature in REC_FEATURES:
    reproduced_values = (
        readback_join[
            f"{feature}_reconstructed"
        ].to_numpy(
            dtype=float
        )
        + readback_join[
            f"{feature}_offset"
        ].to_numpy(
            dtype=float
        )
    )

    original_values = readback_join[
        feature
    ].to_numpy(
        dtype=float
    )

    readback_mismatch_values += int(
        (
            ~np.isclose(
                reproduced_values,
                original_values,
                rtol=ANCHOR_RTOL,
                atol=ANCHOR_ATOL,
                equal_nan=False,
            )
        ).sum()
    )


if readback_mismatch_values != 0:
    raise RuntimeError(
        "Frozen clean anchor failed readback reproduction."
    )


# --------------------------------------------------------------------------------------------------
# 11. REPORT, CHECKPOINT, AND STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


output_paths = [
    UNMATCHED_MAPPING_AUDIT_PATH,
    TIMESTAMP_TIE_GROUPS_PATH,
    TEST_ORDER_SEARCH_AUDIT_PATH,
    INFERRED_EXECUTION_ORDER_PATH,
    GLOBAL_AGE_ORDER_SEARCH_PATH,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    CLEAN_COMPARISON_SUMMARY_PATH,
    CLEAN_MISMATCH_EXAMPLES_PATH,
    CLEAN_ANCHOR_VALIDATION_PATH,
    STEP2B_VALIDATION_PATH,
]


output_manifest = [
    {
        "Path":
            str(
                path
            ),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2B_STATUS,

    "ImplementationVersion":
        IMPLEMENTATION_VERSION,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "SelectionCheckpointSHA256":
        selection_sha256,

    "OfficialVerdictSemantics": {
        "Success":
            SUCCESS_VERDICT_CODE,

        "Exception":
            EXCEPTION_VERDICT_CODE,

        "Assertion":
            ASSERTION_VERDICT_CODE,
    },

    "TimestampTieGroups":
        timestamp_tie_groups_count,

    "TimestampTieBuilds":
        timestamp_tie_builds,

    "ModelReadyTests":
        model_ready_tests,

    "RawOnlyTests":
        raw_only_tests,

    "TestsTouchingTimestampTies":
        tests_with_timestamp_ties,

    "TestsWithNonZeroOrderMismatches":
        tests_with_nonzero_order_mismatches,

    "TestsWithMultipleZeroMismatchOrders":
        tests_with_ambiguous_zero_orders,

    "GlobalAgeOrderCombinations":
        global_age_combination_count,

    "GlobalAgeZeroMismatchCandidates":
        zero_age_candidates,

    "GlobalAgeMinimumMismatchRows":
        best_age_mismatches,

    "RawSortSeconds":
        sort_seconds,

    "RECReconstructionSeconds":
        reconstruction_seconds,

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "RawExecutionRows":
        len(
            inferred_execution_order_for_storage
        ),

    "ModelReadyRows":
        len(
            dataset
        ),

    "ReconstructedRows":
        len(
            clean_reconstructed
        ),

    "GlobalBuildOrderRows":
        len(
            frozen_global_build_order
        ),

    "CommitTokenRows":
        len(
            commit_audit
        ),

    "ExactCommitMatches":
        exact_matches,

    "UniquePrefixMatches":
        prefix_matches,

    "UnmatchedCommitTokens":
        unmatched_tokens,

    "AmbiguousCommitTokens":
        ambiguous_tokens,

    "BuildsWithMappedEntities":
        len(
            builds_with_entities
        ),

    "BuildsWithoutMappedEntities":
        len(
            builds_without_entities
        ),

    "MappingIncompleteBuilds":
        mapping_incomplete_builds,

    "MappingIncompletePartitions":
        mapping_incomplete_source_partitions,

    "MappingIncompleteRowsWithMappedEntities":
        mapping_incomplete_source_rows_with_entities,

    "DirectMismatchValues":
        direct_mismatch_values,

    "VerdictDependentDirectMismatches":
        verdict_dependent_direct_mismatches,

    "VerdictIndependentDirectMismatches":
        verdict_independent_direct_mismatches,

    "FileHistoryDirectMismatches":
        file_history_direct_mismatches,

    "NonFileDirectMismatches":
        non_file_direct_mismatches,

    "AnchorAllowedDirectResidualFeatures":
        ANCHOR_ALLOWED_DIRECT_RESIDUAL_FEATURES,

    "AnchorAllowedDirectResidualValues":
        anchor_allowed_direct_mismatches,

    "AnchorDisallowedNonFileDirectMismatches":
        anchor_disallowed_non_file_direct_mismatches,

    "DirectResidualsConfinedToAnchorAllowedFeatures":
        direct_residuals_confined_to_anchor_allowed_features,

    "TieInferenceFeatures":
        TIE_INFERENCE_FEATURES,

    "FileHistoryMismatchesOutsideMappingIncompleteBuilds":
        file_mismatches_outside_mapping_incomplete_build,

    "UnmatchedMappingEffectConfined":
        unmatched_mapping_effect_is_confined,

    "RowsWithAnyNonZeroAnchorOffset":
        rows_with_any_nonzero_anchor_offset,

    "NonZeroAnchorOffsetValues":
        nonzero_anchor_offset_values,

    "FailedAnchorFeatures":
        failed_anchor_features,

    "AnchoredMismatchValues":
        anchored_mismatch_values,

    "ReadbackMismatchValues":
        readback_mismatch_values,

    "ZeroPercentCleanDatasetReproducedExactly":
        zero_percent_clean_reproduced_exactly,

    "OutputManifest":
        output_manifest,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "ActiveReservations":
        active_reservations,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "RegistryModified":
        False,

    "Projects1To21Modified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "NoiseInjected":
        False,

    "ModelsTrained":
        False,
}


atomic_json(
    STEP2B_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "CheckpointType":
        "PROJECT_22_CLEAN_REC_RECONSTRUCTION",

    "RECReconstructionFrozen":
        True,

    "PerTestExecutionOrderFrozen":
        True,

    "GlobalBuildFirstAppearanceOrderFrozen":
        True,

    "CleanAnchorFrozen":
        True,

    "EvaluationCohortImmutable":
        True,

    "ProceedToNoisePlanAllowed":
        True,
}


atomic_json(
    REC_CHECKPOINT_PATH,
    checkpoint_payload,
)


rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2B_STATUS,

    "ImplementationVersion":
        IMPLEMENTATION_VERSION,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "TimestampTieGroups":
        timestamp_tie_groups_count,

    "TestsTouchingTimestampTies":
        tests_with_timestamp_ties,

    "TestsWithNonZeroOrderMismatches":
        tests_with_nonzero_order_mismatches,

    "GlobalAgeMinimumMismatchRows":
        best_age_mismatches,

    "NonFileDirectMismatches":
        non_file_direct_mismatches,

    "AnchorAllowedDirectResidualFeatures":
        ANCHOR_ALLOWED_DIRECT_RESIDUAL_FEATURES,

    "AnchorAllowedDirectResidualValues":
        anchor_allowed_direct_mismatches,

    "AnchorDisallowedNonFileDirectMismatches":
        anchor_disallowed_non_file_direct_mismatches,

    "DirectResidualsConfinedToAnchorAllowedFeatures":
        direct_residuals_confined_to_anchor_allowed_features,

    "TieInferenceFeatures":
        TIE_INFERENCE_FEATURES,

    "FileHistoryMismatchesOutsideMappingIncompleteBuilds":
        file_mismatches_outside_mapping_incomplete_build,

    "UnmatchedMappingEffectConfined":
        unmatched_mapping_effect_is_confined,

    "FailedAnchorFeatures":
        failed_anchor_features,

    "AnchoredMismatchValues":
        anchored_mismatch_values,

    "ZeroPercentCleanDatasetReproducedExactly":
        zero_percent_clean_reproduced_exactly,

    "Checkpoint":
        str(
            REC_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        rec_checkpoint_sha256,

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,
}


atomic_json(
    STEP2B_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 12. FINAL IMMUTABILITY AND READBACK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Project 22 Step 2B."
    )


final_source_manifest_records = []

for row in current_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_source_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_source_manifest = pd.DataFrame(
    final_source_manifest_records
)


if source_root_hash(
    final_source_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 22 source changed during Step 2B."
    )


checkpoint_readback = load_json(
    REC_CHECKPOINT_PATH
)


status_readback = load_json(
    STEP2B_STATUS_PATH
)


if (
    checkpoint_readback.get(
        "Status"
    )
    != STEP2B_STATUS
    or status_readback.get(
        "Status"
    )
    != STEP2B_STATUS
):
    raise RuntimeError(
        "Project 22 Step 2B checkpoint/status readback failed."
    )


# --------------------------------------------------------------------------------------------------
# 13. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 136)
print("=== PROJECT 22 CELL 5 / STEP 2B RESULT ===")
print("=" * 136)


print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print(
    "Project 11 identity:",
    required_registered_identities[
        11
    ],
)

print(
    "Project 12 identity:",
    required_registered_identities[
        12
    ],
)

print(
    "Project 13 identity:",
    required_registered_identities[
        13
    ],
)

print(
    "Project 14 identity:",
    required_registered_identities[
        14
    ],
)

print(
    "Project 15 identity:",
    required_registered_identities[
        15
    ],
)

print(
    "Project 16 identity:",
    required_registered_identities[
        16
    ],
)

print(
    "Project 17 identity:",
    required_registered_identities[
        17
    ],
)

print(
    "Project 18 identity:",
    required_registered_identities[
        18
    ],
)

print(
    "Project 19 identity:",
    required_registered_identities[
        19
    ],
)

print(
    "Project 20 identity:",
    required_registered_identities[
        20
    ],
)

print(
    "Active reservations:",
    active_reservations,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)

print(
    "Source root SHA-256:",
    current_source_root_sha256,
)


print(
    "\nOfficial verdict semantics:"
)

print(
    "Success:",
    SUCCESS_VERDICT_CODE,
)

print(
    "Exception:",
    EXCEPTION_VERDICT_CODE,
)

print(
    "Assertion:",
    ASSERTION_VERDICT_CODE,
)


print(
    "\nDeterministic execution-order freeze:"
)

print(
    "Timestamp tie groups:",
    timestamp_tie_groups_count,
)

print(
    "Raw execution-order rows:",
    execution_order_readback_rows,
)

print(
    "Global build-order rows:",
    len(
        frozen_global_build_order
    ),
)

print(
    "Tests:",
    total_tests,
)

print(
    "Model-ready tests:",
    model_ready_tests,
)

print(
    "Raw-only tests:",
    raw_only_tests,
)

print(
    "Tests with non-zero order mismatches:",
    tests_with_nonzero_order_mismatches,
)

print(
    "Global REC_Age mismatch rows:",
    best_age_mismatches,
)


print(
    "\nClean REC reconstruction:"
)

print(
    "Raw history rows:",
    len(
        inferred_execution_order_for_storage
    ),
)

print(
    "Model rows requested/reconstructed:",
    len(
        dataset
    ),
    "/",
    len(
        clean_reconstructed
    ),
)

print(
    "Direct mismatch values:",
    direct_mismatch_values,
)

print(
    "Non-file direct mismatch values:",
    non_file_direct_mismatches,
)

print(
    "Anchor-allowed timing direct residual values:",
    anchor_allowed_direct_mismatches,
)

print(
    "Anchor-disallowed non-file direct mismatch values:",
    anchor_disallowed_non_file_direct_mismatches,
)

print(
    "File-history direct mismatch values:",
    file_history_direct_mismatches,
)

print(
    "File-history mismatches outside mapping-incomplete builds:",
    file_mismatches_outside_mapping_incomplete_build,
)

print(
    "Rows with any non-zero anchor offset:",
    rows_with_any_nonzero_anchor_offset,
)

print(
    "Non-zero anchor-offset values:",
    nonzero_anchor_offset_values,
)

print(
    "Failed anchor features:",
    failed_anchor_features,
)

print(
    "Anchored mismatch values:",
    anchored_mismatch_values,
)

print(
    "Readback mismatch values:",
    readback_mismatch_values,
)

print(
    "0% clean dataset reproduced exactly:",
    zero_percent_clean_reproduced_exactly,
)


print(
    "\nMapping audit:"
)

print(
    "Commit-token rows:",
    len(
        commit_audit
    ),
)

print(
    "Exact / prefix / unmatched / ambiguous:",
    exact_matches,
    "/",
    prefix_matches,
    "/",
    unmatched_tokens,
    "/",
    ambiguous_tokens,
)

print(
    "Builds with / without mapped entities:",
    len(
        builds_with_entities
    ),
    "/",
    len(
        builds_without_entities
    ),
)

print(
    "Mapping-incomplete builds:",
    len(
        mapping_incomplete_builds
    ),
)

print(
    "Mapping-incomplete partitions:",
    mapping_incomplete_source_partitions,
)

print(
    "Mapping-incomplete rows with mapped entities:",
    mapping_incomplete_source_rows_with_entities,
)

print(
    "Unmatched mapping effect confined:",
    unmatched_mapping_effect_is_confined,
)


print(
    "\nRuntime:"
)

print(
    "Raw sort seconds:",
    round(
        sort_seconds,
        2,
    ),
)

print(
    "REC reconstruction seconds:",
    round(
        reconstruction_seconds,
        2,
    ),
)


print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–21 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Noise injected:",
    False,
)

print(
    "Models trained:",
    False,
)


print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print(
    "\nREC reconstruction checkpoint:"
)

print(
    REC_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    rec_checkpoint_sha256,
)


print(
    "\nSTATUS:",
    STEP2B_STATUS,
)

print("=" * 136)


=== PROJECT 22 CELL 5 / STEP 2B V2: ANCHOR-AWARE DETERMINISTIC CLEAN REC RECONSTRUCTION ===
Loading the 59,155-row clean execution history.
Sorting raw execution history by Test and frozen chronology.

Timestamp tie groups:


,TieGroup,StartedAtUTC,BuildCount,BuildIDsJSON,PermutationCount
0,1,2020-08-02T20:05:11+00:00,2,"[714276122, 714276120]",2



Unmatched mapping audit:


,BuildID,ChronologyOrder,Partition,UnmatchedCommitTokens,MappedEntityCount,HasMappedEntities,RawExecutionRows,RawFailureRows,ModelReadyRows,ModelFailureRows


Per-test tie-order inference progress: 100 / 689 tests
Per-test tie-order inference progress: 200 / 689 tests
Per-test tie-order inference progress: 300 / 689 tests
Per-test tie-order inference progress: 400 / 689 tests
Per-test tie-order inference progress: 500 / 689 tests
Per-test tie-order inference progress: 600 / 689 tests
Per-test tie-order inference progress: 689 / 689 tests

Per-test tie-order inference summary:


...


Clean REC comparison:


,Feature,FeatureClass,FileHistoryFeature,Rows,DirectMatchingRows,DirectMismatchingRows,DirectMismatchesAtMappingIncompleteBuild,DirectMismatchesOutsideMappingIncompleteBuild,NonZeroAnchorOffsets,AnchoredMatchingRows,AnchoredMismatchingRows,MaximumAbsoluteDirectDifference,MeanAbsoluteDirectDifference,MaximumAbsoluteAnchoredDifference
0,REC_Age,VERDICT_INDEPENDENT,False,117968,117968,0,0,0,0,117968,0,0.000000e+00,0.000000e+00,0.0
1,REC_LastFailureAge,VERDICT_DEPENDENT,False,117968,117968,0,0,0,0,117968,0,0.000000e+00,0.000000e+00,0.0
2,REC_LastTransitionAge,VERDICT_DEPENDENT,False,117968,117968,0,0,0,0,117968,0,0.000000e+00,0.000000e+00,0.0
3,REC_RecentAvgExeTime,VERDICT_INDEPENDENT,False,117968,117967,1,0,1,15633,117968,0,1.103333e+02,9.352819e-04,0.0
4,REC_RecentMaxExeTime,VERDICT_INDEPENDENT,False,117968,117967,1,0,1,1,117968,0,2.450000e+02,2.076834e-03,0.0
5,REC_RecentFailRate,VERDICT_DEPENDENT,False,117968,117968,0,0,0,77,117968,0,5.551115e-17,3.623320e-20,0.0
6,REC_RecentAssertRate,VERDICT_DEPENDENT,False,117968,117968,0,0,0,73,117968,0,5.551115e-17,3.435096e-20,0.0
7,REC_RecentExcRate,VERDICT_DEPENDENT,False,117968,117968,0,0,0,4,117968,0,5.551115e-17,1.882244e-21,0.0
8,REC_RecentTransitionRate,VERDICT_DEPENDENT,False,117968,117968,0,0,0,60,117968,0,5.551115e-17,2.823367e-20,0.0
9,REC_TotalAvgExeTime,VERDICT_INDEPENDENT,False,117968,117906,62,0,62,15733,117968,0,2.000000e+00,7.755392e-04,0.0



Clean-anchor validation:


,Feature,FeatureClass,Rows,MatchingRows,MismatchingRows,MaximumAbsoluteAnchoredDifference,Pass
0,REC_Age,VERDICT_INDEPENDENT,117968,117968,0,0.0,True
1,REC_LastFailureAge,VERDICT_DEPENDENT,117968,117968,0,0.0,True
2,REC_LastTransitionAge,VERDICT_DEPENDENT,117968,117968,0,0.0,True
3,REC_RecentAvgExeTime,VERDICT_INDEPENDENT,117968,117968,0,0.0,True
4,REC_RecentMaxExeTime,VERDICT_INDEPENDENT,117968,117968,0,0.0,True
5,REC_RecentFailRate,VERDICT_DEPENDENT,117968,117968,0,0.0,True
6,REC_RecentAssertRate,VERDICT_DEPENDENT,117968,117968,0,0.0,True
7,REC_RecentExcRate,VERDICT_DEPENDENT,117968,117968,0,0.0,True
8,REC_RecentTransitionRate,VERDICT_DEPENDENT,117968,117968,0,0.0,True
9,REC_TotalAvgExeTime,VERDICT_INDEPENDENT,117968,117968,0,0.0,True



Writing the frozen 59,155-row execution-order parquet.


=== PROJECT 22 CELL 5 / STEP 2B RESULT ===
Project: apache@logging-log4j2
Project slug: apache__logging-log4j2
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Project 17 identity: yamcs@Yamcs
Project 18 identity: cantaloupe-project@cantaloupe
Project 19 identity: EMResearch@EvoMaster
Project 20 identity: apache@curator
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']
Source root SHA-256: 281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64f842964c896c6ac334

Official verdict semantics:
Success: 0
Exception: 1
Assertion: 2

Deterministic execution-order freeze:
Timestamp tie groups: 1
Raw execution-order rows: 240253
Global build-order rows: 441
Test

In [8]:
# ==================================================================================================
# PROJECT 22 — CELL 6 / STEP 3A
# DETERMINISTIC NOISE PLAN AND COHORT FREEZE
#
# PROJECT:
#   apache@logging-log4j2
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME Thesis_project_22.ipynb NOTEBOOK.
#
# PURPOSE:
# - verify the frozen Project 22 selection, source, REC reconstruction, and clean anchor;
# - freeze the raw and model-ready training/evaluation cohorts;
# - freeze the Project 22 failure-subtype distribution;
# - generate deterministic project/seed random streams for label-noise injection;
# - prove nested masks across all noise levels for all 30 repetition seeds;
# - freeze all 270 condition coordinates and expected noisy-label hashes;
# - leave the evaluation partition clean and immutable;
# - perform no model fitting and no registry write.
#
# SAFETY:
# - Projects 1–21 must remain COMPLETE_AND_FROZEN and unchanged;
# - Project 22 must remain absent from the completion registry;
# - no prior-project condition output is accessed or modified.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


print("=" * 132)
print("=== PROJECT 22 CELL 6 / STEP 3A: DETERMINISTIC NOISE PLAN AND COHORT FREEZE ===")
print("=" * 132)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 22
PROJECT_NAME = "apache@logging-log4j2"
PROJECT_SLUG = "apache__logging-log4j2"
PROJECT_SHORT = "LOG4J2"

SOURCE_DIR = Path(
    "/content/datasets/datasets/apache@logging-log4j2"
)

EXPECTED_SELECTION_STATUS = (
    "PASS_PROJECT_22_SELECTION_AND_SOURCE_FROZEN"
)

EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_22_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

STEP3A_STATUS = (
    "PASS_PROJECT_22_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

EXPECTED_SELECTION_SHA256 = (
    "a7387d9495c71dce5d2c21ff08b0afd80d9c5251b5885a82e4052c7506ee7890"
)

EXPECTED_REC_CHECKPOINT_SHA256 = (
    "8417249bc74e2a50e2f61cc3b7776dab7f731f62993e3ed15e415745be178731"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64f842964c896c6ac334"
)

EXPECTED_REGISTRY_SHA256 = (
    "79cd6ecb595c5e8ae91a9494e469792716338d144308560a62caf1b9342306b2"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 21

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 101_286_750

EXPECTED_BUILDS = 441
EXPECTED_TRAIN_BUILDS = 330
EXPECTED_EVAL_BUILDS = 111

EXPECTED_RAW_ROWS = 240_253
EXPECTED_RAW_TRAIN_ROWS = 172_628
EXPECTED_RAW_EVAL_ROWS = 67_625
EXPECTED_RAW_TRAIN_FAILURES = 208
EXPECTED_RAW_EVAL_FAILURES = 40

EXPECTED_MODEL_ROWS = 117_968
EXPECTED_MODEL_TRAIN_ROWS = 95_812
EXPECTED_MODEL_EVAL_ROWS = 22_156
EXPECTED_MODEL_TRAIN_FAILURES = 207
EXPECTED_MODEL_EVAL_FAILURES = 40
EXPECTED_MODEL_FAILING_EVAL_BUILDS = 39

EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

REPETITION_SEEDS = list(
    range(1, 31)
)

EXPECTED_CONDITIONS = (
    len(NOISE_LEVELS)
    * len(REPETITION_SEEDS)
)

EXPECTED_RNG_ROWS = (
    EXPECTED_RAW_TRAIN_ROWS
    * len(REPETITION_SEEDS)
)

RECENT_WINDOW = 6

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_22_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_22_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_22_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_22_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_22_fixed_chronological_builds.csv"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

REC_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

STEP2B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2b_status.json"
)

STEP2B_REPORT_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2b_report.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_22_rec_reconstruction_checkpoint.json"
)

CLEAN_RECONSTRUCTED_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
)

CLEAN_ANCHOR_OFFSETS_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
)

INFERRED_EXECUTION_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_inferred_execution_order.parquet"
)

FROZEN_GLOBAL_BUILD_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_global_build_order.csv"
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

FAILURE_SUBTYPE_PROFILE_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_failure_subtype_profile.csv"
)

SEED_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_seed_manifest.csv"
)

RNG_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_rng_manifest.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

NESTED_MASK_AUDIT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_nested_mask_audit.csv"
)

PROTOCOL_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_frozen_experiment_protocol.json"
)

STEP3A_VALIDATION_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_validation.csv"
)

STEP3A_REPORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_report.json"
)

STEP3A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step3a_status.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_22_noise_plan_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def sha256_array(
    array,
    dtype,
):
    canonical = np.asarray(
        array,
        dtype=dtype,
        order="C",
    )

    return hashlib.sha256(
        canonical.tobytes(
            order="C"
        )
    ).hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_parquet(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_parquet(
        temporary_path,
        index=False,
        compression="zstd",
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def parse_int(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing or non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(
            array
        ),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype(
        "int64"
    )


def deterministic_seed(
    repetition_seed,
    stream_name,
):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode(
        "utf-8"
    )

    digest = hashlib.sha256(
        material
    ).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


def checkpoint_output_sha256(
    checkpoint,
    path,
):
    target_path = str(
        Path(
            path
        )
    )

    matches = [
        entry
        for entry in checkpoint.get(
            "OutputManifest",
            [],
        )
        if str(
            entry.get(
                "Path",
                "",
            )
        ) == target_path
    ]

    if len(
        matches
    ) != 1:
        raise RuntimeError(
            "The Project 22 REC checkpoint does not contain exactly "
            f"one manifest entry for {target_path}."
        )

    return str(
        matches[
            0
        ][
            "SHA256"
        ]
    )


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN-STATE VALIDATION
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    STEP2B_STATUS_PATH,
    STEP2B_REPORT_PATH,
    REC_CHECKPOINT_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    INFERRED_EXECUTION_ORDER_PATH,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    SOURCE_DIR / "dataset.csv",
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 22 Step 3A inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


selection_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)

selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = load_json(
    STEP1B_STATUS_PATH
)

step2b_status = load_json(
    STEP2B_STATUS_PATH
)

step2b_report = load_json(
    STEP2B_REPORT_PATH
)

rec_checkpoint = load_json(
    REC_CHECKPOINT_PATH
)


if selection_sha256 != EXPECTED_SELECTION_SHA256:
    raise RuntimeError(
        "Project 22 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_SHA256}\n"
        f"Actual:   {selection_sha256}"
    )


if rec_checkpoint_sha256 != EXPECTED_REC_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 22 REC checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_REC_CHECKPOINT_SHA256}\n"
        f"Actual:   {rec_checkpoint_sha256}"
    )


if (
    selection_checkpoint.get(
        "Status"
    ) != EXPECTED_SELECTION_STATUS
    or step1b_status.get(
        "Status"
    ) != EXPECTED_SELECTION_STATUS
):
    raise RuntimeError(
        "Project 22 selection is not frozen successfully."
    )


if (
    step2b_status.get(
        "Status"
    ) != EXPECTED_STEP2B_STATUS
    or step2b_report.get(
        "Status"
    ) != EXPECTED_STEP2B_STATUS
    or rec_checkpoint.get(
        "Status"
    ) != EXPECTED_STEP2B_STATUS
):
    raise RuntimeError(
        "Project 22 Step 2B is not frozen successfully."
    )


if not bool(
    rec_checkpoint.get(
        "ZeroPercentCleanDatasetReproducedExactly",
        False,
    )
):
    raise RuntimeError(
        "The Project 22 REC checkpoint does not confirm "
        "exact clean-anchor reproduction."
    )


expected_rec_freeze_flags = {
    "ImplementationVersion":
        "PROJECT_22_V2_ANCHOR_AWARE_TIE_INFERENCE_FULL_MAPPING",

    "CheckpointVersion":
        1,

    # Frozen exactly as written by Project 22 Step 2B.
    "CheckpointType":
        "PROJECT_22_CLEAN_REC_RECONSTRUCTION",

    "RECReconstructionFrozen":
        True,

    "PerTestExecutionOrderFrozen":
        True,

    "GlobalBuildFirstAppearanceOrderFrozen":
        True,

    "CleanAnchorFrozen":
        True,

    "EvaluationCohortImmutable":
        True,

    "ProceedToNoisePlanAllowed":
        True,
}


for flag_name, expected_value in expected_rec_freeze_flags.items():
    if rec_checkpoint.get(
        flag_name
    ) != expected_value:
        raise RuntimeError(
            "The Project 22 REC checkpoint does not match the frozen "
            f"Step 2B contract: {flag_name}={expected_value!r}."
        )


if (
    selection_checkpoint.get(
        "Project"
    ) != PROJECT_NAME
    or selection_checkpoint.get(
        "ProjectSlug"
    ) != PROJECT_SLUG
):
    raise RuntimeError(
        "The frozen Project 22 identity differs."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(int)


if (
    len(
        registry
    ) != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    ) != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–21."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–21 are not all COMPLETE_AND_FROZEN."
    )


required_registered_identities = {
    11:
        "apache@shardingsphere",

    12:
        "zolyfarkas@spf4j",

    13:
        "jcabi@jcabi-github",

    14:
        "JMRI@JMRI",

    15:
        "eclipse@steady",

    16:
        "apache@rocketmq",

    17:
        "yamcs@Yamcs",

    18:
        "cantaloupe-project@cantaloupe",

    19:
        "EMResearch@EvoMaster",

    20:
        "apache@curator",

    21:
        "facebook@buck",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 22 is unexpectedly already registered."
    )


selection_active_reservations = selection_checkpoint.get(
    "ActiveReservations",
    None,
)


if selection_active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Project 22 selection checkpoint active reservations differ."
    )


if selection_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Project 22 selection checkpoint runtime-priority rule differs."
    )


if rec_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Project 22 REC checkpoint active reservations differ."
    )


frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_manifest = pd.DataFrame([
    {
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                (
                    SOURCE_DIR
                    / str(
                        row.RelativePath
                    )
                ).stat().st_size
            ),

        "SHA256":
            sha256_file(
                SOURCE_DIR
                / str(
                    row.RelativePath
                )
            ),
    }
    for row in frozen_source_manifest.itertuples(
        index=False
    )
])


current_source_root_sha256 = source_root_hash(
    current_source_manifest
)

current_source_bytes = int(
    current_source_manifest[
        "SizeBytes"
    ].sum()
)


if current_source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 22 source root differs.\n"
        f"Expected: {EXPECTED_SOURCE_ROOT_SHA256}\n"
        f"Actual:   {current_source_root_sha256}"
    )


expected_inferred_execution_order_sha256 = checkpoint_output_sha256(
    rec_checkpoint,
    INFERRED_EXECUTION_ORDER_PATH,
)

expected_global_build_order_sha256 = checkpoint_output_sha256(
    rec_checkpoint,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
)

expected_clean_reconstructed_sha256 = checkpoint_output_sha256(
    rec_checkpoint,
    CLEAN_RECONSTRUCTED_PATH,
)

expected_clean_anchor_offsets_sha256 = checkpoint_output_sha256(
    rec_checkpoint,
    CLEAN_ANCHOR_OFFSETS_PATH,
)

actual_inferred_execution_order_sha256 = sha256_file(
    INFERRED_EXECUTION_ORDER_PATH
)

actual_global_build_order_sha256 = sha256_file(
    FROZEN_GLOBAL_BUILD_ORDER_PATH
)

actual_clean_reconstructed_sha256 = sha256_file(
    CLEAN_RECONSTRUCTED_PATH
)

actual_clean_anchor_offsets_sha256 = sha256_file(
    CLEAN_ANCHOR_OFFSETS_PATH
)


if (
    actual_inferred_execution_order_sha256
    != expected_inferred_execution_order_sha256
    or actual_global_build_order_sha256
    != expected_global_build_order_sha256
    or actual_clean_reconstructed_sha256
    != expected_clean_reconstructed_sha256
    or actual_clean_anchor_offsets_sha256
    != expected_clean_anchor_offsets_sha256
):
    raise RuntimeError(
        "One or more frozen Project 22 Step 2B artifacts "
        "do not match the REC checkpoint manifest."
    )


# --------------------------------------------------------------------------------------------------
# 5. LOAD CHRONOLOGY, MODEL DATA, AND THE FROZEN V6 RAW EXECUTION ORDER
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)


chronology[
    "BuildID"
] = parse_int(
    chronology[
        "BuildID"
    ],
    "chronology.BuildID",
)


training_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "TRAIN"
        ),
        "BuildID",
    ].astype(
        int
    )
)


evaluation_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq(
            "EVALUATION"
        ),
        "BuildID",
    ].astype(
        int
    )
)


dataset_header = pd.read_csv(
    SOURCE_DIR
    / "dataset.csv",
    nrows=0,
).columns.tolist()


dataset_build_column = resolve_column(
    dataset_header,
    "Build",
    "dataset Build",
)

dataset_test_column = resolve_column(
    dataset_header,
    "Test",
    "dataset Test",
)

dataset_verdict_column = resolve_column(
    dataset_header,
    "Verdict",
    "dataset Verdict",
)


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in dataset_header
]


if missing_rec_features:
    raise RuntimeError(
        "dataset.csv is missing REC features:\n"
        + "\n".join(
            missing_rec_features
        )
    )


predictor_columns = [
    column
    for column in dataset_header
    if column not in {
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    }
]


dataset = pd.read_csv(
    SOURCE_DIR
    / "dataset.csv",
    low_memory=False,
)


dataset = dataset.rename(
    columns={
        dataset_build_column:
            "Build",

        dataset_test_column:
            "Test",

        dataset_verdict_column:
            "Verdict",
    }
)


dataset[
    "Build"
] = parse_int(
    dataset[
        "Build"
    ],
    "dataset.Build",
)

dataset[
    "Test"
] = parse_int(
    dataset[
        "Test"
    ],
    "dataset.Test",
)

dataset[
    "Verdict"
] = parse_int(
    dataset[
        "Verdict"
    ],
    "dataset.Verdict",
)


# V6 froze the exact per-test execution order required to reproduce all 19 REC features.
# This is the canonical raw-history cohort for every Project 22 noise condition.
inferred_execution_order = pd.read_parquet(
    INFERRED_EXECUTION_ORDER_PATH
)


required_inferred_columns = {
    "Build",
    "Test",
    "Job",
    "Verdict",
    "Duration",
    "StartedAtUTC",
    "InferredTestOrder",
}


missing_inferred_columns = (
    required_inferred_columns
    - set(
        inferred_execution_order.columns
    )
)


if missing_inferred_columns:
    raise RuntimeError(
        "The frozen V6 inferred execution-order file is missing columns:\n"
        + "\n".join(
            sorted(
                missing_inferred_columns
            )
        )
    )


inferred_execution_order[
    "Build"
] = parse_int(
    inferred_execution_order[
        "Build"
    ],
    "inferred_execution_order.Build",
)

inferred_execution_order[
    "Test"
] = parse_int(
    inferred_execution_order[
        "Test"
    ],
    "inferred_execution_order.Test",
)

inferred_execution_order[
    "Verdict"
] = parse_int(
    inferred_execution_order[
        "Verdict"
    ],
    "inferred_execution_order.Verdict",
)

inferred_execution_order[
    "InferredTestOrder"
] = parse_int(
    inferred_execution_order[
        "InferredTestOrder"
    ],
    "inferred_execution_order.InferredTestOrder",
)

inferred_execution_order[
    "Job"
] = pd.to_numeric(
    inferred_execution_order[
        "Job"
    ],
    errors="coerce",
)

inferred_execution_order[
    "Duration"
] = pd.to_numeric(
    inferred_execution_order[
        "Duration"
    ],
    errors="coerce",
)


if (
    inferred_execution_order[
        "Job"
    ].isna().any()
    or inferred_execution_order[
        "Duration"
    ].isna().any()
):
    raise RuntimeError(
        "The frozen raw execution order contains missing/non-numeric "
        "job or duration values."
    )


if not np.isfinite(
    inferred_execution_order[
        "Duration"
    ].to_numpy(
        dtype=float
    )
).all():
    raise RuntimeError(
        "The frozen raw execution order contains non-finite durations."
    )


if inferred_execution_order[
    "Duration"
].lt(
    0
).any():
    raise RuntimeError(
        "The frozen raw execution order contains negative durations."
    )


raw_duplicate_build_test_rows = int(
    inferred_execution_order.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


raw_duplicate_test_order_rows = int(
    inferred_execution_order.duplicated(
        subset=[
            "Test",
            "InferredTestOrder",
        ],
        keep=False,
    ).sum()
)


if (
    raw_duplicate_build_test_rows
    or raw_duplicate_test_order_rows
):
    raise RuntimeError(
        "The frozen V6 execution order contains duplicate keys."
    )


exe = (
    inferred_execution_order.sort_values(
        [
            "Test",
            "InferredTestOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
    .copy()
)


frozen_global_build_order = pd.read_csv(
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    low_memory=False,
)


required_global_order_columns = {
    "GlobalBuildOrder",
    "BuildID",
}


if not required_global_order_columns.issubset(
    frozen_global_build_order.columns
):
    raise RuntimeError(
        "The frozen V6 global build-order file is missing required columns."
    )


frozen_global_build_order[
    "GlobalBuildOrder"
] = parse_int(
    frozen_global_build_order[
        "GlobalBuildOrder"
    ],
    "frozen_global_build_order.GlobalBuildOrder",
)

frozen_global_build_order[
    "BuildID"
] = parse_int(
    frozen_global_build_order[
        "BuildID"
    ],
    "frozen_global_build_order.BuildID",
)


global_build_order_valid = bool(
    len(
        frozen_global_build_order
    )
    == EXPECTED_BUILDS
    and frozen_global_build_order[
        "BuildID"
    ].nunique()
    == EXPECTED_BUILDS
    and set(
        frozen_global_build_order[
            "BuildID"
        ].astype(
            int
        )
    )
    == (
        training_builds
        | evaluation_builds
    )
    and sorted(
        frozen_global_build_order[
            "GlobalBuildOrder"
        ].astype(
            int
        ).tolist()
    )
    == list(
        range(
            1,
            EXPECTED_BUILDS
            + 1,
        )
    )
)


if not global_build_order_valid:
    raise RuntimeError(
        "The frozen V6 global build order is invalid."
    )


# 6. FREEZE RAW AND MODEL COHORTS
# --------------------------------------------------------------------------------------------------

raw_training = (
    exe.loc[
        exe[
            "Build"
        ].isin(
            training_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


raw_training.insert(
    0,
    "RawTrainingRowOrder",
    np.arange(
        1,
        len(
            raw_training
        )
        + 1,
        dtype=np.int64,
    ),
)


raw_evaluation = (
    exe.loc[
        exe[
            "Build"
        ].isin(
            evaluation_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


raw_evaluation.insert(
    0,
    "RawEvaluationRowOrder",
    np.arange(
        1,
        len(
            raw_evaluation
        )
        + 1,
        dtype=np.int64,
    ),
)


model_training = (
    dataset.loc[
        dataset[
            "Build"
        ].isin(
            training_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


model_training.insert(
    0,
    "ModelTrainingRowOrder",
    np.arange(
        1,
        len(
            model_training
        )
        + 1,
        dtype=np.int64,
    ),
)


model_evaluation = (
    dataset.loc[
        dataset[
            "Build"
        ].isin(
            evaluation_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


model_evaluation.insert(
    0,
    "ModelEvaluationRowOrder",
    np.arange(
        1,
        len(
            model_evaluation
        )
        + 1,
        dtype=np.int64,
    ),
)


raw_training_failures = int(
    raw_training[
        "Verdict"
    ].ne(
        0
    ).sum()
)

raw_evaluation_failures = int(
    raw_evaluation[
        "Verdict"
    ].ne(
        0
    ).sum()
)

model_training_failures = int(
    model_training[
        "Verdict"
    ].ne(
        0
    ).sum()
)

model_evaluation_failures = int(
    model_evaluation[
        "Verdict"
    ].ne(
        0
    ).sum()
)

model_failing_evaluation_builds = int(
    model_evaluation.loc[
        model_evaluation[
            "Verdict"
        ].ne(
            0
        ),
        "Build",
    ].nunique()
)


raw_training_link_source = raw_training[
    [
        "RawTrainingRowOrder",
        "Build",
        "Test",
        "Verdict",
    ]
].rename(
    columns={
        "Verdict":
            "RawVerdict",
    }
)


model_training_link = (
    model_training[
        [
            "ModelTrainingRowOrder",
            "Build",
            "Test",
            "Verdict",
        ]
    ]
    .rename(
        columns={
            "Verdict":
                "ModelVerdict",
        }
    )
    .merge(
        raw_training_link_source,
        on=[
            "Build",
            "Test",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    .sort_values(
        "ModelTrainingRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


raw_evaluation_link_source = raw_evaluation[
    [
        "RawEvaluationRowOrder",
        "Build",
        "Test",
        "Verdict",
    ]
].rename(
    columns={
        "Verdict":
            "RawVerdict",
    }
)


model_evaluation_link = (
    model_evaluation[
        [
            "ModelEvaluationRowOrder",
            "Build",
            "Test",
            "Verdict",
        ]
    ]
    .rename(
        columns={
            "Verdict":
                "ModelVerdict",
        }
    )
    .merge(
        raw_evaluation_link_source,
        on=[
            "Build",
            "Test",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    .sort_values(
        "ModelEvaluationRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


missing_model_training_links = int(
    model_training_link[
        "_merge"
    ].ne(
        "both"
    ).sum()
)

missing_model_evaluation_links = int(
    model_evaluation_link[
        "_merge"
    ].ne(
        "both"
    ).sum()
)

model_training_verdict_mismatches = int(
    model_training_link[
        "ModelVerdict"
    ].ne(
        model_training_link[
            "RawVerdict"
        ]
    ).sum()
)

model_evaluation_verdict_mismatches = int(
    model_evaluation_link[
        "ModelVerdict"
    ].ne(
        model_evaluation_link[
            "RawVerdict"
        ]
    ).sum()
)


if (
    missing_model_training_links
    or missing_model_evaluation_links
    or model_training_verdict_mismatches
    or model_evaluation_verdict_mismatches
):
    raise RuntimeError(
        "Fixed model/raw cohort linkage failed."
    )


model_training_raw_indices = (
    model_training_link[
        "RawTrainingRowOrder"
    ].astype(
        np.int64
    ).to_numpy()
    - 1
)


# --------------------------------------------------------------------------------------------------
# 7. VERIFY THE FROZEN CLEAN ANCHOR
# --------------------------------------------------------------------------------------------------

clean_reconstructed = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)

clean_anchor_offsets = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)


clean_anchor_join = (
    clean_reconstructed.merge(
        clean_anchor_offsets,
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_reconstructed",
            "_offset",
        ),
    )
    .merge(
        dataset[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="one_to_one",
    )
)


clean_anchor_mismatch_values = 0


for feature in REC_FEATURES:
    reproduced = (
        clean_anchor_join[
            f"{feature}_reconstructed"
        ].to_numpy(
            dtype=float
        )
        + clean_anchor_join[
            f"{feature}_offset"
        ].to_numpy(
            dtype=float
        )
    )

    original = clean_anchor_join[
        feature
    ].to_numpy(
        dtype=float
    )

    clean_anchor_mismatch_values += int(
        (
            ~np.isclose(
                reproduced,
                original,
                rtol=0.0,
                atol=1e-12,
            )
        ).sum()
    )


clean_anchor_reproduced_dataset = bool(
    len(
        clean_anchor_join
    ) == EXPECTED_MODEL_ROWS
    and clean_anchor_mismatch_values == 0
)


if not clean_anchor_reproduced_dataset:
    raise RuntimeError(
        "Frozen clean anchor no longer reproduces dataset.csv exactly."
    )


# --------------------------------------------------------------------------------------------------
# 8. PROJECT-SPECIFIC FAILURE-SUBTYPE PROFILE
# --------------------------------------------------------------------------------------------------

failure_subtype_counts = (
    raw_training.loc[
        raw_training[
            "Verdict"
        ].ne(
            0
        ),
        "Verdict",
    ]
    .value_counts()
    .sort_index()
)


if failure_subtype_counts.empty:
    raise RuntimeError(
        "No clean raw training failure subtypes were found."
    )


failure_subtypes = (
    failure_subtype_counts.index.astype(
        int
    ).to_numpy(
        dtype=np.int16
    )
)


if (
    failure_subtypes.min()
    < np.iinfo(
        np.int16
    ).min
    or failure_subtypes.max()
    > np.iinfo(
        np.int16
    ).max
):
    raise RuntimeError(
        "Failure subtype values do not fit int16."
    )


failure_subtype_probabilities = (
    failure_subtype_counts.to_numpy(
        dtype=float
    )
    / failure_subtype_counts.sum()
)


failure_subtype_profile = pd.DataFrame({
    "FailureSubtype":
        failure_subtypes.astype(
            int
        ),

    "CleanTrainingRows":
        failure_subtype_counts.to_numpy(
            dtype=int
        ),

    "Probability":
        failure_subtype_probabilities,
})


failure_subtype_values_list = failure_subtypes.astype(
    int
).tolist()


failure_subtype_values_valid = bool(
    len(
        failure_subtype_values_list
    )
    > 0
    and set(
        failure_subtype_values_list
    ).issubset({
        1,
        2,
    })
)


failure_subtype_profile_sum_valid = bool(
    int(
        failure_subtype_profile[
            "CleanTrainingRows"
        ].sum()
    )
    == raw_training_failures
    and np.isclose(
        failure_subtype_profile[
            "Probability"
        ].sum(),
        1.0,
        rtol=0.0,
        atol=1e-12,
    )
)


if not failure_subtype_values_valid:
    raise RuntimeError(
        "Project 22 clean training failures must use a non-empty subset "
        "of the frozen exception/assertion codes [1, 2]."
    )


if not failure_subtype_profile_sum_valid:
    raise RuntimeError(
        "Project 22 failure-subtype profile does not reproduce "
        "the clean raw training failure count."
    )


# --------------------------------------------------------------------------------------------------
# 9. GENERATE THE 30 DETERMINISTIC RNG STREAMS AND 270 CONDITION PLAN
# --------------------------------------------------------------------------------------------------

NOISE_PLAN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


rng_temporary_path = RNG_MANIFEST_PATH.with_name(
    f".{RNG_MANIFEST_PATH.name}.tmp_{os.getpid()}"
)


if rng_temporary_path.exists():
    rng_temporary_path.unlink()


rng_schema = pa.schema([
    pa.field(
        "RepetitionSeed",
        pa.int16(),
    ),

    pa.field(
        "RawTrainingRowOrder",
        pa.int32(),
    ),

    pa.field(
        "FlipUniform",
        pa.float64(),
    ),

    pa.field(
        "SampledFailureSubtype",
        pa.int16(),
    ),
])


rng_writer = pq.ParquetWriter(
    rng_temporary_path,
    schema=rng_schema,
    compression="zstd",
)


seed_records = []
condition_records = []
nested_mask_records = []

clean_raw_verdict = raw_training[
    "Verdict"
].to_numpy(
    dtype=np.int16
)

clean_model_verdict = model_training[
    "Verdict"
].to_numpy(
    dtype=np.int16
)

raw_row_order_int32 = np.arange(
    1,
    len(
        raw_training
    )
    + 1,
    dtype=np.int32,
)


try:
    condition_order = 0

    for seed_order, repetition_seed in enumerate(
        REPETITION_SEEDS,
        start=1,
    ):
        flip_seed = deterministic_seed(
            repetition_seed,
            "flip_mask",
        )

        failure_subtype_seed = deterministic_seed(
            repetition_seed,
            "failure_subtype",
        )

        flip_uniform = np.random.default_rng(
            flip_seed
        ).random(
            len(
                raw_training
            )
        )

        sampled_failure_subtype = np.random.default_rng(
            failure_subtype_seed
        ).choice(
            failure_subtypes,
            size=len(
                raw_training
            ),
            replace=True,
            p=failure_subtype_probabilities,
        ).astype(
            np.int16
        )


        rng_table = pa.Table.from_arrays(
            [
                pa.array(
                    np.full(
                        len(
                            raw_training
                        ),
                        repetition_seed,
                        dtype=np.int16,
                    ),
                    type=pa.int16(),
                ),

                pa.array(
                    raw_row_order_int32,
                    type=pa.int32(),
                ),

                pa.array(
                    flip_uniform,
                    type=pa.float64(),
                ),

                pa.array(
                    sampled_failure_subtype,
                    type=pa.int16(),
                ),
            ],
            schema=rng_schema,
        )


        rng_writer.write_table(
            rng_table
        )


        regenerated_uniform = np.random.default_rng(
            flip_seed
        ).random(
            len(
                raw_training
            )
        )

        regenerated_subtype = np.random.default_rng(
            failure_subtype_seed
        ).choice(
            failure_subtypes,
            size=len(
                raw_training
            ),
            replace=True,
            p=failure_subtype_probabilities,
        ).astype(
            np.int16
        )


        uniforms_reproduced = bool(
            np.array_equal(
                flip_uniform,
                regenerated_uniform,
            )
        )

        failure_subtypes_reproduced = bool(
            np.array_equal(
                sampled_failure_subtype,
                regenerated_subtype,
            )
        )


        seed_records.append({
            "RepetitionSeed":
                repetition_seed,

            "FlipSeed":
                flip_seed,

            "FailureSubtypeSeed":
                failure_subtype_seed,

            "NoiseRows":
                len(
                    raw_training
                ),

            "FlipUniformSHA256":
                sha256_array(
                    flip_uniform,
                    "<f8",
                ),

            "SampledFailureSubtypeSHA256":
                sha256_array(
                    sampled_failure_subtype,
                    "<i2",
                ),

            "UniformsReproduced":
                uniforms_reproduced,

            "FailureSubtypesReproduced":
                failure_subtypes_reproduced,
        })


        previous_mask = None
        previous_noise = None


        for noise_order, noise_percent in enumerate(
            NOISE_LEVELS,
            start=1,
        ):
            condition_order += 1

            condition_id = (
                f"noise_{noise_percent:02d}"
                f"__seed_{repetition_seed:02d}"
            )

            flip_mask = (
                flip_uniform
                < (
                    noise_percent
                    / 100.0
                )
            )

            noisy_raw_verdict = clean_raw_verdict.copy()

            pass_to_failure_mask = (
                flip_mask
                & (
                    clean_raw_verdict
                    == 0
                )
            )

            failure_to_pass_mask = (
                flip_mask
                & (
                    clean_raw_verdict
                    != 0
                )
            )

            noisy_raw_verdict[
                pass_to_failure_mask
            ] = sampled_failure_subtype[
                pass_to_failure_mask
            ]

            noisy_raw_verdict[
                failure_to_pass_mask
            ] = 0

            noisy_model_verdict = noisy_raw_verdict[
                model_training_raw_indices
            ]

            number_flipped = int(
                flip_mask.sum()
            )

            pass_to_failure = int(
                pass_to_failure_mask.sum()
            )

            failure_to_pass = int(
                failure_to_pass_mask.sum()
            )

            noisy_raw_failures = int(
                (
                    noisy_raw_verdict
                    != 0
                ).sum()
            )

            model_label_changes = int(
                (
                    noisy_model_verdict
                    != clean_model_verdict
                ).sum()
            )

            noisy_model_failures = int(
                (
                    noisy_model_verdict
                    != 0
                ).sum()
            )

            condition_records.append({
                "ConditionOrder":
                    condition_order,

                "ConditionID":
                    condition_id,

                "SeedOrder":
                    seed_order,

                "NoiseOrderWithinSeed":
                    noise_order,

                "NoisePercent":
                    noise_percent,

                "RepetitionSeed":
                    repetition_seed,

                "FlipSeed":
                    flip_seed,

                "FailureSubtypeSeed":
                    failure_subtype_seed,

                "RawTrainingRows":
                    len(
                        raw_training
                    ),

                "NumberFlipped":
                    number_flipped,

                "RealisedNoisePercent":
                    (
                        100.0
                        * number_flipped
                        / len(
                            raw_training
                        )
                    ),

                "PassToFailure":
                    pass_to_failure,

                "FailureToPass":
                    failure_to_pass,

                "CleanRawFailures":
                    raw_training_failures,

                "NoisyRawFailures":
                    noisy_raw_failures,

                "ModelTrainingRows":
                    len(
                        model_training
                    ),

                "ModelLabelChanges":
                    model_label_changes,

                "CleanModelFailures":
                    model_training_failures,

                "NoisyModelFailures":
                    noisy_model_failures,

                "FlipMaskSHA256":
                    sha256_array(
                        flip_mask.astype(
                            np.uint8
                        ),
                        "u1",
                    ),

                "NoisyRawVerdictSHA256":
                    sha256_array(
                        noisy_raw_verdict,
                        "<i2",
                    ),

                "NoisyModelVerdictSHA256":
                    sha256_array(
                        noisy_model_verdict,
                        "<i2",
                    ),
            })


            if previous_mask is not None:
                violations = int(
                    (
                        previous_mask
                        & (
                            ~flip_mask
                        )
                    ).sum()
                )

                nested_mask_records.append({
                    "RepetitionSeed":
                        repetition_seed,

                    "LowerNoisePercent":
                        previous_noise,

                    "HigherNoisePercent":
                        noise_percent,

                    "Violations":
                        violations,

                    "Pass":
                        violations == 0,
                })


            previous_mask = flip_mask
            previous_noise = noise_percent

finally:
    rng_writer.close()


os.replace(
    rng_temporary_path,
    RNG_MANIFEST_PATH,
)


seed_manifest = pd.DataFrame(
    seed_records
)


condition_plan = pd.DataFrame(
    condition_records
)


nested_mask_audit = pd.DataFrame(
    nested_mask_records
)


nested_mask_violations = int(
    nested_mask_audit[
        "Violations"
    ].sum()
)


zero_noise_conditions = condition_plan[
    condition_plan[
        "NoisePercent"
    ].eq(
        0
    )
]


zero_noise_flip_violations = int(
    zero_noise_conditions[
        "NumberFlipped"
    ].ne(
        0
    ).sum()
)


zero_noise_raw_label_violations = int(
    zero_noise_conditions[
        "NoisyRawFailures"
    ].ne(
        raw_training_failures
    ).sum()
)


zero_noise_model_label_violations = int(
    zero_noise_conditions[
        "ModelLabelChanges"
    ].ne(
        0
    ).sum()
)


positive_noise_conditions = condition_plan[
    condition_plan[
        "NoisePercent"
    ].gt(
        0
    )
]


positive_noise_without_raw_changes = int(
    positive_noise_conditions[
        "NumberFlipped"
    ].le(
        0
    ).sum()
)


positive_noise_without_model_changes = int(
    positive_noise_conditions[
        "ModelLabelChanges"
    ].le(
        0
    ).sum()
)


duplicate_condition_ids = int(
    condition_plan[
        "ConditionID"
    ].duplicated(
        keep=False
    ).sum()
)


duplicate_condition_coordinates = int(
    condition_plan.duplicated(
        subset=[
            "NoisePercent",
            "RepetitionSeed",
        ],
        keep=False,
    ).sum()
)


seed_streams_reproduced = bool(
    seed_manifest[
        [
            "UniformsReproduced",
            "FailureSubtypesReproduced",
        ]
    ].all().all()
)


# --------------------------------------------------------------------------------------------------
# 10. WRITE FROZEN COHORTS AND PLAN OUTPUTS
# --------------------------------------------------------------------------------------------------

atomic_parquet(
    RAW_TRAINING_COHORT_PATH,
    raw_training,
)

atomic_parquet(
    RAW_EVALUATION_COHORT_PATH,
    raw_evaluation,
)

atomic_parquet(
    MODEL_TRAINING_COHORT_PATH,
    model_training,
)

atomic_parquet(
    MODEL_EVALUATION_COHORT_PATH,
    model_evaluation,
)

atomic_parquet(
    MODEL_RAW_TRAIN_LINK_PATH,
    model_training_link.drop(
        columns=[
            "_merge",
        ]
    ),
)

atomic_parquet(
    MODEL_RAW_EVAL_LINK_PATH,
    model_evaluation_link.drop(
        columns=[
            "_merge",
        ]
    ),
)

atomic_csv(
    FAILURE_SUBTYPE_PROFILE_PATH,
    failure_subtype_profile,
)

atomic_csv(
    SEED_MANIFEST_PATH,
    seed_manifest,
)

atomic_csv(
    CONDITION_PLAN_PATH,
    condition_plan,
)

atomic_csv(
    NESTED_MASK_AUDIT_PATH,
    nested_mask_audit,
)


protocol_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ProtocolState":
        "FROZEN",

    "Chronology":
        (
            "fixed chronological split: started_at ascending; "
            "Build ID descending for timestamp ties"
        ),

    "RawHistoryOrder":
        (
            "Project 22 Step 2B frozen per-test execution order; "
            "no timestamp ties are present in the frozen chronology"
        ),

    "GlobalRECAgeBuildOrder":
        (
            "Project 22 Step 2B frozen global build first-appearance order"
        ),

    "Split":
        {
            "Type":
                "chronological_fixed_holdout",

            "TrainingFraction":
                0.75,

            "EvaluationFraction":
                0.25,

            "TrainingBuilds":
                len(
                    training_builds
                ),

            "EvaluationBuilds":
                len(
                    evaluation_builds
                ),
        },

    "NoiseLevelsPercent":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "Conditions":
        EXPECTED_CONDITIONS,

    "RecentExecutionWindow":
        RECENT_WINDOW,

    "NoisePartition":
        "training only",

    "EvaluationPartition":
        "clean and immutable",

    "NoiseUnit":
        "individual raw training execution verdict",

    "FlipRule":
        {
            "PassToFailure":
                (
                    "0 is replaced by a failure subtype "
                    "sampled from the clean project-specific "
                    "failure-subtype distribution"
                ),

            "FailureToPass":
                (
                    "every non-zero verdict selected by "
                    "the mask is replaced by 0"
                ),
        },

    "Randomisation":
        {
            "SeedDerivation":
                (
                    "first little-endian uint32 of "
                    "SHA-256(project|repetition_seed|stream)"
                ),

            "FlipMaskStream":
                "flip_mask",

            "FailureSubtypeStream":
                "failure_subtype",

            "NestedMasks":
                True,

            "SameSeedUsesSameStreamsAcrossNoise":
                True,
        },

    "FeatureHandling":
        {
            "VerdictDependentRECRecomputed":
                VERDICT_DEPENDENT_REC,

            "VerdictIndependentRECPreserved":
                VERDICT_INDEPENDENT_REC,

            "AllRECFeatures":
                REC_FEATURES,

            "CleanAnchorApplied":
                True,
        },

    "TrainingInstanceCohort":
        "fixed TCP-CI model-ready training rows",

    "EvaluationMetrics":
        [
            "APFDc",
            "APFD",
        ],

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "MLTechniques":
        ML_TECHNIQUES,

    "Baselines":
        BASELINES,

    "SameCorruptedHistoryUsedBy":
        ML_TECHNIQUES
        + [
            "LatestFail",
        ],

    "QTFAvgNoiseIndependent":
        True,

    "RandomConstantAcrossNoiseForSameSeedAndBuild":
        True,

    "NoRollingRetraining":
        True,

    "RankingTieBreak":
        "score, then Test ascending",
}


atomic_json(
    PROTOCOL_PATH,
    protocol_payload,
)


# --------------------------------------------------------------------------------------------------
# 11. READBACK AND REPRODUCIBILITY VALIDATION
# --------------------------------------------------------------------------------------------------

raw_training_readback = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)

raw_evaluation_readback = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)

model_training_readback = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)

model_evaluation_readback = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)

condition_plan_readback = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)

seed_manifest_readback = pd.read_csv(
    SEED_MANIFEST_PATH,
    low_memory=False,
)

nested_mask_readback = pd.read_csv(
    NESTED_MASK_AUDIT_PATH,
    low_memory=False,
)

rng_readback_rows = int(
    pq.ParquetFile(
        RNG_MANIFEST_PATH
    ).metadata.num_rows
)


nested_mask_readback_violations = int(
    nested_mask_readback[
        "Violations"
    ].sum()
)


# Reproduce every stream again from the frozen seed manifest.
stream_reproduction_failures = 0


for row in seed_manifest_readback.itertuples(
    index=False
):
    repetition_seed = int(
        row.RepetitionSeed
    )

    flip_seed = deterministic_seed(
        repetition_seed,
        "flip_mask",
    )

    subtype_seed = deterministic_seed(
        repetition_seed,
        "failure_subtype",
    )

    reproduced_uniform = np.random.default_rng(
        flip_seed
    ).random(
        EXPECTED_RAW_TRAIN_ROWS
    )

    reproduced_subtype = np.random.default_rng(
        subtype_seed
    ).choice(
        failure_subtypes,
        size=EXPECTED_RAW_TRAIN_ROWS,
        replace=True,
        p=failure_subtype_probabilities,
    ).astype(
        np.int16
    )

    if (
        int(
            row.FlipSeed
        ) != flip_seed
        or int(
            row.FailureSubtypeSeed
        ) != subtype_seed
        or str(
            row.FlipUniformSHA256
        ) != sha256_array(
            reproduced_uniform,
            "<f8",
        )
        or str(
            row.SampledFailureSubtypeSHA256
        ) != sha256_array(
            reproduced_subtype,
            "<i2",
        )
    ):
        stream_reproduction_failures += 1


# --------------------------------------------------------------------------------------------------
# 12. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 1B passed",
    EXPECTED_SELECTION_STATUS,
    step1b_status.get(
        "Status"
    ),
    step1b_status.get(
        "Status"
    ) == EXPECTED_SELECTION_STATUS,
)

add_check(
    validation_records,
    "Step 2B passed",
    EXPECTED_STEP2B_STATUS,
    step2b_status.get(
        "Status"
    ),
    step2b_status.get(
        "Status"
    ) == EXPECTED_STEP2B_STATUS,
)

add_check(
    validation_records,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_SHA256,
    selection_sha256,
    selection_sha256
    == EXPECTED_SELECTION_SHA256,
)

add_check(
    validation_records,
    "REC checkpoint SHA-256",
    EXPECTED_REC_CHECKPOINT_SHA256,
    rec_checkpoint_sha256,
    rec_checkpoint_sha256
    == EXPECTED_REC_CHECKPOINT_SHA256,
)

add_check(
    validation_records,
    "REC checkpoint implementation",
    "PROJECT_22_V2_ANCHOR_AWARE_TIE_INFERENCE_FULL_MAPPING",
    rec_checkpoint.get(
        "ImplementationVersion"
    ),
    rec_checkpoint.get(
        "ImplementationVersion"
    )
    == "PROJECT_22_V2_ANCHOR_AWARE_TIE_INFERENCE_FULL_MAPPING",
)

add_check(
    validation_records,
    "REC checkpoint schema version",
    1,
    rec_checkpoint.get(
        "CheckpointVersion"
    ),
    rec_checkpoint.get(
        "CheckpointVersion"
    )
    == 1,
)

add_check(
    validation_records,
    "Frozen inferred execution-order SHA-256",
    expected_inferred_execution_order_sha256,
    actual_inferred_execution_order_sha256,
    actual_inferred_execution_order_sha256
    == expected_inferred_execution_order_sha256,
)

add_check(
    validation_records,
    "Frozen global build-order SHA-256",
    expected_global_build_order_sha256,
    actual_global_build_order_sha256,
    actual_global_build_order_sha256
    == expected_global_build_order_sha256,
)

add_check(
    validation_records,
    "Frozen clean reconstruction SHA-256",
    expected_clean_reconstructed_sha256,
    actual_clean_reconstructed_sha256,
    actual_clean_reconstructed_sha256
    == expected_clean_reconstructed_sha256,
)

add_check(
    validation_records,
    "Frozen clean anchor-offset SHA-256",
    expected_clean_anchor_offsets_sha256,
    actual_clean_anchor_offsets_sha256,
    actual_clean_anchor_offsets_sha256
    == expected_clean_anchor_offsets_sha256,
)

add_check(
    validation_records,
    "Clean anchor reproduced dataset",
    True,
    clean_anchor_reproduced_dataset,
    clean_anchor_reproduced_dataset,
)

add_check(
    validation_records,
    "Source files",
    EXPECTED_SOURCE_FILES,
    len(
        current_source_manifest
    ),
    len(
        current_source_manifest
    ) == EXPECTED_SOURCE_FILES,
)

add_check(
    validation_records,
    "Source bytes",
    EXPECTED_SOURCE_BYTES,
    current_source_bytes,
    current_source_bytes
    == EXPECTED_SOURCE_BYTES,
)

add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)

add_check(
    validation_records,
    "Builds",
    EXPECTED_BUILDS,
    len(
        chronology
    ),
    len(
        chronology
    ) == EXPECTED_BUILDS,
)

add_check(
    validation_records,
    "Training builds",
    EXPECTED_TRAIN_BUILDS,
    len(
        training_builds
    ),
    len(
        training_builds
    ) == EXPECTED_TRAIN_BUILDS,
)

add_check(
    validation_records,
    "Evaluation builds",
    EXPECTED_EVAL_BUILDS,
    len(
        evaluation_builds
    ),
    len(
        evaluation_builds
    ) == EXPECTED_EVAL_BUILDS,
)

add_check(
    validation_records,
    "Raw rows",
    EXPECTED_RAW_ROWS,
    len(
        exe
    ),
    len(
        exe
    ) == EXPECTED_RAW_ROWS,
)

add_check(
    validation_records,
    "Frozen V6 raw duplicate Build-Test rows",
    0,
    raw_duplicate_build_test_rows,
    raw_duplicate_build_test_rows == 0,
)

add_check(
    validation_records,
    "Frozen V6 raw duplicate Test-order rows",
    0,
    raw_duplicate_test_order_rows,
    raw_duplicate_test_order_rows == 0,
)

add_check(
    validation_records,
    "Frozen V6 global build order valid",
    True,
    global_build_order_valid,
    global_build_order_valid,
)

add_check(
    validation_records,
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training
    ),
    len(
        raw_training
    ) == EXPECTED_RAW_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation
    ),
    len(
        raw_evaluation
    ) == EXPECTED_RAW_EVAL_ROWS,
)

add_check(
    validation_records,
    "Raw training failures",
    EXPECTED_RAW_TRAIN_FAILURES,
    raw_training_failures,
    raw_training_failures
    == EXPECTED_RAW_TRAIN_FAILURES,
)

add_check(
    validation_records,
    "Raw evaluation failures",
    EXPECTED_RAW_EVAL_FAILURES,
    raw_evaluation_failures,
    raw_evaluation_failures
    == EXPECTED_RAW_EVAL_FAILURES,
)

add_check(
    validation_records,
    "Failure subtype values",
    "non-empty subset of [1, 2]",
    failure_subtype_values_list,
    failure_subtype_values_valid,
)

add_check(
    validation_records,
    "Failure subtype profile sum",
    True,
    failure_subtype_profile_sum_valid,
    failure_subtype_profile_sum_valid,
)

add_check(
    validation_records,
    "Model rows",
    EXPECTED_MODEL_ROWS,
    len(
        dataset
    ),
    len(
        dataset
    ) == EXPECTED_MODEL_ROWS,
)

add_check(
    validation_records,
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training
    ),
    len(
        model_training
    ) == EXPECTED_MODEL_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation
    ),
    len(
        model_evaluation
    ) == EXPECTED_MODEL_EVAL_ROWS,
)

add_check(
    validation_records,
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    model_training_failures,
    model_training_failures
    == EXPECTED_MODEL_TRAIN_FAILURES,
)

add_check(
    validation_records,
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    model_evaluation_failures,
    model_evaluation_failures
    == EXPECTED_MODEL_EVAL_FAILURES,
)

add_check(
    validation_records,
    "Model failing evaluation builds",
    EXPECTED_MODEL_FAILING_EVAL_BUILDS,
    model_failing_evaluation_builds,
    model_failing_evaluation_builds
    == EXPECTED_MODEL_FAILING_EVAL_BUILDS,
)

add_check(
    validation_records,
    "Dataset columns",
    EXPECTED_DATASET_COLUMNS,
    len(
        dataset_header
    ),
    len(
        dataset_header
    ) == EXPECTED_DATASET_COLUMNS,
)

add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    ) == EXPECTED_PREDICTORS,
)

add_check(
    validation_records,
    "REC features",
    EXPECTED_REC_FEATURES,
    len(
        REC_FEATURES
    ),
    (
        len(
            REC_FEATURES
        ) == EXPECTED_REC_FEATURES
        and not missing_rec_features
    ),
)

add_check(
    validation_records,
    "Missing model training links",
    0,
    missing_model_training_links,
    missing_model_training_links == 0,
)

add_check(
    validation_records,
    "Missing model evaluation links",
    0,
    missing_model_evaluation_links,
    missing_model_evaluation_links == 0,
)

add_check(
    validation_records,
    "Model training verdict mismatches",
    0,
    model_training_verdict_mismatches,
    model_training_verdict_mismatches == 0,
)

add_check(
    validation_records,
    "Model evaluation verdict mismatches",
    0,
    model_evaluation_verdict_mismatches,
    model_evaluation_verdict_mismatches == 0,
)

add_check(
    validation_records,
    "Noise levels",
    NOISE_LEVELS,
    sorted(
        condition_plan[
            "NoisePercent"
        ].unique().tolist()
    ),
    sorted(
        condition_plan[
            "NoisePercent"
        ].unique().tolist()
    ) == NOISE_LEVELS,
)

add_check(
    validation_records,
    "Repetition seeds",
    REPETITION_SEEDS,
    sorted(
        condition_plan[
            "RepetitionSeed"
        ].unique().tolist()
    ),
    sorted(
        condition_plan[
            "RepetitionSeed"
        ].unique().tolist()
    ) == REPETITION_SEEDS,
)

add_check(
    validation_records,
    "Condition rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan
    ),
    len(
        condition_plan
    ) == EXPECTED_CONDITIONS,
)

add_check(
    validation_records,
    "Duplicate condition IDs",
    0,
    duplicate_condition_ids,
    duplicate_condition_ids == 0,
)

add_check(
    validation_records,
    "Duplicate condition coordinates",
    0,
    duplicate_condition_coordinates,
    duplicate_condition_coordinates == 0,
)

add_check(
    validation_records,
    "Nested-mask violations",
    0,
    nested_mask_violations,
    nested_mask_violations == 0,
)

add_check(
    validation_records,
    "Zero-noise conditions",
    len(
        REPETITION_SEEDS
    ),
    len(
        zero_noise_conditions
    ),
    len(
        zero_noise_conditions
    ) == len(
        REPETITION_SEEDS
    ),
)

add_check(
    validation_records,
    "Zero-noise flip violations",
    0,
    zero_noise_flip_violations,
    zero_noise_flip_violations == 0,
)

add_check(
    validation_records,
    "Zero-noise raw-label violations",
    0,
    zero_noise_raw_label_violations,
    zero_noise_raw_label_violations == 0,
)

add_check(
    validation_records,
    "Zero-noise model-label violations",
    0,
    zero_noise_model_label_violations,
    zero_noise_model_label_violations == 0,
)

add_check(
    validation_records,
    "Positive-noise conditions without raw changes",
    0,
    positive_noise_without_raw_changes,
    positive_noise_without_raw_changes == 0,
)

add_check(
    validation_records,
    "Positive-noise conditions without model changes",
    0,
    positive_noise_without_model_changes,
    positive_noise_without_model_changes == 0,
)

add_check(
    validation_records,
    "RNG-manifest rows",
    EXPECTED_RNG_ROWS,
    rng_readback_rows,
    rng_readback_rows == EXPECTED_RNG_ROWS,
)

add_check(
    validation_records,
    "Seed-manifest rows",
    len(
        REPETITION_SEEDS
    ),
    len(
        seed_manifest
    ),
    len(
        seed_manifest
    ) == len(
        REPETITION_SEEDS
    ),
)

add_check(
    validation_records,
    "Seed streams reproduced",
    True,
    (
        seed_streams_reproduced
        and stream_reproduction_failures == 0
    ),
    (
        seed_streams_reproduced
        and stream_reproduction_failures == 0
    ),
)

add_check(
    validation_records,
    "Raw-training readback rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training_readback
    ),
    len(
        raw_training_readback
    ) == EXPECTED_RAW_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Raw-evaluation readback rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation_readback
    ),
    len(
        raw_evaluation_readback
    ) == EXPECTED_RAW_EVAL_ROWS,
)

add_check(
    validation_records,
    "Model-training readback rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training_readback
    ),
    len(
        model_training_readback
    ) == EXPECTED_MODEL_TRAIN_ROWS,
)

add_check(
    validation_records,
    "Model-evaluation readback rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation_readback
    ),
    len(
        model_evaluation_readback
    ) == EXPECTED_MODEL_EVAL_ROWS,
)

add_check(
    validation_records,
    "Condition-plan readback rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan_readback
    ),
    len(
        condition_plan_readback
    ) == EXPECTED_CONDITIONS,
)

add_check(
    validation_records,
    "Seed-manifest readback rows",
    len(
        REPETITION_SEEDS
    ),
    len(
        seed_manifest_readback
    ),
    len(
        seed_manifest_readback
    ) == len(
        REPETITION_SEEDS
    ),
)

add_check(
    validation_records,
    "Nested-mask readback violations",
    0,
    nested_mask_readback_violations,
    nested_mask_readback_violations == 0,
)

add_check(
    validation_records,
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(
        registry
    ),
    len(
        registry
    ) == EXPECTED_REGISTERED_PROJECTS,
)

for required_number, required_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            registry_project_numbers.eq(
                required_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {required_number} frozen identity",
        required_project,
        actual_project,
        actual_project == required_project,
    )

add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    selection_active_reservations,
    selection_active_reservations
    == EXPECTED_ACTIVE_RESERVATIONS,
)

add_check(
    validation_records,
    "Project 22 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)

add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ),
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ) == EXPECTED_RUNTIME_PRIORITY_RULE,
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print("\nProject 22 Step 3A validation:")

display(
    validation
)


if not failed_validation.empty:
    print("\nFailed Step 3A checks:")

    display(
        failed_validation
    )

    print(
        "\nNo Step 3A checkpoint or PASS status was written."
    )

    raise RuntimeError(
        "PROJECT 22 STEP 3A VALIDATION FAILED."
    )


atomic_csv(
    STEP3A_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 13. REPORT, CHECKPOINT, AND STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


output_paths = [
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    FAILURE_SUBTYPE_PROFILE_PATH,
    SEED_MANIFEST_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    NESTED_MASK_AUDIT_PATH,
    PROTOCOL_PATH,
    STEP3A_VALIDATION_PATH,
]


output_manifest = [
    {
        "Path":
            str(
                path
            ),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SelectionCheckpointSHA256":
        selection_sha256,

    "RECCheckpointSHA256":
        rec_checkpoint_sha256,

    "SourceRootSHA256":
        current_source_root_sha256,

    "CleanAnchorReproducedDataset":
        clean_anchor_reproduced_dataset,

    "FrozenInferredExecutionOrder":
        str(
            INFERRED_EXECUTION_ORDER_PATH
        ),

    "FrozenInferredExecutionOrderSHA256":
        sha256_file(
            INFERRED_EXECUTION_ORDER_PATH
        ),

    "FrozenGlobalBuildOrder":
        str(
            FROZEN_GLOBAL_BUILD_ORDER_PATH
        ),

    "FrozenGlobalBuildOrderSHA256":
        sha256_file(
            FROZEN_GLOBAL_BUILD_ORDER_PATH
        ),

    "RawHistoryOrdering":
        "Project 22 Step 2B deterministic per-test execution order",

    "RawTrainingRows":
        len(
            raw_training
        ),

    "RawEvaluationRows":
        len(
            raw_evaluation
        ),

    "RawTrainingFailures":
        raw_training_failures,

    "RawEvaluationFailures":
        raw_evaluation_failures,

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "ModelTrainingFailures":
        model_training_failures,

    "ModelEvaluationFailures":
        model_evaluation_failures,

    "ModelFailingEvaluationBuilds":
        model_failing_evaluation_builds,

    "FailureSubtypes":
        failure_subtypes.astype(
            int
        ).tolist(),

    "FailureSubtypeProbabilities":
        failure_subtype_probabilities.tolist(),

    "NoiseLevels":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "Conditions":
        len(
            condition_plan
        ),

    "RNGManifestRows":
        rng_readback_rows,

    "NestedMaskViolations":
        nested_mask_violations,

    "ZeroNoiseFlipViolations":
        zero_noise_flip_violations,

    "ZeroNoiseModelLabelViolations":
        zero_noise_model_label_violations,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "OutputManifest":
        output_manifest,

    "RegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To21Modified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "PriorProjectConditionOutputsAccessed":
        False,

    "NoisePlanFrozen":
        True,

    "ModelsTrained":
        False,
}


atomic_json(
    STEP3A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "NoisePlanCheckpoint":
        True,

    "DoNotChangeCohorts":
        True,

    "DoNotChangeRandomStreams":
        True,

    "DoNotChangeConditionCoordinates":
        True,

    "EvaluationCohortImmutable":
        True,
}


atomic_json(
    NOISE_PLAN_CHECKPOINT_PATH,
    checkpoint_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "RawTrainingRows":
        len(
            raw_training
        ),

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "Conditions":
        len(
            condition_plan
        ),

    "RNGManifestRows":
        rng_readback_rows,

    "NestedMaskViolations":
        nested_mask_violations,

    "Checkpoint":
        str(
            NOISE_PLAN_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        sha256_file(
            NOISE_PLAN_CHECKPOINT_PATH
        ),

    "RegistryModified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "PriorProjectConditionOutputsAccessed":
        False,

    "ModelsTrained":
        False,
}


atomic_json(
    STEP3A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 14. FINAL IMMUTABILITY AND READBACK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 22 Step 3A."
    )


final_source_manifest = pd.DataFrame([
    {
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                (
                    SOURCE_DIR
                    / str(
                        row.RelativePath
                    )
                ).stat().st_size
            ),

        "SHA256":
            sha256_file(
                SOURCE_DIR
                / str(
                    row.RelativePath
                )
            ),
    }
    for row in frozen_source_manifest.itertuples(
        index=False
    )
])


if source_root_hash(
    final_source_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "The frozen Project 22 source changed during Step 3A."
    )


checkpoint_readback = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP3A_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP3A_STATUS:
    raise RuntimeError(
        "Project 22 noise-plan checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP3A_STATUS:
    raise RuntimeError(
        "Project 22 Step 3A status readback failed."
    )


# --------------------------------------------------------------------------------------------------
# 15. DISPLAY
# --------------------------------------------------------------------------------------------------

print("\nFailure-subtype profile:")

display(
    failure_subtype_profile
)


print("\nSeed manifest:")

display(
    seed_manifest
)


print("\nCondition-plan sample:")

display(
    pd.concat(
        [
            condition_plan.head(
                9
            ),
            condition_plan.tail(
                9
            ),
        ],
        ignore_index=True,
    )
)


print("\nNested-mask audit summary:")

display(
    nested_mask_audit.groupby(
        [
            "LowerNoisePercent",
            "HigherNoisePercent",
        ],
        as_index=False,
    ).agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        TotalViolations=(
            "Violations",
            "sum",
        ),

        AllPassed=(
            "Pass",
            "all",
        ),
    )
)


# --------------------------------------------------------------------------------------------------
# 16. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 132)
print("=== PROJECT 22 CELL 6 / STEP 3A RESULT ===")
print("=" * 132)


print("\nProject:")

print(
    PROJECT_NAME
)

print(
    "Project 11 identity:",
    required_registered_identities[
        11
    ],
)

print(
    "Project 12 identity:",
    required_registered_identities[
        12
    ],
)

print(
    "Project 13 identity:",
    required_registered_identities[
        13
    ],
)

print(
    "Project 14 identity:",
    required_registered_identities[
        14
    ],
)

print(
    "Project 15 identity:",
    required_registered_identities[
        15
    ],
)

print(
    "Project 16 identity:",
    required_registered_identities[
        16
    ],
)

print(
    "Project 17 identity:",
    required_registered_identities[
        17
    ],
)

print(
    "Project 18 identity:",
    required_registered_identities[
        18
    ],
)

print(
    "Project 19 identity:",
    required_registered_identities[
        19
    ],
)

print(
    "Project 20 identity:",
    required_registered_identities[
        20
    ],
)

print(
    "Project 21 identity:",
    required_registered_identities[
        21
    ],
)

print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)


print("\nFrozen clean-history order:")

print(
    "Inferred execution-order rows:",
    len(
        exe
    ),
)

print(
    "Global build-order rows:",
    len(
        frozen_global_build_order
    ),
)

print(
    "Raw duplicate Build-Test rows:",
    raw_duplicate_build_test_rows,
)

print(
    "Raw duplicate Test-order rows:",
    raw_duplicate_test_order_rows,
)

print("\nFixed cohorts:")

print(
    "Raw training rows:",
    len(
        raw_training
    ),
)

print(
    "Raw evaluation rows:",
    len(
        raw_evaluation
    ),
)

print(
    "Raw training failures:",
    raw_training_failures,
)

print(
    "Raw evaluation failures:",
    raw_evaluation_failures,
)

print(
    "Model training rows:",
    len(
        model_training
    ),
)

print(
    "Model evaluation rows:",
    len(
        model_evaluation
    ),
)

print(
    "Model training failures:",
    model_training_failures,
)

print(
    "Model evaluation failures:",
    model_evaluation_failures,
)

print(
    "Model failing evaluation builds:",
    model_failing_evaluation_builds,
)


print("\nNoise plan:")

print(
    "Noise levels:",
    NOISE_LEVELS,
)

print(
    "Repetition seeds:",
    len(
        REPETITION_SEEDS
    ),
)

print(
    "Conditions:",
    len(
        condition_plan
    ),
)

print(
    "RNG-manifest rows:",
    rng_readback_rows,
)

print(
    "Failure subtypes:",
    failure_subtypes.astype(
        int
    ).tolist(),
)

print(
    "Failure-subtype probabilities:",
    failure_subtype_probabilities.tolist(),
)

print(
    "Nested-mask violations:",
    nested_mask_violations,
)


print("\nZero-noise audit:")

print(
    "Zero-noise conditions:",
    len(
        zero_noise_conditions
    ),
)

print(
    "Zero-noise flip violations:",
    zero_noise_flip_violations,
)

print(
    "Zero-noise raw-label violations:",
    zero_noise_raw_label_violations,
)

print(
    "Zero-noise model-label violations:",
    zero_noise_model_label_violations,
)


print("\nImmutability and isolation:")

print(
    "Project 22 source unchanged:",
    source_root_hash(
        final_source_manifest
    ) == EXPECTED_SOURCE_ROOT_SHA256,
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–21 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Models trained:",
    False,
)


print("\nValidation:")

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print("\nNoise-plan checkpoint:")

print(
    NOISE_PLAN_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    sha256_file(
        NOISE_PLAN_CHECKPOINT_PATH
    ),
)


print(
    "\nSTATUS:",
    STEP3A_STATUS,
)

print("=" * 132)


=== PROJECT 22 CELL 6 / STEP 3A: DETERMINISTIC NOISE PLAN AND COHORT FREEZE ===

Project 22 Step 3A validation:


,Check,Expected,Actual,Pass
0,Step 1B passed,PASS_PROJECT_22_SELECTION_AND_SOURCE_FROZEN,PASS_PROJECT_22_SELECTION_AND_SOURCE_FROZEN,True
1,Step 2B passed,PASS_PROJECT_22_CLEAN_REC_RECONSTRUCTION_AND_A...,PASS_PROJECT_22_CLEAN_REC_RECONSTRUCTION_AND_A...,True
2,Selection checkpoint SHA-256,a7387d9495c71dce5d2c21ff08b0afd80d9c5251b5885a...,a7387d9495c71dce5d2c21ff08b0afd80d9c5251b5885a...,True
3,REC checkpoint SHA-256,8417249bc74e2a50e2f61cc3b7776dab7f731f62993e3e...,8417249bc74e2a50e2f61cc3b7776dab7f731f62993e3e...,True
4,REC checkpoint implementation,PROJECT_22_V2_ANCHOR_AWARE_TIE_INFERENCE_FULL_...,PROJECT_22_V2_ANCHOR_AWARE_TIE_INFERENCE_FULL_...,True
...,...,...,...,...
72,Project 20 frozen identity,apache@curator,apache@curator,True
73,Project 21 frozen identity,facebook@buck,facebook@buck,True
74,Active reservations,[],[],True
75,Project 22 registry rows,0,0,True



Failure-subtype profile:


,FailureSubtype,CleanTrainingRows,Probability
0,1,2,0.009615
1,2,206,0.990385



Seed manifest:


...


Condition-plan sample:


,ConditionOrder,ConditionID,SeedOrder,NoiseOrderWithinSeed,NoisePercent,RepetitionSeed,FlipSeed,FailureSubtypeSeed,RawTrainingRows,NumberFlipped,...,FailureToPass,CleanRawFailures,NoisyRawFailures,ModelTrainingRows,ModelLabelChanges,CleanModelFailures,NoisyModelFailures,FlipMaskSHA256,NoisyRawVerdictSHA256,NoisyModelVerdictSHA256
0,1,noise_00__seed_01,1,1,0,1,569658115,3621606888,172628,0,...,0,208,208,95812,0,207,207,44ba26b8a50bc62ddf6ae8437d93c32690adea96694e78...,d17c04b5e3393ed5d1a07085dd4b0cfdc4fe3c7257f508...,d0c30684c7d9e93579ed67b4d5919f2264a0bf9fdfa611...
1,2,noise_05__seed_01,1,2,5,1,569658115,3621606888,172628,8769,...,9,208,8959,95812,4862,207,5051,0b9d4619a0a130a13cd96baba2988f22c5df4d22fdb429...,11c9d0261059fc12c923a6964735f9d7bc97259341395d...,d3a2dcd391d07786bf32373ef48dcd07edfb13a81ed5e5...
2,3,noise_10__seed_01,1,3,10,1,569658115,3621606888,172628,17495,...,17,208,17669,95812,9740,207,9913,8a1ed938dc6a82f212080356a5b7668d0403021b7cf296...,c409c5dac9001e66e773be5e36b38fe323aa15ec3176c3...,155927adeb57bdf13e1f8bdfa98ae6ecdb2515755fd9e6...
3,4,noise_15__seed_01,1,4,15,1,569658115,3621606888,172628,26177,...,31,208,26323,95812,14472,207,14617,4aee172f6820ee16ad8e5a1ade2b919545f63adebb3b5b...,b91b19ede90b019a72e34898ff29f7fe77ab7dbb227b30...,81c93b6f5e4281da19f4ea2e3f4f33f71357b2c71c903d...
4,5,noise_20__seed_01,1,5,20,1,569658115,3621606888,172628,34950,...,43,208,35072,95812,19381,207,19502,e0db26b205b7b9b0745d2f5da0fe8f48050a2436f950b2...,9822b43c6b9e37ac6d75af65856f2b88f18b7249d528d2...,9b3486d6d315f46a400da830dde664a925146aae9935cc...
5,6,noise_25__seed_01,1,6,25,1,569658115,3621606888,172628,43419,...,54,208,43519,95812,24127,207,24226,66a331f822a71244edcac0794f93ddb02d434228e9fbb3...,e8e774f5f4e843d98773ef5178f0886e10477a6a832e2d...,f74f906e4badb418db737649fcd0925b79cd83a6f3c59e...
6,7,noise_30__seed_01,1,7,30,1,569658115,3621606888,172628,51968,...,62,208,52052,95812,28819,207,28902,15ac5fddfccf851ae5428e6f577485df0428c4e7054e04...,981f9b4a39b0477e4ae0c9daa5eb75f11196487742bf69...,997330bb64c9b407e64393b74fa45537b96c3d354acede...
7,8,noise_40__seed_01,1,8,40,1,569658115,3621606888,172628,69250,...,91,208,69276,95812,38354,207,38379,68d920cb637b34ca0d32125f54951e863c207af2a41fcc...,fcff7d170f09f9d1e925788d2d6ac626d4715344b05d47...,f791927ef1a625292c468a7f0f01125afcacb5b982b81d...
8,9,noise_50__seed_01,1,9,50,1,569658115,3621606888,172628,86584,...,113,208,86566,95812,47909,207,47890,69a924875f9d498875687219ad5ca2cc5544dc0a67f21d...,4f3620f40819ee20e82327469a7fdeda3bacc8bef3fe32...,85989ce61ef05d972aaf73187f9dc5348bd6e6fc51e603...
9,262,noise_00__seed_30,30,1,0,30,2331445157,3265812134,172628,0,...,0,208,208,95812,0,207,207,44ba26b8a50bc62ddf6ae8437d93c32690adea96694e78...,d17c04b5e3393ed5d1a07085dd4b0cfdc4fe3c7257f508...,d0c30684c7d9e93579ed67b4d5919f2264a0bf9fdfa611...



Nested-mask audit summary:


,LowerNoisePercent,HigherNoisePercent,Seeds,TotalViolations,AllPassed
0,0,5,30,0,True
1,5,10,30,0,True
2,10,15,30,0,True
3,15,20,30,0,True
4,20,25,30,0,True
5,25,30,30,0,True
6,30,40,30,0,True
7,40,50,30,0,True




=== PROJECT 22 CELL 6 / STEP 3A RESULT ===

Project:
apache@logging-log4j2
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Project 17 identity: yamcs@Yamcs
Project 18 identity: cantaloupe-project@cantaloupe
Project 19 identity: EMResearch@EvoMaster
Project 20 identity: apache@curator
Project 21 identity: facebook@buck
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']

Frozen clean-history order:
Inferred execution-order rows: 240253
Global build-order rows: 441
Raw duplicate Build-Test rows: 0
Raw duplicate Test-order rows: 0

Fixed cohorts:
Raw training rows: 172628
Raw evaluation rows: 67625
Raw training failures: 208
Raw evaluation failures: 40
Model training rows: 95812
Model evaluation rows: 22

In [9]:
# ==================================================================================================
# PROJECT 22 — CELL 7 / STEP 4A
# EXPERIMENT RUNTIME, MODEL, BASELINE, METRIC, AND PREDICTOR CONTRACT FREEZE
#
# PROJECT:
#   apache@logging-log4j2
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME Thesis_project_22.ipynb NOTEBOOK.
#
# PURPOSE:
# - verify the frozen Project 22 Step 3A noise plan and every output in its manifest;
# - validate the fixed 151-predictor training/evaluation matrices;
# - freeze runtime versions, median-imputation, labels, ranking, model, baseline,
#   APFD, and APFDc contracts;
# - validate all four required model implementations without fitting Project 22 models;
# - write the runtime-contract checkpoint required before the two-condition smoke test.
#
# SAFETY:
# - no model fitting;
# - no condition execution;
# - no completion-registry write;
# - no prior-project condition-output access.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from IPython.display import display

import hashlib
import json
import os
import platform
import sys

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


print("=" * 136)
print("=== PROJECT 22 CELL 7 / STEP 4A: EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 21
PROJECT_NAME = "apache@logging-log4j2"
PROJECT_SLUG = "apache__logging-log4j2"
PROJECT_SHORT = "LOG4J2"

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_22_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

STEP4A_STATUS = (
    "PASS_PROJECT_22_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
)

EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "3dd2b3f38a8d7b77c179e9f518399c1dc5b1c89a8a29607ea31fd78a91f40666"
)

EXPECTED_REGISTRY_SHA256 = (
    "79cd6ecb595c5e8ae91a9494e469792716338d144308560a62caf1b9342306b2"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64f842964c896c6ac334"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

EXPECTED_REGISTERED_PROJECTS = 21

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_RAW_TRAIN_ROWS = 172_628
EXPECTED_RAW_EVAL_ROWS = 67_625
EXPECTED_MODEL_TRAIN_ROWS = 95_812
EXPECTED_MODEL_EVAL_ROWS = 22_156
EXPECTED_MODEL_TRAIN_FAILURES = 207
EXPECTED_MODEL_EVAL_FAILURES = 40
EXPECTED_MODEL_FAILING_EVAL_BUILDS = 39

EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_CONDITIONS = 270

EXPECTED_RUNTIME_VERSIONS = {
    "Python": "3.12.13",
    "numpy": "2.0.2",
    "pandas": "2.2.2",
    "scikit-learn": "1.6.1",
    "xgboost": "3.3.0",
    "lightgbm": "4.6.0",
    "pyarrow": "18.1.0",
}

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "max_features": "sqrt",
        "bootstrap": True,
        "n_jobs": -1,
    },
    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
        "eval_metric": "logloss",
    },
    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    },
    "NaiveBayes": {
        "var_smoothing": 1e-9,
    },
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_22_selection"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_22_frozen_source_manifest.csv"
)

SOURCE_DIR = Path(
    "/content/datasets/datasets/apache@logging-log4j2"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

STEP3A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step3a_status.json"
)

STEP3A_REPORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_report.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_22_noise_plan_checkpoint.json"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

PROTOCOL_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_frozen_experiment_protocol.json"
)

RUNTIME_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_runtime_contract"
)

RUNTIME_VERSION_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_runtime_versions.csv"
)

PREDICTOR_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_predictor_contract.csv"
)

CLEAN_MEDIAN_REFERENCE_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_clean_training_median_reference.csv"
)

MODEL_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_model_contract.json"
)

BASELINE_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_baseline_contract.json"
)

METRIC_SELF_TEST_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_metric_self_test.csv"
)

RANKING_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_ranking_contract.json"
)

STEP4A_VALIDATION_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_validation.csv"
)

STEP4A_REPORT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_report.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step4a_status.json"
)

RUNTIME_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_22_runtime_contract_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_json(
    path,
    payload,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def deterministic_seed(
    repetition_seed,
    stream_name,
):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(
        material
    ).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )


def deterministic_random_build_seed(
    repetition_seed,
    build_id,
):
    return deterministic_seed(
        repetition_seed,
        f"Random_baseline_build_{int(build_id)}",
    )


def create_models(
    repetition_seed,
):
    return {
        "RandomForest":
            RandomForestClassifier(
                **MODEL_CONFIG[
                    "RandomForest"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "RandomForest_model",
                ),
            ),

        "XGBoost":
            XGBClassifier(
                **MODEL_CONFIG[
                    "XGBoost"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "XGBoost_model",
                ),
            ),

        "LightGBM":
            LGBMClassifier(
                **MODEL_CONFIG[
                    "LightGBM"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "LightGBM_model",
                ),
            ),

        "NaiveBayes":
            GaussianNB(
                **MODEL_CONFIG[
                    "NaiveBayes"
                ]
            ),
    }


def calculate_apfd(
    failures,
):
    failures = np.asarray(
        failures,
        dtype=np.int8,
    )

    number_of_tests = len(
        failures
    )

    number_of_failures = int(
        failures.sum()
    )

    if (
        number_of_tests == 0
        or number_of_failures == 0
    ):
        return np.nan

    failure_positions = (
        np.flatnonzero(
            failures == 1
        )
        + 1
    )

    return float(
        1.0
        - (
            failure_positions.sum()
            / (
                number_of_tests
                * number_of_failures
            )
        )
        + (
            1.0
            / (
                2.0
                * number_of_tests
            )
        )
    )


def calculate_apfdc(
    failures,
    durations,
):
    failures = np.asarray(
        failures,
        dtype=np.int8,
    )

    durations = np.asarray(
        durations,
        dtype=float,
    )

    if len(failures) != len(durations):
        raise ValueError(
            "failures and durations must have equal length."
        )

    if (
        len(failures) == 0
        or failures.sum() == 0
    ):
        return np.nan

    if not np.isfinite(
        durations
    ).all():
        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (
        durations < 0
    ).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(
        durations.sum()
    )

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array([0.0]),
        np.cumsum(
            durations
        )[:-1],
    ])

    failure_mask = (
        failures == 1
    )

    midpoint_detection_times = (
        cumulative_before[
            failure_mask
        ]
        + (
            0.5
            * durations[
                failure_mask
            ]
        )
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(passed),
    })


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND STEP 3A CHECKPOINT
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    STEP3A_STATUS_PATH,
    STEP3A_REPORT_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    CONDITION_PLAN_PATH,
    PROTOCOL_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 22 Step 4A inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


noise_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)

noise_checkpoint = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)

step3a_status = load_json(
    STEP3A_STATUS_PATH
)

step3a_report = load_json(
    STEP3A_REPORT_PATH
)

protocol = load_json(
    PROTOCOL_PATH
)


if (
    noise_checkpoint_sha256
    != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 22 noise-plan checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256}\n"
        f"Actual:   {noise_checkpoint_sha256}"
    )


for label, payload in [
    (
        "noise checkpoint",
        noise_checkpoint,
    ),
    (
        "Step 3A status",
        step3a_status,
    ),
    (
        "Step 3A report",
        step3a_report,
    ),
]:
    if payload.get(
        "Status"
    ) != EXPECTED_STEP3A_STATUS:
        raise RuntimeError(
            f"{label} does not contain the expected Step 3A PASS status."
        )


if (
    noise_checkpoint.get(
        "Project"
    )
    != PROJECT_NAME
    or noise_checkpoint.get(
        "ProjectSlug"
    )
    != PROJECT_SLUG
):
    raise RuntimeError(
        "The frozen Step 3A Project 22 identity differs."
    )


if (
    noise_checkpoint.get(
        "SourceRootSHA256"
    )
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The Step 3A checkpoint source root differs."
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY STEP 3A OUTPUT MANIFEST
# --------------------------------------------------------------------------------------------------

output_manifest = noise_checkpoint.get(
    "OutputManifest",
    []
)


if not isinstance(
    output_manifest,
    list,
) or not output_manifest:
    raise RuntimeError(
        "The Step 3A checkpoint contains no output manifest."
    )


output_manifest_records = []


for item in output_manifest:
    path = Path(
        item[
            "Path"
        ]
    )

    expected_bytes = int(
        item[
            "Bytes"
        ]
    )

    expected_sha256 = str(
        item[
            "SHA256"
        ]
    )

    exists = path.is_file()

    actual_bytes = (
        int(
            path.stat().st_size
        )
        if exists
        else -1
    )

    actual_sha256 = (
        sha256_file(
            path
        )
        if exists
        else "MISSING"
    )

    output_manifest_records.append({
        "Path":
            str(path),

        "ExpectedBytes":
            expected_bytes,

        "ActualBytes":
            actual_bytes,

        "ExpectedSHA256":
            expected_sha256,

        "ActualSHA256":
            actual_sha256,

        "Pass":
            (
                exists
                and actual_bytes
                == expected_bytes
                and actual_sha256
                == expected_sha256
            ),
    })


output_manifest_audit = pd.DataFrame(
    output_manifest_records
)


output_manifest_failures = int(
    (
        ~output_manifest_audit[
            "Pass"
        ]
    ).sum()
)


if output_manifest_failures:
    print(
        "\nFailed Step 3A output-manifest checks:"
    )

    display(
        output_manifest_audit.loc[
            ~output_manifest_audit[
                "Pass"
            ]
        ]
    )

    raise RuntimeError(
        "One or more frozen Step 3A outputs changed."
    )


# --------------------------------------------------------------------------------------------------
# 6. SOURCE AND REGISTRY IMMUTABILITY
# --------------------------------------------------------------------------------------------------

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_records = []


for row in frozen_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            "A frozen Project 22 source file is missing:\n"
            f"{source_path}"
        )

    current_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


current_source_manifest = pd.DataFrame(
    current_source_records
)


current_source_root_sha256 = source_root_hash(
    current_source_manifest
)


if (
    current_source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The frozen Project 22 source root differs."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "The registry does not contain exactly Projects 1–21."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–21 are not all COMPLETE_AND_FROZEN."
    )


if project_numbers.eq(PROJECT_NUMBER).any():
    raise RuntimeError(
        "Project 22 is unexpectedly already registered."
    )


if registry[project_column].eq(PROJECT_NAME).any():
    raise RuntimeError(
        "The selected Project 22 identity is already registered."
    )


required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


active_reservations = []


if active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "The active-reservation state differs from the Project 22 freeze."
    )


# --------------------------------------------------------------------------------------------------
# 7. LOAD AND VALIDATE FIXED COHORTS
# --------------------------------------------------------------------------------------------------

raw_training = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)

raw_evaluation = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)

model_training = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)

model_evaluation = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)

model_raw_train_link = pd.read_parquet(
    MODEL_RAW_TRAIN_LINK_PATH
)

model_raw_eval_link = pd.read_parquet(
    MODEL_RAW_EVAL_LINK_PATH
)

condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)


required_model_columns = {
    "Build",
    "Test",
    "Verdict",
}


if not required_model_columns.issubset(
    model_training.columns
) or not required_model_columns.issubset(
    model_evaluation.columns
):
    raise RuntimeError(
        "Fixed model cohorts are missing Build, Test, or Verdict."
    )


training_metadata_columns = {
    "ModelTrainingRowOrder",
    "Build",
    "Test",
    "Verdict",
}


evaluation_metadata_columns = {
    "ModelEvaluationRowOrder",
    "Build",
    "Test",
    "Verdict",
}


predictor_columns = [
    column
    for column in model_training.columns
    if column not in training_metadata_columns
]


evaluation_predictor_columns = [
    column
    for column in model_evaluation.columns
    if column not in evaluation_metadata_columns
]


if (
    predictor_columns
    != evaluation_predictor_columns
):
    raise RuntimeError(
        "Training and evaluation predictor order differs."
    )


if len(
    predictor_columns
) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "The fixed predictor count differs.\n"
        f"Expected: {EXPECTED_PREDICTORS}\n"
        f"Actual:   {len(predictor_columns)}"
    )


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in predictor_columns
]


if missing_rec_features:
    raise RuntimeError(
        "Fixed predictor cohorts are missing REC features:\n"
        + "\n".join(
            missing_rec_features
        )
    )


model_training_failures = int(
    pd.to_numeric(
        model_training[
            "Verdict"
        ],
        errors="raise",
    ).ne(
        0
    ).sum()
)


model_evaluation_failures = int(
    pd.to_numeric(
        model_evaluation[
            "Verdict"
        ],
        errors="raise",
    ).ne(
        0
    ).sum()
)


model_failing_evaluation_builds = int(
    model_evaluation.loc[
        pd.to_numeric(
            model_evaluation[
                "Verdict"
            ],
            errors="raise",
        ).ne(
            0
        ),
        "Build",
    ].nunique()
)


# --------------------------------------------------------------------------------------------------
# 8. NUMERIC PREDICTORS AND MEDIAN IMPUTATION
# --------------------------------------------------------------------------------------------------

training_numeric = pd.DataFrame(
    index=model_training.index
)

evaluation_numeric = pd.DataFrame(
    index=model_evaluation.index
)

predictor_profile_records = []


for predictor_order, column in enumerate(
    predictor_columns,
    start=1,
):
    training_values = pd.to_numeric(
        model_training[
            column
        ],
        errors="coerce",
    ).astype(
        float
    ).replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )

    evaluation_values = pd.to_numeric(
        model_evaluation[
            column
        ],
        errors="coerce",
    ).astype(
        float
    ).replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )

    training_numeric[
        column
    ] = training_values

    evaluation_numeric[
        column
    ] = evaluation_values

    predictor_profile_records.append({
        "PredictorOrder":
            predictor_order,

        "Predictor":
            column,

        "IsREC":
            column in REC_FEATURES,

        "RECClass":
            (
                "VERDICT_DEPENDENT"
                if column
                in VERDICT_DEPENDENT_REC
                else (
                    "VERDICT_INDEPENDENT"
                    if column
                    in VERDICT_INDEPENDENT_REC
                    else ""
                )
            ),

        "TrainingRows":
            len(
                training_values
            ),

        "TrainingNonMissing":
            int(
                training_values.notna().sum()
            ),

        "TrainingMissing":
            int(
                training_values.isna().sum()
            ),

        "EvaluationRows":
            len(
                evaluation_values
            ),

        "EvaluationMissing":
            int(
                evaluation_values.isna().sum()
            ),

        "AllTrainingValuesMissing":
            bool(
                training_values.notna().sum()
                == 0
            ),
    })


predictor_contract = pd.DataFrame(
    predictor_profile_records
)


all_missing_predictors = predictor_contract.loc[
    predictor_contract[
        "AllTrainingValuesMissing"
    ],
    "Predictor",
].tolist()


if all_missing_predictors:
    raise RuntimeError(
        "One or more predictors are entirely missing in training:\n"
        + "\n".join(
            all_missing_predictors
        )
    )


clean_training_medians = training_numeric.median(
    axis=0,
    skipna=True,
)


if (
    clean_training_medians.isna().any()
    or not np.isfinite(
        clean_training_medians.to_numpy(
            dtype=float
        )
    ).all()
):
    raise RuntimeError(
        "Clean training medians contain missing or infinite values."
    )


training_imputed = training_numeric.fillna(
    clean_training_medians
)

evaluation_imputed = evaluation_numeric.fillna(
    clean_training_medians
)


training_nonfinite_after_imputation = int(
    (
        ~np.isfinite(
            training_imputed.to_numpy(
                dtype=float
            )
        )
    ).sum()
)


evaluation_nonfinite_after_imputation = int(
    (
        ~np.isfinite(
            evaluation_imputed.to_numpy(
                dtype=float
            )
        )
    ).sum()
)


clean_median_reference = pd.DataFrame({
    "PredictorOrder":
        np.arange(
            1,
            len(
                predictor_columns
            )
            + 1,
            dtype=np.int64,
        ),

    "Predictor":
        predictor_columns,

    "CleanTrainingMedian":
        clean_training_medians[
            predictor_columns
        ].to_numpy(
            dtype=float
        ),
})


# --------------------------------------------------------------------------------------------------
# 9. LABEL CONTRACT
# --------------------------------------------------------------------------------------------------

training_binary_labels = (
    pd.to_numeric(
        model_training[
            "Verdict"
        ],
        errors="raise",
    )
    .ne(
        0
    )
    .astype(
        np.int8
    )
)


evaluation_binary_labels = (
    pd.to_numeric(
        model_evaluation[
            "Verdict"
        ],
        errors="raise",
    )
    .ne(
        0
    )
    .astype(
        np.int8
    )
)


training_label_values = sorted(
    training_binary_labels.unique().tolist()
)


evaluation_label_values = sorted(
    evaluation_binary_labels.unique().tolist()
)


# --------------------------------------------------------------------------------------------------
# 10. RUNTIME VERSION CONTRACT
# --------------------------------------------------------------------------------------------------

runtime_versions = pd.DataFrame([
    {
        "Component":
            "Python",

        "Version":
            platform.python_version(),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "Python"
            ],
    },
    {
        "Component":
            "numpy",

        "Version":
            np.__version__,

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "numpy"
            ],
    },
    {
        "Component":
            "pandas",

        "Version":
            pd.__version__,

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "pandas"
            ],
    },
    {
        "Component":
            "scikit-learn",

        "Version":
            metadata.version(
                "scikit-learn"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "scikit-learn"
            ],
    },
    {
        "Component":
            "xgboost",

        "Version":
            metadata.version(
                "xgboost"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "xgboost"
            ],
    },
    {
        "Component":
            "lightgbm",

        "Version":
            metadata.version(
                "lightgbm"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "lightgbm"
            ],
    },
    {
        "Component":
            "pyarrow",

        "Version":
            metadata.version(
                "pyarrow"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "pyarrow"
            ],
    },
])


runtime_versions[
    "Pass"
] = runtime_versions[
    "Version"
].eq(
    runtime_versions[
        "ExpectedVersion"
    ]
)


runtime_version_failures = int(
    (
        ~runtime_versions[
            "Pass"
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 11. MODEL IMPLEMENTATION CONTRACT
# --------------------------------------------------------------------------------------------------

models_seed_1 = create_models(
    repetition_seed=1
)

models_seed_1_repeat = create_models(
    repetition_seed=1
)

models_seed_2 = create_models(
    repetition_seed=2
)


model_contract_records = []


for technique in ML_TECHNIQUES:
    model_a = models_seed_1[
        technique
    ]

    model_b = models_seed_1_repeat[
        technique
    ]

    model_c = models_seed_2[
        technique
    ]

    parameters_a = model_a.get_params(
        deep=False
    )

    parameters_b = model_b.get_params(
        deep=False
    )

    parameters_c = model_c.get_params(
        deep=False
    )

    random_state_a = parameters_a.get(
        "random_state",
        None,
    )

    random_state_c = parameters_c.get(
        "random_state",
        None,
    )

    model_contract_records.append({
        "Technique":
            technique,

        "EstimatorClass":
            (
                f"{model_a.__class__.__module__}."
                f"{model_a.__class__.__name__}"
            ),

        "Seed1RandomState":
            random_state_a,

        "Seed2RandomState":
            random_state_c,

        "SameSeedSameConfiguration":
            parameters_a
            == parameters_b,

        "DifferentSeedStateAsExpected":
            (
                True
                if technique
                == "NaiveBayes"
                else random_state_a
                != random_state_c
            ),

        "ConfigurationJSON":
            json.dumps(
                parameters_a,
                sort_keys=True,
                default=str,
            ),
    })


model_contract_table = pd.DataFrame(
    model_contract_records
)


rf_params = models_seed_1[
    "RandomForest"
].get_params(
    deep=False
)

xgb_params = models_seed_1[
    "XGBoost"
].get_params(
    deep=False
)

lgbm_params = models_seed_1[
    "LightGBM"
].get_params(
    deep=False
)

nb_params = models_seed_1[
    "NaiveBayes"
].get_params(
    deep=False
)


model_parameter_checks = {
    "RandomForest": (
        rf_params.get(
            "n_estimators"
        )
        == 100
        and rf_params.get(
            "max_features"
        )
        == "sqrt"
        and rf_params.get(
            "bootstrap"
        )
        is True
        and rf_params.get(
            "n_jobs"
        )
        == -1
    ),

    "XGBoost": (
        xgb_params.get(
            "n_estimators"
        )
        == 100
        and xgb_params.get(
            "max_depth"
        )
        == 6
        and np.isclose(
            float(
                xgb_params.get(
                    "learning_rate"
                )
            ),
            0.1,
        )
        and xgb_params.get(
            "tree_method"
        )
        == "hist"
        and xgb_params.get(
            "n_jobs"
        )
        == -1
    ),

    "LightGBM": (
        lgbm_params.get(
            "n_estimators"
        )
        == 100
        and np.isclose(
            float(
                lgbm_params.get(
                    "learning_rate"
                )
            ),
            0.1,
        )
        and lgbm_params.get(
            "num_leaves"
        )
        == 31
        and lgbm_params.get(
            "deterministic"
        )
        is True
        and lgbm_params.get(
            "force_col_wise"
        )
        is True
        and lgbm_params.get(
            "n_jobs"
        )
        == -1
    ),

    "NaiveBayes": (
        np.isclose(
            float(
                nb_params.get(
                    "var_smoothing"
                )
            ),
            1e-9,
        )
    ),
}


model_contract_failures = int(
    (
        ~model_contract_table[
            "SameSeedSameConfiguration"
        ]
        | ~model_contract_table[
            "DifferentSeedStateAsExpected"
        ]
    ).sum()
    + sum(
        not bool(value)
        for value in model_parameter_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 12. BASELINE, RANKING, AND RANDOM CONTRACT
# --------------------------------------------------------------------------------------------------

sample_build_id = int(
    model_evaluation[
        "Build"
    ].iloc[
        0
    ]
)


random_seed_1_a = deterministic_random_build_seed(
    repetition_seed=1,
    build_id=sample_build_id,
)

random_seed_1_b = deterministic_random_build_seed(
    repetition_seed=1,
    build_id=sample_build_id,
)

random_seed_2 = deterministic_random_build_seed(
    repetition_seed=2,
    build_id=sample_build_id,
)


sample_random_a = np.random.default_rng(
    random_seed_1_a
).random(
    100
)

sample_random_b = np.random.default_rng(
    random_seed_1_b
).random(
    100
)

sample_random_c = np.random.default_rng(
    random_seed_2
).random(
    100
)


random_same_seed_reproduced = bool(
    np.array_equal(
        sample_random_a,
        sample_random_b,
    )
)


random_different_seed_differs = bool(
    not np.array_equal(
        sample_random_a,
        sample_random_c,
    )
)


ranking_contract = {
    "ML": {
        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",
    },

    "Random": {
        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",

        "SeedRule":
            (
                "first little-endian uint32 of "
                "SHA-256(project|repetition_seed|"
                "Random_baseline_build_<BuildID>)"
            ),

        "ConstantAcrossNoiseForSameSeedAndBuild":
            True,
    },

    "LatestFail": {
        "SourceFeature":
            "REC_LastFailureAge",

        "ScoreFormula":
            "-REC_LastFailureAge",

        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",

        "NoiseDependent":
            True,

        "UsesSameCorruptedHistoryAsML":
            True,
    },

    "QTF-Avg": {
        "SourceFeature":
            "REC_TotalAvgExeTime",

        "Direction":
            "ascending",

        "TieBreak":
            "Test ascending",

        "NoiseDependent":
            False,
    },
}


baseline_contract = {
    "Techniques":
        BASELINE_TECHNIQUES,

    "Random":
        ranking_contract[
            "Random"
        ],

    "LatestFail":
        ranking_contract[
            "LatestFail"
        ],

    "QTF-Avg":
        ranking_contract[
            "QTF-Avg"
        ],

    "NoRollingRetraining":
        True,

    "CleanEvaluationPartition":
        True,
}


# --------------------------------------------------------------------------------------------------
# 13. APFD/APFDc SELF-TESTS
# --------------------------------------------------------------------------------------------------

manual_failures = np.array([
    1,
    1,
    0,
    0,
    0,
], dtype=np.int8)


manual_apfd = calculate_apfd(
    manual_failures
)


manual_apfdc_slow_failure_first = calculate_apfdc(
    manual_failures,
    np.array([
        5.0,
        1.0,
        1.0,
        1.0,
        1.0,
    ]),
)


manual_apfdc_fast_failure_first = calculate_apfdc(
    manual_failures,
    np.array([
        1.0,
        5.0,
        1.0,
        1.0,
        1.0,
    ]),
)


all_pass_apfd = calculate_apfd(
    np.array([
        0,
        0,
        0,
    ])
)


all_pass_apfdc = calculate_apfdc(
    np.array([
        0,
        0,
        0,
    ]),
    np.array([
        1.0,
        1.0,
        1.0,
    ]),
)


metric_self_test = pd.DataFrame([
    {
        "Check":
            "Manual APFD",

        "Expected":
            0.8,

        "Actual":
            manual_apfd,

        "Pass":
            np.isclose(
                manual_apfd,
                0.8,
                rtol=0,
                atol=1e-15,
            ),
    },

    {
        "Check":
            "APFDc rewards quick failing test first",

        "Expected":
            True,

        "Actual":
            manual_apfdc_fast_failure_first
            > manual_apfdc_slow_failure_first,

        "Pass":
            manual_apfdc_fast_failure_first
            > manual_apfdc_slow_failure_first,
    },

    {
        "Check":
            "All-pass APFD is NaN",

        "Expected":
            True,

        "Actual":
            bool(
                np.isnan(
                    all_pass_apfd
                )
            ),

        "Pass":
            bool(
                np.isnan(
                    all_pass_apfd
                )
            ),
    },

    {
        "Check":
            "All-pass APFDc is NaN",

        "Expected":
            True,

        "Actual":
            bool(
                np.isnan(
                    all_pass_apfdc
                )
            ),

        "Pass":
            bool(
                np.isnan(
                    all_pass_apfdc
                )
            ),
    },
])


metric_self_test_failures = int(
    (
        ~metric_self_test[
            "Pass"
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 14. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 3A status",
    EXPECTED_STEP3A_STATUS,
    step3a_status.get(
        "Status"
    ),
    step3a_status.get(
        "Status"
    )
    == EXPECTED_STEP3A_STATUS,
)


add_check(
    validation_records,
    "Noise-plan checkpoint SHA-256",
    EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
    noise_checkpoint_sha256,
    noise_checkpoint_sha256
    == EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
)


add_check(
    validation_records,
    "Step 3A output-manifest failures",
    0,
    output_manifest_failures,
    output_manifest_failures
    == 0,
)


add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)


add_check(
    validation_records,
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training
    ),
    len(
        raw_training
    )
    == EXPECTED_RAW_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation
    ),
    len(
        raw_evaluation
    )
    == EXPECTED_RAW_EVAL_ROWS,
)


add_check(
    validation_records,
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training
    ),
    len(
        model_training
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation
    ),
    len(
        model_evaluation
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)


add_check(
    validation_records,
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    model_training_failures,
    model_training_failures
    == EXPECTED_MODEL_TRAIN_FAILURES,
)


add_check(
    validation_records,
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    model_evaluation_failures,
    model_evaluation_failures
    == EXPECTED_MODEL_EVAL_FAILURES,
)


add_check(
    validation_records,
    "Model failing evaluation builds",
    EXPECTED_MODEL_FAILING_EVAL_BUILDS,
    model_failing_evaluation_builds,
    model_failing_evaluation_builds
    == EXPECTED_MODEL_FAILING_EVAL_BUILDS,
)


add_check(
    validation_records,
    "Condition-plan rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan
    ),
    len(
        condition_plan
    )
    == EXPECTED_CONDITIONS,
)


add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    )
    == EXPECTED_PREDICTORS,
)


add_check(
    validation_records,
    "REC features",
    EXPECTED_REC_FEATURES,
    len(
        REC_FEATURES
    ),
    (
        len(
            REC_FEATURES
        )
        == EXPECTED_REC_FEATURES
        and not missing_rec_features
    ),
)


add_check(
    validation_records,
    "Raw-training link rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_raw_train_link
    ),
    len(
        model_raw_train_link
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Raw-evaluation link rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_raw_eval_link
    ),
    len(
        model_raw_eval_link
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)


add_check(
    validation_records,
    "Training binary label values",
    [0, 1],
    training_label_values,
    training_label_values
    == [
        0,
        1,
    ],
)


add_check(
    validation_records,
    "Evaluation binary label values",
    [0, 1],
    evaluation_label_values,
    evaluation_label_values
    == [
        0,
        1,
    ],
)


add_check(
    validation_records,
    "All-missing predictors",
    0,
    len(
        all_missing_predictors
    ),
    len(
        all_missing_predictors
    )
    == 0,
)


add_check(
    validation_records,
    "Training non-finite values after imputation",
    0,
    training_nonfinite_after_imputation,
    training_nonfinite_after_imputation
    == 0,
)


add_check(
    validation_records,
    "Evaluation non-finite values after imputation",
    0,
    evaluation_nonfinite_after_imputation,
    evaluation_nonfinite_after_imputation
    == 0,
)


add_check(
    validation_records,
    "Runtime-version failures",
    0,
    runtime_version_failures,
    runtime_version_failures
    == 0,
)


add_check(
    validation_records,
    "Model contract failures",
    0,
    model_contract_failures,
    model_contract_failures
    == 0,
)


add_check(
    validation_records,
    "Metric self-test failures",
    0,
    metric_self_test_failures,
    metric_self_test_failures
    == 0,
)


add_check(
    validation_records,
    "Random same-seed reproducible",
    True,
    random_same_seed_reproduced,
    random_same_seed_reproduced,
)


add_check(
    validation_records,
    "Random different-seed differs",
    True,
    random_different_seed_differs,
    random_different_seed_differs,
)


add_check(
    validation_records,
    "LatestFail feature present",
    True,
    (
        "REC_LastFailureAge"
        in predictor_columns
    ),
    (
        "REC_LastFailureAge"
        in predictor_columns
    ),
)


add_check(
    validation_records,
    "QTF-Avg feature present",
    True,
    (
        "REC_TotalAvgExeTime"
        in predictor_columns
    ),
    (
        "REC_TotalAvgExeTime"
        in predictor_columns
    ),
)


add_check(
    validation_records,
    "Registry rows",
    20,
    len(
        registry
    ),
    len(
        registry
    )
    == 20,
)


for predecessor_number, predecessor_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            project_numbers.eq(
                predecessor_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        actual_project,
        actual_project == predecessor_project,
    )


add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    active_reservations,
    active_reservations == EXPECTED_ACTIVE_RESERVATIONS,
)


add_check(
    validation_records,
    "Project 22 registry rows",
    0,
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)


add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    EXPECTED_RUNTIME_PRIORITY_RULE,
    True,
)


add_check(
    validation_records,
    "No rolling retraining",
    True,
    bool(
        baseline_contract[
            "NoRollingRetraining"
        ]
    ),
    bool(
        baseline_contract[
            "NoRollingRetraining"
        ]
    ),
)


add_check(
    validation_records,
    "Evaluation partition clean and fixed",
    True,
    bool(
        baseline_contract[
            "CleanEvaluationPartition"
        ]
    ),
    bool(
        baseline_contract[
            "CleanEvaluationPartition"
        ]
    ),
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 22 Step 4A validation:"
)

display(
    validation
)


if not failed_validation.empty:
    print(
        "\nFailed Project 22 Step 4A checks:"
    )

    display(
        failed_validation
    )

    print(
        "\nNo Step 4A PASS status or checkpoint was written."
    )

    raise RuntimeError(
        "PROJECT 22 STEP 4A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 15. WRITE CONTRACT OUTPUTS
# --------------------------------------------------------------------------------------------------

RUNTIME_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_csv(
    RUNTIME_VERSION_PATH,
    runtime_versions,
)


atomic_csv(
    PREDICTOR_CONTRACT_PATH,
    predictor_contract,
)


atomic_csv(
    CLEAN_MEDIAN_REFERENCE_PATH,
    clean_median_reference,
)


atomic_csv(
    METRIC_SELF_TEST_PATH,
    metric_self_test,
)


atomic_csv(
    STEP4A_VALIDATION_PATH,
    validation,
)


model_contract_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "PositiveClass": {
        "Name":
            "failure",

        "Value":
            1,

        "Conversion":
            "binary target = (Verdict != 0).astype(int)",
    },

    "PredictorCount":
        len(
            predictor_columns
        ),

    "Imputation": {
        "Rule":
            (
                "For every condition, compute one median per active "
                "predictor from that condition's training matrix only. "
                "Replace +/-infinity with missing before computing medians. "
                "Use those training medians to fill training and clean "
                "evaluation missing values."
            ),

        "Scaling":
            "none",

        "ActivePredictors":
            "all 151 fixed predictor columns",
    },

    "Models":
        MODEL_CONFIG,

    "ModelSeeds": {
        "RandomForest":
            "SHA-256(project|repetition_seed|RandomForest_model)",

        "XGBoost":
            "SHA-256(project|repetition_seed|XGBoost_model)",

        "LightGBM":
            "SHA-256(project|repetition_seed|LightGBM_model)",

        "NaiveBayes":
            "deterministic; no random_state parameter",
    },

    "PositiveProbabilityExtraction":
        (
            "Use predict_proba and select the column whose "
            "fitted classes_ value equals 1."
        ),

    "NoRollingRetraining":
        True,

    "ModelImplementationAudit":
        model_contract_table.to_dict(
            orient="records"
        ),
}


atomic_json(
    MODEL_CONTRACT_PATH,
    model_contract_payload,
)


atomic_json(
    BASELINE_CONTRACT_PATH,
    baseline_contract,
)


atomic_json(
    RANKING_CONTRACT_PATH,
    ranking_contract,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


output_paths = [
    RUNTIME_VERSION_PATH,
    PREDICTOR_CONTRACT_PATH,
    CLEAN_MEDIAN_REFERENCE_PATH,
    MODEL_CONTRACT_PATH,
    BASELINE_CONTRACT_PATH,
    METRIC_SELF_TEST_PATH,
    RANKING_CONTRACT_PATH,
    STEP4A_VALIDATION_PATH,
]


runtime_output_manifest = [
    {
        "Path":
            str(path),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "NoisePlanCheckpointSHA256":
        noise_checkpoint_sha256,

    "RuntimeVersions":
        runtime_versions.to_dict(
            orient="records"
        ),

    "PredictorCount":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "MLTechniques":
        ML_TECHNIQUES,

    "BaselineTechniques":
        BASELINE_TECHNIQUES,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "PositiveClass":
        "failure = 1",

    "MedianImputation":
        "condition-training medians",

    "RankingTieBreak":
        "Test ascending",

    "NoRollingRetraining":
        True,

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "ModelTrainingFailures":
        model_training_failures,

    "ModelEvaluationFailures":
        model_evaluation_failures,

    "ModelFailingEvaluationBuilds":
        model_failing_evaluation_builds,

    "RuntimeOutputManifest":
        runtime_output_manifest,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To21Modified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project20ModelsFitted":
        False,

    "FullExperimentStarted":
        False,
}


atomic_json(
    STEP4A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "RuntimeContractCheckpoint":
        True,

    "DoNotChangePredictorSet":
        True,

    "DoNotChangeModelConfiguration":
        True,

    "DoNotChangeBaselineDefinitions":
        True,

    "DoNotChangeRankingRules":
        True,

    "DoNotChangeMetricDefinitions":
        True,

    "ReadyForTwoConditionSmokeTest":
        True,
}


atomic_json(
    RUNTIME_CHECKPOINT_PATH,
    checkpoint_payload,
)


runtime_checkpoint_sha256 = sha256_file(
    RUNTIME_CHECKPOINT_PATH
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "NoisePlanCheckpointSHA256":
        noise_checkpoint_sha256,

    "PredictorCount":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "MLTechniques":
        ML_TECHNIQUES,

    "BaselineTechniques":
        BASELINE_TECHNIQUES,

    "Checkpoint":
        str(
            RUNTIME_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        runtime_checkpoint_sha256,

    "ReadyForTwoConditionSmokeTest":
        True,

    "RegistryModified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project20ModelsFitted":
        False,
}


atomic_json(
    STEP4A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 16. READBACK AND IMMUTABILITY
# --------------------------------------------------------------------------------------------------

checkpoint_readback = load_json(
    RUNTIME_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP4A_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP4A_STATUS:
    raise RuntimeError(
        "Project 22 runtime checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP4A_STATUS:
    raise RuntimeError(
        "Project 22 Step 4A status readback failed."
    )


runtime_manifest_readback_failures = 0


for item in checkpoint_readback.get(
    "RuntimeOutputManifest",
    [],
):
    path = Path(
        item[
            "Path"
        ]
    )

    if (
        not path.is_file()
        or int(
            path.stat().st_size
        )
        != int(
            item[
                "Bytes"
            ]
        )
        or sha256_file(
            path
        )
        != str(
            item[
                "SHA256"
            ]
        )
    ):
        runtime_manifest_readback_failures += 1


if runtime_manifest_readback_failures != 0:
    raise RuntimeError(
        "One or more frozen runtime-contract outputs failed readback."
    )


registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 22 Step 4A."
    )


if sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
) != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256:
    raise RuntimeError(
        "The frozen Project 22 noise-plan checkpoint changed during Step 4A."
    )


final_source_records = []


for row in current_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_source_manifest = pd.DataFrame(
    final_source_records
)


if source_root_hash(
    final_source_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "The frozen Project 22 source changed during Step 4A."
    )


# --------------------------------------------------------------------------------------------------
# 17. DISPLAY
# --------------------------------------------------------------------------------------------------

print(
    "\nRuntime versions:"
)

display(
    runtime_versions
)


print(
    "\nPredictor contract summary:"
)

display(
    predictor_contract.groupby(
        [
            "IsREC",
            "RECClass",
        ],
        dropna=False,
        as_index=False,
    ).agg(
        Predictors=(
            "Predictor",
            "count",
        ),

        TrainingMissingValues=(
            "TrainingMissing",
            "sum",
        ),

        EvaluationMissingValues=(
            "EvaluationMissing",
            "sum",
        ),
    )
)


print(
    "\nModel implementation contract:"
)

display(
    model_contract_table[
        [
            "Technique",
            "EstimatorClass",
            "Seed1RandomState",
            "Seed2RandomState",
            "SameSeedSameConfiguration",
            "DifferentSeedStateAsExpected",
        ]
    ]
)


print(
    "\nMetric self-tests:"
)

display(
    metric_self_test
)


print(
    "\nStep 3A output-manifest audit:"
)

display(
    output_manifest_audit[
        [
            "Path",
            "ExpectedBytes",
            "ActualBytes",
            "Pass",
        ]
    ]
)


# --------------------------------------------------------------------------------------------------
# 18. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 136)
print("=== PROJECT 22 CELL 7 / STEP 4A RESULT ===")
print("=" * 136)


print(
    "\nProject:"
)

print(
    PROJECT_NAME
)

for predecessor_number in sorted(required_registered_identities):
    print(
        f"Project {predecessor_number} identity:",
        required_registered_identities[
            predecessor_number
        ],
    )

print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)


print(
    "\nFrozen experiment contract:"
)

print(
    "Predictors:",
    len(
        predictor_columns
    ),
)

print(
    "REC features:",
    len(
        REC_FEATURES
    ),
)

print(
    "ML techniques:",
    ML_TECHNIQUES,
)

print(
    "Baselines:",
    BASELINE_TECHNIQUES,
)

print(
    "Primary / secondary metrics:",
    "APFDc / APFD",
)

print(
    "Positive class:",
    "failure = 1",
)

print(
    "Median imputation:",
    "condition-training medians",
)

print(
    "Ranking tie-break:",
    "Test ascending",
)

print(
    "Rolling retraining:",
    False,
)


print(
    "\nFixed cohorts:"
)

print(
    "Model training rows:",
    len(
        model_training
    ),
)

print(
    "Model evaluation rows:",
    len(
        model_evaluation
    ),
)

print(
    "Training failures:",
    model_training_failures,
)

print(
    "Evaluation failures:",
    model_evaluation_failures,
)

print(
    "Failing evaluation builds:",
    model_failing_evaluation_builds,
)


print(
    "\nRuntime validation:"
)

print(
    "Step 3A output-manifest failures:",
    output_manifest_failures,
)

print(
    "Runtime-version failures:",
    runtime_version_failures,
)

print(
    "All-missing predictors:",
    len(
        all_missing_predictors
    ),
)

print(
    "Training non-finite values after imputation:",
    training_nonfinite_after_imputation,
)

print(
    "Evaluation non-finite values after imputation:",
    evaluation_nonfinite_after_imputation,
)

print(
    "Model contract failures:",
    model_contract_failures,
)

print(
    "Metric self-test failures:",
    metric_self_test_failures,
)

print(
    "Random same-seed reproducible:",
    random_same_seed_reproduced,
)


print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–21 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Project 22 models fitted:",
    False,
)

print(
    "Full experiment started:",
    False,
)


print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print(
    "\nRuntime-contract checkpoint:"
)

print(
    RUNTIME_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    runtime_checkpoint_sha256,
)


print(
    "\nSTATUS:",
    STEP4A_STATUS,
)

print("=" * 136)


=== PROJECT 22 CELL 7 / STEP 4A: EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE ===


RuntimeError: Project 22 is unexpectedly already registered.

In [10]:
# ==================================================================================================
# PROJECT 22 — CELL 7 / STEP 4A
# EXPERIMENT RUNTIME, MODEL, BASELINE, METRIC, AND PREDICTOR CONTRACT FREEZE
#
# PROJECT:
#   apache@logging-log4j2
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME Thesis_project_22.ipynb NOTEBOOK.
#
# PURPOSE:
# - verify the frozen Project 22 Step 3A noise plan and every output in its manifest;
# - validate the fixed 151-predictor training/evaluation matrices;
# - freeze runtime versions, median-imputation, labels, ranking, model, baseline,
#   APFD, and APFDc contracts;
# - validate all four required model implementations without fitting Project 22 models;
# - write the runtime-contract checkpoint required before the two-condition smoke test.
#
# SAFETY:
# - no model fitting;
# - no condition execution;
# - no completion-registry write;
# - no prior-project condition-output access.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from IPython.display import display

import hashlib
import json
import os
import platform
import sys

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


print("=" * 136)
print("=== PROJECT 22 CELL 7 / STEP 4A: EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 22
PROJECT_NAME = "apache@logging-log4j2"
PROJECT_SLUG = "apache__logging-log4j2"
PROJECT_SHORT = "LOG4J2"

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_22_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

STEP4A_STATUS = (
    "PASS_PROJECT_22_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
)

EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "3dd2b3f38a8d7b77c179e9f518399c1dc5b1c89a8a29607ea31fd78a91f40666"
)

EXPECTED_REGISTRY_SHA256 = (
    "79cd6ecb595c5e8ae91a9494e469792716338d144308560a62caf1b9342306b2"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64f842964c896c6ac334"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

EXPECTED_REGISTERED_PROJECTS = 21

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_RAW_TRAIN_ROWS = 172_628
EXPECTED_RAW_EVAL_ROWS = 67_625
EXPECTED_MODEL_TRAIN_ROWS = 95_812
EXPECTED_MODEL_EVAL_ROWS = 22_156
EXPECTED_MODEL_TRAIN_FAILURES = 207
EXPECTED_MODEL_EVAL_FAILURES = 40
EXPECTED_MODEL_FAILING_EVAL_BUILDS = 39

EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_CONDITIONS = 270

EXPECTED_RUNTIME_VERSIONS = {
    "Python": "3.12.13",
    "numpy": "2.0.2",
    "pandas": "2.2.2",
    "scikit-learn": "1.6.1",
    "xgboost": "3.3.0",
    "lightgbm": "4.6.0",
    "pyarrow": "18.1.0",
}

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "max_features": "sqrt",
        "bootstrap": True,
        "n_jobs": -1,
    },
    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
        "eval_metric": "logloss",
    },
    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    },
    "NaiveBayes": {
        "var_smoothing": 1e-9,
    },
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_22_selection"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_22_frozen_source_manifest.csv"
)

SOURCE_DIR = Path(
    "/content/datasets/datasets/apache@logging-log4j2"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

STEP3A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step3a_status.json"
)

STEP3A_REPORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_report.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_22_noise_plan_checkpoint.json"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

PROTOCOL_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_frozen_experiment_protocol.json"
)

RUNTIME_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_runtime_contract"
)

RUNTIME_VERSION_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_runtime_versions.csv"
)

PREDICTOR_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_predictor_contract.csv"
)

CLEAN_MEDIAN_REFERENCE_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_clean_training_median_reference.csv"
)

MODEL_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_model_contract.json"
)

BASELINE_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_baseline_contract.json"
)

METRIC_SELF_TEST_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_metric_self_test.csv"
)

RANKING_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_ranking_contract.json"
)

STEP4A_VALIDATION_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_validation.csv"
)

STEP4A_REPORT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_report.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step4a_status.json"
)

RUNTIME_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_22_runtime_contract_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_json(
    path,
    payload,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def deterministic_seed(
    repetition_seed,
    stream_name,
):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(
        material
    ).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )


def deterministic_random_build_seed(
    repetition_seed,
    build_id,
):
    return deterministic_seed(
        repetition_seed,
        f"Random_baseline_build_{int(build_id)}",
    )


def create_models(
    repetition_seed,
):
    return {
        "RandomForest":
            RandomForestClassifier(
                **MODEL_CONFIG[
                    "RandomForest"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "RandomForest_model",
                ),
            ),

        "XGBoost":
            XGBClassifier(
                **MODEL_CONFIG[
                    "XGBoost"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "XGBoost_model",
                ),
            ),

        "LightGBM":
            LGBMClassifier(
                **MODEL_CONFIG[
                    "LightGBM"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "LightGBM_model",
                ),
            ),

        "NaiveBayes":
            GaussianNB(
                **MODEL_CONFIG[
                    "NaiveBayes"
                ]
            ),
    }


def calculate_apfd(
    failures,
):
    failures = np.asarray(
        failures,
        dtype=np.int8,
    )

    number_of_tests = len(
        failures
    )

    number_of_failures = int(
        failures.sum()
    )

    if (
        number_of_tests == 0
        or number_of_failures == 0
    ):
        return np.nan

    failure_positions = (
        np.flatnonzero(
            failures == 1
        )
        + 1
    )

    return float(
        1.0
        - (
            failure_positions.sum()
            / (
                number_of_tests
                * number_of_failures
            )
        )
        + (
            1.0
            / (
                2.0
                * number_of_tests
            )
        )
    )


def calculate_apfdc(
    failures,
    durations,
):
    failures = np.asarray(
        failures,
        dtype=np.int8,
    )

    durations = np.asarray(
        durations,
        dtype=float,
    )

    if len(failures) != len(durations):
        raise ValueError(
            "failures and durations must have equal length."
        )

    if (
        len(failures) == 0
        or failures.sum() == 0
    ):
        return np.nan

    if not np.isfinite(
        durations
    ).all():
        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (
        durations < 0
    ).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(
        durations.sum()
    )

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array([0.0]),
        np.cumsum(
            durations
        )[:-1],
    ])

    failure_mask = (
        failures == 1
    )

    midpoint_detection_times = (
        cumulative_before[
            failure_mask
        ]
        + (
            0.5
            * durations[
                failure_mask
            ]
        )
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(passed),
    })


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND STEP 3A CHECKPOINT
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    STEP3A_STATUS_PATH,
    STEP3A_REPORT_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    CONDITION_PLAN_PATH,
    PROTOCOL_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 22 Step 4A inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


noise_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)

noise_checkpoint = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)

step3a_status = load_json(
    STEP3A_STATUS_PATH
)

step3a_report = load_json(
    STEP3A_REPORT_PATH
)

protocol = load_json(
    PROTOCOL_PATH
)


if (
    noise_checkpoint_sha256
    != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 22 noise-plan checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256}\n"
        f"Actual:   {noise_checkpoint_sha256}"
    )


for label, payload in [
    (
        "noise checkpoint",
        noise_checkpoint,
    ),
    (
        "Step 3A status",
        step3a_status,
    ),
    (
        "Step 3A report",
        step3a_report,
    ),
]:
    if payload.get(
        "Status"
    ) != EXPECTED_STEP3A_STATUS:
        raise RuntimeError(
            f"{label} does not contain the expected Step 3A PASS status."
        )


if (
    noise_checkpoint.get(
        "Project"
    )
    != PROJECT_NAME
    or noise_checkpoint.get(
        "ProjectSlug"
    )
    != PROJECT_SLUG
):
    raise RuntimeError(
        "The frozen Step 3A Project 22 identity differs."
    )


if (
    noise_checkpoint.get(
        "SourceRootSHA256"
    )
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The Step 3A checkpoint source root differs."
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY STEP 3A OUTPUT MANIFEST
# --------------------------------------------------------------------------------------------------

output_manifest = noise_checkpoint.get(
    "OutputManifest",
    []
)


if not isinstance(
    output_manifest,
    list,
) or not output_manifest:
    raise RuntimeError(
        "The Step 3A checkpoint contains no output manifest."
    )


output_manifest_records = []


for item in output_manifest:
    path = Path(
        item[
            "Path"
        ]
    )

    expected_bytes = int(
        item[
            "Bytes"
        ]
    )

    expected_sha256 = str(
        item[
            "SHA256"
        ]
    )

    exists = path.is_file()

    actual_bytes = (
        int(
            path.stat().st_size
        )
        if exists
        else -1
    )

    actual_sha256 = (
        sha256_file(
            path
        )
        if exists
        else "MISSING"
    )

    output_manifest_records.append({
        "Path":
            str(path),

        "ExpectedBytes":
            expected_bytes,

        "ActualBytes":
            actual_bytes,

        "ExpectedSHA256":
            expected_sha256,

        "ActualSHA256":
            actual_sha256,

        "Pass":
            (
                exists
                and actual_bytes
                == expected_bytes
                and actual_sha256
                == expected_sha256
            ),
    })


output_manifest_audit = pd.DataFrame(
    output_manifest_records
)


output_manifest_failures = int(
    (
        ~output_manifest_audit[
            "Pass"
        ]
    ).sum()
)


if output_manifest_failures:
    print(
        "\nFailed Step 3A output-manifest checks:"
    )

    display(
        output_manifest_audit.loc[
            ~output_manifest_audit[
                "Pass"
            ]
        ]
    )

    raise RuntimeError(
        "One or more frozen Step 3A outputs changed."
    )


# --------------------------------------------------------------------------------------------------
# 6. SOURCE AND REGISTRY IMMUTABILITY
# --------------------------------------------------------------------------------------------------

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_records = []


for row in frozen_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            "A frozen Project 22 source file is missing:\n"
            f"{source_path}"
        )

    current_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


current_source_manifest = pd.DataFrame(
    current_source_records
)


current_source_root_sha256 = source_root_hash(
    current_source_manifest
)


if (
    current_source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The frozen Project 22 source root differs."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "The registry does not contain exactly Projects 1–21."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–21 are not all COMPLETE_AND_FROZEN."
    )


if project_numbers.eq(PROJECT_NUMBER).any():
    raise RuntimeError(
        "Project 22 is unexpectedly already registered."
    )


if registry[project_column].eq(PROJECT_NAME).any():
    raise RuntimeError(
        "The selected Project 22 identity is already registered."
    )


required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


active_reservations = []


if active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "The active-reservation state differs from the Project 22 freeze."
    )


# --------------------------------------------------------------------------------------------------
# 7. LOAD AND VALIDATE FIXED COHORTS
# --------------------------------------------------------------------------------------------------

raw_training = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)

raw_evaluation = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)

model_training = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)

model_evaluation = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)

model_raw_train_link = pd.read_parquet(
    MODEL_RAW_TRAIN_LINK_PATH
)

model_raw_eval_link = pd.read_parquet(
    MODEL_RAW_EVAL_LINK_PATH
)

condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)


required_model_columns = {
    "Build",
    "Test",
    "Verdict",
}


if not required_model_columns.issubset(
    model_training.columns
) or not required_model_columns.issubset(
    model_evaluation.columns
):
    raise RuntimeError(
        "Fixed model cohorts are missing Build, Test, or Verdict."
    )


training_metadata_columns = {
    "ModelTrainingRowOrder",
    "Build",
    "Test",
    "Verdict",
}


evaluation_metadata_columns = {
    "ModelEvaluationRowOrder",
    "Build",
    "Test",
    "Verdict",
}


predictor_columns = [
    column
    for column in model_training.columns
    if column not in training_metadata_columns
]


evaluation_predictor_columns = [
    column
    for column in model_evaluation.columns
    if column not in evaluation_metadata_columns
]


if (
    predictor_columns
    != evaluation_predictor_columns
):
    raise RuntimeError(
        "Training and evaluation predictor order differs."
    )


if len(
    predictor_columns
) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "The fixed predictor count differs.\n"
        f"Expected: {EXPECTED_PREDICTORS}\n"
        f"Actual:   {len(predictor_columns)}"
    )


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in predictor_columns
]


if missing_rec_features:
    raise RuntimeError(
        "Fixed predictor cohorts are missing REC features:\n"
        + "\n".join(
            missing_rec_features
        )
    )


model_training_failures = int(
    pd.to_numeric(
        model_training[
            "Verdict"
        ],
        errors="raise",
    ).ne(
        0
    ).sum()
)


model_evaluation_failures = int(
    pd.to_numeric(
        model_evaluation[
            "Verdict"
        ],
        errors="raise",
    ).ne(
        0
    ).sum()
)


model_failing_evaluation_builds = int(
    model_evaluation.loc[
        pd.to_numeric(
            model_evaluation[
                "Verdict"
            ],
            errors="raise",
        ).ne(
            0
        ),
        "Build",
    ].nunique()
)


# --------------------------------------------------------------------------------------------------
# 8. NUMERIC PREDICTORS AND MEDIAN IMPUTATION
# --------------------------------------------------------------------------------------------------

training_numeric = pd.DataFrame(
    index=model_training.index
)

evaluation_numeric = pd.DataFrame(
    index=model_evaluation.index
)

predictor_profile_records = []


for predictor_order, column in enumerate(
    predictor_columns,
    start=1,
):
    training_values = pd.to_numeric(
        model_training[
            column
        ],
        errors="coerce",
    ).astype(
        float
    ).replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )

    evaluation_values = pd.to_numeric(
        model_evaluation[
            column
        ],
        errors="coerce",
    ).astype(
        float
    ).replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )

    training_numeric[
        column
    ] = training_values

    evaluation_numeric[
        column
    ] = evaluation_values

    predictor_profile_records.append({
        "PredictorOrder":
            predictor_order,

        "Predictor":
            column,

        "IsREC":
            column in REC_FEATURES,

        "RECClass":
            (
                "VERDICT_DEPENDENT"
                if column
                in VERDICT_DEPENDENT_REC
                else (
                    "VERDICT_INDEPENDENT"
                    if column
                    in VERDICT_INDEPENDENT_REC
                    else ""
                )
            ),

        "TrainingRows":
            len(
                training_values
            ),

        "TrainingNonMissing":
            int(
                training_values.notna().sum()
            ),

        "TrainingMissing":
            int(
                training_values.isna().sum()
            ),

        "EvaluationRows":
            len(
                evaluation_values
            ),

        "EvaluationMissing":
            int(
                evaluation_values.isna().sum()
            ),

        "AllTrainingValuesMissing":
            bool(
                training_values.notna().sum()
                == 0
            ),
    })


predictor_contract = pd.DataFrame(
    predictor_profile_records
)


all_missing_predictors = predictor_contract.loc[
    predictor_contract[
        "AllTrainingValuesMissing"
    ],
    "Predictor",
].tolist()


if all_missing_predictors:
    raise RuntimeError(
        "One or more predictors are entirely missing in training:\n"
        + "\n".join(
            all_missing_predictors
        )
    )


clean_training_medians = training_numeric.median(
    axis=0,
    skipna=True,
)


if (
    clean_training_medians.isna().any()
    or not np.isfinite(
        clean_training_medians.to_numpy(
            dtype=float
        )
    ).all()
):
    raise RuntimeError(
        "Clean training medians contain missing or infinite values."
    )


training_imputed = training_numeric.fillna(
    clean_training_medians
)

evaluation_imputed = evaluation_numeric.fillna(
    clean_training_medians
)


training_nonfinite_after_imputation = int(
    (
        ~np.isfinite(
            training_imputed.to_numpy(
                dtype=float
            )
        )
    ).sum()
)


evaluation_nonfinite_after_imputation = int(
    (
        ~np.isfinite(
            evaluation_imputed.to_numpy(
                dtype=float
            )
        )
    ).sum()
)


clean_median_reference = pd.DataFrame({
    "PredictorOrder":
        np.arange(
            1,
            len(
                predictor_columns
            )
            + 1,
            dtype=np.int64,
        ),

    "Predictor":
        predictor_columns,

    "CleanTrainingMedian":
        clean_training_medians[
            predictor_columns
        ].to_numpy(
            dtype=float
        ),
})


# --------------------------------------------------------------------------------------------------
# 9. LABEL CONTRACT
# --------------------------------------------------------------------------------------------------

training_binary_labels = (
    pd.to_numeric(
        model_training[
            "Verdict"
        ],
        errors="raise",
    )
    .ne(
        0
    )
    .astype(
        np.int8
    )
)


evaluation_binary_labels = (
    pd.to_numeric(
        model_evaluation[
            "Verdict"
        ],
        errors="raise",
    )
    .ne(
        0
    )
    .astype(
        np.int8
    )
)


training_label_values = sorted(
    training_binary_labels.unique().tolist()
)


evaluation_label_values = sorted(
    evaluation_binary_labels.unique().tolist()
)


# --------------------------------------------------------------------------------------------------
# 10. RUNTIME VERSION CONTRACT
# --------------------------------------------------------------------------------------------------

runtime_versions = pd.DataFrame([
    {
        "Component":
            "Python",

        "Version":
            platform.python_version(),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "Python"
            ],
    },
    {
        "Component":
            "numpy",

        "Version":
            np.__version__,

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "numpy"
            ],
    },
    {
        "Component":
            "pandas",

        "Version":
            pd.__version__,

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "pandas"
            ],
    },
    {
        "Component":
            "scikit-learn",

        "Version":
            metadata.version(
                "scikit-learn"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "scikit-learn"
            ],
    },
    {
        "Component":
            "xgboost",

        "Version":
            metadata.version(
                "xgboost"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "xgboost"
            ],
    },
    {
        "Component":
            "lightgbm",

        "Version":
            metadata.version(
                "lightgbm"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "lightgbm"
            ],
    },
    {
        "Component":
            "pyarrow",

        "Version":
            metadata.version(
                "pyarrow"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "pyarrow"
            ],
    },
])


runtime_versions[
    "Pass"
] = runtime_versions[
    "Version"
].eq(
    runtime_versions[
        "ExpectedVersion"
    ]
)


runtime_version_failures = int(
    (
        ~runtime_versions[
            "Pass"
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 11. MODEL IMPLEMENTATION CONTRACT
# --------------------------------------------------------------------------------------------------

models_seed_1 = create_models(
    repetition_seed=1
)

models_seed_1_repeat = create_models(
    repetition_seed=1
)

models_seed_2 = create_models(
    repetition_seed=2
)


model_contract_records = []


for technique in ML_TECHNIQUES:
    model_a = models_seed_1[
        technique
    ]

    model_b = models_seed_1_repeat[
        technique
    ]

    model_c = models_seed_2[
        technique
    ]

    parameters_a = model_a.get_params(
        deep=False
    )

    parameters_b = model_b.get_params(
        deep=False
    )

    parameters_c = model_c.get_params(
        deep=False
    )

    random_state_a = parameters_a.get(
        "random_state",
        None,
    )

    random_state_c = parameters_c.get(
        "random_state",
        None,
    )

    model_contract_records.append({
        "Technique":
            technique,

        "EstimatorClass":
            (
                f"{model_a.__class__.__module__}."
                f"{model_a.__class__.__name__}"
            ),

        "Seed1RandomState":
            random_state_a,

        "Seed2RandomState":
            random_state_c,

        "SameSeedSameConfiguration":
            parameters_a
            == parameters_b,

        "DifferentSeedStateAsExpected":
            (
                True
                if technique
                == "NaiveBayes"
                else random_state_a
                != random_state_c
            ),

        "ConfigurationJSON":
            json.dumps(
                parameters_a,
                sort_keys=True,
                default=str,
            ),
    })


model_contract_table = pd.DataFrame(
    model_contract_records
)


rf_params = models_seed_1[
    "RandomForest"
].get_params(
    deep=False
)

xgb_params = models_seed_1[
    "XGBoost"
].get_params(
    deep=False
)

lgbm_params = models_seed_1[
    "LightGBM"
].get_params(
    deep=False
)

nb_params = models_seed_1[
    "NaiveBayes"
].get_params(
    deep=False
)


model_parameter_checks = {
    "RandomForest": (
        rf_params.get(
            "n_estimators"
        )
        == 100
        and rf_params.get(
            "max_features"
        )
        == "sqrt"
        and rf_params.get(
            "bootstrap"
        )
        is True
        and rf_params.get(
            "n_jobs"
        )
        == -1
    ),

    "XGBoost": (
        xgb_params.get(
            "n_estimators"
        )
        == 100
        and xgb_params.get(
            "max_depth"
        )
        == 6
        and np.isclose(
            float(
                xgb_params.get(
                    "learning_rate"
                )
            ),
            0.1,
        )
        and xgb_params.get(
            "tree_method"
        )
        == "hist"
        and xgb_params.get(
            "n_jobs"
        )
        == -1
    ),

    "LightGBM": (
        lgbm_params.get(
            "n_estimators"
        )
        == 100
        and np.isclose(
            float(
                lgbm_params.get(
                    "learning_rate"
                )
            ),
            0.1,
        )
        and lgbm_params.get(
            "num_leaves"
        )
        == 31
        and lgbm_params.get(
            "deterministic"
        )
        is True
        and lgbm_params.get(
            "force_col_wise"
        )
        is True
        and lgbm_params.get(
            "n_jobs"
        )
        == -1
    ),

    "NaiveBayes": (
        np.isclose(
            float(
                nb_params.get(
                    "var_smoothing"
                )
            ),
            1e-9,
        )
    ),
}


model_contract_failures = int(
    (
        ~model_contract_table[
            "SameSeedSameConfiguration"
        ]
        | ~model_contract_table[
            "DifferentSeedStateAsExpected"
        ]
    ).sum()
    + sum(
        not bool(value)
        for value in model_parameter_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 12. BASELINE, RANKING, AND RANDOM CONTRACT
# --------------------------------------------------------------------------------------------------

sample_build_id = int(
    model_evaluation[
        "Build"
    ].iloc[
        0
    ]
)


random_seed_1_a = deterministic_random_build_seed(
    repetition_seed=1,
    build_id=sample_build_id,
)

random_seed_1_b = deterministic_random_build_seed(
    repetition_seed=1,
    build_id=sample_build_id,
)

random_seed_2 = deterministic_random_build_seed(
    repetition_seed=2,
    build_id=sample_build_id,
)


sample_random_a = np.random.default_rng(
    random_seed_1_a
).random(
    100
)

sample_random_b = np.random.default_rng(
    random_seed_1_b
).random(
    100
)

sample_random_c = np.random.default_rng(
    random_seed_2
).random(
    100
)


random_same_seed_reproduced = bool(
    np.array_equal(
        sample_random_a,
        sample_random_b,
    )
)


random_different_seed_differs = bool(
    not np.array_equal(
        sample_random_a,
        sample_random_c,
    )
)


ranking_contract = {
    "ML": {
        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",
    },

    "Random": {
        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",

        "SeedRule":
            (
                "first little-endian uint32 of "
                "SHA-256(project|repetition_seed|"
                "Random_baseline_build_<BuildID>)"
            ),

        "ConstantAcrossNoiseForSameSeedAndBuild":
            True,
    },

    "LatestFail": {
        "SourceFeature":
            "REC_LastFailureAge",

        "ScoreFormula":
            "-REC_LastFailureAge",

        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",

        "NoiseDependent":
            True,

        "UsesSameCorruptedHistoryAsML":
            True,
    },

    "QTF-Avg": {
        "SourceFeature":
            "REC_TotalAvgExeTime",

        "Direction":
            "ascending",

        "TieBreak":
            "Test ascending",

        "NoiseDependent":
            False,
    },
}


baseline_contract = {
    "Techniques":
        BASELINE_TECHNIQUES,

    "Random":
        ranking_contract[
            "Random"
        ],

    "LatestFail":
        ranking_contract[
            "LatestFail"
        ],

    "QTF-Avg":
        ranking_contract[
            "QTF-Avg"
        ],

    "NoRollingRetraining":
        True,

    "CleanEvaluationPartition":
        True,
}


# --------------------------------------------------------------------------------------------------
# 13. APFD/APFDc SELF-TESTS
# --------------------------------------------------------------------------------------------------

manual_failures = np.array([
    1,
    1,
    0,
    0,
    0,
], dtype=np.int8)


manual_apfd = calculate_apfd(
    manual_failures
)


manual_apfdc_slow_failure_first = calculate_apfdc(
    manual_failures,
    np.array([
        5.0,
        1.0,
        1.0,
        1.0,
        1.0,
    ]),
)


manual_apfdc_fast_failure_first = calculate_apfdc(
    manual_failures,
    np.array([
        1.0,
        5.0,
        1.0,
        1.0,
        1.0,
    ]),
)


all_pass_apfd = calculate_apfd(
    np.array([
        0,
        0,
        0,
    ])
)


all_pass_apfdc = calculate_apfdc(
    np.array([
        0,
        0,
        0,
    ]),
    np.array([
        1.0,
        1.0,
        1.0,
    ]),
)


metric_self_test = pd.DataFrame([
    {
        "Check":
            "Manual APFD",

        "Expected":
            0.8,

        "Actual":
            manual_apfd,

        "Pass":
            np.isclose(
                manual_apfd,
                0.8,
                rtol=0,
                atol=1e-15,
            ),
    },

    {
        "Check":
            "APFDc rewards quick failing test first",

        "Expected":
            True,

        "Actual":
            manual_apfdc_fast_failure_first
            > manual_apfdc_slow_failure_first,

        "Pass":
            manual_apfdc_fast_failure_first
            > manual_apfdc_slow_failure_first,
    },

    {
        "Check":
            "All-pass APFD is NaN",

        "Expected":
            True,

        "Actual":
            bool(
                np.isnan(
                    all_pass_apfd
                )
            ),

        "Pass":
            bool(
                np.isnan(
                    all_pass_apfd
                )
            ),
    },

    {
        "Check":
            "All-pass APFDc is NaN",

        "Expected":
            True,

        "Actual":
            bool(
                np.isnan(
                    all_pass_apfdc
                )
            ),

        "Pass":
            bool(
                np.isnan(
                    all_pass_apfdc
                )
            ),
    },
])


metric_self_test_failures = int(
    (
        ~metric_self_test[
            "Pass"
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 14. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 3A status",
    EXPECTED_STEP3A_STATUS,
    step3a_status.get(
        "Status"
    ),
    step3a_status.get(
        "Status"
    )
    == EXPECTED_STEP3A_STATUS,
)


add_check(
    validation_records,
    "Noise-plan checkpoint SHA-256",
    EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
    noise_checkpoint_sha256,
    noise_checkpoint_sha256
    == EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
)


add_check(
    validation_records,
    "Step 3A output-manifest failures",
    0,
    output_manifest_failures,
    output_manifest_failures
    == 0,
)


add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)


add_check(
    validation_records,
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training
    ),
    len(
        raw_training
    )
    == EXPECTED_RAW_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation
    ),
    len(
        raw_evaluation
    )
    == EXPECTED_RAW_EVAL_ROWS,
)


add_check(
    validation_records,
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training
    ),
    len(
        model_training
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation
    ),
    len(
        model_evaluation
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)


add_check(
    validation_records,
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    model_training_failures,
    model_training_failures
    == EXPECTED_MODEL_TRAIN_FAILURES,
)


add_check(
    validation_records,
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    model_evaluation_failures,
    model_evaluation_failures
    == EXPECTED_MODEL_EVAL_FAILURES,
)


add_check(
    validation_records,
    "Model failing evaluation builds",
    EXPECTED_MODEL_FAILING_EVAL_BUILDS,
    model_failing_evaluation_builds,
    model_failing_evaluation_builds
    == EXPECTED_MODEL_FAILING_EVAL_BUILDS,
)


add_check(
    validation_records,
    "Condition-plan rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan
    ),
    len(
        condition_plan
    )
    == EXPECTED_CONDITIONS,
)


add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    )
    == EXPECTED_PREDICTORS,
)


add_check(
    validation_records,
    "REC features",
    EXPECTED_REC_FEATURES,
    len(
        REC_FEATURES
    ),
    (
        len(
            REC_FEATURES
        )
        == EXPECTED_REC_FEATURES
        and not missing_rec_features
    ),
)


add_check(
    validation_records,
    "Raw-training link rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_raw_train_link
    ),
    len(
        model_raw_train_link
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Raw-evaluation link rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_raw_eval_link
    ),
    len(
        model_raw_eval_link
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)


add_check(
    validation_records,
    "Training binary label values",
    [0, 1],
    training_label_values,
    training_label_values
    == [
        0,
        1,
    ],
)


add_check(
    validation_records,
    "Evaluation binary label values",
    [0, 1],
    evaluation_label_values,
    evaluation_label_values
    == [
        0,
        1,
    ],
)


add_check(
    validation_records,
    "All-missing predictors",
    0,
    len(
        all_missing_predictors
    ),
    len(
        all_missing_predictors
    )
    == 0,
)


add_check(
    validation_records,
    "Training non-finite values after imputation",
    0,
    training_nonfinite_after_imputation,
    training_nonfinite_after_imputation
    == 0,
)


add_check(
    validation_records,
    "Evaluation non-finite values after imputation",
    0,
    evaluation_nonfinite_after_imputation,
    evaluation_nonfinite_after_imputation
    == 0,
)


add_check(
    validation_records,
    "Runtime-version failures",
    0,
    runtime_version_failures,
    runtime_version_failures
    == 0,
)


add_check(
    validation_records,
    "Model contract failures",
    0,
    model_contract_failures,
    model_contract_failures
    == 0,
)


add_check(
    validation_records,
    "Metric self-test failures",
    0,
    metric_self_test_failures,
    metric_self_test_failures
    == 0,
)


add_check(
    validation_records,
    "Random same-seed reproducible",
    True,
    random_same_seed_reproduced,
    random_same_seed_reproduced,
)


add_check(
    validation_records,
    "Random different-seed differs",
    True,
    random_different_seed_differs,
    random_different_seed_differs,
)


add_check(
    validation_records,
    "LatestFail feature present",
    True,
    (
        "REC_LastFailureAge"
        in predictor_columns
    ),
    (
        "REC_LastFailureAge"
        in predictor_columns
    ),
)


add_check(
    validation_records,
    "QTF-Avg feature present",
    True,
    (
        "REC_TotalAvgExeTime"
        in predictor_columns
    ),
    (
        "REC_TotalAvgExeTime"
        in predictor_columns
    ),
)


add_check(
    validation_records,
    "Registry rows",
    20,
    len(
        registry
    ),
    len(
        registry
    )
    == 20,
)


for predecessor_number, predecessor_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            project_numbers.eq(
                predecessor_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        actual_project,
        actual_project == predecessor_project,
    )


add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    active_reservations,
    active_reservations == EXPECTED_ACTIVE_RESERVATIONS,
)


add_check(
    validation_records,
    "Project 22 registry rows",
    0,
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)


add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    EXPECTED_RUNTIME_PRIORITY_RULE,
    True,
)


add_check(
    validation_records,
    "No rolling retraining",
    True,
    bool(
        baseline_contract[
            "NoRollingRetraining"
        ]
    ),
    bool(
        baseline_contract[
            "NoRollingRetraining"
        ]
    ),
)


add_check(
    validation_records,
    "Evaluation partition clean and fixed",
    True,
    bool(
        baseline_contract[
            "CleanEvaluationPartition"
        ]
    ),
    bool(
        baseline_contract[
            "CleanEvaluationPartition"
        ]
    ),
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 22 Step 4A validation:"
)

display(
    validation
)


if not failed_validation.empty:
    print(
        "\nFailed Project 22 Step 4A checks:"
    )

    display(
        failed_validation
    )

    print(
        "\nNo Step 4A PASS status or checkpoint was written."
    )

    raise RuntimeError(
        "PROJECT 22 STEP 4A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 15. WRITE CONTRACT OUTPUTS
# --------------------------------------------------------------------------------------------------

RUNTIME_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_csv(
    RUNTIME_VERSION_PATH,
    runtime_versions,
)


atomic_csv(
    PREDICTOR_CONTRACT_PATH,
    predictor_contract,
)


atomic_csv(
    CLEAN_MEDIAN_REFERENCE_PATH,
    clean_median_reference,
)


atomic_csv(
    METRIC_SELF_TEST_PATH,
    metric_self_test,
)


atomic_csv(
    STEP4A_VALIDATION_PATH,
    validation,
)


model_contract_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "PositiveClass": {
        "Name":
            "failure",

        "Value":
            1,

        "Conversion":
            "binary target = (Verdict != 0).astype(int)",
    },

    "PredictorCount":
        len(
            predictor_columns
        ),

    "Imputation": {
        "Rule":
            (
                "For every condition, compute one median per active "
                "predictor from that condition's training matrix only. "
                "Replace +/-infinity with missing before computing medians. "
                "Use those training medians to fill training and clean "
                "evaluation missing values."
            ),

        "Scaling":
            "none",

        "ActivePredictors":
            "all 151 fixed predictor columns",
    },

    "Models":
        MODEL_CONFIG,

    "ModelSeeds": {
        "RandomForest":
            "SHA-256(project|repetition_seed|RandomForest_model)",

        "XGBoost":
            "SHA-256(project|repetition_seed|XGBoost_model)",

        "LightGBM":
            "SHA-256(project|repetition_seed|LightGBM_model)",

        "NaiveBayes":
            "deterministic; no random_state parameter",
    },

    "PositiveProbabilityExtraction":
        (
            "Use predict_proba and select the column whose "
            "fitted classes_ value equals 1."
        ),

    "NoRollingRetraining":
        True,

    "ModelImplementationAudit":
        model_contract_table.to_dict(
            orient="records"
        ),
}


atomic_json(
    MODEL_CONTRACT_PATH,
    model_contract_payload,
)


atomic_json(
    BASELINE_CONTRACT_PATH,
    baseline_contract,
)


atomic_json(
    RANKING_CONTRACT_PATH,
    ranking_contract,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


output_paths = [
    RUNTIME_VERSION_PATH,
    PREDICTOR_CONTRACT_PATH,
    CLEAN_MEDIAN_REFERENCE_PATH,
    MODEL_CONTRACT_PATH,
    BASELINE_CONTRACT_PATH,
    METRIC_SELF_TEST_PATH,
    RANKING_CONTRACT_PATH,
    STEP4A_VALIDATION_PATH,
]


runtime_output_manifest = [
    {
        "Path":
            str(path),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "NoisePlanCheckpointSHA256":
        noise_checkpoint_sha256,

    "RuntimeVersions":
        runtime_versions.to_dict(
            orient="records"
        ),

    "PredictorCount":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "MLTechniques":
        ML_TECHNIQUES,

    "BaselineTechniques":
        BASELINE_TECHNIQUES,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "PositiveClass":
        "failure = 1",

    "MedianImputation":
        "condition-training medians",

    "RankingTieBreak":
        "Test ascending",

    "NoRollingRetraining":
        True,

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "ModelTrainingFailures":
        model_training_failures,

    "ModelEvaluationFailures":
        model_evaluation_failures,

    "ModelFailingEvaluationBuilds":
        model_failing_evaluation_builds,

    "RuntimeOutputManifest":
        runtime_output_manifest,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To21Modified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project20ModelsFitted":
        False,

    "FullExperimentStarted":
        False,
}


atomic_json(
    STEP4A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "RuntimeContractCheckpoint":
        True,

    "DoNotChangePredictorSet":
        True,

    "DoNotChangeModelConfiguration":
        True,

    "DoNotChangeBaselineDefinitions":
        True,

    "DoNotChangeRankingRules":
        True,

    "DoNotChangeMetricDefinitions":
        True,

    "ReadyForTwoConditionSmokeTest":
        True,
}


atomic_json(
    RUNTIME_CHECKPOINT_PATH,
    checkpoint_payload,
)


runtime_checkpoint_sha256 = sha256_file(
    RUNTIME_CHECKPOINT_PATH
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "NoisePlanCheckpointSHA256":
        noise_checkpoint_sha256,

    "PredictorCount":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "MLTechniques":
        ML_TECHNIQUES,

    "BaselineTechniques":
        BASELINE_TECHNIQUES,

    "Checkpoint":
        str(
            RUNTIME_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        runtime_checkpoint_sha256,

    "ReadyForTwoConditionSmokeTest":
        True,

    "RegistryModified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project20ModelsFitted":
        False,
}


atomic_json(
    STEP4A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 16. READBACK AND IMMUTABILITY
# --------------------------------------------------------------------------------------------------

checkpoint_readback = load_json(
    RUNTIME_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP4A_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP4A_STATUS:
    raise RuntimeError(
        "Project 22 runtime checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP4A_STATUS:
    raise RuntimeError(
        "Project 22 Step 4A status readback failed."
    )


runtime_manifest_readback_failures = 0


for item in checkpoint_readback.get(
    "RuntimeOutputManifest",
    [],
):
    path = Path(
        item[
            "Path"
        ]
    )

    if (
        not path.is_file()
        or int(
            path.stat().st_size
        )
        != int(
            item[
                "Bytes"
            ]
        )
        or sha256_file(
            path
        )
        != str(
            item[
                "SHA256"
            ]
        )
    ):
        runtime_manifest_readback_failures += 1


if runtime_manifest_readback_failures != 0:
    raise RuntimeError(
        "One or more frozen runtime-contract outputs failed readback."
    )


registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 22 Step 4A."
    )


if sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
) != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256:
    raise RuntimeError(
        "The frozen Project 22 noise-plan checkpoint changed during Step 4A."
    )


final_source_records = []


for row in current_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_source_manifest = pd.DataFrame(
    final_source_records
)


if source_root_hash(
    final_source_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "The frozen Project 22 source changed during Step 4A."
    )


# --------------------------------------------------------------------------------------------------
# 17. DISPLAY
# --------------------------------------------------------------------------------------------------

print(
    "\nRuntime versions:"
)

display(
    runtime_versions
)


print(
    "\nPredictor contract summary:"
)

display(
    predictor_contract.groupby(
        [
            "IsREC",
            "RECClass",
        ],
        dropna=False,
        as_index=False,
    ).agg(
        Predictors=(
            "Predictor",
            "count",
        ),

        TrainingMissingValues=(
            "TrainingMissing",
            "sum",
        ),

        EvaluationMissingValues=(
            "EvaluationMissing",
            "sum",
        ),
    )
)


print(
    "\nModel implementation contract:"
)

display(
    model_contract_table[
        [
            "Technique",
            "EstimatorClass",
            "Seed1RandomState",
            "Seed2RandomState",
            "SameSeedSameConfiguration",
            "DifferentSeedStateAsExpected",
        ]
    ]
)


print(
    "\nMetric self-tests:"
)

display(
    metric_self_test
)


print(
    "\nStep 3A output-manifest audit:"
)

display(
    output_manifest_audit[
        [
            "Path",
            "ExpectedBytes",
            "ActualBytes",
            "Pass",
        ]
    ]
)


# --------------------------------------------------------------------------------------------------
# 18. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 136)
print("=== PROJECT 22 CELL 7 / STEP 4A RESULT ===")
print("=" * 136)


print(
    "\nProject:"
)

print(
    PROJECT_NAME
)

for predecessor_number in sorted(required_registered_identities):
    print(
        f"Project {predecessor_number} identity:",
        required_registered_identities[
            predecessor_number
        ],
    )

print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)


print(
    "\nFrozen experiment contract:"
)

print(
    "Predictors:",
    len(
        predictor_columns
    ),
)

print(
    "REC features:",
    len(
        REC_FEATURES
    ),
)

print(
    "ML techniques:",
    ML_TECHNIQUES,
)

print(
    "Baselines:",
    BASELINE_TECHNIQUES,
)

print(
    "Primary / secondary metrics:",
    "APFDc / APFD",
)

print(
    "Positive class:",
    "failure = 1",
)

print(
    "Median imputation:",
    "condition-training medians",
)

print(
    "Ranking tie-break:",
    "Test ascending",
)

print(
    "Rolling retraining:",
    False,
)


print(
    "\nFixed cohorts:"
)

print(
    "Model training rows:",
    len(
        model_training
    ),
)

print(
    "Model evaluation rows:",
    len(
        model_evaluation
    ),
)

print(
    "Training failures:",
    model_training_failures,
)

print(
    "Evaluation failures:",
    model_evaluation_failures,
)

print(
    "Failing evaluation builds:",
    model_failing_evaluation_builds,
)


print(
    "\nRuntime validation:"
)

print(
    "Step 3A output-manifest failures:",
    output_manifest_failures,
)

print(
    "Runtime-version failures:",
    runtime_version_failures,
)

print(
    "All-missing predictors:",
    len(
        all_missing_predictors
    ),
)

print(
    "Training non-finite values after imputation:",
    training_nonfinite_after_imputation,
)

print(
    "Evaluation non-finite values after imputation:",
    evaluation_nonfinite_after_imputation,
)

print(
    "Model contract failures:",
    model_contract_failures,
)

print(
    "Metric self-test failures:",
    metric_self_test_failures,
)

print(
    "Random same-seed reproducible:",
    random_same_seed_reproduced,
)


print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–21 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Project 22 models fitted:",
    False,
)

print(
    "Full experiment started:",
    False,
)


print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print(
    "\nRuntime-contract checkpoint:"
)

print(
    RUNTIME_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    runtime_checkpoint_sha256,
)


print(
    "\nSTATUS:",
    STEP4A_STATUS,
)

print("=" * 136)


=== PROJECT 22 CELL 7 / STEP 4A: EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE ===


/tmp/ipykernel_7490/785055793.py:1378: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  training_numeric[
/tmp/ipykernel_7490/785055793.py:1382: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  evaluation_numeric[
/tmp/ipykernel_7490/785055793.py:1378: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  training


Project 22 Step 4A validation:


,Check,Expected,Actual,Pass
0,Step 3A status,PASS_PROJECT_22_DETERMINISTIC_NOISE_PLAN_AND_C...,PASS_PROJECT_22_DETERMINISTIC_NOISE_PLAN_AND_C...,True
1,Noise-plan checkpoint SHA-256,3dd2b3f38a8d7b77c179e9f518399c1dc5b1c89a8a2960...,3dd2b3f38a8d7b77c179e9f518399c1dc5b1c89a8a2960...,True
2,Step 3A output-manifest failures,0,0,True
3,Source root SHA-256,281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64...,281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64...,True
4,Raw training rows,172628,172628,True
5,Raw evaluation rows,67625,67625,True
6,Model training rows,95812,95812,True
7,Model evaluation rows,22156,22156,True
8,Model training failures,207,207,True
9,Model evaluation failures,40,40,True



Failed Project 22 Step 4A checks:


,Check,Expected,Actual,Pass
28,Registry rows,20,21,False



No Step 4A PASS status or checkpoint was written.


RuntimeError: PROJECT 22 STEP 4A VALIDATION FAILED.

In [11]:
# ==================================================================================================
# PROJECT 22 — CELL 7 / STEP 4A
# EXPERIMENT RUNTIME, MODEL, BASELINE, METRIC, AND PREDICTOR CONTRACT FREEZE
#
# PROJECT:
#   apache@logging-log4j2
#
# RUN THIS AS THE NEXT NEW CELL IN THE SAME Thesis_project_22.ipynb NOTEBOOK.
#
# PURPOSE:
# - verify the frozen Project 22 Step 3A noise plan and every output in its manifest;
# - validate the fixed 151-predictor training/evaluation matrices;
# - freeze runtime versions, median-imputation, labels, ranking, model, baseline,
#   APFD, and APFDc contracts;
# - validate all four required model implementations without fitting Project 22 models;
# - write the runtime-contract checkpoint required before the two-condition smoke test.
#
# SAFETY:
# - no model fitting;
# - no condition execution;
# - no completion-registry write;
# - no prior-project condition-output access.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from IPython.display import display

import hashlib
import json
import os
import platform
import sys

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


print("=" * 136)
print("=== PROJECT 22 CELL 7 / STEP 4A: EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 22
PROJECT_NAME = "apache@logging-log4j2"
PROJECT_SLUG = "apache__logging-log4j2"
PROJECT_SHORT = "LOG4J2"

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_22_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

STEP4A_STATUS = (
    "PASS_PROJECT_22_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
)

EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "3dd2b3f38a8d7b77c179e9f518399c1dc5b1c89a8a29607ea31fd78a91f40666"
)

EXPECTED_REGISTRY_SHA256 = (
    "79cd6ecb595c5e8ae91a9494e469792716338d144308560a62caf1b9342306b2"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64f842964c896c6ac334"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

EXPECTED_REGISTERED_PROJECTS = 21

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_RAW_TRAIN_ROWS = 172_628
EXPECTED_RAW_EVAL_ROWS = 67_625
EXPECTED_MODEL_TRAIN_ROWS = 95_812
EXPECTED_MODEL_EVAL_ROWS = 22_156
EXPECTED_MODEL_TRAIN_FAILURES = 207
EXPECTED_MODEL_EVAL_FAILURES = 40
EXPECTED_MODEL_FAILING_EVAL_BUILDS = 39

EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_CONDITIONS = 270

EXPECTED_RUNTIME_VERSIONS = {
    "Python": "3.12.13",
    "numpy": "2.0.2",
    "pandas": "2.2.2",
    "scikit-learn": "1.6.1",
    "xgboost": "3.3.0",
    "lightgbm": "4.6.0",
    "pyarrow": "18.1.0",
}

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "max_features": "sqrt",
        "bootstrap": True,
        "n_jobs": -1,
    },
    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
        "eval_metric": "logloss",
    },
    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    },
    "NaiveBayes": {
        "var_smoothing": 1e-9,
    },
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_22_selection"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_22_frozen_source_manifest.csv"
)

SOURCE_DIR = Path(
    "/content/datasets/datasets/apache@logging-log4j2"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

STEP3A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step3a_status.json"
)

STEP3A_REPORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_report.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_22_noise_plan_checkpoint.json"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

PROTOCOL_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_frozen_experiment_protocol.json"
)

RUNTIME_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_runtime_contract"
)

RUNTIME_VERSION_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_runtime_versions.csv"
)

PREDICTOR_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_predictor_contract.csv"
)

CLEAN_MEDIAN_REFERENCE_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_clean_training_median_reference.csv"
)

MODEL_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_model_contract.json"
)

BASELINE_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_baseline_contract.json"
)

METRIC_SELF_TEST_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_metric_self_test.csv"
)

RANKING_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_ranking_contract.json"
)

STEP4A_VALIDATION_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_validation.csv"
)

STEP4A_REPORT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_report.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step4a_status.json"
)

RUNTIME_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_22_runtime_contract_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_json(
    path,
    payload,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def deterministic_seed(
    repetition_seed,
    stream_name,
):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(
        material
    ).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )


def deterministic_random_build_seed(
    repetition_seed,
    build_id,
):
    return deterministic_seed(
        repetition_seed,
        f"Random_baseline_build_{int(build_id)}",
    )


def create_models(
    repetition_seed,
):
    return {
        "RandomForest":
            RandomForestClassifier(
                **MODEL_CONFIG[
                    "RandomForest"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "RandomForest_model",
                ),
            ),

        "XGBoost":
            XGBClassifier(
                **MODEL_CONFIG[
                    "XGBoost"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "XGBoost_model",
                ),
            ),

        "LightGBM":
            LGBMClassifier(
                **MODEL_CONFIG[
                    "LightGBM"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "LightGBM_model",
                ),
            ),

        "NaiveBayes":
            GaussianNB(
                **MODEL_CONFIG[
                    "NaiveBayes"
                ]
            ),
    }


def calculate_apfd(
    failures,
):
    failures = np.asarray(
        failures,
        dtype=np.int8,
    )

    number_of_tests = len(
        failures
    )

    number_of_failures = int(
        failures.sum()
    )

    if (
        number_of_tests == 0
        or number_of_failures == 0
    ):
        return np.nan

    failure_positions = (
        np.flatnonzero(
            failures == 1
        )
        + 1
    )

    return float(
        1.0
        - (
            failure_positions.sum()
            / (
                number_of_tests
                * number_of_failures
            )
        )
        + (
            1.0
            / (
                2.0
                * number_of_tests
            )
        )
    )


def calculate_apfdc(
    failures,
    durations,
):
    failures = np.asarray(
        failures,
        dtype=np.int8,
    )

    durations = np.asarray(
        durations,
        dtype=float,
    )

    if len(failures) != len(durations):
        raise ValueError(
            "failures and durations must have equal length."
        )

    if (
        len(failures) == 0
        or failures.sum() == 0
    ):
        return np.nan

    if not np.isfinite(
        durations
    ).all():
        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (
        durations < 0
    ).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(
        durations.sum()
    )

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array([0.0]),
        np.cumsum(
            durations
        )[:-1],
    ])

    failure_mask = (
        failures == 1
    )

    midpoint_detection_times = (
        cumulative_before[
            failure_mask
        ]
        + (
            0.5
            * durations[
                failure_mask
            ]
        )
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(passed),
    })


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND STEP 3A CHECKPOINT
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    STEP3A_STATUS_PATH,
    STEP3A_REPORT_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    CONDITION_PLAN_PATH,
    PROTOCOL_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 22 Step 4A inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


noise_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)

noise_checkpoint = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)

step3a_status = load_json(
    STEP3A_STATUS_PATH
)

step3a_report = load_json(
    STEP3A_REPORT_PATH
)

protocol = load_json(
    PROTOCOL_PATH
)


if (
    noise_checkpoint_sha256
    != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 22 noise-plan checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256}\n"
        f"Actual:   {noise_checkpoint_sha256}"
    )


for label, payload in [
    (
        "noise checkpoint",
        noise_checkpoint,
    ),
    (
        "Step 3A status",
        step3a_status,
    ),
    (
        "Step 3A report",
        step3a_report,
    ),
]:
    if payload.get(
        "Status"
    ) != EXPECTED_STEP3A_STATUS:
        raise RuntimeError(
            f"{label} does not contain the expected Step 3A PASS status."
        )


if (
    noise_checkpoint.get(
        "Project"
    )
    != PROJECT_NAME
    or noise_checkpoint.get(
        "ProjectSlug"
    )
    != PROJECT_SLUG
):
    raise RuntimeError(
        "The frozen Step 3A Project 22 identity differs."
    )


if (
    noise_checkpoint.get(
        "SourceRootSHA256"
    )
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The Step 3A checkpoint source root differs."
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY STEP 3A OUTPUT MANIFEST
# --------------------------------------------------------------------------------------------------

output_manifest = noise_checkpoint.get(
    "OutputManifest",
    []
)


if not isinstance(
    output_manifest,
    list,
) or not output_manifest:
    raise RuntimeError(
        "The Step 3A checkpoint contains no output manifest."
    )


output_manifest_records = []


for item in output_manifest:
    path = Path(
        item[
            "Path"
        ]
    )

    expected_bytes = int(
        item[
            "Bytes"
        ]
    )

    expected_sha256 = str(
        item[
            "SHA256"
        ]
    )

    exists = path.is_file()

    actual_bytes = (
        int(
            path.stat().st_size
        )
        if exists
        else -1
    )

    actual_sha256 = (
        sha256_file(
            path
        )
        if exists
        else "MISSING"
    )

    output_manifest_records.append({
        "Path":
            str(path),

        "ExpectedBytes":
            expected_bytes,

        "ActualBytes":
            actual_bytes,

        "ExpectedSHA256":
            expected_sha256,

        "ActualSHA256":
            actual_sha256,

        "Pass":
            (
                exists
                and actual_bytes
                == expected_bytes
                and actual_sha256
                == expected_sha256
            ),
    })


output_manifest_audit = pd.DataFrame(
    output_manifest_records
)


output_manifest_failures = int(
    (
        ~output_manifest_audit[
            "Pass"
        ]
    ).sum()
)


if output_manifest_failures:
    print(
        "\nFailed Step 3A output-manifest checks:"
    )

    display(
        output_manifest_audit.loc[
            ~output_manifest_audit[
                "Pass"
            ]
        ]
    )

    raise RuntimeError(
        "One or more frozen Step 3A outputs changed."
    )


# --------------------------------------------------------------------------------------------------
# 6. SOURCE AND REGISTRY IMMUTABILITY
# --------------------------------------------------------------------------------------------------

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_records = []


for row in frozen_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            "A frozen Project 22 source file is missing:\n"
            f"{source_path}"
        )

    current_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


current_source_manifest = pd.DataFrame(
    current_source_records
)


current_source_root_sha256 = source_root_hash(
    current_source_manifest
)


if (
    current_source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "The frozen Project 22 source root differs."
    )


registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "The registry does not contain exactly Projects 1–21."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–21 are not all COMPLETE_AND_FROZEN."
    )


if project_numbers.eq(PROJECT_NUMBER).any():
    raise RuntimeError(
        "Project 22 is unexpectedly already registered."
    )


if registry[project_column].eq(PROJECT_NAME).any():
    raise RuntimeError(
        "The selected Project 22 identity is already registered."
    )


required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
}


for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


active_reservations = []


if active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "The active-reservation state differs from the Project 22 freeze."
    )


# --------------------------------------------------------------------------------------------------
# 7. LOAD AND VALIDATE FIXED COHORTS
# --------------------------------------------------------------------------------------------------

raw_training = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)

raw_evaluation = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)

model_training = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)

model_evaluation = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)

model_raw_train_link = pd.read_parquet(
    MODEL_RAW_TRAIN_LINK_PATH
)

model_raw_eval_link = pd.read_parquet(
    MODEL_RAW_EVAL_LINK_PATH
)

condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)


required_model_columns = {
    "Build",
    "Test",
    "Verdict",
}


if not required_model_columns.issubset(
    model_training.columns
) or not required_model_columns.issubset(
    model_evaluation.columns
):
    raise RuntimeError(
        "Fixed model cohorts are missing Build, Test, or Verdict."
    )


training_metadata_columns = {
    "ModelTrainingRowOrder",
    "Build",
    "Test",
    "Verdict",
}


evaluation_metadata_columns = {
    "ModelEvaluationRowOrder",
    "Build",
    "Test",
    "Verdict",
}


predictor_columns = [
    column
    for column in model_training.columns
    if column not in training_metadata_columns
]


evaluation_predictor_columns = [
    column
    for column in model_evaluation.columns
    if column not in evaluation_metadata_columns
]


if (
    predictor_columns
    != evaluation_predictor_columns
):
    raise RuntimeError(
        "Training and evaluation predictor order differs."
    )


if len(
    predictor_columns
) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "The fixed predictor count differs.\n"
        f"Expected: {EXPECTED_PREDICTORS}\n"
        f"Actual:   {len(predictor_columns)}"
    )


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in predictor_columns
]


if missing_rec_features:
    raise RuntimeError(
        "Fixed predictor cohorts are missing REC features:\n"
        + "\n".join(
            missing_rec_features
        )
    )


model_training_failures = int(
    pd.to_numeric(
        model_training[
            "Verdict"
        ],
        errors="raise",
    ).ne(
        0
    ).sum()
)


model_evaluation_failures = int(
    pd.to_numeric(
        model_evaluation[
            "Verdict"
        ],
        errors="raise",
    ).ne(
        0
    ).sum()
)


model_failing_evaluation_builds = int(
    model_evaluation.loc[
        pd.to_numeric(
            model_evaluation[
                "Verdict"
            ],
            errors="raise",
        ).ne(
            0
        ),
        "Build",
    ].nunique()
)


# --------------------------------------------------------------------------------------------------
# 8. NUMERIC PREDICTORS AND MEDIAN IMPUTATION
# --------------------------------------------------------------------------------------------------

training_numeric = pd.DataFrame(
    index=model_training.index
)

evaluation_numeric = pd.DataFrame(
    index=model_evaluation.index
)

predictor_profile_records = []


for predictor_order, column in enumerate(
    predictor_columns,
    start=1,
):
    training_values = pd.to_numeric(
        model_training[
            column
        ],
        errors="coerce",
    ).astype(
        float
    ).replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )

    evaluation_values = pd.to_numeric(
        model_evaluation[
            column
        ],
        errors="coerce",
    ).astype(
        float
    ).replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )

    training_numeric[
        column
    ] = training_values

    evaluation_numeric[
        column
    ] = evaluation_values

    predictor_profile_records.append({
        "PredictorOrder":
            predictor_order,

        "Predictor":
            column,

        "IsREC":
            column in REC_FEATURES,

        "RECClass":
            (
                "VERDICT_DEPENDENT"
                if column
                in VERDICT_DEPENDENT_REC
                else (
                    "VERDICT_INDEPENDENT"
                    if column
                    in VERDICT_INDEPENDENT_REC
                    else ""
                )
            ),

        "TrainingRows":
            len(
                training_values
            ),

        "TrainingNonMissing":
            int(
                training_values.notna().sum()
            ),

        "TrainingMissing":
            int(
                training_values.isna().sum()
            ),

        "EvaluationRows":
            len(
                evaluation_values
            ),

        "EvaluationMissing":
            int(
                evaluation_values.isna().sum()
            ),

        "AllTrainingValuesMissing":
            bool(
                training_values.notna().sum()
                == 0
            ),
    })


predictor_contract = pd.DataFrame(
    predictor_profile_records
)


all_missing_predictors = predictor_contract.loc[
    predictor_contract[
        "AllTrainingValuesMissing"
    ],
    "Predictor",
].tolist()


if all_missing_predictors:
    raise RuntimeError(
        "One or more predictors are entirely missing in training:\n"
        + "\n".join(
            all_missing_predictors
        )
    )


clean_training_medians = training_numeric.median(
    axis=0,
    skipna=True,
)


if (
    clean_training_medians.isna().any()
    or not np.isfinite(
        clean_training_medians.to_numpy(
            dtype=float
        )
    ).all()
):
    raise RuntimeError(
        "Clean training medians contain missing or infinite values."
    )


training_imputed = training_numeric.fillna(
    clean_training_medians
)

evaluation_imputed = evaluation_numeric.fillna(
    clean_training_medians
)


training_nonfinite_after_imputation = int(
    (
        ~np.isfinite(
            training_imputed.to_numpy(
                dtype=float
            )
        )
    ).sum()
)


evaluation_nonfinite_after_imputation = int(
    (
        ~np.isfinite(
            evaluation_imputed.to_numpy(
                dtype=float
            )
        )
    ).sum()
)


clean_median_reference = pd.DataFrame({
    "PredictorOrder":
        np.arange(
            1,
            len(
                predictor_columns
            )
            + 1,
            dtype=np.int64,
        ),

    "Predictor":
        predictor_columns,

    "CleanTrainingMedian":
        clean_training_medians[
            predictor_columns
        ].to_numpy(
            dtype=float
        ),
})


# --------------------------------------------------------------------------------------------------
# 9. LABEL CONTRACT
# --------------------------------------------------------------------------------------------------

training_binary_labels = (
    pd.to_numeric(
        model_training[
            "Verdict"
        ],
        errors="raise",
    )
    .ne(
        0
    )
    .astype(
        np.int8
    )
)


evaluation_binary_labels = (
    pd.to_numeric(
        model_evaluation[
            "Verdict"
        ],
        errors="raise",
    )
    .ne(
        0
    )
    .astype(
        np.int8
    )
)


training_label_values = sorted(
    training_binary_labels.unique().tolist()
)


evaluation_label_values = sorted(
    evaluation_binary_labels.unique().tolist()
)


# --------------------------------------------------------------------------------------------------
# 10. RUNTIME VERSION CONTRACT
# --------------------------------------------------------------------------------------------------

runtime_versions = pd.DataFrame([
    {
        "Component":
            "Python",

        "Version":
            platform.python_version(),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "Python"
            ],
    },
    {
        "Component":
            "numpy",

        "Version":
            np.__version__,

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "numpy"
            ],
    },
    {
        "Component":
            "pandas",

        "Version":
            pd.__version__,

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "pandas"
            ],
    },
    {
        "Component":
            "scikit-learn",

        "Version":
            metadata.version(
                "scikit-learn"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "scikit-learn"
            ],
    },
    {
        "Component":
            "xgboost",

        "Version":
            metadata.version(
                "xgboost"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "xgboost"
            ],
    },
    {
        "Component":
            "lightgbm",

        "Version":
            metadata.version(
                "lightgbm"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "lightgbm"
            ],
    },
    {
        "Component":
            "pyarrow",

        "Version":
            metadata.version(
                "pyarrow"
            ),

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                "pyarrow"
            ],
    },
])


runtime_versions[
    "Pass"
] = runtime_versions[
    "Version"
].eq(
    runtime_versions[
        "ExpectedVersion"
    ]
)


runtime_version_failures = int(
    (
        ~runtime_versions[
            "Pass"
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 11. MODEL IMPLEMENTATION CONTRACT
# --------------------------------------------------------------------------------------------------

models_seed_1 = create_models(
    repetition_seed=1
)

models_seed_1_repeat = create_models(
    repetition_seed=1
)

models_seed_2 = create_models(
    repetition_seed=2
)


model_contract_records = []


for technique in ML_TECHNIQUES:
    model_a = models_seed_1[
        technique
    ]

    model_b = models_seed_1_repeat[
        technique
    ]

    model_c = models_seed_2[
        technique
    ]

    parameters_a = model_a.get_params(
        deep=False
    )

    parameters_b = model_b.get_params(
        deep=False
    )

    parameters_c = model_c.get_params(
        deep=False
    )

    random_state_a = parameters_a.get(
        "random_state",
        None,
    )

    random_state_c = parameters_c.get(
        "random_state",
        None,
    )

    model_contract_records.append({
        "Technique":
            technique,

        "EstimatorClass":
            (
                f"{model_a.__class__.__module__}."
                f"{model_a.__class__.__name__}"
            ),

        "Seed1RandomState":
            random_state_a,

        "Seed2RandomState":
            random_state_c,

        "SameSeedSameConfiguration":
            parameters_a
            == parameters_b,

        "DifferentSeedStateAsExpected":
            (
                True
                if technique
                == "NaiveBayes"
                else random_state_a
                != random_state_c
            ),

        "ConfigurationJSON":
            json.dumps(
                parameters_a,
                sort_keys=True,
                default=str,
            ),
    })


model_contract_table = pd.DataFrame(
    model_contract_records
)


rf_params = models_seed_1[
    "RandomForest"
].get_params(
    deep=False
)

xgb_params = models_seed_1[
    "XGBoost"
].get_params(
    deep=False
)

lgbm_params = models_seed_1[
    "LightGBM"
].get_params(
    deep=False
)

nb_params = models_seed_1[
    "NaiveBayes"
].get_params(
    deep=False
)


model_parameter_checks = {
    "RandomForest": (
        rf_params.get(
            "n_estimators"
        )
        == 100
        and rf_params.get(
            "max_features"
        )
        == "sqrt"
        and rf_params.get(
            "bootstrap"
        )
        is True
        and rf_params.get(
            "n_jobs"
        )
        == -1
    ),

    "XGBoost": (
        xgb_params.get(
            "n_estimators"
        )
        == 100
        and xgb_params.get(
            "max_depth"
        )
        == 6
        and np.isclose(
            float(
                xgb_params.get(
                    "learning_rate"
                )
            ),
            0.1,
        )
        and xgb_params.get(
            "tree_method"
        )
        == "hist"
        and xgb_params.get(
            "n_jobs"
        )
        == -1
    ),

    "LightGBM": (
        lgbm_params.get(
            "n_estimators"
        )
        == 100
        and np.isclose(
            float(
                lgbm_params.get(
                    "learning_rate"
                )
            ),
            0.1,
        )
        and lgbm_params.get(
            "num_leaves"
        )
        == 31
        and lgbm_params.get(
            "deterministic"
        )
        is True
        and lgbm_params.get(
            "force_col_wise"
        )
        is True
        and lgbm_params.get(
            "n_jobs"
        )
        == -1
    ),

    "NaiveBayes": (
        np.isclose(
            float(
                nb_params.get(
                    "var_smoothing"
                )
            ),
            1e-9,
        )
    ),
}


model_contract_failures = int(
    (
        ~model_contract_table[
            "SameSeedSameConfiguration"
        ]
        | ~model_contract_table[
            "DifferentSeedStateAsExpected"
        ]
    ).sum()
    + sum(
        not bool(value)
        for value in model_parameter_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 12. BASELINE, RANKING, AND RANDOM CONTRACT
# --------------------------------------------------------------------------------------------------

sample_build_id = int(
    model_evaluation[
        "Build"
    ].iloc[
        0
    ]
)


random_seed_1_a = deterministic_random_build_seed(
    repetition_seed=1,
    build_id=sample_build_id,
)

random_seed_1_b = deterministic_random_build_seed(
    repetition_seed=1,
    build_id=sample_build_id,
)

random_seed_2 = deterministic_random_build_seed(
    repetition_seed=2,
    build_id=sample_build_id,
)


sample_random_a = np.random.default_rng(
    random_seed_1_a
).random(
    100
)

sample_random_b = np.random.default_rng(
    random_seed_1_b
).random(
    100
)

sample_random_c = np.random.default_rng(
    random_seed_2
).random(
    100
)


random_same_seed_reproduced = bool(
    np.array_equal(
        sample_random_a,
        sample_random_b,
    )
)


random_different_seed_differs = bool(
    not np.array_equal(
        sample_random_a,
        sample_random_c,
    )
)


ranking_contract = {
    "ML": {
        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",
    },

    "Random": {
        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",

        "SeedRule":
            (
                "first little-endian uint32 of "
                "SHA-256(project|repetition_seed|"
                "Random_baseline_build_<BuildID>)"
            ),

        "ConstantAcrossNoiseForSameSeedAndBuild":
            True,
    },

    "LatestFail": {
        "SourceFeature":
            "REC_LastFailureAge",

        "ScoreFormula":
            "-REC_LastFailureAge",

        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",

        "NoiseDependent":
            True,

        "UsesSameCorruptedHistoryAsML":
            True,
    },

    "QTF-Avg": {
        "SourceFeature":
            "REC_TotalAvgExeTime",

        "Direction":
            "ascending",

        "TieBreak":
            "Test ascending",

        "NoiseDependent":
            False,
    },
}


baseline_contract = {
    "Techniques":
        BASELINE_TECHNIQUES,

    "Random":
        ranking_contract[
            "Random"
        ],

    "LatestFail":
        ranking_contract[
            "LatestFail"
        ],

    "QTF-Avg":
        ranking_contract[
            "QTF-Avg"
        ],

    "NoRollingRetraining":
        True,

    "CleanEvaluationPartition":
        True,
}


# --------------------------------------------------------------------------------------------------
# 13. APFD/APFDc SELF-TESTS
# --------------------------------------------------------------------------------------------------

manual_failures = np.array([
    1,
    1,
    0,
    0,
    0,
], dtype=np.int8)


manual_apfd = calculate_apfd(
    manual_failures
)


manual_apfdc_slow_failure_first = calculate_apfdc(
    manual_failures,
    np.array([
        5.0,
        1.0,
        1.0,
        1.0,
        1.0,
    ]),
)


manual_apfdc_fast_failure_first = calculate_apfdc(
    manual_failures,
    np.array([
        1.0,
        5.0,
        1.0,
        1.0,
        1.0,
    ]),
)


all_pass_apfd = calculate_apfd(
    np.array([
        0,
        0,
        0,
    ])
)


all_pass_apfdc = calculate_apfdc(
    np.array([
        0,
        0,
        0,
    ]),
    np.array([
        1.0,
        1.0,
        1.0,
    ]),
)


metric_self_test = pd.DataFrame([
    {
        "Check":
            "Manual APFD",

        "Expected":
            0.8,

        "Actual":
            manual_apfd,

        "Pass":
            np.isclose(
                manual_apfd,
                0.8,
                rtol=0,
                atol=1e-15,
            ),
    },

    {
        "Check":
            "APFDc rewards quick failing test first",

        "Expected":
            True,

        "Actual":
            manual_apfdc_fast_failure_first
            > manual_apfdc_slow_failure_first,

        "Pass":
            manual_apfdc_fast_failure_first
            > manual_apfdc_slow_failure_first,
    },

    {
        "Check":
            "All-pass APFD is NaN",

        "Expected":
            True,

        "Actual":
            bool(
                np.isnan(
                    all_pass_apfd
                )
            ),

        "Pass":
            bool(
                np.isnan(
                    all_pass_apfd
                )
            ),
    },

    {
        "Check":
            "All-pass APFDc is NaN",

        "Expected":
            True,

        "Actual":
            bool(
                np.isnan(
                    all_pass_apfdc
                )
            ),

        "Pass":
            bool(
                np.isnan(
                    all_pass_apfdc
                )
            ),
    },
])


metric_self_test_failures = int(
    (
        ~metric_self_test[
            "Pass"
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 14. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


add_check(
    validation_records,
    "Step 3A status",
    EXPECTED_STEP3A_STATUS,
    step3a_status.get(
        "Status"
    ),
    step3a_status.get(
        "Status"
    )
    == EXPECTED_STEP3A_STATUS,
)


add_check(
    validation_records,
    "Noise-plan checkpoint SHA-256",
    EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
    noise_checkpoint_sha256,
    noise_checkpoint_sha256
    == EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
)


add_check(
    validation_records,
    "Step 3A output-manifest failures",
    0,
    output_manifest_failures,
    output_manifest_failures
    == 0,
)


add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)


add_check(
    validation_records,
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training
    ),
    len(
        raw_training
    )
    == EXPECTED_RAW_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation
    ),
    len(
        raw_evaluation
    )
    == EXPECTED_RAW_EVAL_ROWS,
)


add_check(
    validation_records,
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training
    ),
    len(
        model_training
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation
    ),
    len(
        model_evaluation
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)


add_check(
    validation_records,
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    model_training_failures,
    model_training_failures
    == EXPECTED_MODEL_TRAIN_FAILURES,
)


add_check(
    validation_records,
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    model_evaluation_failures,
    model_evaluation_failures
    == EXPECTED_MODEL_EVAL_FAILURES,
)


add_check(
    validation_records,
    "Model failing evaluation builds",
    EXPECTED_MODEL_FAILING_EVAL_BUILDS,
    model_failing_evaluation_builds,
    model_failing_evaluation_builds
    == EXPECTED_MODEL_FAILING_EVAL_BUILDS,
)


add_check(
    validation_records,
    "Condition-plan rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan
    ),
    len(
        condition_plan
    )
    == EXPECTED_CONDITIONS,
)


add_check(
    validation_records,
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    )
    == EXPECTED_PREDICTORS,
)


add_check(
    validation_records,
    "REC features",
    EXPECTED_REC_FEATURES,
    len(
        REC_FEATURES
    ),
    (
        len(
            REC_FEATURES
        )
        == EXPECTED_REC_FEATURES
        and not missing_rec_features
    ),
)


add_check(
    validation_records,
    "Raw-training link rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_raw_train_link
    ),
    len(
        model_raw_train_link
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)


add_check(
    validation_records,
    "Raw-evaluation link rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_raw_eval_link
    ),
    len(
        model_raw_eval_link
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)


add_check(
    validation_records,
    "Training binary label values",
    [0, 1],
    training_label_values,
    training_label_values
    == [
        0,
        1,
    ],
)


add_check(
    validation_records,
    "Evaluation binary label values",
    [0, 1],
    evaluation_label_values,
    evaluation_label_values
    == [
        0,
        1,
    ],
)


add_check(
    validation_records,
    "All-missing predictors",
    0,
    len(
        all_missing_predictors
    ),
    len(
        all_missing_predictors
    )
    == 0,
)


add_check(
    validation_records,
    "Training non-finite values after imputation",
    0,
    training_nonfinite_after_imputation,
    training_nonfinite_after_imputation
    == 0,
)


add_check(
    validation_records,
    "Evaluation non-finite values after imputation",
    0,
    evaluation_nonfinite_after_imputation,
    evaluation_nonfinite_after_imputation
    == 0,
)


add_check(
    validation_records,
    "Runtime-version failures",
    0,
    runtime_version_failures,
    runtime_version_failures
    == 0,
)


add_check(
    validation_records,
    "Model contract failures",
    0,
    model_contract_failures,
    model_contract_failures
    == 0,
)


add_check(
    validation_records,
    "Metric self-test failures",
    0,
    metric_self_test_failures,
    metric_self_test_failures
    == 0,
)


add_check(
    validation_records,
    "Random same-seed reproducible",
    True,
    random_same_seed_reproduced,
    random_same_seed_reproduced,
)


add_check(
    validation_records,
    "Random different-seed differs",
    True,
    random_different_seed_differs,
    random_different_seed_differs,
)


add_check(
    validation_records,
    "LatestFail feature present",
    True,
    (
        "REC_LastFailureAge"
        in predictor_columns
    ),
    (
        "REC_LastFailureAge"
        in predictor_columns
    ),
)


add_check(
    validation_records,
    "QTF-Avg feature present",
    True,
    (
        "REC_TotalAvgExeTime"
        in predictor_columns
    ),
    (
        "REC_TotalAvgExeTime"
        in predictor_columns
    ),
)


add_check(
    validation_records,
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(
        registry
    ),
    len(
        registry
    )
    == EXPECTED_REGISTERED_PROJECTS,
)


for predecessor_number, predecessor_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            project_numbers.eq(
                predecessor_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        actual_project,
        actual_project == predecessor_project,
    )


add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    active_reservations,
    active_reservations == EXPECTED_ACTIVE_RESERVATIONS,
)


add_check(
    validation_records,
    "Project 22 registry rows",
    0,
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)


add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    EXPECTED_RUNTIME_PRIORITY_RULE,
    True,
)


add_check(
    validation_records,
    "No rolling retraining",
    True,
    bool(
        baseline_contract[
            "NoRollingRetraining"
        ]
    ),
    bool(
        baseline_contract[
            "NoRollingRetraining"
        ]
    ),
)


add_check(
    validation_records,
    "Evaluation partition clean and fixed",
    True,
    bool(
        baseline_contract[
            "CleanEvaluationPartition"
        ]
    ),
    bool(
        baseline_contract[
            "CleanEvaluationPartition"
        ]
    ),
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 22 Step 4A validation:"
)

display(
    validation
)


if not failed_validation.empty:
    print(
        "\nFailed Project 22 Step 4A checks:"
    )

    display(
        failed_validation
    )

    print(
        "\nNo Step 4A PASS status or checkpoint was written."
    )

    raise RuntimeError(
        "PROJECT 22 STEP 4A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 15. WRITE CONTRACT OUTPUTS
# --------------------------------------------------------------------------------------------------

RUNTIME_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_csv(
    RUNTIME_VERSION_PATH,
    runtime_versions,
)


atomic_csv(
    PREDICTOR_CONTRACT_PATH,
    predictor_contract,
)


atomic_csv(
    CLEAN_MEDIAN_REFERENCE_PATH,
    clean_median_reference,
)


atomic_csv(
    METRIC_SELF_TEST_PATH,
    metric_self_test,
)


atomic_csv(
    STEP4A_VALIDATION_PATH,
    validation,
)


model_contract_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "PositiveClass": {
        "Name":
            "failure",

        "Value":
            1,

        "Conversion":
            "binary target = (Verdict != 0).astype(int)",
    },

    "PredictorCount":
        len(
            predictor_columns
        ),

    "Imputation": {
        "Rule":
            (
                "For every condition, compute one median per active "
                "predictor from that condition's training matrix only. "
                "Replace +/-infinity with missing before computing medians. "
                "Use those training medians to fill training and clean "
                "evaluation missing values."
            ),

        "Scaling":
            "none",

        "ActivePredictors":
            "all 151 fixed predictor columns",
    },

    "Models":
        MODEL_CONFIG,

    "ModelSeeds": {
        "RandomForest":
            "SHA-256(project|repetition_seed|RandomForest_model)",

        "XGBoost":
            "SHA-256(project|repetition_seed|XGBoost_model)",

        "LightGBM":
            "SHA-256(project|repetition_seed|LightGBM_model)",

        "NaiveBayes":
            "deterministic; no random_state parameter",
    },

    "PositiveProbabilityExtraction":
        (
            "Use predict_proba and select the column whose "
            "fitted classes_ value equals 1."
        ),

    "NoRollingRetraining":
        True,

    "ModelImplementationAudit":
        model_contract_table.to_dict(
            orient="records"
        ),
}


atomic_json(
    MODEL_CONTRACT_PATH,
    model_contract_payload,
)


atomic_json(
    BASELINE_CONTRACT_PATH,
    baseline_contract,
)


atomic_json(
    RANKING_CONTRACT_PATH,
    ranking_contract,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


output_paths = [
    RUNTIME_VERSION_PATH,
    PREDICTOR_CONTRACT_PATH,
    CLEAN_MEDIAN_REFERENCE_PATH,
    MODEL_CONTRACT_PATH,
    BASELINE_CONTRACT_PATH,
    METRIC_SELF_TEST_PATH,
    RANKING_CONTRACT_PATH,
    STEP4A_VALIDATION_PATH,
]


runtime_output_manifest = [
    {
        "Path":
            str(path),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "NoisePlanCheckpointSHA256":
        noise_checkpoint_sha256,

    "RuntimeVersions":
        runtime_versions.to_dict(
            orient="records"
        ),

    "PredictorCount":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "MLTechniques":
        ML_TECHNIQUES,

    "BaselineTechniques":
        BASELINE_TECHNIQUES,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "PositiveClass":
        "failure = 1",

    "MedianImputation":
        "condition-training medians",

    "RankingTieBreak":
        "Test ascending",

    "NoRollingRetraining":
        True,

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "ModelTrainingFailures":
        model_training_failures,

    "ModelEvaluationFailures":
        model_evaluation_failures,

    "ModelFailingEvaluationBuilds":
        model_failing_evaluation_builds,

    "RuntimeOutputManifest":
        runtime_output_manifest,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To21Modified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project20ModelsFitted":
        False,

    "FullExperimentStarted":
        False,
}


atomic_json(
    STEP4A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "RuntimeContractCheckpoint":
        True,

    "DoNotChangePredictorSet":
        True,

    "DoNotChangeModelConfiguration":
        True,

    "DoNotChangeBaselineDefinitions":
        True,

    "DoNotChangeRankingRules":
        True,

    "DoNotChangeMetricDefinitions":
        True,

    "ReadyForTwoConditionSmokeTest":
        True,
}


atomic_json(
    RUNTIME_CHECKPOINT_PATH,
    checkpoint_payload,
)


runtime_checkpoint_sha256 = sha256_file(
    RUNTIME_CHECKPOINT_PATH
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "NoisePlanCheckpointSHA256":
        noise_checkpoint_sha256,

    "PredictorCount":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "MLTechniques":
        ML_TECHNIQUES,

    "BaselineTechniques":
        BASELINE_TECHNIQUES,

    "Checkpoint":
        str(
            RUNTIME_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        runtime_checkpoint_sha256,

    "ReadyForTwoConditionSmokeTest":
        True,

    "RegistryModified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "Project20ModelsFitted":
        False,
}


atomic_json(
    STEP4A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 16. READBACK AND IMMUTABILITY
# --------------------------------------------------------------------------------------------------

checkpoint_readback = load_json(
    RUNTIME_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP4A_STATUS_PATH
)


if checkpoint_readback.get(
    "Status"
) != STEP4A_STATUS:
    raise RuntimeError(
        "Project 22 runtime checkpoint readback failed."
    )


if status_readback.get(
    "Status"
) != STEP4A_STATUS:
    raise RuntimeError(
        "Project 22 Step 4A status readback failed."
    )


runtime_manifest_readback_failures = 0


for item in checkpoint_readback.get(
    "RuntimeOutputManifest",
    [],
):
    path = Path(
        item[
            "Path"
        ]
    )

    if (
        not path.is_file()
        or int(
            path.stat().st_size
        )
        != int(
            item[
                "Bytes"
            ]
        )
        or sha256_file(
            path
        )
        != str(
            item[
                "SHA256"
            ]
        )
    ):
        runtime_manifest_readback_failures += 1


if runtime_manifest_readback_failures != 0:
    raise RuntimeError(
        "One or more frozen runtime-contract outputs failed readback."
    )


registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "The completion registry changed during Project 22 Step 4A."
    )


if sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
) != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256:
    raise RuntimeError(
        "The frozen Project 22 noise-plan checkpoint changed during Step 4A."
    )


final_source_records = []


for row in current_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_source_manifest = pd.DataFrame(
    final_source_records
)


if source_root_hash(
    final_source_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "The frozen Project 22 source changed during Step 4A."
    )


# --------------------------------------------------------------------------------------------------
# 17. DISPLAY
# --------------------------------------------------------------------------------------------------

print(
    "\nRuntime versions:"
)

display(
    runtime_versions
)


print(
    "\nPredictor contract summary:"
)

display(
    predictor_contract.groupby(
        [
            "IsREC",
            "RECClass",
        ],
        dropna=False,
        as_index=False,
    ).agg(
        Predictors=(
            "Predictor",
            "count",
        ),

        TrainingMissingValues=(
            "TrainingMissing",
            "sum",
        ),

        EvaluationMissingValues=(
            "EvaluationMissing",
            "sum",
        ),
    )
)


print(
    "\nModel implementation contract:"
)

display(
    model_contract_table[
        [
            "Technique",
            "EstimatorClass",
            "Seed1RandomState",
            "Seed2RandomState",
            "SameSeedSameConfiguration",
            "DifferentSeedStateAsExpected",
        ]
    ]
)


print(
    "\nMetric self-tests:"
)

display(
    metric_self_test
)


print(
    "\nStep 3A output-manifest audit:"
)

display(
    output_manifest_audit[
        [
            "Path",
            "ExpectedBytes",
            "ActualBytes",
            "Pass",
        ]
    ]
)


# --------------------------------------------------------------------------------------------------
# 18. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print("\n")
print("=" * 136)
print("=== PROJECT 22 CELL 7 / STEP 4A RESULT ===")
print("=" * 136)


print(
    "\nProject:"
)

print(
    PROJECT_NAME
)

for predecessor_number in sorted(required_registered_identities):
    print(
        f"Project {predecessor_number} identity:",
        required_registered_identities[
            predecessor_number
        ],
    )

print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)


print(
    "\nFrozen experiment contract:"
)

print(
    "Predictors:",
    len(
        predictor_columns
    ),
)

print(
    "REC features:",
    len(
        REC_FEATURES
    ),
)

print(
    "ML techniques:",
    ML_TECHNIQUES,
)

print(
    "Baselines:",
    BASELINE_TECHNIQUES,
)

print(
    "Primary / secondary metrics:",
    "APFDc / APFD",
)

print(
    "Positive class:",
    "failure = 1",
)

print(
    "Median imputation:",
    "condition-training medians",
)

print(
    "Ranking tie-break:",
    "Test ascending",
)

print(
    "Rolling retraining:",
    False,
)


print(
    "\nFixed cohorts:"
)

print(
    "Model training rows:",
    len(
        model_training
    ),
)

print(
    "Model evaluation rows:",
    len(
        model_evaluation
    ),
)

print(
    "Training failures:",
    model_training_failures,
)

print(
    "Evaluation failures:",
    model_evaluation_failures,
)

print(
    "Failing evaluation builds:",
    model_failing_evaluation_builds,
)


print(
    "\nRuntime validation:"
)

print(
    "Step 3A output-manifest failures:",
    output_manifest_failures,
)

print(
    "Runtime-version failures:",
    runtime_version_failures,
)

print(
    "All-missing predictors:",
    len(
        all_missing_predictors
    ),
)

print(
    "Training non-finite values after imputation:",
    training_nonfinite_after_imputation,
)

print(
    "Evaluation non-finite values after imputation:",
    evaluation_nonfinite_after_imputation,
)

print(
    "Model contract failures:",
    model_contract_failures,
)

print(
    "Metric self-test failures:",
    metric_self_test_failures,
)

print(
    "Random same-seed reproducible:",
    random_same_seed_reproduced,
)


print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–21 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Project 22 models fitted:",
    False,
)

print(
    "Full experiment started:",
    False,
)


print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)


print(
    "\nRuntime-contract checkpoint:"
)

print(
    RUNTIME_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    runtime_checkpoint_sha256,
)


print(
    "\nSTATUS:",
    STEP4A_STATUS,
)

print("=" * 136)


=== PROJECT 22 CELL 7 / STEP 4A: EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE ===


/tmp/ipykernel_7490/2069752735.py:1378: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  training_numeric[
/tmp/ipykernel_7490/2069752735.py:1382: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  evaluation_numeric[
/tmp/ipykernel_7490/2069752735.py:1378: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train


Project 22 Step 4A validation:


,Check,Expected,Actual,Pass
0,Step 3A status,PASS_PROJECT_22_DETERMINISTIC_NOISE_PLAN_AND_C...,PASS_PROJECT_22_DETERMINISTIC_NOISE_PLAN_AND_C...,True
1,Noise-plan checkpoint SHA-256,3dd2b3f38a8d7b77c179e9f518399c1dc5b1c89a8a2960...,3dd2b3f38a8d7b77c179e9f518399c1dc5b1c89a8a2960...,True
2,Step 3A output-manifest failures,0,0,True
3,Source root SHA-256,281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64...,281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64...,True
4,Raw training rows,172628,172628,True
5,Raw evaluation rows,67625,67625,True
6,Model training rows,95812,95812,True
7,Model evaluation rows,22156,22156,True
8,Model training failures,207,207,True
9,Model evaluation failures,40,40,True



Runtime versions:


...


Metric self-tests:


,Check,Expected,Actual,Pass
0,Manual APFD,0.8,0.8,True
1,APFDc rewards quick failing test first,True,True,True
2,All-pass APFD is NaN,True,True,True
3,All-pass APFDc is NaN,True,True,True



Step 3A output-manifest audit:


,Path,ExpectedBytes,ActualBytes,Pass
0,/content/drive/MyDrive/Thesis_Experiment/Resul...,778868,778868,True
1,/content/drive/MyDrive/Thesis_Experiment/Resul...,354079,354079,True
2,/content/drive/MyDrive/Thesis_Experiment/Resul...,5135132,5135132,True
3,/content/drive/MyDrive/Thesis_Experiment/Resul...,1674187,1674187,True
4,/content/drive/MyDrive/Thesis_Experiment/Resul...,756250,756250,True
5,/content/drive/MyDrive/Thesis_Experiment/Resul...,168928,168928,True
6,/content/drive/MyDrive/Thesis_Experiment/Resul...,95,95,True
7,/content/drive/MyDrive/Thesis_Experiment/Resul...,5281,5281,True
8,/content/drive/MyDrive/Thesis_Experiment/Resul...,71791004,71791004,True
9,/content/drive/MyDrive/Thesis_Experiment/Resul...,85817,85817,True




=== PROJECT 22 CELL 7 / STEP 4A RESULT ===

Project:
apache@logging-log4j2
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Project 17 identity: yamcs@Yamcs
Project 18 identity: cantaloupe-project@cantaloupe
Project 19 identity: EMResearch@EvoMaster
Project 20 identity: apache@curator
Project 21 identity: facebook@buck
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']

Frozen experiment contract:
Predictors: 151
REC features: 19
ML techniques: ['RandomForest', 'XGBoost', 'LightGBM', 'NaiveBayes']
Baselines: ['Random', 'LatestFail', 'QTF-Avg']
Primary / secondary metrics: APFDc / APFD
Positive class: failure = 1
Median imputation: condition-training medians
Ranking tie-break: Test ascending
Rolling re

In [12]:
# ==================================================================================================
# PROJECT 22 — CELL 8 / STEP 4B
# TWO-CONDITION END-TO-END SMOKE TEST
#
# PROJECT:
#   apache@logging-log4j2
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_22.ipynb.
#
# SMOKE CONDITIONS:
# - 0% noise, repetition seed 1
# - 50% noise, repetition seed 1
#
# THIS CELL:
# - verifies the frozen Step 4A runtime/model contract;
# - reconstructs condition-specific dependent REC features;
# - preserves all six verdict-independent REC features;
# - applies the frozen clean-anchor offsets;
# - trains all four ML techniques once per smoke condition;
# - evaluates ML plus Random, LatestFail, and QTF-Avg;
# - validates APFDc/APFD outputs and baseline invariance;
# - writes only Project 22 smoke-test outputs and checkpoint/status files;
# - does not modify the registry or full 270-condition raw-result root.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import gc
import hashlib
import json
import os
import shutil
import time
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from pandas.errors import PerformanceWarning
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.simplefilter("ignore", PerformanceWarning)


print("=" * 136)
print("=== PROJECT 22 CELL 8 / STEP 4B: TWO-CONDITION END-TO-END SMOKE TEST ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 22
PROJECT_NAME = "apache@logging-log4j2"
PROJECT_SLUG = "apache__logging-log4j2"
PROJECT_SHORT = "LOG4J2"

EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_22_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_22_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

EXPECTED_STEP4A_STATUS = (
    "PASS_PROJECT_22_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
)

EXPECTED_REC_IMPLEMENTATION_VERSION = (
    "PROJECT_22_V2_ANCHOR_AWARE_TIE_INFERENCE_FULL_MAPPING"
)

STEP4B_STATUS = (
    "PASS_PROJECT_22_TWO_CONDITION_END_TO_END_SMOKE_TEST"
)

CONDITION_STATUS = (
    "PASS_PROJECT_22_SMOKE_CONDITION"
)

EXPECTED_RUNTIME_CHECKPOINT_SHA256 = (
    "3ef12ada698780da80e05fd6899b0fd98fc5dcc4f3e74c87ec8ea326e3f4f593"
)

EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "3dd2b3f38a8d7b77c179e9f518399c1dc5b1c89a8a29607ea31fd78a91f40666"
)

EXPECTED_REC_CHECKPOINT_SHA256 = (
    "8417249bc74e2a50e2f61cc3b7776dab7f731f62993e3ed15e415745be178731"
)

EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "a7387d9495c71dce5d2c21ff08b0afd80d9c5251b5885a82e4052c7506ee7890"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64f842964c896c6ac334"
)

EXPECTED_REGISTRY_SHA256 = (
    "79cd6ecb595c5e8ae91a9494e469792716338d144308560a62caf1b9342306b2"
)

EXPECTED_BUILDS = 441
EXPECTED_RAW_ROWS = 240_253
EXPECTED_RAW_TRAIN_ROWS = 172_628
EXPECTED_RAW_EVAL_ROWS = 67_625
EXPECTED_MODEL_TRAIN_ROWS = 95_812
EXPECTED_MODEL_EVAL_ROWS = 22_156
EXPECTED_MODEL_ROWS = 117_968
EXPECTED_MODEL_TRAIN_FAILURES = 207
EXPECTED_MODEL_EVAL_FAILURES = 40
EXPECTED_FAILING_EVAL_BUILDS = 39
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_EVALUATION_BUILDS = 111
EXPECTED_SMOKE_CONDITIONS = 2
EXPECTED_TECHNIQUES = 7
EXPECTED_ML_TECHNIQUES = 4
EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_MODEL_EVAL_ROWS * EXPECTED_TECHNIQUES
)
EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = (
    EXPECTED_FAILING_EVAL_BUILDS * EXPECTED_TECHNIQUES
)
EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = EXPECTED_TECHNIQUES
EXPECTED_MODEL_FIT_ROWS_PER_CONDITION = EXPECTED_ML_TECHNIQUES
EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION = EXPECTED_PREDICTORS
EXPECTED_RNG_MANIFEST_ROWS = 5_178_840
EXPECTED_REGISTERED_PROJECTS = 21
EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

SMOKE_CONDITION_IDS = [
    "noise_00__seed_01",
    "noise_50__seed_01",
]

SMOKE_NOISE_LEVELS = [
    0,
    50,
]

SMOKE_REPETITION_SEED = 1
RECENT_WINDOW = 6

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "max_features": "sqrt",
        "bootstrap": True,
        "n_jobs": -1,
    },
    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
        "eval_metric": "logloss",
    },
    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    },
    "NaiveBayes": {
        "var_smoothing": 1e-9,
    },
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SOURCE_DIR = Path(
    "/content/datasets/datasets/apache@logging-log4j2"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_22_selection"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_22_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_22_fixed_chronological_builds.csv"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_22_selection_checkpoint.json"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

REC_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

BUILD_ENTITY_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
)

CLEAN_RECONSTRUCTED_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
)

CLEAN_ANCHOR_OFFSETS_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
)

INFERRED_EXECUTION_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_inferred_execution_order.parquet"
)

FROZEN_GLOBAL_BUILD_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_global_build_order.csv"
)

REC_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_22_rec_reconstruction_checkpoint.json"
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

RNG_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_rng_manifest.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_22_noise_plan_checkpoint.json"
)

RUNTIME_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_runtime_contract"
)

PREDICTOR_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_predictor_contract.csv"
)

MODEL_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_model_contract.json"
)

BASELINE_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_baseline_contract.json"
)

RANKING_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_ranking_contract.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step4a_status.json"
)

STEP4A_REPORT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_report.json"
)

RUNTIME_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_22_runtime_contract_checkpoint.json"
)

SMOKE_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_smoke_test"
)

SMOKE_CONDITION_INVENTORY_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_condition_inventory.csv"
)

SMOKE_COMBINED_CONDITION_AUDIT_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_combined_condition_audit.csv"
)

SMOKE_COMBINED_PROJECT_RUNS_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_combined_project_runs.csv"
)

SMOKE_COMBINED_BUILD_METRICS_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_combined_build_metrics.csv"
)

SMOKE_COMBINED_MODEL_FITS_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_combined_model_fits.csv"
)

SMOKE_BASELINE_INVARIANCE_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_smoke_baseline_invariance.csv"
)

SMOKE_VALIDATION_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_step4b_validation.csv"
)

SMOKE_REPORT_PATH = (
    SMOKE_ROOT
    / f"{PROJECT_SHORT}_step4b_report.json"
)

STEP4B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step4b_status.json"
)

SMOKE_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_22_smoke_test_checkpoint.json"
)

# The smoke test must never write to this future full-run root.
FULL_RAW_RESULT_ROOT = (
    RESULTS_ROOT
    / "Raw"
    / PROJECT_SLUG
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def sha256_array(values, dtype):
    array = np.asarray(values).astype(dtype, copy=False)
    return hashlib.sha256(array.tobytes(order="C")).hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    with temporary_path.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(temporary_path, path)


def atomic_csv(path, frame, compression=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
        compression=compression,
    )

    os.replace(temporary_path, path)


def atomic_parquet(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_parquet(
        temporary_path,
        index=False,
    )

    os.replace(temporary_path, path)


def source_root_hash(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )
        digest.update(line.encode("utf-8"))

    return digest.hexdigest()


def parse_int(values, label):
    numeric = pd.to_numeric(values, errors="coerce")

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains {int(numeric.isna().sum())} missing/non-numeric values."
        )

    array = numeric.to_numpy(dtype=float)

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")


def add_check(rows, check, expected, actual, passed):
    rows.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })


def deterministic_seed(repetition_seed, stream_name):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(material).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )


def deterministic_random_build_seed(repetition_seed, build_id):
    return deterministic_seed(
        repetition_seed,
        f"Random_baseline_build_{int(build_id)}",
    )


def create_models(repetition_seed):
    return {
        "RandomForest": RandomForestClassifier(
            **MODEL_CONFIG["RandomForest"],
            random_state=deterministic_seed(
                repetition_seed,
                "RandomForest_model",
            ),
        ),
        "XGBoost": XGBClassifier(
            **MODEL_CONFIG["XGBoost"],
            random_state=deterministic_seed(
                repetition_seed,
                "XGBoost_model",
            ),
        ),
        "LightGBM": LGBMClassifier(
            **MODEL_CONFIG["LightGBM"],
            random_state=deterministic_seed(
                repetition_seed,
                "LightGBM_model",
            ),
        ),
        "NaiveBayes": GaussianNB(
            **MODEL_CONFIG["NaiveBayes"]
        ),
    }


def calculate_apfd(failures):
    failures = np.asarray(failures, dtype=np.int8)
    number_of_tests = len(failures)
    number_of_failures = int(failures.sum())

    if number_of_tests == 0 or number_of_failures == 0:
        return np.nan

    failure_positions = np.flatnonzero(failures == 1) + 1

    return float(
        1.0
        - (
            failure_positions.sum()
            / (number_of_tests * number_of_failures)
        )
        + (1.0 / (2.0 * number_of_tests))
    )


def calculate_apfdc(failures, durations):
    failures = np.asarray(failures, dtype=np.int8)
    durations = np.asarray(durations, dtype=float)

    if len(failures) != len(durations):
        raise ValueError(
            "failures and durations must have equal length."
        )

    if len(failures) == 0 or failures.sum() == 0:
        return np.nan

    if not np.isfinite(durations).all():
        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (durations < 0).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(durations.sum())

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array([0.0]),
        np.cumsum(durations)[:-1],
    ])

    failure_mask = failures == 1
    midpoint_detection_times = (
        cumulative_before[failure_mask]
        + (0.5 * durations[failure_mask])
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


def calculate_rates(history):
    history_length = len(history)

    if history_length == 0:
        raise ValueError(
            "Rate calculation requires non-empty history."
        )

    verdicts = history["verdict"]

    return (
        float(verdicts.ne(0).sum() / history_length),
        float(verdicts.eq(2).sum() / history_length),
        float(verdicts.eq(1).sum() / history_length),
        float(history["transition"].eq(1).sum() / history_length),
    )


def calculate_max_test_file_rate(
    history,
    target_column,
    current_changed_entities,
    entity_changed_builds,
):
    target_builds = (
        history.loc[
            history[target_column].gt(0),
            "build",
        ]
        .drop_duplicates()
        .astype(int)
        .tolist()
    )

    if len(target_builds) == 0:
        return -1.0

    target_build_set = set(target_builds)
    maximum_frequency = 0

    for entity_id in current_changed_entities:
        changed_builds = entity_changed_builds.get(
            int(entity_id),
            set(),
        )

        overlap_count = len(
            changed_builds.intersection(target_build_set)
        )

        maximum_frequency = max(
            maximum_frequency,
            overlap_count,
        )

    if maximum_frequency == 0:
        return 0.0

    return float(
        maximum_frequency
        / len(target_builds)
    )


def reconstruct_rec_features(
    execution_history,
    requested_rows,
    global_build_position,
    changed_entities_by_build,
    entity_changed_builds,
    recent_window=6,
):
    requested_pairs = set(
        zip(
            requested_rows["Build"].astype(int),
            requested_rows["Test"].astype(int),
        )
    )

    reconstructed_records = []
    test_groups = execution_history.groupby(
        "test",
        sort=False,
    )
    total_tests = int(
        execution_history["test"].nunique()
    )

    for test_index, (test_id, test_history) in enumerate(
        test_groups,
        start=1,
    ):
        test_history = (
            test_history.sort_values(
                "inferred_test_order",
                kind="mergesort",
            )
            .reset_index(drop=True)
            .copy()
        )

        test_history["transition"] = (
            test_history["verdict"]
            .diff()
            .fillna(0)
            .ne(0)
            .astype(int)
        )

        first_test_build = int(
            test_history.iloc[0]["build"]
        )

        for current_position in range(len(test_history)):
            current_row = test_history.iloc[current_position]
            current_build = int(current_row["build"])
            current_test = int(test_id)
            pair = (current_build, current_test)

            if pair not in requested_pairs:
                continue

            history = (
                test_history.iloc[:current_position]
                .copy()
                .reset_index(drop=True)
            )

            record = {
                "Build": current_build,
                "Test": current_test,
            }

            if history.empty:
                for feature in REC_FEATURES:
                    record[feature] = -1.0

                record["REC_Age"] = 0.0
                reconstructed_records.append(record)
                continue

            recent_history = history.tail(recent_window).copy()

            age = float(
                global_build_position[current_build]
                - global_build_position[first_test_build]
            )

            failure_positions = np.flatnonzero(
                history["verdict"].to_numpy() > 0
            )

            last_failure_age = (
                -1.0
                if len(failure_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(failure_positions[-1])
                )
            )

            transition_positions = np.flatnonzero(
                history["transition"].to_numpy() > 0
            )

            last_transition_age = (
                -1.0
                if len(transition_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(transition_positions[-1])
                )
            )

            (
                recent_fail_rate,
                recent_assert_rate,
                recent_exc_rate,
                recent_transition_rate,
            ) = calculate_rates(recent_history)

            (
                total_fail_rate,
                total_assert_rate,
                total_exc_rate,
                total_transition_rate,
            ) = calculate_rates(history)

            current_changed_entities = (
                changed_entities_by_build.get(
                    current_build,
                    set(),
                )
            )

            max_file_fail_rate = calculate_max_test_file_rate(
                history=history,
                target_column="verdict",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            max_file_transition_rate = calculate_max_test_file_rate(
                history=history,
                target_column="transition",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            record.update({
                "REC_Age": age,
                "REC_LastFailureAge": last_failure_age,
                "REC_LastTransitionAge": last_transition_age,
                "REC_RecentAvgExeTime": float(
                    recent_history["duration"].mean()
                ),
                "REC_RecentMaxExeTime": float(
                    recent_history["duration"].max()
                ),
                "REC_RecentFailRate": recent_fail_rate,
                "REC_RecentAssertRate": recent_assert_rate,
                "REC_RecentExcRate": recent_exc_rate,
                "REC_RecentTransitionRate": recent_transition_rate,
                "REC_TotalAvgExeTime": float(
                    history["duration"].mean()
                ),
                "REC_TotalMaxExeTime": float(
                    history["duration"].max()
                ),
                "REC_TotalFailRate": total_fail_rate,
                "REC_TotalAssertRate": total_assert_rate,
                "REC_TotalExcRate": total_exc_rate,
                "REC_TotalTransitionRate": total_transition_rate,
                "REC_LastVerdict": float(
                    recent_history.iloc[-1]["verdict"]
                ),
                "REC_LastExeTime": float(
                    recent_history.iloc[-1]["duration"]
                ),
                "REC_MaxTestFileFailRate": max_file_fail_rate,
                "REC_MaxTestFileTransitionRate": (
                    max_file_transition_rate
                ),
            })

            reconstructed_records.append(record)

        if test_index % 100 == 0 or test_index == total_tests:
            print(
                "    REC reconstruction progress:",
                test_index,
                "/",
                total_tests,
                "tests | reconstructed rows:",
                len(reconstructed_records),
            )

    return pd.DataFrame(reconstructed_records)


def positive_probability(estimator, matrix):
    probabilities = estimator.predict_proba(matrix)
    classes = np.asarray(estimator.classes_)
    positive_columns = np.flatnonzero(classes == 1)

    if len(positive_columns) != 1:
        raise RuntimeError(
            "Fitted estimator does not expose exactly one class-1 probability column."
        )

    scores = probabilities[:, int(positive_columns[0])]

    if not np.isfinite(scores).all():
        raise RuntimeError(
            "Model produced non-finite failure probabilities."
        )

    if ((scores < 0) | (scores > 1)).any():
        raise RuntimeError(
            "Model produced probabilities outside [0,1]."
        )

    return scores.astype(float, copy=False)


def make_ranking(
    evaluation_meta,
    technique,
    scores,
    ascending_score,
):
    ranking = evaluation_meta.copy()
    ranking["Technique"] = technique
    ranking["Score"] = np.asarray(scores, dtype=float)

    if len(ranking) != EXPECTED_MODEL_EVAL_ROWS:
        raise RuntimeError(
            f"{technique} ranking input has the wrong row count."
        )

    if not np.isfinite(ranking["Score"].to_numpy(dtype=float)).all():
        raise RuntimeError(
            f"{technique} ranking contains non-finite scores."
        )

    ranking = (
        ranking.sort_values(
            [
                "Build",
                "Score",
                "Test",
            ],
            ascending=[
                True,
                bool(ascending_score),
                True,
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    ranking["Rank"] = (
        ranking.groupby(
            "Build",
            sort=False,
        )
        .cumcount()
        .add(1)
        .astype("int64")
    )

    return ranking[
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "ConditionKey",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "Build",
            "Test",
            "Rank",
            "Score",
            "CleanVerdict",
            "CleanFailure",
            "Duration",
        ]
    ]


def calculate_condition_metrics(rankings):
    build_metric_records = []

    failing_rankings = rankings.loc[
        rankings["Build"].isin(failing_evaluation_builds)
    ].copy()

    for (technique, build_id), build_ranking in failing_rankings.groupby(
        [
            "Technique",
            "Build",
        ],
        sort=False,
    ):
        build_ranking = build_ranking.sort_values(
            "Rank",
            kind="mergesort",
        )

        failures = build_ranking[
            "CleanFailure"
        ].to_numpy(dtype=np.int8)

        durations = build_ranking[
            "Duration"
        ].to_numpy(dtype=float)

        number_of_failures = int(failures.sum())

        if number_of_failures <= 0:
            raise RuntimeError(
                "A supposedly failing evaluation build has no failures."
            )

        apfd = calculate_apfd(failures)
        apfdc = calculate_apfdc(failures, durations)

        build_metric_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                build_ranking["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                build_ranking["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                build_ranking["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "Build": int(build_id),
            "Tests": int(len(build_ranking)),
            "Failures": number_of_failures,
            "TotalDuration": float(durations.sum()),
            "APFDc": float(apfdc),
            "APFD": float(apfd),
        })

    build_metrics = pd.DataFrame(build_metric_records)

    project_run_records = []

    for technique, technique_metrics in build_metrics.groupby(
        "Technique",
        sort=False,
    ):
        project_run_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                technique_metrics["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                technique_metrics["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                technique_metrics["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "EvaluationBuilds": EXPECTED_EVALUATION_BUILDS,
            "ScoredFailingBuilds": int(len(technique_metrics)),
            "EvaluationRows": EXPECTED_MODEL_EVAL_ROWS,
            "EvaluationFailures": EXPECTED_MODEL_EVAL_FAILURES,
            "MeanAPFDc": float(
                technique_metrics["APFDc"].mean()
            ),
            "MedianAPFDc": float(
                technique_metrics["APFDc"].median()
            ),
            "MeanAPFD": float(
                technique_metrics["APFD"].mean()
            ),
            "MedianAPFD": float(
                technique_metrics["APFD"].median()
            ),
        })

    project_runs = pd.DataFrame(project_run_records)

    return build_metrics, project_runs


def audit_checkpoint_manifest(
    payload,
    manifest_key,
    label,
):
    manifest = payload.get(
        manifest_key,
        [],
    )

    if not isinstance(
        manifest,
        list,
    ) or not manifest:
        raise RuntimeError(
            f"{label} contains no {manifest_key}."
        )

    records = []

    for item in manifest:
        path = Path(
            item[
                "Path"
            ]
        )

        expected_bytes = int(
            item[
                "Bytes"
            ]
        )

        expected_sha256 = str(
            item[
                "SHA256"
            ]
        ).lower()

        exists = path.is_file()

        actual_bytes = (
            int(
                path.stat().st_size
            )
            if exists
            else -1
        )

        actual_sha256 = (
            sha256_file(
                path
            )
            if exists
            else "MISSING"
        )

        records.append({
            "Checkpoint":
                label,

            "Path":
                str(
                    path
                ),

            "ExpectedBytes":
                expected_bytes,

            "ActualBytes":
                actual_bytes,

            "ExpectedSHA256":
                expected_sha256,

            "ActualSHA256":
                actual_sha256,

            "Pass":
                bool(
                    exists
                    and actual_bytes
                    == expected_bytes
                    and actual_sha256
                    == expected_sha256
                ),
        })

    audit = pd.DataFrame(
        records
    )

    failures = int(
        (
            ~audit[
                "Pass"
            ]
        ).sum()
    )

    return (
        audit,
        failures,
    )


def directory_manifest(root):
    root = Path(root)
    rows = []

    if not root.exists():
        return pd.DataFrame(
            columns=[
                "RelativePath",
                "Bytes",
                "SHA256",
            ]
        )

    for path in sorted(
        [
            candidate
            for candidate in root.rglob("*")
            if candidate.is_file()
        ],
        key=lambda candidate: candidate.relative_to(root).as_posix(),
    ):
        rows.append({
            "RelativePath": path.relative_to(root).as_posix(),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        })

    return pd.DataFrame(rows)


def directory_root_hash(manifest):
    digest = hashlib.sha256()

    for row in manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode("utf-8")
        )

    return digest.hexdigest()


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN CHECKPOINTS
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SOURCE_DIR / "builds.csv",
    SOURCE_DIR / "contributors.csv",
    SOURCE_DIR / "dataset.csv",
    SOURCE_DIR / "entity_change_history.csv",
    SOURCE_DIR / "exe.csv",
    SOURCE_DIR / "id_map.csv",
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    SELECTION_CHECKPOINT_PATH,
    BUILD_ENTITY_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    INFERRED_EXECUTION_ORDER_PATH,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    REC_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    PREDICTOR_CONTRACT_PATH,
    MODEL_CONTRACT_PATH,
    BASELINE_CONTRACT_PATH,
    RANKING_CONTRACT_PATH,
    STEP4A_STATUS_PATH,
    STEP4A_REPORT_PATH,
    RUNTIME_CHECKPOINT_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 22 Step 4B inputs are missing:\n"
        + "\n".join(missing_paths)
    )

selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)
rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)
noise_plan_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)
runtime_checkpoint_sha256 = sha256_file(
    RUNTIME_CHECKPOINT_PATH
)

if selection_checkpoint_sha256 != EXPECTED_SELECTION_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 22 selection checkpoint SHA-256 differs."
    )

if rec_checkpoint_sha256 != EXPECTED_REC_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 22 REC checkpoint SHA-256 differs."
    )

if noise_plan_checkpoint_sha256 != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 22 noise-plan checkpoint SHA-256 differs."
    )

if runtime_checkpoint_sha256 != EXPECTED_RUNTIME_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 22 runtime-contract checkpoint SHA-256 differs."
    )

selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)
rec_checkpoint = load_json(
    REC_CHECKPOINT_PATH
)
noise_plan_checkpoint = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)
runtime_checkpoint = load_json(
    RUNTIME_CHECKPOINT_PATH
)
step4a_status = load_json(
    STEP4A_STATUS_PATH
)
step4a_report = load_json(
    STEP4A_REPORT_PATH
)

if selection_checkpoint.get(
    "Status"
) != "PASS_PROJECT_22_SELECTION_AND_SOURCE_FROZEN":
    raise RuntimeError(
        "Selection checkpoint does not contain the frozen Step 1B PASS status."
    )

if rec_checkpoint.get(
    "Status"
) != EXPECTED_STEP2B_STATUS:
    raise RuntimeError(
        "REC checkpoint does not contain the frozen Step 2B PASS status."
    )

if rec_checkpoint.get(
    "ImplementationVersion"
) != EXPECTED_REC_IMPLEMENTATION_VERSION:
    raise RuntimeError(
        "REC checkpoint implementation version differs from the frozen Project 22 V2 contract."
    )

if rec_checkpoint.get(
    "MappingIncompleteBuilds"
) != []:
    raise RuntimeError(
        "Project 22 Step 2B no longer reports full build/entity mapping coverage."
    )

if int(rec_checkpoint.get(
    "VerdictDependentDirectMismatches",
    -1,
)) != 0:
    raise RuntimeError(
        "Project 22 Step 2B verdict-dependent direct mismatches are no longer zero."
    )

if int(rec_checkpoint.get(
    "FileHistoryDirectMismatches",
    -1,
)) != 0:
    raise RuntimeError(
        "Project 22 Step 2B file-history direct mismatches are no longer zero."
    )

if int(rec_checkpoint.get(
    "AnchorDisallowedNonFileDirectMismatches",
    -1,
)) != 0:
    raise RuntimeError(
        "Project 22 Step 2B has anchor-disallowed non-file residuals."
    )

if rec_checkpoint.get(
    "DirectResidualsConfinedToAnchorAllowedFeatures"
) is not True:
    raise RuntimeError(
        "Project 22 Step 2B direct residuals are not confined to the approved timing aggregates."
    )

if int(rec_checkpoint.get(
    "AnchoredMismatchValues",
    -1,
)) != 0:
    raise RuntimeError(
        "Project 22 Step 2B anchored REC mismatches are no longer zero."
    )

if rec_checkpoint.get(
    "ZeroPercentCleanDatasetReproducedExactly"
) is not True:
    raise RuntimeError(
        "Project 22 Step 2B no longer reproduces the clean 0% dataset exactly."
    )

if noise_plan_checkpoint.get(
    "Status"
) != EXPECTED_STEP3A_STATUS:
    raise RuntimeError(
        "Noise-plan checkpoint does not contain the frozen Step 3A PASS status."
    )

for label, payload in [
    ("runtime checkpoint", runtime_checkpoint),
    ("Step 4A status", step4a_status),
    ("Step 4A report", step4a_report),
]:
    if payload.get("Status") != EXPECTED_STEP4A_STATUS:
        raise RuntimeError(
            f"{label} does not contain the frozen Step 4A PASS status."
        )

if runtime_checkpoint.get("Project") != PROJECT_NAME:
    raise RuntimeError(
        "Runtime checkpoint project identity differs."
    )

if runtime_checkpoint.get("ProjectSlug") != PROJECT_SLUG:
    raise RuntimeError(
        "Runtime checkpoint project slug differs."
    )

if runtime_checkpoint.get("SourceRootSHA256") != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Runtime checkpoint source root differs."
    )

if runtime_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Runtime checkpoint active-reservation state differs."
    )

if runtime_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Runtime checkpoint runtime-priority rule differs."
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY SOURCE ROOT, REGISTRY, AND STEP 4A OUTPUT MANIFEST
# --------------------------------------------------------------------------------------------------

registry_sha256_before = sha256_file(REGISTRY_PATH)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs before Step 4B."
    )

registry = pd.read_csv(
    REGISTRY_PATH,
    low_memory=False,
)

project_number_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "projectnumber",
            "project_number",
            "project no",
            "projectno",
        }
    ),
    None,
)

project_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "project",
            "projectname",
            "project_name",
        }
    ),
    None,
)

status_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "status",
            "projectstatus",
            "project_status",
        }
    ),
    None,
)

if (
    project_number_column is None
    or project_column is None
    or status_column is None
):
    raise RuntimeError(
        "Could not resolve ProjectNumber, Project, and Status "
        "columns in the completion registry."
    )

registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)

if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Completion registry does not contain exactly Projects 1–21."
    )

if not registry[
    status_column
].astype(
    str
).eq(
    "COMPLETE_AND_FROZEN"
).all():
    raise RuntimeError(
        "Projects 1–21 are not all COMPLETE_AND_FROZEN."
    )

required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
}

for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or str(
            matching_rows.iloc[
                0
            ][
                project_column
            ]
        )
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )

if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].astype(
        str
    ).eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 22 is already present in the completion registry."
    )

active_reservations = []

if active_reservations != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "The active-reservation state differs from the Project 22 freeze."
    )

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)

current_source_rows = []

for row in frozen_source_manifest.itertuples(index=False):
    source_path = (
        SOURCE_DIR
        / str(row.RelativePath)
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            f"Frozen Project 22 source file is missing: {source_path}"
        )

    current_source_rows.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })

current_source_manifest = pd.DataFrame(current_source_rows)
current_source_root_sha256 = source_root_hash(
    current_source_manifest
)

if current_source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 22 source root differs before Step 4B."
    )

rec_manifest_audit, rec_manifest_failures = (
    audit_checkpoint_manifest(
        rec_checkpoint,
        "OutputManifest",
        "REC checkpoint",
    )
)

noise_manifest_audit, noise_manifest_failures = (
    audit_checkpoint_manifest(
        noise_plan_checkpoint,
        "OutputManifest",
        "Noise-plan checkpoint",
    )
)

if rec_manifest_failures != 0:
    print(
        "\nFailed REC output-manifest checks:"
    )

    display(
        rec_manifest_audit.loc[
            ~rec_manifest_audit[
                "Pass"
            ]
        ]
    )

    raise RuntimeError(
        "One or more frozen REC outputs changed."
    )

if noise_manifest_failures != 0:
    print(
        "\nFailed noise-plan output-manifest checks:"
    )

    display(
        noise_manifest_audit.loc[
            ~noise_manifest_audit[
                "Pass"
            ]
        ]
    )

    raise RuntimeError(
        "One or more frozen noise-plan outputs changed."
    )

runtime_output_manifest = runtime_checkpoint.get(
    "RuntimeOutputManifest",
    [],
)

if not isinstance(runtime_output_manifest, list) or not runtime_output_manifest:
    raise RuntimeError(
        "Runtime checkpoint has no output manifest."
    )

runtime_manifest_records = []

for item in runtime_output_manifest:
    path = Path(item["Path"])
    expected_bytes = int(item["Bytes"])
    expected_sha256 = str(item["SHA256"]).lower()
    exists = path.is_file()
    actual_bytes = int(path.stat().st_size) if exists else -1
    actual_sha256 = sha256_file(path) if exists else "MISSING"
    passed = (
        exists
        and actual_bytes == expected_bytes
        and actual_sha256 == expected_sha256
    )

    runtime_manifest_records.append({
        "Path": str(path),
        "ExpectedBytes": expected_bytes,
        "ActualBytes": actual_bytes,
        "ExpectedSHA256": expected_sha256,
        "ActualSHA256": actual_sha256,
        "Pass": passed,
    })

runtime_manifest_audit = pd.DataFrame(
    runtime_manifest_records
)
runtime_manifest_failures = int(
    (~runtime_manifest_audit["Pass"]).sum()
)

if runtime_manifest_failures != 0:
    print("\nFailed Step 4A output-manifest checks:")
    display(
        runtime_manifest_audit.loc[
            ~runtime_manifest_audit["Pass"]
        ]
    )
    raise RuntimeError(
        "Step 4A output manifest no longer validates."
    )

full_raw_result_root_existed_before = FULL_RAW_RESULT_ROOT.exists()
full_raw_result_manifest_before = directory_manifest(
    FULL_RAW_RESULT_ROOT
)
full_raw_result_root_hash_before = directory_root_hash(
    full_raw_result_manifest_before
)


# --------------------------------------------------------------------------------------------------
# 6. LOAD FROZEN COHORTS, LINKS, CONDITION PLAN, AND RNG STREAM
# --------------------------------------------------------------------------------------------------

print("\nLoading frozen Project 22 cohorts and contracts.")

raw_training = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)
raw_evaluation = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)
model_training = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)
model_evaluation = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)
model_train_link = pd.read_parquet(
    MODEL_RAW_TRAIN_LINK_PATH
)
model_eval_link = pd.read_parquet(
    MODEL_RAW_EVAL_LINK_PATH
)
condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)
predictor_contract = pd.read_csv(
    PREDICTOR_CONTRACT_PATH,
    low_memory=False,
)
inferred_execution_order = pd.read_parquet(
    INFERRED_EXECUTION_ORDER_PATH
)
frozen_global_build_order = pd.read_csv(
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    low_memory=False,
)
build_entity = pd.read_csv(
    BUILD_ENTITY_PATH,
    compression="gzip",
    low_memory=False,
)
anchor_offsets = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)
clean_reconstructed = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)

if len(raw_training) != EXPECTED_RAW_TRAIN_ROWS:
    raise RuntimeError("Raw training cohort row count differs.")
if len(raw_evaluation) != EXPECTED_RAW_EVAL_ROWS:
    raise RuntimeError("Raw evaluation cohort row count differs.")
if len(model_training) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model training cohort row count differs.")
if len(model_evaluation) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model evaluation cohort row count differs.")
if len(model_train_link) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model/raw training link row count differs.")
if len(model_eval_link) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model/raw evaluation link row count differs.")
if len(anchor_offsets) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean anchor-offset row count differs.")
if len(clean_reconstructed) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean reconstructed REC row count differs.")
if len(inferred_execution_order) != EXPECTED_RAW_ROWS:
    raise RuntimeError("Frozen inferred execution-order row count differs.")
if len(frozen_global_build_order) != EXPECTED_BUILDS:
    raise RuntimeError("Frozen global build-order row count differs.")

required_cohort_columns = {
    "Build",
    "Test",
    "Verdict",
}

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    missing = required_cohort_columns - set(frame.columns)
    if missing:
        raise RuntimeError(
            f"{label} cohort is missing columns: {sorted(missing)}"
        )

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    frame["Build"] = parse_int(
        frame["Build"],
        f"{label}.Build",
    )
    frame["Test"] = parse_int(
        frame["Test"],
        f"{label}.Test",
    )
    frame["Verdict"] = parse_int(
        frame["Verdict"],
        f"{label}.Verdict",
    )

raw_order_column = "RawTrainingRowOrder"
raw_eval_order_column = "RawEvaluationRowOrder"
model_train_order_column = "ModelTrainingRowOrder"
model_eval_order_column = "ModelEvaluationRowOrder"

for column, frame, expected_rows, label in [
    (
        raw_order_column,
        raw_training,
        EXPECTED_RAW_TRAIN_ROWS,
        "raw training",
    ),
    (
        raw_eval_order_column,
        raw_evaluation,
        EXPECTED_RAW_EVAL_ROWS,
        "raw evaluation",
    ),
    (
        model_train_order_column,
        model_training,
        EXPECTED_MODEL_TRAIN_ROWS,
        "model training",
    ),
    (
        model_eval_order_column,
        model_evaluation,
        EXPECTED_MODEL_EVAL_ROWS,
        "model evaluation",
    ),
]:
    if column not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing {column}."
        )

    frame[column] = parse_int(
        frame[column],
        f"{label}.{column}",
    )

    frame.sort_values(
        column,
        kind="mergesort",
        inplace=True,
    )
    frame.reset_index(drop=True, inplace=True)

    expected_sequence = np.arange(
        1,
        expected_rows + 1,
        dtype=np.int64,
    )

    if not np.array_equal(
        frame[column].to_numpy(dtype=np.int64),
        expected_sequence,
    ):
        raise RuntimeError(
            f"{label} row-order sequence is not canonical."
        )

model_train_link[model_train_order_column] = parse_int(
    model_train_link[model_train_order_column],
    "model_train_link.ModelTrainingRowOrder",
)
model_train_link[raw_order_column] = parse_int(
    model_train_link[raw_order_column],
    "model_train_link.RawTrainingRowOrder",
)
model_eval_link[model_eval_order_column] = parse_int(
    model_eval_link[model_eval_order_column],
    "model_eval_link.ModelEvaluationRowOrder",
)
model_eval_link[raw_eval_order_column] = parse_int(
    model_eval_link[raw_eval_order_column],
    "model_eval_link.RawEvaluationRowOrder",
)

model_train_link = (
    model_train_link.sort_values(
        model_train_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)
model_eval_link = (
    model_eval_link.sort_values(
        model_eval_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

model_training_raw_indices = (
    model_train_link[raw_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)
model_evaluation_raw_indices = (
    model_eval_link[raw_eval_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)

if (
    model_training_raw_indices.min() < 0
    or model_training_raw_indices.max() >= EXPECTED_RAW_TRAIN_ROWS
):
    raise RuntimeError(
        "Model/raw training indices are outside the frozen raw cohort."
    )

if (
    model_evaluation_raw_indices.min() < 0
    or model_evaluation_raw_indices.max() >= EXPECTED_RAW_EVAL_ROWS
):
    raise RuntimeError(
        "Model/raw evaluation indices are outside the frozen raw cohort."
    )

linked_train_build = raw_training.iloc[
    model_training_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_train_test = raw_training.iloc[
    model_training_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_train_verdict = raw_training.iloc[
    model_training_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

linked_eval_build = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_eval_test = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_eval_verdict = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

if not np.array_equal(
    linked_train_build,
    model_training["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Build links differ.")
if not np.array_equal(
    linked_train_test,
    model_training["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Test links differ.")
if not np.array_equal(
    linked_train_verdict,
    model_training["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training verdict links differ.")
if not np.array_equal(
    linked_eval_build,
    model_evaluation["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Build links differ.")
if not np.array_equal(
    linked_eval_test,
    model_evaluation["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Test links differ.")
if not np.array_equal(
    linked_eval_verdict,
    model_evaluation["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation verdict links differ.")

smoke_plan = (
    condition_plan.loc[
        condition_plan["ConditionID"].isin(
            SMOKE_CONDITION_IDS
        )
    ]
    .copy()
    .sort_values(
        "NoisePercent",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(smoke_plan) != EXPECTED_SMOKE_CONDITIONS:
    raise RuntimeError(
        "The frozen condition plan does not contain exactly the two smoke conditions."
    )

if smoke_plan["ConditionID"].tolist() != SMOKE_CONDITION_IDS:
    raise RuntimeError(
        "Smoke-condition order differs from the frozen contract."
    )

if smoke_plan["NoisePercent"].astype(int).tolist() != SMOKE_NOISE_LEVELS:
    raise RuntimeError(
        "Smoke noise levels differ from the frozen contract."
    )

if not smoke_plan["RepetitionSeed"].astype(int).eq(
    SMOKE_REPETITION_SEED
).all():
    raise RuntimeError(
        "Smoke repetition seed differs from the frozen contract."
    )

rng_metadata_rows = int(
    pq.ParquetFile(RNG_MANIFEST_PATH).metadata.num_rows
)

if rng_metadata_rows != EXPECTED_RNG_MANIFEST_ROWS:
    raise RuntimeError(
        "Frozen RNG manifest row count differs."
    )

rng_seed = pd.read_parquet(
    RNG_MANIFEST_PATH,
    filters=[
        (
            "RepetitionSeed",
            "==",
            SMOKE_REPETITION_SEED,
        ),
    ],
)

rng_seed[raw_order_column] = parse_int(
    rng_seed[raw_order_column],
    "rng_seed.RawTrainingRowOrder",
)

rng_seed = (
    rng_seed.sort_values(
        raw_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(rng_seed) != EXPECTED_RAW_TRAIN_ROWS:
    raise RuntimeError(
        "Seed-1 RNG stream has the wrong row count."
    )

if not np.array_equal(
    rng_seed[raw_order_column].to_numpy(dtype=np.int64),
    np.arange(
        1,
        EXPECTED_RAW_TRAIN_ROWS + 1,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Seed-1 RNG stream row order differs."
    )

flip_uniform = rng_seed[
    "FlipUniform"
].to_numpy(dtype=np.float64)
sampled_failure_subtype = rng_seed[
    "SampledFailureSubtype"
].to_numpy(dtype=np.int16)

if not np.isfinite(flip_uniform).all():
    raise RuntimeError(
        "Seed-1 flip-uniform stream contains non-finite values."
    )

if ((flip_uniform < 0) | (flip_uniform >= 1)).any():
    raise RuntimeError(
        "Seed-1 flip-uniform values are outside [0,1)."
    )

expected_failure_subtypes = sorted(
    int(value)
    for value in noise_plan_checkpoint.get(
        "FailureSubtypes",
        [],
    )
)

if (
    not expected_failure_subtypes
    or not set(expected_failure_subtypes).issubset({1, 2})
):
    raise RuntimeError(
        "Frozen failure-subtype contract is empty or contains unknown codes."
    )

if sorted(np.unique(sampled_failure_subtype).tolist()) != expected_failure_subtypes:
    raise RuntimeError(
        "Seed-1 failure-subtype stream differs from the frozen noise-plan contract."
    )


# --------------------------------------------------------------------------------------------------
# 7. PREDICTOR ORDER, NUMERIC MATRICES, CHRONOLOGY, ENTITY MAP, AND EVALUATION META
# --------------------------------------------------------------------------------------------------

if "Predictor" not in predictor_contract.columns:
    raise RuntimeError(
        "Predictor contract is missing the Predictor column."
    )

predictor_columns = predictor_contract[
    "Predictor"
].astype(str).tolist()

if len(predictor_columns) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract does not contain the frozen predictor count."
    )

if len(set(predictor_columns)) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract contains duplicate predictors."
    )

missing_training_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_training.columns
]
missing_evaluation_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_evaluation.columns
]

if missing_training_predictors or missing_evaluation_predictors:
    raise RuntimeError(
        "Frozen model cohorts are missing contract predictors."
    )

if any(feature not in predictor_columns for feature in REC_FEATURES):
    raise RuntimeError(
        "The 19 REC features are not all present in the predictor contract."
    )

if set(VERDICT_DEPENDENT_REC).intersection(
    VERDICT_INDEPENDENT_REC
):
    raise RuntimeError(
        "Dependent and independent REC sets overlap."
    )

if set(VERDICT_DEPENDENT_REC + VERDICT_INDEPENDENT_REC) != set(
    REC_FEATURES
):
    raise RuntimeError(
        "Dependent and independent REC sets do not partition all 19 REC features."
    )

print("Converting the fixed predictor cohorts to one numeric matrix.")

training_numeric_frame = model_training[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

evaluation_numeric_frame = model_evaluation[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

training_base_numeric = training_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)
evaluation_base_numeric = evaluation_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)

training_base_numeric[
    ~np.isfinite(training_base_numeric)
] = np.nan
evaluation_base_numeric[
    ~np.isfinite(evaluation_base_numeric)
] = np.nan

all_base_numeric = np.vstack([
    training_base_numeric,
    evaluation_base_numeric,
])

predictor_index = {
    feature: index
    for index, feature in enumerate(predictor_columns)
}

dependent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_DEPENDENT_REC
    ],
    dtype=np.int64,
)

independent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_INDEPENDENT_REC
    ],
    dtype=np.int64,
)

model_all = pd.concat(
    [
        model_training[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        model_evaluation[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
    ],
    ignore_index=True,
)

if model_all.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Model cohort contains duplicate Build-Test rows."
    )

model_key_index = pd.MultiIndex.from_frame(
    model_all[["Build", "Test"]]
)

anchor_offsets = anchor_offsets.copy()
anchor_offsets["Build"] = parse_int(
    anchor_offsets["Build"],
    "anchor_offsets.Build",
)
anchor_offsets["Test"] = parse_int(
    anchor_offsets["Test"],
    "anchor_offsets.Test",
)

if anchor_offsets.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Anchor offsets contain duplicate Build-Test rows."
    )

anchor_indexed = anchor_offsets.set_index(
    [
        "Build",
        "Test",
    ]
)

missing_anchor_keys = model_key_index.difference(
    anchor_indexed.index
)

if len(missing_anchor_keys) != 0:
    raise RuntimeError(
        "Anchor offsets do not cover the full model cohort."
    )

anchor_values_all = anchor_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_reconstructed["Build"] = parse_int(
    clean_reconstructed["Build"],
    "clean_reconstructed.Build",
)
clean_reconstructed["Test"] = parse_int(
    clean_reconstructed["Test"],
    "clean_reconstructed.Test",
)

clean_reconstructed_indexed = clean_reconstructed.set_index(
    [
        "Build",
        "Test",
    ]
)

clean_reconstructed_all = clean_reconstructed_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_anchored_all = (
    clean_reconstructed_all
    + anchor_values_all
)

clean_original_rec_all = model_all[
    REC_FEATURES
].to_numpy(dtype=np.float64)

clean_anchor_mismatch_values = int(
    (~np.isclose(
        clean_anchored_all,
        clean_original_rec_all,
        rtol=0,
        atol=1e-12,
    )).sum()
)

if clean_anchor_mismatch_values != 0:
    raise RuntimeError(
        "Frozen clean REC reconstruction plus anchor no longer reproduces the model cohort."
    )

required_inferred_order_columns = {
    "Build",
    "Test",
    "Job",
    "Verdict",
    "Duration",
    "InferredTestOrder",
}

missing_inferred_order_columns = (
    required_inferred_order_columns
    - set(inferred_execution_order.columns)
)

if missing_inferred_order_columns:
    raise RuntimeError(
        "Frozen inferred execution order is missing columns: "
        f"{sorted(missing_inferred_order_columns)}"
    )

for column in [
    "Build",
    "Test",
    "Verdict",
    "InferredTestOrder",
]:
    inferred_execution_order[column] = parse_int(
        inferred_execution_order[column],
        f"inferred_execution_order.{column}",
    )

inferred_execution_order["Job"] = pd.to_numeric(
    inferred_execution_order["Job"],
    errors="coerce",
)

inferred_execution_order["Duration"] = pd.to_numeric(
    inferred_execution_order["Duration"],
    errors="coerce",
)

if not np.isfinite(
    inferred_execution_order["Job"].to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "Frozen inferred execution order contains non-finite jobs."
    )

if not np.isfinite(
    inferred_execution_order["Duration"].to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "Frozen inferred execution order contains non-finite durations."
    )

if inferred_execution_order["Duration"].lt(0).any():
    raise RuntimeError(
        "Frozen inferred execution order contains negative durations."
    )

if inferred_execution_order.duplicated(
    subset=[
        "Build",
        "Test",
    ],
    keep=False,
).any():
    raise RuntimeError(
        "Frozen inferred execution order contains duplicate Build-Test rows."
    )

if inferred_execution_order.duplicated(
    subset=[
        "Test",
        "InferredTestOrder",
    ],
    keep=False,
).any():
    raise RuntimeError(
        "Frozen inferred execution order contains duplicate per-test order rows."
    )

required_global_order_columns = {
    "GlobalBuildOrder",
    "BuildID",
}

missing_global_order_columns = (
    required_global_order_columns
    - set(frozen_global_build_order.columns)
)

if missing_global_order_columns:
    raise RuntimeError(
        "Frozen global build order is missing columns: "
        f"{sorted(missing_global_order_columns)}"
    )

frozen_global_build_order["GlobalBuildOrder"] = parse_int(
    frozen_global_build_order["GlobalBuildOrder"],
    "frozen_global_build_order.GlobalBuildOrder",
)

frozen_global_build_order["BuildID"] = parse_int(
    frozen_global_build_order["BuildID"],
    "frozen_global_build_order.BuildID",
)

frozen_global_build_order = (
    frozen_global_build_order.sort_values(
        "GlobalBuildOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if not np.array_equal(
    frozen_global_build_order[
        "GlobalBuildOrder"
    ].to_numpy(dtype=np.int64),
    np.arange(
        1,
        EXPECTED_BUILDS + 1,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Frozen global build-order sequence is not canonical."
    )

if frozen_global_build_order["BuildID"].nunique() != EXPECTED_BUILDS:
    raise RuntimeError(
        "Frozen global build order contains duplicate build IDs."
    )

ordered_builds = (
    frozen_global_build_order[
        "BuildID"
    ]
    .astype(int)
    .tolist()
)

global_build_position = {
    int(build_id): position
    for position, build_id in enumerate(ordered_builds)
}

build_entity["BuildID"] = parse_int(
    build_entity["BuildID"],
    "build_entity.BuildID",
)
build_entity["EntityId"] = parse_int(
    build_entity["EntityId"],
    "build_entity.EntityId",
)

changed_entities_by_build = (
    build_entity.groupby(
        "BuildID"
    )["EntityId"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

changed_entities_by_build = {
    int(build_id): set(
        int(entity_id)
        for entity_id in changed_entities_by_build.get(
            int(build_id),
            set(),
        )
    )
    for build_id in ordered_builds
}

entity_changed_builds = (
    build_entity.groupby(
        "EntityId"
    )["BuildID"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

entity_changed_builds = {
    int(entity_id): set(
        int(build_id)
        for build_id in build_ids
    )
    for entity_id, build_ids in entity_changed_builds.items()
}

for frame, label in [
    (raw_training, "raw training"),
    (raw_evaluation, "raw evaluation"),
]:
    if "Job" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Job."
        )
    if "Duration" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Duration."
        )
    if "InferredTestOrder" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing InferredTestOrder."
        )

    frame["InferredTestOrder"] = parse_int(
        frame["InferredTestOrder"],
        f"{label}.InferredTestOrder",
    )

    frame["Duration"] = pd.to_numeric(
        frame["Duration"],
        errors="coerce",
    )

    if not np.isfinite(
        frame["Duration"].to_numpy(dtype=float)
    ).all():
        raise RuntimeError(
            f"{label} cohort contains non-finite durations."
        )

    if frame["Duration"].lt(0).any():
        raise RuntimeError(
            f"{label} cohort contains negative durations."
        )

combined_raw_order = (
    pd.concat(
        [
            raw_training[
                [
                    "Build",
                    "Test",
                    "Job",
                    "Verdict",
                    "Duration",
                    "InferredTestOrder",
                ]
            ],
            raw_evaluation[
                [
                    "Build",
                    "Test",
                    "Job",
                    "Verdict",
                    "Duration",
                    "InferredTestOrder",
                ]
            ],
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "Test",
            "InferredTestOrder",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

inferred_order_reference = (
    inferred_execution_order[
        [
            "Build",
            "Test",
            "Job",
            "Verdict",
            "Duration",
            "InferredTestOrder",
        ]
    ]
    .sort_values(
        [
            "Test",
            "InferredTestOrder",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

raw_order_key_mismatches = (
    EXPECTED_RAW_ROWS
    if len(combined_raw_order) != len(inferred_order_reference)
    else int(
        (
            combined_raw_order[
                [
                    "Build",
                    "Test",
                    "Verdict",
                    "InferredTestOrder",
                ]
            ].to_numpy(dtype=np.int64)
            != inferred_order_reference[
                [
                    "Build",
                    "Test",
                    "Verdict",
                    "InferredTestOrder",
                ]
            ].to_numpy(dtype=np.int64)
        ).sum()
    )
)

raw_order_numeric_mismatches = (
    EXPECTED_RAW_ROWS
    if len(combined_raw_order) != len(inferred_order_reference)
    else int(
        (
            ~np.isclose(
                combined_raw_order[
                    [
                        "Job",
                        "Duration",
                    ]
                ].to_numpy(dtype=float),
                inferred_order_reference[
                    [
                        "Job",
                        "Duration",
                    ]
                ].to_numpy(dtype=float),
                rtol=0,
                atol=0,
                equal_nan=False,
            )
        ).sum()
    )
)

if (
    raw_order_key_mismatches != 0
    or raw_order_numeric_mismatches != 0
):
    raise RuntimeError(
        "The fixed raw cohorts no longer reproduce the frozen V6 "
        "inferred execution order."
    )

clean_raw_training_verdict = raw_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_raw_evaluation_verdict = raw_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_training_verdict = model_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_verdict = model_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_binary = (
    clean_model_evaluation_verdict != 0
).astype(np.int8)

if int((clean_model_training_verdict != 0).sum()) != EXPECTED_MODEL_TRAIN_FAILURES:
    raise RuntimeError(
        "Clean model-training failure count differs."
    )

if int(clean_model_evaluation_binary.sum()) != EXPECTED_MODEL_EVAL_FAILURES:
    raise RuntimeError(
        "Clean model-evaluation failure count differs."
    )

evaluation_duration = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Duration"].to_numpy(dtype=float)

if not np.isfinite(evaluation_duration).all():
    raise RuntimeError(
        "Model evaluation durations are non-finite."
    )

failing_evaluation_builds = sorted(
    model_evaluation.loc[
        clean_model_evaluation_binary == 1,
        "Build",
    ]
    .astype(int)
    .unique()
    .tolist()
)

if len(failing_evaluation_builds) != EXPECTED_FAILING_EVAL_BUILDS:
    raise RuntimeError(
        "Failing evaluation-build count differs."
    )

evaluation_meta_base = pd.DataFrame({
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Build": model_evaluation["Build"].to_numpy(dtype=np.int64),
    "Test": model_evaluation["Test"].to_numpy(dtype=np.int64),
    "CleanVerdict": clean_model_evaluation_verdict.astype(np.int64),
    "CleanFailure": clean_model_evaluation_binary.astype(np.int8),
    "Duration": evaluation_duration.astype(float),
})

if evaluation_meta_base.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Evaluation metadata contains duplicate Build-Test rows."
    )

random_scores = np.empty(
    EXPECTED_MODEL_EVAL_ROWS,
    dtype=np.float64,
)

for build_id in sorted(
    evaluation_meta_base["Build"].unique()
):
    build_indices = np.flatnonzero(
        evaluation_meta_base["Build"].to_numpy(dtype=np.int64)
        == int(build_id)
    )

    random_scores[build_indices] = np.random.default_rng(
        deterministic_random_build_seed(
            SMOKE_REPETITION_SEED,
            int(build_id),
        )
    ).random(len(build_indices))

if not np.isfinite(random_scores).all():
    raise RuntimeError(
        "Random baseline produced non-finite scores."
    )


# --------------------------------------------------------------------------------------------------
# 8. RUN THE TWO END-TO-END SMOKE CONDITIONS
# --------------------------------------------------------------------------------------------------

# Remove only incomplete/previous Project 22 smoke-test outputs.
# Frozen Steps 0–4A and the future full-result root are untouched.
if SMOKE_ROOT.exists():
    shutil.rmtree(
        SMOKE_ROOT
    )

SMOKE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

raw_training_hash_before = sha256_file(
    RAW_TRAINING_COHORT_PATH
)
raw_evaluation_hash_before = sha256_file(
    RAW_EVALUATION_COHORT_PATH
)
model_training_hash_before = sha256_file(
    MODEL_TRAINING_COHORT_PATH
)
model_evaluation_hash_before = sha256_file(
    MODEL_EVALUATION_COHORT_PATH
)

condition_inventory_records = []
all_condition_audits = []
all_project_runs = []
all_build_metrics = []
all_model_fits = []
all_rankings_for_invariance = []

smoke_execution_started = time.perf_counter()

for smoke_index, plan_row in enumerate(
    smoke_plan.itertuples(index=False),
    start=1,
):
    condition_started = time.perf_counter()
    condition_key = str(plan_row.ConditionID)
    noise_percent = int(plan_row.NoisePercent)
    repetition_seed = int(plan_row.RepetitionSeed)

    print("\n" + "-" * 136)
    print(
        f"[{smoke_index}/{EXPECTED_SMOKE_CONDITIONS}] "
        f"Running {condition_key}"
    )
    print("-" * 136)

    condition_dir = (
        SMOKE_ROOT
        / condition_key
    )
    condition_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    ranking_path = (
        condition_dir
        / "rankings.csv.gz"
    )
    build_metrics_path = (
        condition_dir
        / "build_metrics.csv"
    )
    project_runs_path = (
        condition_dir
        / "project_runs.csv"
    )
    model_fits_path = (
        condition_dir
        / "model_fits.csv"
    )
    training_medians_path = (
        condition_dir
        / "training_medians.csv"
    )
    condition_audit_path = (
        condition_dir
        / "condition_audit.csv"
    )
    condition_summary_path = (
        condition_dir
        / "condition_summary.json"
    )
    completion_marker_path = (
        condition_dir
        / "COMPLETE.json"
    )

    flip_mask = (
        flip_uniform
        < (noise_percent / 100.0)
    )

    noisy_raw_training_verdict = clean_raw_training_verdict.copy()

    pass_to_failure_mask = (
        flip_mask
        & (clean_raw_training_verdict == 0)
    )
    failure_to_pass_mask = (
        flip_mask
        & (clean_raw_training_verdict != 0)
    )

    noisy_raw_training_verdict[
        pass_to_failure_mask
    ] = sampled_failure_subtype[
        pass_to_failure_mask
    ]
    noisy_raw_training_verdict[
        failure_to_pass_mask
    ] = 0

    noisy_model_training_verdict = noisy_raw_training_verdict[
        model_training_raw_indices
    ]

    actual_flip_mask_sha256 = sha256_array(
        flip_mask.astype(np.uint8),
        "u1",
    )
    actual_noisy_raw_sha256 = sha256_array(
        noisy_raw_training_verdict,
        "<i2",
    )
    actual_noisy_model_sha256 = sha256_array(
        noisy_model_training_verdict,
        "<i2",
    )

    expected_flip_mask_sha256 = str(
        plan_row.FlipMaskSHA256
    )
    expected_noisy_raw_sha256 = str(
        plan_row.NoisyRawVerdictSHA256
    )
    expected_noisy_model_sha256 = str(
        plan_row.NoisyModelVerdictSHA256
    )

    if actual_flip_mask_sha256 != expected_flip_mask_sha256:
        raise RuntimeError(
            f"{condition_key}: flip-mask SHA-256 differs from Step 3A."
        )

    if actual_noisy_raw_sha256 != expected_noisy_raw_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy raw-verdict SHA-256 differs from Step 3A."
        )

    if actual_noisy_model_sha256 != expected_noisy_model_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy model-verdict SHA-256 differs from Step 3A."
        )

    number_flipped = int(flip_mask.sum())
    pass_to_failure = int(pass_to_failure_mask.sum())
    failure_to_pass = int(failure_to_pass_mask.sum())
    model_label_changes = int(
        (
            noisy_model_training_verdict
            != clean_model_training_verdict
        ).sum()
    )
    noisy_model_training_binary = (
        noisy_model_training_verdict != 0
    ).astype(np.int8)
    noisy_model_training_failures = int(
        noisy_model_training_binary.sum()
    )

    if number_flipped != int(plan_row.NumberFlipped):
        raise RuntimeError(
            f"{condition_key}: NumberFlipped differs from Step 3A."
        )
    if pass_to_failure != int(plan_row.PassToFailure):
        raise RuntimeError(
            f"{condition_key}: PassToFailure differs from Step 3A."
        )
    if failure_to_pass != int(plan_row.FailureToPass):
        raise RuntimeError(
            f"{condition_key}: FailureToPass differs from Step 3A."
        )
    if model_label_changes != int(plan_row.ModelLabelChanges):
        raise RuntimeError(
            f"{condition_key}: ModelLabelChanges differs from Step 3A."
        )
    if noisy_model_training_failures != int(plan_row.NoisyModelFailures):
        raise RuntimeError(
            f"{condition_key}: NoisyModelFailures differs from Step 3A."
        )

    print(
        "  Reconstructing REC features from the condition-specific history."
    )

    train_history = pd.DataFrame({
        "build": raw_training["Build"].to_numpy(dtype=np.int64),
        "test": raw_training["Test"].to_numpy(dtype=np.int64),
        "job": raw_training["Job"].to_numpy(),
        "verdict": noisy_raw_training_verdict.astype(np.int16),
        "duration": raw_training["Duration"].to_numpy(dtype=float),
        "inferred_test_order": raw_training[
            "InferredTestOrder"
        ].to_numpy(dtype=np.int64),
    })

    evaluation_history = pd.DataFrame({
        "build": raw_evaluation["Build"].to_numpy(dtype=np.int64),
        "test": raw_evaluation["Test"].to_numpy(dtype=np.int64),
        "job": raw_evaluation["Job"].to_numpy(),
        "verdict": clean_raw_evaluation_verdict.astype(np.int16),
        "duration": raw_evaluation["Duration"].to_numpy(dtype=float),
        "inferred_test_order": raw_evaluation[
            "InferredTestOrder"
        ].to_numpy(dtype=np.int64),
    })

    execution_history = pd.concat(
        [
            train_history,
            evaluation_history,
        ],
        ignore_index=True,
    )

    if execution_history.duplicated(
        subset=[
            "build",
            "test",
        ],
        keep=False,
    ).any():
        raise RuntimeError(
            f"{condition_key}: execution history has duplicate Build-Test rows."
        )

    if execution_history.duplicated(
        subset=[
            "test",
            "inferred_test_order",
        ],
        keep=False,
    ).any():
        raise RuntimeError(
            f"{condition_key}: execution history has duplicate per-test order rows."
        )

    execution_history = (
        execution_history.sort_values(
            [
                "test",
                "inferred_test_order",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    rec_started = time.perf_counter()

    reconstructed = reconstruct_rec_features(
        execution_history=execution_history,
        requested_rows=model_all[["Build", "Test"]],
        global_build_position=global_build_position,
        changed_entities_by_build=changed_entities_by_build,
        entity_changed_builds=entity_changed_builds,
        recent_window=RECENT_WINDOW,
    )

    rec_seconds = time.perf_counter() - rec_started

    if len(reconstructed) != EXPECTED_MODEL_ROWS:
        raise RuntimeError(
            f"{condition_key}: reconstructed REC row count differs."
        )

    if reconstructed.duplicated(
        subset=["Build", "Test"],
        keep=False,
    ).any():
        raise RuntimeError(
            f"{condition_key}: reconstructed REC contains duplicate keys."
        )

    reconstructed["Build"] = parse_int(
        reconstructed["Build"],
        f"{condition_key}.reconstructed.Build",
    )
    reconstructed["Test"] = parse_int(
        reconstructed["Test"],
        f"{condition_key}.reconstructed.Test",
    )

    reconstructed_indexed = reconstructed.set_index(
        [
            "Build",
            "Test",
        ]
    )

    missing_reconstructed_keys = model_key_index.difference(
        reconstructed_indexed.index
    )

    if len(missing_reconstructed_keys) != 0:
        raise RuntimeError(
            f"{condition_key}: reconstruction does not cover all model rows."
        )

    reconstructed_values_all = reconstructed_indexed.loc[
        model_key_index,
        REC_FEATURES,
    ].to_numpy(dtype=np.float64)

    anchored_values_all = (
        reconstructed_values_all
        + anchor_values_all
    )

    independent_reconstruction_mismatches = int(
        (~np.isclose(
            anchored_values_all[:, [
                REC_FEATURES.index(feature)
                for feature in VERDICT_INDEPENDENT_REC
            ]],
            clean_original_rec_all[:, [
                REC_FEATURES.index(feature)
                for feature in VERDICT_INDEPENDENT_REC
            ]],
            rtol=0,
            atol=1e-12,
        )).sum()
    )

    if independent_reconstruction_mismatches != 0:
        raise RuntimeError(
            f"{condition_key}: verdict-independent REC reconstruction changed."
        )

    condition_numeric_all = all_base_numeric.copy()

    dependent_rec_values_all = anchored_values_all[:, [
        REC_FEATURES.index(feature)
        for feature in VERDICT_DEPENDENT_REC
    ]]

    condition_numeric_all[:, dependent_predictor_indices] = (
        dependent_rec_values_all
    )

    condition_training_numeric = condition_numeric_all[
        :EXPECTED_MODEL_TRAIN_ROWS
    ].copy()
    condition_evaluation_numeric = condition_numeric_all[
        EXPECTED_MODEL_TRAIN_ROWS:
    ].copy()

    condition_original_dependent_all = all_base_numeric[
        :, dependent_predictor_indices
    ]

    dependent_rec_changes = int(
        (~np.isclose(
            condition_numeric_all[:, dependent_predictor_indices],
            condition_original_dependent_all,
            rtol=0,
            atol=1e-12,
            equal_nan=True,
        )).sum()
    )

    independent_rec_changes = int(
        (~np.isclose(
            condition_numeric_all[:, independent_predictor_indices],
            all_base_numeric[:, independent_predictor_indices],
            rtol=0,
            atol=0,
            equal_nan=True,
        )).sum()
    )

    if independent_rec_changes != 0:
        raise RuntimeError(
            f"{condition_key}: preserved independent REC predictors changed."
        )

    if noise_percent == 0:
        zero_rec_mismatches = int(
            (~np.isclose(
                condition_numeric_all[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                all_base_numeric[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                rtol=0,
                atol=1e-12,
                equal_nan=True,
            )).sum()
        )

        if zero_rec_mismatches != 0:
            raise RuntimeError(
                "0% smoke condition did not reproduce the clean REC cohort."
            )

        if number_flipped != 0:
            raise RuntimeError(
                "0% smoke condition unexpectedly flipped raw labels."
            )

        if model_label_changes != 0:
            raise RuntimeError(
                "0% smoke condition unexpectedly changed model labels."
            )

        if dependent_rec_changes != 0:
            raise RuntimeError(
                "0% smoke condition unexpectedly changed dependent REC values."
            )

    if noise_percent > 0:
        if number_flipped <= 0:
            raise RuntimeError(
                "Positive-noise smoke condition changed no raw labels."
            )
        if model_label_changes <= 0:
            raise RuntimeError(
                "Positive-noise smoke condition changed no model labels."
            )
        if dependent_rec_changes <= 0:
            raise RuntimeError(
                "Positive-noise smoke condition changed no dependent REC values."
            )

    medians = np.nanmedian(
        condition_training_numeric,
        axis=0,
    )

    nonfinite_median_indices = np.flatnonzero(
        ~np.isfinite(medians)
    )

    if len(nonfinite_median_indices) != 0:
        bad_features = [
            predictor_columns[index]
            for index in nonfinite_median_indices
        ]
        raise RuntimeError(
            f"{condition_key}: non-finite training medians for {bad_features}."
        )

    training_missing_mask = ~np.isfinite(
        condition_training_numeric
    )
    evaluation_missing_mask = ~np.isfinite(
        condition_evaluation_numeric
    )

    if training_missing_mask.any():
        row_indices, column_indices = np.where(
            training_missing_mask
        )
        condition_training_numeric[
            row_indices,
            column_indices,
        ] = medians[column_indices]

    if evaluation_missing_mask.any():
        row_indices, column_indices = np.where(
            evaluation_missing_mask
        )
        condition_evaluation_numeric[
            row_indices,
            column_indices,
        ] = medians[column_indices]

    if not np.isfinite(condition_training_numeric).all():
        raise RuntimeError(
            f"{condition_key}: training matrix remains non-finite after imputation."
        )

    if not np.isfinite(condition_evaluation_numeric).all():
        raise RuntimeError(
            f"{condition_key}: evaluation matrix remains non-finite after imputation."
        )

    # Keep float64 throughout the smoke test. This matches the frozen Step 4A
    # numeric/imputation contract and avoids changing ranking/model behaviour
    # through an unapproved dtype conversion.

    training_medians = pd.DataFrame({
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "PredictorOrder": np.arange(
            1,
            EXPECTED_PREDICTORS + 1,
            dtype=np.int64,
        ),
        "Predictor": predictor_columns,
        "TrainingMedian": medians.astype(float),
    })

    evaluation_meta = evaluation_meta_base.copy()
    evaluation_meta["ConditionKey"] = condition_key
    evaluation_meta["NoisePercent"] = noise_percent
    evaluation_meta["RepetitionSeed"] = repetition_seed

    technique_scores = {}
    model_fit_records = []
    models = create_models(repetition_seed)

    for technique in ML_TECHNIQUES:
        print(f"  Fitting: {technique}")
        model = models[technique]
        fit_started = time.perf_counter()
        fit_status = "PASS_MODEL_FIT"
        fit_error = ""

        try:
            model.fit(
                condition_training_numeric,
                noisy_model_training_binary,
            )

            fit_seconds = time.perf_counter() - fit_started
            scores = positive_probability(
                model,
                condition_evaluation_numeric,
            )

            technique_scores[technique] = scores

        except Exception as error:
            fit_seconds = time.perf_counter() - fit_started
            fit_status = "FAIL_MODEL_FIT"
            fit_error = repr(error)

            model_fit_records.append({
                "ProjectNumber": PROJECT_NUMBER,
                "Project": PROJECT_NAME,
                "ProjectSlug": PROJECT_SLUG,
                "ConditionKey": condition_key,
                "NoisePercent": noise_percent,
                "RepetitionSeed": repetition_seed,
                "Technique": technique,
                "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
                "TrainingFailures": noisy_model_training_failures,
                "Predictors": EXPECTED_PREDICTORS,
                "FitSeconds": float(fit_seconds),
                "ClassesJSON": "[]",
                "Status": fit_status,
                "Error": fit_error,
            })

            raise RuntimeError(
                f"{condition_key}: {technique} fitting failed: {error!r}"
            ) from error

        model_fit_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": condition_key,
            "NoisePercent": noise_percent,
            "RepetitionSeed": repetition_seed,
            "Technique": technique,
            "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
            "TrainingFailures": noisy_model_training_failures,
            "Predictors": EXPECTED_PREDICTORS,
            "FitSeconds": float(fit_seconds),
            "ClassesJSON": json.dumps(
                [
                    int(value)
                    for value in np.asarray(model.classes_).tolist()
                ]
            ),
            "Status": fit_status,
            "Error": fit_error,
        })

        del model
        gc.collect()

    model_fits = pd.DataFrame(model_fit_records)

    if len(model_fits) != EXPECTED_MODEL_FIT_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: model-fit row count differs."
        )

    if not model_fits["Status"].eq("PASS_MODEL_FIT").all():
        raise RuntimeError(
            f"{condition_key}: one or more model fits failed."
        )

    condition_eval_last_failure_age = condition_evaluation_numeric[
        :,
        predictor_index["REC_LastFailureAge"],
    ].astype(float)

    condition_eval_qtf = condition_evaluation_numeric[
        :,
        predictor_index["REC_TotalAvgExeTime"],
    ].astype(float)

    technique_scores["Random"] = random_scores.copy()
    technique_scores["LatestFail"] = (
        -condition_eval_last_failure_age
    )
    technique_scores["QTF-Avg"] = condition_eval_qtf

    ranking_frames = []

    for technique in ALL_TECHNIQUES:
        ranking_frames.append(
            make_ranking(
                evaluation_meta=evaluation_meta,
                technique=technique,
                scores=technique_scores[technique],
                ascending_score=(technique == "QTF-Avg"),
            )
        )

    rankings = pd.concat(
        ranking_frames,
        ignore_index=True,
    )

    if len(rankings) != EXPECTED_RANKING_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: ranking row count differs."
        )

    ranking_techniques = sorted(
        rankings["Technique"].unique().tolist()
    )

    if ranking_techniques != sorted(ALL_TECHNIQUES):
        raise RuntimeError(
            f"{condition_key}: ranking technique set differs."
        )

    ranking_rows_per_technique = rankings.groupby(
        "Technique"
    ).size()

    if not ranking_rows_per_technique.eq(
        EXPECTED_MODEL_EVAL_ROWS
    ).all():
        raise RuntimeError(
            f"{condition_key}: ranking rows per technique differ."
        )

    duplicate_ranking_rows = int(
        rankings.duplicated(
            subset=[
                "Technique",
                "Build",
                "Test",
            ],
            keep=False,
        ).sum()
    )

    if duplicate_ranking_rows != 0:
        raise RuntimeError(
            f"{condition_key}: duplicate ranking rows found."
        )

    build_metrics, project_runs = calculate_condition_metrics(
        rankings
    )

    if len(build_metrics) != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: build-metric row count differs."
        )

    if len(project_runs) != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: project-run row count differs."
        )

    if sorted(project_runs["Technique"].tolist()) != sorted(
        ALL_TECHNIQUES
    ):
        raise RuntimeError(
            f"{condition_key}: project-run technique set differs."
        )

    metric_columns = [
        "APFDc",
        "APFD",
    ]

    build_metric_values = build_metrics[
        metric_columns
    ].to_numpy(dtype=float)

    if not np.isfinite(build_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: build metrics contain non-finite values."
        )

    if ((build_metric_values < 0) | (build_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: build metrics fall outside [0,1]."
        )

    project_metric_columns = [
        "MeanAPFDc",
        "MedianAPFDc",
        "MeanAPFD",
        "MedianAPFD",
    ]

    project_metric_values = project_runs[
        project_metric_columns
    ].to_numpy(dtype=float)

    if not np.isfinite(project_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: project metrics contain non-finite values."
        )

    if ((project_metric_values < 0) | (project_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: project metrics fall outside [0,1]."
        )

    condition_seconds = time.perf_counter() - condition_started

    condition_audit = pd.DataFrame([{
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "RawTrainingRows": EXPECTED_RAW_TRAIN_ROWS,
        "NumberFlipped": number_flipped,
        "ExpectedNumberFlipped": int(plan_row.NumberFlipped),
        "RealisedNoisePercent": float(
            100.0 * number_flipped / EXPECTED_RAW_TRAIN_ROWS
        ),
        "PassToFailure": pass_to_failure,
        "FailureToPass": failure_to_pass,
        "ModelTrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
        "ModelLabelChanges": model_label_changes,
        "ExpectedModelLabelChanges": int(plan_row.ModelLabelChanges),
        "TrainingFailures": noisy_model_training_failures,
        "ExpectedTrainingFailures": int(plan_row.NoisyModelFailures),
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "IndependentReconstructionMismatches": independent_reconstruction_mismatches,
        "ReconstructedRows": int(len(reconstructed)),
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ExpectedFlipMaskSHA256": expected_flip_mask_sha256,
        "ActualFlipMaskSHA256": actual_flip_mask_sha256,
        "ExpectedNoisyRawVerdictSHA256": expected_noisy_raw_sha256,
        "ActualNoisyRawVerdictSHA256": actual_noisy_raw_sha256,
        "ExpectedNoisyModelVerdictSHA256": expected_noisy_model_sha256,
        "ActualNoisyModelVerdictSHA256": actual_noisy_model_sha256,
        "RECSeconds": float(rec_seconds),
        "ConditionSeconds": float(condition_seconds),
        "Status": CONDITION_STATUS,
    }])

    atomic_csv(
        ranking_path,
        rankings,
        compression="gzip",
    )
    atomic_csv(
        build_metrics_path,
        build_metrics,
    )
    atomic_csv(
        project_runs_path,
        project_runs,
    )
    atomic_csv(
        model_fits_path,
        model_fits,
    )
    atomic_csv(
        training_medians_path,
        training_medians,
    )
    atomic_csv(
        condition_audit_path,
        condition_audit,
    )

    condition_output_paths = [
        ranking_path,
        build_metrics_path,
        project_runs_path,
        model_fits_path,
        training_medians_path,
        condition_audit_path,
    ]

    condition_output_manifest = [
        {
            "Path": str(path),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        }
        for path in condition_output_paths
    ]

    condition_summary = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "Status": CONDITION_STATUS,
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
        "NumberFlipped": number_flipped,
        "ModelLabelChanges": model_label_changes,
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "TrainingFailures": noisy_model_training_failures,
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ConditionSeconds": float(condition_seconds),
        "OutputManifest": condition_output_manifest,
    }

    atomic_json(
        condition_summary_path,
        condition_summary,
    )

    completion_marker = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "Status": CONDITION_STATUS,
        "ConditionSummaryPath": str(condition_summary_path),
        "ConditionSummarySHA256": sha256_file(condition_summary_path),
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
    }

    atomic_json(
        completion_marker_path,
        completion_marker,
    )

    condition_manifest = directory_manifest(
        condition_dir
    )
    condition_root_sha256 = directory_root_hash(
        condition_manifest
    )

    condition_inventory_records.append({
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": smoke_index,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "ConditionDirectory": str(condition_dir),
        "Status": CONDITION_STATUS,
        "Files": int(len(condition_manifest)),
        "ConditionBytes": int(condition_manifest["Bytes"].sum()),
        "ConditionRootSHA256": condition_root_sha256,
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "ModelFitRows": int(len(model_fits)),
        "TrainingMedianRows": int(len(training_medians)),
        "ConditionSeconds": float(condition_seconds),
    })

    all_condition_audits.append(condition_audit)
    all_project_runs.append(project_runs)
    all_build_metrics.append(build_metrics)
    all_model_fits.append(model_fits)
    all_rankings_for_invariance.append(
        rankings.loc[
            rankings["Technique"].isin(
                [
                    "Random",
                    "QTF-Avg",
                ]
            )
        ].copy()
    )

    print(
        f"  Completed: {condition_key}\n"
        f"  Raw flips: {number_flipped} | "
        f"model-label changes: {model_label_changes} | "
        f"dependent REC changes: {dependent_rec_changes}\n"
        f"  Training failures: {noisy_model_training_failures} | "
        f"condition seconds: {condition_seconds:.2f}"
    )

    del train_history
    del evaluation_history
    del execution_history
    del reconstructed
    del reconstructed_indexed
    del reconstructed_values_all
    del anchored_values_all
    del condition_numeric_all
    del condition_training_numeric
    del condition_evaluation_numeric
    del rankings
    del ranking_frames
    del technique_scores
    del models
    gc.collect()


# --------------------------------------------------------------------------------------------------
# 9. COMBINE SMOKE OUTPUTS AND VERIFY BASELINE INVARIANCE
# --------------------------------------------------------------------------------------------------

condition_inventory = pd.DataFrame(
    condition_inventory_records
)
combined_condition_audit = pd.concat(
    all_condition_audits,
    ignore_index=True,
)
combined_project_runs = pd.concat(
    all_project_runs,
    ignore_index=True,
)
combined_build_metrics = pd.concat(
    all_build_metrics,
    ignore_index=True,
)
combined_model_fits = pd.concat(
    all_model_fits,
    ignore_index=True,
)
combined_invariance_rankings = pd.concat(
    all_rankings_for_invariance,
    ignore_index=True,
)

baseline_invariance_records = []

for technique in [
    "Random",
    "QTF-Avg",
]:
    zero_rows = (
        combined_invariance_rankings.loc[
            (
                combined_invariance_rankings["Technique"].eq(technique)
                & combined_invariance_rankings["NoisePercent"].eq(0)
            )
        ]
        .sort_values(
            [
                "Build",
                "Test",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    noisy_rows = (
        combined_invariance_rankings.loc[
            (
                combined_invariance_rankings["Technique"].eq(technique)
                & combined_invariance_rankings["NoisePercent"].eq(50)
            )
        ]
        .sort_values(
            [
                "Build",
                "Test",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    same_keys = bool(
        zero_rows[["Build", "Test"]].equals(
            noisy_rows[["Build", "Test"]]
        )
    )

    score_mismatches = (
        EXPECTED_MODEL_EVAL_ROWS
        if not same_keys
        else int(
            (~np.isclose(
                zero_rows["Score"].to_numpy(dtype=float),
                noisy_rows["Score"].to_numpy(dtype=float),
                rtol=0,
                atol=0,
            )).sum()
        )
    )

    rank_mismatches = (
        EXPECTED_MODEL_EVAL_ROWS
        if not same_keys
        else int(
            (
                zero_rows["Rank"].to_numpy(dtype=np.int64)
                != noisy_rows["Rank"].to_numpy(dtype=np.int64)
            ).sum()
        )
    )

    baseline_invariance_records.append({
        "Technique": technique,
        "Rows": int(len(zero_rows)),
        "SameBuildTestKeys": same_keys,
        "ScoreMismatches": score_mismatches,
        "RankMismatches": rank_mismatches,
        "Pass": (
            len(zero_rows) == EXPECTED_MODEL_EVAL_ROWS
            and len(noisy_rows) == EXPECTED_MODEL_EVAL_ROWS
            and same_keys
            and score_mismatches == 0
            and rank_mismatches == 0
        ),
    })

baseline_invariance = pd.DataFrame(
    baseline_invariance_records
)
baseline_invariance_failures = int(
    (~baseline_invariance["Pass"]).sum()
)

if baseline_invariance_failures != 0:
    print("\nBaseline invariance failures:")
    display(
        baseline_invariance.loc[
            ~baseline_invariance["Pass"]
        ]
    )
    raise RuntimeError(
        "Random or QTF-Avg changed across the two smoke noise levels."
    )

atomic_csv(
    SMOKE_CONDITION_INVENTORY_PATH,
    condition_inventory,
)
atomic_csv(
    SMOKE_COMBINED_CONDITION_AUDIT_PATH,
    combined_condition_audit,
)
atomic_csv(
    SMOKE_COMBINED_PROJECT_RUNS_PATH,
    combined_project_runs,
)
atomic_csv(
    SMOKE_COMBINED_BUILD_METRICS_PATH,
    combined_build_metrics,
)
atomic_csv(
    SMOKE_COMBINED_MODEL_FITS_PATH,
    combined_model_fits,
)
atomic_csv(
    SMOKE_BASELINE_INVARIANCE_PATH,
    baseline_invariance,
)


# --------------------------------------------------------------------------------------------------
# 10. FINAL VALIDATION
# --------------------------------------------------------------------------------------------------

raw_training_hash_after = sha256_file(
    RAW_TRAINING_COHORT_PATH
)
raw_evaluation_hash_after = sha256_file(
    RAW_EVALUATION_COHORT_PATH
)
model_training_hash_after = sha256_file(
    MODEL_TRAINING_COHORT_PATH
)
model_evaluation_hash_after = sha256_file(
    MODEL_EVALUATION_COHORT_PATH
)
registry_sha256_after = sha256_file(
    REGISTRY_PATH
)

current_source_rows_after = []

for row in frozen_source_manifest.itertuples(index=False):
    source_path = (
        SOURCE_DIR
        / str(row.RelativePath)
    )
    current_source_rows_after.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })

source_root_sha256_after = source_root_hash(
    pd.DataFrame(current_source_rows_after)
)

full_raw_result_root_exists_after = FULL_RAW_RESULT_ROOT.exists()
full_raw_result_manifest_after = directory_manifest(
    FULL_RAW_RESULT_ROOT
)
full_raw_result_root_hash_after = directory_root_hash(
    full_raw_result_manifest_after
)

full_raw_result_unchanged = bool(
    full_raw_result_root_existed_before
    == full_raw_result_root_exists_after
    and full_raw_result_root_hash_before
    == full_raw_result_root_hash_after
    and len(full_raw_result_manifest_before)
    == len(full_raw_result_manifest_after)
)

zero_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].eq(0)
].iloc[0]

positive_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].eq(50)
].iloc[0]

validation_records = []

add_check(
    validation_records,
    "Step 4A status",
    EXPECTED_STEP4A_STATUS,
    step4a_status.get("Status"),
    step4a_status.get("Status") == EXPECTED_STEP4A_STATUS,
)
add_check(
    validation_records,
    "Runtime checkpoint SHA-256",
    EXPECTED_RUNTIME_CHECKPOINT_SHA256,
    runtime_checkpoint_sha256,
    runtime_checkpoint_sha256 == EXPECTED_RUNTIME_CHECKPOINT_SHA256,
)
add_check(
    validation_records,
    "Noise-plan checkpoint SHA-256",
    EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
    noise_plan_checkpoint_sha256,
    noise_plan_checkpoint_sha256 == EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
)
add_check(
    validation_records,
    "REC checkpoint SHA-256",
    EXPECTED_REC_CHECKPOINT_SHA256,
    rec_checkpoint_sha256,
    rec_checkpoint_sha256 == EXPECTED_REC_CHECKPOINT_SHA256,
)
add_check(
    validation_records,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_CHECKPOINT_SHA256,
    selection_checkpoint_sha256,
    selection_checkpoint_sha256 == EXPECTED_SELECTION_CHECKPOINT_SHA256,
)
add_check(
    validation_records,
    "REC output-manifest failures",
    0,
    rec_manifest_failures,
    rec_manifest_failures == 0,
)
add_check(
    validation_records,
    "Noise-plan output-manifest failures",
    0,
    noise_manifest_failures,
    noise_manifest_failures == 0,
)
add_check(
    validation_records,
    "Step 4A output-manifest failures",
    0,
    runtime_manifest_failures,
    runtime_manifest_failures == 0,
)
add_check(
    validation_records,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    source_root_sha256_after,
    source_root_sha256_after == EXPECTED_SOURCE_ROOT_SHA256,
)
add_check(
    validation_records,
    "Smoke conditions",
    EXPECTED_SMOKE_CONDITIONS,
    len(condition_inventory),
    len(condition_inventory) == EXPECTED_SMOKE_CONDITIONS,
)
add_check(
    validation_records,
    "Smoke condition keys",
    SMOKE_CONDITION_IDS,
    condition_inventory["ConditionKey"].tolist(),
    condition_inventory["ConditionKey"].tolist() == SMOKE_CONDITION_IDS,
)
add_check(
    validation_records,
    "Condition statuses",
    CONDITION_STATUS,
    sorted(condition_inventory["Status"].unique().tolist()),
    condition_inventory["Status"].eq(CONDITION_STATUS).all(),
)
add_check(
    validation_records,
    "Condition-audit rows",
    EXPECTED_SMOKE_CONDITIONS,
    len(combined_condition_audit),
    len(combined_condition_audit) == EXPECTED_SMOKE_CONDITIONS,
)
add_check(
    validation_records,
    "Total ML fits",
    EXPECTED_SMOKE_CONDITIONS * EXPECTED_ML_TECHNIQUES,
    len(combined_model_fits),
    len(combined_model_fits)
    == EXPECTED_SMOKE_CONDITIONS * EXPECTED_ML_TECHNIQUES,
)
add_check(
    validation_records,
    "Model-fit failures",
    0,
    int((~combined_model_fits["Status"].eq("PASS_MODEL_FIT")).sum()),
    combined_model_fits["Status"].eq("PASS_MODEL_FIT").all(),
)
add_check(
    validation_records,
    "Total project-run rows",
    EXPECTED_SMOKE_CONDITIONS * EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
    len(combined_project_runs),
    len(combined_project_runs)
    == EXPECTED_SMOKE_CONDITIONS * EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
)
add_check(
    validation_records,
    "Total build-metric rows",
    EXPECTED_SMOKE_CONDITIONS * EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
    len(combined_build_metrics),
    len(combined_build_metrics)
    == EXPECTED_SMOKE_CONDITIONS * EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
)
add_check(
    validation_records,
    "Technique set",
    sorted(ALL_TECHNIQUES),
    sorted(combined_project_runs["Technique"].unique().tolist()),
    sorted(combined_project_runs["Technique"].unique().tolist())
    == sorted(ALL_TECHNIQUES),
)
add_check(
    validation_records,
    "Project-run rows per condition violations",
    0,
    int((
        combined_project_runs.groupby("ConditionKey").size()
        != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
    ).sum()),
    bool((
        combined_project_runs.groupby("ConditionKey").size()
        == EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
    ).all()),
)
add_check(
    validation_records,
    "Build-metric rows per condition violations",
    0,
    int((
        combined_build_metrics.groupby("ConditionKey").size()
        != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
    ).sum()),
    bool((
        combined_build_metrics.groupby("ConditionKey").size()
        == EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
    ).all()),
)
add_check(
    validation_records,
    "Zero-noise raw flips",
    0,
    int(zero_audit["NumberFlipped"]),
    int(zero_audit["NumberFlipped"]) == 0,
)
add_check(
    validation_records,
    "Zero-noise model-label changes",
    0,
    int(zero_audit["ModelLabelChanges"]),
    int(zero_audit["ModelLabelChanges"]) == 0,
)
add_check(
    validation_records,
    "Zero-noise dependent REC changes",
    0,
    int(zero_audit["DependentRECChanges"]),
    int(zero_audit["DependentRECChanges"]) == 0,
)
add_check(
    validation_records,
    "Positive-noise raw flips",
    "> 0",
    int(positive_audit["NumberFlipped"]),
    int(positive_audit["NumberFlipped"]) > 0,
)
add_check(
    validation_records,
    "Positive-noise model-label changes",
    "> 0",
    int(positive_audit["ModelLabelChanges"]),
    int(positive_audit["ModelLabelChanges"]) > 0,
)
add_check(
    validation_records,
    "Positive-noise dependent REC changes",
    "> 0",
    int(positive_audit["DependentRECChanges"]),
    int(positive_audit["DependentRECChanges"]) > 0,
)
add_check(
    validation_records,
    "Independent REC changes",
    0,
    int(combined_condition_audit["IndependentRECChanges"].sum()),
    int(combined_condition_audit["IndependentRECChanges"].sum()) == 0,
)
add_check(
    validation_records,
    "Frozen raw-order key mismatches",
    0,
    raw_order_key_mismatches,
    raw_order_key_mismatches == 0,
)
add_check(
    validation_records,
    "Frozen raw-order numeric mismatches",
    0,
    raw_order_numeric_mismatches,
    raw_order_numeric_mismatches == 0,
)
add_check(
    validation_records,
    "Independent REC reconstruction mismatches",
    0,
    int(combined_condition_audit[
        "IndependentReconstructionMismatches"
    ].sum()),
    int(combined_condition_audit[
        "IndependentReconstructionMismatches"
    ].sum()) == 0,
)
add_check(
    validation_records,
    "Noise-plan hash mismatches",
    0,
    int((
        combined_condition_audit["ExpectedFlipMaskSHA256"]
        != combined_condition_audit["ActualFlipMaskSHA256"]
    ).sum())
    + int((
        combined_condition_audit["ExpectedNoisyRawVerdictSHA256"]
        != combined_condition_audit["ActualNoisyRawVerdictSHA256"]
    ).sum())
    + int((
        combined_condition_audit["ExpectedNoisyModelVerdictSHA256"]
        != combined_condition_audit["ActualNoisyModelVerdictSHA256"]
    ).sum()),
    bool(
        (
            combined_condition_audit["ExpectedFlipMaskSHA256"]
            == combined_condition_audit["ActualFlipMaskSHA256"]
        ).all()
        and (
            combined_condition_audit["ExpectedNoisyRawVerdictSHA256"]
            == combined_condition_audit["ActualNoisyRawVerdictSHA256"]
        ).all()
        and (
            combined_condition_audit["ExpectedNoisyModelVerdictSHA256"]
            == combined_condition_audit["ActualNoisyModelVerdictSHA256"]
        ).all()
    ),
)
add_check(
    validation_records,
    "Baseline invariance failures",
    0,
    baseline_invariance_failures,
    baseline_invariance_failures == 0,
)
add_check(
    validation_records,
    "Project metrics non-finite",
    0,
    int((~np.isfinite(
        combined_project_runs[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float)
    )).sum()),
    bool(np.isfinite(
        combined_project_runs[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float)
    ).all()),
)
add_check(
    validation_records,
    "Project metrics outside [0,1]",
    0,
    int((
        (
            combined_project_runs[[
                "MeanAPFDc",
                "MedianAPFDc",
                "MeanAPFD",
                "MedianAPFD",
            ]].to_numpy(dtype=float)
            < 0
        )
        | (
            combined_project_runs[[
                "MeanAPFDc",
                "MedianAPFDc",
                "MeanAPFD",
                "MedianAPFD",
            ]].to_numpy(dtype=float)
            > 1
        )
    ).sum()),
    bool((
        (
            combined_project_runs[[
                "MeanAPFDc",
                "MedianAPFDc",
                "MeanAPFD",
                "MedianAPFD",
            ]].to_numpy(dtype=float)
            >= 0
        )
        & (
            combined_project_runs[[
                "MeanAPFDc",
                "MedianAPFDc",
                "MeanAPFD",
                "MedianAPFD",
            ]].to_numpy(dtype=float)
            <= 1
        )
    ).all()),
)
add_check(
    validation_records,
    "Raw training cohort unchanged",
    raw_training_hash_before,
    raw_training_hash_after,
    raw_training_hash_after == raw_training_hash_before,
)
add_check(
    validation_records,
    "Raw evaluation cohort unchanged",
    raw_evaluation_hash_before,
    raw_evaluation_hash_after,
    raw_evaluation_hash_after == raw_evaluation_hash_before,
)
add_check(
    validation_records,
    "Model training cohort unchanged",
    model_training_hash_before,
    model_training_hash_after,
    model_training_hash_after == model_training_hash_before,
)
add_check(
    validation_records,
    "Model evaluation cohort unchanged",
    model_evaluation_hash_before,
    model_evaluation_hash_after,
    model_evaluation_hash_after == model_evaluation_hash_before,
)
add_check(
    validation_records,
    "Completion registry unchanged",
    registry_sha256_before,
    registry_sha256_after,
    registry_sha256_after == registry_sha256_before,
)
add_check(
    validation_records,
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(
        registry
    ),
    len(
        registry
    )
    == EXPECTED_REGISTERED_PROJECTS,
)

for predecessor_number, predecessor_project in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            registry_project_numbers.eq(
                predecessor_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {predecessor_number} frozen identity",
        predecessor_project,
        actual_project,
        actual_project == predecessor_project,
    )

add_check(
    validation_records,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    active_reservations,
    active_reservations == EXPECTED_ACTIVE_RESERVATIONS,
)

add_check(
    validation_records,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    runtime_checkpoint.get(
        "RuntimePriorityRule"
    ),
    runtime_checkpoint.get(
        "RuntimePriorityRule"
    )
    == EXPECTED_RUNTIME_PRIORITY_RULE,
)

add_check(
    validation_records,
    "Registry Project 22 rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)
add_check(
    validation_records,
    "Full raw-result root unchanged",
    True,
    full_raw_result_unchanged,
    full_raw_result_unchanged,
)

validation = pd.DataFrame(
    validation_records
)
failed_validation = validation.loc[
    ~validation["Pass"]
]

print("\nProject 22 Step 4B validation:")
display(validation)

print("\nBaseline invariance audit:")
display(baseline_invariance)

print("\nSmoke project-run results:")
display(
    combined_project_runs.sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    ).reset_index(drop=True)
)

if not failed_validation.empty:
    print("\nFailed Project 22 Step 4B checks:")
    display(failed_validation)
    print("\nNo Step 4B PASS status or checkpoint was written.")
    raise RuntimeError(
        "PROJECT 22 STEP 4B VALIDATION FAILED. DO NOT START THE FULL EXPERIMENT."
    )

atomic_csv(
    SMOKE_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 11. REPORT, CHECKPOINT, STATUS, AND FINAL READBACK
# --------------------------------------------------------------------------------------------------

smoke_execution_seconds = (
    time.perf_counter()
    - smoke_execution_started
)
completed_at_utc = datetime.now(
    timezone.utc
).isoformat()

smoke_output_paths = [
    SMOKE_CONDITION_INVENTORY_PATH,
    SMOKE_COMBINED_CONDITION_AUDIT_PATH,
    SMOKE_COMBINED_PROJECT_RUNS_PATH,
    SMOKE_COMBINED_BUILD_METRICS_PATH,
    SMOKE_COMBINED_MODEL_FITS_PATH,
    SMOKE_BASELINE_INVARIANCE_PATH,
    SMOKE_VALIDATION_PATH,
]

for condition_key in SMOKE_CONDITION_IDS:
    condition_dir = SMOKE_ROOT / condition_key
    smoke_output_paths.extend([
        path
        for path in condition_dir.rglob("*")
        if path.is_file()
    ])

smoke_output_paths = sorted(
    set(smoke_output_paths),
    key=lambda path: str(path),
)

smoke_output_manifest = [
    {
        "Path": str(path),
        "Bytes": int(path.stat().st_size),
        "SHA256": sha256_file(path),
    }
    for path in smoke_output_paths
]

report_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP4B_STATUS,
    "CompletedAtUTC": completed_at_utc,
    "RuntimeCheckpointSHA256": runtime_checkpoint_sha256,
    "NoisePlanCheckpointSHA256": noise_plan_checkpoint_sha256,
    "RECCheckpointSHA256": rec_checkpoint_sha256,
    "SelectionCheckpointSHA256": selection_checkpoint_sha256,
    "SourceRootSHA256": source_root_sha256_after,
    "SmokeConditionKeys": SMOKE_CONDITION_IDS,
    "SmokeConditions": int(len(condition_inventory)),
    "MLFits": int(len(combined_model_fits)),
    "RankingRows": int(condition_inventory["RankingRows"].sum()),
    "BuildMetricRows": int(len(combined_build_metrics)),
    "ProjectRunRows": int(len(combined_project_runs)),
    "TrainingMedianRows": int(
        condition_inventory["TrainingMedianRows"].sum()
    ),
    "ZeroNoiseRawFlips": int(zero_audit["NumberFlipped"]),
    "ZeroNoiseModelLabelChanges": int(
        zero_audit["ModelLabelChanges"]
    ),
    "ZeroNoiseDependentRECChanges": int(
        zero_audit["DependentRECChanges"]
    ),
    "PositiveNoiseRawFlips": int(
        positive_audit["NumberFlipped"]
    ),
    "PositiveNoiseModelLabelChanges": int(
        positive_audit["ModelLabelChanges"]
    ),
    "PositiveNoiseDependentRECChanges": int(
        positive_audit["DependentRECChanges"]
    ),
    "IndependentRECChanges": int(
        combined_condition_audit["IndependentRECChanges"].sum()
    ),
    "BaselineInvarianceFailures": baseline_invariance_failures,
    "ValidationChecks": int(len(validation)),
    "FailedValidationChecks": int(len(failed_validation)),
    "SmokeExecutionSeconds": float(smoke_execution_seconds),
    "OutputManifest": smoke_output_manifest,
    "RegistrySHA256": registry_sha256_after,
    "RegistryModified": False,
    "Projects1To21Modified": False,
    "ActiveReservations": EXPECTED_ACTIVE_RESERVATIONS,
    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectConditionOutputsModified": False,
    "FullRawResultRootModified": False,
    "FullExperimentStarted": False,
}

atomic_json(
    SMOKE_REPORT_PATH,
    report_payload,
)

checkpoint_payload = {
    **report_payload,
    "CheckpointVersion": 1,
    "SmokeTestPassed": True,
    "RuntimeContractFrozen": True,
    "NoisePlanFrozen": True,
    "EvaluationCohortImmutable": True,
    "ReadyForFull270ConditionExperiment": True,
}

atomic_json(
    SMOKE_CHECKPOINT_PATH,
    checkpoint_payload,
)

status_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP4B_STATUS,
    "CompletedAtUTC": completed_at_utc,
    "SmokeConditions": int(len(condition_inventory)),
    "MLFits": int(len(combined_model_fits)),
    "FailedValidationChecks": int(len(failed_validation)),
    "Checkpoint": str(SMOKE_CHECKPOINT_PATH),
    "CheckpointSHA256": sha256_file(SMOKE_CHECKPOINT_PATH),
    "RegistryModified": False,
    "PriorProjectConditionOutputsAccessed": False,
    "FullExperimentStarted": False,
}

atomic_json(
    STEP4B_STATUS_PATH,
    status_payload,
)

checkpoint_readback = load_json(
    SMOKE_CHECKPOINT_PATH
)
status_readback = load_json(
    STEP4B_STATUS_PATH
)

if checkpoint_readback.get("Status") != STEP4B_STATUS:
    raise RuntimeError(
        "Step 4B checkpoint readback failed."
    )

if status_readback.get("Status") != STEP4B_STATUS:
    raise RuntimeError(
        "Step 4B status readback failed."
    )

if sha256_file(REGISTRY_PATH) != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Step 4B finalisation."
    )

final_source_rows = []
for row in frozen_source_manifest.itertuples(index=False):
    source_path = (
        SOURCE_DIR
        / str(row.RelativePath)
    )
    final_source_rows.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })

if source_root_hash(pd.DataFrame(final_source_rows)) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Project 22 source changed during Step 4B finalisation."
    )

print("\n" + "=" * 136)
print("=== PROJECT 22 CELL 8 / STEP 4B RESULT ===")
print("=" * 136)
print()
print("Project:")
print(PROJECT_NAME)
print(
    "Project 11 identity:",
    required_registered_identities[
        11
    ],
)
print(
    "Project 12 identity:",
    required_registered_identities[
        12
    ],
)
print(
    "Project 13 identity:",
    required_registered_identities[
        13
    ],
)
print(
    "Project 14 identity:",
    required_registered_identities[
        14
    ],
)
print(
    "Project 15 identity:",
    required_registered_identities[
        15
    ],
)
print(
    "Project 16 identity:",
    required_registered_identities[
        16
    ],
)
print(
    "Project 17 identity:",
    required_registered_identities[
        17
    ],
)
print(
    "Project 18 identity:",
    required_registered_identities[
        18
    ],
)
print(
    "Project 19 identity:",
    required_registered_identities[
        19
    ],
)
print(
    "Project 20 identity:",
    required_registered_identities[
        20
    ],
)
print(
    "Project 21 identity:",
    required_registered_identities[
        21
    ],
)
print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)
print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)
print()
print("Two-condition end-to-end smoke test:")
print("Conditions:", SMOKE_CONDITION_IDS)
print("Conditions passed:", len(condition_inventory), "/", EXPECTED_SMOKE_CONDITIONS)
print("ML fits:", len(combined_model_fits), "/", EXPECTED_SMOKE_CONDITIONS * EXPECTED_ML_TECHNIQUES)
print("Ranking rows:", int(condition_inventory["RankingRows"].sum()))
print("Build-metric rows:", len(combined_build_metrics))
print("Project-run rows:", len(combined_project_runs))
print("Training-median rows:", int(condition_inventory["TrainingMedianRows"].sum()))
print()
print("Noise and REC audit:")
print("0% raw flips:", int(zero_audit["NumberFlipped"]))
print("0% model-label changes:", int(zero_audit["ModelLabelChanges"]))
print("0% dependent REC changes:", int(zero_audit["DependentRECChanges"]))
print("50% raw flips:", int(positive_audit["NumberFlipped"]))
print("50% model-label changes:", int(positive_audit["ModelLabelChanges"]))
print("50% dependent REC changes:", int(positive_audit["DependentRECChanges"]))
print("Independent REC changes:", int(combined_condition_audit["IndependentRECChanges"].sum()))
print()
print("Baselines and metrics:")
print("Random/QTF-Avg invariance failures:", baseline_invariance_failures)
print("Techniques:", ALL_TECHNIQUES)
print("Primary / secondary metrics: APFDc / APFD")
print()
print("Immutability and isolation:")
print("Project 22 source unchanged:", True)
print("Completion registry unchanged:", True)
print("Projects 1–21 modified:", 0)
print("Prior project condition outputs accessed:", False)
print("Prior project condition outputs modified:", False)
print("Full experiment raw-result root modified:", False)
print("Full 270-condition experiment started:", False)
print()
print("Validation:")
print("Checks:", len(validation))
print("Failed checks:", len(failed_validation))
print()
print("Smoke-test checkpoint:")
print(SMOKE_CHECKPOINT_PATH)
print("Checkpoint SHA-256:", sha256_file(SMOKE_CHECKPOINT_PATH))
print()
print("Runtime seconds:", round(smoke_execution_seconds, 2))
print()
print("STATUS:", STEP4B_STATUS)
print("=" * 136)


=== PROJECT 22 CELL 8 / STEP 4B: TWO-CONDITION END-TO-END SMOKE TEST ===

Loading frozen Project 22 cohorts and contracts.
Converting the fixed predictor cohorts to one numeric matrix.

----------------------------------------------------------------------------------------------------------------------------------------
[1/2] Running noise_00__seed_01
----------------------------------------------------------------------------------------------------------------------------------------
  Reconstructing REC features from the condition-specific history.
    REC reconstruction progress: 100 / 689 tests | reconstructed rows: 20322
    REC reconstruction progress: 200 / 689 tests | reconstructed rows: 38208
    REC reconstruction progress: 300 / 689 tests | reconstructed rows: 55459
    REC reconstruction progress: 400 / 689 tests | reconstructed rows: 76853
    REC reconstruction progress: 500 / 689 tests | reconstructed rows: 97248
    REC reconstruction progress: 600 / 689 tests | recon

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_00__seed_01
  Raw flips: 0 | model-label changes: 0 | dependent REC changes: 0
  Training failures: 207 | condition seconds: 404.06

----------------------------------------------------------------------------------------------------------------------------------------
[2/2] Running noise_50__seed_01
----------------------------------------------------------------------------------------------------------------------------------------
  Reconstructing REC features from the condition-specific history.
    REC reconstruction progress: 100 / 689 tests | reconstructed rows: 20322
    REC reconstruction progress: 200 / 689 tests | reconstructed rows: 38208
    REC reconstruction progress: 300 / 689 tests | reconstructed rows: 55459
    REC reconstruction progress: 400 / 689 tests | reconstructed rows: 76853
    REC reconstruction progress: 500 / 689 tests | reconstructed rows: 97248
    REC reconstruction progress: 600 / 689 tests | reconstructed row

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_50__seed_01
  Raw flips: 86584 | model-label changes: 47909 | dependent REC changes: 1212489
  Training failures: 47890 | condition seconds: 451.59

Project 22 Step 4B validation:


...


Baseline invariance audit:


,Technique,Rows,SameBuildTestKeys,ScoreMismatches,RankMismatches,Pass
0,Random,22156,True,0,0,True
1,QTF-Avg,22156,True,0,0,True



Smoke project-run results:


,ProjectNumber,Project,ProjectSlug,ConditionKey,NoisePercent,RepetitionSeed,Technique,EvaluationBuilds,ScoredFailingBuilds,EvaluationRows,EvaluationFailures,MeanAPFDc,MedianAPFDc,MeanAPFD,MedianAPFD
0,22,apache@logging-log4j2,apache__logging-log4j2,noise_00__seed_01,0,1,LatestFail,111,39,22156,40,0.048835,0.044033,0.032709,0.025578
1,22,apache@logging-log4j2,apache__logging-log4j2,noise_00__seed_01,0,1,LightGBM,111,39,22156,40,0.922421,0.999024,0.940615,0.999175
2,22,apache@logging-log4j2,apache__logging-log4j2,noise_00__seed_01,0,1,NaiveBayes,111,39,22156,40,0.883923,0.969066,0.834451,0.964580
3,22,apache@logging-log4j2,apache__logging-log4j2,noise_00__seed_01,0,1,QTF-Avg,111,39,22156,40,0.686528,0.703948,0.149534,0.131188
4,22,apache@logging-log4j2,apache__logging-log4j2,noise_00__seed_01,0,1,Random,111,39,22156,40,0.504862,0.512803,0.501200,0.485227
5,22,apache@logging-log4j2,apache__logging-log4j2,noise_00__seed_01,0,1,RandomForest,111,39,22156,40,0.941984,0.999026,0.941776,0.999175
6,22,apache@logging-log4j2,apache__logging-log4j2,noise_00__seed_01,0,1,XGBoost,111,39,22156,40,0.913470,0.999024,0.960992,0.999175
7,22,apache@logging-log4j2,apache__logging-log4j2,noise_50__seed_01,50,1,LatestFail,111,39,22156,40,0.907682,0.985977,0.911198,0.989773
8,22,apache@logging-log4j2,apache__logging-log4j2,noise_50__seed_01,50,1,LightGBM,111,39,22156,40,0.342519,0.297689,0.383743,0.413991
9,22,apache@logging-log4j2,apache__logging-log4j2,noise_50__seed_01,50,1,NaiveBayes,111,39,22156,40,0.089618,0.040149,0.124158,0.031863



=== PROJECT 22 CELL 8 / STEP 4B RESULT ===

Project:
apache@logging-log4j2
Project 11 identity: apache@shardingsphere
Project 12 identity: zolyfarkas@spf4j
Project 13 identity: jcabi@jcabi-github
Project 14 identity: JMRI@JMRI
Project 15 identity: eclipse@steady
Project 16 identity: apache@rocketmq
Project 17 identity: yamcs@Yamcs
Project 18 identity: cantaloupe-project@cantaloupe
Project 19 identity: EMResearch@EvoMaster
Project 20 identity: apache@curator
Project 21 identity: facebook@buck
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']

Two-condition end-to-end smoke test:
Conditions: ['noise_00__seed_01', 'noise_50__seed_01']
Conditions passed: 2 / 2
ML fits: 8 / 8
Ranking rows: 310184
Build-metric rows: 546
Project-run rows: 14
Training-median rows: 302

Noise and REC audit:
0% raw flips: 0
0% model-label changes: 0
0% dependent REC changes: 0
50% raw flips: 86584
50

In [1]:
# ==================================================================================================
# PROJECT 22 — CELL 9 / STEP 5A PARALLEL MASTER FINALIZATION
# ZERO-FIT REVALIDATION, AGGREGATION, RAW-ROOT FREEZE, AND OFFICIAL CHECKPOINT
#
# PROJECT:
#   apache@logging-log4j2
#
# RUN THIS AS THE NEXT NEW CELL IN THE RECONNECTED MASTER Thesis_project_22.ipynb NOTEBOOK.
#
# PURPOSE:
# - verify all six frozen Step 5A worker checkpoints and output manifests;
# - prove worker seed shards are disjoint and cover seeds 1–30 exactly;
# - independently revalidate all 270 completed condition directories without fitting any model;
# - rebuild the official aggregate outputs from the frozen raw condition files;
# - hash and freeze the complete 2,160-file Project 22 raw-result root;
# - create the official Project 22 Step 5A report/status/checkpoint for Step 5B.
#
# SAFETY:
# - ZERO model fitting and ZERO noise/REC condition execution in this cell;
# - no registry write and no modification of Projects 1–21;
# - no prior-project condition-output access;
# - all 270 condition directories are read-only during finalization;
# - official Step 5A outputs are written only after every validation passes.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import re
import tarfile
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from google.colab import drive

print("=" * 136)
print("=== PROJECT 22 CELL 9 / STEP 5A: PARALLEL MASTER FINALIZATION ===")
print("=" * 136)

# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 22
PROJECT_NAME = "apache@logging-log4j2"
PROJECT_SLUG = "apache__logging-log4j2"
PROJECT_SHORT = "LOG4J2"

EXPECTED_SELECTION_STATUS = "PASS_PROJECT_22_SELECTION_AND_SOURCE_FROZEN"
EXPECTED_STEP2B_STATUS = "PASS_PROJECT_22_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
EXPECTED_STEP3A_STATUS = "PASS_PROJECT_22_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
EXPECTED_STEP4A_STATUS = "PASS_PROJECT_22_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
EXPECTED_STEP4B_STATUS = "PASS_PROJECT_22_TWO_CONDITION_END_TO_END_SMOKE_TEST"
STEP5A_STATUS = "PASS_PROJECT_22_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
CONDITION_STATUS = "PASS_FULL_CONDITION"
MASTER_FINALIZER_IMPLEMENTATION = "PROJECT_22_PARALLEL_STEP5A_MASTER_FINALIZER_V1_ZERO_FIT"

EXPECTED_RUNTIME_CHECKPOINT_SHA256 = "3ef12ada698780da80e05fd6899b0fd98fc5dcc4f3e74c87ec8ea326e3f4f593"
EXPECTED_SMOKE_CHECKPOINT_SHA256 = "91b912640da6dc4b661fabd35632b5fa001ce164bc4857f2f6e139550925e00d"
EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = "3dd2b3f38a8d7b77c179e9f518399c1dc5b1c89a8a29607ea31fd78a91f40666"
EXPECTED_REC_CHECKPOINT_SHA256 = "8417249bc74e2a50e2f61cc3b7776dab7f731f62993e3ed15e415745be178731"
EXPECTED_SELECTION_CHECKPOINT_SHA256 = "a7387d9495c71dce5d2c21ff08b0afd80d9c5251b5885a82e4052c7506ee7890"
EXPECTED_SOURCE_ROOT_SHA256 = "281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64f842964c896c6ac334"
EXPECTED_REGISTRY_SHA256 = "79cd6ecb595c5e8ae91a9494e469792716338d144308560a62caf1b9342306b2"

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 21
EXPECTED_ACTIVE_RESERVATIONS = []
EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_WORKER_CHECKPOINTS = {
    "seed_01_05": {
        "Status": "PASS_PROJECT_22_STEP5A_WORKER_SEEDS_01_05_COMPLETE",
        "Seeds": list(range(1, 6)),
        "SHA256": "6edc0ecf1d33c3fd2263f968f7d953726e3b0ddf3a0a2b40b2cc34ecbb15bb44",
        "RawRootSHA256": "144501c9ae9d261d310e17bb0e5c3c86e596813a3c3d39b0e0354e6ca744f25e",
        "RawBytes": 107_998_067,
    },
    "seed_06_10": {
        "Status": "PASS_PROJECT_22_STEP5A_WORKER_SEEDS_06_10_COMPLETE",
        "Seeds": list(range(6, 11)),
        "SHA256": "1a8ebadb53e01acd4408ac58c8235b4c6504bee2a6bb0213966563a4243aa0c8",
        "RawRootSHA256": "dd718cad6981f126505d70415f57ff7caa8f6025f88e6c6af1fafb270ebab85c",
        "RawBytes": 107_584_568,
    },
    "seed_11_15": {
        "Status": "PASS_PROJECT_22_STEP5A_WORKER_SEEDS_11_15_COMPLETE",
        "Seeds": list(range(11, 16)),
        "SHA256": "f02df169293ce4d1ae143552547a89654ce21e1084b641b4b3ce48ce9afbbc05",
        "RawRootSHA256": "a23ab53f39cfb431e3b34ff1996cfff9d35ab297858cdc69fd53c143cd392e30",
        "RawBytes": 107_570_605,
    },
    "seed_16_20": {
        "Status": "PASS_PROJECT_22_STEP5A_WORKER_SEEDS_16_20_COMPLETE",
        "Seeds": list(range(16, 21)),
        "SHA256": "0643261cc0bd54cadba8354fa88e726ab2e199b940aa2b3f609a581121663db4",
        "RawRootSHA256": "e35458b61a8f803fd28bceae2bcb180150fde655cea38aae9fb42a2d7395b07f",
        "RawBytes": 107_709_615,
    },
    "seed_21_25": {
        "Status": "PASS_PROJECT_22_STEP5A_WORKER_SEEDS_21_25_COMPLETE",
        "Seeds": list(range(21, 26)),
        "SHA256": "23806c7c5252cb4f4b5f6f5faee559678b8c6d02a023437405f7551eaa6c3718",
        "RawRootSHA256": "9fa32797786104d0b6eddbc5190dfe784cbe464f257b4a8e1c0b21e1f27a0f33",
        "RawBytes": 107_713_150,
    },
    "seed_26_30": {
        "Status": "PASS_PROJECT_22_STEP5A_WORKER_SEEDS_26_30_COMPLETE",
        "Seeds": list(range(26, 31)),
        "SHA256": "f7e03bb7c57f29782b057554bbe8116a25e14cce7336361154e7c60e3a0af173",
        "RawRootSHA256": "1ca645ced25f816336359993d10e664ecccbcd4b88db3baa5b90778556ab8c5e",
        "RawBytes": 107_554_648,
    },
}

EXPECTED_WORKER_CONDITIONS = 45
EXPECTED_WORKER_MODEL_FITS = 180
EXPECTED_WORKER_RAW_FILES = 360
EXPECTED_WORKER_RANKING_ROWS = 6_979_140
EXPECTED_WORKER_BUILD_METRIC_ROWS = 12_285
EXPECTED_WORKER_PROJECT_RUN_ROWS = 315
EXPECTED_WORKER_TRAINING_MEDIAN_ROWS = 6_795

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 101_286_750
EXPECTED_RAW_TRAIN_ROWS = 172_628
EXPECTED_RAW_EVAL_ROWS = 67_625
EXPECTED_MODEL_TRAIN_ROWS = 95_812
EXPECTED_MODEL_EVAL_ROWS = 22_156
EXPECTED_MODEL_TRAIN_FAILURES = 207
EXPECTED_MODEL_EVAL_FAILURES = 40
EXPECTED_FAILING_EVAL_BUILDS = 39
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_EVALUATION_BUILDS = 111
EXPECTED_TECHNIQUES = 7
EXPECTED_ML_TECHNIQUES = 4
EXPECTED_CONDITIONS = 270
EXPECTED_FILES_PER_CONDITION = 8
EXPECTED_RAW_FILES = 2_160
EXPECTED_RAW_BYTES = 646_130_653
EXPECTED_RNG_ROWS = 5_178_840

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
REPETITION_SEEDS = list(range(1, 31))
ML_TECHNIQUES = ["RandomForest", "XGBoost", "LightGBM", "NaiveBayes"]
BASELINE_TECHNIQUES = ["Random", "LatestFail", "QTF-Avg"]
ALL_TECHNIQUES = ML_TECHNIQUES + BASELINE_TECHNIQUES
INVARIANT_BASELINES = ["Random", "QTF-Avg"]
ACCELERATED_ENGINE_VERSION = "PROJECT_22_FAST_DEPENDENT_REC_V3_SMOKE_EQUIVALENT_FROZEN_INFERRED_TEST_ORDER_ANCHOR_AWARE"
SMOKE_EQUIVALENCE_KEYS = ["noise_00__seed_01", "noise_50__seed_01"]

EXPECTED_RANKING_ROWS_PER_CONDITION = EXPECTED_MODEL_EVAL_ROWS * EXPECTED_TECHNIQUES
EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = EXPECTED_FAILING_EVAL_BUILDS * EXPECTED_TECHNIQUES
EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = EXPECTED_TECHNIQUES
EXPECTED_MODEL_FIT_ROWS_PER_CONDITION = EXPECTED_ML_TECHNIQUES
EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION = EXPECTED_PREDICTORS

EXPECTED_TOTAL_RANKING_ROWS = EXPECTED_RANKING_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
EXPECTED_TOTAL_BUILD_METRIC_ROWS = EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
EXPECTED_TOTAL_PROJECT_RUN_ROWS = EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
EXPECTED_TOTAL_MODEL_FITS = EXPECTED_MODEL_FIT_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS = EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
EXPECTED_TOTAL_CONDITION_AUDIT_ROWS = EXPECTED_CONDITIONS

# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES_ROOT = THESIS_ROOT / "Notes"
RESULTS_ROOT = THESIS_ROOT / "Results"
REGISTRY_PATH = NOTES_ROOT / "completed_project_registry.csv"
ARCHIVE_PATH = THESIS_ROOT / "Data" / "Raw" / "TCP-CI-main-dataset.tar.gz"
SOURCE_DIR = Path("/content/datasets/datasets/apache@logging-log4j2")

SELECTION_ROOT = RESULTS_ROOT / "Aggregated" / "project_22_selection"
FROZEN_SOURCE_MANIFEST_PATH = SELECTION_ROOT / "project_22_frozen_source_manifest.csv"
SELECTION_CHECKPOINT_PATH = NOTES_ROOT / "project_22_selection_checkpoint.json"
STEP1B_STATUS_PATH = SELECTION_ROOT / "project_22_step1b_status.json"

PROJECT_ROOT = RESULTS_ROOT / "Aggregated" / PROJECT_SLUG
REC_PREFLIGHT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_rec_preflight"
STEP2B_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step2b_status.json"
REC_CHECKPOINT_PATH = NOTES_ROOT / "project_22_rec_reconstruction_checkpoint.json"

NOISE_PLAN_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_noise_plan"
RAW_TRAINING_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
RAW_EVALUATION_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
MODEL_TRAINING_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
MODEL_EVALUATION_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
RNG_MANIFEST_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_rng_manifest.parquet"
CONDITION_PLAN_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_condition_plan.csv"
STEP3A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step3a_status.json"
NOISE_PLAN_CHECKPOINT_PATH = NOTES_ROOT / "project_22_noise_plan_checkpoint.json"

RUNTIME_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_runtime_contract"
STEP4A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step4a_status.json"
STEP4A_REPORT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_step4a_report.json"
RUNTIME_CHECKPOINT_PATH = NOTES_ROOT / "project_22_runtime_contract_checkpoint.json"

SMOKE_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_smoke_test"
STEP4B_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step4b_status.json"
SMOKE_CHECKPOINT_PATH = NOTES_ROOT / "project_22_smoke_test_checkpoint.json"

FULL_RAW_RESULT_ROOT = RESULTS_ROOT / "Raw" / PROJECT_SLUG
FULL_EXPERIMENT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_full_experiment"
CONDITION_INVENTORY_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_condition_inventory.csv"
RAW_MANIFEST_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_raw_manifest.csv"
BASELINE_INVARIANCE_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_baseline_invariance.csv"
COMBINED_CONDITION_AUDIT_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_condition_audit.csv"
COMBINED_PROJECT_RUNS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_project_runs.csv"
COMBINED_BUILD_METRICS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_build_metrics.csv"
COMBINED_MODEL_FITS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_model_fits.csv"
WORKER_CHECKPOINT_AUDIT_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_worker_checkpoint_audit.csv"
STEP5A_VALIDATION_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_validation.csv"
STEP5A_REPORT_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_report.json"
RUN_PROGRESS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_run_progress.json"
STEP5A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5a_status.json"
STEP5A_CHECKPOINT_PATH = NOTES_ROOT / "project_22_step5a_checkpoint.json"
ACCELERATED_EQUIVALENCE_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_accelerated_engine_equivalence.csv"

WORKER_CHECKPOINT_PATHS = {
    tag: NOTES_ROOT / f"project_22_step5a_worker_{tag}_checkpoint.json"
    for tag in EXPECTED_WORKER_CHECKPOINTS
}

# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def atomic_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, sort_keys=True, ensure_ascii=False, default=str)
        f.write("\n")
    os.replace(tmp, path)


def atomic_csv(path, df):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    df.to_csv(tmp, index=False, lineterminator="\n")
    os.replace(tmp, path)


def source_root_hash(frame):
    h = hashlib.sha256()
    for r in frame.sort_values("RelativePath", kind="mergesort").itertuples(index=False):
        size = getattr(r, "SizeBytes", getattr(r, "Bytes", None))
        h.update(f"{r.RelativePath}\0{int(size)}\0{str(r.SHA256).lower()}\n".encode("utf-8"))
    return h.hexdigest()


def directory_manifest(root):
    root = Path(root)
    if not root.exists():
        return pd.DataFrame(columns=["RelativePath", "Bytes", "SHA256"])
    records = []
    for path in sorted((p for p in root.rglob("*") if p.is_file()), key=lambda p: p.as_posix()):
        records.append({
            "RelativePath": path.relative_to(root).as_posix(),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        })
    return pd.DataFrame(records, columns=["RelativePath", "Bytes", "SHA256"])


def directory_root_hash(manifest):
    h = hashlib.sha256()
    for r in manifest.sort_values("RelativePath", kind="mergesort").itertuples(index=False):
        h.update(f"{r.RelativePath}\0{int(r.Bytes)}\0{str(r.SHA256).lower()}\n".encode("utf-8"))
    return h.hexdigest()


def add_check(rows, check, expected, actual, passed):
    rows.append({"Check": check, "Expected": expected, "Actual": actual, "Pass": bool(passed)})


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def verify_manifest_entries(entries, label):
    failures = []
    if not isinstance(entries, list) or not entries:
        return [{"Label": label, "Path": "<manifest>", "Reason": "missing/empty manifest"}]
    for item in entries:
        path = Path(str(item.get("Path", "")))
        if not path.is_file():
            failures.append({"Label": label, "Path": str(path), "Reason": "missing"})
            continue
        expected_bytes = int(item.get("Bytes", -1))
        expected_sha = str(item.get("SHA256", "")).lower()
        actual_bytes = int(path.stat().st_size)
        actual_sha = sha256_file(path)
        if actual_bytes != expected_bytes or actual_sha != expected_sha:
            failures.append({"Label": label, "Path": str(path), "Reason": "size/hash mismatch"})
    return failures


def resolve_registry_columns(registry):
    lookup = {str(c).strip().lower(): c for c in registry.columns}
    def pick(names, label):
        for name in names:
            if name in lookup:
                return lookup[name]
        raise RuntimeError(f"Could not resolve registry {label}; columns={list(registry.columns)}")
    return (
        pick(["projectnumber", "project_number", "project no", "projectno"], "ProjectNumber"),
        pick(["project", "projectname", "project_name"], "Project"),
        pick(["status", "projectstatus", "project_status"], "Status"),
    )


def restore_source_if_needed():
    if SOURCE_DIR.is_dir():
        return False
    if not ARCHIVE_PATH.is_file():
        raise FileNotFoundError(f"Project 22 source is absent and archive is missing: {ARCHIVE_PATH}")
    target_root = Path("/content/datasets")
    target_root.mkdir(parents=True, exist_ok=True)
    prefix = "datasets/apache@logging-log4j2/"
    print("\nRestoring only the frozen Project 22 source from the thesis archive...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        members = [m for m in tar.getmembers() if m.name.startswith(prefix)]
        if not members:
            raise RuntimeError(f"Archive contains no members under {prefix}")
        root_resolved = target_root.resolve()
        for member in members:
            destination = (target_root / member.name).resolve()
            if root_resolved not in destination.parents and destination != root_resolved:
                raise RuntimeError(f"Unsafe archive member: {member.name}")
        tar.extractall(target_root, members=members)
    if not SOURCE_DIR.is_dir():
        raise RuntimeError("Selective Project 22 source restoration failed.")
    return True


EXPECTED_CONDITION_FILES = {
    "rankings.csv.gz",
    "build_metrics.csv",
    "project_runs.csv",
    "model_fits.csv",
    "training_medians.csv",
    "condition_audit.csv",
    "condition_summary.json",
    "COMPLETE.json",
}


def validate_completed_condition(condition_dir, plan_row):
    condition_dir = Path(condition_dir)
    condition_key = str(plan_row.ConditionID)
    if not condition_dir.is_dir():
        return None
    actual_files = {p.name for p in condition_dir.iterdir() if p.is_file()}
    if actual_files != EXPECTED_CONDITION_FILES:
        return None
    completion_path = condition_dir / "COMPLETE.json"
    summary_path = condition_dir / "condition_summary.json"
    try:
        completion = load_json(completion_path)
        summary = load_json(summary_path)
    except Exception:
        return None
    if completion.get("Status") != CONDITION_STATUS or summary.get("Status") != CONDITION_STATUS:
        return None
    if completion.get("ConditionKey") != condition_key or summary.get("ConditionKey") != condition_key:
        return None
    if int(summary.get("NoisePercent", -1)) != int(plan_row.NoisePercent):
        return None
    if int(summary.get("RepetitionSeed", -1)) != int(plan_row.RepetitionSeed):
        return None
    if int(summary.get("ConditionOrder", -1)) != int(plan_row.ConditionOrder):
        return None
    if summary.get("Project") != PROJECT_NAME or summary.get("ProjectSlug") != PROJECT_SLUG:
        return None
    if summary.get("AcceleratedEngineVersion") != ACCELERATED_ENGINE_VERSION:
        return None
    if str(completion.get("ConditionSummaryPath")) != str(summary_path):
        return None
    if str(completion.get("ConditionSummarySHA256")) != sha256_file(summary_path):
        return None
    output_manifest = summary.get("OutputManifest", [])
    if not isinstance(output_manifest, list) or len(output_manifest) != 6:
        return None
    for item in output_manifest:
        path = Path(str(item.get("Path", "")))
        if path.parent != condition_dir or not path.is_file():
            return None
        if int(path.stat().st_size) != int(item.get("Bytes", -1)):
            return None
        if sha256_file(path) != str(item.get("SHA256", "")):
            return None
    expected_counts = {
        "MLFits": EXPECTED_MODEL_FIT_ROWS_PER_CONDITION,
        "RankingRows": EXPECTED_RANKING_ROWS_PER_CONDITION,
        "BuildMetricRows": EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
        "ProjectRunRows": EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
        "TrainingMedianRows": EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION,
    }
    for key, expected in expected_counts.items():
        if int(summary.get(key, -1)) != expected:
            return None
    if int(summary.get("IndependentRECChanges", -1)) != 0:
        return None
    fingerprints = summary.get("BaselineFingerprints", {})
    if sorted(fingerprints.keys()) != ["QTF-Avg", "Random"]:
        return None
    manifest = directory_manifest(condition_dir)
    if len(manifest) != EXPECTED_FILES_PER_CONDITION:
        return None
    return {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": int(plan_row.ConditionOrder),
        "ConditionKey": condition_key,
        "NoisePercent": int(plan_row.NoisePercent),
        "RepetitionSeed": int(plan_row.RepetitionSeed),
        "ConditionDirectory": str(condition_dir),
        "Status": CONDITION_STATUS,
        "Files": int(len(manifest)),
        "ConditionBytes": int(manifest["Bytes"].sum()),
        "ConditionRootSHA256": directory_root_hash(manifest),
        "RankingRows": int(summary["RankingRows"]),
        "BuildMetricRows": int(summary["BuildMetricRows"]),
        "ProjectRunRows": int(summary["ProjectRunRows"]),
        "MLFits": int(summary["MLFits"]),
        "TrainingMedianRows": int(summary["TrainingMedianRows"]),
        "NumberFlipped": int(summary.get("NumberFlipped", -1)),
        "ModelLabelChanges": int(summary.get("ModelLabelChanges", -1)),
        "DependentRECChanges": int(summary.get("DependentRECChanges", -1)),
        "IndependentRECChanges": int(summary.get("IndependentRECChanges", -1)),
        "TrainingFailures": int(summary.get("TrainingFailures", -1)),
        "ConditionSeconds": float(summary.get("ConditionSeconds", np.nan)),
        "BaselineFingerprints": fingerprints,
    }

# --------------------------------------------------------------------------------------------------
# 4. MOUNT, RESTORE SELECTED SOURCE IF NEEDED, AND VERIFY UPSTREAM FREEZES
# --------------------------------------------------------------------------------------------------

started = time.perf_counter()
drive.mount("/content/drive", force_remount=False)
source_restored = restore_source_if_needed()

required_paths = [
    REGISTRY_PATH, FROZEN_SOURCE_MANIFEST_PATH, SELECTION_CHECKPOINT_PATH, STEP1B_STATUS_PATH,
    STEP2B_STATUS_PATH, REC_CHECKPOINT_PATH, NOISE_PLAN_CHECKPOINT_PATH, STEP3A_STATUS_PATH,
    RUNTIME_CHECKPOINT_PATH, STEP4A_STATUS_PATH, STEP4A_REPORT_PATH, SMOKE_CHECKPOINT_PATH,
    STEP4B_STATUS_PATH, CONDITION_PLAN_PATH, RAW_TRAINING_COHORT_PATH, RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH, MODEL_EVALUATION_COHORT_PATH, RNG_MANIFEST_PATH,
    ACCELERATED_EQUIVALENCE_PATH, FULL_RAW_RESULT_ROOT,
] + list(WORKER_CHECKPOINT_PATHS.values())
missing = [str(p) for p in required_paths if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing Project 22 master-finalization inputs:\n" + "\n".join(missing))

sha_expectations = {
    "selection": (SELECTION_CHECKPOINT_PATH, EXPECTED_SELECTION_CHECKPOINT_SHA256),
    "REC": (REC_CHECKPOINT_PATH, EXPECTED_REC_CHECKPOINT_SHA256),
    "noise plan": (NOISE_PLAN_CHECKPOINT_PATH, EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256),
    "runtime": (RUNTIME_CHECKPOINT_PATH, EXPECTED_RUNTIME_CHECKPOINT_SHA256),
    "smoke": (SMOKE_CHECKPOINT_PATH, EXPECTED_SMOKE_CHECKPOINT_SHA256),
}
for label, (path, expected) in sha_expectations.items():
    actual = sha256_file(path)
    if actual != expected:
        raise RuntimeError(f"Frozen Project 22 {label} checkpoint SHA-256 differs.\nExpected: {expected}\nActual:   {actual}")

selection_checkpoint = load_json(SELECTION_CHECKPOINT_PATH)
step1b_status = load_json(STEP1B_STATUS_PATH)
rec_checkpoint = load_json(REC_CHECKPOINT_PATH)
step2b_status = load_json(STEP2B_STATUS_PATH)
noise_checkpoint = load_json(NOISE_PLAN_CHECKPOINT_PATH)
step3a_status = load_json(STEP3A_STATUS_PATH)
runtime_checkpoint = load_json(RUNTIME_CHECKPOINT_PATH)
step4a_status = load_json(STEP4A_STATUS_PATH)
step4a_report = load_json(STEP4A_REPORT_PATH)
smoke_checkpoint = load_json(SMOKE_CHECKPOINT_PATH)
step4b_status = load_json(STEP4B_STATUS_PATH)

status_pairs = [
    ("selection checkpoint", selection_checkpoint.get("Status"), EXPECTED_SELECTION_STATUS),
    ("Step 1B status", step1b_status.get("Status"), EXPECTED_SELECTION_STATUS),
    ("REC checkpoint", rec_checkpoint.get("Status"), EXPECTED_STEP2B_STATUS),
    ("Step 2B status", step2b_status.get("Status"), EXPECTED_STEP2B_STATUS),
    ("noise checkpoint", noise_checkpoint.get("Status"), EXPECTED_STEP3A_STATUS),
    ("Step 3A status", step3a_status.get("Status"), EXPECTED_STEP3A_STATUS),
    ("runtime checkpoint", runtime_checkpoint.get("Status"), EXPECTED_STEP4A_STATUS),
    ("Step 4A status", step4a_status.get("Status"), EXPECTED_STEP4A_STATUS),
    ("Step 4A report", step4a_report.get("Status"), EXPECTED_STEP4A_STATUS),
    ("smoke checkpoint", smoke_checkpoint.get("Status"), EXPECTED_STEP4B_STATUS),
    ("Step 4B status", step4b_status.get("Status"), EXPECTED_STEP4B_STATUS),
]
for label, actual, expected in status_pairs:
    if actual != expected:
        raise RuntimeError(f"{label} differs: expected {expected!r}, actual {actual!r}")
if not bool(smoke_checkpoint.get("ReadyForFull270ConditionExperiment", False)):
    raise RuntimeError("Frozen Step 4B checkpoint does not authorize the full experiment.")

manifest_failures = []
manifest_failures += verify_manifest_entries(runtime_checkpoint.get("RuntimeOutputManifest", []), "runtime")
manifest_failures += verify_manifest_entries(smoke_checkpoint.get("OutputManifest", []), "smoke")
if manifest_failures:
    display(pd.DataFrame(manifest_failures))
    raise RuntimeError("One or more frozen upstream output manifests no longer validate.")

# Registry immutability and identities.
registry_sha_before = sha256_file(REGISTRY_PATH)
if registry_sha_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError("Completion registry SHA-256 differs before Project 22 Step 5A finalization.")
registry = pd.read_csv(REGISTRY_PATH, low_memory=False)
pn_col, project_col, status_col = resolve_registry_columns(registry)
project_numbers = pd.to_numeric(registry[pn_col], errors="raise").astype(int)
if len(registry) != EXPECTED_REGISTERED_PROJECTS or sorted(project_numbers.tolist()) != list(range(1, 22)):
    raise RuntimeError("Completion registry does not contain exactly Projects 1–21.")
if not registry[status_col].astype(str).eq(EXPECTED_COMPLETE_STATUS).all():
    raise RuntimeError("Projects 1–21 are not all COMPLETE_AND_FROZEN.")
required_identities = {
    11: "apache@shardingsphere", 12: "zolyfarkas@spf4j", 13: "jcabi@jcabi-github",
    14: "JMRI@JMRI", 15: "eclipse@steady", 16: "apache@rocketmq", 17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe", 19: "EMResearch@EvoMaster", 20: "apache@curator",
    21: "facebook@buck",
}
for number, identity in required_identities.items():
    rows = registry.loc[project_numbers.eq(number)]
    if len(rows) != 1 or str(rows.iloc[0][project_col]) != identity:
        raise RuntimeError(f"Frozen predecessor identity differs for Project {number}: expected {identity}")
if project_numbers.eq(PROJECT_NUMBER).any() or registry[project_col].astype(str).eq(PROJECT_NAME).any():
    raise RuntimeError("Project 22 is already present in the completion registry.")

# Source immutability.
frozen_source_manifest = pd.read_csv(FROZEN_SOURCE_MANIFEST_PATH, low_memory=False)
current_source = []
for row in frozen_source_manifest.itertuples(index=False):
    path = SOURCE_DIR / str(row.RelativePath)
    if not path.is_file():
        raise FileNotFoundError(f"Frozen Project 22 source file is missing: {path}")
    current_source.append({"RelativePath": str(row.RelativePath), "SizeBytes": int(path.stat().st_size), "SHA256": sha256_file(path)})
current_source = pd.DataFrame(current_source)
source_sha_before = source_root_hash(current_source)
if source_sha_before != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError("Frozen Project 22 source root differs before finalization.")
if len(current_source) != EXPECTED_SOURCE_FILES or int(current_source["SizeBytes"].sum()) != EXPECTED_SOURCE_BYTES:
    raise RuntimeError("Frozen Project 22 source file/byte counts differ.")

# Record cohort hashes before any official output write.
cohort_paths = [RAW_TRAINING_COHORT_PATH, RAW_EVALUATION_COHORT_PATH, MODEL_TRAINING_COHORT_PATH, MODEL_EVALUATION_COHORT_PATH, RNG_MANIFEST_PATH, CONDITION_PLAN_PATH]
cohort_sha_before = {str(p): sha256_file(p) for p in cohort_paths}
if pq.ParquetFile(RAW_TRAINING_COHORT_PATH).metadata.num_rows != EXPECTED_RAW_TRAIN_ROWS:
    raise RuntimeError("Frozen raw training cohort row count differs.")
if pq.ParquetFile(RAW_EVALUATION_COHORT_PATH).metadata.num_rows != EXPECTED_RAW_EVAL_ROWS:
    raise RuntimeError("Frozen raw evaluation cohort row count differs.")
if pq.ParquetFile(MODEL_TRAINING_COHORT_PATH).metadata.num_rows != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Frozen model training cohort row count differs.")
if pq.ParquetFile(MODEL_EVALUATION_COHORT_PATH).metadata.num_rows != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Frozen model evaluation cohort row count differs.")
if pq.ParquetFile(RNG_MANIFEST_PATH).metadata.num_rows != EXPECTED_RNG_ROWS:
    raise RuntimeError("Frozen RNG manifest row count differs.")

# --------------------------------------------------------------------------------------------------
# 5. VALIDATE FROZEN CONDITION PLAN
# --------------------------------------------------------------------------------------------------

condition_plan = pd.read_csv(CONDITION_PLAN_PATH, low_memory=False)
required_plan_cols = {"ConditionOrder", "ConditionID", "NoisePercent", "RepetitionSeed"}
if not required_plan_cols.issubset(condition_plan.columns):
    raise RuntimeError(f"Condition plan missing required columns: {sorted(required_plan_cols - set(condition_plan.columns))}")
condition_plan = condition_plan.sort_values("ConditionOrder", kind="mergesort").reset_index(drop=True)
expected_grid = [(seed, noise, f"noise_{noise:02d}__seed_{seed:02d}") for seed in REPETITION_SEEDS for noise in NOISE_LEVELS]
actual_grid = [(int(r.RepetitionSeed), int(r.NoisePercent), str(r.ConditionID)) for r in condition_plan.itertuples(index=False)]
if len(condition_plan) != EXPECTED_CONDITIONS or actual_grid != expected_grid:
    raise RuntimeError("Frozen Project 22 condition grid/order differs from the thesis contract.")
if condition_plan["ConditionID"].duplicated().any() or condition_plan.duplicated(["NoisePercent", "RepetitionSeed"]).any():
    raise RuntimeError("Frozen condition plan contains duplicate coordinates.")

# --------------------------------------------------------------------------------------------------
# 6. VERIFY ALL SIX WORKER CHECKPOINTS AND SHARD COVERAGE
# --------------------------------------------------------------------------------------------------

worker_records = []
worker_seed_union = []
worker_manifest_failures = []
for tag, expected in EXPECTED_WORKER_CHECKPOINTS.items():
    path = WORKER_CHECKPOINT_PATHS[tag]
    actual_sha = sha256_file(path)
    if actual_sha != expected["SHA256"]:
        raise RuntimeError(f"Worker {tag} checkpoint SHA differs. Expected {expected['SHA256']}; actual {actual_sha}")
    cp = load_json(path)
    if cp.get("Project") != PROJECT_NAME or cp.get("ProjectSlug") != PROJECT_SLUG or int(cp.get("ProjectNumber", -1)) != PROJECT_NUMBER:
        raise RuntimeError(f"Worker {tag} project identity differs.")
    if cp.get("Status") != expected["Status"]:
        raise RuntimeError(f"Worker {tag} PASS status differs.")
    seeds = [int(x) for x in cp.get("AssignedSeeds", [])]
    if seeds != expected["Seeds"]:
        raise RuntimeError(f"Worker {tag} assigned seeds differ: {seeds}")
    worker_seed_union.extend(seeds)
    scalar_checks = {
        "AssignedConditions": EXPECTED_WORKER_CONDITIONS,
        "CompletedConditions": EXPECTED_WORKER_CONDITIONS,
        "MLFits": EXPECTED_WORKER_MODEL_FITS,
        "RankingRows": EXPECTED_WORKER_RANKING_ROWS,
        "BuildMetricRows": EXPECTED_WORKER_BUILD_METRIC_ROWS,
        "ProjectRunRows": EXPECTED_WORKER_PROJECT_RUN_ROWS,
        "TrainingMedianRows": EXPECTED_WORKER_TRAINING_MEDIAN_ROWS,
        "WorkerRawFiles": EXPECTED_WORKER_RAW_FILES,
        "WorkerRawBytes": expected["RawBytes"],
    }
    for key, value in scalar_checks.items():
        if int(cp.get(key, -1)) != int(value):
            raise RuntimeError(f"Worker {tag} {key} differs: expected {value}; actual {cp.get(key)}")
    if str(cp.get("WorkerRawRootSHA256", "")) != expected["RawRootSHA256"]:
        raise RuntimeError(f"Worker {tag} raw-root hash differs.")
    if not bool(cp.get("WorkerShardComplete", False)) or bool(cp.get("OfficialStep5AComplete", True)):
        raise RuntimeError(f"Worker {tag} shard-completion/master-state flags differ.")
    if not bool(cp.get("MasterFinalizationRequired", False)):
        raise RuntimeError(f"Worker {tag} does not require master finalization.")
    if int(cp.get("BaselineInvarianceFailures", -1)) != 0 or int(cp.get("FailedValidationChecks", -1)) != 0:
        raise RuntimeError(f"Worker {tag} reports validation/baseline failures.")
    if int(cp.get("MasterStep5AWriteGuardFailures", -1)) != 0:
        raise RuntimeError(f"Worker {tag} reports master write-guard failures.")
    if bool(cp.get("RegistryModified", True)) or bool(cp.get("PriorProjectConditionOutputsAccessed", True)) or bool(cp.get("PriorProjectConditionOutputsModified", True)):
        raise RuntimeError(f"Worker {tag} reports an immutability violation.")
    if str(cp.get("SourceRootSHA256", "")) != EXPECTED_SOURCE_ROOT_SHA256 or str(cp.get("RegistrySHA256", "")) != EXPECTED_REGISTRY_SHA256:
        raise RuntimeError(f"Worker {tag} source/registry freeze linkage differs.")
    worker_manifest_failures += verify_manifest_entries(cp.get("WorkerOutputManifest", []), f"worker {tag}")
    report_path = Path(str(cp.get("WorkerReportPath", "")))
    if not report_path.is_file() or sha256_file(report_path) != str(cp.get("WorkerReportSHA256", "")):
        raise RuntimeError(f"Worker {tag} report linkage does not validate.")

    # Independently reconstruct the worker raw root from its frozen private raw manifest.
    raw_manifest_entries = [i for i in cp.get("WorkerOutputManifest", []) if "raw_manifest" in Path(str(i.get("Path", ""))).name]
    if len(raw_manifest_entries) != 1:
        raise RuntimeError(f"Worker {tag} output manifest does not identify exactly one private raw manifest.")
    wm_path = Path(str(raw_manifest_entries[0]["Path"]))
    wm = pd.read_csv(wm_path, low_memory=False)
    if not {"RelativePath", "Bytes", "SHA256"}.issubset(wm.columns):
        raise RuntimeError(f"Worker {tag} raw manifest schema differs.")
    wm = wm[["RelativePath", "Bytes", "SHA256"]].copy()
    wm["Bytes"] = pd.to_numeric(wm["Bytes"], errors="raise").astype("int64")
    if len(wm) != EXPECTED_WORKER_RAW_FILES or int(wm["Bytes"].sum()) != expected["RawBytes"]:
        raise RuntimeError(f"Worker {tag} private raw manifest counts differ.")
    if directory_root_hash(wm) != expected["RawRootSHA256"]:
        raise RuntimeError(f"Worker {tag} private raw manifest does not reproduce its frozen raw-root hash.")

    worker_records.append({
        "WorkerTag": tag,
        "Status": cp.get("Status"),
        "AssignedSeeds": json.dumps(seeds),
        "Conditions": int(cp["CompletedConditions"]),
        "MLFits": int(cp["MLFits"]),
        "RankingRows": int(cp["RankingRows"]),
        "BuildMetricRows": int(cp["BuildMetricRows"]),
        "ProjectRunRows": int(cp["ProjectRunRows"]),
        "TrainingMedianRows": int(cp["TrainingMedianRows"]),
        "RawFiles": int(cp["WorkerRawFiles"]),
        "RawBytes": int(cp["WorkerRawBytes"]),
        "RawRootSHA256": str(cp["WorkerRawRootSHA256"]),
        "CheckpointSHA256": actual_sha,
        "ValidationChecks": int(cp.get("ValidationChecks", 0)),
        "FailedValidationChecks": int(cp.get("FailedValidationChecks", 0)),
    })

if worker_manifest_failures:
    display(pd.DataFrame(worker_manifest_failures))
    raise RuntimeError("One or more worker output manifests no longer validate.")
if sorted(worker_seed_union) != REPETITION_SEEDS or len(worker_seed_union) != len(set(worker_seed_union)):
    raise RuntimeError(f"Worker shards do not cover seeds 1–30 exactly once: {worker_seed_union}")
worker_audit = pd.DataFrame(worker_records)

# --------------------------------------------------------------------------------------------------
# 7. VERIFY SHARED SMOKE-EQUIVALENCE RECORD
# --------------------------------------------------------------------------------------------------

equivalence = pd.read_csv(ACCELERATED_EQUIVALENCE_PATH, low_memory=False)
if "ConditionKey" not in equivalence.columns or "Pass" not in equivalence.columns:
    raise RuntimeError("Accelerated-equivalence file schema differs.")
if sorted(equivalence["ConditionKey"].astype(str).tolist()) != sorted(SMOKE_EQUIVALENCE_KEYS):
    raise RuntimeError("Accelerated-equivalence file does not contain exactly the two frozen smoke conditions.")
if not equivalence["Pass"].map(as_bool).all():
    raise RuntimeError("At least one accelerated-engine smoke-equivalence check failed.")

# --------------------------------------------------------------------------------------------------
# 8. INDEPENDENT READ-ONLY REVALIDATION OF ALL 270 CONDITIONS
# --------------------------------------------------------------------------------------------------

print("\nIndependently revalidating all 270 frozen condition directories (ZERO fitting).")
inventory_records = []
audit_frames = []
project_run_frames = []
build_metric_frames = []
model_fit_frames = []
baseline_records = []

for idx, plan_row in enumerate(condition_plan.itertuples(index=False), start=1):
    condition_dir = FULL_RAW_RESULT_ROOT / str(plan_row.ConditionID)
    validated = validate_completed_condition(condition_dir, plan_row)
    if validated is None:
        raise RuntimeError(f"Condition failed independent readback validation: {plan_row.ConditionID}")
    fingerprints = validated.pop("BaselineFingerprints")
    inventory_records.append(validated)

    audit = pd.read_csv(condition_dir / "condition_audit.csv", low_memory=False)
    project_runs = pd.read_csv(condition_dir / "project_runs.csv", low_memory=False)
    build_metrics = pd.read_csv(condition_dir / "build_metrics.csv", low_memory=False)
    model_fits = pd.read_csv(condition_dir / "model_fits.csv", low_memory=False)
    medians = pd.read_csv(condition_dir / "training_medians.csv", low_memory=False)
    if len(audit) != 1 or len(project_runs) != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION or len(build_metrics) != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION or len(model_fits) != EXPECTED_MODEL_FIT_ROWS_PER_CONDITION or len(medians) != EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION:
        raise RuntimeError(f"Compact output row counts differ for {plan_row.ConditionID}")
    for df, label in [(audit, "audit"), (project_runs, "project runs"), (build_metrics, "build metrics"), (model_fits, "model fits")]:
        if "ConditionKey" in df.columns and not df["ConditionKey"].astype(str).eq(str(plan_row.ConditionID)).all():
            raise RuntimeError(f"{plan_row.ConditionID}: {label} ConditionKey differs.")
    if "Technique" in project_runs.columns and sorted(project_runs["Technique"].astype(str).tolist()) != sorted(ALL_TECHNIQUES):
        raise RuntimeError(f"{plan_row.ConditionID}: project-run technique set differs.")
    if "Technique" in model_fits.columns and sorted(model_fits["Technique"].astype(str).tolist()) != sorted(ML_TECHNIQUES):
        raise RuntimeError(f"{plan_row.ConditionID}: model-fit technique set differs.")

    audit_frames.append(audit)
    project_run_frames.append(project_runs)
    build_metric_frames.append(build_metrics)
    model_fit_frames.append(model_fits)

    for technique in INVARIANT_BASELINES:
        fp = fingerprints[technique]
        baseline_records.append({
            "ConditionKey": str(plan_row.ConditionID),
            "NoisePercent": int(plan_row.NoisePercent),
            "RepetitionSeed": int(plan_row.RepetitionSeed),
            "Technique": technique,
            "FingerprintJSON": json.dumps(fp, sort_keys=True, separators=(",", ":")),
        })

    if idx % 30 == 0 or idx == EXPECTED_CONDITIONS:
        print(f"  Validated {idx}/{EXPECTED_CONDITIONS} conditions")

condition_inventory = pd.DataFrame(inventory_records).sort_values("ConditionOrder", kind="mergesort").reset_index(drop=True)
combined_condition_audit = pd.concat(audit_frames, ignore_index=True)
combined_project_runs = pd.concat(project_run_frames, ignore_index=True)
combined_build_metrics = pd.concat(build_metric_frames, ignore_index=True)
combined_model_fits = pd.concat(model_fit_frames, ignore_index=True)
baseline_detail = pd.DataFrame(baseline_records)

# Noise-invariant baseline fingerprints must be identical across all nine noise levels within each seed.
baseline_invariance = (
    baseline_detail.groupby(["RepetitionSeed", "Technique"], as_index=False)
    .agg(Conditions=("ConditionKey", "size"), FingerprintVariants=("FingerprintJSON", "nunique"))
)
baseline_invariance["ExpectedConditions"] = len(NOISE_LEVELS)
baseline_invariance["Pass"] = (
    baseline_invariance["Conditions"].eq(len(NOISE_LEVELS))
    & baseline_invariance["FingerprintVariants"].eq(1)
)

# --------------------------------------------------------------------------------------------------
# 9. HASH AND FREEZE THE COMPLETE RAW ROOT
# --------------------------------------------------------------------------------------------------

print("\nHashing the complete Project 22 raw-result root (2,160 files).")
raw_manifest = directory_manifest(FULL_RAW_RESULT_ROOT)
raw_root_sha256 = directory_root_hash(raw_manifest)
raw_files = int(len(raw_manifest))
raw_bytes = int(raw_manifest["Bytes"].sum())

# --------------------------------------------------------------------------------------------------
# 10. MASTER VALIDATION
# --------------------------------------------------------------------------------------------------

validation = []
add_check(validation, "Worker checkpoints", 6, len(worker_audit), len(worker_audit) == 6)
add_check(validation, "Worker seed coverage", list(range(1, 31)), sorted(worker_seed_union), sorted(worker_seed_union) == list(range(1, 31)) and len(worker_seed_union) == len(set(worker_seed_union)))
add_check(validation, "Conditions", EXPECTED_CONDITIONS, len(condition_inventory), len(condition_inventory) == EXPECTED_CONDITIONS)
add_check(validation, "Raw files", EXPECTED_RAW_FILES, raw_files, raw_files == EXPECTED_RAW_FILES)
add_check(validation, "Raw bytes", EXPECTED_RAW_BYTES, raw_bytes, raw_bytes == EXPECTED_RAW_BYTES)
add_check(validation, "Ranking rows", EXPECTED_TOTAL_RANKING_ROWS, int(condition_inventory["RankingRows"].sum()), int(condition_inventory["RankingRows"].sum()) == EXPECTED_TOTAL_RANKING_ROWS)
add_check(validation, "Build-metric rows", EXPECTED_TOTAL_BUILD_METRIC_ROWS, len(combined_build_metrics), len(combined_build_metrics) == EXPECTED_TOTAL_BUILD_METRIC_ROWS)
add_check(validation, "Project-run rows", EXPECTED_TOTAL_PROJECT_RUN_ROWS, len(combined_project_runs), len(combined_project_runs) == EXPECTED_TOTAL_PROJECT_RUN_ROWS)
add_check(validation, "Model-fit rows", EXPECTED_TOTAL_MODEL_FITS, len(combined_model_fits), len(combined_model_fits) == EXPECTED_TOTAL_MODEL_FITS)
add_check(validation, "Condition-audit rows", EXPECTED_TOTAL_CONDITION_AUDIT_ROWS, len(combined_condition_audit), len(combined_condition_audit) == EXPECTED_TOTAL_CONDITION_AUDIT_ROWS)
add_check(validation, "Training-median rows", EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS, int(condition_inventory["TrainingMedianRows"].sum()), int(condition_inventory["TrainingMedianRows"].sum()) == EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS)
add_check(validation, "Baseline invariance rows", 60, len(baseline_invariance), len(baseline_invariance) == 60)
add_check(validation, "Baseline invariance failures", 0, int((~baseline_invariance["Pass"]).sum()), int((~baseline_invariance["Pass"]).sum()) == 0)
add_check(validation, "Independent REC changes", 0, int(condition_inventory["IndependentRECChanges"].sum()), int(condition_inventory["IndependentRECChanges"].sum()) == 0)

zero = condition_inventory["NoisePercent"].eq(0)
pos = condition_inventory["NoisePercent"].gt(0)
add_check(validation, "Zero-noise raw flips", 0, int(condition_inventory.loc[zero, "NumberFlipped"].sum()), int(condition_inventory.loc[zero, "NumberFlipped"].sum()) == 0)
add_check(validation, "Zero-noise model-label changes", 0, int(condition_inventory.loc[zero, "ModelLabelChanges"].sum()), int(condition_inventory.loc[zero, "ModelLabelChanges"].sum()) == 0)
add_check(validation, "Zero-noise dependent REC changes", 0, int(condition_inventory.loc[zero, "DependentRECChanges"].sum()), int(condition_inventory.loc[zero, "DependentRECChanges"].sum()) == 0)
add_check(validation, "Positive-noise conditions without raw flips", 0, int(condition_inventory.loc[pos, "NumberFlipped"].le(0).sum()), int(condition_inventory.loc[pos, "NumberFlipped"].le(0).sum()) == 0)
add_check(validation, "Positive-noise conditions without model-label changes", 0, int(condition_inventory.loc[pos, "ModelLabelChanges"].le(0).sum()), int(condition_inventory.loc[pos, "ModelLabelChanges"].le(0).sum()) == 0)

# Audit hash equality and independent reconstruction invariance.
for expected_col, actual_col, label in [
    ("ExpectedFlipMaskSHA256", "ActualFlipMaskSHA256", "Flip-mask hash mismatches"),
    ("ExpectedNoisyRawVerdictSHA256", "ActualNoisyRawVerdictSHA256", "Noisy raw-verdict hash mismatches"),
    ("ExpectedNoisyModelVerdictSHA256", "ActualNoisyModelVerdictSHA256", "Noisy model-verdict hash mismatches"),
]:
    if expected_col not in combined_condition_audit.columns or actual_col not in combined_condition_audit.columns:
        add_check(validation, label, 0, "missing audit columns", False)
    else:
        mismatches = int((combined_condition_audit[expected_col].astype(str) != combined_condition_audit[actual_col].astype(str)).sum())
        add_check(validation, label, 0, mismatches, mismatches == 0)
if "IndependentReconstructionMismatches" in combined_condition_audit.columns:
    value = int(pd.to_numeric(combined_condition_audit["IndependentReconstructionMismatches"], errors="raise").sum())
    add_check(validation, "Independent reconstruction mismatches", 0, value, value == 0)
else:
    add_check(validation, "Independent reconstruction mismatches", 0, "missing column", False)

# Metric sanity: thesis metrics must be finite and within [0,1].
for df, cols, label in [
    (combined_project_runs, ["MeanAPFDc", "MedianAPFDc", "MeanAPFD", "MedianAPFD"], "Project metrics"),
    (combined_build_metrics, ["APFDc", "APFD"], "Build metrics"),
]:
    missing_cols = [c for c in cols if c not in df.columns]
    if missing_cols:
        add_check(validation, f"{label} columns", cols, missing_cols, False)
    else:
        arr = df[cols].apply(pd.to_numeric, errors="coerce").to_numpy(float)
        bad = int((~np.isfinite(arr)).sum() + ((arr < 0) | (arr > 1)).sum())
        add_check(validation, f"{label} non-finite/out-of-range values", 0, bad, bad == 0)

# Model fits must be successful if a Status column exists.
if "Status" in combined_model_fits.columns:
    bad_status = int(~combined_model_fits["Status"].astype(str).str.upper().isin({"PASS", "OK", "SUCCESS", "FITTED"})).sum()
    # Some frozen implementations use a longer status token; accept anything containing PASS/SUCCESS/FIT.
    status_text = combined_model_fits["Status"].astype(str).str.upper()
    bad_status = int((~status_text.str.contains("PASS|SUCCESS|FIT", regex=True)).sum())
    add_check(validation, "Model-fit status failures", 0, bad_status, bad_status == 0)

add_check(validation, "Registry SHA unchanged", EXPECTED_REGISTRY_SHA256, sha256_file(REGISTRY_PATH), sha256_file(REGISTRY_PATH) == EXPECTED_REGISTRY_SHA256)
add_check(validation, "Source root unchanged", EXPECTED_SOURCE_ROOT_SHA256, source_sha_before, source_sha_before == EXPECTED_SOURCE_ROOT_SHA256)
add_check(validation, "Shared smoke-equivalence failures", 0, int((~equivalence["Pass"].map(as_bool)).sum()), int((~equivalence["Pass"].map(as_bool)).sum()) == 0)
add_check(validation, "Master finalizer model fits", 0, 0, True)
add_check(validation, "Master finalizer condition executions", 0, 0, True)

step5a_validation = pd.DataFrame(validation)
failed_validation = step5a_validation.loc[~step5a_validation["Pass"]]
print("\nMaster Step 5A validation:")
display(step5a_validation)
if not failed_validation.empty:
    print("\nFAILED CHECKS:")
    display(failed_validation)
    raise RuntimeError("PROJECT 22 STEP 5A MASTER FINALIZATION VALIDATION FAILED. No official Step 5A freeze was written.")

# --------------------------------------------------------------------------------------------------
# 11. WRITE OFFICIAL STEP 5A AGGREGATES ONLY AFTER ALL CHECKS PASS
# --------------------------------------------------------------------------------------------------

atomic_csv(CONDITION_INVENTORY_PATH, condition_inventory)
atomic_csv(RAW_MANIFEST_PATH, raw_manifest)
atomic_csv(BASELINE_INVARIANCE_PATH, baseline_invariance)
atomic_csv(COMBINED_CONDITION_AUDIT_PATH, combined_condition_audit)
atomic_csv(COMBINED_PROJECT_RUNS_PATH, combined_project_runs)
atomic_csv(COMBINED_BUILD_METRICS_PATH, combined_build_metrics)
atomic_csv(COMBINED_MODEL_FITS_PATH, combined_model_fits)
atomic_csv(WORKER_CHECKPOINT_AUDIT_PATH, worker_audit)
atomic_csv(STEP5A_VALIDATION_PATH, step5a_validation)

aggregate_output_paths = [
    CONDITION_INVENTORY_PATH, RAW_MANIFEST_PATH, BASELINE_INVARIANCE_PATH,
    COMBINED_CONDITION_AUDIT_PATH, COMBINED_PROJECT_RUNS_PATH,
    COMBINED_BUILD_METRICS_PATH, COMBINED_MODEL_FITS_PATH,
    WORKER_CHECKPOINT_AUDIT_PATH, STEP5A_VALIDATION_PATH,
    ACCELERATED_EQUIVALENCE_PATH,
]
aggregate_output_manifest = [
    {"Path": str(p), "Bytes": int(p.stat().st_size), "SHA256": sha256_file(p)}
    for p in aggregate_output_paths
]

completed_at = datetime.now(timezone.utc).isoformat()
execution_seconds = time.perf_counter() - started
report = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5A_STATUS,
    "Implementation": MASTER_FINALIZER_IMPLEMENTATION,
    "CompletedAtUTC": completed_at,
    "ExecutionMode": "PARALLEL_SIX_WORKERS_THEN_ZERO_FIT_MASTER_FINALIZATION",
    "AcceleratedEngineVersion": ACCELERATED_ENGINE_VERSION,
    "SelectionCheckpointSHA256": EXPECTED_SELECTION_CHECKPOINT_SHA256,
    "RECCheckpointSHA256": EXPECTED_REC_CHECKPOINT_SHA256,
    "NoisePlanCheckpointSHA256": EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
    "RuntimeCheckpointSHA256": EXPECTED_RUNTIME_CHECKPOINT_SHA256,
    "SmokeCheckpointSHA256": EXPECTED_SMOKE_CHECKPOINT_SHA256,
    "ExpectedWorkerCheckpointSHA256": {tag: data["SHA256"] for tag, data in EXPECTED_WORKER_CHECKPOINTS.items()},
    "WorkerCheckpoints": worker_records,
    "Conditions": len(condition_inventory),
    "NoiseLevels": NOISE_LEVELS,
    "RepetitionSeeds": REPETITION_SEEDS,
    "MLFits": len(combined_model_fits),
    "RankingRows": int(condition_inventory["RankingRows"].sum()),
    "BuildMetricRows": len(combined_build_metrics),
    "ProjectRunRows": len(combined_project_runs),
    "ConditionAuditRows": len(combined_condition_audit),
    "TrainingMedianRows": int(condition_inventory["TrainingMedianRows"].sum()),
    "RawFiles": raw_files,
    "RawBytes": raw_bytes,
    "RawRootSHA256": raw_root_sha256,
    "AggregateOutputManifest": aggregate_output_manifest,
    "ValidationChecks": len(step5a_validation),
    "FailedValidationChecks": 0,
    "SourceRootSHA256": EXPECTED_SOURCE_ROOT_SHA256,
    "RegistrySHA256": EXPECTED_REGISTRY_SHA256,
    "RegistryModified": False,
    "Projects1To21Modified": False,
    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectConditionOutputsModified": False,
    "EvaluationCohortImmutable": True,
    "ActiveReservations": EXPECTED_ACTIVE_RESERVATIONS,
    "RuntimePriorityRule": EXPECTED_RUNTIME_PRIORITY_RULE,
    "ResumeSafe": True,
    "ParallelExecution": True,
    "MasterFinalizationModelFits": 0,
    "MasterFinalizationConditionExecutions": 0,
    "SourceRestoredInThisInvocation": source_restored,
    "ExecutionSecondsThisInvocation": float(execution_seconds),
}
atomic_json(STEP5A_REPORT_PATH, report)

checkpoint = {
    **report,
    "CheckpointVersion": 1,
    "Step5AReportPath": str(STEP5A_REPORT_PATH),
    "Step5AReportSHA256": sha256_file(STEP5A_REPORT_PATH),
    "Full270ConditionExperimentComplete": True,
    "RawResultRootFrozen": True,
    "DoNotRerunCompletedConditions": True,
    "OfficialStep5AComplete": True,
    "NextRequiredStep": "STEP_5B_RAW_REVALIDATION_AND_COMPACT_AGGREGATION",
}
atomic_json(STEP5A_CHECKPOINT_PATH, checkpoint)

status_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5A_STATUS,
    "CompletedAtUTC": completed_at,
    "Conditions": EXPECTED_CONDITIONS,
    "MLFits": EXPECTED_TOTAL_MODEL_FITS,
    "RawFiles": EXPECTED_RAW_FILES,
    "RawBytes": EXPECTED_RAW_BYTES,
    "RawRootSHA256": raw_root_sha256,
    "Checkpoint": str(STEP5A_CHECKPOINT_PATH),
    "CheckpointSHA256": sha256_file(STEP5A_CHECKPOINT_PATH),
    "RegistryModified": False,
    "OfficialStep5AComplete": True,
}
atomic_json(STEP5A_STATUS_PATH, status_payload)

atomic_json(RUN_PROGRESS_PATH, {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5A_STATUS,
    "UpdatedAtUTC": completed_at,
    "CompletedConditions": EXPECTED_CONDITIONS,
    "ExpectedConditions": EXPECTED_CONDITIONS,
    "CheckpointSHA256": sha256_file(STEP5A_CHECKPOINT_PATH),
    "ResumeSafe": True,
    "OfficialStep5AComplete": True,
    "NextRequiredStep": "STEP_5B_RAW_REVALIDATION_AND_COMPACT_AGGREGATION",
})

# --------------------------------------------------------------------------------------------------
# 12. FINAL READBACK / IMMUTABILITY PROOF
# --------------------------------------------------------------------------------------------------

if load_json(STEP5A_CHECKPOINT_PATH).get("Status") != STEP5A_STATUS:
    raise RuntimeError("Official Project 22 Step 5A checkpoint readback failed.")
if load_json(STEP5A_STATUS_PATH).get("Status") != STEP5A_STATUS:
    raise RuntimeError("Official Project 22 Step 5A status readback failed.")
final_manifest_failures = verify_manifest_entries(load_json(STEP5A_CHECKPOINT_PATH).get("AggregateOutputManifest", []), "official Step5A aggregate")
if final_manifest_failures:
    display(pd.DataFrame(final_manifest_failures))
    raise RuntimeError("Official Step 5A aggregate-output manifest readback failed.")
if sha256_file(REGISTRY_PATH) != registry_sha_before:
    raise RuntimeError("Completion registry changed during master finalization.")
for p, before in cohort_sha_before.items():
    if sha256_file(Path(p)) != before:
        raise RuntimeError(f"Frozen Project 22 cohort/plan changed during master finalization: {p}")
final_source = pd.DataFrame([
    {"RelativePath": str(r.RelativePath), "SizeBytes": int((SOURCE_DIR / str(r.RelativePath)).stat().st_size), "SHA256": sha256_file(SOURCE_DIR / str(r.RelativePath))}
    for r in frozen_source_manifest.itertuples(index=False)
])
if source_root_hash(final_source) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError("Frozen Project 22 source changed during master finalization.")

# Raw files are never written by this finalizer; verify count/bytes again after official writes.
raw_file_paths_after = [p for p in FULL_RAW_RESULT_ROOT.rglob("*") if p.is_file()]
if len(raw_file_paths_after) != EXPECTED_RAW_FILES or sum(p.stat().st_size for p in raw_file_paths_after) != EXPECTED_RAW_BYTES:
    raise RuntimeError("Project 22 raw root count/bytes changed during master finalization.")

print("\n" + "=" * 136)
print("=== PROJECT 22 CELL 9 / STEP 5A RESULT ===")
print("=" * 136)
print("Project:", PROJECT_NAME)
print("Workers verified:", len(worker_audit), "/ 6")
print("Seeds covered exactly once:", sorted(worker_seed_union) == REPETITION_SEEDS)
print("Conditions:", len(condition_inventory), "/", EXPECTED_CONDITIONS)
print("ML fits:", len(combined_model_fits), "/", EXPECTED_TOTAL_MODEL_FITS)
print("Ranking rows:", int(condition_inventory["RankingRows"].sum()), "/", EXPECTED_TOTAL_RANKING_ROWS)
print("Build-metric rows:", len(combined_build_metrics), "/", EXPECTED_TOTAL_BUILD_METRIC_ROWS)
print("Project-run rows:", len(combined_project_runs), "/", EXPECTED_TOTAL_PROJECT_RUN_ROWS)
print("Raw files:", raw_files, "/", EXPECTED_RAW_FILES)
print("Raw bytes:", raw_bytes, "/", EXPECTED_RAW_BYTES)
print("Raw root SHA-256:", raw_root_sha256)
print("Baseline invariance failures:", int((~baseline_invariance["Pass"]).sum()))
print("Validation checks:", len(step5a_validation))
print("Failed checks:", int((~step5a_validation["Pass"]).sum()))
print("Master-finalizer model fits:", 0)
print("Master-finalizer condition executions:", 0)
print("Registry modified:", False)
print("Checkpoint:", STEP5A_CHECKPOINT_PATH)
print("Checkpoint SHA-256:", sha256_file(STEP5A_CHECKPOINT_PATH))
print("Next required step: STEP 5B — RAW REVALIDATION AND COMPACT AGGREGATION")
print("STATUS:", STEP5A_STATUS)
print("=" * 136)


=== PROJECT 22 CELL 9 / STEP 5A: PARALLEL MASTER FINALIZATION ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Restoring only the frozen Project 22 source from the thesis archive...


/tmp/ipykernel_1273/147906895.py:366: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(target_root, members=members)



Independently revalidating all 270 frozen condition directories (ZERO fitting).
  Validated 30/270 conditions
  Validated 60/270 conditions
  Validated 90/270 conditions
  Validated 120/270 conditions
  Validated 150/270 conditions
  Validated 180/270 conditions
  Validated 210/270 conditions
  Validated 240/270 conditions
  Validated 270/270 conditions

Hashing the complete Project 22 raw-result root (2,160 files).


TypeError: cannot convert the series to <class 'int'>

In [2]:
# ==================================================================================================
# PROJECT 22 — CELL 9 / STEP 5A PARALLEL MASTER FINALIZATION
# ZERO-FIT REVALIDATION, AGGREGATION, RAW-ROOT FREEZE, AND OFFICIAL CHECKPOINT
#
# PROJECT:
#   apache@logging-log4j2
#
# RUN THIS AS THE NEXT NEW CELL IN THE RECONNECTED MASTER Thesis_project_22.ipynb NOTEBOOK.
#
# PURPOSE:
# - verify all six frozen Step 5A worker checkpoints and output manifests;
# - prove worker seed shards are disjoint and cover seeds 1–30 exactly;
# - independently revalidate all 270 completed condition directories without fitting any model;
# - rebuild the official aggregate outputs from the frozen raw condition files;
# - hash and freeze the complete 2,160-file Project 22 raw-result root;
# - create the official Project 22 Step 5A report/status/checkpoint for Step 5B.
#
# SAFETY:
# - ZERO model fitting and ZERO noise/REC condition execution in this cell;
# - no registry write and no modification of Projects 1–21;
# - no prior-project condition-output access;
# - all 270 condition directories are read-only during finalization;
# - official Step 5A outputs are written only after every validation passes.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import re
import tarfile
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from google.colab import drive

print("=" * 136)
print("=== PROJECT 22 CELL 9 / STEP 5A: PARALLEL MASTER FINALIZATION ===")
print("=" * 136)

# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 22
PROJECT_NAME = "apache@logging-log4j2"
PROJECT_SLUG = "apache__logging-log4j2"
PROJECT_SHORT = "LOG4J2"

EXPECTED_SELECTION_STATUS = "PASS_PROJECT_22_SELECTION_AND_SOURCE_FROZEN"
EXPECTED_STEP2B_STATUS = "PASS_PROJECT_22_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
EXPECTED_STEP3A_STATUS = "PASS_PROJECT_22_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
EXPECTED_STEP4A_STATUS = "PASS_PROJECT_22_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
EXPECTED_STEP4B_STATUS = "PASS_PROJECT_22_TWO_CONDITION_END_TO_END_SMOKE_TEST"
STEP5A_STATUS = "PASS_PROJECT_22_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
CONDITION_STATUS = "PASS_FULL_CONDITION"
MASTER_FINALIZER_IMPLEMENTATION = "PROJECT_22_PARALLEL_STEP5A_MASTER_FINALIZER_V2_STATUS_CHECK_FIX_ZERO_FIT"

EXPECTED_RUNTIME_CHECKPOINT_SHA256 = "3ef12ada698780da80e05fd6899b0fd98fc5dcc4f3e74c87ec8ea326e3f4f593"
EXPECTED_SMOKE_CHECKPOINT_SHA256 = "91b912640da6dc4b661fabd35632b5fa001ce164bc4857f2f6e139550925e00d"
EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = "3dd2b3f38a8d7b77c179e9f518399c1dc5b1c89a8a29607ea31fd78a91f40666"
EXPECTED_REC_CHECKPOINT_SHA256 = "8417249bc74e2a50e2f61cc3b7776dab7f731f62993e3ed15e415745be178731"
EXPECTED_SELECTION_CHECKPOINT_SHA256 = "a7387d9495c71dce5d2c21ff08b0afd80d9c5251b5885a82e4052c7506ee7890"
EXPECTED_SOURCE_ROOT_SHA256 = "281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64f842964c896c6ac334"
EXPECTED_REGISTRY_SHA256 = "79cd6ecb595c5e8ae91a9494e469792716338d144308560a62caf1b9342306b2"

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 21
EXPECTED_ACTIVE_RESERVATIONS = []
EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_WORKER_CHECKPOINTS = {
    "seed_01_05": {
        "Status": "PASS_PROJECT_22_STEP5A_WORKER_SEEDS_01_05_COMPLETE",
        "Seeds": list(range(1, 6)),
        "SHA256": "6edc0ecf1d33c3fd2263f968f7d953726e3b0ddf3a0a2b40b2cc34ecbb15bb44",
        "RawRootSHA256": "144501c9ae9d261d310e17bb0e5c3c86e596813a3c3d39b0e0354e6ca744f25e",
        "RawBytes": 107_998_067,
    },
    "seed_06_10": {
        "Status": "PASS_PROJECT_22_STEP5A_WORKER_SEEDS_06_10_COMPLETE",
        "Seeds": list(range(6, 11)),
        "SHA256": "1a8ebadb53e01acd4408ac58c8235b4c6504bee2a6bb0213966563a4243aa0c8",
        "RawRootSHA256": "dd718cad6981f126505d70415f57ff7caa8f6025f88e6c6af1fafb270ebab85c",
        "RawBytes": 107_584_568,
    },
    "seed_11_15": {
        "Status": "PASS_PROJECT_22_STEP5A_WORKER_SEEDS_11_15_COMPLETE",
        "Seeds": list(range(11, 16)),
        "SHA256": "f02df169293ce4d1ae143552547a89654ce21e1084b641b4b3ce48ce9afbbc05",
        "RawRootSHA256": "a23ab53f39cfb431e3b34ff1996cfff9d35ab297858cdc69fd53c143cd392e30",
        "RawBytes": 107_570_605,
    },
    "seed_16_20": {
        "Status": "PASS_PROJECT_22_STEP5A_WORKER_SEEDS_16_20_COMPLETE",
        "Seeds": list(range(16, 21)),
        "SHA256": "0643261cc0bd54cadba8354fa88e726ab2e199b940aa2b3f609a581121663db4",
        "RawRootSHA256": "e35458b61a8f803fd28bceae2bcb180150fde655cea38aae9fb42a2d7395b07f",
        "RawBytes": 107_709_615,
    },
    "seed_21_25": {
        "Status": "PASS_PROJECT_22_STEP5A_WORKER_SEEDS_21_25_COMPLETE",
        "Seeds": list(range(21, 26)),
        "SHA256": "23806c7c5252cb4f4b5f6f5faee559678b8c6d02a023437405f7551eaa6c3718",
        "RawRootSHA256": "9fa32797786104d0b6eddbc5190dfe784cbe464f257b4a8e1c0b21e1f27a0f33",
        "RawBytes": 107_713_150,
    },
    "seed_26_30": {
        "Status": "PASS_PROJECT_22_STEP5A_WORKER_SEEDS_26_30_COMPLETE",
        "Seeds": list(range(26, 31)),
        "SHA256": "f7e03bb7c57f29782b057554bbe8116a25e14cce7336361154e7c60e3a0af173",
        "RawRootSHA256": "1ca645ced25f816336359993d10e664ecccbcd4b88db3baa5b90778556ab8c5e",
        "RawBytes": 107_554_648,
    },
}

EXPECTED_WORKER_CONDITIONS = 45
EXPECTED_WORKER_MODEL_FITS = 180
EXPECTED_WORKER_RAW_FILES = 360
EXPECTED_WORKER_RANKING_ROWS = 6_979_140
EXPECTED_WORKER_BUILD_METRIC_ROWS = 12_285
EXPECTED_WORKER_PROJECT_RUN_ROWS = 315
EXPECTED_WORKER_TRAINING_MEDIAN_ROWS = 6_795

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 101_286_750
EXPECTED_RAW_TRAIN_ROWS = 172_628
EXPECTED_RAW_EVAL_ROWS = 67_625
EXPECTED_MODEL_TRAIN_ROWS = 95_812
EXPECTED_MODEL_EVAL_ROWS = 22_156
EXPECTED_MODEL_TRAIN_FAILURES = 207
EXPECTED_MODEL_EVAL_FAILURES = 40
EXPECTED_FAILING_EVAL_BUILDS = 39
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_EVALUATION_BUILDS = 111
EXPECTED_TECHNIQUES = 7
EXPECTED_ML_TECHNIQUES = 4
EXPECTED_CONDITIONS = 270
EXPECTED_FILES_PER_CONDITION = 8
EXPECTED_RAW_FILES = 2_160
EXPECTED_RAW_BYTES = 646_130_653
EXPECTED_RNG_ROWS = 5_178_840

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
REPETITION_SEEDS = list(range(1, 31))
ML_TECHNIQUES = ["RandomForest", "XGBoost", "LightGBM", "NaiveBayes"]
BASELINE_TECHNIQUES = ["Random", "LatestFail", "QTF-Avg"]
ALL_TECHNIQUES = ML_TECHNIQUES + BASELINE_TECHNIQUES
INVARIANT_BASELINES = ["Random", "QTF-Avg"]
ACCELERATED_ENGINE_VERSION = "PROJECT_22_FAST_DEPENDENT_REC_V3_SMOKE_EQUIVALENT_FROZEN_INFERRED_TEST_ORDER_ANCHOR_AWARE"
SMOKE_EQUIVALENCE_KEYS = ["noise_00__seed_01", "noise_50__seed_01"]

EXPECTED_RANKING_ROWS_PER_CONDITION = EXPECTED_MODEL_EVAL_ROWS * EXPECTED_TECHNIQUES
EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = EXPECTED_FAILING_EVAL_BUILDS * EXPECTED_TECHNIQUES
EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = EXPECTED_TECHNIQUES
EXPECTED_MODEL_FIT_ROWS_PER_CONDITION = EXPECTED_ML_TECHNIQUES
EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION = EXPECTED_PREDICTORS

EXPECTED_TOTAL_RANKING_ROWS = EXPECTED_RANKING_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
EXPECTED_TOTAL_BUILD_METRIC_ROWS = EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
EXPECTED_TOTAL_PROJECT_RUN_ROWS = EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
EXPECTED_TOTAL_MODEL_FITS = EXPECTED_MODEL_FIT_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS = EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
EXPECTED_TOTAL_CONDITION_AUDIT_ROWS = EXPECTED_CONDITIONS

# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES_ROOT = THESIS_ROOT / "Notes"
RESULTS_ROOT = THESIS_ROOT / "Results"
REGISTRY_PATH = NOTES_ROOT / "completed_project_registry.csv"
ARCHIVE_PATH = THESIS_ROOT / "Data" / "Raw" / "TCP-CI-main-dataset.tar.gz"
SOURCE_DIR = Path("/content/datasets/datasets/apache@logging-log4j2")

SELECTION_ROOT = RESULTS_ROOT / "Aggregated" / "project_22_selection"
FROZEN_SOURCE_MANIFEST_PATH = SELECTION_ROOT / "project_22_frozen_source_manifest.csv"
SELECTION_CHECKPOINT_PATH = NOTES_ROOT / "project_22_selection_checkpoint.json"
STEP1B_STATUS_PATH = SELECTION_ROOT / "project_22_step1b_status.json"

PROJECT_ROOT = RESULTS_ROOT / "Aggregated" / PROJECT_SLUG
REC_PREFLIGHT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_rec_preflight"
STEP2B_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step2b_status.json"
REC_CHECKPOINT_PATH = NOTES_ROOT / "project_22_rec_reconstruction_checkpoint.json"

NOISE_PLAN_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_noise_plan"
RAW_TRAINING_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
RAW_EVALUATION_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
MODEL_TRAINING_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
MODEL_EVALUATION_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
RNG_MANIFEST_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_rng_manifest.parquet"
CONDITION_PLAN_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_condition_plan.csv"
STEP3A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step3a_status.json"
NOISE_PLAN_CHECKPOINT_PATH = NOTES_ROOT / "project_22_noise_plan_checkpoint.json"

RUNTIME_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_runtime_contract"
STEP4A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step4a_status.json"
STEP4A_REPORT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_step4a_report.json"
RUNTIME_CHECKPOINT_PATH = NOTES_ROOT / "project_22_runtime_contract_checkpoint.json"

SMOKE_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_smoke_test"
STEP4B_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step4b_status.json"
SMOKE_CHECKPOINT_PATH = NOTES_ROOT / "project_22_smoke_test_checkpoint.json"

FULL_RAW_RESULT_ROOT = RESULTS_ROOT / "Raw" / PROJECT_SLUG
FULL_EXPERIMENT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_full_experiment"
CONDITION_INVENTORY_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_condition_inventory.csv"
RAW_MANIFEST_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_raw_manifest.csv"
BASELINE_INVARIANCE_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_baseline_invariance.csv"
COMBINED_CONDITION_AUDIT_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_condition_audit.csv"
COMBINED_PROJECT_RUNS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_project_runs.csv"
COMBINED_BUILD_METRICS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_build_metrics.csv"
COMBINED_MODEL_FITS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_model_fits.csv"
WORKER_CHECKPOINT_AUDIT_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_worker_checkpoint_audit.csv"
STEP5A_VALIDATION_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_validation.csv"
STEP5A_REPORT_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_report.json"
RUN_PROGRESS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_run_progress.json"
STEP5A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5a_status.json"
STEP5A_CHECKPOINT_PATH = NOTES_ROOT / "project_22_step5a_checkpoint.json"
ACCELERATED_EQUIVALENCE_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_accelerated_engine_equivalence.csv"

WORKER_CHECKPOINT_PATHS = {
    tag: NOTES_ROOT / f"project_22_step5a_worker_{tag}_checkpoint.json"
    for tag in EXPECTED_WORKER_CHECKPOINTS
}

# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def atomic_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, sort_keys=True, ensure_ascii=False, default=str)
        f.write("\n")
    os.replace(tmp, path)


def atomic_csv(path, df):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    df.to_csv(tmp, index=False, lineterminator="\n")
    os.replace(tmp, path)


def source_root_hash(frame):
    h = hashlib.sha256()
    for r in frame.sort_values("RelativePath", kind="mergesort").itertuples(index=False):
        size = getattr(r, "SizeBytes", getattr(r, "Bytes", None))
        h.update(f"{r.RelativePath}\0{int(size)}\0{str(r.SHA256).lower()}\n".encode("utf-8"))
    return h.hexdigest()


def directory_manifest(root):
    root = Path(root)
    if not root.exists():
        return pd.DataFrame(columns=["RelativePath", "Bytes", "SHA256"])
    records = []
    for path in sorted((p for p in root.rglob("*") if p.is_file()), key=lambda p: p.as_posix()):
        records.append({
            "RelativePath": path.relative_to(root).as_posix(),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        })
    return pd.DataFrame(records, columns=["RelativePath", "Bytes", "SHA256"])


def directory_root_hash(manifest):
    h = hashlib.sha256()
    for r in manifest.sort_values("RelativePath", kind="mergesort").itertuples(index=False):
        h.update(f"{r.RelativePath}\0{int(r.Bytes)}\0{str(r.SHA256).lower()}\n".encode("utf-8"))
    return h.hexdigest()


def add_check(rows, check, expected, actual, passed):
    rows.append({"Check": check, "Expected": expected, "Actual": actual, "Pass": bool(passed)})


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def verify_manifest_entries(entries, label):
    failures = []
    if not isinstance(entries, list) or not entries:
        return [{"Label": label, "Path": "<manifest>", "Reason": "missing/empty manifest"}]
    for item in entries:
        path = Path(str(item.get("Path", "")))
        if not path.is_file():
            failures.append({"Label": label, "Path": str(path), "Reason": "missing"})
            continue
        expected_bytes = int(item.get("Bytes", -1))
        expected_sha = str(item.get("SHA256", "")).lower()
        actual_bytes = int(path.stat().st_size)
        actual_sha = sha256_file(path)
        if actual_bytes != expected_bytes or actual_sha != expected_sha:
            failures.append({"Label": label, "Path": str(path), "Reason": "size/hash mismatch"})
    return failures


def resolve_registry_columns(registry):
    lookup = {str(c).strip().lower(): c for c in registry.columns}
    def pick(names, label):
        for name in names:
            if name in lookup:
                return lookup[name]
        raise RuntimeError(f"Could not resolve registry {label}; columns={list(registry.columns)}")
    return (
        pick(["projectnumber", "project_number", "project no", "projectno"], "ProjectNumber"),
        pick(["project", "projectname", "project_name"], "Project"),
        pick(["status", "projectstatus", "project_status"], "Status"),
    )


def restore_source_if_needed():
    if SOURCE_DIR.is_dir():
        return False
    if not ARCHIVE_PATH.is_file():
        raise FileNotFoundError(f"Project 22 source is absent and archive is missing: {ARCHIVE_PATH}")
    target_root = Path("/content/datasets")
    target_root.mkdir(parents=True, exist_ok=True)
    prefix = "datasets/apache@logging-log4j2/"
    print("\nRestoring only the frozen Project 22 source from the thesis archive...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        members = [m for m in tar.getmembers() if m.name.startswith(prefix)]
        if not members:
            raise RuntimeError(f"Archive contains no members under {prefix}")
        root_resolved = target_root.resolve()
        for member in members:
            destination = (target_root / member.name).resolve()
            if root_resolved not in destination.parents and destination != root_resolved:
                raise RuntimeError(f"Unsafe archive member: {member.name}")
        tar.extractall(target_root, members=members)
    if not SOURCE_DIR.is_dir():
        raise RuntimeError("Selective Project 22 source restoration failed.")
    return True


EXPECTED_CONDITION_FILES = {
    "rankings.csv.gz",
    "build_metrics.csv",
    "project_runs.csv",
    "model_fits.csv",
    "training_medians.csv",
    "condition_audit.csv",
    "condition_summary.json",
    "COMPLETE.json",
}


def validate_completed_condition(condition_dir, plan_row):
    condition_dir = Path(condition_dir)
    condition_key = str(plan_row.ConditionID)
    if not condition_dir.is_dir():
        return None
    actual_files = {p.name for p in condition_dir.iterdir() if p.is_file()}
    if actual_files != EXPECTED_CONDITION_FILES:
        return None
    completion_path = condition_dir / "COMPLETE.json"
    summary_path = condition_dir / "condition_summary.json"
    try:
        completion = load_json(completion_path)
        summary = load_json(summary_path)
    except Exception:
        return None
    if completion.get("Status") != CONDITION_STATUS or summary.get("Status") != CONDITION_STATUS:
        return None
    if completion.get("ConditionKey") != condition_key or summary.get("ConditionKey") != condition_key:
        return None
    if int(summary.get("NoisePercent", -1)) != int(plan_row.NoisePercent):
        return None
    if int(summary.get("RepetitionSeed", -1)) != int(plan_row.RepetitionSeed):
        return None
    if int(summary.get("ConditionOrder", -1)) != int(plan_row.ConditionOrder):
        return None
    if summary.get("Project") != PROJECT_NAME or summary.get("ProjectSlug") != PROJECT_SLUG:
        return None
    if summary.get("AcceleratedEngineVersion") != ACCELERATED_ENGINE_VERSION:
        return None
    if str(completion.get("ConditionSummaryPath")) != str(summary_path):
        return None
    if str(completion.get("ConditionSummarySHA256")) != sha256_file(summary_path):
        return None
    output_manifest = summary.get("OutputManifest", [])
    if not isinstance(output_manifest, list) or len(output_manifest) != 6:
        return None
    for item in output_manifest:
        path = Path(str(item.get("Path", "")))
        if path.parent != condition_dir or not path.is_file():
            return None
        if int(path.stat().st_size) != int(item.get("Bytes", -1)):
            return None
        if sha256_file(path) != str(item.get("SHA256", "")):
            return None
    expected_counts = {
        "MLFits": EXPECTED_MODEL_FIT_ROWS_PER_CONDITION,
        "RankingRows": EXPECTED_RANKING_ROWS_PER_CONDITION,
        "BuildMetricRows": EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
        "ProjectRunRows": EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
        "TrainingMedianRows": EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION,
    }
    for key, expected in expected_counts.items():
        if int(summary.get(key, -1)) != expected:
            return None
    if int(summary.get("IndependentRECChanges", -1)) != 0:
        return None
    fingerprints = summary.get("BaselineFingerprints", {})
    if sorted(fingerprints.keys()) != ["QTF-Avg", "Random"]:
        return None
    manifest = directory_manifest(condition_dir)
    if len(manifest) != EXPECTED_FILES_PER_CONDITION:
        return None
    return {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": int(plan_row.ConditionOrder),
        "ConditionKey": condition_key,
        "NoisePercent": int(plan_row.NoisePercent),
        "RepetitionSeed": int(plan_row.RepetitionSeed),
        "ConditionDirectory": str(condition_dir),
        "Status": CONDITION_STATUS,
        "Files": int(len(manifest)),
        "ConditionBytes": int(manifest["Bytes"].sum()),
        "ConditionRootSHA256": directory_root_hash(manifest),
        "RankingRows": int(summary["RankingRows"]),
        "BuildMetricRows": int(summary["BuildMetricRows"]),
        "ProjectRunRows": int(summary["ProjectRunRows"]),
        "MLFits": int(summary["MLFits"]),
        "TrainingMedianRows": int(summary["TrainingMedianRows"]),
        "NumberFlipped": int(summary.get("NumberFlipped", -1)),
        "ModelLabelChanges": int(summary.get("ModelLabelChanges", -1)),
        "DependentRECChanges": int(summary.get("DependentRECChanges", -1)),
        "IndependentRECChanges": int(summary.get("IndependentRECChanges", -1)),
        "TrainingFailures": int(summary.get("TrainingFailures", -1)),
        "ConditionSeconds": float(summary.get("ConditionSeconds", np.nan)),
        "BaselineFingerprints": fingerprints,
    }

# --------------------------------------------------------------------------------------------------
# 4. MOUNT, RESTORE SELECTED SOURCE IF NEEDED, AND VERIFY UPSTREAM FREEZES
# --------------------------------------------------------------------------------------------------

started = time.perf_counter()
drive.mount("/content/drive", force_remount=False)
source_restored = restore_source_if_needed()

required_paths = [
    REGISTRY_PATH, FROZEN_SOURCE_MANIFEST_PATH, SELECTION_CHECKPOINT_PATH, STEP1B_STATUS_PATH,
    STEP2B_STATUS_PATH, REC_CHECKPOINT_PATH, NOISE_PLAN_CHECKPOINT_PATH, STEP3A_STATUS_PATH,
    RUNTIME_CHECKPOINT_PATH, STEP4A_STATUS_PATH, STEP4A_REPORT_PATH, SMOKE_CHECKPOINT_PATH,
    STEP4B_STATUS_PATH, CONDITION_PLAN_PATH, RAW_TRAINING_COHORT_PATH, RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH, MODEL_EVALUATION_COHORT_PATH, RNG_MANIFEST_PATH,
    ACCELERATED_EQUIVALENCE_PATH, FULL_RAW_RESULT_ROOT,
] + list(WORKER_CHECKPOINT_PATHS.values())
missing = [str(p) for p in required_paths if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing Project 22 master-finalization inputs:\n" + "\n".join(missing))

sha_expectations = {
    "selection": (SELECTION_CHECKPOINT_PATH, EXPECTED_SELECTION_CHECKPOINT_SHA256),
    "REC": (REC_CHECKPOINT_PATH, EXPECTED_REC_CHECKPOINT_SHA256),
    "noise plan": (NOISE_PLAN_CHECKPOINT_PATH, EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256),
    "runtime": (RUNTIME_CHECKPOINT_PATH, EXPECTED_RUNTIME_CHECKPOINT_SHA256),
    "smoke": (SMOKE_CHECKPOINT_PATH, EXPECTED_SMOKE_CHECKPOINT_SHA256),
}
for label, (path, expected) in sha_expectations.items():
    actual = sha256_file(path)
    if actual != expected:
        raise RuntimeError(f"Frozen Project 22 {label} checkpoint SHA-256 differs.\nExpected: {expected}\nActual:   {actual}")

selection_checkpoint = load_json(SELECTION_CHECKPOINT_PATH)
step1b_status = load_json(STEP1B_STATUS_PATH)
rec_checkpoint = load_json(REC_CHECKPOINT_PATH)
step2b_status = load_json(STEP2B_STATUS_PATH)
noise_checkpoint = load_json(NOISE_PLAN_CHECKPOINT_PATH)
step3a_status = load_json(STEP3A_STATUS_PATH)
runtime_checkpoint = load_json(RUNTIME_CHECKPOINT_PATH)
step4a_status = load_json(STEP4A_STATUS_PATH)
step4a_report = load_json(STEP4A_REPORT_PATH)
smoke_checkpoint = load_json(SMOKE_CHECKPOINT_PATH)
step4b_status = load_json(STEP4B_STATUS_PATH)

status_pairs = [
    ("selection checkpoint", selection_checkpoint.get("Status"), EXPECTED_SELECTION_STATUS),
    ("Step 1B status", step1b_status.get("Status"), EXPECTED_SELECTION_STATUS),
    ("REC checkpoint", rec_checkpoint.get("Status"), EXPECTED_STEP2B_STATUS),
    ("Step 2B status", step2b_status.get("Status"), EXPECTED_STEP2B_STATUS),
    ("noise checkpoint", noise_checkpoint.get("Status"), EXPECTED_STEP3A_STATUS),
    ("Step 3A status", step3a_status.get("Status"), EXPECTED_STEP3A_STATUS),
    ("runtime checkpoint", runtime_checkpoint.get("Status"), EXPECTED_STEP4A_STATUS),
    ("Step 4A status", step4a_status.get("Status"), EXPECTED_STEP4A_STATUS),
    ("Step 4A report", step4a_report.get("Status"), EXPECTED_STEP4A_STATUS),
    ("smoke checkpoint", smoke_checkpoint.get("Status"), EXPECTED_STEP4B_STATUS),
    ("Step 4B status", step4b_status.get("Status"), EXPECTED_STEP4B_STATUS),
]
for label, actual, expected in status_pairs:
    if actual != expected:
        raise RuntimeError(f"{label} differs: expected {expected!r}, actual {actual!r}")
if not bool(smoke_checkpoint.get("ReadyForFull270ConditionExperiment", False)):
    raise RuntimeError("Frozen Step 4B checkpoint does not authorize the full experiment.")

manifest_failures = []
manifest_failures += verify_manifest_entries(runtime_checkpoint.get("RuntimeOutputManifest", []), "runtime")
manifest_failures += verify_manifest_entries(smoke_checkpoint.get("OutputManifest", []), "smoke")
if manifest_failures:
    display(pd.DataFrame(manifest_failures))
    raise RuntimeError("One or more frozen upstream output manifests no longer validate.")

# Registry immutability and identities.
registry_sha_before = sha256_file(REGISTRY_PATH)
if registry_sha_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError("Completion registry SHA-256 differs before Project 22 Step 5A finalization.")
registry = pd.read_csv(REGISTRY_PATH, low_memory=False)
pn_col, project_col, status_col = resolve_registry_columns(registry)
project_numbers = pd.to_numeric(registry[pn_col], errors="raise").astype(int)
if len(registry) != EXPECTED_REGISTERED_PROJECTS or sorted(project_numbers.tolist()) != list(range(1, 22)):
    raise RuntimeError("Completion registry does not contain exactly Projects 1–21.")
if not registry[status_col].astype(str).eq(EXPECTED_COMPLETE_STATUS).all():
    raise RuntimeError("Projects 1–21 are not all COMPLETE_AND_FROZEN.")
required_identities = {
    11: "apache@shardingsphere", 12: "zolyfarkas@spf4j", 13: "jcabi@jcabi-github",
    14: "JMRI@JMRI", 15: "eclipse@steady", 16: "apache@rocketmq", 17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe", 19: "EMResearch@EvoMaster", 20: "apache@curator",
    21: "facebook@buck",
}
for number, identity in required_identities.items():
    rows = registry.loc[project_numbers.eq(number)]
    if len(rows) != 1 or str(rows.iloc[0][project_col]) != identity:
        raise RuntimeError(f"Frozen predecessor identity differs for Project {number}: expected {identity}")
if project_numbers.eq(PROJECT_NUMBER).any() or registry[project_col].astype(str).eq(PROJECT_NAME).any():
    raise RuntimeError("Project 22 is already present in the completion registry.")

# Source immutability.
frozen_source_manifest = pd.read_csv(FROZEN_SOURCE_MANIFEST_PATH, low_memory=False)
current_source = []
for row in frozen_source_manifest.itertuples(index=False):
    path = SOURCE_DIR / str(row.RelativePath)
    if not path.is_file():
        raise FileNotFoundError(f"Frozen Project 22 source file is missing: {path}")
    current_source.append({"RelativePath": str(row.RelativePath), "SizeBytes": int(path.stat().st_size), "SHA256": sha256_file(path)})
current_source = pd.DataFrame(current_source)
source_sha_before = source_root_hash(current_source)
if source_sha_before != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError("Frozen Project 22 source root differs before finalization.")
if len(current_source) != EXPECTED_SOURCE_FILES or int(current_source["SizeBytes"].sum()) != EXPECTED_SOURCE_BYTES:
    raise RuntimeError("Frozen Project 22 source file/byte counts differ.")

# Record cohort hashes before any official output write.
cohort_paths = [RAW_TRAINING_COHORT_PATH, RAW_EVALUATION_COHORT_PATH, MODEL_TRAINING_COHORT_PATH, MODEL_EVALUATION_COHORT_PATH, RNG_MANIFEST_PATH, CONDITION_PLAN_PATH]
cohort_sha_before = {str(p): sha256_file(p) for p in cohort_paths}
if pq.ParquetFile(RAW_TRAINING_COHORT_PATH).metadata.num_rows != EXPECTED_RAW_TRAIN_ROWS:
    raise RuntimeError("Frozen raw training cohort row count differs.")
if pq.ParquetFile(RAW_EVALUATION_COHORT_PATH).metadata.num_rows != EXPECTED_RAW_EVAL_ROWS:
    raise RuntimeError("Frozen raw evaluation cohort row count differs.")
if pq.ParquetFile(MODEL_TRAINING_COHORT_PATH).metadata.num_rows != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Frozen model training cohort row count differs.")
if pq.ParquetFile(MODEL_EVALUATION_COHORT_PATH).metadata.num_rows != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Frozen model evaluation cohort row count differs.")
if pq.ParquetFile(RNG_MANIFEST_PATH).metadata.num_rows != EXPECTED_RNG_ROWS:
    raise RuntimeError("Frozen RNG manifest row count differs.")

# --------------------------------------------------------------------------------------------------
# 5. VALIDATE FROZEN CONDITION PLAN
# --------------------------------------------------------------------------------------------------

condition_plan = pd.read_csv(CONDITION_PLAN_PATH, low_memory=False)
required_plan_cols = {"ConditionOrder", "ConditionID", "NoisePercent", "RepetitionSeed"}
if not required_plan_cols.issubset(condition_plan.columns):
    raise RuntimeError(f"Condition plan missing required columns: {sorted(required_plan_cols - set(condition_plan.columns))}")
condition_plan = condition_plan.sort_values("ConditionOrder", kind="mergesort").reset_index(drop=True)
expected_grid = [(seed, noise, f"noise_{noise:02d}__seed_{seed:02d}") for seed in REPETITION_SEEDS for noise in NOISE_LEVELS]
actual_grid = [(int(r.RepetitionSeed), int(r.NoisePercent), str(r.ConditionID)) for r in condition_plan.itertuples(index=False)]
if len(condition_plan) != EXPECTED_CONDITIONS or actual_grid != expected_grid:
    raise RuntimeError("Frozen Project 22 condition grid/order differs from the thesis contract.")
if condition_plan["ConditionID"].duplicated().any() or condition_plan.duplicated(["NoisePercent", "RepetitionSeed"]).any():
    raise RuntimeError("Frozen condition plan contains duplicate coordinates.")

# --------------------------------------------------------------------------------------------------
# 6. VERIFY ALL SIX WORKER CHECKPOINTS AND SHARD COVERAGE
# --------------------------------------------------------------------------------------------------

worker_records = []
worker_seed_union = []
worker_manifest_failures = []
for tag, expected in EXPECTED_WORKER_CHECKPOINTS.items():
    path = WORKER_CHECKPOINT_PATHS[tag]
    actual_sha = sha256_file(path)
    if actual_sha != expected["SHA256"]:
        raise RuntimeError(f"Worker {tag} checkpoint SHA differs. Expected {expected['SHA256']}; actual {actual_sha}")
    cp = load_json(path)
    if cp.get("Project") != PROJECT_NAME or cp.get("ProjectSlug") != PROJECT_SLUG or int(cp.get("ProjectNumber", -1)) != PROJECT_NUMBER:
        raise RuntimeError(f"Worker {tag} project identity differs.")
    if cp.get("Status") != expected["Status"]:
        raise RuntimeError(f"Worker {tag} PASS status differs.")
    seeds = [int(x) for x in cp.get("AssignedSeeds", [])]
    if seeds != expected["Seeds"]:
        raise RuntimeError(f"Worker {tag} assigned seeds differ: {seeds}")
    worker_seed_union.extend(seeds)
    scalar_checks = {
        "AssignedConditions": EXPECTED_WORKER_CONDITIONS,
        "CompletedConditions": EXPECTED_WORKER_CONDITIONS,
        "MLFits": EXPECTED_WORKER_MODEL_FITS,
        "RankingRows": EXPECTED_WORKER_RANKING_ROWS,
        "BuildMetricRows": EXPECTED_WORKER_BUILD_METRIC_ROWS,
        "ProjectRunRows": EXPECTED_WORKER_PROJECT_RUN_ROWS,
        "TrainingMedianRows": EXPECTED_WORKER_TRAINING_MEDIAN_ROWS,
        "WorkerRawFiles": EXPECTED_WORKER_RAW_FILES,
        "WorkerRawBytes": expected["RawBytes"],
    }
    for key, value in scalar_checks.items():
        if int(cp.get(key, -1)) != int(value):
            raise RuntimeError(f"Worker {tag} {key} differs: expected {value}; actual {cp.get(key)}")
    if str(cp.get("WorkerRawRootSHA256", "")) != expected["RawRootSHA256"]:
        raise RuntimeError(f"Worker {tag} raw-root hash differs.")
    if not bool(cp.get("WorkerShardComplete", False)) or bool(cp.get("OfficialStep5AComplete", True)):
        raise RuntimeError(f"Worker {tag} shard-completion/master-state flags differ.")
    if not bool(cp.get("MasterFinalizationRequired", False)):
        raise RuntimeError(f"Worker {tag} does not require master finalization.")
    if int(cp.get("BaselineInvarianceFailures", -1)) != 0 or int(cp.get("FailedValidationChecks", -1)) != 0:
        raise RuntimeError(f"Worker {tag} reports validation/baseline failures.")
    if int(cp.get("MasterStep5AWriteGuardFailures", -1)) != 0:
        raise RuntimeError(f"Worker {tag} reports master write-guard failures.")
    if bool(cp.get("RegistryModified", True)) or bool(cp.get("PriorProjectConditionOutputsAccessed", True)) or bool(cp.get("PriorProjectConditionOutputsModified", True)):
        raise RuntimeError(f"Worker {tag} reports an immutability violation.")
    if str(cp.get("SourceRootSHA256", "")) != EXPECTED_SOURCE_ROOT_SHA256 or str(cp.get("RegistrySHA256", "")) != EXPECTED_REGISTRY_SHA256:
        raise RuntimeError(f"Worker {tag} source/registry freeze linkage differs.")
    worker_manifest_failures += verify_manifest_entries(cp.get("WorkerOutputManifest", []), f"worker {tag}")
    report_path = Path(str(cp.get("WorkerReportPath", "")))
    if not report_path.is_file() or sha256_file(report_path) != str(cp.get("WorkerReportSHA256", "")):
        raise RuntimeError(f"Worker {tag} report linkage does not validate.")

    # Independently reconstruct the worker raw root from its frozen private raw manifest.
    raw_manifest_entries = [i for i in cp.get("WorkerOutputManifest", []) if "raw_manifest" in Path(str(i.get("Path", ""))).name]
    if len(raw_manifest_entries) != 1:
        raise RuntimeError(f"Worker {tag} output manifest does not identify exactly one private raw manifest.")
    wm_path = Path(str(raw_manifest_entries[0]["Path"]))
    wm = pd.read_csv(wm_path, low_memory=False)
    if not {"RelativePath", "Bytes", "SHA256"}.issubset(wm.columns):
        raise RuntimeError(f"Worker {tag} raw manifest schema differs.")
    wm = wm[["RelativePath", "Bytes", "SHA256"]].copy()
    wm["Bytes"] = pd.to_numeric(wm["Bytes"], errors="raise").astype("int64")
    if len(wm) != EXPECTED_WORKER_RAW_FILES or int(wm["Bytes"].sum()) != expected["RawBytes"]:
        raise RuntimeError(f"Worker {tag} private raw manifest counts differ.")
    if directory_root_hash(wm) != expected["RawRootSHA256"]:
        raise RuntimeError(f"Worker {tag} private raw manifest does not reproduce its frozen raw-root hash.")

    worker_records.append({
        "WorkerTag": tag,
        "Status": cp.get("Status"),
        "AssignedSeeds": json.dumps(seeds),
        "Conditions": int(cp["CompletedConditions"]),
        "MLFits": int(cp["MLFits"]),
        "RankingRows": int(cp["RankingRows"]),
        "BuildMetricRows": int(cp["BuildMetricRows"]),
        "ProjectRunRows": int(cp["ProjectRunRows"]),
        "TrainingMedianRows": int(cp["TrainingMedianRows"]),
        "RawFiles": int(cp["WorkerRawFiles"]),
        "RawBytes": int(cp["WorkerRawBytes"]),
        "RawRootSHA256": str(cp["WorkerRawRootSHA256"]),
        "CheckpointSHA256": actual_sha,
        "ValidationChecks": int(cp.get("ValidationChecks", 0)),
        "FailedValidationChecks": int(cp.get("FailedValidationChecks", 0)),
    })

if worker_manifest_failures:
    display(pd.DataFrame(worker_manifest_failures))
    raise RuntimeError("One or more worker output manifests no longer validate.")
if sorted(worker_seed_union) != REPETITION_SEEDS or len(worker_seed_union) != len(set(worker_seed_union)):
    raise RuntimeError(f"Worker shards do not cover seeds 1–30 exactly once: {worker_seed_union}")
worker_audit = pd.DataFrame(worker_records)

# --------------------------------------------------------------------------------------------------
# 7. VERIFY SHARED SMOKE-EQUIVALENCE RECORD
# --------------------------------------------------------------------------------------------------

equivalence = pd.read_csv(ACCELERATED_EQUIVALENCE_PATH, low_memory=False)
if "ConditionKey" not in equivalence.columns or "Pass" not in equivalence.columns:
    raise RuntimeError("Accelerated-equivalence file schema differs.")
if sorted(equivalence["ConditionKey"].astype(str).tolist()) != sorted(SMOKE_EQUIVALENCE_KEYS):
    raise RuntimeError("Accelerated-equivalence file does not contain exactly the two frozen smoke conditions.")
if not equivalence["Pass"].map(as_bool).all():
    raise RuntimeError("At least one accelerated-engine smoke-equivalence check failed.")

# --------------------------------------------------------------------------------------------------
# 8. INDEPENDENT READ-ONLY REVALIDATION OF ALL 270 CONDITIONS
# --------------------------------------------------------------------------------------------------

print("\nIndependently revalidating all 270 frozen condition directories (ZERO fitting).")
inventory_records = []
audit_frames = []
project_run_frames = []
build_metric_frames = []
model_fit_frames = []
baseline_records = []

for idx, plan_row in enumerate(condition_plan.itertuples(index=False), start=1):
    condition_dir = FULL_RAW_RESULT_ROOT / str(plan_row.ConditionID)
    validated = validate_completed_condition(condition_dir, plan_row)
    if validated is None:
        raise RuntimeError(f"Condition failed independent readback validation: {plan_row.ConditionID}")
    fingerprints = validated.pop("BaselineFingerprints")
    inventory_records.append(validated)

    audit = pd.read_csv(condition_dir / "condition_audit.csv", low_memory=False)
    project_runs = pd.read_csv(condition_dir / "project_runs.csv", low_memory=False)
    build_metrics = pd.read_csv(condition_dir / "build_metrics.csv", low_memory=False)
    model_fits = pd.read_csv(condition_dir / "model_fits.csv", low_memory=False)
    medians = pd.read_csv(condition_dir / "training_medians.csv", low_memory=False)
    if len(audit) != 1 or len(project_runs) != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION or len(build_metrics) != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION or len(model_fits) != EXPECTED_MODEL_FIT_ROWS_PER_CONDITION or len(medians) != EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION:
        raise RuntimeError(f"Compact output row counts differ for {plan_row.ConditionID}")
    for df, label in [(audit, "audit"), (project_runs, "project runs"), (build_metrics, "build metrics"), (model_fits, "model fits")]:
        if "ConditionKey" in df.columns and not df["ConditionKey"].astype(str).eq(str(plan_row.ConditionID)).all():
            raise RuntimeError(f"{plan_row.ConditionID}: {label} ConditionKey differs.")
    if "Technique" in project_runs.columns and sorted(project_runs["Technique"].astype(str).tolist()) != sorted(ALL_TECHNIQUES):
        raise RuntimeError(f"{plan_row.ConditionID}: project-run technique set differs.")
    if "Technique" in model_fits.columns and sorted(model_fits["Technique"].astype(str).tolist()) != sorted(ML_TECHNIQUES):
        raise RuntimeError(f"{plan_row.ConditionID}: model-fit technique set differs.")

    audit_frames.append(audit)
    project_run_frames.append(project_runs)
    build_metric_frames.append(build_metrics)
    model_fit_frames.append(model_fits)

    for technique in INVARIANT_BASELINES:
        fp = fingerprints[technique]
        baseline_records.append({
            "ConditionKey": str(plan_row.ConditionID),
            "NoisePercent": int(plan_row.NoisePercent),
            "RepetitionSeed": int(plan_row.RepetitionSeed),
            "Technique": technique,
            "FingerprintJSON": json.dumps(fp, sort_keys=True, separators=(",", ":")),
        })

    if idx % 30 == 0 or idx == EXPECTED_CONDITIONS:
        print(f"  Validated {idx}/{EXPECTED_CONDITIONS} conditions")

condition_inventory = pd.DataFrame(inventory_records).sort_values("ConditionOrder", kind="mergesort").reset_index(drop=True)
combined_condition_audit = pd.concat(audit_frames, ignore_index=True)
combined_project_runs = pd.concat(project_run_frames, ignore_index=True)
combined_build_metrics = pd.concat(build_metric_frames, ignore_index=True)
combined_model_fits = pd.concat(model_fit_frames, ignore_index=True)
baseline_detail = pd.DataFrame(baseline_records)

# Noise-invariant baseline fingerprints must be identical across all nine noise levels within each seed.
baseline_invariance = (
    baseline_detail.groupby(["RepetitionSeed", "Technique"], as_index=False)
    .agg(Conditions=("ConditionKey", "size"), FingerprintVariants=("FingerprintJSON", "nunique"))
)
baseline_invariance["ExpectedConditions"] = len(NOISE_LEVELS)
baseline_invariance["Pass"] = (
    baseline_invariance["Conditions"].eq(len(NOISE_LEVELS))
    & baseline_invariance["FingerprintVariants"].eq(1)
)

# --------------------------------------------------------------------------------------------------
# 9. HASH AND FREEZE THE COMPLETE RAW ROOT
# --------------------------------------------------------------------------------------------------

print("\nHashing the complete Project 22 raw-result root (2,160 files).")
raw_manifest = directory_manifest(FULL_RAW_RESULT_ROOT)
raw_root_sha256 = directory_root_hash(raw_manifest)
raw_files = int(len(raw_manifest))
raw_bytes = int(raw_manifest["Bytes"].sum())

# --------------------------------------------------------------------------------------------------
# 10. MASTER VALIDATION
# --------------------------------------------------------------------------------------------------

validation = []
add_check(validation, "Worker checkpoints", 6, len(worker_audit), len(worker_audit) == 6)
add_check(validation, "Worker seed coverage", list(range(1, 31)), sorted(worker_seed_union), sorted(worker_seed_union) == list(range(1, 31)) and len(worker_seed_union) == len(set(worker_seed_union)))
add_check(validation, "Conditions", EXPECTED_CONDITIONS, len(condition_inventory), len(condition_inventory) == EXPECTED_CONDITIONS)
add_check(validation, "Raw files", EXPECTED_RAW_FILES, raw_files, raw_files == EXPECTED_RAW_FILES)
add_check(validation, "Raw bytes", EXPECTED_RAW_BYTES, raw_bytes, raw_bytes == EXPECTED_RAW_BYTES)
add_check(validation, "Ranking rows", EXPECTED_TOTAL_RANKING_ROWS, int(condition_inventory["RankingRows"].sum()), int(condition_inventory["RankingRows"].sum()) == EXPECTED_TOTAL_RANKING_ROWS)
add_check(validation, "Build-metric rows", EXPECTED_TOTAL_BUILD_METRIC_ROWS, len(combined_build_metrics), len(combined_build_metrics) == EXPECTED_TOTAL_BUILD_METRIC_ROWS)
add_check(validation, "Project-run rows", EXPECTED_TOTAL_PROJECT_RUN_ROWS, len(combined_project_runs), len(combined_project_runs) == EXPECTED_TOTAL_PROJECT_RUN_ROWS)
add_check(validation, "Model-fit rows", EXPECTED_TOTAL_MODEL_FITS, len(combined_model_fits), len(combined_model_fits) == EXPECTED_TOTAL_MODEL_FITS)
add_check(validation, "Condition-audit rows", EXPECTED_TOTAL_CONDITION_AUDIT_ROWS, len(combined_condition_audit), len(combined_condition_audit) == EXPECTED_TOTAL_CONDITION_AUDIT_ROWS)
add_check(validation, "Training-median rows", EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS, int(condition_inventory["TrainingMedianRows"].sum()), int(condition_inventory["TrainingMedianRows"].sum()) == EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS)
add_check(validation, "Baseline invariance rows", 60, len(baseline_invariance), len(baseline_invariance) == 60)
add_check(validation, "Baseline invariance failures", 0, int((~baseline_invariance["Pass"]).sum()), int((~baseline_invariance["Pass"]).sum()) == 0)
add_check(validation, "Independent REC changes", 0, int(condition_inventory["IndependentRECChanges"].sum()), int(condition_inventory["IndependentRECChanges"].sum()) == 0)

zero = condition_inventory["NoisePercent"].eq(0)
pos = condition_inventory["NoisePercent"].gt(0)
add_check(validation, "Zero-noise raw flips", 0, int(condition_inventory.loc[zero, "NumberFlipped"].sum()), int(condition_inventory.loc[zero, "NumberFlipped"].sum()) == 0)
add_check(validation, "Zero-noise model-label changes", 0, int(condition_inventory.loc[zero, "ModelLabelChanges"].sum()), int(condition_inventory.loc[zero, "ModelLabelChanges"].sum()) == 0)
add_check(validation, "Zero-noise dependent REC changes", 0, int(condition_inventory.loc[zero, "DependentRECChanges"].sum()), int(condition_inventory.loc[zero, "DependentRECChanges"].sum()) == 0)
add_check(validation, "Positive-noise conditions without raw flips", 0, int(condition_inventory.loc[pos, "NumberFlipped"].le(0).sum()), int(condition_inventory.loc[pos, "NumberFlipped"].le(0).sum()) == 0)
add_check(validation, "Positive-noise conditions without model-label changes", 0, int(condition_inventory.loc[pos, "ModelLabelChanges"].le(0).sum()), int(condition_inventory.loc[pos, "ModelLabelChanges"].le(0).sum()) == 0)

# Audit hash equality and independent reconstruction invariance.
for expected_col, actual_col, label in [
    ("ExpectedFlipMaskSHA256", "ActualFlipMaskSHA256", "Flip-mask hash mismatches"),
    ("ExpectedNoisyRawVerdictSHA256", "ActualNoisyRawVerdictSHA256", "Noisy raw-verdict hash mismatches"),
    ("ExpectedNoisyModelVerdictSHA256", "ActualNoisyModelVerdictSHA256", "Noisy model-verdict hash mismatches"),
]:
    if expected_col not in combined_condition_audit.columns or actual_col not in combined_condition_audit.columns:
        add_check(validation, label, 0, "missing audit columns", False)
    else:
        mismatches = int((combined_condition_audit[expected_col].astype(str) != combined_condition_audit[actual_col].astype(str)).sum())
        add_check(validation, label, 0, mismatches, mismatches == 0)
if "IndependentReconstructionMismatches" in combined_condition_audit.columns:
    value = int(pd.to_numeric(combined_condition_audit["IndependentReconstructionMismatches"], errors="raise").sum())
    add_check(validation, "Independent reconstruction mismatches", 0, value, value == 0)
else:
    add_check(validation, "Independent reconstruction mismatches", 0, "missing column", False)

# Metric sanity: thesis metrics must be finite and within [0,1].
for df, cols, label in [
    (combined_project_runs, ["MeanAPFDc", "MedianAPFDc", "MeanAPFD", "MedianAPFD"], "Project metrics"),
    (combined_build_metrics, ["APFDc", "APFD"], "Build metrics"),
]:
    missing_cols = [c for c in cols if c not in df.columns]
    if missing_cols:
        add_check(validation, f"{label} columns", cols, missing_cols, False)
    else:
        arr = df[cols].apply(pd.to_numeric, errors="coerce").to_numpy(float)
        bad = int((~np.isfinite(arr)).sum() + ((arr < 0) | (arr > 1)).sum())
        add_check(validation, f"{label} non-finite/out-of-range values", 0, bad, bad == 0)

# Model fits must match the exact frozen Project 22 worker contract.
# Every worker condition validates Status == "PASS_MODEL_FIT" before it can freeze.
if "Status" in combined_model_fits.columns:
    status_text = combined_model_fits["Status"].astype(str).str.strip()
    bad_status = int((~status_text.eq("PASS_MODEL_FIT")).sum())
    add_check(validation, "Model-fit status failures", 0, bad_status, bad_status == 0)
else:
    add_check(validation, "Model-fit Status column present", True, False, False)

add_check(validation, "Registry SHA unchanged", EXPECTED_REGISTRY_SHA256, sha256_file(REGISTRY_PATH), sha256_file(REGISTRY_PATH) == EXPECTED_REGISTRY_SHA256)
add_check(validation, "Source root unchanged", EXPECTED_SOURCE_ROOT_SHA256, source_sha_before, source_sha_before == EXPECTED_SOURCE_ROOT_SHA256)
add_check(validation, "Shared smoke-equivalence failures", 0, int((~equivalence["Pass"].map(as_bool)).sum()), int((~equivalence["Pass"].map(as_bool)).sum()) == 0)
add_check(validation, "Master finalizer model fits", 0, 0, True)
add_check(validation, "Master finalizer condition executions", 0, 0, True)

step5a_validation = pd.DataFrame(validation)
failed_validation = step5a_validation.loc[~step5a_validation["Pass"]]
print("\nMaster Step 5A validation:")
display(step5a_validation)
if not failed_validation.empty:
    print("\nFAILED CHECKS:")
    display(failed_validation)
    raise RuntimeError("PROJECT 22 STEP 5A MASTER FINALIZATION VALIDATION FAILED. No official Step 5A freeze was written.")

# --------------------------------------------------------------------------------------------------
# 11. WRITE OFFICIAL STEP 5A AGGREGATES ONLY AFTER ALL CHECKS PASS
# --------------------------------------------------------------------------------------------------

atomic_csv(CONDITION_INVENTORY_PATH, condition_inventory)
atomic_csv(RAW_MANIFEST_PATH, raw_manifest)
atomic_csv(BASELINE_INVARIANCE_PATH, baseline_invariance)
atomic_csv(COMBINED_CONDITION_AUDIT_PATH, combined_condition_audit)
atomic_csv(COMBINED_PROJECT_RUNS_PATH, combined_project_runs)
atomic_csv(COMBINED_BUILD_METRICS_PATH, combined_build_metrics)
atomic_csv(COMBINED_MODEL_FITS_PATH, combined_model_fits)
atomic_csv(WORKER_CHECKPOINT_AUDIT_PATH, worker_audit)
atomic_csv(STEP5A_VALIDATION_PATH, step5a_validation)

aggregate_output_paths = [
    CONDITION_INVENTORY_PATH, RAW_MANIFEST_PATH, BASELINE_INVARIANCE_PATH,
    COMBINED_CONDITION_AUDIT_PATH, COMBINED_PROJECT_RUNS_PATH,
    COMBINED_BUILD_METRICS_PATH, COMBINED_MODEL_FITS_PATH,
    WORKER_CHECKPOINT_AUDIT_PATH, STEP5A_VALIDATION_PATH,
    ACCELERATED_EQUIVALENCE_PATH,
]
aggregate_output_manifest = [
    {"Path": str(p), "Bytes": int(p.stat().st_size), "SHA256": sha256_file(p)}
    for p in aggregate_output_paths
]

completed_at = datetime.now(timezone.utc).isoformat()
execution_seconds = time.perf_counter() - started
report = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5A_STATUS,
    "Implementation": MASTER_FINALIZER_IMPLEMENTATION,
    "CompletedAtUTC": completed_at,
    "ExecutionMode": "PARALLEL_SIX_WORKERS_THEN_ZERO_FIT_MASTER_FINALIZATION",
    "AcceleratedEngineVersion": ACCELERATED_ENGINE_VERSION,
    "SelectionCheckpointSHA256": EXPECTED_SELECTION_CHECKPOINT_SHA256,
    "RECCheckpointSHA256": EXPECTED_REC_CHECKPOINT_SHA256,
    "NoisePlanCheckpointSHA256": EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
    "RuntimeCheckpointSHA256": EXPECTED_RUNTIME_CHECKPOINT_SHA256,
    "SmokeCheckpointSHA256": EXPECTED_SMOKE_CHECKPOINT_SHA256,
    "ExpectedWorkerCheckpointSHA256": {tag: data["SHA256"] for tag, data in EXPECTED_WORKER_CHECKPOINTS.items()},
    "WorkerCheckpoints": worker_records,
    "Conditions": len(condition_inventory),
    "NoiseLevels": NOISE_LEVELS,
    "RepetitionSeeds": REPETITION_SEEDS,
    "MLFits": len(combined_model_fits),
    "RankingRows": int(condition_inventory["RankingRows"].sum()),
    "BuildMetricRows": len(combined_build_metrics),
    "ProjectRunRows": len(combined_project_runs),
    "ConditionAuditRows": len(combined_condition_audit),
    "TrainingMedianRows": int(condition_inventory["TrainingMedianRows"].sum()),
    "RawFiles": raw_files,
    "RawBytes": raw_bytes,
    "RawRootSHA256": raw_root_sha256,
    "AggregateOutputManifest": aggregate_output_manifest,
    "ValidationChecks": len(step5a_validation),
    "FailedValidationChecks": 0,
    "SourceRootSHA256": EXPECTED_SOURCE_ROOT_SHA256,
    "RegistrySHA256": EXPECTED_REGISTRY_SHA256,
    "RegistryModified": False,
    "Projects1To21Modified": False,
    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectConditionOutputsModified": False,
    "EvaluationCohortImmutable": True,
    "ActiveReservations": EXPECTED_ACTIVE_RESERVATIONS,
    "RuntimePriorityRule": EXPECTED_RUNTIME_PRIORITY_RULE,
    "ResumeSafe": True,
    "ParallelExecution": True,
    "MasterFinalizationModelFits": 0,
    "MasterFinalizationConditionExecutions": 0,
    "SourceRestoredInThisInvocation": source_restored,
    "ExecutionSecondsThisInvocation": float(execution_seconds),
}
atomic_json(STEP5A_REPORT_PATH, report)

checkpoint = {
    **report,
    "CheckpointVersion": 1,
    "Step5AReportPath": str(STEP5A_REPORT_PATH),
    "Step5AReportSHA256": sha256_file(STEP5A_REPORT_PATH),
    "Full270ConditionExperimentComplete": True,
    "RawResultRootFrozen": True,
    "DoNotRerunCompletedConditions": True,
    "OfficialStep5AComplete": True,
    "NextRequiredStep": "STEP_5B_RAW_REVALIDATION_AND_COMPACT_AGGREGATION",
}
atomic_json(STEP5A_CHECKPOINT_PATH, checkpoint)

status_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5A_STATUS,
    "CompletedAtUTC": completed_at,
    "Conditions": EXPECTED_CONDITIONS,
    "MLFits": EXPECTED_TOTAL_MODEL_FITS,
    "RawFiles": EXPECTED_RAW_FILES,
    "RawBytes": EXPECTED_RAW_BYTES,
    "RawRootSHA256": raw_root_sha256,
    "Checkpoint": str(STEP5A_CHECKPOINT_PATH),
    "CheckpointSHA256": sha256_file(STEP5A_CHECKPOINT_PATH),
    "RegistryModified": False,
    "OfficialStep5AComplete": True,
}
atomic_json(STEP5A_STATUS_PATH, status_payload)

atomic_json(RUN_PROGRESS_PATH, {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5A_STATUS,
    "UpdatedAtUTC": completed_at,
    "CompletedConditions": EXPECTED_CONDITIONS,
    "ExpectedConditions": EXPECTED_CONDITIONS,
    "CheckpointSHA256": sha256_file(STEP5A_CHECKPOINT_PATH),
    "ResumeSafe": True,
    "OfficialStep5AComplete": True,
    "NextRequiredStep": "STEP_5B_RAW_REVALIDATION_AND_COMPACT_AGGREGATION",
})

# --------------------------------------------------------------------------------------------------
# 12. FINAL READBACK / IMMUTABILITY PROOF
# --------------------------------------------------------------------------------------------------

if load_json(STEP5A_CHECKPOINT_PATH).get("Status") != STEP5A_STATUS:
    raise RuntimeError("Official Project 22 Step 5A checkpoint readback failed.")
if load_json(STEP5A_STATUS_PATH).get("Status") != STEP5A_STATUS:
    raise RuntimeError("Official Project 22 Step 5A status readback failed.")
final_manifest_failures = verify_manifest_entries(load_json(STEP5A_CHECKPOINT_PATH).get("AggregateOutputManifest", []), "official Step5A aggregate")
if final_manifest_failures:
    display(pd.DataFrame(final_manifest_failures))
    raise RuntimeError("Official Step 5A aggregate-output manifest readback failed.")
if sha256_file(REGISTRY_PATH) != registry_sha_before:
    raise RuntimeError("Completion registry changed during master finalization.")
for p, before in cohort_sha_before.items():
    if sha256_file(Path(p)) != before:
        raise RuntimeError(f"Frozen Project 22 cohort/plan changed during master finalization: {p}")
final_source = pd.DataFrame([
    {"RelativePath": str(r.RelativePath), "SizeBytes": int((SOURCE_DIR / str(r.RelativePath)).stat().st_size), "SHA256": sha256_file(SOURCE_DIR / str(r.RelativePath))}
    for r in frozen_source_manifest.itertuples(index=False)
])
if source_root_hash(final_source) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError("Frozen Project 22 source changed during master finalization.")

# Raw files are never written by this finalizer; verify count/bytes again after official writes.
raw_file_paths_after = [p for p in FULL_RAW_RESULT_ROOT.rglob("*") if p.is_file()]
if len(raw_file_paths_after) != EXPECTED_RAW_FILES or sum(p.stat().st_size for p in raw_file_paths_after) != EXPECTED_RAW_BYTES:
    raise RuntimeError("Project 22 raw root count/bytes changed during master finalization.")

print("\n" + "=" * 136)
print("=== PROJECT 22 CELL 9 / STEP 5A RESULT ===")
print("=" * 136)
print("Project:", PROJECT_NAME)
print("Workers verified:", len(worker_audit), "/ 6")
print("Seeds covered exactly once:", sorted(worker_seed_union) == REPETITION_SEEDS)
print("Conditions:", len(condition_inventory), "/", EXPECTED_CONDITIONS)
print("ML fits:", len(combined_model_fits), "/", EXPECTED_TOTAL_MODEL_FITS)
print("Ranking rows:", int(condition_inventory["RankingRows"].sum()), "/", EXPECTED_TOTAL_RANKING_ROWS)
print("Build-metric rows:", len(combined_build_metrics), "/", EXPECTED_TOTAL_BUILD_METRIC_ROWS)
print("Project-run rows:", len(combined_project_runs), "/", EXPECTED_TOTAL_PROJECT_RUN_ROWS)
print("Raw files:", raw_files, "/", EXPECTED_RAW_FILES)
print("Raw bytes:", raw_bytes, "/", EXPECTED_RAW_BYTES)
print("Raw root SHA-256:", raw_root_sha256)
print("Baseline invariance failures:", int((~baseline_invariance["Pass"]).sum()))
print("Validation checks:", len(step5a_validation))
print("Failed checks:", int((~step5a_validation["Pass"]).sum()))
print("Master-finalizer model fits:", 0)
print("Master-finalizer condition executions:", 0)
print("Registry modified:", False)
print("Checkpoint:", STEP5A_CHECKPOINT_PATH)
print("Checkpoint SHA-256:", sha256_file(STEP5A_CHECKPOINT_PATH))
print("Next required step: STEP 5B — RAW REVALIDATION AND COMPACT AGGREGATION")
print("STATUS:", STEP5A_STATUS)
print("=" * 136)


=== PROJECT 22 CELL 9 / STEP 5A: PARALLEL MASTER FINALIZATION ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Independently revalidating all 270 frozen condition directories (ZERO fitting).
  Validated 30/270 conditions
  Validated 60/270 conditions
  Validated 90/270 conditions
  Validated 120/270 conditions
  Validated 150/270 conditions
  Validated 180/270 conditions
  Validated 210/270 conditions
  Validated 240/270 conditions
  Validated 270/270 conditions

Hashing the complete Project 22 raw-result root (2,160 files).

Master Step 5A validation:


,Check,Expected,Actual,Pass
0,Worker checkpoints,6,6,True
1,Worker seed coverage,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",True
2,Conditions,270,270,True
3,Raw files,2160,2160,True
4,Raw bytes,646130653,646130653,True
5,Ranking rows,41874840,41874840,True
6,Build-metric rows,73710,73710,True
7,Project-run rows,1890,1890,True
8,Model-fit rows,1080,1080,True
9,Condition-audit rows,270,270,True



=== PROJECT 22 CELL 9 / STEP 5A RESULT ===
Project: apache@logging-log4j2
Workers verified: 6 / 6
Seeds covered exactly once: True
Conditions: 270 / 270
ML fits: 1080 / 1080
Ranking rows: 41874840 / 41874840
Build-metric rows: 73710 / 73710
Project-run rows: 1890 / 1890
Raw files: 2160 / 2160
Raw bytes: 646130653 / 646130653
Raw root SHA-256: 374e4e1eaab266451539c9c7a4751fa6fe50250dccf57b19a5bc19cf8fad3f11
Baseline invariance failures: 0
Validation checks: 31
Failed checks: 0
Master-finalizer model fits: 0
Master-finalizer condition executions: 0
Registry modified: False
Checkpoint: /content/drive/MyDrive/Thesis_Experiment/Notes/project_22_step5a_checkpoint.json
Checkpoint SHA-256: 600035cfbf9f8b2dfd0fd83d49050466e3c71e6187c1d631b24cfe57a6107a26
Next required step: STEP 5B — RAW REVALIDATION AND COMPACT AGGREGATION
STATUS: PASS_PROJECT_22_FULL_270_CONDITION_EXPERIMENT_COMPLETE


In [3]:
# ==================================================================================================
# PROJECT 22 — CELL 10 / STEP 5B
# PROJECT-SPECIFIC RAW REVALIDATION AND COMPACT AGGREGATION
#
# PROJECT:
#   apache@logging-log4j2
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_22.ipynb.
#
# CURRENT REGISTRY CONTRACT:
# - Projects 1–21 must be present exactly once and COMPLETE_AND_FROZEN.
# - Project 22 must still be absent.
#
# THIS CELL:
# - independently hashes all 2,160 Project 22 raw files;
# - validates every condition checkpoint, summary, embedded manifest, and compact output;
# - independently recounts all 41,874,840 compressed ranking rows;
# - independently validates noise hashes, REC invariance, APFDc/APFD metrics, and baselines;
# - creates analysis-ready aggregates across all 30 seeds;
# - writes the Project 22 Step 5B checkpoint;
# - does not rerun conditions or fit models;
# - does not access or modify prior-project condition outputs;
# - does not register Project 22.
#
# BASELINE-INVARIANCE CONTRACT:
# - Random and QTF-Avg are compared independently within each metric;
# - APFDc and APFD are never compared against one another.
# ==================================================================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import gzip
import hashlib
import json
import os
import time

import numpy as np
import pandas as pd


print("=" * 136)
print("=== PROJECT 22 CELL 10 / STEP 5B: RAW REVALIDATION AND COMPACT AGGREGATION ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN PROJECT 22 CONTRACT
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 22
PROJECT_NAME = "apache@logging-log4j2"
PROJECT_SLUG = "apache__logging-log4j2"
PROJECT_SHORT = "LOG4J2"

STEP5A_STATUS = "PASS_PROJECT_22_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
CONDITION_STATUS = "PASS_FULL_CONDITION"
STEP5B_STATUS = "PASS_PROJECT_22_RAW_RESULTS_REVALIDATED_AND_COMPACT_AGGREGATES_FROZEN"

EXPECTED_STEP5A_SHA = "600035cfbf9f8b2dfd0fd83d49050466e3c71e6187c1d631b24cfe57a6107a26"
EXPECTED_RAW_ROOT_SHA = "374e4e1eaab266451539c9c7a4751fa6fe50250dccf57b19a5bc19cf8fad3f11"
EXPECTED_REGISTRY_SHA = "79cd6ecb595c5e8ae91a9494e469792716338d144308560a62caf1b9342306b2"
EXPECTED_SOURCE_ROOT_SHA = "281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64f842964c896c6ac334"

EXPECTED_ACTIVE_RESERVATIONS = []
EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
SEEDS = list(range(1, 31))

TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
    "Random",
    "LatestFail",
    "QTF-Avg",
]
ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]
INVARIANT_BASELINES = [
    "Random",
    "QTF-Avg",
]

PROJECT_METRICS = [
    "MeanAPFDc",
    "MedianAPFDc",
    "MeanAPFD",
    "MedianAPFD",
]
BUILD_METRICS = [
    "APFDc",
    "APFD",
]

EXPECTED_CONDITIONS = 270
EXPECTED_FILES_PER_CONDITION = 8
EXPECTED_RAW_FILES = 2_160
EXPECTED_RAW_BYTES = 646_130_653

EXPECTED_EVALUATION_ROWS = 22_156
EXPECTED_SCORED_FAILING_BUILDS = 39
EXPECTED_EVALUATION_BUILDS = 111
EXPECTED_EVALUATION_FAILURES = 40
EXPECTED_PREDICTORS = 151

EXPECTED_RANKING_ROWS_PER_CONDITION = EXPECTED_EVALUATION_ROWS * len(TECHNIQUES)       # 155,092
EXPECTED_BUILD_ROWS_PER_CONDITION = EXPECTED_SCORED_FAILING_BUILDS * len(TECHNIQUES) # 273
EXPECTED_PROJECT_ROWS_PER_CONDITION = len(TECHNIQUES)                                # 7
EXPECTED_FIT_ROWS_PER_CONDITION = len(ML_TECHNIQUES)                                 # 4
EXPECTED_MEDIAN_ROWS_PER_CONDITION = EXPECTED_PREDICTORS                             # 151

EXPECTED_TOTAL_RANKING_ROWS = 41_874_840
EXPECTED_TOTAL_BUILD_ROWS = 73_710
EXPECTED_TOTAL_PROJECT_ROWS = 1_890
EXPECTED_TOTAL_FIT_ROWS = 1_080
EXPECTED_TOTAL_AUDIT_ROWS = 270
EXPECTED_TOTAL_MEDIAN_ROWS = 40_770

EXPECTED_NOISE_TECHNIQUE_SUMMARY_ROWS = len(NOISE_LEVELS) * len(TECHNIQUES) # 63
EXPECTED_SEED_DELTA_ROWS = EXPECTED_TOTAL_PROJECT_ROWS                       # 1,890
EXPECTED_NOISE_DELTA_SUMMARY_ROWS = EXPECTED_NOISE_TECHNIQUE_SUMMARY_ROWS    # 63

EXPECTED_CONDITION_FILES = {
    "rankings.csv.gz",
    "build_metrics.csv",
    "project_runs.csv",
    "model_fits.csv",
    "training_medians.csv",
    "condition_audit.csv",
    "condition_summary.json",
    "COMPLETE.json",
}
CONDITION_OUTPUT_FILES = EXPECTED_CONDITION_FILES - {
    "condition_summary.json",
    "COMPLETE.json",
}

REQUIRED_REGISTERED_IDENTITIES = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
}


# --------------------------------------------------------------------------------------------------
# 2. DRIVE PATHS
# --------------------------------------------------------------------------------------------------

drive.mount("/content/drive", force_remount=False)

ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES = ROOT / "Notes"
RESULTS = ROOT / "Results"

REGISTRY = NOTES / "completed_project_registry.csv"

PROJECT_ROOT = RESULTS / "Aggregated" / PROJECT_SLUG
RAW_ROOT = RESULTS / "Raw" / PROJECT_SLUG
FULL_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_full_experiment"

PLAN = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

STEP5A_CHECKPOINT = NOTES / "project_22_step5a_checkpoint.json"
STEP5A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5a_status.json"
STEP5A_REPORT = FULL_ROOT / f"{PROJECT_SHORT}_step5a_report.json"
STEP5A_RAW_MANIFEST = FULL_ROOT / f"{PROJECT_SHORT}_raw_manifest.csv"
STEP5A_BASELINE = FULL_ROOT / f"{PROJECT_SHORT}_baseline_invariance.csv"

OUT = PROJECT_ROOT / f"{PROJECT_SHORT}_step5b"

CURRENT_MANIFEST = OUT / f"{PROJECT_SHORT}_independent_raw_manifest.csv"
CONDITION_INVENTORY = OUT / f"{PROJECT_SHORT}_independent_condition_inventory.csv"
REVALIDATED_PROJECT_RUNS = OUT / f"{PROJECT_SHORT}_revalidated_project_runs.csv"
REVALIDATED_BUILD_METRICS = OUT / f"{PROJECT_SHORT}_revalidated_build_metrics.csv"
REVALIDATED_MODEL_FITS = OUT / f"{PROJECT_SHORT}_revalidated_model_fits.csv"
REVALIDATED_CONDITION_AUDIT = OUT / f"{PROJECT_SHORT}_revalidated_condition_audit.csv"
REVALIDATED_MEDIANS = OUT / f"{PROJECT_SHORT}_revalidated_training_medians.csv"
NOISE_SUMMARY = OUT / f"{PROJECT_SHORT}_noise_technique_summary.csv"
SEED_DELTAS = OUT / f"{PROJECT_SHORT}_seed_level_noise_deltas.csv"
DELTA_SUMMARY = OUT / f"{PROJECT_SHORT}_noise_delta_summary.csv"
VALIDATION_PATH = OUT / f"{PROJECT_SHORT}_step5b_validation.csv"
REPORT_PATH = OUT / f"{PROJECT_SHORT}_step5b_report.json"

STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5b_status.json"
CHECKPOINT_PATH = NOTES / "project_22_step5b_checkpoint.json"


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    with tmp.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(tmp, path)


def atomic_csv(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_csv(
        tmp,
        index=False,
        lineterminator="\n",
    )

    os.replace(tmp, path)


def resolve_col(columns, names, label):
    lookup = {
        str(column).strip().lower(): column
        for column in columns
    }

    for name in names:
        if str(name).lower() in lookup:
            return lookup[str(name).lower()]

    raise RuntimeError(
        f"Could not resolve {label}; columns={list(columns)}"
    )


def normalize_manifest(frame, label):
    path_col = resolve_col(
        frame.columns,
        ["RelativePath"],
        f"{label} path",
    )
    bytes_col = resolve_col(
        frame.columns,
        ["Bytes", "SizeBytes"],
        f"{label} bytes",
    )
    sha_col = resolve_col(
        frame.columns,
        ["SHA256"],
        f"{label} SHA256",
    )

    out = frame[
        [
            path_col,
            bytes_col,
            sha_col,
        ]
    ].copy()

    out.columns = [
        "RelativePath",
        "Bytes",
        "SHA256",
    ]

    out["RelativePath"] = (
        out["RelativePath"]
        .astype(str)
        .str.replace("\\", "/", regex=False)
    )
    out["Bytes"] = pd.to_numeric(
        out["Bytes"],
        errors="raise",
    ).astype("int64")
    out["SHA256"] = (
        out["SHA256"]
        .astype(str)
        .str.lower()
    )

    return (
        out.sort_values(
            "RelativePath",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


def root_hash(manifest):
    digest = hashlib.sha256()

    for row in manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode("utf-8")
        )

    return digest.hexdigest()


def gzip_rows(path):
    # Independent physical row recount of the compressed ranking CSV.
    rows = 0

    with gzip.open(path, "rb") as handle:
        for _ in handle:
            rows += 1

    return max(0, rows - 1)


def add_check(rows, name, expected, actual, passed):
    rows.append({
        "Check": name,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })


def metric_nonfinite(frame, columns):
    values = (
        frame[columns]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .to_numpy(dtype=float)
    )

    return int((~np.isfinite(values)).sum())


def metric_outside(frame, columns):
    values = (
        frame[columns]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .to_numpy(dtype=float)
    )

    return int(
        (
            (values < 0)
            | (values > 1)
        ).sum()
    )


def pass_series(values):
    if values.dtype == bool:
        return values

    parsed = (
        values.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
    )

    if parsed.isna().any():
        raise RuntimeError(
            "Could not parse a frozen baseline Pass column."
        )

    return parsed.astype(bool)


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY,
    PLAN,
    STEP5A_CHECKPOINT,
    STEP5A_STATUS_PATH,
    STEP5A_REPORT,
    STEP5A_RAW_MANIFEST,
    STEP5A_BASELINE,
    RAW_ROOT,
]

missing = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing Project 22 Step 5B inputs:\n"
        + "\n".join(missing)
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY THE OFFICIAL STEP 5A FREEZE AND REGISTRY
# --------------------------------------------------------------------------------------------------

step5a_sha = sha256_file(
    STEP5A_CHECKPOINT
)
step5a = load_json(
    STEP5A_CHECKPOINT
)
step5a_status = load_json(
    STEP5A_STATUS_PATH
)
step5a_report = load_json(
    STEP5A_REPORT
)

if step5a_sha != EXPECTED_STEP5A_SHA:
    raise RuntimeError(
        "Step 5A checkpoint SHA differs.\n"
        f"Expected: {EXPECTED_STEP5A_SHA}\n"
        f"Actual:   {step5a_sha}"
    )

for label, payload in [
    ("checkpoint", step5a),
    ("status", step5a_status),
    ("report", step5a_report),
]:
    if payload.get("Status") != STEP5A_STATUS:
        raise RuntimeError(
            f"Step 5A {label} is not in the frozen PASS state."
        )

if step5a.get("Project") != PROJECT_NAME:
    raise RuntimeError(
        "Step 5A checkpoint project identity differs."
    )

if step5a.get("ProjectSlug") != PROJECT_SLUG:
    raise RuntimeError(
        "Step 5A checkpoint project slug differs."
    )

if step5a.get("SourceRootSHA256") != EXPECTED_SOURCE_ROOT_SHA:
    raise RuntimeError(
        "Step 5A source-root SHA differs."
    )

if step5a.get("RawRootSHA256") != EXPECTED_RAW_ROOT_SHA:
    raise RuntimeError(
        "Step 5A frozen raw-root SHA differs."
    )

if int(step5a.get("RawFiles", -1)) != EXPECTED_RAW_FILES:
    raise RuntimeError(
        "Step 5A raw-file count differs."
    )

if int(step5a.get("RawBytes", -1)) != EXPECTED_RAW_BYTES:
    raise RuntimeError(
        "Step 5A raw-byte count differs."
    )

if int(step5a.get("Conditions", -1)) != EXPECTED_CONDITIONS:
    raise RuntimeError(
        "Step 5A condition count differs."
    )

if int(step5a.get("MLFits", -1)) != EXPECTED_TOTAL_FIT_ROWS:
    raise RuntimeError(
        "Step 5A model-fit count differs."
    )

if step5a.get("ActiveReservations") != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Step 5A active-reservation contract differs."
    )

if step5a.get("RuntimePriorityRule") != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Step 5A runtime-priority rule differs."
    )

if not bool(
    step5a.get(
        "OfficialStep5AComplete",
        False,
    )
):
    raise RuntimeError(
        "Step 5A checkpoint is not marked officially complete."
    )

registry_sha_before = sha256_file(
    REGISTRY
)

if registry_sha_before != EXPECTED_REGISTRY_SHA:
    raise RuntimeError(
        "Registry SHA differs before Project 22 Step 5B.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA}\n"
        f"Actual:   {registry_sha_before}"
    )

registry = pd.read_csv(
    REGISTRY,
    dtype=str,
).fillna("")

project_number_col = resolve_col(
    registry.columns,
    [
        "ProjectNumber",
        "project_number",
        "ProjectNo",
        "Project No",
    ],
    "registry ProjectNumber",
)
project_col = resolve_col(
    registry.columns,
    [
        "Project",
        "ProjectName",
        "project_name",
    ],
    "registry Project",
)
status_col = resolve_col(
    registry.columns,
    [
        "Status",
        "ProjectStatus",
        "project_status",
    ],
    "registry Status",
)

project_numbers = pd.to_numeric(
    registry[project_number_col],
    errors="raise",
).astype(int)

if (
    len(registry) != 21
    or sorted(project_numbers.tolist())
    != list(range(1, 22))
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–21 "
        "before Project 22 Step 5B."
    )

if not registry[
    status_col
].astype(str).eq(
    "COMPLETE_AND_FROZEN"
).all():
    raise RuntimeError(
        "Projects 1–21 are not all COMPLETE_AND_FROZEN."
    )

for required_number, required_project in REQUIRED_REGISTERED_IDENTITIES.items():
    matching = registry.loc[
        project_numbers.eq(
            required_number
        )
    ]

    if (
        len(matching) != 1
        or str(
            matching.iloc[
                0
            ][
                project_col
            ]
        )
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different "
            "registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )

if (
    project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_col
    ].astype(str).eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 22 is unexpectedly already present "
        "in the completion registry."
    )


# --------------------------------------------------------------------------------------------------
# 6. VERIFY STEP 5A AGGREGATE MANIFEST
# --------------------------------------------------------------------------------------------------

aggregate_manifest = step5a.get(
    "AggregateOutputManifest",
    [],
)

if (
    not isinstance(
        aggregate_manifest,
        list,
    )
    or not aggregate_manifest
):
    raise RuntimeError(
        "Step 5A checkpoint has no AggregateOutputManifest."
    )

aggregate_manifest_failures = 0

for item in aggregate_manifest:
    path = Path(
        item["Path"]
    )

    passed = bool(
        path.is_file()
        and int(
            path.stat().st_size
        )
        == int(
            item["Bytes"]
        )
        and sha256_file(
            path
        )
        == str(
            item["SHA256"]
        ).lower()
    )

    aggregate_manifest_failures += int(
        not passed
    )

if aggregate_manifest_failures:
    raise RuntimeError(
        f"{aggregate_manifest_failures} frozen "
        "Step 5A aggregate outputs changed."
    )


# --------------------------------------------------------------------------------------------------
# 7. INDEPENDENTLY HASH ALL 2,160 RAW FILES
# --------------------------------------------------------------------------------------------------

print(
    "\nIndependently hashing all 2,160 raw files."
)

hash_start = time.perf_counter()

raw_paths = sorted(
    [
        path
        for path in RAW_ROOT.rglob("*")
        if path.is_file()
    ],
    key=lambda path: (
        path.relative_to(
            RAW_ROOT
        ).as_posix()
    ),
)

manifest_rows = []

for index, path in enumerate(
    raw_paths,
    start=1,
):
    manifest_rows.append({
        "RelativePath": (
            path.relative_to(
                RAW_ROOT
            ).as_posix()
        ),
        "Bytes": int(
            path.stat().st_size
        ),
        "SHA256": sha256_file(
            path
        ),
    })

    if (
        index % 200 == 0
        or index == len(
            raw_paths
        )
    ):
        print(
            "  Raw hashing progress:",
            index,
            "/",
            len(
                raw_paths
            ),
            "files",
        )

current_manifest = normalize_manifest(
    pd.DataFrame(
        manifest_rows
    ),
    "current manifest",
)

hash_seconds = (
    time.perf_counter()
    - hash_start
)

frozen_manifest = normalize_manifest(
    pd.read_csv(
        STEP5A_RAW_MANIFEST,
        low_memory=False,
    ),
    "frozen Step 5A manifest",
)

current_raw_sha = root_hash(
    current_manifest
)
current_raw_bytes = int(
    current_manifest[
        "Bytes"
    ].sum()
)

manifest_compare = frozen_manifest.merge(
    current_manifest,
    on="RelativePath",
    how="outer",
    suffixes=(
        "_frozen",
        "_current",
    ),
    indicator=True,
)

missing_raw = int(
    manifest_compare[
        "_merge"
    ].eq(
        "left_only"
    ).sum()
)
unexpected_raw = int(
    manifest_compare[
        "_merge"
    ].eq(
        "right_only"
    ).sum()
)

both = manifest_compare[
    "_merge"
].eq(
    "both"
)

size_mismatch = int(
    (
        both
        & manifest_compare[
            "Bytes_frozen"
        ].ne(
            manifest_compare[
                "Bytes_current"
            ]
        )
    ).sum()
)

hash_mismatch = int(
    (
        both
        & manifest_compare[
            "SHA256_frozen"
        ].ne(
            manifest_compare[
                "SHA256_current"
            ]
        )
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 8. CONDITION-BY-CONDITION INDEPENDENT REVALIDATION
# --------------------------------------------------------------------------------------------------

condition_plan = pd.read_csv(
    PLAN,
    low_memory=False,
)

condition_id_col = resolve_col(
    condition_plan.columns,
    [
        "ConditionID",
        "ConditionKey",
    ],
    "condition identifier",
)
condition_order_col = resolve_col(
    condition_plan.columns,
    [
        "ConditionOrder",
    ],
    "condition order",
)
noise_col = resolve_col(
    condition_plan.columns,
    [
        "NoisePercent",
    ],
    "noise percent",
)
seed_col = resolve_col(
    condition_plan.columns,
    [
        "RepetitionSeed",
    ],
    "repetition seed",
)

for column in [
    condition_order_col,
    noise_col,
    seed_col,
]:
    condition_plan[
        column
    ] = pd.to_numeric(
        condition_plan[
            column
        ],
        errors="raise",
    ).astype(int)

condition_plan = (
    condition_plan.sort_values(
        condition_order_col,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

inventory_rows = []
project_frames = []
build_frames = []
fit_frames = []
audit_frames = []
median_frames = []

marker_failures = 0
summary_failures = 0
file_set_failures = 0
embedded_manifest_failures = 0
ranking_count_failures = 0

print(
    "\nRevalidating all 270 condition directories."
)

condition_start = time.perf_counter()

for index, (_, row) in enumerate(
    condition_plan.iterrows(),
    start=1,
):
    condition_key = str(
        row[
            condition_id_col
        ]
    )
    condition_order = int(
        row[
            condition_order_col
        ]
    )
    noise_percent = int(
        row[
            noise_col
        ]
    )
    repetition_seed = int(
        row[
            seed_col
        ]
    )

    condition_dir = (
        RAW_ROOT
        / condition_key
    )

    if not condition_dir.is_dir():
        raise FileNotFoundError(
            f"Missing condition directory: {condition_dir}"
        )

    actual_files = {
        path.name
        for path in condition_dir.iterdir()
        if path.is_file()
    }

    file_set_pass = (
        actual_files
        == EXPECTED_CONDITION_FILES
    )
    file_set_failures += int(
        not file_set_pass
    )

    complete_path = (
        condition_dir
        / "COMPLETE.json"
    )
    summary_path = (
        condition_dir
        / "condition_summary.json"
    )

    complete = load_json(
        complete_path
    )
    summary = load_json(
        summary_path
    )

    complete_pass = bool(
        complete.get(
            "Status"
        )
        == CONDITION_STATUS
        and complete.get(
            "ConditionKey"
        )
        == condition_key
        and str(
            complete.get(
                "ConditionSummaryPath"
            )
        )
        == str(
            summary_path
        )
        and str(
            complete.get(
                "ConditionSummarySHA256"
            )
        ).lower()
        == sha256_file(
            summary_path
        )
    )

    summary_pass = bool(
        summary.get(
            "Status"
        )
        == CONDITION_STATUS
        and summary.get(
            "ConditionKey"
        )
        == condition_key
        and int(
            summary.get(
                "NoisePercent",
                -1,
            )
        )
        == noise_percent
        and int(
            summary.get(
                "RepetitionSeed",
                -1,
            )
        )
        == repetition_seed
    )

    marker_failures += int(
        not complete_pass
    )
    summary_failures += int(
        not summary_pass
    )

    output_manifest = summary.get(
        "OutputManifest",
        [],
    )

    local_embedded_failures = 0
    output_names = set()

    if (
        not isinstance(
            output_manifest,
            list,
        )
        or len(
            output_manifest
        )
        != 6
    ):
        local_embedded_failures += 1
    else:
        for item in output_manifest:
            path = Path(
                item.get(
                    "Path",
                    "",
                )
            )

            output_names.add(
                path.name
            )

            passed = bool(
                path.parent
                == condition_dir
                and path.is_file()
                and int(
                    path.stat().st_size
                )
                == int(
                    item.get(
                        "Bytes",
                        -1,
                    )
                )
                and sha256_file(
                    path
                )
                == str(
                    item.get(
                        "SHA256",
                        "",
                    )
                ).lower()
            )

            local_embedded_failures += int(
                not passed
            )

        local_embedded_failures += int(
            output_names
            != CONDITION_OUTPUT_FILES
        )

    embedded_manifest_failures += (
        local_embedded_failures
    )

    ranking_rows = gzip_rows(
        condition_dir
        / "rankings.csv.gz"
    )

    ranking_count_failures += int(
        ranking_rows
        != EXPECTED_RANKING_ROWS_PER_CONDITION
    )

    build_metrics = pd.read_csv(
        condition_dir
        / "build_metrics.csv",
        low_memory=False,
    )
    project_runs = pd.read_csv(
        condition_dir
        / "project_runs.csv",
        low_memory=False,
    )
    model_fits = pd.read_csv(
        condition_dir
        / "model_fits.csv",
        low_memory=False,
    )
    condition_audit = pd.read_csv(
        condition_dir
        / "condition_audit.csv",
        low_memory=False,
    )
    training_medians = pd.read_csv(
        condition_dir
        / "training_medians.csv",
        low_memory=False,
    )

    expected_counts = [
        EXPECTED_BUILD_ROWS_PER_CONDITION,
        EXPECTED_PROJECT_ROWS_PER_CONDITION,
        EXPECTED_FIT_ROWS_PER_CONDITION,
        1,
        EXPECTED_MEDIAN_ROWS_PER_CONDITION,
    ]

    actual_counts = [
        len(
            build_metrics
        ),
        len(
            project_runs
        ),
        len(
            model_fits
        ),
        len(
            condition_audit
        ),
        len(
            training_medians
        ),
    ]

    if actual_counts != expected_counts:
        raise RuntimeError(
            f"{condition_key}: compact output counts differ.\n"
            f"Expected: {expected_counts}\n"
            f"Actual:   {actual_counts}"
        )

    for field, count in [
        (
            "RankingRows",
            ranking_rows,
        ),
        (
            "BuildMetricRows",
            len(
                build_metrics
            ),
        ),
        (
            "ProjectRunRows",
            len(
                project_runs
            ),
        ),
        (
            "MLFits",
            len(
                model_fits
            ),
        ),
        (
            "TrainingMedianRows",
            len(
                training_medians
            ),
        ),
    ]:
        if int(
            summary.get(
                field,
                -1,
            )
        ) != int(
            count
        ):
            raise RuntimeError(
                f"{condition_key}: "
                f"condition_summary {field} differs."
            )

    project_frames.append(
        project_runs
    )
    build_frames.append(
        build_metrics
    )
    fit_frames.append(
        model_fits
    )
    audit_frames.append(
        condition_audit
    )
    median_frames.append(
        training_medians
    )

    inventory_rows.append({
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": condition_order,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "ConditionDirectory": str(
            condition_dir
        ),
        "CompletionStatus": complete.get(
            "Status"
        ),
        "SummaryStatus": summary.get(
            "Status"
        ),
        "Files": len(
            actual_files
        ),
        "ConditionBytes": int(
            sum(
                path.stat().st_size
                for path in condition_dir.iterdir()
                if path.is_file()
            )
        ),
        "RankingRows": ranking_rows,
        "BuildMetricRows": len(
            build_metrics
        ),
        "ProjectRunRows": len(
            project_runs
        ),
        "ModelFits": len(
            model_fits
        ),
        "ConditionAuditRows": len(
            condition_audit
        ),
        "TrainingMedianRows": len(
            training_medians
        ),
        "FileSetPass": file_set_pass,
        "CompletionMarkerPass": complete_pass,
        "ConditionSummaryPass": summary_pass,
        "EmbeddedManifestFailures": local_embedded_failures,
        "CompletionMarkerSHA256": sha256_file(
            complete_path
        ),
        "ConditionSummarySHA256": sha256_file(
            summary_path
        ),
    })

    if (
        index % 30 == 0
        or index == len(
            condition_plan
        )
    ):
        print(
            "  Condition revalidation progress:",
            index,
            "/",
            len(
                condition_plan
            ),
        )

condition_seconds = (
    time.perf_counter()
    - condition_start
)

inventory = (
    pd.DataFrame(
        inventory_rows
    )
    .sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

project_runs = pd.concat(
    project_frames,
    ignore_index=True,
)
build_metrics = pd.concat(
    build_frames,
    ignore_index=True,
)
model_fits = pd.concat(
    fit_frames,
    ignore_index=True,
)
condition_audit = pd.concat(
    audit_frames,
    ignore_index=True,
)
training_medians = pd.concat(
    median_frames,
    ignore_index=True,
)


# --------------------------------------------------------------------------------------------------
# 9. INDEPENDENT CONTRACT AUDITS
# --------------------------------------------------------------------------------------------------

coordinate_count = len(
    inventory[
        [
            "NoisePercent",
            "RepetitionSeed",
        ]
    ].drop_duplicates()
)

duplicate_condition_keys = int(
    inventory.duplicated(
        [
            "ConditionKey",
        ],
        keep=False,
    ).sum()
)

duplicate_coordinates = int(
    inventory.duplicated(
        [
            "NoisePercent",
            "RepetitionSeed",
        ],
        keep=False,
    ).sum()
)

condition_order_violations = int(
    (
        inventory[
            "ConditionOrder"
        ].to_numpy(
            dtype=int
        )
        != np.arange(
            1,
            EXPECTED_CONDITIONS + 1,
        )
    ).sum()
)

files_per_condition_violations = int(
    inventory[
        "Files"
    ].ne(
        EXPECTED_FILES_PER_CONDITION
    ).sum()
)

ranking_rows_per_condition_violations = int(
    inventory[
        "RankingRows"
    ].ne(
        EXPECTED_RANKING_ROWS_PER_CONDITION
    ).sum()
)

small_rows_per_condition_violations = int(
    inventory[
        "BuildMetricRows"
    ].ne(
        EXPECTED_BUILD_ROWS_PER_CONDITION
    ).sum()
    + inventory[
        "ProjectRunRows"
    ].ne(
        EXPECTED_PROJECT_ROWS_PER_CONDITION
    ).sum()
    + inventory[
        "ModelFits"
    ].ne(
        EXPECTED_FIT_ROWS_PER_CONDITION
    ).sum()
    + inventory[
        "ConditionAuditRows"
    ].ne(
        1
    ).sum()
    + inventory[
        "TrainingMedianRows"
    ].ne(
        EXPECTED_MEDIAN_ROWS_PER_CONDITION
    ).sum()
)

project_techniques = sorted(
    project_runs[
        "Technique"
    ].astype(
        str
    ).unique().tolist()
)

fit_techniques = sorted(
    model_fits[
        "Technique"
    ].astype(
        str
    ).unique().tolist()
)

duplicate_project_rows = int(
    project_runs.duplicated(
        [
            "ConditionKey",
            "Technique",
        ],
        keep=False,
    ).sum()
)

duplicate_build_rows = int(
    build_metrics.duplicated(
        [
            "ConditionKey",
            "Technique",
            "Build",
        ],
        keep=False,
    ).sum()
)

duplicate_fit_rows = int(
    model_fits.duplicated(
        [
            "ConditionKey",
            "Technique",
        ],
        keep=False,
    ).sum()
)

duplicate_audit_rows = int(
    condition_audit.duplicated(
        [
            "ConditionKey",
        ],
        keep=False,
    ).sum()
)

duplicate_median_rows = int(
    training_medians.duplicated(
        [
            "ConditionKey",
            "PredictorOrder",
        ],
        keep=False,
    ).sum()
)

model_fit_failures = int(
    (
        ~model_fits[
            "Status"
        ].astype(
            str
        ).eq(
            "PASS_MODEL_FIT"
        )
    ).sum()
)

model_fit_errors = int(
    model_fits[
        "Error"
    ].fillna(
        ""
    ).astype(
        str
    ).str.len().gt(
        0
    ).sum()
)

project_nonfinite = metric_nonfinite(
    project_runs,
    PROJECT_METRICS,
)
project_outside = metric_outside(
    project_runs,
    PROJECT_METRICS,
)
build_nonfinite = metric_nonfinite(
    build_metrics,
    BUILD_METRICS,
)
build_outside = metric_outside(
    build_metrics,
    BUILD_METRICS,
)

median_values = pd.to_numeric(
    training_medians[
        "TrainingMedian"
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)

median_nonfinite = int(
    (
        ~np.isfinite(
            median_values
        )
    ).sum()
)

median_predictor_violations = int(
    training_medians.groupby(
        "ConditionKey"
    )[
        "Predictor"
    ].nunique().ne(
        EXPECTED_PREDICTORS
    ).sum()
)

scored_build_violations = int(
    project_runs[
        "ScoredFailingBuilds"
    ].ne(
        EXPECTED_SCORED_FAILING_BUILDS
    ).sum()
)

evaluation_build_violations = int(
    project_runs[
        "EvaluationBuilds"
    ].ne(
        EXPECTED_EVALUATION_BUILDS
    ).sum()
)

evaluation_failure_violations = int(
    project_runs[
        "EvaluationFailures"
    ].ne(
        EXPECTED_EVALUATION_FAILURES
    ).sum()
)

zero_noise = condition_audit.loc[
    condition_audit[
        "NoisePercent"
    ].eq(
        0
    )
].copy()

positive_noise = condition_audit.loc[
    condition_audit[
        "NoisePercent"
    ].gt(
        0
    )
].copy()

zero_flip_violations = int(
    zero_noise[
        "NumberFlipped"
    ].ne(
        0
    ).sum()
)

zero_model_violations = int(
    zero_noise[
        "ModelLabelChanges"
    ].ne(
        0
    ).sum()
)

zero_rec_violations = int(
    zero_noise[
        "DependentRECChanges"
    ].ne(
        0
    ).sum()
)

positive_raw_violations = int(
    positive_noise[
        "NumberFlipped"
    ].le(
        0
    ).sum()
)

positive_model_violations = int(
    positive_noise[
        "ModelLabelChanges"
    ].le(
        0
    ).sum()
)

positive_rec_violations = int(
    positive_noise[
        "DependentRECChanges"
    ].le(
        0
    ).sum()
)

independent_rec_violations = int(
    condition_audit[
        "IndependentRECChanges"
    ].ne(
        0
    ).sum()
)

independent_reconstruction_violations = int(
    condition_audit[
        "IndependentReconstructionMismatches"
    ].ne(
        0
    ).sum()
)

noise_hash_mismatches = 0

for expected_column, actual_column in [
    (
        "ExpectedFlipMaskSHA256",
        "ActualFlipMaskSHA256",
    ),
    (
        "ExpectedNoisyRawVerdictSHA256",
        "ActualNoisyRawVerdictSHA256",
    ),
    (
        "ExpectedNoisyModelVerdictSHA256",
        "ActualNoisyModelVerdictSHA256",
    ),
]:
    noise_hash_mismatches += int(
        (
            condition_audit[
                expected_column
            ].astype(
                str
            )
            != condition_audit[
                actual_column
            ].astype(
                str
            )
        ).sum()
    )


# --------------------------------------------------------------------------------------------------
# 10. BASELINE INVARIANCE
# --------------------------------------------------------------------------------------------------

baseline_invariance = pd.read_csv(
    STEP5A_BASELINE,
    low_memory=False,
)

if "Pass" in baseline_invariance.columns:
    frozen_baseline_pass = pass_series(
        baseline_invariance[
            "Pass"
        ]
    )

    ranking_baseline_failures = int(
        (
            ~frozen_baseline_pass
        ).sum()
    )
else:
    mismatch_columns = [
        column
        for column in baseline_invariance.columns
        if "mismatch" in str(
            column
        ).lower()
    ]

    if not mismatch_columns:
        raise RuntimeError(
            "Frozen Step 5A baseline-invariance output "
            "has neither Pass nor mismatch columns."
        )

    ranking_baseline_failures = int(
        baseline_invariance[
            mismatch_columns
        ].apply(
            pd.to_numeric,
            errors="coerce",
        ).fillna(
            0
        ).to_numpy(
            dtype=float
        ).sum()
    )

# IMPORTANT:
# Each metric is evaluated independently.
# APFDc and APFD must never be numerically compared against one another.
project_metric_baseline_failures = 0
project_metric_baseline_max_range = 0.0

for _, group in project_runs.loc[
    project_runs[
        "Technique"
    ].isin(
        INVARIANT_BASELINES
    )
].groupby(
    [
        "RepetitionSeed",
        "Technique",
    ],
    sort=False,
):
    values = group[
        PROJECT_METRICS
    ].to_numpy(
        dtype=float
    )

    per_metric_ranges = (
        np.max(
            values,
            axis=0,
        )
        - np.min(
            values,
            axis=0,
        )
    )

    project_metric_baseline_max_range = max(
        project_metric_baseline_max_range,
        float(
            np.max(
                per_metric_ranges
            )
        ),
    )

    project_metric_baseline_failures += int(
        (
            per_metric_ranges
            > 1e-15
        ).any()
    )


# --------------------------------------------------------------------------------------------------
# 11. ANALYSIS-READY COMPACT AGGREGATES
# --------------------------------------------------------------------------------------------------

aggregation_start = time.perf_counter()

noise_summary = (
    project_runs.groupby(
        [
            "NoisePercent",
            "Technique",
        ],
        as_index=False,
        sort=True,
    )
    .agg(
        Runs=(
            "ConditionKey",
            "count",
        ),
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),
        Mean_MeanAPFDc=(
            "MeanAPFDc",
            "mean",
        ),
        SD_MeanAPFDc=(
            "MeanAPFDc",
            "std",
        ),
        Median_MeanAPFDc=(
            "MeanAPFDc",
            "median",
        ),
        Mean_MedianAPFDc=(
            "MedianAPFDc",
            "mean",
        ),
        SD_MedianAPFDc=(
            "MedianAPFDc",
            "std",
        ),
        Median_MedianAPFDc=(
            "MedianAPFDc",
            "median",
        ),
        Mean_MeanAPFD=(
            "MeanAPFD",
            "mean",
        ),
        SD_MeanAPFD=(
            "MeanAPFD",
            "std",
        ),
        Median_MeanAPFD=(
            "MeanAPFD",
            "median",
        ),
        Mean_MedianAPFD=(
            "MedianAPFD",
            "mean",
        ),
        SD_MedianAPFD=(
            "MedianAPFD",
            "std",
        ),
        Median_MedianAPFD=(
            "MedianAPFD",
            "median",
        ),
    )
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

clean_project_runs = (
    project_runs.loc[
        project_runs[
            "NoisePercent"
        ].eq(
            0
        ),
        [
            "RepetitionSeed",
            "Technique",
        ]
        + PROJECT_METRICS,
    ]
    .rename(
        columns={
            column: f"Clean_{column}"
            for column in PROJECT_METRICS
        }
    )
)

seed_deltas = project_runs.merge(
    clean_project_runs,
    on=[
        "RepetitionSeed",
        "Technique",
    ],
    how="left",
    validate="many_to_one",
)

for column in PROJECT_METRICS:
    seed_deltas[
        f"Delta_{column}"
    ] = (
        seed_deltas[
            column
        ]
        - seed_deltas[
            f"Clean_{column}"
        ]
    )

delta_columns = [
    f"Delta_{column}"
    for column in PROJECT_METRICS
]

seed_deltas = (
    seed_deltas[
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "ConditionKey",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ]
        + PROJECT_METRICS
        + [
            f"Clean_{column}"
            for column in PROJECT_METRICS
        ]
        + delta_columns
    ]
    .sort_values(
        [
            "NoisePercent",
            "Technique",
            "RepetitionSeed",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

delta_summary = (
    seed_deltas.groupby(
        [
            "NoisePercent",
            "Technique",
        ],
        as_index=False,
        sort=True,
    )
    .agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),
        Mean_Delta_MeanAPFDc=(
            "Delta_MeanAPFDc",
            "mean",
        ),
        SD_Delta_MeanAPFDc=(
            "Delta_MeanAPFDc",
            "std",
        ),
        Median_Delta_MeanAPFDc=(
            "Delta_MeanAPFDc",
            "median",
        ),
        Mean_Delta_MedianAPFDc=(
            "Delta_MedianAPFDc",
            "mean",
        ),
        SD_Delta_MedianAPFDc=(
            "Delta_MedianAPFDc",
            "std",
        ),
        Median_Delta_MedianAPFDc=(
            "Delta_MedianAPFDc",
            "median",
        ),
        Mean_Delta_MeanAPFD=(
            "Delta_MeanAPFD",
            "mean",
        ),
        SD_Delta_MeanAPFD=(
            "Delta_MeanAPFD",
            "std",
        ),
        Median_Delta_MeanAPFD=(
            "Delta_MeanAPFD",
            "median",
        ),
        Mean_Delta_MedianAPFD=(
            "Delta_MedianAPFD",
            "mean",
        ),
        SD_Delta_MedianAPFD=(
            "Delta_MedianAPFD",
            "std",
        ),
        Median_Delta_MedianAPFD=(
            "Delta_MedianAPFD",
            "median",
        ),
    )
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

aggregation_seconds = (
    time.perf_counter()
    - aggregation_start
)

summary_numeric_columns = [
    column
    for column in noise_summary.columns
    if column
    not in {
        "NoisePercent",
        "Technique",
    }
]

delta_summary_numeric_columns = [
    column
    for column in delta_summary.columns
    if column
    not in {
        "NoisePercent",
        "Technique",
    }
]

summary_nonfinite = metric_nonfinite(
    noise_summary,
    summary_numeric_columns,
)

delta_summary_nonfinite = metric_nonfinite(
    delta_summary,
    delta_summary_numeric_columns,
)

clean_delta_nonzero = int(
    (
        np.abs(
            seed_deltas.loc[
                seed_deltas[
                    "NoisePercent"
                ].eq(
                    0
                ),
                delta_columns,
            ].to_numpy(
                dtype=float
            )
        )
        > 1e-15
    ).sum()
)

invariant_baseline_delta_nonzero = int(
    (
        np.abs(
            seed_deltas.loc[
                seed_deltas[
                    "Technique"
                ].isin(
                    INVARIANT_BASELINES
                ),
                delta_columns,
            ].to_numpy(
                dtype=float
            )
        )
        > 1e-15
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 12. VALIDATION TABLE
# --------------------------------------------------------------------------------------------------

checks = []

add_check(
    checks,
    "Step 5A status",
    STEP5A_STATUS,
    step5a.get(
        "Status"
    ),
    step5a.get(
        "Status"
    )
    == STEP5A_STATUS,
)

add_check(
    checks,
    "Step 5A checkpoint SHA-256",
    EXPECTED_STEP5A_SHA,
    step5a_sha,
    step5a_sha
    == EXPECTED_STEP5A_SHA,
)

add_check(
    checks,
    "Frozen raw-root SHA-256",
    EXPECTED_RAW_ROOT_SHA,
    step5a.get(
        "RawRootSHA256"
    ),
    step5a.get(
        "RawRootSHA256"
    )
    == EXPECTED_RAW_ROOT_SHA,
)

add_check(
    checks,
    "Independent current raw-root SHA-256",
    EXPECTED_RAW_ROOT_SHA,
    current_raw_sha,
    current_raw_sha
    == EXPECTED_RAW_ROOT_SHA,
)

add_check(
    checks,
    "Step 5A aggregate-manifest failures",
    0,
    aggregate_manifest_failures,
    aggregate_manifest_failures
    == 0,
)

add_check(
    checks,
    "Condition marker failures",
    0,
    marker_failures,
    marker_failures
    == 0,
)

add_check(
    checks,
    "Condition summary failures",
    0,
    summary_failures,
    summary_failures
    == 0,
)

add_check(
    checks,
    "Condition file-set failures",
    0,
    file_set_failures,
    file_set_failures
    == 0,
)

add_check(
    checks,
    "Embedded output-manifest failures",
    0,
    embedded_manifest_failures,
    embedded_manifest_failures
    == 0,
)

add_check(
    checks,
    "Conditions",
    EXPECTED_CONDITIONS,
    len(
        inventory
    ),
    len(
        inventory
    )
    == EXPECTED_CONDITIONS,
)

add_check(
    checks,
    "Condition coordinates",
    EXPECTED_CONDITIONS,
    coordinate_count,
    coordinate_count
    == EXPECTED_CONDITIONS,
)

add_check(
    checks,
    "Duplicate condition keys",
    0,
    duplicate_condition_keys,
    duplicate_condition_keys
    == 0,
)

add_check(
    checks,
    "Duplicate condition coordinates",
    0,
    duplicate_coordinates,
    duplicate_coordinates
    == 0,
)

add_check(
    checks,
    "Condition-order violations",
    0,
    condition_order_violations,
    condition_order_violations
    == 0,
)

add_check(
    checks,
    "Files-per-condition violations",
    0,
    files_per_condition_violations,
    files_per_condition_violations
    == 0,
)

add_check(
    checks,
    "Raw files",
    EXPECTED_RAW_FILES,
    len(
        current_manifest
    ),
    len(
        current_manifest
    )
    == EXPECTED_RAW_FILES,
)

add_check(
    checks,
    "Raw bytes",
    EXPECTED_RAW_BYTES,
    current_raw_bytes,
    current_raw_bytes
    == EXPECTED_RAW_BYTES,
)

add_check(
    checks,
    "Missing raw files",
    0,
    missing_raw,
    missing_raw
    == 0,
)

add_check(
    checks,
    "Unexpected raw files",
    0,
    unexpected_raw,
    unexpected_raw
    == 0,
)

add_check(
    checks,
    "Raw size mismatches",
    0,
    size_mismatch,
    size_mismatch
    == 0,
)

add_check(
    checks,
    "Raw SHA-256 mismatches",
    0,
    hash_mismatch,
    hash_mismatch
    == 0,
)

add_check(
    checks,
    "Ranking rows",
    EXPECTED_TOTAL_RANKING_ROWS,
    int(
        inventory[
            "RankingRows"
        ].sum()
    ),
    int(
        inventory[
            "RankingRows"
        ].sum()
    )
    == EXPECTED_TOTAL_RANKING_ROWS,
)

add_check(
    checks,
    "Ranking row-count failures",
    0,
    ranking_count_failures
    + ranking_rows_per_condition_violations,
    ranking_count_failures
    + ranking_rows_per_condition_violations
    == 0,
)

add_check(
    checks,
    "Project-run rows",
    EXPECTED_TOTAL_PROJECT_ROWS,
    len(
        project_runs
    ),
    len(
        project_runs
    )
    == EXPECTED_TOTAL_PROJECT_ROWS,
)

add_check(
    checks,
    "Build-metric rows",
    EXPECTED_TOTAL_BUILD_ROWS,
    len(
        build_metrics
    ),
    len(
        build_metrics
    )
    == EXPECTED_TOTAL_BUILD_ROWS,
)

add_check(
    checks,
    "Model-fit rows",
    EXPECTED_TOTAL_FIT_ROWS,
    len(
        model_fits
    ),
    len(
        model_fits
    )
    == EXPECTED_TOTAL_FIT_ROWS,
)

add_check(
    checks,
    "Condition-audit rows",
    EXPECTED_TOTAL_AUDIT_ROWS,
    len(
        condition_audit
    ),
    len(
        condition_audit
    )
    == EXPECTED_TOTAL_AUDIT_ROWS,
)

add_check(
    checks,
    "Training-median rows",
    EXPECTED_TOTAL_MEDIAN_ROWS,
    len(
        training_medians
    ),
    len(
        training_medians
    )
    == EXPECTED_TOTAL_MEDIAN_ROWS,
)

add_check(
    checks,
    "Small rows-per-condition violations",
    0,
    small_rows_per_condition_violations,
    small_rows_per_condition_violations
    == 0,
)

add_check(
    checks,
    "Project-run technique set",
    sorted(
        TECHNIQUES
    ),
    project_techniques,
    project_techniques
    == sorted(
        TECHNIQUES
    ),
)

add_check(
    checks,
    "Model-fit technique set",
    sorted(
        ML_TECHNIQUES
    ),
    fit_techniques,
    fit_techniques
    == sorted(
        ML_TECHNIQUES
    ),
)

duplicate_compact_rows = (
    duplicate_project_rows
    + duplicate_build_rows
    + duplicate_fit_rows
    + duplicate_audit_rows
    + duplicate_median_rows
)

add_check(
    checks,
    "Duplicate project/build/fit/audit/median rows",
    0,
    duplicate_compact_rows,
    duplicate_compact_rows
    == 0,
)

add_check(
    checks,
    "Model-fit failures",
    0,
    model_fit_failures
    + model_fit_errors,
    model_fit_failures
    + model_fit_errors
    == 0,
)

add_check(
    checks,
    "Scored-failing-build count violations",
    0,
    scored_build_violations,
    scored_build_violations
    == 0,
)

add_check(
    checks,
    "Evaluation-build count violations",
    0,
    evaluation_build_violations,
    evaluation_build_violations
    == 0,
)

add_check(
    checks,
    "Evaluation-failure count violations",
    0,
    evaluation_failure_violations,
    evaluation_failure_violations
    == 0,
)

combined_evaluation_count_violations = (
    scored_build_violations
    + evaluation_build_violations
    + evaluation_failure_violations
)

add_check(
    checks,
    "Combined scored/evaluated/failure count violations",
    0,
    combined_evaluation_count_violations,
    combined_evaluation_count_violations
    == 0,
)

add_check(
    checks,
    "Project metric invalid values",
    0,
    project_nonfinite
    + project_outside,
    project_nonfinite
    + project_outside
    == 0,
)

add_check(
    checks,
    "Build metric invalid values",
    0,
    build_nonfinite
    + build_outside,
    build_nonfinite
    + build_outside
    == 0,
)

add_check(
    checks,
    "Training-median invalid values",
    0,
    median_nonfinite
    + median_predictor_violations,
    median_nonfinite
    + median_predictor_violations
    == 0,
)

add_check(
    checks,
    "Zero-noise conditions",
    30,
    len(
        zero_noise
    ),
    len(
        zero_noise
    )
    == 30,
)

zero_noise_violations = (
    zero_flip_violations
    + zero_model_violations
    + zero_rec_violations
)

add_check(
    checks,
    "Zero-noise violations",
    0,
    zero_noise_violations,
    zero_noise_violations
    == 0,
)

positive_noise_violations = (
    positive_raw_violations
    + positive_model_violations
    + positive_rec_violations
)

add_check(
    checks,
    "Positive-noise violations",
    0,
    positive_noise_violations,
    positive_noise_violations
    == 0,
)

independent_violations = (
    independent_rec_violations
    + independent_reconstruction_violations
)

add_check(
    checks,
    "Independent REC violations",
    0,
    independent_violations,
    independent_violations
    == 0,
)

add_check(
    checks,
    "Noise-plan hash mismatches",
    0,
    noise_hash_mismatches,
    noise_hash_mismatches
    == 0,
)

add_check(
    checks,
    "Ranking-level baseline-invariance failures",
    0,
    ranking_baseline_failures,
    ranking_baseline_failures
    == 0,
)

add_check(
    checks,
    "Project-metric baseline-invariance failures",
    0,
    project_metric_baseline_failures,
    project_metric_baseline_failures
    == 0,
)

add_check(
    checks,
    "Noise-technique summary rows",
    EXPECTED_NOISE_TECHNIQUE_SUMMARY_ROWS,
    len(
        noise_summary
    ),
    len(
        noise_summary
    )
    == EXPECTED_NOISE_TECHNIQUE_SUMMARY_ROWS,
)

noise_summary_violations = int(
    noise_summary[
        "Runs"
    ].ne(
        30
    ).sum()
    + noise_summary[
        "Seeds"
    ].ne(
        30
    ).sum()
) + summary_nonfinite

add_check(
    checks,
    "Noise-technique summary count/nonfinite violations",
    0,
    noise_summary_violations,
    noise_summary_violations
    == 0,
)

add_check(
    checks,
    "Seed-level delta rows",
    EXPECTED_SEED_DELTA_ROWS,
    len(
        seed_deltas
    ),
    len(
        seed_deltas
    )
    == EXPECTED_SEED_DELTA_ROWS,
)

add_check(
    checks,
    "Clean delta non-zero values",
    0,
    clean_delta_nonzero,
    clean_delta_nonzero
    == 0,
)

add_check(
    checks,
    "Invariant-baseline delta non-zero values",
    0,
    invariant_baseline_delta_nonzero,
    invariant_baseline_delta_nonzero
    == 0,
)

add_check(
    checks,
    "Noise-delta summary rows",
    EXPECTED_NOISE_DELTA_SUMMARY_ROWS,
    len(
        delta_summary
    ),
    len(
        delta_summary
    )
    == EXPECTED_NOISE_DELTA_SUMMARY_ROWS,
)

delta_summary_violations = int(
    delta_summary[
        "Seeds"
    ].ne(
        30
    ).sum()
) + delta_summary_nonfinite

add_check(
    checks,
    "Noise-delta summary count/nonfinite violations",
    0,
    delta_summary_violations,
    delta_summary_violations
    == 0,
)

add_check(
    checks,
    "Registry rows",
    21,
    len(
        registry
    ),
    len(
        registry
    )
    == 21,
)

add_check(
    checks,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    step5a.get(
        "ActiveReservations"
    ),
    step5a.get(
        "ActiveReservations"
    )
    == EXPECTED_ACTIVE_RESERVATIONS,
)

add_check(
    checks,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    step5a.get(
        "RuntimePriorityRule"
    ),
    step5a.get(
        "RuntimePriorityRule"
    )
    == EXPECTED_RUNTIME_PRIORITY_RULE,
)

for required_number, required_project in REQUIRED_REGISTERED_IDENTITIES.items():
    matching = registry.loc[
        project_numbers.eq(
            required_number
        )
    ]

    add_check(
        checks,
        f"Registry Project {required_number} rows",
        1,
        len(
            matching
        ),
        len(
            matching
        )
        == 1,
    )

    add_check(
        checks,
        f"Project {required_number} frozen identity",
        required_project,
        (
            str(
                matching.iloc[
                    0
                ][
                    project_col
                ]
            )
            if len(
                matching
            )
            == 1
            else None
        ),
        (
            len(
                matching
            )
            == 1
            and str(
                matching.iloc[
                    0
                ][
                    project_col
                ]
            )
            == required_project
        ),
    )

add_check(
    checks,
    "Registry Project 22 rows",
    0,
    int(
        project_numbers.eq(
            22
        ).sum()
    ),
    int(
        project_numbers.eq(
            22
        ).sum()
    )
    == 0,
)

validation = pd.DataFrame(
    checks
)

failed = validation.loc[
    ~validation[
        "Pass"
    ]
].copy()

print(
    "\nProject 22 Step 5B validation:"
)
display(
    validation
)

if not failed.empty:
    print(
        "\nFailed checks:"
    )
    display(
        failed
    )

    raise RuntimeError(
        "PROJECT 22 STEP 5B VALIDATION FAILED. "
        "No PASS checkpoint was written."
    )


# --------------------------------------------------------------------------------------------------
# 13. FREEZE STEP 5B OUTPUTS
# --------------------------------------------------------------------------------------------------

OUT.mkdir(
    parents=True,
    exist_ok=True,
)

for path, frame in [
    (
        CURRENT_MANIFEST,
        current_manifest,
    ),
    (
        CONDITION_INVENTORY,
        inventory,
    ),
    (
        REVALIDATED_PROJECT_RUNS,
        project_runs,
    ),
    (
        REVALIDATED_BUILD_METRICS,
        build_metrics,
    ),
    (
        REVALIDATED_MODEL_FITS,
        model_fits,
    ),
    (
        REVALIDATED_CONDITION_AUDIT,
        condition_audit,
    ),
    (
        REVALIDATED_MEDIANS,
        training_medians,
    ),
    (
        NOISE_SUMMARY,
        noise_summary,
    ),
    (
        SEED_DELTAS,
        seed_deltas,
    ),
    (
        DELTA_SUMMARY,
        delta_summary,
    ),
    (
        VALIDATION_PATH,
        validation,
    ),
]:
    atomic_csv(
        path,
        frame,
    )

output_paths = [
    CURRENT_MANIFEST,
    CONDITION_INVENTORY,
    REVALIDATED_PROJECT_RUNS,
    REVALIDATED_BUILD_METRICS,
    REVALIDATED_MODEL_FITS,
    REVALIDATED_CONDITION_AUDIT,
    REVALIDATED_MEDIANS,
    NOISE_SUMMARY,
    SEED_DELTAS,
    DELTA_SUMMARY,
    VALIDATION_PATH,
]

output_manifest = [
    {
        "Path": str(
            path
        ),
        "Bytes": int(
            path.stat().st_size
        ),
        "SHA256": sha256_file(
            path
        ),
    }
    for path in output_paths
]

completed_at = datetime.now(
    timezone.utc
).isoformat()

report = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5B_STATUS,
    "CompletedAtUTC": completed_at,
    "Step5ACheckpointSHA256": step5a_sha,
    "FrozenRawRootSHA256": EXPECTED_RAW_ROOT_SHA,
    "IndependentRawRootSHA256": current_raw_sha,
    "RawFiles": len(
        current_manifest
    ),
    "RawBytes": current_raw_bytes,
    "Conditions": len(
        inventory
    ),
    "ExpectedScoredFailingBuilds": EXPECTED_SCORED_FAILING_BUILDS,
    "ExpectedEvaluationBuilds": EXPECTED_EVALUATION_BUILDS,
    "ExpectedEvaluationFailures": EXPECTED_EVALUATION_FAILURES,
    "MLFits": len(
        model_fits
    ),
    "RankingRows": int(
        inventory[
            "RankingRows"
        ].sum()
    ),
    "BuildMetricRows": len(
        build_metrics
    ),
    "ProjectRunRows": len(
        project_runs
    ),
    "ConditionAuditRows": len(
        condition_audit
    ),
    "TrainingMedianRows": len(
        training_medians
    ),
    "NoiseTechniqueSummaryRows": len(
        noise_summary
    ),
    "SeedLevelNoiseDeltaRows": len(
        seed_deltas
    ),
    "NoiseDeltaSummaryRows": len(
        delta_summary
    ),
    "StandardDeviationDefinition": (
        "Sample SD across 30 seeds; pandas std, ddof=1"
    ),
    "RankingLevelBaselineInvarianceFailures": int(
        ranking_baseline_failures
    ),
    "ProjectMetricBaselineInvarianceFailures": int(
        project_metric_baseline_failures
    ),
    "ProjectMetricBaselineMaximumWithinMetricRange": float(
        project_metric_baseline_max_range
    ),
    "RawHashingSeconds": float(
        hash_seconds
    ),
    "ConditionRevalidationSeconds": float(
        condition_seconds
    ),
    "AggregationSeconds": float(
        aggregation_seconds
    ),
    "OutputManifest": output_manifest,
    "ValidationChecks": len(
        validation
    ),
    "FailedValidationChecks": len(
        failed
    ),
    "SourceRootSHA256": EXPECTED_SOURCE_ROOT_SHA,
    "RegistrySHA256": registry_sha_before,
    "RegistryModified": False,
    "Projects1To21Modified": False,
    "ActiveReservations": EXPECTED_ACTIVE_RESERVATIONS,
    "RuntimePriorityRule": EXPECTED_RUNTIME_PRIORITY_RULE,
    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectConditionOutputsModified": False,
    "PriorProjectWriteAttempted": False,
    "ModelsFitted": False,
    "ConditionsRerun": False,
}

atomic_json(
    REPORT_PATH,
    report,
)

checkpoint = {
    **report,
    "CheckpointVersion": 1,
    "CheckpointType": (
        "PROJECT_22_RAW_REVALIDATION_AND_COMPACT_AGGREGATION"
    ),
    "RawResultsRevalidated": True,
    "CompactAggregatesFrozen": True,
    "ReadyForFinalPackageAndRegistration": True,
    "NextRequiredStep": (
        "STEP_5C_FINAL_PACKAGE_FREEZE_AND_COMPLETION_REGISTRY_REGISTRATION"
    ),
}

atomic_json(
    CHECKPOINT_PATH,
    checkpoint,
)

checkpoint_sha = sha256_file(
    CHECKPOINT_PATH
)

status_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5B_STATUS,
    "CompletedAtUTC": completed_at,
    "Step5ACheckpointSHA256": step5a_sha,
    "RawRootSHA256": current_raw_sha,
    "RawFiles": len(
        current_manifest
    ),
    "RawBytes": current_raw_bytes,
    "Conditions": len(
        inventory
    ),
    "MLFits": len(
        model_fits
    ),
    "Checkpoint": str(
        CHECKPOINT_PATH
    ),
    "CheckpointSHA256": checkpoint_sha,
    "ReadyForFinalPackageAndRegistration": True,
    "RegistryModified": False,
    "NextRequiredStep": (
        "STEP_5C_FINAL_PACKAGE_FREEZE_AND_COMPLETION_REGISTRY_REGISTRATION"
    ),
}

atomic_json(
    STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 14. READBACK AND IMMUTABILITY
# --------------------------------------------------------------------------------------------------

if (
    load_json(
        CHECKPOINT_PATH
    ).get(
        "Status"
    )
    != STEP5B_STATUS
    or load_json(
        STATUS_PATH
    ).get(
        "Status"
    )
    != STEP5B_STATUS
):
    raise RuntimeError(
        "Step 5B checkpoint/status readback failed."
    )

for item in output_manifest:
    path = Path(
        item[
            "Path"
        ]
    )

    if (
        not path.is_file()
        or int(
            path.stat().st_size
        )
        != int(
            item[
                "Bytes"
            ]
        )
        or sha256_file(
            path
        )
        != str(
            item[
                "SHA256"
            ]
        )
    ):
        raise RuntimeError(
            f"Step 5B output readback failed: {path}"
        )

registry_sha_after = sha256_file(
    REGISTRY
)

if registry_sha_after != registry_sha_before:
    raise RuntimeError(
        "Completion registry changed during Project 22 Step 5B."
    )

if sha256_file(
    STEP5A_CHECKPOINT
) != EXPECTED_STEP5A_SHA:
    raise RuntimeError(
        "Step 5A checkpoint changed during Step 5B."
    )

# Raw result files are read-only in Step 5B.
# Count and byte totals are checked again after all Step 5B writes.
raw_paths_after = [
    path
    for path in RAW_ROOT.rglob("*")
    if path.is_file()
]

if len(
    raw_paths_after
) != EXPECTED_RAW_FILES:
    raise RuntimeError(
        "Project 22 raw-file count changed during Step 5B."
    )

if int(
    sum(
        path.stat().st_size
        for path in raw_paths_after
    )
) != EXPECTED_RAW_BYTES:
    raise RuntimeError(
        "Project 22 raw-byte total changed during Step 5B."
    )


# --------------------------------------------------------------------------------------------------
# 15. RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\nNoise-technique summary:"
)
display(
    noise_summary
)

print(
    "\nNoise-delta summary:"
)
display(
    delta_summary
)

print(
    "\n"
    + "=" * 136
)
print(
    "=== PROJECT 22 CELL 10 / STEP 5B RESULT ==="
)
print(
    "=" * 136
)

print(
    "Project:",
    PROJECT_NAME,
)
print(
    "Step 5A checkpoint SHA-256:",
    step5a_sha,
)
print(
    "Frozen raw-root SHA-256:",
    EXPECTED_RAW_ROOT_SHA,
)
print(
    "Independent current raw-root SHA-256:",
    current_raw_sha,
)

print(
    "\nRaw-output revalidation:"
)
print(
    "Conditions:",
    len(
        inventory
    ),
    "/",
    EXPECTED_CONDITIONS,
)
print(
    "Raw files:",
    len(
        current_manifest
    ),
    "/",
    EXPECTED_RAW_FILES,
)
print(
    "Raw bytes:",
    current_raw_bytes,
    "/",
    EXPECTED_RAW_BYTES,
)
print(
    "Missing / unexpected / size / SHA mismatches:",
    missing_raw,
    "/",
    unexpected_raw,
    "/",
    size_mismatch,
    "/",
    hash_mismatch,
)
print(
    "Embedded output-manifest failures:",
    embedded_manifest_failures,
)

print(
    "\nExperiment totals:"
)
print(
    "ML fits:",
    len(
        model_fits
    ),
    "/",
    EXPECTED_TOTAL_FIT_ROWS,
)
print(
    "Ranking rows:",
    int(
        inventory[
            "RankingRows"
        ].sum()
    ),
    "/",
    EXPECTED_TOTAL_RANKING_ROWS,
)
print(
    "Build-metric rows:",
    len(
        build_metrics
    ),
    "/",
    EXPECTED_TOTAL_BUILD_ROWS,
)
print(
    "Project-run rows:",
    len(
        project_runs
    ),
    "/",
    EXPECTED_TOTAL_PROJECT_ROWS,
)
print(
    "Condition-audit rows:",
    len(
        condition_audit
    ),
    "/",
    EXPECTED_TOTAL_AUDIT_ROWS,
)
print(
    "Training-median rows:",
    len(
        training_medians
    ),
    "/",
    EXPECTED_TOTAL_MEDIAN_ROWS,
)

print(
    "\nAnalysis-ready aggregates:"
)
print(
    "Noise-technique summary rows:",
    len(
        noise_summary
    ),
)
print(
    "Seed-level noise-delta rows:",
    len(
        seed_deltas
    ),
)
print(
    "Noise-delta summary rows:",
    len(
        delta_summary
    ),
)
print(
    "Sample SD calculated with ddof=1:",
    True,
)
print(
    "Ranking-level baseline-invariance failures:",
    ranking_baseline_failures,
)
print(
    "Project-metric baseline-invariance failures:",
    project_metric_baseline_failures,
)
print(
    "Maximum invariant-baseline within-metric range:",
    project_metric_baseline_max_range,
)

print(
    "\nSafety:"
)
print(
    "Models fitted in Step 5B:",
    False,
)
print(
    "Conditions rerun in Step 5B:",
    False,
)
print(
    "Prior-project condition outputs accessed:",
    False,
)
print(
    "Registry modified:",
    False,
)

print(
    "\nValidation checks:",
    len(
        validation
    ),
)
print(
    "Failed checks:",
    len(
        failed
    ),
)
print(
    "Step 5B checkpoint:",
    CHECKPOINT_PATH,
)
print(
    "Step 5B checkpoint SHA-256:",
    checkpoint_sha,
)
print(
    "Next required step: STEP 5C — FINAL PACKAGE FREEZE AND COMPLETION-REGISTRY REGISTRATION"
)
print(
    "STATUS:",
    STEP5B_STATUS,
)
print(
    "=" * 136
)


=== PROJECT 22 CELL 10 / STEP 5B: RAW REVALIDATION AND COMPACT AGGREGATION ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Independently hashing all 2,160 raw files.
  Raw hashing progress: 200 / 2160 files
  Raw hashing progress: 400 / 2160 files
  Raw hashing progress: 600 / 2160 files
  Raw hashing progress: 800 / 2160 files
  Raw hashing progress: 1000 / 2160 files
  Raw hashing progress: 1200 / 2160 files
  Raw hashing progress: 1400 / 2160 files
  Raw hashing progress: 1600 / 2160 files
  Raw hashing progress: 1800 / 2160 files
  Raw hashing progress: 2000 / 2160 files
  Raw hashing progress: 2160 / 2160 files

Revalidating all 270 condition directories.
  Condition revalidation progress: 30 / 270
  Condition revalidation progress: 60 / 270
  Condition revalidation progress: 90 / 270
  Condition revalidation progress: 120 / 270
  Condition revalidation progress: 150 / 270
  Condition revalidatio

,Check,Expected,Actual,Pass
0,Step 5A status,PASS_PROJECT_22_FULL_270_CONDITION_EXPERIMENT_...,PASS_PROJECT_22_FULL_270_CONDITION_EXPERIMENT_...,True
1,Step 5A checkpoint SHA-256,600035cfbf9f8b2dfd0fd83d49050466e3c71e6187c1d6...,600035cfbf9f8b2dfd0fd83d49050466e3c71e6187c1d6...,True
2,Frozen raw-root SHA-256,374e4e1eaab266451539c9c7a4751fa6fe50250dccf57b...,374e4e1eaab266451539c9c7a4751fa6fe50250dccf57b...,True
3,Independent current raw-root SHA-256,374e4e1eaab266451539c9c7a4751fa6fe50250dccf57b...,374e4e1eaab266451539c9c7a4751fa6fe50250dccf57b...,True
4,Step 5A aggregate-manifest failures,0,0,True
...,...,...,...,...
75,Registry Project 20 rows,1,1,True
76,Project 20 frozen identity,apache@curator,apache@curator,True
77,Registry Project 21 rows,1,1,True
78,Project 21 frozen identity,facebook@buck,facebook@buck,True



Noise-technique summary:


,NoisePercent,Technique,Runs,Seeds,Mean_MeanAPFDc,SD_MeanAPFDc,Median_MeanAPFDc,Mean_MedianAPFDc,SD_MedianAPFDc,Median_MedianAPFDc,Mean_MeanAPFD,SD_MeanAPFD,Median_MeanAPFD,Mean_MedianAPFD,SD_MedianAPFD,Median_MedianAPFD
0,0,LatestFail,30,30,0.048835,0.000000,0.048835,0.044033,0.000000,0.044033,0.032709,0.000000,0.032709,0.025578,0.000000,0.025578
1,0,LightGBM,30,30,0.922421,0.000000,0.922421,0.999024,0.000000,0.999024,0.940615,0.000000,0.940615,0.999175,0.000000,0.999175
2,0,NaiveBayes,30,30,0.883923,0.000000,0.883923,0.969066,0.000000,0.969066,0.834451,0.000000,0.834451,0.964580,0.000000,0.964580
3,0,QTF-Avg,30,30,0.686528,0.000000,0.686528,0.703948,0.000000,0.703948,0.149534,0.000000,0.149534,0.131188,0.000000,0.131188
4,0,Random,30,30,0.513169,0.047476,0.506934,0.511745,0.077823,0.512341,0.513268,0.046323,0.500406,0.516121,0.082730,0.514666
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,50,NaiveBayes,30,30,0.440206,0.362888,0.226587,0.435183,0.460034,0.107125,0.441693,0.375925,0.153457,0.437673,0.480426,0.035880
59,50,QTF-Avg,30,30,0.686528,0.000000,0.686528,0.703948,0.000000,0.703948,0.149534,0.000000,0.149534,0.131188,0.000000,0.131188
60,50,Random,30,30,0.513169,0.047476,0.506934,0.511745,0.077823,0.512341,0.513268,0.046323,0.500406,0.516121,0.082730,0.514666
61,50,RandomForest,30,30,0.390441,0.109476,0.379736,0.331763,0.161483,0.326212,0.366739,0.108097,0.337316,0.303610,0.167837,0.246075



Noise-delta summary:


,NoisePercent,Technique,Seeds,Mean_Delta_MeanAPFDc,SD_Delta_MeanAPFDc,Median_Delta_MeanAPFDc,Mean_Delta_MedianAPFDc,SD_Delta_MedianAPFDc,Median_Delta_MedianAPFDc,Mean_Delta_MeanAPFD,SD_Delta_MeanAPFD,Median_Delta_MeanAPFD,Mean_Delta_MedianAPFD,SD_Delta_MedianAPFD,Median_Delta_MedianAPFD
0,0,LatestFail,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0,LightGBM,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0,NaiveBayes,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0,QTF-Avg,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0,Random,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,50,NaiveBayes,30,-0.443717,0.362888,-0.657336,-0.533883,0.460034,-0.861941,-0.392758,0.375925,-0.680994,-0.526907,0.480426,-0.928700
59,50,QTF-Avg,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
60,50,Random,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
61,50,RandomForest,30,-0.553470,0.109948,-0.551379,-0.667262,0.161483,-0.672813,-0.577821,0.107313,-0.598542,-0.695565,0.167837,-0.753100



=== PROJECT 22 CELL 10 / STEP 5B RESULT ===
Project: apache@logging-log4j2
Step 5A checkpoint SHA-256: 600035cfbf9f8b2dfd0fd83d49050466e3c71e6187c1d631b24cfe57a6107a26
Frozen raw-root SHA-256: 374e4e1eaab266451539c9c7a4751fa6fe50250dccf57b19a5bc19cf8fad3f11
Independent current raw-root SHA-256: 374e4e1eaab266451539c9c7a4751fa6fe50250dccf57b19a5bc19cf8fad3f11

Raw-output revalidation:
Conditions: 270 / 270
Raw files: 2160 / 2160
Raw bytes: 646130653 / 646130653
Missing / unexpected / size / SHA mismatches: 0 / 0 / 0 / 0
Embedded output-manifest failures: 0

Experiment totals:
ML fits: 1080 / 1080
Ranking rows: 41874840 / 41874840
Build-metric rows: 73710 / 73710
Project-run rows: 1890 / 1890
Condition-audit rows: 270 / 270
Training-median rows: 40770 / 40770

Analysis-ready aggregates:
Noise-technique summary rows: 63
Seed-level noise-delta rows: 1890
Noise-delta summary rows: 63
Sample SD calculated with ddof=1: True
Ranking-level baseline-invariance failures: 0
Project-metric baselin

In [4]:
# ==================================================================================================
# PROJECT 22 — CELL 11 / STEP 5C
# REGISTRY-SCHEMA-COMPLETE, CROSS-FILESYSTEM-SAFE FINAL PACKAGE AND REGISTRATION
#
# PROJECT:
#   apache@logging-log4j2
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_22.ipynb.
#
# REQUIRED FROZEN INPUTS:
# - Project 22 Step 5A checkpoint SHA-256:
#   600035cfbf9f8b2dfd0fd83d49050466e3c71e6187c1d631b24cfe57a6107a26
# - Project 22 Step 5B checkpoint SHA-256:
#   9259c486debcda1794423bd16d4333083e14d203216df596327ec4658bfb04fb
# - Project 22 raw-root SHA-256:
#   374e4e1eaab266451539c9c7a4751fa6fe50250dccf57b19a5bc19cf8fad3f11
# - Project 22 source-root SHA-256:
#   281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64f842964c896c6ac334
# - Registry before registration:
#   exactly Projects 1–21, all COMPLETE_AND_FROZEN
# - Registry SHA-256 before registration:
#   79cd6ecb595c5e8ae91a9494e469792716338d144308560a62caf1b9342306b2
#
# SAFETY:
# - no model fitting;
# - no condition reruns;
# - no raw-result modification or deletion;
# - no prior-project condition-output access or write;
# - registry write only after package and candidate-row validation;
# - cross-filesystem-safe Google Drive staging and readback;
# - six frozen Project 22 Step 5A worker checkpoints are preserved and revalidated.
# ==================================================================================================

from google.colab import drive
from pathlib import Path
from IPython.display import display
import hashlib
import json
import os
import re
import shutil
import tempfile
import pandas as pd

print("=" * 136)
print("=== PROJECT 22 CELL 11 / STEP 5C: FINAL PACKAGE FREEZE AND REGISTRY REGISTRATION ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN IDENTITY, HASHES, AND COUNTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 22
PROJECT_NAME = "apache@logging-log4j2"
PROJECT_SLUG = "apache__logging-log4j2"
PROJECT_SHORT = "LOG4J2"

COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

STEP5C_STATUS = "PASS_PROJECT_22_FINAL_PACKAGE_FROZEN_AND_REGISTERED"
STEP5A_STATUS = "PASS_PROJECT_22_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
STEP5B_STATUS = "PASS_PROJECT_22_RAW_RESULTS_REVALIDATED_AND_COMPACT_AGGREGATES_FROZEN"

REGISTRY_SHA_BEFORE_EXPECTED = (
    "79cd6ecb595c5e8ae91a9494e469792716338d144308560a62caf1b9342306b2"
)

STEP5A_SHA_EXPECTED = (
    "600035cfbf9f8b2dfd0fd83d49050466e3c71e6187c1d631b24cfe57a6107a26"
)

STEP5B_SHA_EXPECTED = (
    "9259c486debcda1794423bd16d4333083e14d203216df596327ec4658bfb04fb"
)

SOURCE_ROOT_SHA = (
    "281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64f842964c896c6ac334"
)

RAW_ROOT_SHA = (
    "374e4e1eaab266451539c9c7a4751fa6fe50250dccf57b19a5bc19cf8fad3f11"
)

COUNTS = {
    "RawFiles": 2160,
    "RawBytes": 646130653,
    "Conditions": 270,
    "MLFits": 1080,
    "RankingRows": 41874840,
    "BuildMetricRows": 73710,
    "ProjectRunRows": 1890,
    "ConditionAuditRows": 270,
    "TrainingMedianRows": 40770,

    "Builds": 441,
    "TrainingBuilds": 330,
    "EvaluationBuilds": 111,

    "RawRows": 240253,
    "RawTrainingRows": 172628,
    "RawEvaluationRows": 67625,
    "RawTrainingFailures": 208,
    "RawEvaluationFailures": 40,

    "ModelRows": 117968,
    "ModelTrainingRows": 95812,
    "ModelEvaluationRows": 22156,
    "ModelTrainingFailures": 207,
    "ModelEvaluationFailures": 40,
    "ModelFailingEvaluationBuilds": 39,

    "Predictors": 151,
    "RECFeatures": 19,
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

drive.mount(
    "/content/drive",
    force_remount=False,
)

ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES = (
    ROOT
    / "Notes"
)

RESULTS = (
    ROOT
    / "Results"
)

REGISTRY = (
    NOTES
    / "completed_project_registry.csv"
)

PROJECT_ROOT = (
    RESULTS
    / "Aggregated"
    / PROJECT_SLUG
)

RAW_ROOT = (
    RESULTS
    / "Raw"
    / PROJECT_SLUG
)

FINAL_ROOT = (
    RESULTS
    / "Final"
    / PROJECT_SLUG
)

MANIFEST_PATH = (
    FINAL_ROOT
    / "final_package_manifest.csv"
)

SUMMARY_PATH = (
    FINAL_ROOT
    / "final_package_summary.json"
)

README_PATH = (
    FINAL_ROOT
    / "README.txt"
)

STEP5C_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5c"
)

VALIDATION_PATH = (
    STEP5C_ROOT
    / f"{PROJECT_SHORT}_step5c_validation.csv"
)

REPORT_PATH = (
    STEP5C_ROOT
    / f"{PROJECT_SHORT}_step5c_report.json"
)

STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5c_status.json"
)

CHECKPOINT_PATH = (
    NOTES
    / "project_22_step5c_checkpoint.json"
)

BACKUP_PATH = (
    NOTES
    / "completed_project_registry_before_project_22.csv"
)

STEP5B_RAW_MANIFEST_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5b"
    / f"{PROJECT_SHORT}_independent_raw_manifest.csv"
)

STEP5A_CP = (
    NOTES
    / "project_22_step5a_checkpoint.json"
)

STEP5B_CP = (
    NOTES
    / "project_22_step5b_checkpoint.json"
)

STEP5A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5a_status.json"
)

STEP5B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5b_status.json"
)

UPSTREAM_CPS = [
    NOTES / "project_22_selection_checkpoint.json",
    NOTES / "project_22_rec_reconstruction_checkpoint.json",
    NOTES / "project_22_noise_plan_checkpoint.json",
    NOTES / "project_22_runtime_contract_checkpoint.json",
    NOTES / "project_22_smoke_test_checkpoint.json",
    STEP5A_CP,
    STEP5B_CP,
]

WORKER_CHECKPOINTS = {
    NOTES / "project_22_step5a_worker_seed_01_05_checkpoint.json":
        "6edc0ecf1d33c3fd2263f968f7d953726e3b0ddf3a0a2b40b2cc34ecbb15bb44",

    NOTES / "project_22_step5a_worker_seed_06_10_checkpoint.json":
        "1a8ebadb53e01acd4408ac58c8235b4c6504bee2a6bb0213966563a4243aa0c8",

    NOTES / "project_22_step5a_worker_seed_11_15_checkpoint.json":
        "f02df169293ce4d1ae143552547a89654ce21e1084b641b4b3ce48ce9afbbc05",

    NOTES / "project_22_step5a_worker_seed_16_20_checkpoint.json":
        "0643261cc0bd54cadba8354fa88e726ab2e199b940aa2b3f609a581121663db4",

    NOTES / "project_22_step5a_worker_seed_21_25_checkpoint.json":
        "23806c7c5252cb4f4b5f6f5faee559678b8c6d02a023437405f7551eaa6c3718",

    NOTES / "project_22_step5a_worker_seed_26_30_checkpoint.json":
        "f7e03bb7c57f29782b057554bbe8116a25e14cce7336361154e7c60e3a0af173",
}


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha(
    path,
    chunk=8 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(
        path
    ).open(
        "rb"
    ) as f:
        for block in iter(
            lambda: f.read(
                chunk
            ),
            b"",
        ):
            h.update(
                block
            )

    return h.hexdigest()


def load_json(
    path,
):
    with Path(
        path
    ).open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(
            f
        )


def atomic_json(
    path,
    obj,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        f.write(
            "\n"
        )

    os.replace(
        tmp,
        path,
    )


def atomic_csv(
    path,
    df,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    df.to_csv(
        tmp,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        tmp,
        path,
    )


def atomic_text(
    path,
    text,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    tmp.write_text(
        text,
        encoding="utf-8",
    )

    os.replace(
        tmp,
        path,
    )


def resolve(
    cols,
    expected,
):
    matches = [
        c
        for c in cols
        if str(
            c
        ).strip().lower()
        == expected.lower()
    ]

    if len(
        matches
    ) != 1:
        raise RuntimeError(
            f"Could not resolve registry column {expected!r}; "
            f"matches={matches}; columns={list(cols)}"
        )

    return matches[
        0
    ]


def norm(
    value,
):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(
            value
        ).lower(),
    )


def manifest(
    root,
    exclude=(),
):
    root = Path(
        root
    )

    exclude = set(
        exclude
    )

    rows = []

    files = sorted(
        (
            path
            for path in root.rglob(
                "*"
            )
            if path.is_file()
        ),
        key=lambda path:
            path.relative_to(
                root
            ).as_posix(),
    )

    for path in files:
        rel = path.relative_to(
            root
        ).as_posix()

        if rel in exclude:
            continue

        rows.append(
            {
                "RelativePath": rel,
                "Bytes": int(
                    path.stat().st_size
                ),
                "SHA256": sha(
                    path
                ),
            }
        )

    return pd.DataFrame(
        rows,
        columns=[
            "RelativePath",
            "Bytes",
            "SHA256",
        ],
    )


def root_hash(
    df,
):
    h = hashlib.sha256()

    ordered = df.sort_values(
        "RelativePath",
        kind="mergesort",
    )

    for row in ordered.itertuples(
        index=False
    ):
        h.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode()
        )

    return h.hexdigest()


def verify_manifest(
    items,
    label,
):
    if (
        not isinstance(
            items,
            list,
        )
        or not items
    ):
        raise RuntimeError(
            f"{label} has no output manifest."
        )

    rows = []

    for item in items:
        path = Path(
            item[
                "Path"
            ]
        )

        exists = path.is_file()

        expected_bytes = int(
            item[
                "Bytes"
            ]
        )

        expected_sha = str(
            item[
                "SHA256"
            ]
        ).lower()

        if exists:
            actual_bytes = int(
                path.stat().st_size
            )

            actual_sha = sha(
                path
            )
        else:
            actual_bytes = -1
            actual_sha = "MISSING"

        rows.append(
            {
                "Path": str(
                    path
                ),
                "ExpectedBytes": expected_bytes,
                "ActualBytes": actual_bytes,
                "ExpectedSHA256": expected_sha,
                "ActualSHA256": actual_sha,
                "Pass": bool(
                    exists
                    and expected_bytes
                    == actual_bytes
                    and expected_sha
                    == actual_sha
                ),
            }
        )

    out = pd.DataFrame(
        rows
    )

    if not out[
        "Pass"
    ].all():
        display(
            out.loc[
                ~out[
                    "Pass"
                ]
            ]
        )

        raise RuntimeError(
            f"{label} manifest verification failed."
        )

    return out


def check(
    rows,
    name,
    expected,
    actual,
    passed,
):
    rows.append(
        {
            "Check": name,
            "Expected": expected,
            "Actual": actual,
            "Pass": bool(
                passed
            ),
        }
    )


# --------------------------------------------------------------------------------------------------
# 4. VERIFY ALL REQUIRED INPUTS BEFORE ANY PACKAGE OR REGISTRY WRITE
# --------------------------------------------------------------------------------------------------

required = [
    REGISTRY,
    RAW_ROOT,
    STEP5A_CP,
    STEP5B_CP,
    STEP5A_STATUS_PATH,
    STEP5B_STATUS_PATH,
    *UPSTREAM_CPS,
    *WORKER_CHECKPOINTS,
]

missing = [
    str(
        path
    )
    for path in required
    if not Path(
        path
    ).exists()
]

if missing:
    raise FileNotFoundError(
        "Missing Project 22 Step 5C inputs:\n"
        + "\n".join(
            missing
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY STEP 5A / STEP 5B FREEZES
# --------------------------------------------------------------------------------------------------

step5a_sha = sha(
    STEP5A_CP
)

step5b_sha = sha(
    STEP5B_CP
)

if step5a_sha != STEP5A_SHA_EXPECTED:
    raise RuntimeError(
        "Step 5A checkpoint SHA differs.\n"
        f"Expected: {STEP5A_SHA_EXPECTED}\n"
        f"Actual:   {step5a_sha}"
    )

if step5b_sha != STEP5B_SHA_EXPECTED:
    raise RuntimeError(
        "Step 5B checkpoint SHA differs.\n"
        f"Expected: {STEP5B_SHA_EXPECTED}\n"
        f"Actual:   {step5b_sha}"
    )

step5a = load_json(
    STEP5A_CP
)

step5b = load_json(
    STEP5B_CP
)

step5a_status_payload = load_json(
    STEP5A_STATUS_PATH
)

step5b_status_payload = load_json(
    STEP5B_STATUS_PATH
)

for label, payload, expected in [
    (
        "Step 5A checkpoint",
        step5a,
        STEP5A_STATUS,
    ),
    (
        "Step 5A status",
        step5a_status_payload,
        STEP5A_STATUS,
    ),
    (
        "Step 5B checkpoint",
        step5b,
        STEP5B_STATUS,
    ),
    (
        "Step 5B status",
        step5b_status_payload,
        STEP5B_STATUS,
    ),
]:
    if payload.get(
        "Status"
    ) != expected:
        raise RuntimeError(
            f"{label} is not in expected PASS state."
        )

if (
    step5a.get(
        "SourceRootSHA256"
    )
    != SOURCE_ROOT_SHA
):
    raise RuntimeError(
        "Step 5A source-root SHA differs."
    )

if (
    step5a.get(
        "RawRootSHA256"
    )
    != RAW_ROOT_SHA
):
    raise RuntimeError(
        "Step 5A raw-root SHA differs."
    )

if (
    step5b.get(
        "SourceRootSHA256"
    )
    != SOURCE_ROOT_SHA
):
    raise RuntimeError(
        "Step 5B source-root SHA differs."
    )

if (
    step5b.get(
        "FrozenRawRootSHA256"
    )
    != RAW_ROOT_SHA
):
    raise RuntimeError(
        "Step 5B frozen raw-root SHA differs."
    )

if (
    step5b.get(
        "IndependentRawRootSHA256"
    )
    != RAW_ROOT_SHA
):
    raise RuntimeError(
        "Step 5B independent raw-root SHA differs."
    )

if (
    step5b.get(
        "Step5ACheckpointSHA256"
    )
    != STEP5A_SHA_EXPECTED
):
    raise RuntimeError(
        "Step 5B does not link to the frozen Step 5A checkpoint."
    )

if not bool(
    step5b.get(
        "ReadyForFinalPackageAndRegistration",
        False,
    )
):
    raise RuntimeError(
        "Step 5B is not marked ready for final package and registration."
    )

if bool(
    step5b.get(
        "RegistryModified",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports that the completion registry was modified."
    )

if bool(
    step5b.get(
        "PriorProjectConditionOutputsAccessed",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports access to prior-project condition outputs."
    )

if bool(
    step5b.get(
        "PriorProjectConditionOutputsModified",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports modification of prior-project condition outputs."
    )

if bool(
    step5b.get(
        "PriorProjectWriteAttempted",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports a prior-project write attempt."
    )

if bool(
    step5b.get(
        "ModelsFitted",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports model fitting."
    )

if bool(
    step5b.get(
        "ConditionsRerun",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports condition reruns."
    )


# --------------------------------------------------------------------------------------------------
# 6. VERIFY SIX FROZEN PARALLEL WORKER CHECKPOINTS
# --------------------------------------------------------------------------------------------------

worker_checkpoint_sha_mismatches = 0

for worker_path, expected_worker_sha in WORKER_CHECKPOINTS.items():
    actual_worker_sha = sha(
        worker_path
    )

    worker_checkpoint_sha_mismatches += int(
        actual_worker_sha
        != expected_worker_sha
    )

expected_worker_hashes_by_tag = {
    "seed_01_05":
        WORKER_CHECKPOINTS[
            NOTES
            / "project_22_step5a_worker_seed_01_05_checkpoint.json"
        ],

    "seed_06_10":
        WORKER_CHECKPOINTS[
            NOTES
            / "project_22_step5a_worker_seed_06_10_checkpoint.json"
        ],

    "seed_11_15":
        WORKER_CHECKPOINTS[
            NOTES
            / "project_22_step5a_worker_seed_11_15_checkpoint.json"
        ],

    "seed_16_20":
        WORKER_CHECKPOINTS[
            NOTES
            / "project_22_step5a_worker_seed_16_20_checkpoint.json"
        ],

    "seed_21_25":
        WORKER_CHECKPOINTS[
            NOTES
            / "project_22_step5a_worker_seed_21_25_checkpoint.json"
        ],

    "seed_26_30":
        WORKER_CHECKPOINTS[
            NOTES
            / "project_22_step5a_worker_seed_26_30_checkpoint.json"
        ],
}

step5a_worker_hashes = step5a.get(
    "ParallelWorkerCheckpointSHA256",
    {},
)

if (
    step5a_worker_hashes
    != expected_worker_hashes_by_tag
):
    raise RuntimeError(
        "Step 5A parallel-worker checkpoint SHA map differs "
        "from the six frozen Project 22 workers."
    )

if worker_checkpoint_sha_mismatches:
    raise RuntimeError(
        f"{worker_checkpoint_sha_mismatches} frozen Project 22 "
        "parallel-worker checkpoints changed."
    )


# --------------------------------------------------------------------------------------------------
# 7. VERIFY STEP 5A / STEP 5B OUTPUT MANIFESTS
# --------------------------------------------------------------------------------------------------

step5a_audit = verify_manifest(
    step5a.get(
        "AggregateOutputManifest",
        [],
    ),
    "Step 5A aggregate",
)

step5b_audit = verify_manifest(
    step5b.get(
        "OutputManifest",
        [],
    ),
    "Step 5B",
)


# --------------------------------------------------------------------------------------------------
# 8. VERIFY COMPLETION REGISTRY PRE-STATE
# --------------------------------------------------------------------------------------------------

registry_sha_before = sha(
    REGISTRY
)

if (
    registry_sha_before
    != REGISTRY_SHA_BEFORE_EXPECTED
):
    raise RuntimeError(
        "Registry SHA differs before Project 22 registration:\n"
        f"Expected: {REGISTRY_SHA_BEFORE_EXPECTED}\n"
        f"Actual:   {registry_sha_before}"
    )

if (
    step5b.get(
        "RegistrySHA256"
    )
    != REGISTRY_SHA_BEFORE_EXPECTED
):
    raise RuntimeError(
        "The frozen Step 5B checkpoint does not reference "
        "the expected pre-Project-22 registry SHA."
    )

reg_before = pd.read_csv(
    REGISTRY,
    dtype=str,
).fillna(
    ""
)

pn_col = resolve(
    reg_before.columns,
    "ProjectNumber",
)

project_col = resolve(
    reg_before.columns,
    "Project",
)

status_col = resolve(
    reg_before.columns,
    "Status",
)

pnums = pd.to_numeric(
    reg_before[
        pn_col
    ],
    errors="raise",
).astype(
    int
)

if (
    len(
        reg_before
    )
    != 21
    or sorted(
        pnums.tolist()
    )
    != list(
        range(
            1,
            22,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–21."
    )

if not reg_before[
    status_col
].eq(
    COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–21 are not all COMPLETE_AND_FROZEN."
    )

required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
}

for (
    required_number,
    required_project,
) in required_registered_identities.items():
    matching_rows = reg_before.loc[
        pnums.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_col
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )

if (
    pnums.eq(
        PROJECT_NUMBER
    ).any()
    or reg_before[
        project_col
    ].eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 22 is already present in the completion registry."
    )


# --------------------------------------------------------------------------------------------------
# 9. FREEZE A PRE-PROJECT-22 REGISTRY BACKUP
# --------------------------------------------------------------------------------------------------

if not BACKUP_PATH.exists():
    shutil.copy2(
        REGISTRY,
        BACKUP_PATH,
    )

if sha(
    BACKUP_PATH
) != registry_sha_before:
    raise RuntimeError(
        "Pre-Project-22 registry backup does not match the live registry."
    )


# --------------------------------------------------------------------------------------------------
# 10. ASSEMBLE FINAL COMPACT PACKAGE
# --------------------------------------------------------------------------------------------------

sources = set(
    Path(
        path
    )
    for path in UPSTREAM_CPS
)

sources.update(
    WORKER_CHECKPOINTS.keys()
)

sources.update(
    [
        STEP5A_STATUS_PATH,
        STEP5B_STATUS_PATH,
    ]
)

sources.update(
    Path(
        item[
            "Path"
        ]
    )
    for item in step5a.get(
        "AggregateOutputManifest",
        [],
    )
)

sources.update(
    Path(
        item[
            "Path"
        ]
    )
    for item in step5b.get(
        "OutputManifest",
        [],
    )
)

sources = sorted(
    sources,
    key=str,
)

missing_sources = [
    str(
        path
    )
    for path in sources
    if not path.is_file()
]

if missing_sources:
    raise FileNotFoundError(
        "Missing compact-package sources:\n"
        + "\n".join(
            missing_sources
        )
    )

created_at = str(
    step5b.get(
        "CompletedAtUTC",
        "",
    )
).strip()

if not created_at:
    raise RuntimeError(
        "The frozen Step 5B checkpoint contains no CompletedAtUTC timestamp."
    )

tmp_root = Path(
    tempfile.mkdtemp(
        prefix="project22_package_",
        dir="/content",
    )
)

drive_staging_root = None

try:
    for src in sources:
        try:
            rel = src.relative_to(
                ROOT
            )
        except ValueError as exc:
            raise RuntimeError(
                f"Package source is outside thesis root: {src}"
            ) from exc

        dst = (
            tmp_root
            / rel
        )

        dst.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        shutil.copy2(
            src,
            dst,
        )

    atomic_text(
        tmp_root
        / "README.txt",
        f"""PROJECT 22 FINAL COMPACT PACKAGE

Project number: {PROJECT_NUMBER}
Project: {PROJECT_NAME}
Project slug: {PROJECT_SLUG}
Status: {COMPLETE_STATUS}
Created at UTC: {created_at}

The 2,160 raw condition-output files are not duplicated here.
Raw results: {RAW_ROOT}
Raw-root SHA-256: {RAW_ROOT_SHA}
Source-root SHA-256: {SOURCE_ROOT_SHA}
Primary metric: APFDc
Secondary metric: APFD
Parallel Step 5A worker checkpoints preserved in this package: 6
""",
    )

    before_summary = manifest(
        tmp_root
    )

    atomic_json(
        tmp_root
        / "final_package_summary.json",
        {
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "Status": COMPLETE_STATUS,
            "CreatedAtUTC": created_at,
            "SourceRootSHA256": SOURCE_ROOT_SHA,
            "RawRootSHA256": RAW_ROOT_SHA,
            **COUNTS,
            "Step5ACheckpointSHA256": step5a_sha,
            "Step5BCheckpointSHA256": step5b_sha,
            "PayloadRootSHA256BeforeSummary": root_hash(
                before_summary
            ),
            "RawResultsDuplicatedIntoPackage": False,
            "ParallelWorkerCheckpointSHA256":
                expected_worker_hashes_by_tag,
            "PriorProjectConditionOutputsAccessed": False,
            "PriorProjectConditionOutputsModified": False,
            "PriorProjectWriteAttempted": False,
        },
    )

    candidate_manifest = manifest(
        tmp_root,
        {
            "final_package_manifest.csv",
        },
    )

    package_root_sha = root_hash(
        candidate_manifest
    )

    atomic_csv(
        tmp_root
        / "final_package_manifest.csv",
        candidate_manifest,
    )

    package_files = (
        len(
            candidate_manifest
        )
        + 1
    )

    package_bytes = int(
        candidate_manifest[
            "Bytes"
        ].sum()
        + (
            tmp_root
            / "final_package_manifest.csv"
        ).stat().st_size
    )

    if FINAL_ROOT.exists():
        if not MANIFEST_PATH.is_file():
            raise RuntimeError(
                "An existing Project 22 final-package directory has no manifest "
                "and was not modified."
            )

        existing = pd.read_csv(
            MANIFEST_PATH,
            low_memory=False,
        )

        if root_hash(
            existing
        ) != package_root_sha:
            raise RuntimeError(
                "A different Project 22 final package already exists "
                "and was not modified."
            )

        shutil.rmtree(
            tmp_root
        )

        package_already_frozen = True

    else:
        # /content and Google Drive are different filesystems.
        # Publish in two stages:
        #   1. copy completed local package to a sibling Drive staging directory;
        #   2. verify the staged package exactly;
        #   3. rename staging -> FINAL_ROOT within Google Drive.
        FINAL_ROOT.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        drive_staging_root = FINAL_ROOT.with_name(
            f"{FINAL_ROOT.name}__staging_{os.getpid()}"
        )

        if drive_staging_root.exists():
            shutil.rmtree(
                drive_staging_root
            )

        shutil.copytree(
            tmp_root,
            drive_staging_root,
            copy_function=shutil.copy2,
        )

        staged_manifest_path = (
            drive_staging_root
            / "final_package_manifest.csv"
        )

        if not staged_manifest_path.is_file():
            raise RuntimeError(
                "The Google Drive staging package has no manifest."
            )

        staged_manifest = pd.read_csv(
            staged_manifest_path,
            low_memory=False,
        )

        staged_root_sha = root_hash(
            staged_manifest
        )

        if staged_root_sha != package_root_sha:
            raise RuntimeError(
                "The Google Drive staging package root SHA-256 differs.\n"
                f"Expected: {package_root_sha}\n"
                f"Actual:   {staged_root_sha}"
            )

        staged_payload_manifest = manifest(
            drive_staging_root,
            {
                "final_package_manifest.csv",
            },
        )

        candidate_payload_manifest = (
            candidate_manifest.sort_values(
                "RelativePath",
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        staged_payload_manifest = (
            staged_payload_manifest.sort_values(
                "RelativePath",
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        if not staged_payload_manifest.equals(
            candidate_payload_manifest
        ):
            comparison = candidate_payload_manifest.merge(
                staged_payload_manifest,
                on="RelativePath",
                how="outer",
                suffixes=(
                    "_candidate",
                    "_staged",
                ),
                indicator=True,
            )

            failed_comparison = comparison.loc[
                (
                    comparison[
                        "_merge"
                    ].ne(
                        "both"
                    )
                    | comparison[
                        "Bytes_candidate"
                    ].ne(
                        comparison[
                            "Bytes_staged"
                        ]
                    )
                    | comparison[
                        "SHA256_candidate"
                    ].ne(
                        comparison[
                            "SHA256_staged"
                        ]
                    )
                )
            ]

            raise RuntimeError(
                "The Google Drive staging package failed exact file-level "
                "verification.\n"
                + failed_comparison.head(
                    20
                ).to_string(
                    index=False
                )
            )

        os.replace(
            drive_staging_root,
            FINAL_ROOT,
        )

        drive_staging_root = None

        shutil.rmtree(
            tmp_root
        )

        package_already_frozen = False

except Exception:
    if tmp_root.exists():
        shutil.rmtree(
            tmp_root,
            ignore_errors=True,
        )

    if (
        drive_staging_root is not None
        and drive_staging_root.exists()
    ):
        shutil.rmtree(
            drive_staging_root,
            ignore_errors=True,
        )

    raise


# --------------------------------------------------------------------------------------------------
# 11. READ BACK EVERY FINAL-PACKAGE FILE
# --------------------------------------------------------------------------------------------------

pkg_manifest = pd.read_csv(
    MANIFEST_PATH,
    low_memory=False,
)

manifest_paths = set(
    pkg_manifest[
        "RelativePath"
    ].astype(
        str
    )
)

actual_paths = {
    path.relative_to(
        FINAL_ROOT
    ).as_posix()
    for path in FINAL_ROOT.rglob(
        "*"
    )
    if path.is_file()
}

expected_paths = (
    manifest_paths
    | {
        "final_package_manifest.csv",
    }
)

missing_pkg = len(
    expected_paths
    - actual_paths
)

unexpected_pkg = len(
    actual_paths
    - expected_paths
)

size_bad = 0
hash_bad = 0

for row in pkg_manifest.itertuples(
    index=False
):
    path = (
        FINAL_ROOT
        / str(
            row.RelativePath
        )
    )

    if path.is_file():
        size_bad += int(
            path.stat().st_size
            != int(
                row.Bytes
            )
        )

        hash_bad += int(
            sha(
                path
            )
            != str(
                row.SHA256
            )
        )

package_root_readback = root_hash(
    pkg_manifest
)

if (
    package_root_readback
    != package_root_sha
    or any(
        [
            missing_pkg,
            unexpected_pkg,
            size_bad,
            hash_bad,
        ]
    )
):
    raise RuntimeError(
        "Final Project 22 package failed readback validation."
    )


# --------------------------------------------------------------------------------------------------
# 12. BUILD A COMPLETE PROJECT 22 REGISTRY ROW
# --------------------------------------------------------------------------------------------------
#
# The registry schema has evolved across projects.
# Use Project 21's exact formatting for protocol fields whose textual
# representation may have varied historically, while filling Project 22's
# project-specific values explicitly.
# --------------------------------------------------------------------------------------------------

project_21_template_rows = reg_before.loc[
    pd.to_numeric(
        reg_before[
            pn_col
        ],
        errors="raise",
    ).astype(
        int
    ).eq(
        21
    )
]

if len(
    project_21_template_rows
) != 1:
    raise RuntimeError(
        "Could not resolve exactly one Project 21 registry template row."
    )

project_21_template = project_21_template_rows.iloc[
    0
]

protocol_template_values = {}

for registry_column in reg_before.columns:
    normalised_column = norm(
        registry_column
    )

    if normalised_column in {
        "seeds",
        "noiselevels",
        "techniques",
        "donotrerun",
    }:
        protocol_template_values[
            normalised_column
        ] = str(
            project_21_template[
                registry_column
            ]
        ).strip()

protocol_fallback_values = {
    "seeds":
        json.dumps(
            list(
                range(
                    1,
                    31,
                )
            ),
            separators=(
                ",",
                ":",
            ),
        ),

    "noiselevels":
        json.dumps(
            [
                0,
                5,
                10,
                15,
                20,
                25,
                30,
                40,
                50,
            ],
            separators=(
                ",",
                ":",
            ),
        ),

    "techniques":
        json.dumps(
            [
                "RandomForest",
                "XGBoost",
                "LightGBM",
                "NaiveBayes",
                "Random",
                "LatestFail",
                "QTF-Avg",
            ],
            separators=(
                ",",
                ":",
            ),
        ),

    "donotrerun":
        "True",
}

for (
    protocol_key,
    fallback_value,
) in protocol_fallback_values.items():
    if not protocol_template_values.get(
        protocol_key,
        "",
    ):
        protocol_template_values[
            protocol_key
        ] = fallback_value


values = {
    "projectnumber": PROJECT_NUMBER,
    "projectno": PROJECT_NUMBER,
    "project": PROJECT_NAME,
    "projectname": PROJECT_NAME,
    "projectslug": PROJECT_SLUG,
    "slug": PROJECT_SLUG,

    "status": COMPLETE_STATUS,
    "completionstatus": COMPLETE_STATUS,

    "completedatutc": created_at,
    "completedat": created_at,
    "frozenatutc": created_at,
    "frozenat": created_at,
    "registeredatutc": created_at,
    "registeredat": created_at,

    "sourcerootsha256": SOURCE_ROOT_SHA,

    "rawrootsha256": RAW_ROOT_SHA,
    "rawresultrootsha256": RAW_ROOT_SHA,
    "rawresultsrootsha256": RAW_ROOT_SHA,

    "rawroot": str(
        RAW_ROOT
    ),
    "rawresultroot": str(
        RAW_ROOT
    ),

    "finalpackagepath": str(
        FINAL_ROOT
    ),
    "packagepath": str(
        FINAL_ROOT
    ),
    "finalpackageroot": str(
        FINAL_ROOT
    ),

    "finalpackagerootsha256": package_root_sha,
    "packagerootsha256": package_root_sha,
    "packagesha256": package_root_sha,

    "packagefiles": package_files,
    "packagefilecount": package_files,
    "packagebytes": package_bytes,

    "step5acheckpointsha256": step5a_sha,
    "step5bcheckpointsha256": step5b_sha,

    # Complete observed registry schema.
    "seeds":
        protocol_template_values[
            "seeds"
        ],

    "noiselevels":
        protocol_template_values[
            "noiselevels"
        ],

    "techniques":
        protocol_template_values[
            "techniques"
        ],

    "evaluationrows":
        COUNTS[
            "ModelEvaluationRows"
        ],

    "evaluationfailures":
        COUNTS[
            "ModelEvaluationFailures"
        ],

    "finaldirectory":
        str(
            FINAL_ROOT
        ),

    "finalauditreport":
        str(
            REPORT_PATH
        ),

    "donotrerun":
        protocol_template_values[
            "donotrerun"
        ],

    "freezerecord":
        str(
            CHECKPOINT_PATH
        ),

    "rawresultsmanifest":
        str(
            STEP5B_RAW_MANIFEST_PATH
        ),

    "finalpackagemanifest":
        str(
            MANIFEST_PATH
        ),

    "finalauditstatus":
        STEP5C_STATUS,
}

for (
    key,
    value,
) in COUNTS.items():
    values[
        norm(
            key
        )
    ] = value

values.update(
    {
        "rawfilecount":
            COUNTS[
                "RawFiles"
            ],

        "conditioncount":
            COUNTS[
                "Conditions"
            ],

        "modelfits":
            COUNTS[
                "MLFits"
            ],

        "modelreadyrows":
            COUNTS[
                "ModelRows"
            ],

        "predictorcount":
            COUNTS[
                "Predictors"
            ],

        "recfeaturecount":
            COUNTS[
                "RECFeatures"
            ],

        "rawexecutionrows":
            COUNTS[
                "RawRows"
            ],
    }
)

if not STEP5B_RAW_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "The independently frozen Step 5B raw-results manifest is missing:\n"
        f"{STEP5B_RAW_MANIFEST_PATH}"
    )

new_row = {}
unresolved = []

for col in reg_before.columns:
    normalised = norm(
        col
    )

    if normalised in values:
        new_row[
            col
        ] = str(
            values[
                normalised
            ]
        )

    else:
        unique_nonempty = sorted(
            set(
                value
                for value in reg_before[
                    col
                ].astype(
                    str
                ).str.strip()
                if value
            )
        )

        if len(
            unique_nonempty
        ) == 1:
            # Preserve a global protocol constant.
            new_row[
                col
            ] = unique_nonempty[
                0
            ]

        elif reg_before[
            col
        ].astype(
            str
        ).str.strip().eq(
            ""
        ).all():
            new_row[
                col
            ] = ""

        else:
            new_row[
                col
            ] = ""

            unresolved.append(
                col
            )

new_row[
    pn_col
] = str(
    PROJECT_NUMBER
)

new_row[
    project_col
] = PROJECT_NAME

new_row[
    status_col
] = COMPLETE_STATUS

if unresolved:
    raise RuntimeError(
        "Unexpected unmapped registry columns remain; "
        "no registry write was attempted:\n"
        + "\n".join(
            unresolved
        )
    )

reg_candidate = pd.concat(
    [
        reg_before,
        pd.DataFrame(
            [
                new_row
            ]
        ),
    ],
    ignore_index=True,
)

reg_candidate[
    pn_col
] = pd.to_numeric(
    reg_candidate[
        pn_col
    ],
    errors="raise",
).astype(
    int
).astype(
    str
)

candidate_nums = pd.to_numeric(
    reg_candidate[
        pn_col
    ],
    errors="raise",
).astype(
    int
)

project22_candidate = reg_candidate.loc[
    candidate_nums.eq(
        22
    )
]

if (
    len(
        reg_candidate
    )
    != 22
    or sorted(
        candidate_nums.tolist()
    )
    != list(
        range(
            1,
            23,
        )
    )
):
    raise RuntimeError(
        "Candidate registry does not contain exactly Projects 1–22."
    )

if (
    not reg_candidate[
        status_col
    ].eq(
        COMPLETE_STATUS
    ).all()
    or len(
        project22_candidate
    )
    != 1
    or project22_candidate.iloc[
        0
    ][
        project_col
    ]
    != PROJECT_NAME
):
    raise RuntimeError(
        "Candidate Project 22 registry row failed status/identity validation."
    )


# --------------------------------------------------------------------------------------------------
# 13. PRE-WRITE VALIDATION
# --------------------------------------------------------------------------------------------------

rows = []

check(
    rows,
    "Step 5A checkpoint SHA-256",
    STEP5A_SHA_EXPECTED,
    step5a_sha,
    step5a_sha
    == STEP5A_SHA_EXPECTED,
)

check(
    rows,
    "Step 5B checkpoint SHA-256",
    STEP5B_SHA_EXPECTED,
    step5b_sha,
    step5b_sha
    == STEP5B_SHA_EXPECTED,
)

check(
    rows,
    "Source root SHA-256",
    SOURCE_ROOT_SHA,
    step5a.get(
        "SourceRootSHA256"
    ),
    step5a.get(
        "SourceRootSHA256"
    )
    == SOURCE_ROOT_SHA,
)

check(
    rows,
    "Raw root SHA-256",
    RAW_ROOT_SHA,
    step5b.get(
        "IndependentRawRootSHA256"
    ),
    step5b.get(
        "IndependentRawRootSHA256"
    )
    == RAW_ROOT_SHA,
)

check(
    rows,
    "Step 5A manifest failures",
    0,
    int(
        (
            ~step5a_audit[
                "Pass"
            ]
        ).sum()
    ),
    step5a_audit[
        "Pass"
    ].all(),
)

check(
    rows,
    "Step 5B manifest failures",
    0,
    int(
        (
            ~step5b_audit[
                "Pass"
            ]
        ).sum()
    ),
    step5b_audit[
        "Pass"
    ].all(),
)

check(
    rows,
    "Parallel worker checkpoint SHA mismatches",
    0,
    worker_checkpoint_sha_mismatches,
    worker_checkpoint_sha_mismatches
    == 0,
)

check(
    rows,
    "Parallel worker checkpoints",
    6,
    len(
        expected_worker_hashes_by_tag
    ),
    len(
        expected_worker_hashes_by_tag
    )
    == 6,
)

check(
    rows,
    "Step 5B prior-project output access",
    False,
    bool(
        step5b.get(
            "PriorProjectConditionOutputsAccessed",
            True,
        )
    ),
    not bool(
        step5b.get(
            "PriorProjectConditionOutputsAccessed",
            True,
        )
    ),
)

check(
    rows,
    "Step 5B prior-project output modification",
    False,
    bool(
        step5b.get(
            "PriorProjectConditionOutputsModified",
            True,
        )
    ),
    not bool(
        step5b.get(
            "PriorProjectConditionOutputsModified",
            True,
        )
    ),
)

check(
    rows,
    "Package missing files",
    0,
    missing_pkg,
    missing_pkg
    == 0,
)

check(
    rows,
    "Package unexpected files",
    0,
    unexpected_pkg,
    unexpected_pkg
    == 0,
)

check(
    rows,
    "Package size mismatches",
    0,
    size_bad,
    size_bad
    == 0,
)

check(
    rows,
    "Package SHA-256 mismatches",
    0,
    hash_bad,
    hash_bad
    == 0,
)

check(
    rows,
    "Registry rows before",
    21,
    len(
        reg_before
    ),
    len(
        reg_before
    )
    == 21,
)

check(
    rows,
    "Registry rows candidate",
    22,
    len(
        reg_candidate
    ),
    len(
        reg_candidate
    )
    == 22,
)

check(
    rows,
    "Candidate Project 22 rows",
    1,
    len(
        project22_candidate
    ),
    len(
        project22_candidate
    )
    == 1,
)

check(
    rows,
    "Unresolved variable registry columns",
    0,
    len(
        unresolved
    ),
    len(
        unresolved
    )
    == 0,
)

required_registry_field_expectations = {
    "Seeds":
        protocol_template_values[
            "seeds"
        ],

    "NoiseLevels":
        protocol_template_values[
            "noiselevels"
        ],

    "Techniques":
        protocol_template_values[
            "techniques"
        ],

    "EvaluationRows":
        str(
            COUNTS[
                "ModelEvaluationRows"
            ]
        ),

    "EvaluationFailures":
        str(
            COUNTS[
                "ModelEvaluationFailures"
            ]
        ),

    "FinalDirectory":
        str(
            FINAL_ROOT
        ),

    "FinalAuditReport":
        str(
            REPORT_PATH
        ),

    "DoNotRerun":
        protocol_template_values[
            "donotrerun"
        ],

    "FreezeRecord":
        str(
            CHECKPOINT_PATH
        ),

    "RawResultsManifest":
        str(
            STEP5B_RAW_MANIFEST_PATH
        ),

    "FinalPackageManifest":
        str(
            MANIFEST_PATH
        ),

    "RawResultsRootSHA256":
        RAW_ROOT_SHA,

    "FinalAuditStatus":
        STEP5C_STATUS,
}

registry_field_validation_failures = 0

for (
    expected_column_name,
    expected_value,
) in required_registry_field_expectations.items():
    matching_columns = [
        column
        for column in reg_before.columns
        if norm(
            column
        )
        == norm(
            expected_column_name
        )
    ]

    if len(
        matching_columns
    ) != 1:
        registry_field_validation_failures += 1
        continue

    actual_value = str(
        project22_candidate.iloc[
            0
        ][
            matching_columns[
                0
            ]
        ]
    )

    registry_field_validation_failures += int(
        actual_value
        != str(
            expected_value
        )
    )

check(
    rows,
    "Explicit Project 22 registry-field failures",
    0,
    registry_field_validation_failures,
    registry_field_validation_failures
    == 0,
)

pre = pd.DataFrame(
    rows
)

print(
    "\nProject 22 Step 5C pre-write validation:"
)

display(
    pre
)

print(
    "\nProject 22 registry row candidate:"
)

display(
    project22_candidate
)

if not pre[
    "Pass"
].all():
    raise RuntimeError(
        "PROJECT 22 STEP 5C PRE-WRITE VALIDATION FAILED. "
        "Registry not modified."
    )


# --------------------------------------------------------------------------------------------------
# 14. ATOMIC COMPLETION-REGISTRY WRITE
# --------------------------------------------------------------------------------------------------

tmp_reg = REGISTRY.with_name(
    f".{REGISTRY.name}.project22_{os.getpid()}"
)

reg_candidate.to_csv(
    tmp_reg,
    index=False,
    lineterminator="\n",
)

tmp_read = pd.read_csv(
    tmp_reg,
    dtype=str,
).fillna(
    ""
)

tmp_nums = pd.to_numeric(
    tmp_read[
        pn_col
    ],
    errors="raise",
).astype(
    int
)

if (
    len(
        tmp_read
    )
    != 22
    or sorted(
        tmp_nums.tolist()
    )
    != list(
        range(
            1,
            23,
        )
    )
    or not tmp_read[
        status_col
    ].eq(
        COMPLETE_STATUS
    ).all()
    or int(
        tmp_nums.eq(
            22
        ).sum()
    )
    != 1
):
    tmp_reg.unlink(
        missing_ok=True
    )

    raise RuntimeError(
        "Temporary Project 22 registry failed readback; "
        "live registry unchanged."
    )

os.replace(
    tmp_reg,
    REGISTRY,
)


# --------------------------------------------------------------------------------------------------
# 15. POST-WRITE REGISTRY VALIDATION
# --------------------------------------------------------------------------------------------------

reg_after = pd.read_csv(
    REGISTRY,
    dtype=str,
).fillna(
    ""
)

after_nums = pd.to_numeric(
    reg_after[
        pn_col
    ],
    errors="raise",
).astype(
    int
)

project22_after = reg_after.loc[
    after_nums.eq(
        22
    )
]

registry_sha_after = sha(
    REGISTRY
)

if (
    len(
        reg_after
    )
    != 22
    or sorted(
        after_nums.tolist()
    )
    != list(
        range(
            1,
            23,
        )
    )
    or not reg_after[
        status_col
    ].eq(
        COMPLETE_STATUS
    ).all()
    or len(
        project22_after
    )
    != 1
    or project22_after.iloc[
        0
    ][
        project_col
    ]
    != PROJECT_NAME
):
    raise RuntimeError(
        "Live registry failed post-write validation.\n"
        f"Backup: {BACKUP_PATH}"
    )

for (
    required_number,
    required_project,
) in required_registered_identities.items():
    matching_rows = reg_after.loc[
        after_nums.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_col
        ]
        != required_project
    ):
        raise RuntimeError(
            "A predecessor registry identity changed after "
            "Project 22 registration."
        )

check(
    rows,
    "Registry rows after",
    22,
    len(
        reg_after
    ),
    len(
        reg_after
    )
    == 22,
)

check(
    rows,
    "COMPLETE_AND_FROZEN projects after",
    22,
    int(
        reg_after[
            status_col
        ].eq(
            COMPLETE_STATUS
        ).sum()
    ),
    int(
        reg_after[
            status_col
        ].eq(
            COMPLETE_STATUS
        ).sum()
    )
    == 22,
)

check(
    rows,
    "Registry Project 22 rows after",
    1,
    len(
        project22_after
    ),
    len(
        project22_after
    )
    == 1,
)

check(
    rows,
    "Registry SHA changed",
    True,
    registry_sha_after
    != registry_sha_before,
    registry_sha_after
    != registry_sha_before,
)

for number in range(
    11,
    22,
):
    check(
        rows,
        f"Registry Project {number} rows after",
        1,
        int(
            after_nums.eq(
                number
            ).sum()
        ),
        int(
            after_nums.eq(
                number
            ).sum()
        )
        == 1,
    )

validation = pd.DataFrame(
    rows
)

failed = validation.loc[
    ~validation[
        "Pass"
    ]
]

if not failed.empty:
    display(
        failed
    )

    raise RuntimeError(
        "PROJECT 22 STEP 5C POST-WRITE VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 16. WRITE STEP 5C AUDIT / CHECKPOINT / STATUS
# --------------------------------------------------------------------------------------------------

STEP5C_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

atomic_csv(
    VALIDATION_PATH,
    validation,
)

registry_rows_after_by_project = {
    f"Project{number}RegistryRowsAfter":
        int(
            after_nums.eq(
                number
            ).sum()
        )
    for number in range(
        11,
        23,
    )
}

report = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5C_STATUS,
    "CompletedAtUTC": created_at,

    "SourceRootSHA256": SOURCE_ROOT_SHA,
    "RawRootSHA256": RAW_ROOT_SHA,

    **COUNTS,

    "FinalPackageRoot": str(
        FINAL_ROOT
    ),
    "FinalPackageFiles": package_files,
    "FinalPackageBytes": package_bytes,
    "FinalPackageRootSHA256": package_root_sha,
    "PackageAlreadyFrozenBeforeThisCell":
        package_already_frozen,

    "PackageMissingFiles": missing_pkg,
    "PackageUnexpectedFiles": unexpected_pkg,
    "PackageSizeMismatches": size_bad,
    "PackageSHA256Mismatches": hash_bad,

    "RegistrySHA256Before": registry_sha_before,
    "RegistrySHA256After": registry_sha_after,
    "RegistryRowsBefore": len(
        reg_before
    ),
    "RegistryRowsAfter": len(
        reg_after
    ),

    **registry_rows_after_by_project,

    "RegistryBackup": str(
        BACKUP_PATH
    ),

    "Step5ACheckpointSHA256": step5a_sha,
    "Step5BCheckpointSHA256": step5b_sha,

    "ParallelWorkerCheckpointSHA256":
        expected_worker_hashes_by_tag,

    "ValidationChecks": len(
        validation
    ),
    "FailedValidationChecks": len(
        failed
    ),

    "ConditionsRerun": False,
    "ModelsFitted": False,
    "RawResultsModified": False,

    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectConditionOutputsModified": False,
    "PriorProjectWriteAttempted": False,
}

atomic_json(
    REPORT_PATH,
    report,
)

atomic_json(
    CHECKPOINT_PATH,
    {
        **report,
        "CheckpointVersion": 1,
        "CheckpointType":
            "PROJECT_22_FINAL_PACKAGE_AND_REGISTRY",
        "FinalPackageFrozen": True,
        "CompletionRegistryUpdated": True,
        "ProjectCompleteAndFrozen": True,
        "NextRequiredStep":
            "PROJECT_23_MAY_START_IN_A_NEW_NOTEBOOK",
    },
)

step5c_sha = sha(
    CHECKPOINT_PATH
)

atomic_json(
    STATUS_PATH,
    {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "Status": STEP5C_STATUS,
        "CompletedAtUTC": created_at,

        "FinalPackageRoot": str(
            FINAL_ROOT
        ),
        "FinalPackageRootSHA256":
            package_root_sha,

        "RegistryRows": len(
            reg_after
        ),
        "RegistrySHA256":
            registry_sha_after,

        "Checkpoint": str(
            CHECKPOINT_PATH
        ),
        "CheckpointSHA256":
            step5c_sha,

        "ProjectCompleteAndFrozen": True,

        "ParallelWorkerCheckpointSHA256":
            expected_worker_hashes_by_tag,

        "PriorProjectConditionOutputsAccessed": False,
        "PriorProjectConditionOutputsModified": False,

        "NextRequiredStep":
            "PROJECT_23_MAY_START_IN_A_NEW_NOTEBOOK",
    },
)


# --------------------------------------------------------------------------------------------------
# 17. FINAL READBACK AND IMMUTABILITY PROOF
# --------------------------------------------------------------------------------------------------

checkpoint_readback = load_json(
    CHECKPOINT_PATH
)

status_readback = load_json(
    STATUS_PATH
)

if (
    checkpoint_readback.get(
        "Status"
    )
    != STEP5C_STATUS
    or status_readback.get(
        "Status"
    )
    != STEP5C_STATUS
):
    raise RuntimeError(
        "Project 22 Step 5C checkpoint/status readback failed."
    )

if not bool(
    checkpoint_readback.get(
        "ProjectCompleteAndFrozen",
        False,
    )
):
    raise RuntimeError(
        "Project 22 checkpoint is not marked complete and frozen."
    )

if not bool(
    checkpoint_readback.get(
        "CompletionRegistryUpdated",
        False,
    )
):
    raise RuntimeError(
        "Project 22 checkpoint is not marked registry-updated."
    )

if not bool(
    checkpoint_readback.get(
        "FinalPackageFrozen",
        False,
    )
):
    raise RuntimeError(
        "Project 22 checkpoint is not marked final-package frozen."
    )

if sha(
    REGISTRY
) != registry_sha_after:
    raise RuntimeError(
        "Completion registry changed after Project 22 finalisation."
    )

if root_hash(
    pd.read_csv(
        MANIFEST_PATH,
        low_memory=False,
    )
) != package_root_sha:
    raise RuntimeError(
        "Final package changed after Project 22 finalisation."
    )

if sha(
    STEP5A_CP
) != STEP5A_SHA_EXPECTED:
    raise RuntimeError(
        "Step 5A checkpoint changed during Project 22 Step 5C."
    )

if sha(
    STEP5B_CP
) != STEP5B_SHA_EXPECTED:
    raise RuntimeError(
        "Step 5B checkpoint changed during Project 22 Step 5C."
    )

for (
    worker_path,
    expected_worker_sha,
) in WORKER_CHECKPOINTS.items():
    if sha(
        worker_path
    ) != expected_worker_sha:
        raise RuntimeError(
            "A frozen Project 22 worker checkpoint changed "
            "during Step 5C."
        )


# --------------------------------------------------------------------------------------------------
# 18. RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 136
)

print(
    "=== PROJECT 22 CELL 11 / STEP 5C RESULT ==="
)

print(
    "=" * 136
)

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print(
    "\nRaw result freeze:"
)

print(
    "Conditions:",
    COUNTS[
        "Conditions"
    ],
)

print(
    "ML fits:",
    COUNTS[
        "MLFits"
    ],
)

print(
    "Raw files:",
    COUNTS[
        "RawFiles"
    ],
)

print(
    "Raw bytes:",
    COUNTS[
        "RawBytes"
    ],
)

print(
    "Raw root SHA-256:",
    RAW_ROOT_SHA,
)

print(
    "\nParallel execution freeze:"
)

print(
    "Worker checkpoints:",
    len(
        expected_worker_hashes_by_tag
    ),
)

print(
    "Worker checkpoint SHA mismatches:",
    worker_checkpoint_sha_mismatches,
)

print(
    "\nFinal package freeze:"
)

print(
    "Package root:",
    FINAL_ROOT,
)

print(
    "Package files:",
    package_files,
)

print(
    "Package bytes:",
    package_bytes,
)

print(
    "Missing package files:",
    missing_pkg,
)

print(
    "Unexpected package files:",
    unexpected_pkg,
)

print(
    "Package size mismatches:",
    size_bad,
)

print(
    "Package SHA-256 mismatches:",
    hash_bad,
)

print(
    "Final package root SHA-256:",
    package_root_sha,
)

print(
    "\nCompletion registry:"
)

print(
    "Registry rows:",
    len(
        reg_after
    ),
)

print(
    "COMPLETE_AND_FROZEN projects:",
    int(
        reg_after[
            status_col
        ].eq(
            COMPLETE_STATUS
        ).sum()
    ),
)

print(
    "Project 22 registry rows:",
    len(
        project22_after
    ),
)

print(
    "Registry SHA-256 before:",
    registry_sha_before,
)

print(
    "Registry SHA-256 after:",
    registry_sha_after,
)

print(
    "Package already frozen before this cell:",
    package_already_frozen,
)

print(
    "\nStep 5C freeze:"
)

print(
    "Validation checks:",
    len(
        validation
    ),
)

print(
    "Failed validation checks:",
    len(
        failed
    ),
)

print(
    "Step 5C checkpoint:",
    CHECKPOINT_PATH,
)

print(
    "Step 5C checkpoint SHA-256:",
    step5c_sha,
)

print(
    "Project complete and frozen:",
    True,
)

print(
    "Next required step:",
    "PROJECT 23 MAY START IN A NEW NOTEBOOK",
)

print(
    "STATUS:",
    STEP5C_STATUS,
)

print(
    "=" * 136
)


=== PROJECT 22 CELL 11 / STEP 5C: FINAL PACKAGE FREEZE AND REGISTRY REGISTRATION ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


RuntimeError: Step 5A parallel-worker checkpoint SHA map differs from the six frozen Project 22 workers.

In [5]:
# ==================================================================================================
# PROJECT 22 — CELL 11 / STEP 5C
# REGISTRY-SCHEMA-COMPLETE, CROSS-FILESYSTEM-SAFE FINAL PACKAGE AND REGISTRATION
#
# PROJECT:
#   apache@logging-log4j2
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_22.ipynb.
#
# REQUIRED FROZEN INPUTS:
# - Project 22 Step 5A checkpoint SHA-256:
#   600035cfbf9f8b2dfd0fd83d49050466e3c71e6187c1d631b24cfe57a6107a26
# - Project 22 Step 5B checkpoint SHA-256:
#   9259c486debcda1794423bd16d4333083e14d203216df596327ec4658bfb04fb
# - Project 22 raw-root SHA-256:
#   374e4e1eaab266451539c9c7a4751fa6fe50250dccf57b19a5bc19cf8fad3f11
# - Project 22 source-root SHA-256:
#   281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64f842964c896c6ac334
# - Registry before registration:
#   exactly Projects 1–21, all COMPLETE_AND_FROZEN
# - Registry SHA-256 before registration:
#   79cd6ecb595c5e8ae91a9494e469792716338d144308560a62caf1b9342306b2
#
# SAFETY:
# - no model fitting;
# - no condition reruns;
# - no raw-result modification or deletion;
# - no prior-project condition-output access or write;
# - registry write only after package and candidate-row validation;
# - cross-filesystem-safe Google Drive staging and readback;
# - six frozen Project 22 Step 5A worker checkpoints are preserved and revalidated.
# ==================================================================================================

from google.colab import drive
from pathlib import Path
from IPython.display import display
import hashlib
import json
import os
import re
import shutil
import tempfile
import pandas as pd

print("=" * 136)
print("=== PROJECT 22 CELL 11 / STEP 5C: FINAL PACKAGE FREEZE AND REGISTRY REGISTRATION ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN IDENTITY, HASHES, AND COUNTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 22
PROJECT_NAME = "apache@logging-log4j2"
PROJECT_SLUG = "apache__logging-log4j2"
PROJECT_SHORT = "LOG4J2"

COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

STEP5C_STATUS = "PASS_PROJECT_22_FINAL_PACKAGE_FROZEN_AND_REGISTERED"
STEP5A_STATUS = "PASS_PROJECT_22_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
STEP5B_STATUS = "PASS_PROJECT_22_RAW_RESULTS_REVALIDATED_AND_COMPACT_AGGREGATES_FROZEN"

REGISTRY_SHA_BEFORE_EXPECTED = (
    "79cd6ecb595c5e8ae91a9494e469792716338d144308560a62caf1b9342306b2"
)

STEP5A_SHA_EXPECTED = (
    "600035cfbf9f8b2dfd0fd83d49050466e3c71e6187c1d631b24cfe57a6107a26"
)

STEP5B_SHA_EXPECTED = (
    "9259c486debcda1794423bd16d4333083e14d203216df596327ec4658bfb04fb"
)

SOURCE_ROOT_SHA = (
    "281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64f842964c896c6ac334"
)

RAW_ROOT_SHA = (
    "374e4e1eaab266451539c9c7a4751fa6fe50250dccf57b19a5bc19cf8fad3f11"
)

COUNTS = {
    "RawFiles": 2160,
    "RawBytes": 646130653,
    "Conditions": 270,
    "MLFits": 1080,
    "RankingRows": 41874840,
    "BuildMetricRows": 73710,
    "ProjectRunRows": 1890,
    "ConditionAuditRows": 270,
    "TrainingMedianRows": 40770,

    "Builds": 441,
    "TrainingBuilds": 330,
    "EvaluationBuilds": 111,

    "RawRows": 240253,
    "RawTrainingRows": 172628,
    "RawEvaluationRows": 67625,
    "RawTrainingFailures": 208,
    "RawEvaluationFailures": 40,

    "ModelRows": 117968,
    "ModelTrainingRows": 95812,
    "ModelEvaluationRows": 22156,
    "ModelTrainingFailures": 207,
    "ModelEvaluationFailures": 40,
    "ModelFailingEvaluationBuilds": 39,

    "Predictors": 151,
    "RECFeatures": 19,
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

drive.mount(
    "/content/drive",
    force_remount=False,
)

ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES = (
    ROOT
    / "Notes"
)

RESULTS = (
    ROOT
    / "Results"
)

REGISTRY = (
    NOTES
    / "completed_project_registry.csv"
)

PROJECT_ROOT = (
    RESULTS
    / "Aggregated"
    / PROJECT_SLUG
)

RAW_ROOT = (
    RESULTS
    / "Raw"
    / PROJECT_SLUG
)

FINAL_ROOT = (
    RESULTS
    / "Final"
    / PROJECT_SLUG
)

MANIFEST_PATH = (
    FINAL_ROOT
    / "final_package_manifest.csv"
)

SUMMARY_PATH = (
    FINAL_ROOT
    / "final_package_summary.json"
)

README_PATH = (
    FINAL_ROOT
    / "README.txt"
)

STEP5C_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5c"
)

VALIDATION_PATH = (
    STEP5C_ROOT
    / f"{PROJECT_SHORT}_step5c_validation.csv"
)

REPORT_PATH = (
    STEP5C_ROOT
    / f"{PROJECT_SHORT}_step5c_report.json"
)

STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5c_status.json"
)

CHECKPOINT_PATH = (
    NOTES
    / "project_22_step5c_checkpoint.json"
)

BACKUP_PATH = (
    NOTES
    / "completed_project_registry_before_project_22.csv"
)

STEP5B_RAW_MANIFEST_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5b"
    / f"{PROJECT_SHORT}_independent_raw_manifest.csv"
)

STEP5A_CP = (
    NOTES
    / "project_22_step5a_checkpoint.json"
)

STEP5B_CP = (
    NOTES
    / "project_22_step5b_checkpoint.json"
)

STEP5A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5a_status.json"
)

STEP5B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5b_status.json"
)

UPSTREAM_CPS = [
    NOTES / "project_22_selection_checkpoint.json",
    NOTES / "project_22_rec_reconstruction_checkpoint.json",
    NOTES / "project_22_noise_plan_checkpoint.json",
    NOTES / "project_22_runtime_contract_checkpoint.json",
    NOTES / "project_22_smoke_test_checkpoint.json",
    STEP5A_CP,
    STEP5B_CP,
]

WORKER_CHECKPOINTS = {
    NOTES / "project_22_step5a_worker_seed_01_05_checkpoint.json":
        "6edc0ecf1d33c3fd2263f968f7d953726e3b0ddf3a0a2b40b2cc34ecbb15bb44",

    NOTES / "project_22_step5a_worker_seed_06_10_checkpoint.json":
        "1a8ebadb53e01acd4408ac58c8235b4c6504bee2a6bb0213966563a4243aa0c8",

    NOTES / "project_22_step5a_worker_seed_11_15_checkpoint.json":
        "f02df169293ce4d1ae143552547a89654ce21e1084b641b4b3ce48ce9afbbc05",

    NOTES / "project_22_step5a_worker_seed_16_20_checkpoint.json":
        "0643261cc0bd54cadba8354fa88e726ab2e199b940aa2b3f609a581121663db4",

    NOTES / "project_22_step5a_worker_seed_21_25_checkpoint.json":
        "23806c7c5252cb4f4b5f6f5faee559678b8c6d02a023437405f7551eaa6c3718",

    NOTES / "project_22_step5a_worker_seed_26_30_checkpoint.json":
        "f7e03bb7c57f29782b057554bbe8116a25e14cce7336361154e7c60e3a0af173",
}


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha(
    path,
    chunk=8 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(
        path
    ).open(
        "rb"
    ) as f:
        for block in iter(
            lambda: f.read(
                chunk
            ),
            b"",
        ):
            h.update(
                block
            )

    return h.hexdigest()


def load_json(
    path,
):
    with Path(
        path
    ).open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(
            f
        )


def atomic_json(
    path,
    obj,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        f.write(
            "\n"
        )

    os.replace(
        tmp,
        path,
    )


def atomic_csv(
    path,
    df,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    df.to_csv(
        tmp,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        tmp,
        path,
    )


def atomic_text(
    path,
    text,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    tmp.write_text(
        text,
        encoding="utf-8",
    )

    os.replace(
        tmp,
        path,
    )


def resolve(
    cols,
    expected,
):
    matches = [
        c
        for c in cols
        if str(
            c
        ).strip().lower()
        == expected.lower()
    ]

    if len(
        matches
    ) != 1:
        raise RuntimeError(
            f"Could not resolve registry column {expected!r}; "
            f"matches={matches}; columns={list(cols)}"
        )

    return matches[
        0
    ]


def norm(
    value,
):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(
            value
        ).lower(),
    )


def manifest(
    root,
    exclude=(),
):
    root = Path(
        root
    )

    exclude = set(
        exclude
    )

    rows = []

    files = sorted(
        (
            path
            for path in root.rglob(
                "*"
            )
            if path.is_file()
        ),
        key=lambda path:
            path.relative_to(
                root
            ).as_posix(),
    )

    for path in files:
        rel = path.relative_to(
            root
        ).as_posix()

        if rel in exclude:
            continue

        rows.append(
            {
                "RelativePath": rel,
                "Bytes": int(
                    path.stat().st_size
                ),
                "SHA256": sha(
                    path
                ),
            }
        )

    return pd.DataFrame(
        rows,
        columns=[
            "RelativePath",
            "Bytes",
            "SHA256",
        ],
    )


def root_hash(
    df,
):
    h = hashlib.sha256()

    ordered = df.sort_values(
        "RelativePath",
        kind="mergesort",
    )

    for row in ordered.itertuples(
        index=False
    ):
        h.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode()
        )

    return h.hexdigest()


def verify_manifest(
    items,
    label,
):
    if (
        not isinstance(
            items,
            list,
        )
        or not items
    ):
        raise RuntimeError(
            f"{label} has no output manifest."
        )

    rows = []

    for item in items:
        path = Path(
            item[
                "Path"
            ]
        )

        exists = path.is_file()

        expected_bytes = int(
            item[
                "Bytes"
            ]
        )

        expected_sha = str(
            item[
                "SHA256"
            ]
        ).lower()

        if exists:
            actual_bytes = int(
                path.stat().st_size
            )

            actual_sha = sha(
                path
            )
        else:
            actual_bytes = -1
            actual_sha = "MISSING"

        rows.append(
            {
                "Path": str(
                    path
                ),
                "ExpectedBytes": expected_bytes,
                "ActualBytes": actual_bytes,
                "ExpectedSHA256": expected_sha,
                "ActualSHA256": actual_sha,
                "Pass": bool(
                    exists
                    and expected_bytes
                    == actual_bytes
                    and expected_sha
                    == actual_sha
                ),
            }
        )

    out = pd.DataFrame(
        rows
    )

    if not out[
        "Pass"
    ].all():
        display(
            out.loc[
                ~out[
                    "Pass"
                ]
            ]
        )

        raise RuntimeError(
            f"{label} manifest verification failed."
        )

    return out


def check(
    rows,
    name,
    expected,
    actual,
    passed,
):
    rows.append(
        {
            "Check": name,
            "Expected": expected,
            "Actual": actual,
            "Pass": bool(
                passed
            ),
        }
    )


# --------------------------------------------------------------------------------------------------
# 4. VERIFY ALL REQUIRED INPUTS BEFORE ANY PACKAGE OR REGISTRY WRITE
# --------------------------------------------------------------------------------------------------

required = [
    REGISTRY,
    RAW_ROOT,
    STEP5A_CP,
    STEP5B_CP,
    STEP5A_STATUS_PATH,
    STEP5B_STATUS_PATH,
    *UPSTREAM_CPS,
    *WORKER_CHECKPOINTS,
]

missing = [
    str(
        path
    )
    for path in required
    if not Path(
        path
    ).exists()
]

if missing:
    raise FileNotFoundError(
        "Missing Project 22 Step 5C inputs:\n"
        + "\n".join(
            missing
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY STEP 5A / STEP 5B FREEZES
# --------------------------------------------------------------------------------------------------

step5a_sha = sha(
    STEP5A_CP
)

step5b_sha = sha(
    STEP5B_CP
)

if step5a_sha != STEP5A_SHA_EXPECTED:
    raise RuntimeError(
        "Step 5A checkpoint SHA differs.\n"
        f"Expected: {STEP5A_SHA_EXPECTED}\n"
        f"Actual:   {step5a_sha}"
    )

if step5b_sha != STEP5B_SHA_EXPECTED:
    raise RuntimeError(
        "Step 5B checkpoint SHA differs.\n"
        f"Expected: {STEP5B_SHA_EXPECTED}\n"
        f"Actual:   {step5b_sha}"
    )

step5a = load_json(
    STEP5A_CP
)

step5b = load_json(
    STEP5B_CP
)

step5a_status_payload = load_json(
    STEP5A_STATUS_PATH
)

step5b_status_payload = load_json(
    STEP5B_STATUS_PATH
)

for label, payload, expected in [
    (
        "Step 5A checkpoint",
        step5a,
        STEP5A_STATUS,
    ),
    (
        "Step 5A status",
        step5a_status_payload,
        STEP5A_STATUS,
    ),
    (
        "Step 5B checkpoint",
        step5b,
        STEP5B_STATUS,
    ),
    (
        "Step 5B status",
        step5b_status_payload,
        STEP5B_STATUS,
    ),
]:
    if payload.get(
        "Status"
    ) != expected:
        raise RuntimeError(
            f"{label} is not in expected PASS state."
        )

if (
    step5a.get(
        "SourceRootSHA256"
    )
    != SOURCE_ROOT_SHA
):
    raise RuntimeError(
        "Step 5A source-root SHA differs."
    )

if (
    step5a.get(
        "RawRootSHA256"
    )
    != RAW_ROOT_SHA
):
    raise RuntimeError(
        "Step 5A raw-root SHA differs."
    )

if (
    step5b.get(
        "SourceRootSHA256"
    )
    != SOURCE_ROOT_SHA
):
    raise RuntimeError(
        "Step 5B source-root SHA differs."
    )

if (
    step5b.get(
        "FrozenRawRootSHA256"
    )
    != RAW_ROOT_SHA
):
    raise RuntimeError(
        "Step 5B frozen raw-root SHA differs."
    )

if (
    step5b.get(
        "IndependentRawRootSHA256"
    )
    != RAW_ROOT_SHA
):
    raise RuntimeError(
        "Step 5B independent raw-root SHA differs."
    )

if (
    step5b.get(
        "Step5ACheckpointSHA256"
    )
    != STEP5A_SHA_EXPECTED
):
    raise RuntimeError(
        "Step 5B does not link to the frozen Step 5A checkpoint."
    )

if not bool(
    step5b.get(
        "ReadyForFinalPackageAndRegistration",
        False,
    )
):
    raise RuntimeError(
        "Step 5B is not marked ready for final package and registration."
    )

if bool(
    step5b.get(
        "RegistryModified",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports that the completion registry was modified."
    )

if bool(
    step5b.get(
        "PriorProjectConditionOutputsAccessed",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports access to prior-project condition outputs."
    )

if bool(
    step5b.get(
        "PriorProjectConditionOutputsModified",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports modification of prior-project condition outputs."
    )

if bool(
    step5b.get(
        "PriorProjectWriteAttempted",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports a prior-project write attempt."
    )

if bool(
    step5b.get(
        "ModelsFitted",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports model fitting."
    )

if bool(
    step5b.get(
        "ConditionsRerun",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports condition reruns."
    )


# --------------------------------------------------------------------------------------------------
# 6. VERIFY SIX FROZEN PARALLEL WORKER CHECKPOINTS
# --------------------------------------------------------------------------------------------------

worker_checkpoint_sha_mismatches = 0

for worker_path, expected_worker_sha in WORKER_CHECKPOINTS.items():
    actual_worker_sha = sha(
        worker_path
    )

    worker_checkpoint_sha_mismatches += int(
        actual_worker_sha
        != expected_worker_sha
    )

expected_worker_hashes_by_tag = {
    "seed_01_05":
        WORKER_CHECKPOINTS[
            NOTES
            / "project_22_step5a_worker_seed_01_05_checkpoint.json"
        ],

    "seed_06_10":
        WORKER_CHECKPOINTS[
            NOTES
            / "project_22_step5a_worker_seed_06_10_checkpoint.json"
        ],

    "seed_11_15":
        WORKER_CHECKPOINTS[
            NOTES
            / "project_22_step5a_worker_seed_11_15_checkpoint.json"
        ],

    "seed_16_20":
        WORKER_CHECKPOINTS[
            NOTES
            / "project_22_step5a_worker_seed_16_20_checkpoint.json"
        ],

    "seed_21_25":
        WORKER_CHECKPOINTS[
            NOTES
            / "project_22_step5a_worker_seed_21_25_checkpoint.json"
        ],

    "seed_26_30":
        WORKER_CHECKPOINTS[
            NOTES
            / "project_22_step5a_worker_seed_26_30_checkpoint.json"
        ],
}

step5a_worker_hashes = step5a.get(
    "ExpectedWorkerCheckpointSHA256",
    {},
)

if (
    step5a_worker_hashes
    != expected_worker_hashes_by_tag
):
    raise RuntimeError(
        "Step 5A ExpectedWorkerCheckpointSHA256 map differs "
        "from the six frozen Project 22 workers."
    )

if worker_checkpoint_sha_mismatches:
    raise RuntimeError(
        f"{worker_checkpoint_sha_mismatches} frozen Project 22 "
        "parallel-worker checkpoints changed."
    )


# --------------------------------------------------------------------------------------------------
# 7. VERIFY STEP 5A / STEP 5B OUTPUT MANIFESTS
# --------------------------------------------------------------------------------------------------

step5a_audit = verify_manifest(
    step5a.get(
        "AggregateOutputManifest",
        [],
    ),
    "Step 5A aggregate",
)

step5b_audit = verify_manifest(
    step5b.get(
        "OutputManifest",
        [],
    ),
    "Step 5B",
)


# --------------------------------------------------------------------------------------------------
# 8. VERIFY COMPLETION REGISTRY PRE-STATE
# --------------------------------------------------------------------------------------------------

registry_sha_before = sha(
    REGISTRY
)

if (
    registry_sha_before
    != REGISTRY_SHA_BEFORE_EXPECTED
):
    raise RuntimeError(
        "Registry SHA differs before Project 22 registration:\n"
        f"Expected: {REGISTRY_SHA_BEFORE_EXPECTED}\n"
        f"Actual:   {registry_sha_before}"
    )

if (
    step5b.get(
        "RegistrySHA256"
    )
    != REGISTRY_SHA_BEFORE_EXPECTED
):
    raise RuntimeError(
        "The frozen Step 5B checkpoint does not reference "
        "the expected pre-Project-22 registry SHA."
    )

reg_before = pd.read_csv(
    REGISTRY,
    dtype=str,
).fillna(
    ""
)

pn_col = resolve(
    reg_before.columns,
    "ProjectNumber",
)

project_col = resolve(
    reg_before.columns,
    "Project",
)

status_col = resolve(
    reg_before.columns,
    "Status",
)

pnums = pd.to_numeric(
    reg_before[
        pn_col
    ],
    errors="raise",
).astype(
    int
)

if (
    len(
        reg_before
    )
    != 21
    or sorted(
        pnums.tolist()
    )
    != list(
        range(
            1,
            22,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–21."
    )

if not reg_before[
    status_col
].eq(
    COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–21 are not all COMPLETE_AND_FROZEN."
    )

required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
}

for (
    required_number,
    required_project,
) in required_registered_identities.items():
    matching_rows = reg_before.loc[
        pnums.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_col
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )

if (
    pnums.eq(
        PROJECT_NUMBER
    ).any()
    or reg_before[
        project_col
    ].eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 22 is already present in the completion registry."
    )


# --------------------------------------------------------------------------------------------------
# 9. FREEZE A PRE-PROJECT-22 REGISTRY BACKUP
# --------------------------------------------------------------------------------------------------

if not BACKUP_PATH.exists():
    shutil.copy2(
        REGISTRY,
        BACKUP_PATH,
    )

if sha(
    BACKUP_PATH
) != registry_sha_before:
    raise RuntimeError(
        "Pre-Project-22 registry backup does not match the live registry."
    )


# --------------------------------------------------------------------------------------------------
# 10. ASSEMBLE FINAL COMPACT PACKAGE
# --------------------------------------------------------------------------------------------------

sources = set(
    Path(
        path
    )
    for path in UPSTREAM_CPS
)

sources.update(
    WORKER_CHECKPOINTS.keys()
)

sources.update(
    [
        STEP5A_STATUS_PATH,
        STEP5B_STATUS_PATH,
    ]
)

sources.update(
    Path(
        item[
            "Path"
        ]
    )
    for item in step5a.get(
        "AggregateOutputManifest",
        [],
    )
)

sources.update(
    Path(
        item[
            "Path"
        ]
    )
    for item in step5b.get(
        "OutputManifest",
        [],
    )
)

sources = sorted(
    sources,
    key=str,
)

missing_sources = [
    str(
        path
    )
    for path in sources
    if not path.is_file()
]

if missing_sources:
    raise FileNotFoundError(
        "Missing compact-package sources:\n"
        + "\n".join(
            missing_sources
        )
    )

created_at = str(
    step5b.get(
        "CompletedAtUTC",
        "",
    )
).strip()

if not created_at:
    raise RuntimeError(
        "The frozen Step 5B checkpoint contains no CompletedAtUTC timestamp."
    )

tmp_root = Path(
    tempfile.mkdtemp(
        prefix="project22_package_",
        dir="/content",
    )
)

drive_staging_root = None

try:
    for src in sources:
        try:
            rel = src.relative_to(
                ROOT
            )
        except ValueError as exc:
            raise RuntimeError(
                f"Package source is outside thesis root: {src}"
            ) from exc

        dst = (
            tmp_root
            / rel
        )

        dst.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        shutil.copy2(
            src,
            dst,
        )

    atomic_text(
        tmp_root
        / "README.txt",
        f"""PROJECT 22 FINAL COMPACT PACKAGE

Project number: {PROJECT_NUMBER}
Project: {PROJECT_NAME}
Project slug: {PROJECT_SLUG}
Status: {COMPLETE_STATUS}
Created at UTC: {created_at}

The 2,160 raw condition-output files are not duplicated here.
Raw results: {RAW_ROOT}
Raw-root SHA-256: {RAW_ROOT_SHA}
Source-root SHA-256: {SOURCE_ROOT_SHA}
Primary metric: APFDc
Secondary metric: APFD
Parallel Step 5A worker checkpoints preserved in this package: 6
""",
    )

    before_summary = manifest(
        tmp_root
    )

    atomic_json(
        tmp_root
        / "final_package_summary.json",
        {
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "Status": COMPLETE_STATUS,
            "CreatedAtUTC": created_at,
            "SourceRootSHA256": SOURCE_ROOT_SHA,
            "RawRootSHA256": RAW_ROOT_SHA,
            **COUNTS,
            "Step5ACheckpointSHA256": step5a_sha,
            "Step5BCheckpointSHA256": step5b_sha,
            "PayloadRootSHA256BeforeSummary": root_hash(
                before_summary
            ),
            "RawResultsDuplicatedIntoPackage": False,
            "ParallelWorkerCheckpointSHA256":
                expected_worker_hashes_by_tag,
            "PriorProjectConditionOutputsAccessed": False,
            "PriorProjectConditionOutputsModified": False,
            "PriorProjectWriteAttempted": False,
        },
    )

    candidate_manifest = manifest(
        tmp_root,
        {
            "final_package_manifest.csv",
        },
    )

    package_root_sha = root_hash(
        candidate_manifest
    )

    atomic_csv(
        tmp_root
        / "final_package_manifest.csv",
        candidate_manifest,
    )

    package_files = (
        len(
            candidate_manifest
        )
        + 1
    )

    package_bytes = int(
        candidate_manifest[
            "Bytes"
        ].sum()
        + (
            tmp_root
            / "final_package_manifest.csv"
        ).stat().st_size
    )

    if FINAL_ROOT.exists():
        if not MANIFEST_PATH.is_file():
            raise RuntimeError(
                "An existing Project 22 final-package directory has no manifest "
                "and was not modified."
            )

        existing = pd.read_csv(
            MANIFEST_PATH,
            low_memory=False,
        )

        if root_hash(
            existing
        ) != package_root_sha:
            raise RuntimeError(
                "A different Project 22 final package already exists "
                "and was not modified."
            )

        shutil.rmtree(
            tmp_root
        )

        package_already_frozen = True

    else:
        # /content and Google Drive are different filesystems.
        # Publish in two stages:
        #   1. copy completed local package to a sibling Drive staging directory;
        #   2. verify the staged package exactly;
        #   3. rename staging -> FINAL_ROOT within Google Drive.
        FINAL_ROOT.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        drive_staging_root = FINAL_ROOT.with_name(
            f"{FINAL_ROOT.name}__staging_{os.getpid()}"
        )

        if drive_staging_root.exists():
            shutil.rmtree(
                drive_staging_root
            )

        shutil.copytree(
            tmp_root,
            drive_staging_root,
            copy_function=shutil.copy2,
        )

        staged_manifest_path = (
            drive_staging_root
            / "final_package_manifest.csv"
        )

        if not staged_manifest_path.is_file():
            raise RuntimeError(
                "The Google Drive staging package has no manifest."
            )

        staged_manifest = pd.read_csv(
            staged_manifest_path,
            low_memory=False,
        )

        staged_root_sha = root_hash(
            staged_manifest
        )

        if staged_root_sha != package_root_sha:
            raise RuntimeError(
                "The Google Drive staging package root SHA-256 differs.\n"
                f"Expected: {package_root_sha}\n"
                f"Actual:   {staged_root_sha}"
            )

        staged_payload_manifest = manifest(
            drive_staging_root,
            {
                "final_package_manifest.csv",
            },
        )

        candidate_payload_manifest = (
            candidate_manifest.sort_values(
                "RelativePath",
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        staged_payload_manifest = (
            staged_payload_manifest.sort_values(
                "RelativePath",
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        if not staged_payload_manifest.equals(
            candidate_payload_manifest
        ):
            comparison = candidate_payload_manifest.merge(
                staged_payload_manifest,
                on="RelativePath",
                how="outer",
                suffixes=(
                    "_candidate",
                    "_staged",
                ),
                indicator=True,
            )

            failed_comparison = comparison.loc[
                (
                    comparison[
                        "_merge"
                    ].ne(
                        "both"
                    )
                    | comparison[
                        "Bytes_candidate"
                    ].ne(
                        comparison[
                            "Bytes_staged"
                        ]
                    )
                    | comparison[
                        "SHA256_candidate"
                    ].ne(
                        comparison[
                            "SHA256_staged"
                        ]
                    )
                )
            ]

            raise RuntimeError(
                "The Google Drive staging package failed exact file-level "
                "verification.\n"
                + failed_comparison.head(
                    20
                ).to_string(
                    index=False
                )
            )

        os.replace(
            drive_staging_root,
            FINAL_ROOT,
        )

        drive_staging_root = None

        shutil.rmtree(
            tmp_root
        )

        package_already_frozen = False

except Exception:
    if tmp_root.exists():
        shutil.rmtree(
            tmp_root,
            ignore_errors=True,
        )

    if (
        drive_staging_root is not None
        and drive_staging_root.exists()
    ):
        shutil.rmtree(
            drive_staging_root,
            ignore_errors=True,
        )

    raise


# --------------------------------------------------------------------------------------------------
# 11. READ BACK EVERY FINAL-PACKAGE FILE
# --------------------------------------------------------------------------------------------------

pkg_manifest = pd.read_csv(
    MANIFEST_PATH,
    low_memory=False,
)

manifest_paths = set(
    pkg_manifest[
        "RelativePath"
    ].astype(
        str
    )
)

actual_paths = {
    path.relative_to(
        FINAL_ROOT
    ).as_posix()
    for path in FINAL_ROOT.rglob(
        "*"
    )
    if path.is_file()
}

expected_paths = (
    manifest_paths
    | {
        "final_package_manifest.csv",
    }
)

missing_pkg = len(
    expected_paths
    - actual_paths
)

unexpected_pkg = len(
    actual_paths
    - expected_paths
)

size_bad = 0
hash_bad = 0

for row in pkg_manifest.itertuples(
    index=False
):
    path = (
        FINAL_ROOT
        / str(
            row.RelativePath
        )
    )

    if path.is_file():
        size_bad += int(
            path.stat().st_size
            != int(
                row.Bytes
            )
        )

        hash_bad += int(
            sha(
                path
            )
            != str(
                row.SHA256
            )
        )

package_root_readback = root_hash(
    pkg_manifest
)

if (
    package_root_readback
    != package_root_sha
    or any(
        [
            missing_pkg,
            unexpected_pkg,
            size_bad,
            hash_bad,
        ]
    )
):
    raise RuntimeError(
        "Final Project 22 package failed readback validation."
    )


# --------------------------------------------------------------------------------------------------
# 12. BUILD A COMPLETE PROJECT 22 REGISTRY ROW
# --------------------------------------------------------------------------------------------------
#
# The registry schema has evolved across projects.
# Use Project 21's exact formatting for protocol fields whose textual
# representation may have varied historically, while filling Project 22's
# project-specific values explicitly.
# --------------------------------------------------------------------------------------------------

project_21_template_rows = reg_before.loc[
    pd.to_numeric(
        reg_before[
            pn_col
        ],
        errors="raise",
    ).astype(
        int
    ).eq(
        21
    )
]

if len(
    project_21_template_rows
) != 1:
    raise RuntimeError(
        "Could not resolve exactly one Project 21 registry template row."
    )

project_21_template = project_21_template_rows.iloc[
    0
]

protocol_template_values = {}

for registry_column in reg_before.columns:
    normalised_column = norm(
        registry_column
    )

    if normalised_column in {
        "seeds",
        "noiselevels",
        "techniques",
        "donotrerun",
    }:
        protocol_template_values[
            normalised_column
        ] = str(
            project_21_template[
                registry_column
            ]
        ).strip()

protocol_fallback_values = {
    "seeds":
        json.dumps(
            list(
                range(
                    1,
                    31,
                )
            ),
            separators=(
                ",",
                ":",
            ),
        ),

    "noiselevels":
        json.dumps(
            [
                0,
                5,
                10,
                15,
                20,
                25,
                30,
                40,
                50,
            ],
            separators=(
                ",",
                ":",
            ),
        ),

    "techniques":
        json.dumps(
            [
                "RandomForest",
                "XGBoost",
                "LightGBM",
                "NaiveBayes",
                "Random",
                "LatestFail",
                "QTF-Avg",
            ],
            separators=(
                ",",
                ":",
            ),
        ),

    "donotrerun":
        "True",
}

for (
    protocol_key,
    fallback_value,
) in protocol_fallback_values.items():
    if not protocol_template_values.get(
        protocol_key,
        "",
    ):
        protocol_template_values[
            protocol_key
        ] = fallback_value


values = {
    "projectnumber": PROJECT_NUMBER,
    "projectno": PROJECT_NUMBER,
    "project": PROJECT_NAME,
    "projectname": PROJECT_NAME,
    "projectslug": PROJECT_SLUG,
    "slug": PROJECT_SLUG,

    "status": COMPLETE_STATUS,
    "completionstatus": COMPLETE_STATUS,

    "completedatutc": created_at,
    "completedat": created_at,
    "frozenatutc": created_at,
    "frozenat": created_at,
    "registeredatutc": created_at,
    "registeredat": created_at,

    "sourcerootsha256": SOURCE_ROOT_SHA,

    "rawrootsha256": RAW_ROOT_SHA,
    "rawresultrootsha256": RAW_ROOT_SHA,
    "rawresultsrootsha256": RAW_ROOT_SHA,

    "rawroot": str(
        RAW_ROOT
    ),
    "rawresultroot": str(
        RAW_ROOT
    ),

    "finalpackagepath": str(
        FINAL_ROOT
    ),
    "packagepath": str(
        FINAL_ROOT
    ),
    "finalpackageroot": str(
        FINAL_ROOT
    ),

    "finalpackagerootsha256": package_root_sha,
    "packagerootsha256": package_root_sha,
    "packagesha256": package_root_sha,

    "packagefiles": package_files,
    "packagefilecount": package_files,
    "packagebytes": package_bytes,

    "step5acheckpointsha256": step5a_sha,
    "step5bcheckpointsha256": step5b_sha,

    # Complete observed registry schema.
    "seeds":
        protocol_template_values[
            "seeds"
        ],

    "noiselevels":
        protocol_template_values[
            "noiselevels"
        ],

    "techniques":
        protocol_template_values[
            "techniques"
        ],

    "evaluationrows":
        COUNTS[
            "ModelEvaluationRows"
        ],

    "evaluationfailures":
        COUNTS[
            "ModelEvaluationFailures"
        ],

    "finaldirectory":
        str(
            FINAL_ROOT
        ),

    "finalauditreport":
        str(
            REPORT_PATH
        ),

    "donotrerun":
        protocol_template_values[
            "donotrerun"
        ],

    "freezerecord":
        str(
            CHECKPOINT_PATH
        ),

    "rawresultsmanifest":
        str(
            STEP5B_RAW_MANIFEST_PATH
        ),

    "finalpackagemanifest":
        str(
            MANIFEST_PATH
        ),

    "finalauditstatus":
        STEP5C_STATUS,
}

for (
    key,
    value,
) in COUNTS.items():
    values[
        norm(
            key
        )
    ] = value

values.update(
    {
        "rawfilecount":
            COUNTS[
                "RawFiles"
            ],

        "conditioncount":
            COUNTS[
                "Conditions"
            ],

        "modelfits":
            COUNTS[
                "MLFits"
            ],

        "modelreadyrows":
            COUNTS[
                "ModelRows"
            ],

        "predictorcount":
            COUNTS[
                "Predictors"
            ],

        "recfeaturecount":
            COUNTS[
                "RECFeatures"
            ],

        "rawexecutionrows":
            COUNTS[
                "RawRows"
            ],
    }
)

if not STEP5B_RAW_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "The independently frozen Step 5B raw-results manifest is missing:\n"
        f"{STEP5B_RAW_MANIFEST_PATH}"
    )

new_row = {}
unresolved = []

for col in reg_before.columns:
    normalised = norm(
        col
    )

    if normalised in values:
        new_row[
            col
        ] = str(
            values[
                normalised
            ]
        )

    else:
        unique_nonempty = sorted(
            set(
                value
                for value in reg_before[
                    col
                ].astype(
                    str
                ).str.strip()
                if value
            )
        )

        if len(
            unique_nonempty
        ) == 1:
            # Preserve a global protocol constant.
            new_row[
                col
            ] = unique_nonempty[
                0
            ]

        elif reg_before[
            col
        ].astype(
            str
        ).str.strip().eq(
            ""
        ).all():
            new_row[
                col
            ] = ""

        else:
            new_row[
                col
            ] = ""

            unresolved.append(
                col
            )

new_row[
    pn_col
] = str(
    PROJECT_NUMBER
)

new_row[
    project_col
] = PROJECT_NAME

new_row[
    status_col
] = COMPLETE_STATUS

if unresolved:
    raise RuntimeError(
        "Unexpected unmapped registry columns remain; "
        "no registry write was attempted:\n"
        + "\n".join(
            unresolved
        )
    )

reg_candidate = pd.concat(
    [
        reg_before,
        pd.DataFrame(
            [
                new_row
            ]
        ),
    ],
    ignore_index=True,
)

reg_candidate[
    pn_col
] = pd.to_numeric(
    reg_candidate[
        pn_col
    ],
    errors="raise",
).astype(
    int
).astype(
    str
)

candidate_nums = pd.to_numeric(
    reg_candidate[
        pn_col
    ],
    errors="raise",
).astype(
    int
)

project22_candidate = reg_candidate.loc[
    candidate_nums.eq(
        22
    )
]

if (
    len(
        reg_candidate
    )
    != 22
    or sorted(
        candidate_nums.tolist()
    )
    != list(
        range(
            1,
            23,
        )
    )
):
    raise RuntimeError(
        "Candidate registry does not contain exactly Projects 1–22."
    )

if (
    not reg_candidate[
        status_col
    ].eq(
        COMPLETE_STATUS
    ).all()
    or len(
        project22_candidate
    )
    != 1
    or project22_candidate.iloc[
        0
    ][
        project_col
    ]
    != PROJECT_NAME
):
    raise RuntimeError(
        "Candidate Project 22 registry row failed status/identity validation."
    )


# --------------------------------------------------------------------------------------------------
# 13. PRE-WRITE VALIDATION
# --------------------------------------------------------------------------------------------------

rows = []

check(
    rows,
    "Step 5A checkpoint SHA-256",
    STEP5A_SHA_EXPECTED,
    step5a_sha,
    step5a_sha
    == STEP5A_SHA_EXPECTED,
)

check(
    rows,
    "Step 5B checkpoint SHA-256",
    STEP5B_SHA_EXPECTED,
    step5b_sha,
    step5b_sha
    == STEP5B_SHA_EXPECTED,
)

check(
    rows,
    "Source root SHA-256",
    SOURCE_ROOT_SHA,
    step5a.get(
        "SourceRootSHA256"
    ),
    step5a.get(
        "SourceRootSHA256"
    )
    == SOURCE_ROOT_SHA,
)

check(
    rows,
    "Raw root SHA-256",
    RAW_ROOT_SHA,
    step5b.get(
        "IndependentRawRootSHA256"
    ),
    step5b.get(
        "IndependentRawRootSHA256"
    )
    == RAW_ROOT_SHA,
)

check(
    rows,
    "Step 5A manifest failures",
    0,
    int(
        (
            ~step5a_audit[
                "Pass"
            ]
        ).sum()
    ),
    step5a_audit[
        "Pass"
    ].all(),
)

check(
    rows,
    "Step 5B manifest failures",
    0,
    int(
        (
            ~step5b_audit[
                "Pass"
            ]
        ).sum()
    ),
    step5b_audit[
        "Pass"
    ].all(),
)

check(
    rows,
    "Parallel worker checkpoint SHA mismatches",
    0,
    worker_checkpoint_sha_mismatches,
    worker_checkpoint_sha_mismatches
    == 0,
)

check(
    rows,
    "Parallel worker checkpoints",
    6,
    len(
        expected_worker_hashes_by_tag
    ),
    len(
        expected_worker_hashes_by_tag
    )
    == 6,
)

check(
    rows,
    "Step 5B prior-project output access",
    False,
    bool(
        step5b.get(
            "PriorProjectConditionOutputsAccessed",
            True,
        )
    ),
    not bool(
        step5b.get(
            "PriorProjectConditionOutputsAccessed",
            True,
        )
    ),
)

check(
    rows,
    "Step 5B prior-project output modification",
    False,
    bool(
        step5b.get(
            "PriorProjectConditionOutputsModified",
            True,
        )
    ),
    not bool(
        step5b.get(
            "PriorProjectConditionOutputsModified",
            True,
        )
    ),
)

check(
    rows,
    "Package missing files",
    0,
    missing_pkg,
    missing_pkg
    == 0,
)

check(
    rows,
    "Package unexpected files",
    0,
    unexpected_pkg,
    unexpected_pkg
    == 0,
)

check(
    rows,
    "Package size mismatches",
    0,
    size_bad,
    size_bad
    == 0,
)

check(
    rows,
    "Package SHA-256 mismatches",
    0,
    hash_bad,
    hash_bad
    == 0,
)

check(
    rows,
    "Registry rows before",
    21,
    len(
        reg_before
    ),
    len(
        reg_before
    )
    == 21,
)

check(
    rows,
    "Registry rows candidate",
    22,
    len(
        reg_candidate
    ),
    len(
        reg_candidate
    )
    == 22,
)

check(
    rows,
    "Candidate Project 22 rows",
    1,
    len(
        project22_candidate
    ),
    len(
        project22_candidate
    )
    == 1,
)

check(
    rows,
    "Unresolved variable registry columns",
    0,
    len(
        unresolved
    ),
    len(
        unresolved
    )
    == 0,
)

required_registry_field_expectations = {
    "Seeds":
        protocol_template_values[
            "seeds"
        ],

    "NoiseLevels":
        protocol_template_values[
            "noiselevels"
        ],

    "Techniques":
        protocol_template_values[
            "techniques"
        ],

    "EvaluationRows":
        str(
            COUNTS[
                "ModelEvaluationRows"
            ]
        ),

    "EvaluationFailures":
        str(
            COUNTS[
                "ModelEvaluationFailures"
            ]
        ),

    "FinalDirectory":
        str(
            FINAL_ROOT
        ),

    "FinalAuditReport":
        str(
            REPORT_PATH
        ),

    "DoNotRerun":
        protocol_template_values[
            "donotrerun"
        ],

    "FreezeRecord":
        str(
            CHECKPOINT_PATH
        ),

    "RawResultsManifest":
        str(
            STEP5B_RAW_MANIFEST_PATH
        ),

    "FinalPackageManifest":
        str(
            MANIFEST_PATH
        ),

    "RawResultsRootSHA256":
        RAW_ROOT_SHA,

    "FinalAuditStatus":
        STEP5C_STATUS,
}

registry_field_validation_failures = 0

for (
    expected_column_name,
    expected_value,
) in required_registry_field_expectations.items():
    matching_columns = [
        column
        for column in reg_before.columns
        if norm(
            column
        )
        == norm(
            expected_column_name
        )
    ]

    if len(
        matching_columns
    ) != 1:
        registry_field_validation_failures += 1
        continue

    actual_value = str(
        project22_candidate.iloc[
            0
        ][
            matching_columns[
                0
            ]
        ]
    )

    registry_field_validation_failures += int(
        actual_value
        != str(
            expected_value
        )
    )

check(
    rows,
    "Explicit Project 22 registry-field failures",
    0,
    registry_field_validation_failures,
    registry_field_validation_failures
    == 0,
)

pre = pd.DataFrame(
    rows
)

print(
    "\nProject 22 Step 5C pre-write validation:"
)

display(
    pre
)

print(
    "\nProject 22 registry row candidate:"
)

display(
    project22_candidate
)

if not pre[
    "Pass"
].all():
    raise RuntimeError(
        "PROJECT 22 STEP 5C PRE-WRITE VALIDATION FAILED. "
        "Registry not modified."
    )


# --------------------------------------------------------------------------------------------------
# 14. ATOMIC COMPLETION-REGISTRY WRITE
# --------------------------------------------------------------------------------------------------

tmp_reg = REGISTRY.with_name(
    f".{REGISTRY.name}.project22_{os.getpid()}"
)

reg_candidate.to_csv(
    tmp_reg,
    index=False,
    lineterminator="\n",
)

tmp_read = pd.read_csv(
    tmp_reg,
    dtype=str,
).fillna(
    ""
)

tmp_nums = pd.to_numeric(
    tmp_read[
        pn_col
    ],
    errors="raise",
).astype(
    int
)

if (
    len(
        tmp_read
    )
    != 22
    or sorted(
        tmp_nums.tolist()
    )
    != list(
        range(
            1,
            23,
        )
    )
    or not tmp_read[
        status_col
    ].eq(
        COMPLETE_STATUS
    ).all()
    or int(
        tmp_nums.eq(
            22
        ).sum()
    )
    != 1
):
    tmp_reg.unlink(
        missing_ok=True
    )

    raise RuntimeError(
        "Temporary Project 22 registry failed readback; "
        "live registry unchanged."
    )

os.replace(
    tmp_reg,
    REGISTRY,
)


# --------------------------------------------------------------------------------------------------
# 15. POST-WRITE REGISTRY VALIDATION
# --------------------------------------------------------------------------------------------------

reg_after = pd.read_csv(
    REGISTRY,
    dtype=str,
).fillna(
    ""
)

after_nums = pd.to_numeric(
    reg_after[
        pn_col
    ],
    errors="raise",
).astype(
    int
)

project22_after = reg_after.loc[
    after_nums.eq(
        22
    )
]

registry_sha_after = sha(
    REGISTRY
)

if (
    len(
        reg_after
    )
    != 22
    or sorted(
        after_nums.tolist()
    )
    != list(
        range(
            1,
            23,
        )
    )
    or not reg_after[
        status_col
    ].eq(
        COMPLETE_STATUS
    ).all()
    or len(
        project22_after
    )
    != 1
    or project22_after.iloc[
        0
    ][
        project_col
    ]
    != PROJECT_NAME
):
    raise RuntimeError(
        "Live registry failed post-write validation.\n"
        f"Backup: {BACKUP_PATH}"
    )

for (
    required_number,
    required_project,
) in required_registered_identities.items():
    matching_rows = reg_after.loc[
        after_nums.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_col
        ]
        != required_project
    ):
        raise RuntimeError(
            "A predecessor registry identity changed after "
            "Project 22 registration."
        )

check(
    rows,
    "Registry rows after",
    22,
    len(
        reg_after
    ),
    len(
        reg_after
    )
    == 22,
)

check(
    rows,
    "COMPLETE_AND_FROZEN projects after",
    22,
    int(
        reg_after[
            status_col
        ].eq(
            COMPLETE_STATUS
        ).sum()
    ),
    int(
        reg_after[
            status_col
        ].eq(
            COMPLETE_STATUS
        ).sum()
    )
    == 22,
)

check(
    rows,
    "Registry Project 22 rows after",
    1,
    len(
        project22_after
    ),
    len(
        project22_after
    )
    == 1,
)

check(
    rows,
    "Registry SHA changed",
    True,
    registry_sha_after
    != registry_sha_before,
    registry_sha_after
    != registry_sha_before,
)

for number in range(
    11,
    22,
):
    check(
        rows,
        f"Registry Project {number} rows after",
        1,
        int(
            after_nums.eq(
                number
            ).sum()
        ),
        int(
            after_nums.eq(
                number
            ).sum()
        )
        == 1,
    )

validation = pd.DataFrame(
    rows
)

failed = validation.loc[
    ~validation[
        "Pass"
    ]
]

if not failed.empty:
    display(
        failed
    )

    raise RuntimeError(
        "PROJECT 22 STEP 5C POST-WRITE VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 16. WRITE STEP 5C AUDIT / CHECKPOINT / STATUS
# --------------------------------------------------------------------------------------------------

STEP5C_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

atomic_csv(
    VALIDATION_PATH,
    validation,
)

registry_rows_after_by_project = {
    f"Project{number}RegistryRowsAfter":
        int(
            after_nums.eq(
                number
            ).sum()
        )
    for number in range(
        11,
        23,
    )
}

report = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5C_STATUS,
    "CompletedAtUTC": created_at,

    "SourceRootSHA256": SOURCE_ROOT_SHA,
    "RawRootSHA256": RAW_ROOT_SHA,

    **COUNTS,

    "FinalPackageRoot": str(
        FINAL_ROOT
    ),
    "FinalPackageFiles": package_files,
    "FinalPackageBytes": package_bytes,
    "FinalPackageRootSHA256": package_root_sha,
    "PackageAlreadyFrozenBeforeThisCell":
        package_already_frozen,

    "PackageMissingFiles": missing_pkg,
    "PackageUnexpectedFiles": unexpected_pkg,
    "PackageSizeMismatches": size_bad,
    "PackageSHA256Mismatches": hash_bad,

    "RegistrySHA256Before": registry_sha_before,
    "RegistrySHA256After": registry_sha_after,
    "RegistryRowsBefore": len(
        reg_before
    ),
    "RegistryRowsAfter": len(
        reg_after
    ),

    **registry_rows_after_by_project,

    "RegistryBackup": str(
        BACKUP_PATH
    ),

    "Step5ACheckpointSHA256": step5a_sha,
    "Step5BCheckpointSHA256": step5b_sha,

    "ParallelWorkerCheckpointSHA256":
        expected_worker_hashes_by_tag,

    "ValidationChecks": len(
        validation
    ),
    "FailedValidationChecks": len(
        failed
    ),

    "ConditionsRerun": False,
    "ModelsFitted": False,
    "RawResultsModified": False,

    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectConditionOutputsModified": False,
    "PriorProjectWriteAttempted": False,
}

atomic_json(
    REPORT_PATH,
    report,
)

atomic_json(
    CHECKPOINT_PATH,
    {
        **report,
        "CheckpointVersion": 1,
        "CheckpointType":
            "PROJECT_22_FINAL_PACKAGE_AND_REGISTRY",
        "FinalPackageFrozen": True,
        "CompletionRegistryUpdated": True,
        "ProjectCompleteAndFrozen": True,
        "NextRequiredStep":
            "PROJECT_23_MAY_START_IN_A_NEW_NOTEBOOK",
    },
)

step5c_sha = sha(
    CHECKPOINT_PATH
)

atomic_json(
    STATUS_PATH,
    {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "Status": STEP5C_STATUS,
        "CompletedAtUTC": created_at,

        "FinalPackageRoot": str(
            FINAL_ROOT
        ),
        "FinalPackageRootSHA256":
            package_root_sha,

        "RegistryRows": len(
            reg_after
        ),
        "RegistrySHA256":
            registry_sha_after,

        "Checkpoint": str(
            CHECKPOINT_PATH
        ),
        "CheckpointSHA256":
            step5c_sha,

        "ProjectCompleteAndFrozen": True,

        "ParallelWorkerCheckpointSHA256":
            expected_worker_hashes_by_tag,

        "PriorProjectConditionOutputsAccessed": False,
        "PriorProjectConditionOutputsModified": False,

        "NextRequiredStep":
            "PROJECT_23_MAY_START_IN_A_NEW_NOTEBOOK",
    },
)


# --------------------------------------------------------------------------------------------------
# 17. FINAL READBACK AND IMMUTABILITY PROOF
# --------------------------------------------------------------------------------------------------

checkpoint_readback = load_json(
    CHECKPOINT_PATH
)

status_readback = load_json(
    STATUS_PATH
)

if (
    checkpoint_readback.get(
        "Status"
    )
    != STEP5C_STATUS
    or status_readback.get(
        "Status"
    )
    != STEP5C_STATUS
):
    raise RuntimeError(
        "Project 22 Step 5C checkpoint/status readback failed."
    )

if not bool(
    checkpoint_readback.get(
        "ProjectCompleteAndFrozen",
        False,
    )
):
    raise RuntimeError(
        "Project 22 checkpoint is not marked complete and frozen."
    )

if not bool(
    checkpoint_readback.get(
        "CompletionRegistryUpdated",
        False,
    )
):
    raise RuntimeError(
        "Project 22 checkpoint is not marked registry-updated."
    )

if not bool(
    checkpoint_readback.get(
        "FinalPackageFrozen",
        False,
    )
):
    raise RuntimeError(
        "Project 22 checkpoint is not marked final-package frozen."
    )

if sha(
    REGISTRY
) != registry_sha_after:
    raise RuntimeError(
        "Completion registry changed after Project 22 finalisation."
    )

if root_hash(
    pd.read_csv(
        MANIFEST_PATH,
        low_memory=False,
    )
) != package_root_sha:
    raise RuntimeError(
        "Final package changed after Project 22 finalisation."
    )

if sha(
    STEP5A_CP
) != STEP5A_SHA_EXPECTED:
    raise RuntimeError(
        "Step 5A checkpoint changed during Project 22 Step 5C."
    )

if sha(
    STEP5B_CP
) != STEP5B_SHA_EXPECTED:
    raise RuntimeError(
        "Step 5B checkpoint changed during Project 22 Step 5C."
    )

for (
    worker_path,
    expected_worker_sha,
) in WORKER_CHECKPOINTS.items():
    if sha(
        worker_path
    ) != expected_worker_sha:
        raise RuntimeError(
            "A frozen Project 22 worker checkpoint changed "
            "during Step 5C."
        )


# --------------------------------------------------------------------------------------------------
# 18. RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 136
)

print(
    "=== PROJECT 22 CELL 11 / STEP 5C RESULT ==="
)

print(
    "=" * 136
)

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print(
    "\nRaw result freeze:"
)

print(
    "Conditions:",
    COUNTS[
        "Conditions"
    ],
)

print(
    "ML fits:",
    COUNTS[
        "MLFits"
    ],
)

print(
    "Raw files:",
    COUNTS[
        "RawFiles"
    ],
)

print(
    "Raw bytes:",
    COUNTS[
        "RawBytes"
    ],
)

print(
    "Raw root SHA-256:",
    RAW_ROOT_SHA,
)

print(
    "\nParallel execution freeze:"
)

print(
    "Worker checkpoints:",
    len(
        expected_worker_hashes_by_tag
    ),
)

print(
    "Worker checkpoint SHA mismatches:",
    worker_checkpoint_sha_mismatches,
)

print(
    "\nFinal package freeze:"
)

print(
    "Package root:",
    FINAL_ROOT,
)

print(
    "Package files:",
    package_files,
)

print(
    "Package bytes:",
    package_bytes,
)

print(
    "Missing package files:",
    missing_pkg,
)

print(
    "Unexpected package files:",
    unexpected_pkg,
)

print(
    "Package size mismatches:",
    size_bad,
)

print(
    "Package SHA-256 mismatches:",
    hash_bad,
)

print(
    "Final package root SHA-256:",
    package_root_sha,
)

print(
    "\nCompletion registry:"
)

print(
    "Registry rows:",
    len(
        reg_after
    ),
)

print(
    "COMPLETE_AND_FROZEN projects:",
    int(
        reg_after[
            status_col
        ].eq(
            COMPLETE_STATUS
        ).sum()
    ),
)

print(
    "Project 22 registry rows:",
    len(
        project22_after
    ),
)

print(
    "Registry SHA-256 before:",
    registry_sha_before,
)

print(
    "Registry SHA-256 after:",
    registry_sha_after,
)

print(
    "Package already frozen before this cell:",
    package_already_frozen,
)

print(
    "\nStep 5C freeze:"
)

print(
    "Validation checks:",
    len(
        validation
    ),
)

print(
    "Failed validation checks:",
    len(
        failed
    ),
)

print(
    "Step 5C checkpoint:",
    CHECKPOINT_PATH,
)

print(
    "Step 5C checkpoint SHA-256:",
    step5c_sha,
)

print(
    "Project complete and frozen:",
    True,
)

print(
    "Next required step:",
    "PROJECT 23 MAY START IN A NEW NOTEBOOK",
)

print(
    "STATUS:",
    STEP5C_STATUS,
)

print(
    "=" * 136
)


=== PROJECT 22 CELL 11 / STEP 5C: FINAL PACKAGE FREEZE AND REGISTRY REGISTRATION ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Project 22 Step 5C pre-write validation:


,Check,Expected,Actual,Pass
0,Step 5A checkpoint SHA-256,600035cfbf9f8b2dfd0fd83d49050466e3c71e6187c1d6...,600035cfbf9f8b2dfd0fd83d49050466e3c71e6187c1d6...,True
1,Step 5B checkpoint SHA-256,9259c486debcda1794423bd16d4333083e14d203216df5...,9259c486debcda1794423bd16d4333083e14d203216df5...,True
2,Source root SHA-256,281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64...,281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64...,True
3,Raw root SHA-256,374e4e1eaab266451539c9c7a4751fa6fe50250dccf57b...,374e4e1eaab266451539c9c7a4751fa6fe50250dccf57b...,True
4,Step 5A manifest failures,0,0,True
5,Step 5B manifest failures,0,0,True
6,Parallel worker checkpoint SHA mismatches,0,0,True
7,Parallel worker checkpoints,6,6,True
8,Step 5B prior-project output access,False,False,True
9,Step 5B prior-project output modification,False,False,True



Project 22 registry row candidate:


,ProjectNumber,Project,ProjectSlug,Status,Conditions,Seeds,NoiseLevels,Techniques,EvaluationBuilds,EvaluationRows,...,FreezeRecord,ChecksumManifest,LastFreezeValidationAtUTC,RawResultsManifest,FinalPackageManifest,RawResultsRootSHA256,FinalPackageRootSHA256,ModelFits,ManifestRowsAudited,FinalAuditStatus
21,22,apache@logging-log4j2,apache__logging-log4j2,COMPLETE_AND_FROZEN,270,30,9,7,111,22156,...,/content/drive/MyDrive/Thesis_Experiment/Notes...,/content/drive/MyDrive/Thesis_Experiment/Resul...,2026-07-25T03:49:02.436302+00:00,/content/drive/MyDrive/Thesis_Experiment/Resul...,/content/drive/MyDrive/Thesis_Experiment/Resul...,374e4e1eaab266451539c9c7a4751fa6fe50250dccf57b...,e1d28d4b3b9ea97fb9c203fad5b4b90af102f41db0b664...,1080,5358150.0,PASS_PROJECT_22_FINAL_PACKAGE_FROZEN_AND_REGIS...



=== PROJECT 22 CELL 11 / STEP 5C RESULT ===
Project number: 22
Project: apache@logging-log4j2
Project slug: apache__logging-log4j2

Raw result freeze:
Conditions: 270
ML fits: 1080
Raw files: 2160
Raw bytes: 646130653
Raw root SHA-256: 374e4e1eaab266451539c9c7a4751fa6fe50250dccf57b19a5bc19cf8fad3f11

Parallel execution freeze:
Worker checkpoints: 6
Worker checkpoint SHA mismatches: 0

Final package freeze:
Package root: /content/drive/MyDrive/Thesis_Experiment/Results/Final/apache__logging-log4j2
Package files: 39
Package bytes: 28220460
Missing package files: 0
Unexpected package files: 0
Package size mismatches: 0
Package SHA-256 mismatches: 0
Final package root SHA-256: e1d28d4b3b9ea97fb9c203fad5b4b90af102f41db0b664f5588adfe91b0d93af

Completion registry:
Registry rows: 22
COMPLETE_AND_FROZEN projects: 22
Project 22 registry rows: 1
Registry SHA-256 before: 79cd6ecb595c5e8ae91a9494e469792716338d144308560a62caf1b9342306b2
Registry SHA-256 after: 914e02a81f9b2f0e51559d8040ac59afc70f7